# Phase 1 — Python Language Fundamentals

**Outcome:** predict the output of any core Python snippet and explain it in terms of
objects, names, types and frames; use every built-in data type and its API fluently.

| Unit | Topic |
| --- | --- |
| 1.1 | How Python executes code |
| 1.2 | Objects, names and mutability |
| 1.3 | Numeric and boolean data types |
| 1.4 | Text and binary data types |
| 1.5 | Collection data types I: lists, tuples and ranges |
| 1.6 | Collection data types II: dictionaries, sets and the `collections` module |
| 1.7 | Copying: shallow copies, deep copies and structured clones |
| 1.8 | Operators, expressions and control flow |
| 1.9 | Iteration, comprehensions and built-in functions |
| 1.10 | Functions: definition and arguments |
| 1.11 | Scope, namespaces and closures |
| 1.12 | Decorators, `functools` and the call stack |
| 1.13 | Iterators, generators and `itertools` |
| 1.14 | Exceptions and context managers |
| Project | Log analyzer |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- *Predict the output* questions hide the answer in a collapsible block: decide first,
  then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check cell.
  Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.15*, …).

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 1.1 — How Python Executes Code

**Objectives**

- Explain what "compiled" and "interpreted" each mean for CPython.
- Follow one line of source through every stage: tokens, AST, symbol table, code
  object, bytecode, evaluation.
- Find and recognise the cached bytecode files Python writes in `__pycache__`.
- Tell statements from expressions and read Python's block syntax.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| V8 Ignition bytecode | CPython bytecode (`dis.dis`) | You can print CPython's bytecode from inside the program |
| TurboFan optimising JIT | Specializing adaptive interpreter (3.11+) and experimental JIT (3.13+) | CPython's JIT is off by default; most speed-ups come from specialisation |
| Esprima / acorn AST | `ast` module (built in) | The AST is part of the standard library |
| V8 code cache | `__pycache__/*.pyc` | The cache is a file next to your source, one per interpreter version |
| `{ … }` blocks, semicolons | Indentation, newlines | Indentation is syntax, not style |

In [2]:
import ast
import dis
import importlib
import importlib.util
import io
import os
import subprocess
import symtable
import sys
import tokenize
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


repo_root = find_repo_root()
scratch_dir = repo_root / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{sys.implementation.name=}, {sys.version_info[:2]=}")

sys.implementation.name='cpython', sys.version_info[:2]=(3, 15)


### 1.1.1 Compiled and interpreted

CPython does both. It **compiles** the whole source file to **bytecode** (instructions
for an imaginary machine), then **interprets** that bytecode in a loop. The imaginary
machine is a **stack-based virtual machine**: instructions push values onto a stack,
and operators pop their operands off it and push the result.

The cell disassembles a one-line function. Read it as a stack machine:

| Instruction | Stack after it |
| --- | --- |
| load `price` | `[price]` |
| load `tax` | `[price, tax]` |
| `BINARY_OP (+)` | `[price + tax]` |
| `RETURN_VALUE` | `[]` (value returned) |

> **Note:** exact instruction names change between Python versions (3.15 fuses the two
> loads into one `LOAD_FAST_BORROW_LOAD_FAST_BORROW` instruction). The shape stays the same.

In [3]:
def add_tax(price: int, tax: int) -> int:
    """Return price plus tax."""
    return price + tax


dis.dis(add_tax)

  1           RESUME                   0

  3           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (price, tax)
              BINARY_OP                0 (+)
              RETURN_VALUE


**Specializing adaptive interpreter (new in 3.11, PEP 659).** After a function has run
a few times, CPython rewrites ("quickens") its bytecode in place with versions
specialised for the types it actually saw. `BINARY_OP (+)` becomes
`BINARY_OP_ADD_INT` when both operands were always `int`. If a different type turns up,
the instruction falls back to the generic version.

`dis.dis(..., adaptive=True)` shows the specialised bytecode. The cell calls `add_tax`
1000 times with integers first.

In [4]:
for _ in range(1_000):
    add_tax(100, 18)

dis.dis(add_tax, adaptive=True)

  1           RESUME_CHECK             0

  3           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (price, tax)
              BINARY_OP_ADD_INT        0 (+)
              RETURN_VALUE


**JIT compiler (experimental, new in 3.13; faster in 3.14 and 3.15, PEP 744).** On top
of specialisation, CPython can compile hot traces of specialised bytecode to machine
code. The JIT is built in to the uv builds of 3.15 but **off by default**; set the
environment variable `PYTHON_JIT=1` to turn it on. `sys._jit` (3.14+) reports the state.

The cell checks this kernel, then starts a child interpreter with `PYTHON_JIT=1`.
Interpreters built without the JIT print a skip message.

In [5]:
if hasattr(sys, "_jit"):
    print(f"{sys._jit.is_available()=}  (built with the JIT)")
    print(f"{sys._jit.is_enabled()=}  (turned on in this kernel)")
    child_result = subprocess.run(
        [sys.executable, "-c", "import sys; print(sys._jit.is_enabled())"],
        env={**os.environ, "PYTHON_JIT": "1"},
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"with PYTHON_JIT=1: is_enabled() -> {child_result.stdout.strip()}")
else:
    print("Skipped: sys._jit needs Python 3.14+")

sys._jit.is_available()=True  (built with the JIT)
sys._jit.is_enabled()=False  (turned on in this kernel)
with PYTHON_JIT=1: is_enabled() -> True


> **JS bridge:** V8 works the same way in outline. **Ignition** compiles JavaScript to
> bytecode and interprets it; **TurboFan** compiles hot functions to optimised machine
> code using type feedback. CPython's *adaptive interpreter* plays the role of
> Ignition's type feedback, and its JIT is a much younger, simpler TurboFan.

### 1.1.2 The pipeline

```mermaid
flowchart LR
    source["source text"] --> tokenizer["tokenizer"]
    tokenizer -- "tokens" --> parser["PEG parser"]
    parser -- "AST" --> compiler["compiler + symbol table"]
    compiler -- "code object (bytecode)" --> eval_loop["eval loop (VM)"]
    eval_loop --> result["side effects / result"]
```

<details><summary>ASCII fallback</summary>

```text
source --tokenize--> tokens --PEG parser--> AST --compiler (symbol table)--> code object --eval loop--> result
```

</details>

| Stage | What it produces | Module to look at it |
| --- | --- | --- |
| Tokenizer | `NAME`, `OP`, `NUMBER`… tokens | `tokenize` |
| PEG parser (3.9+, PEP 617) | Abstract syntax tree | `ast.parse`, `ast.dump` |
| Symbol table | Which names are local, global, free | `symtable` |
| Compiler | A **code object**: bytecode + constants + names | `compile`, `code.co_*` |
| Eval loop | Runs the bytecode on the VM | `exec`, `eval`, `dis` |

**Stage 1: tokenizer.** It splits source text into tokens.

In [6]:
source_line = "total_in_rupees = price * 2 + 18\n"
for token in tokenize.generate_tokens(io.StringIO(source_line).readline):
    print(f"{tokenize.tok_name[token.type]:<10} {token.string!r}")

NAME       'total_in_rupees'
OP         '='
NAME       'price'
OP         '*'
NUMBER     '2'
OP         '+'
NUMBER     '18'
NEWLINE    '\n'
ENDMARKER  ''


**Stage 2: parser.** It builds an **abstract syntax tree** from the tokens.
`ast.dump(ast.parse(...))` prints the tree. Notice that operator precedence is already
resolved: `price * 2` is nested inside the `+`.

In [7]:
print(ast.dump(ast.parse(source_line), indent=2))

Module(
  body=[
    Assign(
      targets=[
        Name(id='total_in_rupees', ctx=Store())],
      value=BinOp(
        left=BinOp(
          left=Name(id='price'),
          op=Mult(),
          right=Constant(value=2)),
        op=Add(),
        right=Constant(value=18)))])


**Stage 3: symbol table.** Before generating bytecode, the compiler decides the scope of
every name. Inside a function, a name that is assigned is **local**; a name that is only
read is **global** (or a closure variable). This is decided at compile time, which is
why Unit 1.11's `UnboundLocalError` exists.

In [8]:
function_source = """
def add_gst(price):
    taxed_price = price * gst_rate
    return taxed_price
"""
module_table = symtable.symtable(function_source, "<demo>", "exec")
function_table = module_table.lookup("add_gst").get_namespace()
print(f"{function_table.get_parameters()=}")
print(f"{function_table.get_locals()=}")
print(f"{function_table.get_globals()=}")

function_table.get_parameters()=('price',)
function_table.get_locals()=('price', 'taxed_price')
function_table.get_globals()=('gst_rate',)


**Stage 4: compiler.** `compile(source, filename, mode)` returns a **code object**. The
mode decides what source is allowed: `"exec"` for statements (a module), `"eval"` for a
single expression. The code object holds the bytecode (`co_code`) and the tables it
refers to.

In [9]:
module_code = compile(source_line, "<demo>", "exec")
expression_code = compile("price * 2 + 18", "<demo>", "eval")
print(f"{type(module_code).__name__=}")
print(f"{module_code.co_names=}")
print(f"{module_code.co_consts=}")
print(f"{len(module_code.co_code)=} bytes of bytecode")
print(f"{expression_code.co_names=}")

type(module_code).__name__='code'
module_code.co_names=('price', 'total_in_rupees')
module_code.co_consts=(2,)
len(module_code.co_code)=40 bytes of bytecode
expression_code.co_names=('price',)


`dis.dis` turns the code object's bytes back into readable instructions. At module level,
names are stored with `STORE_NAME` (a dict write) instead of `STORE_FAST` (a slot in the
frame), which is one reason code inside functions runs faster.

In [10]:
dis.dis(module_code)

  0           RESUME                   0

  1           LOAD_NAME                0 (price)
              LOAD_SMALL_INT           2
              BINARY_OP                5 (*)
              LOAD_SMALL_INT          18
              BINARY_OP                0 (+)
              STORE_NAME               1 (total_in_rupees)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


**Stage 5: eval loop.** `exec(code, namespace)` runs a code object with a dict you
supply as its global namespace. The assignment lands in that dict, not in the notebook's
globals. `eval` runs an expression and returns its value.

In [11]:
demo_namespace: dict[str, object] = {"price": 500}
exec(module_code, demo_namespace)
print(f"{demo_namespace['total_in_rupees']=}")
print(f"{'total_in_rupees' in globals()=}")
print(f"{eval(expression_code, {'price': 10})=}")

demo_namespace['total_in_rupees']=1018
'total_in_rupees' in globals()=False
eval(expression_code, {'price': 10})=38


> **Warning:** never `exec` or `eval` text that came from a user. It runs with the
> full power of your program. Use `ast.literal_eval` to parse literals safely.

### 1.1.3 Bytecode caching

Compiling is not free, so when a module is **imported**, CPython saves the code object
to `__pycache__/<module>.cpython-315.pyc` next to the source. The next import loads
the `.pyc` directly, unless the source changed. (The script you run directly, and
notebook cells, are never cached.)

The cell writes a small module into `notebooks/_scratch/` and imports it.
`importlib.util.cache_from_source` computes where its cache file goes.

In [12]:
module_path = scratch_dir / "price_rules.py"
module_path.write_text(
    "GST_RATE = 18\n\n\ndef with_gst(price: int) -> float:\n"
    '    """Return price including GST."""\n'
    "    return price * (100 + GST_RATE) / 100\n",
    encoding="utf-8",
)
if str(scratch_dir) not in sys.path:
    sys.path.insert(0, str(scratch_dir))
importlib.invalidate_caches()
price_rules = importlib.import_module("price_rules")  # same as: import price_rules

cache_path = Path(importlib.util.cache_from_source(str(module_path)))
print(f"{price_rules.with_gst(1000)=}")
print(f"{cache_path.relative_to(scratch_dir)=}")
print(f"{cache_path.exists()=}")

price_rules.with_gst(1000)=1180.0
cache_path.relative_to(scratch_dir)=PosixPath('__pycache__/price_rules.cpython-315.pyc')
cache_path.exists()=True


Every `.pyc` starts with a 4-byte **magic number** that changes whenever the bytecode
format changes. A `.pyc` with the wrong magic number is ignored and rebuilt, which is
why each Python version gets its own file name tag (`sys.implementation.cache_tag`).
The next 12 bytes hold flags and the source's modification time and size.

In [13]:
pyc_header = cache_path.read_bytes()[:16]
print(f"{importlib.util.MAGIC_NUMBER=}")
print(f"{pyc_header[:4]=}")
print(f"{pyc_header[:4] == importlib.util.MAGIC_NUMBER=}")
print(f"{sys.implementation.cache_tag=}")

importlib.util.MAGIC_NUMBER=b'R\x0e\r\n'
pyc_header[:4]=b'R\x0e\r\n'
pyc_header[:4] == importlib.util.MAGIC_NUMBER=True
sys.implementation.cache_tag='cpython-315'


> **Note:** `__pycache__/` belongs in `.gitignore` (this repository has it). Set
> `PYTHONDONTWRITEBYTECODE=1` to stop Python writing caches, e.g. in read-only containers.

### 1.1.4 Statements vs expressions

An **expression** produces a value (`price * 2`, `f(x)`, `a if c else b`). A
**statement** does something (`x = 1`, `if`, `for`, `def`, `import`, `return`). Every
expression can be used as a statement, but not the reverse: `eval` accepts only
expressions. The AST labels each kind.

In [14]:
sample_module = ast.parse("discount = 10\nprice * 2\nif discount:\n    pass")
for node in sample_module.body:
    print(f"{type(node).__name__:<6} {ast.unparse(node)!r}")

try:
    eval("discount = 10")
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

Assign 'discount = 10'
Expr   'price * 2'
If     'if discount:\n    pass'
SyntaxError: invalid syntax


**Indentation is block syntax.** A colon opens a block, and the indented lines below it
form the body; there are no braces. PEP 8 uses 4 spaces. Wrong indentation is a
*compile-time* error (`IndentationError`, a subclass of `SyntaxError`), so the cell
compiles bad source as a string to catch it.

In [15]:
badly_indented_source = "if True:\nprint('missing indent')\n"
try:
    compile(badly_indented_source, "<demo>", "exec")
except IndentationError as error:
    print(f"{type(error).__name__}: {error.msg} (line {error.lineno})")
print(f"{issubclass(IndentationError, SyntaxError)=}")

IndentationError: expected an indented block after 'if' statement on line 1 (line 2)
issubclass(IndentationError, SyntaxError)=True


**`pass`** is the statement that does nothing. Python needs it where a block is required
but you have nothing to put there yet (JS would use `{}`).

In [16]:
def not_written_yet() -> None:
    """Placeholder whose body is intentionally empty."""
    pass


class EmptyMarker:
    pass


print(f"{not_written_yet()=}, {EmptyMarker.__name__=}")

not_written_yet()=None, EmptyMarker.__name__='EmptyMarker'


**Line continuation.** A newline ends a statement *unless* it is inside `()`, `[]` or
`{}`. Wrap long expressions in brackets instead of ending a line with a backslash:

```python
order_total = 499 + \
    150          # explicit continuation: breaks if a space follows the backslash
```

`ruff format` (and Black) never produce backslash continuations; they add parentheses.

In [17]:
line_item_prices = [
    499,
    150,
    1_200,
]
order_total = (
    sum(line_item_prices)
    + 40  # delivery charge
    - 100  # coupon
)
print(f"{order_total=}, {line_item_prices=}")

order_total=1789, line_item_prices=[499, 150, 1200]


#### Under the hood: one code object per function

A function is a thin wrapper around a code object compiled once, when its `def` ran.
`__code__` exposes it, with the variable names and constants the bytecode uses.

In [18]:
add_tax_code = add_tax.__code__
print(f"{add_tax_code.co_name=}")
print(f"{add_tax_code.co_varnames=}")
print(f"{add_tax_code.co_argcount=}")
print(f"{add_tax_code.co_consts=}")
print(f"{add_tax.__code__ is add_tax_code=} (compiled once, reused on every call)")

add_tax_code.co_name='add_tax'
add_tax_code.co_varnames=('price', 'tax')
add_tax_code.co_argcount=2
add_tax_code.co_consts=('Return price plus tax.',)
add_tax.__code__ is add_tax_code=True (compiled once, reused on every call)


#### Predict the output

**Predict the output.**

```python
namespace = {}
exec("x = 2\ny = x * 21", namespace)
print(namespace["y"], "y" in globals())
```

<details><summary>Answer</summary>

`42 False`. `exec` used `namespace` as the global namespace of the code it ran, so both
`x` and `y` were stored there and not in the notebook's globals.

</details>

Run the next cell to check your prediction.

In [19]:
namespace: dict[str, object] = {}
exec("x = 2\ny = x * 21", namespace)
print(namespace["y"], "y" in globals())

42 False


**Predict the output.**

```python
print(type(ast.parse("1 + 2").body[0]).__name__)
```

<details><summary>Answer</summary>

`Expr`. `ast.parse` defaults to `"exec"` mode, so the source is a module whose body is a
list of **statements**. A bare expression becomes an *expression statement* (`Expr`)
that wraps the `BinOp`.

</details>

Run the next cell to check your prediction.

In [20]:
print(type(ast.parse("1 + 2").body[0]).__name__)
print(type(ast.parse("1 + 2").body[0].value).__name__)

Expr
BinOp


### Recap — Unit 1.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Compiled *and* interpreted | Source → bytecode once, then a stack VM runs it | V8 Ignition |
| Adaptive interpreter | Hot bytecode is specialised for the types it sees (3.11+) | Inline caches / type feedback |
| JIT | Experimental, off by default, enable with `PYTHON_JIT=1` (3.13+) | TurboFan |
| Pipeline | tokenize → PEG parser → AST → symbol table → code object → eval loop | parse → AST → bytecode → execute |
| `exec` / `eval` | Run code in a namespace dict you control; never on untrusted input | `eval`, `new Function` |
| `__pycache__` | Imported modules cache bytecode in `.pyc`, checked by magic number | V8 code cache |
| Statement vs expression | Only expressions have values; `eval` accepts only expressions | Same split |
| Blocks | Indentation + colon; `pass` for an empty body; wrap long lines in brackets | `{ }` and `;` |

**Further reading**

- [`dis` — Disassembler](https://docs.python.org/3/library/dis.html),
  [`ast`](https://docs.python.org/3/library/ast.html),
  [`symtable`](https://docs.python.org/3/library/symtable.html),
  [`tokenize`](https://docs.python.org/3/library/tokenize.html)
- [Built-in `compile`, `exec`, `eval`](https://docs.python.org/3/library/functions.html#compile),
  [`importlib.util.cache_from_source`](https://docs.python.org/3/library/importlib.html#importlib.util.cache_from_source)
- [Language reference: lexical analysis (indentation, line joining)](https://docs.python.org/3/reference/lexical_analysis.html)
- [PEP 617 — PEG parser](https://peps.python.org/pep-0617/),
  [PEP 659 — Specializing adaptive interpreter](https://peps.python.org/pep-0659/),
  [PEP 744 — JIT compilation](https://peps.python.org/pep-0744/),
  [PEP 3147 — `__pycache__` directories](https://peps.python.org/pep-3147/)

## Unit 1.2 — Objects, Names and Mutability

**Objectives**

- Describe every value as an object with an identity, a type and a value.
- Explain variables as names bound to objects in a namespace dict.
- Predict when a function call changes the caller's data.
- Classify built-in types as mutable or immutable, and use `is` vs `==` correctly.
- Explain why names have no type but objects do.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| Primitives (`number`, `string`) vs objects | Everything is an object | `int` and `str` are objects too, just immutable ones |
| `let x = obj` copies a reference | `x = obj` binds a name | Same idea, but it applies to *every* value |
| `===` | `==` (value) and `is` (identity) | `is` on numbers or strings depends on caching; never use it for that |
| `Object.is(a, b)` | `a is b` | `is` compares identity, not value |
| `typeof x` | `type(x)` / `isinstance(x, T)` | `isinstance` respects subclasses |
| `delete obj.key` | `del name` | `del` unbinds a name; the object lives on while referenced |

The next cell is this unit's setup.

In [21]:
import inspect
import math
import sys
import types
import warnings


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.2.1 Everything is an object

Every object has three things:

| Property | How to read it | Can it change? |
| --- | --- | --- |
| **Identity** | `id(obj)` (CPython: the memory address) | Never, for the object's lifetime |
| **Type** | `type(obj)` | Never (in practice) |
| **Value** | the object itself, `repr(obj)` | Only for mutable types |

> **Note:** `id()` values differ on every run. The demos compare ids; they never rely on
> the number itself.

In [22]:
price_in_rupees = 499
cart_items = ["pen", "notebook"]
for value in (price_in_rupees, "Asha", cart_items, 3.5, None):
    print(f"type={type(value).__name__:<8} value={value!r:<20} has an id: {id(value) > 0}")

type=int      value=499                  has an id: True
type=str      value='Asha'               has an id: True
type=list     value=['pen', 'notebook']  has an id: True
type=float    value=3.5                  has an id: True
type=NoneType value=None                 has an id: True


Types, functions, modules and classes are objects too: you can put them in a list, pass
them to functions and give them attributes. Even `int` is an object, whose type is
`type`.

In [23]:
def apply_discount(amount: float) -> float:
    """Return the amount with a 10% discount."""
    return amount * 0.9


first_class_values = [int, apply_discount, math, type, len]
for value in first_class_values:
    print(f"{getattr(value, '__name__', value)!s:<16} is a {type(value).__name__}")

print(f"{type(int) is type=}, {type(type) is type=}")
print(f"{inspect.isclass(int)=}, {inspect.isfunction(apply_discount)=}, {inspect.ismodule(math)=}")
apply_discount.author = "Asha"  # functions accept new attributes like any object
print(f"{apply_discount.author=}")

int              is a type
apply_discount   is a function
math             is a module
type             is a type
len              is a builtin_function_or_method
type(int) is type=True, type(type) is type=True
inspect.isclass(int)=True, inspect.isfunction(apply_discount)=True, inspect.ismodule(math)=True
apply_discount.author='Asha'


### 1.2.2 Names and binding

A variable is a **name**: a key in a namespace dict, bound to an object. At module level
(and in a notebook) that dict is `globals()`. Writing to the dict is the same as
assigning a variable.

In [24]:
delivery_fee = 40
module_namespace = globals()
print(f"{type(module_namespace).__name__=}")
print(f"{module_namespace['delivery_fee']=}")

module_namespace["service_fee"] = 15  # same effect as: service_fee = 15
print(f"{service_fee=}")  # noqa: F821  (created through the dict, invisible to linters)

type(module_namespace).__name__='dict'
module_namespace['delivery_fee']=40
service_fee=15


**Assignment never copies.** `b = a` binds a second name to the *same* object. If the
object is mutable, a change through one name is visible through the other.

```mermaid
flowchart LR
    subgraph names ["globals()"]
        morning["morning_orders"]
        alias["evening_orders"]
    end
    list_object["list object<br/>['idli', 'dosa', 'vada']"]
    morning --> list_object
    alias --> list_object
```

<details><summary>ASCII fallback (Python Tutor style)</summary>

```text
Frames            Objects
+----------------+
| morning_orders | ----+
| evening_orders | ----+--> list ['idli', 'dosa', 'vada']
+----------------+
```

</details>

In [25]:
morning_orders = ["idli", "dosa"]
evening_orders = morning_orders  # a second name, not a second list
evening_orders.append("vada")
print(f"{morning_orders=}")
print(f"{evening_orders is morning_orders=}")

morning_orders=['idli', 'dosa', 'vada']
evening_orders is morning_orders=True


**`del` removes a name, not the object.** The object survives as long as something
still refers to it. `sys.getrefcount` counts the references (it reports one extra for
its own argument). CPython frees the object when the count reaches zero.

In [26]:
reference_count_before = sys.getrefcount(morning_orders)
del morning_orders
print(f"{'morning_orders' in globals()=}")
print(f"{evening_orders=}  (the list is still alive)")
print(f"{sys.getrefcount(evening_orders) == reference_count_before - 1=}")
try:
    print(morning_orders)  # noqa: F821
except NameError as error:
    print(f"{type(error).__name__}: {error}")

'morning_orders' in globals()=False
evening_orders=['idli', 'dosa', 'vada']  (the list is still alive)
sys.getrefcount(evening_orders) == reference_count_before - 1=True
NameError: name 'morning_orders' is not defined


### 1.2.3 Call by object reference

Calling a function binds each parameter name to the **same object** the caller passed
(sometimes called *call by sharing*). Two consequences:

| Inside the function | Effect on the caller |
| --- | --- |
| **Mutate** the object (`items.append(...)`) | Visible: it is the same object |
| **Rebind** the parameter (`items = [...]`) | Invisible: only the local name moved |

In [27]:
def add_free_gift(order_items: list[str]) -> None:
    """Mutate the caller's list by appending a gift."""
    order_items.append("free sticker")


def replace_order(order_items: list[str]) -> None:
    """Rebind the local name only; the caller's list is untouched."""
    order_items = ["something else"]
    print(f"  inside replace_order: {order_items=}")


customer_order = ["tea"]
add_free_gift(customer_order)
print(f"after add_free_gift: {customer_order=}")
replace_order(customer_order)
print(f"after replace_order: {customer_order=}")

after add_free_gift: customer_order=['tea', 'free sticker']
  inside replace_order: order_items=['something else']
after replace_order: customer_order=['tea', 'free sticker']


The same rule applies to immutable arguments: `count += 1` inside a function rebinds the
local name to a new `int`, so the caller sees no change. Return the new value instead.

> **JS bridge:** Python has **no primitives**. In JS, numbers and strings are passed by
> value and objects by reference-copy. In Python *everything* is passed the same way;
> `int` and `str` only *look* like they are passed by value because they cannot be
> mutated.

In [28]:
def increment(counter: int) -> int:
    """Return counter + 1; rebinding the parameter does not touch the caller."""
    counter += 1
    return counter


visit_count = 10
new_visit_count = increment(visit_count)
print(f"{visit_count=}, {new_visit_count=}")

visit_count=10, new_visit_count=11


### 1.2.4 Mutable vs immutable types

| Immutable (value fixed at creation) | Mutable (value can change in place) |
| --- | --- |
| `int`, `float`, `complex`, `bool` | `list` |
| `decimal.Decimal`, `fractions.Fraction` | `dict` |
| `str` | `set` |
| `bytes` | `bytearray` |
| `tuple` (its items may be mutable) | `memoryview` over a writable buffer |
| `frozenset` | `collections.deque`, `array.array` |
| `frozendict` (new in 3.15) | instances of most user-defined classes |
| `range`, `NoneType`, functions' code objects | `types.SimpleNamespace` |

"Changing" an immutable object really creates a **new** object and rebinds the name.
Watch the identity across `+=` (comparing ids, never printing them):

In [29]:
stock_count = 1_000
id_before = id(stock_count)
stock_count += 1
print(f"int:  same object after += ? {id(stock_count) == id_before}")

greeting_text = "Namaste"
id_before = id(greeting_text)
greeting_text += ", Asha"
print(f"str:  same object after += ? {id(greeting_text) == id_before}")

basket = ["rice"]
id_before = id(basket)
basket += ["dal"]  # list += extends in place
print(f"list: same object after += ? {id(basket) == id_before}")

int:  same object after += ? False
str:  same object after += ? False
list: same object after += ? True


The rule of thumb: an immutable type has no method that modifies it. `str.upper()`
returns a new string; `list.sort()` changes the list and returns `None`. Trying to
assign into an immutable object raises `TypeError`.

In [30]:
coordinates = (12.97, 77.59)
try:
    coordinates[0] = 13.0
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

city = "bengaluru"
print(f"{city.upper()=}, {city=}  (unchanged)")
print(f"{basket.sort()=}  (sorted in place, returns None)")

TypeError: 'tuple' object does not support item assignment
city.upper()='BENGALURU', city='bengaluru'  (unchanged)
basket.sort()=None  (sorted in place, returns None)


### 1.2.5 Identity vs equality

| Operator | Question | Implemented by |
| --- | --- | --- |
| `a == b` | Do they have equal **values**? | `a.__eq__(b)`, which each type defines |
| `a is b` | Are they the **same object**? | Identity comparison; cannot be overridden |

Use `is` only for singletons: `None`, `True`, `False`, `...`, and sentinel objects you
create yourself. Use `==` for everything else.

In [31]:
class Price:
    """An amount in paise that compares by value."""

    def __init__(self, paise: int) -> None:
        self.paise = paise

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Price):
            return NotImplemented
        return self.paise == other.paise


first_price = Price(49_900)
second_price = Price(49_900)
print(f"{first_price == second_price=}  (__eq__ compares paise)")
print(f"{first_price is second_price=}  (two distinct objects)")
first_list, second_list = [1, 2], [1, 2]
print(f"{first_list == second_list=}, {first_list is second_list=}")

first_price == second_price=True  (__eq__ compares paise)
first_price is second_price=False  (two distinct objects)
first_list == second_list=True, first_list is second_list=False


**Small-integer cache.** CPython pre-creates small `int` objects and reuses them, so two
names bound to the same small value can be the *same object*. The range changed:

| Python | Cached range |
| --- | --- |
| up to 3.14 | `-5` … `256` |
| **3.15** (*changed in 3.15*) | `-5` … `1024` |

The cell builds numbers at run time with `int("…")`. Literals in one cell would be
merged into a single constant by the compiler and hide the effect. This is an
implementation detail, which is exactly why you must not use `is` for numbers.

In [32]:
for number_text in ["256", "257", "1024", "1025"]:
    first_number, second_number = int(number_text), int(number_text)
    is_same_object = first_number is second_number
    print(f"{number_text:>5}: equal={first_number == second_number}, same object={is_same_object}")
if sys.version_info < (3, 15):
    print("Note: on this Python the cache ends at 256, so 1024 is not shared")

  256: equal=True, same object=True
  257: equal=True, same object=True
 1024: equal=True, same object=True
 1025: equal=True, same object=False


**String interning.** CPython also shares some strings: identifier-like literals and
names are *interned* automatically. Strings built at run time usually are not.
`sys.intern` interns one on request, which can save memory and makes later comparisons
identity checks (used inside dicts of attribute names).

In [33]:
built_status = "".join(["sh", "ipped"])
literal_status = "shipped"
print(f"{built_status == literal_status=}, {built_status is literal_status=}")

interned_status = sys.intern(built_status)
print(f"{interned_status is literal_status=}  (sys.intern returned the shared copy)")

built_status == literal_status=True, built_status is literal_status=False
interned_status is literal_status=True  (sys.intern returned the shared copy)


Because results like these vary by version and context, the compiler warns when you use
`is` with a literal. The cell compiles such code and captures the **`SyntaxWarning`**
(new in 3.8) instead of letting it print.

In [34]:
suspicious_source = "order_count = 5\nprint(order_count is 5)"
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    compile(suspicious_source, "<demo>", "exec")
for caught_warning in caught_warnings:
    print(f"{caught_warning.category.__name__}: {caught_warning.message}")

### 1.2.6 Dynamic typing

**Names have no types; objects do.** One name can be rebound to objects of different
types. Type hints (`price: int`) document intent and help tools such as pyright, but
the interpreter does not enforce them (Phase 4).

In [35]:
shipment_reference = 1042
print(f"{type(shipment_reference).__name__=}")
shipment_reference = "SHIP-1042"
print(f"{type(shipment_reference).__name__=}")

hinted_quantity: int = "three"  # legal at run time: hints are not checked
print(f"{hinted_quantity=}, {type(hinted_quantity).__name__=}")

type(shipment_reference).__name__='int'
type(shipment_reference).__name__='str'
hinted_quantity='three', type(hinted_quantity).__name__='str'


**`type()` vs `isinstance()`.** `type(x) is T` matches one exact class.
`isinstance(x, T)` also accepts subclasses, and takes a tuple (or a `X | Y` union) of
types. Prefer `isinstance` for checks.

In [36]:
class DiscountedPrice(Price):
    """A Price subclass."""


sale_price = DiscountedPrice(39_900)
print(f"{type(sale_price) is Price=}")
print(f"{isinstance(sale_price, Price)=}")
print(f"{isinstance(3.5, int | float)=}")
print(f"{isinstance(True, int)=}  (bool is a subclass of int; see 1.3.6)")

type(sale_price) is Price=False
isinstance(sale_price, Price)=True
isinstance(3.5, int | float)=True
isinstance(True, int)=True  (bool is a subclass of int; see 1.3.6)


**Duck typing preview.** Python code usually does not check types at all: it calls the
method it needs and lets anything that has it work ("if it walks like a duck…").
Phase 2 formalises this with protocols.

In [37]:
class PaperReceipt:
    """A receipt that is printed."""

    def render(self) -> str:
        """Return a description of the receipt."""
        return "Printed receipt"


class EmailReceipt:
    """A receipt that is emailed; unrelated to PaperReceipt."""

    def render(self) -> str:
        """Return a description of the receipt."""
        return "Emailed receipt"


def send_receipt(receipt: PaperReceipt | EmailReceipt) -> None:
    """Works with any object that has a render() method."""
    print(receipt.render())


ad_hoc_receipt = types.SimpleNamespace(render=lambda: "Ad-hoc receipt")
for receipt in (PaperReceipt(), EmailReceipt(), ad_hoc_receipt):
    send_receipt(receipt)

Printed receipt
Emailed receipt
Ad-hoc receipt


#### Under the hood: names live in `__dict__`, objects carry headers

A module's globals are literally its `__dict__`, and an instance's attributes are in its
own `__dict__`. Every object also carries a header (reference count and type pointer),
which is why even `0` costs bytes. `show_size` (from the shared setup) prints the
shallow size.

In [38]:
print(f"{globals() is sys.modules['__main__'].__dict__=}")
print(f"{vars(first_price)=}")
show_size("small int 0", 0)
show_size("None", None)
show_size("empty list", [])

globals() is sys.modules['__main__'].__dict__=True
vars(first_price)={'paise': 49900}
small int 0: 28 bytes, type=int, id=22124928
None: 16 bytes, type=NoneType, id=22109848
empty list: 56 bytes, type=list, id=140266921353408


#### Predict the output

**Predict the output.**

```python
shopping_list = ["milk"]
other_list = shopping_list
other_list = other_list + ["bread"]
print(shopping_list)
```

<details><summary>Answer</summary>

`['milk']`. `other_list + [...]` builds a **new** list and rebinds `other_list` to it;
`shopping_list` still points at the original. Compare `basket += [...]` in 1.2.4,
which extends the list in place.

</details>

Run the next cell to check your prediction.

In [39]:
shopping_list = ["milk"]
other_list = shopping_list
other_list = other_list + ["bread"]
print(shopping_list)

['milk']


**Predict the output.**

```python
def reset(scores):
    scores.clear()
    scores = [100]

team_scores = [7, 9]
reset(team_scores)
print(team_scores)
```

<details><summary>Answer</summary>

`[]`. `clear()` mutates the caller's list; the rebinding on the next line only moves
the local name.

</details>

Run the next cell to check your prediction.

In [40]:
def reset(scores: list[int]) -> None:
    """Clear the caller's list, then rebind the local name."""
    scores.clear()
    scores = [100]


team_scores = [7, 9]
reset(team_scores)
print(team_scores)

[]


### Recap — Unit 1.2

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Everything is an object | Identity, type and value; classes, functions, modules too | Objects, but JS also has primitives |
| Names and binding | A variable is a key in a namespace dict bound to an object | Variable holds a reference |
| Assignment | Never copies; two names can share one object | Same for objects |
| `del` | Removes a name; the object dies when unreferenced | `delete` removes a property |
| Call by object reference | Mutation is visible to the caller, rebinding is not | Same for objects; primitives copy |
| Mutable vs immutable | `list`/`dict`/`set` change in place; `int`/`str`/`tuple` never do | Objects vs primitives |
| `is` vs `==` | `is` for singletons only; small-int cache is `-5..1024` in 3.15 | `Object.is` vs `===` |
| Dynamic typing | Names untyped, objects typed; prefer `isinstance` and duck typing | `typeof`, `instanceof` |

**Further reading**

- [Data model: objects, values and types](https://docs.python.org/3/reference/datamodel.html#objects-values-and-types)
- [Naming and binding](https://docs.python.org/3/reference/executionmodel.html#naming-and-binding)
- [Programming FAQ: how do I write a function with output parameters?](https://docs.python.org/3/faq/programming.html#how-do-i-write-a-function-with-output-parameters-call-by-reference)
- [`sys.intern`](https://docs.python.org/3/library/sys.html#sys.intern),
  [`isinstance`](https://docs.python.org/3/library/functions.html#isinstance),
  [`id`](https://docs.python.org/3/library/functions.html#id)
- [PEP 8 — comparisons to singletons](https://peps.python.org/pep-0008/#programming-recommendations)

## Unit 1.3 — Numeric and Boolean Data Types

**Objectives**

- Use `int`, `float`, `Decimal`, `Fraction` and `complex`, and choose the right one for
  money, measurements and exact ratios.
- Predict `//`, `%` and `round()` results, including negative numbers and halves.
- Explain why `bool` is an `int`, what `None` is, and which values are falsy.
- Convert between types explicitly, knowing what raises.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `number` (one type) | `int` and `float` (two types) | `7 / 2 == 3.5`, but `7 // 2 == 3`; ints never lose precision |
| `BigInt` (`10n`) | `int` | Every `int` is a big int; no suffix, no mixing errors |
| `-7 % 2 === -1` | `-7 % 2 == 1` | `%` takes the sign of the divisor |
| `Math.round(2.5) === 3` | `round(2.5) == 2` | Banker's rounding (round half to even) |
| `1 / 0 === Infinity` | `1 / 0` raises `ZeroDivisionError` | Division by zero is an error |
| `null` and `undefined` | `None` only | Missing dict keys raise `KeyError` |
| `[]` and `{}` are truthy | `[]` and `{}` are falsy | Empty containers are false |
| `1 + "1" === "11"` | `TypeError` | No implicit coercion |

The next cell is this unit's setup.

In [41]:
import cmath
import math
import sys
import timeit
from collections.abc import Callable
from decimal import ROUND_HALF_EVEN, ROUND_HALF_UP, Decimal, getcontext, localcontext
from fractions import Fraction


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


PAISE = Decimal("0.01")  # quantize() target for rupee amounts
print(f"{sys.version_info[:2]=}, {getcontext().prec=}")

sys.version_info[:2]=(3, 15), getcontext().prec=28


### 1.3.0 Python's built-in data types at a glance

Units 1.3–1.7 teach these types:

| Category | Type | Mutable | Ordered | Hashable | Literal / constructor | Section |
| --- | --- | --- | --- | --- | --- | --- |
| Numeric | `int` | No | — | Yes | `42`, `0xff`, `1_000` | 1.3.1 |
| Numeric | `float` | No | — | Yes | `3.14`, `1e-3` | 1.3.2 |
| Numeric | `complex` | No | — | Yes | `3 + 4j` | 1.3.5 |
| Numeric (stdlib) | `Decimal`, `Fraction` | No | — | Yes | `Decimal("0.1")`, `Fraction(1, 3)` | 1.3.3–1.3.4 |
| Boolean | `bool` | No | — | Yes | `True`, `False` | 1.3.6 |
| Null | `NoneType` | No | — | Yes | `None` | 1.3.7 |
| Text | `str` | No | Yes | Yes | `"hi"`, `f"{name}"` | 1.4 |
| Binary | `bytes` | No | Yes | Yes | `b"hi"` | 1.4.5 |
| Binary | `bytearray` | Yes | Yes | No | `bytearray(b"hi")` | 1.4.5 |
| Binary | `memoryview` | Depends on buffer | Yes | Read-only views only | `memoryview(data)` | 1.4.6 |
| Sequence | `list` | Yes | Yes | No | `[1, 2]` | 1.5 |
| Sequence | `tuple` | No | Yes | If all items are | `(1, 2)`, `(1,)` | 1.5.5 |
| Sequence | `range` | No | Yes | Yes | `range(10)` | 1.5.7 |
| Mapping | `dict` | Yes | Insertion order | No | `{"a": 1}` | 1.6 |
| Mapping | `frozendict` (3.15) | No | Insertion order | If all values are | `frozendict(a=1)` | 1.6.5 |
| Set | `set` | Yes | No | No | `{1, 2}`, `set()` | 1.6.6 |
| Set | `frozenset` | No | No | Yes | `frozenset({1, 2})` | 1.6.6 |

**Hashable** means `hash(obj)` works, so the object can be a dict key or a set member
(1.6.2). The cell checks the *Hashable* column for one sample of each row, including the
"it depends" cases. Unhashable objects raise `TypeError`, except a writable
`memoryview`, which raises `ValueError`. `frozendict` is *new in 3.15*, so it is added
only on 3.15+.

In [42]:
hashability_samples: list[tuple[str, object]] = [
    ("int", 42),
    ("float", 3.14),
    ("complex", 3 + 4j),
    ("Decimal", Decimal("0.1")),
    ("Fraction", Fraction(1, 3)),
    ("bool", True),
    ("NoneType", None),
    ("str", "hi"),
    ("bytes", b"hi"),
    ("bytearray", bytearray(b"hi")),
    ("memoryview (read-only)", memoryview(b"hi")),
    ("memoryview (writable)", memoryview(bytearray(b"hi"))),
    ("list", [1, 2]),
    ("tuple of ints", (1, 2)),
    ("tuple holding a list", (1, [2])),
    ("range", range(10)),
    ("dict", {"a": 1}),
    ("set", {1, 2}),
    ("frozenset", frozenset({1, 2})),
]
if sys.version_info >= (3, 15):
    hashability_samples += [
        ("frozendict of ints", frozendict(a=1)),
        ("frozendict holding a list", frozendict(a=[1])),
    ]
else:
    print("Skipped frozendict rows: new in 3.15")

for label, sample in hashability_samples:
    try:
        hash(sample)
        verdict = "hashable"
    except (TypeError, ValueError) as error:  # writable memoryview raises ValueError
        verdict = f"unhashable ({type(error).__name__})"
    print(f"{label:<26} {verdict}")

int                        hashable
float                      hashable
complex                    hashable
Decimal                    hashable
Fraction                   hashable
bool                       hashable
NoneType                   hashable
str                        hashable
bytes                      hashable
bytearray                  unhashable (TypeError)
memoryview (read-only)     hashable
memoryview (writable)      unhashable (ValueError)
list                       unhashable (TypeError)
tuple of ints              hashable
tuple holding a list       unhashable (TypeError)
range                      hashable
dict                       unhashable (TypeError)
set                        unhashable (TypeError)
frozenset                  hashable
frozendict of ints         hashable
frozendict holding a list  unhashable (TypeError)


### 1.3.1 Integers (`int`)

Python's `int` has **arbitrary precision**: it grows to hold any value, so it never
overflows. There is no separate BigInt type and no `n` suffix. JavaScript's `number`
silently loses integer precision above `2**53`; Python does not.

In [43]:
js_max_safe_integer = 2**53 - 1
print(f"{js_max_safe_integer + 2=}  (exact; JS number would print ...992)")
print(f"{2**100=}")
print(f"{math.factorial(30)=}")
print(f"{float(2**53 + 1) == 2**53=}  (float, like JS number, cannot hold it)")

js_max_safe_integer + 2=9007199254740993  (exact; JS number would print ...992)
2**100=1267650600228229401496703205376
math.factorial(30)=265252859812191058636308480000000
float(2**53 + 1) == 2**53=True  (float, like JS number, cannot hold it)


Internally CPython stores an `int` as an array of **30-bit digits** (`sys.int_info`), so
its memory grows by 4 bytes for every 30 bits of magnitude. The growth table:

In [44]:
print(f"{sys.int_info=}")
print(f"{'value':>14} {'bits':>5} {'bytes':>6}")
for label, number in [
    ("0", 0),
    ("1", 1),
    ("2**30 - 1", 2**30 - 1),
    ("2**30", 2**30),
    ("2**60", 2**60),
    ("2**90", 2**90),
    ("10**100", 10**100),
    ("10**1000", 10**1000),
]:
    print(f"{label:>14} {number.bit_length():>5} {sys.getsizeof(number):>6}")

sys.int_info=sys.int_info(bits_per_digit=30, sizeof_digit=4, default_max_str_digits=4300, str_digits_check_threshold=640)
         value  bits  bytes
             0     0     28
             1     1     28
     2**30 - 1    30     28
         2**30    31     32
         2**60    61     36
         2**90    91     40
       10**100   333     72
      10**1000  3322    468


**Literals.** Decimal, binary (`0b`), octal (`0o`, never a bare leading zero) and
hexadecimal (`0x`). Underscores group digits for readability and are ignored.

In [45]:
population_of_india = 1_428_627_663
file_mode = 0o755
flag_byte = 0b1010_0001
brand_colour = 0xFF9933
print(f"{population_of_india=}, {file_mode=}, {flag_byte=}, {brand_colour=}")
try:
    eval("0755")
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

population_of_india=1428627663, file_mode=493, flag_byte=161, brand_colour=16750899
SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers


**Operators.**

| Operator | Meaning | `17 op 5` |
| --- | --- | --- |
| `+ - *` | add, subtract, multiply | `22`, `12`, `85` |
| `/` | true division, **always returns `float`** | `3.4` |
| `//` | floor division: rounds **down** (towards −∞) | `3` |
| `%` | modulo: the result takes the **divisor's** sign | `2` |
| `**` | power | `1419857` |

Floor means *down*, not *towards zero*, so `-7 // 2 == -4`. And `a == (a // b) * b + a % b`
always holds, which forces `-7 % 2 == 1`. JavaScript's `%` is a *remainder* that keeps the
dividend's sign (`-7 % 2 === -1`).

In [46]:
print(f"{17 + 5=}, {17 - 5=}, {17 * 5=}, {17 ** 5=}")
print(f"{17 / 5=}, {10 / 2=}  (always float)")
print(f"{17 // 5=}, {17 % 5=}")
print(f"{-7 // 2=}, {-7 % 2=}  (JS: Math.trunc gives -3, % gives -1)")
print(f"{7 // -2=}, {7 % -2=}")
dividend, divisor = -7, 2
print(f"{(dividend // divisor) * divisor + dividend % divisor == dividend=}")
print(f"{math.fmod(-7, 2)=}  (JS-style remainder, when you need it)")

17 + 5=22, 17 - 5=12, 17 * 5=85, 17 ** 5=1419857
17 / 5=3.4, 10 / 2=5.0  (always float)
17 // 5=3, 17 % 5=2
-7 // 2=-4, -7 % 2=1  (JS: Math.trunc gives -3, % gives -1)
7 // -2=-4, 7 % -2=-1
(dividend // divisor) * divisor + dividend % divisor == dividend=True
math.fmod(-7, 2)=-1.0  (JS-style remainder, when you need it)


**Functions and methods.** Each gets its own demo below.

| Name | What it does |
| --- | --- |
| `abs(x)` | absolute value |
| `divmod(a, b)` | `(a // b, a % b)` in one call |
| `pow(base, exp, mod)` | modular power, fast even for huge exponents |
| `round(x, ndigits)` | rounds to a multiple of `10**-ndigits` (round half to even) |
| `int.bit_length()` | bits needed, ignoring the sign |
| `int.bit_count()` | number of 1 bits (*popcount*) |
| `int.to_bytes` / `int.from_bytes` | convert to and from raw bytes |
| `bin` / `oct` / `hex` | text in base 2 / 8 / 16 |
| `int(text, base)` | parse text in any base 2–36 |
| `int.is_integer()` | always `True`; exists so code can treat `int` and `float` alike |

`abs` works on every numeric type.

In [47]:
account_balance_change = -2_500
print(f"{abs(account_balance_change)=}, {abs(-3.5)=}, {abs(Decimal('-0.10'))=}")

abs(account_balance_change)=2500, abs(-3.5)=3.5, abs(Decimal('-0.10'))=Decimal('0.10')


`divmod` returns quotient and remainder together, e.g. to split minutes into hours.

In [48]:
hours, minutes = divmod(135, 60)
print(f"{hours=}, {minutes=}")
print(f"{divmod(-7, 2)=}")

hours=2, minutes=15
divmod(-7, 2)=(-4, 1)


Three-argument `pow(base, exp, mod)` computes `base ** exp % mod` without building the
huge intermediate number. It is the core of RSA. With `exp=-1` it returns the modular
inverse (3.8+).

In [49]:
print(f"{pow(2, 10)=}")
print(f"{pow(7, 10**18, 13)=}  (instant; 7 ** 10**18 would never finish)")
print(f"{pow(3, -1, 11)=}  (3 * 4 % 11 == 1)")

pow(2, 10)=1024
pow(7, 10**18, 13)=9  (instant; 7 ** 10**18 would never finish)
pow(3, -1, 11)=4  (3 * 4 % 11 == 1)


`round(int, ndigits)` with a **negative** `ndigits` rounds to tens, hundreds, …. Ties
go to the even neighbour, so `1250` becomes `1200`.

In [50]:
print(f"{round(1_234, -2)=}, {round(1_250, -2)=}, {round(1_350, -2)=}")
print(f"{round(7)=}, {type(round(7.6)).__name__=}  (round() with no ndigits returns int)")

round(1_234, -2)=1200, round(1_250, -2)=1200, round(1_350, -2)=1400
round(7)=7, type(round(7.6)).__name__='int'  (round() with no ndigits returns int)


`bit_length()` is the number of bits needed for the magnitude; `bit_count()` counts the
ones (*new in 3.10*).

In [51]:
sample_number = 0b1011_0000
print(f"{sample_number=}, {sample_number.bit_length()=}, {(-sample_number).bit_length()=}")
print(f"{sample_number.bit_count()=}, {(255).bit_count()=}")

sample_number=176, sample_number.bit_length()=8, (-sample_number).bit_length()=8
sample_number.bit_count()=3, (255).bit_count()=8


`to_bytes(length, byteorder, signed=...)` and `int.from_bytes` convert between an `int`
and its raw bytes, e.g. for binary file formats and network protocols. Both default to
big-endian (3.11+).

In [52]:
port_number = 8080
port_bytes = port_number.to_bytes(2, "big")
print(f"{port_bytes=}, {port_bytes.hex()=}")
print(f"{int.from_bytes(port_bytes, 'big')=}")
print(f"{int.from_bytes(port_bytes, 'little')=}  (byte order matters)")
print(f"{(-1).to_bytes(2, 'big', signed=True)=}")
try:
    (-1).to_bytes(2, "big")
except OverflowError as error:
    print(f"{type(error).__name__}: {error}")

port_bytes=b'\x1f\x90', port_bytes.hex()='1f90'
int.from_bytes(port_bytes, 'big')=8080
int.from_bytes(port_bytes, 'little')=36895  (byte order matters)
(-1).to_bytes(2, 'big', signed=True)=b'\xff\xff'
OverflowError: can't convert negative int to unsigned


`bin`, `oct` and `hex` return prefixed strings. `format()` with `b`, `o`, `x` does the
same without the prefix, and can pad.

In [53]:
print(f"{bin(10)=}, {oct(493)=}, {hex(16_750_899)=}")
print(f"{format(10, '08b')=}, {format(255, 'X')=}")

bin(10)='0b1010', oct(493)='0o755', hex(16_750_899)='0xff9933'
format(10, '08b')='00001010', format(255, 'X')='FF'


`int(text, base)` parses text in bases 2 to 36. Base `0` reads the prefix from the text.
Surrounding whitespace and underscores between digits are allowed.

In [54]:
print(f"{int('ff', 16)=}, {int('1010', 2)=}, {int('z', 36)=}")
print(f"{int('0x1F', 0)=}, {int('0b101', 0)=}")
print(f"{int('  1_000  ')=}")

int('ff', 16)=255, int('1010', 2)=10, int('z', 36)=35
int('0x1F', 0)=31, int('0b101', 0)=5
int('  1_000  ')=1000


`int.is_integer()` (*new in 3.12*) always returns `True`. It exists so code that receives
"a number" can ask the question without checking the type first.

In [55]:
for quantity in (3, 3.0, 3.5):
    print(f"{quantity!r:>4}: {quantity.is_integer()=}")

   3: quantity.is_integer()=True
 3.0: quantity.is_integer()=True
 3.5: quantity.is_integer()=False


**Bitwise operators** work on the binary digits. Their classic use is a set of on/off
flags packed into one integer, such as Unix file permissions. (Phase 2 shows
`enum.Flag`, the type-safe version.)

| Operator | Meaning | Use with flags |
| --- | --- | --- |
| `a \| b` | OR | grant a permission |
| `a & b` | AND | test a permission |
| `a & ~b` | AND NOT | revoke a permission |
| `a ^ b` | XOR | toggle a permission |
| `a << n`, `a >> n` | shift left / right | build masks, multiply / divide by `2**n` |

In [56]:
READ, WRITE, EXECUTE = 0b100, 0b010, 0b001

permissions = READ | WRITE
print(f"{bin(permissions)=}, can write: {bool(permissions & WRITE)}")
permissions &= ~WRITE
print(f"{bin(permissions)=}, can write: {bool(permissions & WRITE)}")
permissions ^= EXECUTE
print(f"{bin(permissions)=}, can execute: {bool(permissions & EXECUTE)}")
print(f"{~5=}  (~x == -x - 1: Python ints behave as infinite two's complement)")
print(f"{1 << 10=}, {1024 >> 3=}")
owner_group_other = (0o7 << 6) | (0o5 << 3) | 0o5
print(f"{oct(owner_group_other)=}")

bin(permissions)='0b110', can write: True
bin(permissions)='0b100', can write: False
bin(permissions)='0b101', can execute: True
~5=-6  (~x == -x - 1: Python ints behave as infinite two's complement)
1 << 10=1024, 1024 >> 3=128
oct(owner_group_other)='0o755'


**The 4300-digit limit (3.11+, also back-ported to security releases).** Converting a
huge `int` to or from **decimal text** takes time quadratic in the number of digits. A
web service that ran `int(request_body)` could be stalled by a megabyte of digits
(CVE-2020-10735). Python therefore refuses decimal conversions above 4300 digits by
default. Binary bases (`hex`, `bin`) are linear and are not limited. Raise the limit
only for trusted data, and put it back afterwards.

In [57]:
huge_number = 10**5000
try:
    str(huge_number)
except ValueError as error:
    print(f"{type(error).__name__}: {str(error)[:70]}...")
print(f"{len(hex(huge_number))=}  (hex is not limited)")

default_limit = sys.get_int_max_str_digits()
sys.set_int_max_str_digits(10_000)
try:
    print(f"{len(str(huge_number))=}")
finally:
    sys.set_int_max_str_digits(default_limit)
print(f"{sys.get_int_max_str_digits()=}")

ValueError: Exceeds the limit (4300 digits) for integer string conversion; use sys...
len(hex(huge_number))=4155  (hex is not limited)
len(str(huge_number))=5001
sys.get_int_max_str_digits()=4300


### 1.3.2 Floating point (`float`)

A Python `float` is an **IEEE 754 double** (64 bits), exactly the same as a JavaScript
`number`. Most decimal fractions such as `0.1` have no exact binary form, so they are
stored as the nearest double, and the errors show up in arithmetic.
`Decimal(some_float)` reveals the value that is really stored. Compare floats with
`math.isclose`, never with `==`.

In [58]:
print(f"{0.1 + 0.2=}")
print(f"{0.1 + 0.2 == 0.3=}")
print(f"{Decimal(0.1)=}")
print(f"{math.isclose(0.1 + 0.2, 0.3)=}")
print(f"{math.isclose(1e-12, 0.0)=}  (relative tolerance fails near 0)")
print(f"{math.isclose(1e-12, 0.0, abs_tol=1e-9)=}")

0.1 + 0.2=0.30000000000000004
0.1 + 0.2 == 0.3=False
Decimal(0.1)=Decimal('0.1000000000000000055511151231257827021181583404541015625')
math.isclose(0.1 + 0.2, 0.3)=True
math.isclose(1e-12, 0.0)=False  (relative tolerance fails near 0)
math.isclose(1e-12, 0.0, abs_tol=1e-9)=True


**`round()` surprises.** Two effects combine:

1. **Banker's rounding:** an exact tie goes to the **even** neighbour, so `round(2.5) == 2`.
   This avoids an upward bias when many values are rounded. JS `Math.round(2.5)` is `3`.
2. **Representation error:** `2.675` is really stored as `2.67499999…`, so it is not a
   tie at all and rounds *down*.

In [59]:
print(f"{round(0.5)=}, {round(1.5)=}, {round(2.5)=}, {round(-2.5)=}")
print(f"{round(2.675, 2)=}")
print(f"{Decimal(2.675)=}")

round(0.5)=0, round(1.5)=2, round(2.5)=2, round(-2.5)=-2
round(2.675, 2)=2.67
Decimal(2.675)=Decimal('2.67499999999999982236431605997495353221893310546875')


**Special values.** `inf`, `-inf` and `nan` ("not a number") come from `float("inf")` /
`math.inf` / `math.nan` or from overflowing operations. `nan` is not equal to anything,
not even itself, so test it with `math.isnan`. Unlike JS, `1 / 0` raises instead of
returning `Infinity`.

In [60]:
positive_infinity = float("inf")
not_a_number = float("nan")
print(f"{positive_infinity > 10**308=}, {-math.inf < -1e308=}")
print(f"{positive_infinity - positive_infinity=}")
print(f"{not_a_number == not_a_number=}, {not_a_number != not_a_number=}")
print(f"{math.isnan(not_a_number)=}, {math.isinf(-math.inf)=}, {math.isfinite(1e308)=}")
print(f"{1e308 * 10=}")
try:
    ratio = 1 / 0
except ZeroDivisionError as error:
    print(f"{type(error).__name__}: {error}")

positive_infinity > 10**308=True, -math.inf < -1e308=True
positive_infinity - positive_infinity=nan
not_a_number == not_a_number=False, not_a_number != not_a_number=True
math.isnan(not_a_number)=True, math.isinf(-math.inf)=True, math.isfinite(1e308)=True
1e308 * 10=inf
ZeroDivisionError: division by zero


**`float` methods.** `as_integer_ratio()` returns the exact fraction the double stores.
For `0.1` that is not `1/10`.

In [61]:
print(f"{(0.75).as_integer_ratio()=}")
print(f"{(0.1).as_integer_ratio()=}")

(0.75).as_integer_ratio()=(3, 4)
(0.1).as_integer_ratio()=(3602879701896397, 36028797018963968)


`is_integer()` tells whether the float has no fractional part.

In [62]:
print(f"{(4.0).is_integer()=}, {(4.5).is_integer()=}, {(1e20).is_integer()=}")

(4.0).is_integer()=True, (4.5).is_integer()=False, (1e20).is_integer()=True


`hex()` and `float.fromhex()` write and read the exact bits in hexadecimal, with no
decimal rounding. Use them to store or send a float losslessly.

In [63]:
exact_text = (0.1).hex()
print(f"{exact_text=}")
print(f"{float.fromhex(exact_text) == 0.1=}")

exact_text='0x1.999999999999ap-4'
float.fromhex(exact_text) == 0.1=True


`sys.float_info` describes the platform's doubles. `epsilon` is the gap between `1.0`
and the next float, and `dig` is how many decimal digits are always kept.

In [64]:
print(f"{sys.float_info.max=}")
print(f"{sys.float_info.min=}")
print(f"{sys.float_info.epsilon=}")
print(f"{sys.float_info.dig=}, {sys.float_info.mant_dig=}")
print(f"{1.0 + sys.float_info.epsilon > 1.0=}, {1.0 + sys.float_info.epsilon / 2 > 1.0=}")

sys.float_info.max=1.7976931348623157e+308
sys.float_info.min=2.2250738585072014e-308
sys.float_info.epsilon=2.220446049250313e-16
sys.float_info.dig=15, sys.float_info.mant_dig=53
1.0 + sys.float_info.epsilon > 1.0=True, 1.0 + sys.float_info.epsilon / 2 > 1.0=False


**Useful `math` functions.** `floor` rounds down, `ceil` rounds up, `trunc` drops the
fraction (towards zero, like `int()`). They differ only for negative numbers, and all
three return `int`.

In [65]:
for value in (3.7, -3.7):
    rounded_values = (math.floor(value), math.ceil(value), math.trunc(value), int(value))
    print(f"{value:>5}: floor, ceil, trunc, int = {rounded_values}")

  3.7: floor, ceil, trunc, int = (3, 4, 3, 3)
 -3.7: floor, ceil, trunc, int = (-4, -3, -3, -3)


`math.sqrt` returns a float and rejects negatives (use `cmath`, 1.3.5). `math.isqrt`
gives the exact integer square root of any `int`.

In [66]:
print(f"{math.sqrt(2)=}, {math.isqrt(10**20 + 1)=}")
try:
    math.sqrt(-1)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

math.sqrt(2)=1.4142135623730951, math.isqrt(10**20 + 1)=10000000000
ValueError: expected a nonnegative input, got -1.0


`math.fsum` adds floats while tracking every lost low-order bit, so the result is
correctly rounded. A plain `+=` loop (what JS `reduce` does) accumulates error.
*Changed in 3.12:* `sum()` of floats uses compensated (Neumaier) summation, which fixes
the common cases, but `fsum` is still the exact one when values of wildly different
sizes cancel.

In [67]:
ten_dimes = [0.1] * 10
loop_total = 0.0
for dime in ten_dimes:
    loop_total += dime
print(f"{loop_total=}  (naive left-to-right)")
print(f"{sum(ten_dimes)=}, {math.fsum(ten_dimes)=}")

mixed_magnitudes = [1e100, 1.0, -1e100, 1e-100, 1e50, -1.0, -1e50]
print(f"{sum(mixed_magnitudes)=}, {math.fsum(mixed_magnitudes)=}")

loop_total=0.9999999999999999  (naive left-to-right)
sum(ten_dimes)=1.0, math.fsum(ten_dimes)=1.0
sum(mixed_magnitudes)=0.0, math.fsum(mixed_magnitudes)=1e-100


`math.prod` multiplies an iterable, the way `sum` adds one (3.8+).

In [68]:
print(f"{math.prod([2, 3, 7])=}")
print(f"{math.prod([1.05] * 3)=}  (three years of 5% growth)")

math.prod([2, 3, 7])=42
math.prod([1.05] * 3)=1.1576250000000001  (three years of 5% growth)


`math.pi`, `math.e` (and `math.tau`, `math.inf`, `math.nan`) are float constants.

In [69]:
circle_radius_in_metres = 2.5
print(f"{math.pi=}, {math.e=}, {math.tau == 2 * math.pi=}")
print(f"{math.pi * circle_radius_in_metres**2=}")

math.pi=3.141592653589793, math.e=2.718281828459045, math.tau == 2 * math.pi=True
math.pi * circle_radius_in_metres**2=19.634954084936208


### 1.3.3 Exact decimals (`decimal.Decimal`)

`Decimal` stores numbers in base 10, so `0.1` is exactly one tenth. That is what money
needs. **Always construct it from a string (or an `int`), never from a float**: a float
argument copies the float's representation error.

In [70]:
print(f"{Decimal('0.1')=}")
print(f"{Decimal(0.1)=}")
print(f"{Decimal('0.1') + Decimal('0.2') == Decimal('0.3')=}")
print(f"{Decimal(25)=}, {Decimal('1.50')=}  (trailing zeros are kept)")

Decimal('0.1')=Decimal('0.1')
Decimal(0.1)=Decimal('0.1000000000000000055511151231257827021181583404541015625')
Decimal('0.1') + Decimal('0.2') == Decimal('0.3')=True
Decimal(25)=Decimal('25'), Decimal('1.50')=Decimal('1.50')  (trailing zeros are kept)


**Arithmetic.** `Decimal` supports the usual operators and mixes with `int`, but not
with `float` (that would bring the inexactness back). Results are rounded to the
**context precision**: 28 significant digits by default. `localcontext()` changes the
precision (or the rounding) only inside a `with` block.

In [71]:
unit_price = Decimal("19.99")
print(f"{unit_price * 3=}, {unit_price - 5=}, {unit_price / 4=}")
print(f"{Decimal(1) / Decimal(7)=}")
with localcontext() as local_context:
    local_context.prec = 6
    print(f"inside localcontext: {Decimal(1) / Decimal(7)=}")
print(f"after the block: {getcontext().prec=}")
try:
    unit_price + 0.5
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

unit_price * 3=Decimal('59.97'), unit_price - 5=Decimal('14.99'), unit_price / 4=Decimal('4.9975')
Decimal(1) / Decimal(7)=Decimal('0.1428571428571428571428571429')
inside localcontext: Decimal(1) / Decimal(7)=Decimal('0.142857')
after the block: getcontext().prec=28
TypeError: unsupported operand type(s) for +: 'decimal.Decimal' and 'float'


**`quantize`** rounds to a fixed exponent, e.g. `Decimal("0.01")` for paise. The
rounding mode is explicit:

| Mode | Ties go… | Typical use |
| --- | --- | --- |
| `ROUND_HALF_UP` | away from zero (`2.675 → 2.68`) | invoices and tax, as taught in school |
| `ROUND_HALF_EVEN` | to the even digit (`2.665 → 2.66`) | statistics, the `Decimal` default |

In [72]:
print(f"{'value':>7} {'HALF_UP':>8} {'HALF_EVEN':>10}")
for amount_text in ["2.675", "2.665", "0.125", "-1.005"]:
    amount = Decimal(amount_text)
    half_up = amount.quantize(PAISE, rounding=ROUND_HALF_UP)
    half_even = amount.quantize(PAISE, rounding=ROUND_HALF_EVEN)
    print(f"{amount_text:>7} {half_up!s:>8} {half_even!s:>10}")

  value  HALF_UP  HALF_EVEN
  2.675     2.68       2.68
  2.665     2.67       2.66
  0.125     0.13       0.12
 -1.005    -1.01      -1.00


**Money: an 18% GST demo.** One thousand orders of ₹99.99 each, with GST computed and
rounded per order. Float accumulates representation error across the additions;
`Decimal` is exact to the paisa.

In [73]:
order_count = 1_000

float_grand_total = 0.0
for _ in range(order_count):
    float_grand_total += 99.99 + round(99.99 * 0.18, 2)

decimal_grand_total = Decimal("0.00")
for _ in range(order_count):
    gst = (Decimal("99.99") * Decimal("0.18")).quantize(PAISE, rounding=ROUND_HALF_UP)
    decimal_grand_total += Decimal("99.99") + gst

print(f"{float_grand_total=}")
print(f"{decimal_grand_total=}")

float_grand_total=117990.00000000196
decimal_grand_total=Decimal('117990.00')


**Performance trade-off.** `float` arithmetic runs in hardware; `Decimal` is software
(C-accelerated `_decimal`, but still slower). The usual pattern is `Decimal` (or
integer paise) for money and `float` for measurements and science. Timings vary by
machine; the ratio is what matters.

In [74]:
float_operands = {"a": 99.99, "b": 0.18, "c": 1.5}
decimal_operands = {"a": Decimal("99.99"), "b": Decimal("0.18"), "c": Decimal("1.5")}
float_seconds = timeit.timeit("a * b + c", globals=float_operands, number=200_000)
decimal_seconds = timeit.timeit("a * b + c", globals=decimal_operands, number=200_000)
slowdown = decimal_seconds / float_seconds
print(f"float:   {float_seconds * 1000:.1f} ms")
print(f"Decimal: {decimal_seconds * 1000:.1f} ms  (about {slowdown:.0f}x slower)")

float:   3.4 ms
Decimal: 17.1 ms  (about 5x slower)


### 1.3.4 Rational numbers (`fractions.Fraction`)

A `Fraction` is an exact numerator/denominator pair, always kept in lowest terms. One
third really is one third, so `Fraction(1, 3) * 3 == 1`.

In [75]:
one_third = Fraction(1, 3)
print(f"{one_third * 3 == 1=}, {one_third * 3=}")
print(f"{Fraction(6, 8)=}  (reduced automatically)")
print(f"{one_third + Fraction(1, 6)=}")
print(f"{1 / 3 * 3 == 1=}  (float happens to round back here, but not in general)")

one_third * 3 == 1=True, one_third * 3=Fraction(1, 1)
Fraction(6, 8)=Fraction(3, 4)  (reduced automatically)
one_third + Fraction(1, 6)=Fraction(1, 2)
1 / 3 * 3 == 1=True  (float happens to round back here, but not in general)


Build a `Fraction` from a **string** for the exact value you typed. Building it from a
float shows the float's exact binary value. `limit_denominator(max_denominator)` finds
the closest fraction with a small denominator, which recovers the intended value.

In [76]:
print(f"{Fraction('3/7')=}, {Fraction('0.125')=}, {Fraction('1.5e-2')=}")
print(f"{Fraction(0.1)=}")
print(f"{Fraction(0.1).limit_denominator(1000)=}")
print(f"{Fraction(math.pi).limit_denominator(1000)=}")

Fraction('3/7')=Fraction(3, 7), Fraction('0.125')=Fraction(1, 8), Fraction('1.5e-2')=Fraction(3, 200)
Fraction(0.1)=Fraction(3602879701896397, 36028797018963968)
Fraction(0.1).limit_denominator(1000)=Fraction(1, 10)
Fraction(math.pi).limit_denominator(1000)=Fraction(355, 113)


### 1.3.5 Complex numbers (`complex`)

Python has complex numbers built in: write the imaginary part with a `j` suffix. They
are used in signal processing and electrical engineering; you will rarely need them for
web work.

In [77]:
impedance = 3 + 4j
print(f"{impedance=}, {type(impedance).__name__=}")
print(f"{impedance.real=}, {impedance.imag=}")
print(f"{abs(impedance)=}  (magnitude)")
print(f"{impedance.conjugate()=}, {impedance * impedance.conjugate()=}")
print(f"{complex(3, 4) == impedance=}")

impedance=(3+4j), type(impedance).__name__='complex'
impedance.real=3.0, impedance.imag=4.0
abs(impedance)=5.0  (magnitude)
impedance.conjugate()=(3-4j), impedance * impedance.conjugate()=(25+0j)
complex(3, 4) == impedance=True


`cmath` is the complex-number version of `math`: its functions accept negative and
complex inputs.

In [78]:
print(f"{cmath.sqrt(-1)=}")
print(f"{cmath.phase(1j)=}  (pi / 2)")
print(f"{cmath.polar(impedance)=}")
print(f"{cmath.rect(5, 0.9272952180016122)=}")

cmath.sqrt(-1)=1j
cmath.phase(1j)=1.5707963267948966  (pi / 2)
cmath.polar(impedance)=(5.0, 0.9272952180016122)
cmath.rect(5, 0.9272952180016122)=(3.0000000000000004+3.9999999999999996j)


### 1.3.6 Booleans (`bool`)

`bool` is a **subclass of `int`**: `True` is `1` and `False` is `0` in arithmetic. A
useful consequence: `sum(flags)` counts the `True` values.

In [79]:
print(f"{issubclass(bool, int)=}")
print(f"{True + True=}, {True * 10=}, {True == 1=}")
order_is_paid = [True, False, True, True]
print(f"{sum(order_is_paid)=} paid orders")

issubclass(bool, int)=True
True + True=2, True * 10=10, True == 1=True
sum(order_is_paid)=3 paid orders


**The validation trap.** Because `True` *is* an `int`, `isinstance(True, int)` is
`True`, and a check meant to accept quantities also accepts booleans. Reject `bool`
explicitly.

In [80]:
def is_valid_quantity_naive(quantity: object) -> bool:
    """Accept positive ints (buggy: also accepts True)."""
    return isinstance(quantity, int) and quantity > 0


def is_valid_quantity(quantity: object) -> bool:
    """Accept positive ints, rejecting bool."""
    return isinstance(quantity, int) and not isinstance(quantity, bool) and quantity > 0


for candidate in (3, True, 0, 2.0):
    naive_verdict = is_valid_quantity_naive(candidate)
    print(f"{candidate!r:>5}: naive={naive_verdict}, fixed={is_valid_quantity(candidate)}")

    3: naive=True, fixed=True
 True: naive=True, fixed=False
    0: naive=False, fixed=False
  2.0: naive=False, fixed=False


`bool(x)` converts any value using the truthiness rules (1.3.8). Watch out for strings:
any non-empty string is `True`, including `"False"` and `"0"`.

In [81]:
print(f"{bool(0)=}, {bool(42)=}, {bool('')=}, {bool('False')=}, {bool('0')=}")
form_value = "false"
print(f"{form_value.strip().lower() in {'true', '1', 'yes'}=}  (parse text explicitly)")

bool(0)=False, bool(42)=True, bool('')=False, bool('False')=True, bool('0')=True
form_value.strip().lower() in {'true', '1', 'yes'}=False  (parse text explicitly)


### 1.3.7 The null value (`None`)

`None` is the only instance of `NoneType`, so every `None` is the same object; check it
with `is None`. A function without a `return` (or with a bare `return`) returns `None`.

In [82]:
def log_message(message: str) -> None:
    """Print a message; no return statement."""
    print(f"log: {message}")


result = log_message("order created")
print(f"{result is None=}")
print(f"{type(None).__name__=}, {type(None)() is None=}")

log: order created
result is None=True
type(None).__name__='NoneType', type(None)() is None=True


> **JS bridge:** Python has **no `undefined`**. An unassigned name raises `NameError`,
> a missing attribute raises `AttributeError`, and a missing dict key raises `KeyError`.
> `dict.get(key)` returns `None` (or a default) when you want JS-like behaviour.

In [83]:
customer = {"name": "Asha", "city": "Pune"}
try:
    customer["phone"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{customer.get('phone')=}, {customer.get('phone', 'not given')=}")

KeyError: 'phone'
customer.get('phone')=None, customer.get('phone', 'not given')='not given'


### 1.3.8 Truthiness

Every object is true or false in an `if`, `while`, `and`, `or` or `not`. These values are
**falsy**; everything else is truthy:

| Falsy value | Type |
| --- | --- |
| `None` | `NoneType` |
| `False` | `bool` |
| `0`, `0.0`, `0j` | `int`, `float`, `complex` |
| `Decimal(0)`, `Fraction(0)` | `Decimal`, `Fraction` |
| `""`, `b""` | `str`, `bytes` |
| `[]`, `()`, `{}` | `list`, `tuple`, `dict` |
| `set()`, `range(0)` | `set`, `range` |
| an object whose `__bool__` returns `False`, or whose `__len__` returns `0` | user classes |

The cell evaluates every row.

In [84]:
falsy_candidates = [None, False, 0, 0.0, 0j, Decimal(0), Fraction(0)]
falsy_candidates += ["", b"", [], (), {}, set(), range(0)]
for candidate in falsy_candidates:
    print(f"{candidate!r:<14} -> {bool(candidate)}")

None           -> False
False          -> False
0              -> False
0.0            -> False
0j             -> False
Decimal('0')   -> False
Fraction(0, 1) -> False
''             -> False
b''            -> False
[]             -> False
()             -> False
{}             -> False
set()          -> False
range(0, 0)    -> False


User classes decide with `__bool__`, or, if it is missing, with `__len__` (zero length
is false). With neither, every instance is truthy.

In [85]:
class Wallet:
    """A wallet that is truthy only when it holds money."""

    def __init__(self, balance_in_paise: int) -> None:
        self.balance_in_paise = balance_in_paise

    def __bool__(self) -> bool:
        return self.balance_in_paise > 0


class ShoppingBag:
    """A bag whose truthiness comes from its length."""

    def __init__(self, items: list[str]) -> None:
        self.items = items

    def __len__(self) -> int:
        return len(self.items)


class PlainObject:
    """No __bool__ and no __len__."""


print(f"{bool(Wallet(0))=}, {bool(Wallet(500))=}")
print(f"{bool(ShoppingBag([]))=}, {bool(ShoppingBag(['soap']))=}")
print(f"{bool(PlainObject())=}")

bool(Wallet(0))=False, bool(Wallet(500))=True
bool(ShoppingBag([]))=False, bool(ShoppingBag(['soap']))=True
bool(PlainObject())=True


**Differences from JavaScript**

| Value | JavaScript | Python |
| --- | --- | --- |
| `[]`, `{}` | truthy | **falsy** |
| `"0"`, `" "` | truthy | truthy |
| `NaN` | falsy | **truthy** (`nan` is a non-zero float) |
| `0n` / `0` | falsy | falsy |
| `document.all` | falsy (!) | — |

**When to prefer `is not None`.** `if value:` treats `0`, `""` and `[]` the same as a
missing value. When `0` is a legitimate input, test for `None` explicitly.

In [86]:
def price_after_discount_buggy(price: int, discount: int | None = None) -> int:
    """Apply a discount; treats a discount of 0 as 'not given' (bug)."""
    if not discount:
        discount = 10  # default discount
    return price - discount


def price_after_discount(price: int, discount: int | None = None) -> int:
    """Apply a discount; only None means 'not given'."""
    if discount is None:
        discount = 10
    return price - discount


print(f"{bool(float('nan'))=}")
print(f"{price_after_discount_buggy(100, 0)=}  (wanted 100)")
print(f"{price_after_discount(100, 0)=}")

bool(float('nan'))=True
price_after_discount_buggy(100, 0)=90  (wanted 100)
price_after_discount(100, 0)=100


### 1.3.9 Type conversion

Conversions are always **explicit**: call the type. They raise `ValueError` when the
text is not a valid literal for that type. Note that `int("3.5")` fails: convert to
`float` first if decimals are allowed. `int(3.99)` truncates.

In [87]:
print(f"{int('42')=}, {int(' -7 ')=}, {int(3.99)=}, {int(True)=}")
print(f"{float('3.5')=}, {float('1e3')=}, {float('nan')=}, {float(7)=}")
print(f"{str(42)=}, {str(3.5)=}, {str(None)=}")
print(f"{bool(1)=}, {bool('')=}")
print(f"{complex('3+4j')=}, {complex(2)=}")
print(f"{int(float('3.5'))=}")

int('42')=42, int(' -7 ')=-7, int(3.99)=3, int(True)=1
float('3.5')=3.5, float('1e3')=1000.0, float('nan')=nan, float(7)=7.0
str(42)='42', str(3.5)='3.5', str(None)='None'
bool(1)=True, bool('')=False
complex('3+4j')=(3+4j), complex(2)=(2+0j)
int(float('3.5'))=3


Each failing conversion raises `ValueError` with a message that quotes the bad text.

In [88]:
for converter, bad_text in [(int, "3.5"), (int, "12abc"), (float, "abc"), (complex, "3 + 4j")]:
    try:
        converter(bad_text)
    except ValueError as error:
        print(f"{converter.__name__}({bad_text!r}) -> {type(error).__name__}: {error}")

int('3.5') -> ValueError: invalid literal for int() with base 10: '3.5'
int('12abc') -> ValueError: invalid literal for int() with base 10: '12abc'
float('abc') -> ValueError: could not convert string to float: 'abc'
complex('3 + 4j') -> ValueError: complex() arg is a malformed string


**No implicit coercion.** JavaScript converts silently (`1 + "1" === "11"`,
`1 == "1"` is `true`). Python raises `TypeError` for mixed `+` and returns `False` for
`==` between unrelated types. Numeric types do mix with each other (`1 + 2.5`, `1 == 1.0`).

In [89]:
try:
    mixed_total = 1 + "1"
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{1 == '1'=}")
print(f"{1 + int('1')=}, {str(1) + '1'=}")
print(f"{1 + 2.5=}, {1 == 1.0 == Decimal(1) == Fraction(1)=}")

TypeError: unsupported operand type(s) for +: 'int' and 'str'
1 == '1'=False
1 + int('1')=2, str(1) + '1'='11'
1 + 2.5=3.5, 1 == 1.0 == Decimal(1) == Fraction(1)=True


#### Under the hood: object sizes and the int digit formula

Every number is a full object with a header, so even `True` costs bytes. For `int`,
the size is a fixed header plus 4 bytes per 30-bit digit; the cell checks the formula
against `sys.getsizeof` for growing values. `show_size` (shared setup) prints a few
other numeric types for comparison.

In [90]:
int_header_bytes = sys.getsizeof(1) - sys.int_info.sizeof_digit
for exponent in (10, 100, 1_000):
    number = 2**exponent
    digit_count = -(-number.bit_length() // sys.int_info.bits_per_digit)  # ceiling division
    predicted_bytes = int_header_bytes + digit_count * sys.int_info.sizeof_digit
    actual_bytes = sys.getsizeof(number)
    print(f"2**{exponent:<5} digits={digit_count:<3} {predicted_bytes=:<4} {actual_bytes=}")

show_size("float", 1.5)
show_size("complex", 1 + 2j)
show_size("Decimal", Decimal("1.5"))
show_size("Fraction", Fraction(1, 2))
show_size("True", True)

2**10    digits=1   predicted_bytes=28   actual_bytes=28
2**100   digits=4   predicted_bytes=40   actual_bytes=40
2**1000  digits=34  predicted_bytes=160  actual_bytes=160
float: 24 bytes, type=float, id=140267243973232
complex: 32 bytes, type=complex, id=140266941510320
Decimal: 120 bytes, type=Decimal, id=140266921722560
Fraction: 48 bytes, type=Fraction, id=140266941614832
True: 28 bytes, type=bool, id=22110296


### Lab 1.3

Three exercises: an exact bill splitter, an invoice calculator (with a float version to
show drift), and predict-the-output questions. For each coding exercise, replace the
`raise NotImplementedError` in the stub, then run the check cell. Until then the checks
print FAIL with `NotImplementedError`; that is expected.

#### Exercise 1 — `split_bill`

Write `split_bill(total_amount: str, people_count: int) -> list[Decimal]`:

- Split the total (a rupee amount as text, e.g. `"100"` or `"1000.01"`) into
  `people_count` parts **exact to the paisa**: every part has two decimal places and
  the parts sum to exactly the total.
- Parts differ by at most one paisa; the leftover paise go to the **first** people.
  `split_bill("100", 3) == [Decimal("33.34"), Decimal("33.33"), Decimal("33.33")]`.
- Raise `ValueError` when `people_count` is less than 1.

*Hint:* work in whole paise with `divmod`.

In [91]:
def split_bill(total_amount: str, people_count: int) -> list[Decimal]:
    """Split a rupee amount into people_count parts that sum exactly to the total."""
    raise NotImplementedError("Lab 1.3: implement split_bill()")

The check cell defines `raised_error_name(thunk)`, which returns the name of the
exception a call raises (or `"no error"`), so error cases fit the
`(label, thunk, expected)` format of `run_checks`.

In [92]:
def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


run_checks(
    [
        (
            "100 between 3",
            lambda: split_bill("100", 3),
            [Decimal("33.34"), Decimal("33.33"), Decimal("33.33")],
        ),
        ("parts sum to the total", lambda: sum(split_bill("1000.01", 7)), Decimal("1000.01")),
        ("one part per person", lambda: len(split_bill("10", 4)), 4),
        (
            "parts differ by at most one paisa",
            lambda: max(split_bill("0.05", 3)) - min(split_bill("0.05", 3)) <= PAISE,
            True,
        ),
        (
            "every part has two decimal places",
            lambda: {part.as_tuple().exponent for part in split_bill("90", 4)},
            {-2},
        ),
        ("returns Decimal", lambda: type(split_bill("1", 1)[0]).__name__, "Decimal"),
        (
            "zero people raises",
            lambda: raised_error_name(lambda: split_bill("10", 0)),
            "ValueError",
        ),
    ]
)

FAIL  100 between 3: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  parts sum to the total: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  one part per person: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  parts differ by at most one paisa: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  every part has two decimal places: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  returns Decimal: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  zero people raises: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 2 — invoice calculator

An invoice is a list of `(unit_price, quantity)` line items. For **each line**:
net amount = unit price × quantity, GST = net amount × rate, **rounded to paise per
line**. The invoice total is the sum of every line's net amount plus its GST.

1. `invoice_total_decimal(line_items: list[tuple[str, int]], gst_rate_percent: str = "18") -> Decimal`:
   prices and rate are strings, GST is rounded with `quantize(PAISE, rounding=ROUND_HALF_UP)`,
   and the total is returned with two decimal places.
2. `invoice_total_float(line_items: list[tuple[float, int]], gst_rate_percent: float = 18.0) -> float`:
   the same calculation with floats and `round(gst, 2)`. Do **not** round the final
   total, so the drift stays visible.

In [93]:
def invoice_total_decimal(
    line_items: list[tuple[str, int]], gst_rate_percent: str = "18"
) -> Decimal:
    """Return net + per-line GST (ROUND_HALF_UP to paise) for all line items."""
    raise NotImplementedError("Lab 1.3: implement invoice_total_decimal()")


def invoice_total_float(
    line_items: list[tuple[float, int]], gst_rate_percent: float = 18.0
) -> float:
    """Return the same total computed with float and round()."""
    raise NotImplementedError("Lab 1.3: implement invoice_total_float()")

The checks include a line of ₹2.75: its GST is exactly ₹0.495, which `ROUND_HALF_UP`
makes ₹0.50. The float `2.75 * 18 / 100` is stored just *below* 0.495, so `round()`
gives ₹0.49.

In [94]:
sample_line_items = [
    ("19.99", 3),
    ("0.10", 7),
    ("2.75", 1),
    ("12.25", 2),
    ("0.25", 1),
    ("5.25", 1),
]
float_line_items = [(float(price), quantity) for price, quantity in sample_line_items]

run_checks(
    [
        ("empty invoice", lambda: invoice_total_decimal([]), Decimal("0.00")),
        ("one line, 18% GST", lambda: invoice_total_decimal([("100", 2)]), Decimal("236.00")),
        ("GST rounds half up", lambda: invoice_total_decimal([("1.25", 1)]), Decimal("1.48")),
        ("custom rate 5%", lambda: invoice_total_decimal([("10.10", 1)], "5"), Decimal("10.61")),
        ("sample invoice", lambda: invoice_total_decimal(sample_line_items), Decimal("110.25")),
        ("float: one line, 18% GST", lambda: invoice_total_float([(100.0, 2)]), 236.0),
        ("float: round() loses a paisa on 2.75", lambda: invoice_total_float([(2.75, 1)]), 3.24),
        (
            "float: sample invoice drifts from the exact total",
            lambda: (
                invoice_total_float(float_line_items)
                != float(invoice_total_decimal(sample_line_items))
            ),
            True,
        ),
    ]
)

FAIL  empty invoice: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  one line, 18% GST: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  GST rounds half up: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  custom rate 5%: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  sample invoice: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  float: one line, 18% GST: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
FAIL  float: round() loses a paisa on 2.75: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
FAIL  float: sample invoice drifts from the exact total: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
0/8 checks passed


Once both functions work, this cell shows the drift side by side. Until then it prints
a reminder.

In [95]:
try:
    exact_invoice_total = invoice_total_decimal(sample_line_items)
    float_invoice_total = invoice_total_float(float_line_items)
except NotImplementedError as error:
    print(f"Not yet: {error}")
else:
    print(f"{exact_invoice_total=}")
    print(f"{float_invoice_total=}")
    print(f"difference: {Decimal(float_invoice_total) - exact_invoice_total:.17f}")

Not yet: Lab 1.3: implement invoice_total_decimal()


#### Exercise 3 — predict the output

**Predict the output.**

```python
print(-7 // 2, 7 // -2, -7.0 // 2)
```

<details><summary>Answer</summary>

`-4 -4 -4.0`. `//` floors towards negative infinity: −3.5 becomes −4 in both sign
combinations. With a float operand the result is a float.

</details>

Run the next cell to check your prediction.

In [96]:
print(-7 // 2, 7 // -2, -7.0 // 2)

-4 -4 -4.0


**Predict the output.**

```python
print(-7 % 2, 7 % -2, -7 % -2)
```

<details><summary>Answer</summary>

`1 -1 -1`. The result of `%` takes the sign of the **divisor**, so that
`(a // b) * b + a % b == a`. For `-7 % 2`: `(-4) * 2 + 1 == -7`.

</details>

Run the next cell to check your prediction.

In [97]:
print(-7 % 2, 7 % -2, -7 % -2)

1 -1 -1


**Predict the output.**

```python
print(round(0.5), round(1.5), round(2.5), round(2.675, 2))
```

<details><summary>Answer</summary>

`0 2 2 2.67`. Exact ties go to the even neighbour (banker's rounding), and `2.675` is
stored as `2.67499999…`, which is below the tie.

</details>

Run the next cell to check your prediction.

In [98]:
print(round(0.5), round(1.5), round(2.5), round(2.675, 2))

0 2 2 2.67


**Predict the output.**

```python
print(bool([0]), bool("0"), bool(0.0), bool(None), bool(" "), bool({}))
```

<details><summary>Answer</summary>

`True True False False True False`. `[0]` is a non-empty list, and `"0"` and `" "` are
non-empty strings, so all three are truthy. `0.0`, `None` and the empty dict `{}` are
falsy.

</details>

Run the next cell to check your prediction.

In [99]:
print(bool([0]), bool("0"), bool(0.0), bool(None), bool(" "), bool({}))

True True False False True False


### Recap — Unit 1.3

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `int` | Arbitrary precision, 30-bit digits, never overflows | `BigInt` |
| `/`, `//`, `%` | `/` is always float; `//` floors; `%` takes the divisor's sign | `/`, `Math.floor`, remainder `%` |
| 4300-digit limit | Decimal-text conversion of huge ints is capped against DoS | — |
| `float` | IEEE 754 double; compare with `math.isclose`; `1/0` raises | `number` |
| `round()` | Round half to even, plus representation error | `Math.round` rounds half up |
| `Decimal` | Exact base-10; build from strings; `quantize` with explicit rounding for money | decimal.js, big.js |
| `Fraction` | Exact rationals; `limit_denominator` recovers simple ratios | — |
| `complex` | `3 + 4j`, `.real`, `.imag`, `abs`, `cmath` | — |
| `bool` | Subclass of `int`; reject it explicitly when validating ints | `boolean` |
| `None` | The only null; check with `is None`; no `undefined` | `null` / `undefined` |
| Truthiness | Empty containers and zeros are falsy; use `is not None` when 0 is valid | `[]` and `{}` are truthy |
| Conversion | Always explicit; bad text raises `ValueError`; mixed `+` raises `TypeError` | Implicit coercion |

**Further reading**

- [Numeric types — `int`, `float`, `complex`](https://docs.python.org/3/library/stdtypes.html#numeric-types-int-float-complex),
  [integer string conversion length limitation](https://docs.python.org/3/library/stdtypes.html#int-max-str-digits)
- [Floating-point arithmetic: issues and limitations](https://docs.python.org/3/tutorial/floatingpoint.html),
  [`math`](https://docs.python.org/3/library/math.html),
  [`cmath`](https://docs.python.org/3/library/cmath.html)
- [`decimal`](https://docs.python.org/3/library/decimal.html),
  [`fractions`](https://docs.python.org/3/library/fractions.html)
- [Truth value testing](https://docs.python.org/3/library/stdtypes.html#truth-value-testing),
  [`None`](https://docs.python.org/3/library/constants.html#None)
- [PEP 515 — Underscores in numeric literals](https://peps.python.org/pep-0515/),
  [PEP 285 — Adding a bool type](https://peps.python.org/pep-0285/),
  [PEP 327 — Decimal data type](https://peps.python.org/pep-0327/),
  [PEP 237 — Unifying long integers and integers](https://peps.python.org/pep-0237/)

## Unit 1.4 — Text and Binary Data Types

**Objectives**

- Index, slice, search, clean, split and format text with the complete `str` API.
- Explain what `len` counts, how CPython stores a string, and why `"10" < "9"`.
- Compare Unicode text correctly with normalization and `casefold`.
- Move between `str` and `bytes` on purpose, choosing an `errors` policy.
- Render untrusted values safely with t-strings (*new in 3.14*).

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `"😀".length === 2` | `len("😀") == 1` | Python counts code points; JS counts UTF-16 code units |
| `` `Hi ${name}` `` | `f"Hi {name}"` | Same idea; f-strings add a format spec (`{price:,.2f}`) and `=` debugging |
| Tagged templates (`` html`…` ``) | t-strings `t"…"` (3.14) | They give you a `Template` object, not a string; your renderer decides how to escape |
| `"a-b-a".replace("a", "x")` replaces the first match | `str.replace` replaces **every** match | Pass `count=1` for the JS behaviour |
| `"10" < "9"` is `true` | `"10" < "9"` is `True` | Same: strings compare character by character, never numerically |
| `str.trim()` | `str.strip()` | `strip("xy")` removes a *set* of characters, not a prefix |
| `str[99]` is `undefined` | `IndexError` | Slices (`text[99:]`) never raise |
| `Buffer` / `Uint8Array` | `bytes` / `bytearray` | `bytes` is immutable; `str + bytes` raises `TypeError` |

The next cell is this unit's setup.

In [100]:
import base64
import csv
import hashlib
import html
import locale
import sqlite3
import string
import sys
import unicodedata
from collections.abc import Callable
from datetime import date
from decimal import Decimal
from string.templatelib import Interpolation, Template


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


print(f"{sys.version_info[:2]=}, {sys.getdefaultencoding()=}, {unicodedata.unidata_version=}")

sys.version_info[:2]=(3, 15), sys.getdefaultencoding()='utf-8', unicodedata.unidata_version='17.0.0'


### 1.4.1 Strings (`str`) fundamentals

A `str` is an **immutable sequence of Unicode code points**. Single and double quotes
mean the same thing; pick the one that avoids escaping. Triple quotes span lines. A raw
string `r"..."` keeps backslashes as typed (useful for regexes and Windows paths).
Common escapes: `\n` newline, `\t` tab, `\u20b9` a code point by its hex number (₹).

In [101]:
shop_name = 'Annapurna "Tiffin" Centre'
opening_note = "It's open"
menu_text = """Idli
Dosa
Vada"""
raw_windows_path = r"C:\new\table"
escaped_windows_path = "C:\new\table"
rupee_sign = "\u20b9"
print(shop_name, "|", opening_note)
print(f"{menu_text.splitlines()=}")
print(f"{raw_windows_path=}, {len(raw_windows_path)=}")
print(f"{escaped_windows_path=}, {len(escaped_windows_path)=}  (\\n and \\t were escapes)")
print(f"{rupee_sign=}", "Total:\t" + rupee_sign + "120")

Annapurna "Tiffin" Centre | It's open
menu_text.splitlines()=['Idli', 'Dosa', 'Vada']
raw_windows_path='C:\\new\\table', len(raw_windows_path)=12
escaped_windows_path='C:\new\table', len(escaped_windows_path)=10  (\n and \t were escapes)
rupee_sign='₹' Total:	₹120


Immutable means no method or assignment changes a string in place. Every "change"
builds a new string.

In [102]:
dish_name = "dosa"
try:
    dish_name[0] = "D"
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
capitalized_dish_name = "D" + dish_name[1:]
print(f"{dish_name=}, {capitalized_dish_name=}  (a new string; the original is unchanged)")

TypeError: 'str' object does not support item assignment
dish_name='dosa', capitalized_dish_name='Dosa'  (a new string; the original is unchanged)


**Indexing and slicing.** `text[i]` returns a one-character string; negative indexes
count from the end. `text[start:stop:step]` copies a range (stop excluded), and
`[::-1]` reverses. An index out of range raises `IndexError`, but a slice out of range
just returns fewer characters.

In [103]:
order_code = "BLR-2026-0042"
print(f"{order_code[0]=}, {order_code[-1]=}, {order_code[-4:]=}")
print(f"{order_code[:3]=}, {order_code[4:8]=}, {order_code[::2]=}")
print(f"{order_code[::-1]=}")
print(f"{order_code[100:]=}, {order_code[-100:3]=}  (slices clamp, never raise)")
try:
    print(order_code[100])
except IndexError as error:
    print(f"{type(error).__name__}: {error}")

order_code[0]='B', order_code[-1]='2', order_code[-4:]='0042'
order_code[:3]='BLR', order_code[4:8]='2026', order_code[::2]='BR22-02'
order_code[::-1]='2400-6202-RLB'
order_code[100:]='', order_code[-100:3]='BLR'  (slices clamp, never raise)
IndexError: string index out of range


**What `len` counts.** Python's `len` counts **code points**. JavaScript's `.length`
counts UTF-16 code units, so any character outside the Basic Multilingual Plane (most
emoji) counts as 2 in JS. Neither counts what a person sees: 👍🏽 is two code points (a
thumb plus a skin-tone modifier) and 🇮🇳 is two regional-indicator letters.

In [104]:
for sample_text in ["dosa", "₹120", "😀", "👍🏽", "🇮🇳"]:
    code_point_count = len(sample_text)
    utf16_unit_count = len(sample_text.encode("utf-16-le")) // 2
    utf8_byte_count = len(sample_text.encode("utf-8"))
    print(
        f"{sample_text!r:8} len={code_point_count}  JS .length={utf16_unit_count}"
        f"  UTF-8 bytes={utf8_byte_count}"
    )

'dosa'   len=4  JS .length=4  UTF-8 bytes=4
'₹120'   len=4  JS .length=4  UTF-8 bytes=6
'😀'      len=1  JS .length=2  UTF-8 bytes=4
'👍🏽'     len=2  JS .length=4  UTF-8 bytes=8
'🇮🇳'     len=2  JS .length=4  UTF-8 bytes=8


#### Under the hood: PEP 393 compact strings

CPython stores each string with the narrowest fixed width that fits its **largest**
code point: 1 byte per character (Latin-1), 2 bytes (the rest of the BMP, e.g. ₹) or
4 bytes (emoji). Indexing stays O(1). One emoji makes the whole string 4 bytes per
character. Each row adds one more character and measures the growth. Exact sizes vary
by Python version.

In [105]:
for width_label, sample_text in [
    ("ASCII", "a" * 10),
    ("Latin-1 (é)", "é" * 10),
    ("BMP (₹)", "₹" * 10),
    ("astral (😀)", "😀" * 10),
    ("ASCII + one 😀", "a" * 9 + "😀"),
]:
    bytes_per_character = sys.getsizeof(sample_text + sample_text[-1]) - sys.getsizeof(sample_text)
    print(
        f"{width_label:15} {sys.getsizeof(sample_text):>4} bytes total, "
        f"{bytes_per_character} byte(s) per character"
    )

ASCII             51 bytes total, 1 byte(s) per character


Latin-1 (é)       67 bytes total, 1 byte(s) per character
BMP (₹)           78 bytes total, 2 byte(s) per character
astral (😀)       100 bytes total, 4 byte(s) per character
ASCII + one 😀    100 bytes total, 4 byte(s) per character


**Operators.** `+` concatenates, `*` repeats, `in` tests for a substring. Comparison is
**lexicographic by code point**: `"10" < "9"` because `"1"` (U+0031) comes before `"9"`
(U+0039). Uppercase letters sort before lowercase ones. Sort numbers stored as text with
`key=int`.

In [106]:
print(f"{'tea' + 'pot'=}, {'-' * 10=}, {'dos' in 'dosa'=}")
print(f"{'10' < '9'=}  (compares '1' with '9' first)")
print(f"{sorted(['10', '9', '100', '2'])=}")
print(f"{sorted(['10', '9', '100', '2'], key=int)=}")
print(f"{'apple' < 'Banana'=}, {ord('a')=}, {ord('B')=}  (uppercase sorts first)")

'tea' + 'pot'='teapot', '-' * 10='----------', 'dos' in 'dosa'=True
'10' < '9'=True  (compares '1' with '9' first)
sorted(['10', '9', '100', '2'])=['10', '100', '2', '9']
sorted(['10', '9', '100', '2'], key=int)=['2', '9', '10', '100']
'apple' < 'Banana'=False, ord('a')=97, ord('B')=66  (uppercase sorts first)


### 1.4.2 String methods (complete API by group)

| Group | Methods |
| --- | --- |
| Case | `lower`, `upper`, `title`, `capitalize`, `casefold`, `swapcase` |
| Searching | `find`, `rfind`, `index`, `rindex`, `count`, `startswith`, `endswith` (accept tuples), `in` |
| Cleaning | `strip`, `lstrip`, `rstrip` (character sets), `removeprefix`, `removesuffix` |
| Splitting and joining | `split()` vs `split(sep)`, `maxsplit`, `rsplit`, `splitlines`, `partition`, `rpartition`, `"sep".join(iterable)` |
| Replacing | `replace(old, new, count)`, `translate` with `str.maketrans`, `expandtabs` |
| Testing | `isdigit` vs `isdecimal` vs `isnumeric`, `isalpha`, `isalnum`, `isspace`, `isidentifier`, `islower`, `isupper`, `istitle`, `isascii`, `isprintable` |
| Layout | `center`, `ljust`, `rjust`, `zfill` |
| Encoding | `encode` |

Every method returns a **new** object; none of them changes the original string. Each
group below gets its own demo.

#### Case

`casefold` is a stronger `lower` meant for comparisons (1.4.4). `title` capitalizes
after *every* non-letter, including apostrophes.

In [107]:
city_name = "new DELHI"
print(f"{city_name.lower()=}")
print(f"{city_name.upper()=}")
print(f"{city_name.title()=}")
print(f"{city_name.capitalize()=}  (first character only, the rest lowered)")
print(f"{city_name.swapcase()=}")
print(f"{'Straße'.lower()=}, {'Straße'.casefold()=}")
print(f"{"o'neil's café".title()=}  (the apostrophe starts a new word)")

city_name.lower()='new delhi'
city_name.upper()='NEW DELHI'
city_name.title()='New Delhi'
city_name.capitalize()='New delhi'  (first character only, the rest lowered)
city_name.swapcase()='NEW delhi'
'Straße'.lower()='straße', 'Straße'.casefold()='strasse'
"o'neil's café".title()="O'Neil'S Café"  (the apostrophe starts a new word)


#### Searching

`find`/`rfind` return `-1` when nothing matches; `index`/`rindex` raise `ValueError`
instead. `startswith` and `endswith` accept a **tuple** of options. Use `in` when you
only need yes or no.

In [108]:
log_line = "GET /api/orders/42 200 GET"
print(f"{log_line.find('GET')=}, {log_line.rfind('GET')=}, {log_line.find('POST')=}")
print(f"{log_line.index('/api')=}, {log_line.rindex('GET')=}")
try:
    log_line.index("POST")
except ValueError as error:
    print(f"index: {type(error).__name__}: {error}")
print(f"{log_line.count('GET')=}, {log_line.count('/')=}")
file_name = "invoice_2026.PDF"
print(f"{file_name.startswith('invoice')=}")
print(f"{file_name.lower().endswith(('.pdf', '.png', '.jpg'))=}")
print(f"{'orders' in log_line=}")

log_line.find('GET')=0, log_line.rfind('GET')=23, log_line.find('POST')=-1
log_line.index('/api')=4, log_line.rindex('GET')=23
index: ValueError: substring not found
log_line.count('GET')=2, log_line.count('/')=3
file_name.startswith('invoice')=True
file_name.lower().endswith(('.pdf', '.png', '.jpg'))=True
'orders' in log_line=True


#### Cleaning

With no argument, `strip`, `lstrip` and `rstrip` remove whitespace. With an argument
they remove any characters **from that set**, not that exact prefix or suffix, which
surprises everyone once. Use `removeprefix` / `removesuffix` (3.9+) to drop an exact
piece of text.

In [109]:
raw_email = "  \t ramesh@example.com \n"
print(f"{raw_email.strip()=}")
print(f"{raw_email.lstrip()=}")
print(f"{raw_email.rstrip()=}")
print(f"{'xxhixyx'.strip('xy')=}  (any x or y, from both ends)")
site_name = "commerce.com"
print(f"{site_name.strip('.com')=}  (oops: strips the characters . c o m)")
print(f"{site_name.removesuffix('.com')=}")
print(f"{'www.example.in'.removeprefix('www.')=}")
print(f"{'example.in'.removeprefix('www.')=}  (no match: unchanged)")

raw_email.strip()='ramesh@example.com'
raw_email.lstrip()='ramesh@example.com \n'
raw_email.rstrip()='  \t ramesh@example.com'
'xxhixyx'.strip('xy')='hi'  (any x or y, from both ends)
site_name.strip('.com')='erce'  (oops: strips the characters . c o m)
site_name.removesuffix('.com')='commerce'
'www.example.in'.removeprefix('www.')='example.in'
'example.in'.removeprefix('www.')='example.in'  (no match: unchanged)


#### Splitting and joining

`split()` with no argument splits on any run of whitespace and drops empty edges;
`split(sep)` splits on exactly `sep` and keeps empty fields. `maxsplit` limits the
number of cuts (`rsplit` cuts from the right). `partition` always returns a 3-tuple,
so it is safe to unpack. `"sep".join(iterable)` is the inverse of `split`: the
separator is the object, and every item must already be a `str`.

In [110]:
messy_line = "  idli   dosa\tvada  "
print(f"{messy_line.split()=}")
print(f"{'a,,b'.split(',')=}  (empty field kept)")
print(f"{''.split()=}, {''.split(',')=}")
log_entry = "2026-10-30 12:00:01 ERROR payment failed: card declined"
print(f"{log_entry.split(' ', maxsplit=2)=}")
print(f"{'/srv/app/logs/today.log'.rsplit('/', maxsplit=1)=}")
print(f"{'line one\nline two\r\nline three'.splitlines()=}")
print(f"{'timeout=30'.partition('=')=}, {'no equals'.partition('=')=}")
print(f"{'report.final.pdf'.rpartition('.')=}")
print(f"{', '.join(['idli', 'dosa', 'vada'])=}")
print(f"join with ints: {raised_error_name(lambda: ', '.join([1, 2]))}")

messy_line.split()=['idli', 'dosa', 'vada']
'a,,b'.split(',')=['a', '', 'b']  (empty field kept)
''.split()=[], ''.split(',')=['']
log_entry.split(' ', maxsplit=2)=['2026-10-30', '12:00:01', 'ERROR payment failed: card declined']
'/srv/app/logs/today.log'.rsplit('/', maxsplit=1)=['/srv/app/logs', 'today.log']
'line one\nline two\r\nline three'.splitlines()=['line one', 'line two', 'line three']
'timeout=30'.partition('=')=('timeout', '=', '30'), 'no equals'.partition('=')=('no equals', '', '')
'report.final.pdf'.rpartition('.')=('report.final', '.', 'pdf')
', '.join(['idli', 'dosa', 'vada'])='idli, dosa, vada'
join with ints: TypeError


#### Replacing

`replace(old, new, count)` replaces every match unless you pass `count`.
`str.maketrans` builds a table for `translate`: map characters one to one, or map a
character to `None` to delete it. `expandtabs` turns tabs into spaces, aligned to tab
stops.

In [111]:
phone_number = "98-45-67-89-01"
print(f"{phone_number.replace('-', '')=}")
print(f"{phone_number.replace('-', ' ', 2)=}  (count limits the replacements)")
vowel_masker = str.maketrans("aeiou", "*****")
print(f"{'bangalore'.translate(vowel_masker)=}")
amount_cleaner = str.maketrans({"₹": None, ",": None, " ": None})
print(f"{'₹ 1,23,456'.translate(amount_cleaner)=}")
print(f"{str.maketrans('ab', 'xy', '!')=}  (a dict of code points)")
tabbed_row = "id\tname\tcity"
print(f"{tabbed_row.expandtabs(8)=}")
print(f"{tabbed_row.expandtabs(4)=}")

phone_number.replace('-', '')='9845678901'
phone_number.replace('-', ' ', 2)='98 45 67-89-01'  (count limits the replacements)
'bangalore'.translate(vowel_masker)='b*ng*l*r*'
'₹ 1,23,456'.translate(amount_cleaner)='123456'
str.maketrans('ab', 'xy', '!')={97: 120, 98: 121, 33: None}  (a dict of code points)
tabbed_row.expandtabs(8)='id      name    city'
tabbed_row.expandtabs(4)='id  name    city'


#### Testing

The three numeric tests differ, and only `isdecimal` matches what `int()` accepts:

| Method | True for | Example that is True |
| --- | --- | --- |
| `isdecimal` | digits in any script (`int()` accepts these) | `"42"`, `"٣"` (Arabic-Indic three) |
| `isdigit` | decimals plus superscripts and similar | `"²"` |
| `isnumeric` | digits plus fractions and numeral characters | `"½"`, `"四"` |

None of them accepts a sign or a decimal point.

In [112]:
print(f"{'sample':>8} {'isdecimal':>10} {'isdigit':>8} {'isnumeric':>10}")
for numeric_text in ["42", "٣", "²", "½", "四", "4.2", "-1"]:
    print(
        f"{numeric_text!r:>8} {numeric_text.isdecimal()!s:>10} "
        f"{numeric_text.isdigit()!s:>8} {numeric_text.isnumeric()!s:>10}"
    )
print(f"{int('٣')=}  (isdecimal text converts)")
print(f"int('²'): {raised_error_name(lambda: int('²'))}")

  sample  isdecimal  isdigit  isnumeric
    '42'       True     True       True
     '٣'       True     True       True
     '²'      False     True       True
     '½'      False    False       True
     '四'      False    False       True
   '4.2'      False    False      False
    '-1'      False    False      False
int('٣')=3  (isdecimal text converts)
int('²'): ValueError


The other tests. Every `is…` method returns `False` for the empty string, except
`isascii` and `isprintable`. `isidentifier` does not reject keywords; use
`keyword.iskeyword` for that.

In [113]:
print(f"{'dosa'.isalpha()=}, {'dosa2'.isalpha()=}, {'dosa2'.isalnum()=}")
print(f"{' \t\n'.isspace()=}, {''.isspace()=}")
print(f"{'order_id'.isidentifier()=}, {'2nd_order'.isidentifier()=}")
print(f"{'class'.isidentifier()=}  (a keyword, but still a valid identifier shape)")
print(f"{'dosa'.islower()=}, {'DOSA 2'.isupper()=}, {'Masala Dosa'.istitle()=}")
print(f"{'dosa'.isascii()=}, {'café'.isascii()=}, {''.isascii()=}")
print(f"{'plain text'.isprintable()=}, {'tab\there'.isprintable()=}")

'dosa'.isalpha()=True, 'dosa2'.isalpha()=False, 'dosa2'.isalnum()=True
' \t\n'.isspace()=True, ''.isspace()=False
'order_id'.isidentifier()=True, '2nd_order'.isidentifier()=False
'class'.isidentifier()=True  (a keyword, but still a valid identifier shape)
'dosa'.islower()=True, 'DOSA 2'.isupper()=True, 'Masala Dosa'.istitle()=True
'dosa'.isascii()=True, 'café'.isascii()=False, ''.isascii()=True
'plain text'.isprintable()=True, 'tab\there'.isprintable()=False


#### Layout

`ljust`, `rjust` and `center` pad to a width with an optional fill character; they
never truncate. `zfill` pads with zeros after the sign. The format-spec mini-language
in 1.4.3 does the same and more.

In [114]:
for menu_item, price_in_rupees in [("Idli", 40), ("Masala Dosa", 120), ("Coffee", 35)]:
    print(menu_item.ljust(16, ".") + str(price_in_rupees).rjust(6))
print(" Total ".center(22, "="))
print(f"{'42'.zfill(6)=}, {'-42'.zfill(6)=}  (zfill keeps the sign first)")
print(f"{'Masala Dosa'.ljust(5)=}  (never truncates)")

Idli............    40
Masala Dosa.....   120
Coffee..........    35
======= Total ========
'42'.zfill(6)='000042', '-42'.zfill(6)='-00042'  (zfill keeps the sign first)
'Masala Dosa'.ljust(5)='Masala Dosa'  (never truncates)


#### Encoding

`encode` turns text into `bytes` (UTF-8 unless you name another codec). A character
the codec cannot represent raises `UnicodeEncodeError` unless you choose an `errors`
policy; 1.4.5 covers the policies.

In [115]:
greeting_text = "Namaste ₹"
encoded_greeting = greeting_text.encode()
print(f"{encoded_greeting=}, {len(greeting_text)=}, {len(encoded_greeting)=}")
print(f"{greeting_text.encode('utf-16-le')=}")
try:
    greeting_text.encode("ascii")
except UnicodeEncodeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{greeting_text.encode('ascii', errors='replace')=}")

encoded_greeting=b'Namaste \xe2\x82\xb9', len(greeting_text)=9, len(encoded_greeting)=11
greeting_text.encode('utf-16-le')=b'N\x00a\x00m\x00a\x00s\x00t\x00e\x00 \x00\xb9 '
UnicodeEncodeError: 'ascii' codec can't encode character '\u20b9' in position 8: ordinal not in range(128)
greeting_text.encode('ascii', errors='replace')=b'Namaste ?'


### 1.4.3 String formatting

**f-strings** (PEP 498) evaluate any expression inside `{}`. Extras:

| Syntax | Meaning |
| --- | --- |
| `{expr=}` | debug: prints `expr=` and then the `repr` of its value (spaces around `=` are kept) |
| `{value!r}`, `{value!s}`, `{value!a}` | apply `repr()`, `str()` or `ascii()` before formatting |
| `{value:>{width}}` | nested field: the format spec itself contains a replacement field |
| multi-line `{…}` | since 3.12 (PEP 701) an expression may span lines, contain comments and reuse the outer quote character |

In [116]:
item_name = "Masala Dosa"
unit_price_in_rupees = 120
quantity = 3
print(f"{item_name} x {quantity} = ₹{unit_price_in_rupees * quantity}")
print(f"{item_name.upper()=}, {quantity * unit_price_in_rupees = }")
print(f"{item_name!r}, {item_name!s}, {'café ₹'!a}")
column_width = 15
print(f"[{item_name:>{column_width}}]  (width taken from a variable)")
print(f"[{item_name:^{column_width + 5}}]")

Masala Dosa x 3 = ₹360
item_name.upper()='MASALA DOSA', quantity * unit_price_in_rupees = 360
'Masala Dosa', Masala Dosa, 'caf\xe9 \u20b9'
[    Masala Dosa]  (width taken from a variable)
[    Masala Dosa     ]


A multi-line f-string (3.12+): the replacement field spans several lines and contains
a comment, with no backslashes.

In [117]:
order_lines = [("Idli", 2), ("Vada", 1), ("Coffee", 2)]
order_summary = f"Items: {
    ', '.join(
        f'{dish} x{count}'  # a comment inside the replacement field
        for dish, count in order_lines
    )
}"
print(order_summary)

Items: Idli x2, Vada x1, Coffee x2


**Format-spec mini-language** (everything after the `:`):

```text
[[fill]align][sign]["z"]["#"]["0"][width][grouping][.precision][type]
```

| Part | Options | Example → result |
| --- | --- | --- |
| fill + align | `<` left, `>` right, `^` centre, any fill character | `f"{'hi':*^6}"` → `**hi**` |
| sign | `+` always, `-` negatives only (default), space for a leading space | `f"{42:+d}"` → `+42` |
| `#` | alternate form: `0x`, `0b`, `0o` prefixes | `f"{255:#x}"` → `0xff` |
| width | minimum field width | `f"{7:5d}"` → `    7` |
| grouping | `,` or `_` thousands separator | `f"{1234567:,}"` → `1,234,567` |
| precision | digits after the point (or max characters for strings) | `f"{3.14159:.2f}"` → `3.14` |
| type | `d` int, `f` fixed, `e` exponent, `%` percent, `x` hex, `b` binary, `o` octal | `f"{0.256:.1%}"` → `25.6%` |

Dates have their own spec, passed to `strftime`: `f"{today:%Y-%m-%d}"`.

> **Note:** grouping is always in thousands (`12,34,567` lakh style is not built in).

In [118]:
order_total = 1234567.891
print(f"[{'left':<10}] [{'right':>10}] [{'centre':^10}] [{'fill':*^10}]")
print(f"{order_total:,.2f}  {order_total:_.1f}  {1_000_000:,d}  {1_000_000:_d}")
print(f"{order_total:.3e}  {0.1875:.1%}  [{order_total:15,.2f}]")
print(f"{255:x} {255:#x} {255:X} {5:b} {5:#010b} {8:o} {8:#o}")
print(f"[{42:+d}] [{-42:+d}] [{42: d}] [{42:d}]")
invoice_date = date(2026, 10, 30)
print(f"{invoice_date:%Y-%m-%d}  {invoice_date:%d %b %Y, %A}")
print(f"[{'truncated text':.9}]  (precision on a string cuts it)")

[left      ] [     right] [  centre  ] [***fill***]
1,234,567.89  1_234_567.9  1,000,000  1_000_000
1.235e+06  18.8%  [   1,234,567.89]
ff 0xff FF 101 0b00000101 10 0o10
[+42] [-42] [ 42] [42]
2026-10-30  30 Oct 2026, Friday
[truncated]  (precision on a string cuts it)


**Legacy styles** you will meet in older code and libraries:

| Style | Where you still see it |
| --- | --- |
| `"{} {name}".format(...)` | code from before 3.6; building a template once and filling it later |
| `"{name}".format_map(mapping)` | filling from a dict (or a dict subclass with `__missing__`) |
| `"%s %.2f" % (...)` | the `logging` module's lazy messages, old C-style code |
| `string.Template("$name")` | user-editable templates: no expressions, no attribute access |

The `%` line carries `# noqa: UP031` because ruff (rightly) suggests f-strings instead.

In [119]:
customer_record = {"name": "Asha", "balance": 2500.5}
print("{} owes ₹{:,.2f}".format("Asha", 2500.5))
print("{name} owes ₹{balance:,.2f}".format(**customer_record))
print("{name} owes ₹{balance:,.2f}".format_map(customer_record))
print("%s owes ₹%.2f (%d%% overdue)" % ("Asha", 2500.5, 12))  # noqa: UP031
greeting_template = string.Template("Hello $name, order $order_id is ready")
print(greeting_template.substitute(name="Asha", order_id="A-17"))
print(greeting_template.safe_substitute(name="Asha"))
try:
    greeting_template.substitute(name="Asha")
except KeyError as error:
    print(f"substitute: {type(error).__name__}: {error}")

Asha owes ₹2,500.50
Asha owes ₹2,500.50
Asha owes ₹2,500.50
Asha owes ₹2500.50 (12% overdue)
Hello Asha, order A-17 is ready
Hello Asha, order $order_id is ready
substitute: KeyError: 'order_id'


**Template strings, t-strings** (*new in 3.14*, PEP 750). A `t"..."` literal looks like
an f-string but produces a `string.templatelib.Template` object instead of a `str`:

- `.strings`: the literal text parts (always one more than the interpolations).
- `.interpolations`: `Interpolation` objects with `.value`, `.expression`,
  `.conversion` and `.format_spec`.
- Iterating yields the non-empty strings and interpolations in order.

Nothing is joined until **your** code renders it. That is the point: a renderer can
escape every value for HTML, turn values into SQL parameters, and so on.

In [120]:
customer_name = "Asha"
amount_due = 1234.5
reminder_template = t"Dear {customer_name}, you owe ₹{amount_due:,.2f}"
print(f"{type(reminder_template)=}")
print(f"{reminder_template.strings=}")
for interpolation in reminder_template.interpolations:
    print(f"{interpolation.expression=}, {interpolation.value=}, {interpolation.format_spec=}")
print(f"{list(reminder_template)=}")

type(reminder_template)=<class 'string.templatelib.Template'>
reminder_template.strings=('Dear ', ', you owe ₹', '')
interpolation.expression='customer_name', interpolation.value='Asha', interpolation.format_spec=''
interpolation.expression='amount_due', interpolation.value=1234.5, interpolation.format_spec=',.2f'
list(reminder_template)=['Dear ', Interpolation('Asha', 'customer_name', None, ''), ', you owe ₹', Interpolation(1234.5, 'amount_due', None, ',.2f')]


A safe HTML renderer: literal parts are trusted (the developer wrote them),
interpolated values are not, so each value is formatted with its spec and then passed
through `html.escape`. An f-string would have put the script tag straight into the
page.

In [121]:
def render_safe_html(template: Template) -> str:
    """Render a t-string, HTML-escaping every interpolated value."""
    rendered_parts: list[str] = []
    for template_part in template:
        if isinstance(template_part, Interpolation):
            formatted_value = format(template_part.value, template_part.format_spec)
            rendered_parts.append(html.escape(formatted_value))
        else:
            rendered_parts.append(template_part)
    return "".join(rendered_parts)


user_comment = "<script>alert('stolen cookie')</script>"
print("f-string:", f"<p>{user_comment}</p>")
print("t-string:", render_safe_html(t"<p>{user_comment}</p>"))

f-string: <p><script>alert('stolen cookie')</script></p>
t-string: <p>&lt;script&gt;alert(&#x27;stolen cookie&#x27;)&lt;/script&gt;</p>


### 1.4.4 Unicode correctness

The same visible text can be stored as different code-point sequences: `é` can be one
precomposed code point (U+00E9) or `e` followed by a **combining** acute accent
(U+0301). `==` compares code points, so the two are not equal until you
**normalize** them:

| Form | What it does |
| --- | --- |
| NFC | compose: `e` + `\u0301` → `é` (the usual storage form) |
| NFD | decompose: `é` → `e` + `\u0301` |
| NFKC / NFKD | also fold *compatibility* characters: `ﬁ` → `fi`, full-width `２` → `2`, `²` → `2` |

In [122]:
precomposed_name = "Jos\u00e9"
combining_name = "Jose\u0301"
print(f"{precomposed_name=}, {combining_name=}")
print(f"{precomposed_name == combining_name=}")
print(f"{len(precomposed_name)=}, {len(combining_name)=}")
nfc_combining_name = unicodedata.normalize("NFC", combining_name)
print(f"{nfc_combining_name == precomposed_name=}")
print(f"{len(unicodedata.normalize('NFD', precomposed_name))=}")
print(f"{unicodedata.is_normalized('NFC', combining_name)=}")
print(f"{unicodedata.normalize('NFKC', 'ﬁle ２０２６ x²')=}")

precomposed_name='José', combining_name='José'
precomposed_name == combining_name=False
len(precomposed_name)=4, len(combining_name)=5
nfc_combining_name == precomposed_name=True
len(unicodedata.normalize('NFD', precomposed_name))=5
unicodedata.is_normalized('NFC', combining_name)=False
unicodedata.normalize('NFKC', 'ﬁle ２０２６ x²')='file 2026 x2'


**Case-insensitive comparison** needs `casefold`, not `lower`: the German `ß` lowers
to itself but casefolds to `ss`. For a fully correct caseless match, normalize *and*
casefold both sides. `unicodedata.name` tells you exactly which code point you are
looking at, which helps with "identical-looking" bugs.

In [123]:
def texts_match(first_text: str, second_text: str) -> bool:
    """Compare text the Unicode way: NFC-normalize and casefold both sides."""
    first_key = unicodedata.normalize("NFC", first_text).casefold()
    second_key = unicodedata.normalize("NFC", second_text).casefold()
    return first_key == second_key


print(f"{'Straße'.lower() == 'STRASSE'.lower()=}")
print(f"{'Straße'.casefold() == 'STRASSE'.casefold()=}")
print(f"{texts_match('JOSE\u0301', 'josé')=}")
for character in "é₹ß😀\u0301\u00a0":
    print(f"U+{ord(character):04X} {unicodedata.name(character)}")

'Straße'.lower() == 'STRASSE'.lower()=False
'Straße'.casefold() == 'STRASSE'.casefold()=True
texts_match('JOSE\u0301', 'josé')=True
U+00E9 LATIN SMALL LETTER E WITH ACUTE
U+20B9 INDIAN RUPEE SIGN
U+00DF LATIN SMALL LETTER SHARP S
U+1F600 GRINNING FACE
U+0301 COMBINING ACUTE ACCENT
U+00A0 NO-BREAK SPACE


### 1.4.5 Bytes (`bytes`) and byte arrays (`bytearray`)

`str` is text (code points); `bytes` is binary data (integers 0–255). You convert at
the boundary: **encode** text when it leaves your program (files, sockets, hashes) and
**decode** bytes when they come in.

```mermaid
flowchart LR
    text["str<br/>'₹500'"] -- "encode('utf-8')" --> raw["bytes<br/>b'\xe2\x82\xb9500'"]
    raw -- "decode('utf-8')" --> text
```

<details><summary>ASCII fallback</summary>

```text
str '₹500'  --encode('utf-8')-->  bytes b'\xe2\x82\xb9500'
            <--decode('utf-8')--
```

</details>

A `b"..."` literal may contain only ASCII characters and escapes. **Indexing a
`bytes` object returns an `int`**; slicing returns `bytes`. `bytes` is immutable,
`bytearray` is its mutable twin.

In [124]:
png_header = b"\x89PNG\r\n\x1a\n"
http_request_line = b"GET / HTTP/1.1\r\n"
print(f"{png_header=}, {len(png_header)=}")
print(f"{http_request_line[0]=}  (indexing gives an int)")
print(f"{http_request_line[:3]=}  (slicing gives bytes)")
print(f"{list(b'AZ az')=}")
try:
    eval('b"café"')
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

png_header=b'\x89PNG\r\n\x1a\n', len(png_header)=8
http_request_line[0]=71  (indexing gives an int)
http_request_line[:3]=b'GET'  (slicing gives bytes)
list(b'AZ az')=[65, 90, 32, 97, 122]
SyntaxError: bytes can only contain ASCII literal characters


`bytes` cannot change after it is created. `bytearray` supports item assignment,
`extend`, `append` and `del`, and `bytes(…)` freezes a copy of it.

In [125]:
frozen_payload = b"hello"
try:
    frozen_payload[0] = 72
except TypeError as error:
    print(f"bytes: {type(error).__name__}: {error}")
mutable_buffer = bytearray(b"hello")
mutable_buffer[0] = ord("H")
mutable_buffer.extend(b" world")
del mutable_buffer[-1]
print(f"{mutable_buffer=}, {bytes(mutable_buffer)=}")
print(f"{bytearray(4)=}  (four zero bytes)")

bytes: TypeError: 'bytes' object does not support item assignment
mutable_buffer=bytearray(b'Hello worl'), bytes(mutable_buffer)=b'Hello worl'
bytearray(4)=bytearray(b'\x00\x00\x00\x00')  (four zero bytes)


**`encode` / `decode` and the `errors` argument.** The default `errors="strict"`
raises on bad data. The alternatives:

| `errors=` | On a bad character or byte |
| --- | --- |
| `strict` | raise `UnicodeEncodeError` / `UnicodeDecodeError` (default) |
| `replace` | substitute `?` (encode) or `�` U+FFFD (decode) |
| `ignore` | drop it silently (data loss!) |
| `backslashreplace` | write a `\x..` / `\u....` escape so nothing is lost |

The bytes below are cut in the middle of the 3-byte `₹`, which is exactly what happens
when a network read stops halfway through a character.

In [126]:
customer_note = "Paid ₹500 — thanks"
note_bytes = customer_note.encode("utf-8")
print(f"{note_bytes=}")
print(f"{note_bytes.decode('utf-8')=}")
truncated_bytes = note_bytes[:7]
for error_policy in ["strict", "replace", "ignore", "backslashreplace"]:
    try:
        decoded_text = truncated_bytes.decode("utf-8", errors=error_policy)
    except UnicodeDecodeError as error:
        decoded_text = f"{type(error).__name__}: {error}"
    print(f"decode {error_policy:>16}: {decoded_text!r}")
print(f"{customer_note.encode('ascii', errors='backslashreplace')=}")
print(f"{customer_note.encode('ascii', errors='ignore')=}")

note_bytes=b'Paid \xe2\x82\xb9500 \xe2\x80\x94 thanks'
note_bytes.decode('utf-8')='Paid ₹500 — thanks'
decode           strict: "UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 5-6: unexpected end of data"
decode          replace: 'Paid �'
decode           ignore: 'Paid '
decode backslashreplace: 'Paid \\xe2\\x82'
customer_note.encode('ascii', errors='backslashreplace')=b'Paid \\u20b9500 \\u2014 thanks'
customer_note.encode('ascii', errors='ignore')=b'Paid 500  thanks'


**UTF-8 by default** (*changed in 3.15*, PEP 686). Python 3.15 turns on **UTF-8 mode**
by default, so `open()` without `encoding=` reads and writes UTF-8 on every OS, no
matter the locale (before 3.15, Windows used the ANSI code page). `locale.getencoding()`
still reports the OS locale. Set `PYTHONUTF8=0` to opt out. Code that must also run on
older Pythons should keep passing `encoding="utf-8"` explicitly.

In [127]:
print(f"{sys.flags.utf8_mode=}")
print(f"{locale.getpreferredencoding(False)=}  (what open() uses without encoding=)")
print(f"{locale.getencoding()=}  (the OS locale encoding)")
if sys.version_info >= (3, 15):
    print("Python 3.15+: UTF-8 mode is on by default (PEP 686)")
else:
    print("Before 3.15: open() uses the locale encoding; pass encoding='utf-8'")

sys.flags.utf8_mode=1
locale.getpreferredencoding(False)='utf-8'  (what open() uses without encoding=)
locale.getencoding()='UTF-8'  (the OS locale encoding)
Python 3.15+: UTF-8 mode is on by default (PEP 686)


**Methods shared with `str`.** Most `str` methods exist on `bytes` and `bytearray`, but
their arguments must be bytes too. `hex()` and `bytes.fromhex()` convert to and from
hex text, which you see in checksums and protocol dumps.

In [128]:
raw_header = b"Content-Type: text/html; charset=utf-8"
print(f"{raw_header.split(b': ', maxsplit=1)=}")
print(f"{raw_header.replace(b'html', b'plain')=}")
print(f"{raw_header.startswith(b'Content')=}, {raw_header.upper().find(b'UTF')=}")
print(f"{raw_header.partition(b'; ')=}")
print(f"{b'  padded  '.strip()=}, {b'abc'.isalpha()=}")
checksum_bytes = bytes([0xDE, 0xAD, 0xBE, 0xEF])
print(f"{checksum_bytes.hex()=}, {checksum_bytes.hex(':')=}")
print(f"{bytes.fromhex('de ad be ef')=}")

raw_header.split(b': ', maxsplit=1)=[b'Content-Type', b'text/html; charset=utf-8']
raw_header.replace(b'html', b'plain')=b'Content-Type: text/plain; charset=utf-8'
raw_header.startswith(b'Content')=True, raw_header.upper().find(b'UTF')=33
raw_header.partition(b'; ')=(b'Content-Type: text/html', b'; ', b'charset=utf-8')
b'  padded  '.strip()=b'padded', b'abc'.isalpha()=True
checksum_bytes.hex()='deadbeef', checksum_bytes.hex(':')='de:ad:be:ef'
bytes.fromhex('de ad be ef')=b'\xde\xad\xbe\xef'


**Mixing `str` and `bytes` raises `TypeError`.** Python 2 silently mixed them and
corrupted text; Python 3 refuses, so the bug shows up where it starts. Equality is the
one exception: it simply returns `False`. The fix is always to decode (or encode) at
the boundary and then work in one type.

In [129]:
text_part = "order="
byte_part = b"42"
try:
    text_part + byte_part
except TypeError as error:
    print(f"str + bytes: {type(error).__name__}: {error}")
print(f"bytes in str: {raised_error_name(lambda: byte_part in text_part)}")
print(f"str.join([str, bytes]): {raised_error_name(lambda: ''.join([text_part, byte_part]))}")
print(f"{b'42' == '42'=}  (no error, just never equal)")
print(f"{text_part + byte_part.decode()=}  (decode first, then work in str)")

str + bytes: TypeError: can only concatenate str (not "bytes") to str
bytes in str: TypeError
str.join([str, bytes]): TypeError
b'42' == '42'=False  (no error, just never equal)
text_part + byte_part.decode()='order=42'  (decode first, then work in str)


### 1.4.6 Memory views (`memoryview`)

Slicing `bytes` **copies** the data. A `memoryview` exposes another object's buffer
without copying: slicing a view creates another view, and over a `bytearray` you can
write through it. This matters when you parse large binary payloads. Phase 5 returns
to buffers.

> **Note:** while a view exists, the `bytearray` cannot be resized.

In [130]:
frame_buffer = bytearray(b"HEADER:payload-data:CRC")
header_view = memoryview(frame_buffer)[:6]
payload_view = memoryview(frame_buffer)[7:19]
print(f"{bytes(header_view)=}, {bytes(payload_view)=}")
print(f"{payload_view.nbytes=}, {payload_view.obj is frame_buffer=}, {payload_view.readonly=}")
payload_view[:7] = b"PAYLOAD"
print(f"{frame_buffer=}  (written through the view)")
print(f"append while viewed: {raised_error_name(lambda: frame_buffer.extend(b'!'))}")
large_blob = bytes(10_000_000)
print(f"bytes slice:      {sys.getsizeof(large_blob[1:]):>12,} bytes (a copy)")
print(f"memoryview slice: {sys.getsizeof(memoryview(large_blob)[1:]):>12,} bytes")
header_view.release()
payload_view.release()

bytes(header_view)=b'HEADER', bytes(payload_view)=b'payload-data'
payload_view.nbytes=12, payload_view.obj is frame_buffer=True, payload_view.readonly=False
frame_buffer=bytearray(b'HEADER:PAYLOAD-data:CRC')  (written through the view)
append while viewed: BufferError
bytes slice:        10,000,032 bytes (a copy)
memoryview slice:          184 bytes


### 1.4.7 Real-world use cases

**Normalizing user input.** Store and compare a canonical form: strip the edges,
lowercase or casefold, and reject what is not valid. Here, emails: domains are
case-insensitive, and in practice the whole address is treated that way. Lab 1.4 does
the same for usernames and search queries.

In [131]:
def normalize_email(raw_email: str) -> str:
    """Return a trimmed, lowercased email address, or raise ValueError."""
    cleaned_email = raw_email.strip().lower()
    local_part, separator, domain_part = cleaned_email.rpartition("@")
    if not separator or not local_part or "." not in domain_part:
        raise ValueError(f"not an email address: {raw_email!r}")
    return cleaned_email


for raw_email in ["  Asha.K@Example.COM ", "ravi@mail.in", "no-at-sign.com"]:
    try:
        print(f"{raw_email!r:24} -> {normalize_email(raw_email)!r}")
    except ValueError as error:
        print(f"{raw_email!r:24} -> {type(error).__name__}: {error}")

'  Asha.K@Example.COM '  -> 'asha.k@example.com'
'ravi@mail.in'           -> 'ravi@mail.in'
'no-at-sign.com'         -> ValueError: not an email address: 'no-at-sign.com'


**Parsing structured text.** `split(maxsplit=…)` handles fixed-shape log lines, and
`partition` handles `key=value` lines (it never fails to unpack). `parse_log_line`
returns to the Phase 1 project, the log analyzer.

In [132]:
def parse_log_line(log_line: str) -> dict[str, str]:
    """Split 'DATE TIME LEVEL message' into its four named fields."""
    log_date, log_time, level, message = log_line.split(maxsplit=3)
    return {"date": log_date, "time": log_time, "level": level, "message": message}


def parse_key_value_line(config_line: str) -> tuple[str, str] | None:
    """Parse 'key = value'; return None for blank and comment-only lines."""
    content = config_line.partition("#")[0].strip()
    if not content:
        return None
    key, separator, value = content.partition("=")
    if not separator:
        raise ValueError(f"missing '=' in {config_line!r}")
    return key.strip(), value.strip()


print(parse_log_line("2026-10-30 12:00:01 ERROR payment failed: card declined"))

{'date': '2026-10-30', 'time': '12:00:01', 'level': 'ERROR', 'message': 'payment failed: card declined'}


Feeding a small config file through `parse_key_value_line`: blank lines and comments
are skipped, and the spaces around `=` are optional.

In [133]:
config_text = """
# database settings
host = db.internal
port=5432
timeout = 30  # seconds
"""
settings: dict[str, str] = {}
for config_line in config_text.splitlines():
    if (parsed_setting := parse_key_value_line(config_line)) is not None:
        setting_key, setting_value = parsed_setting
        settings[setting_key] = setting_value
print(f"{settings=}")

settings={'host': 'db.internal', 'port': '5432', 'timeout': '30'}


CSV fields: `split(",")` is fine for simple, unquoted data. As soon as a field can
contain a comma inside quotes, use the `csv` module.

In [134]:
simple_csv_row = "A-17,Asha,1200.50"
order_id, customer, amount_text = simple_csv_row.split(",")
print(f"{order_id=}, {customer=}, {Decimal(amount_text)=}")
quoted_csv_row = 'A-18,"Kumar, Asha",950.00'
print(f"{quoted_csv_row.split(',')=}  (wrong: 4 fields)")
print(f"{next(csv.reader([quoted_csv_row]))=}  (right: 3 fields)")

order_id='A-17', customer='Asha', Decimal(amount_text)=Decimal('1200.50')
quoted_csv_row.split(',')=['A-18', '"Kumar', ' Asha"', '950.00']  (wrong: 4 fields)
next(csv.reader([quoted_csv_row]))=['A-18', 'Kumar, Asha', '950.00']  (right: 3 fields)


**Reports with the format-spec mini-language.** Fixed-width columns, thousands
separators and percentages need no string arithmetic.

In [135]:
revenue_by_region = {
    "North": 1_250_000.0,
    "South": 2_480_500.5,
    "East": 975_250.25,
    "West": 1_810_000.0,
}
total_revenue = sum(revenue_by_region.values())
print(f"{'Region':<8}{'Revenue (₹)':>16}{'Share':>9}")
print("-" * 33)
for region_name, region_revenue in revenue_by_region.items():
    print(f"{region_name:<8}{region_revenue:>16,.2f}{region_revenue / total_revenue:>9.1%}")
print("-" * 33)
print(f"{'Total':<8}{total_revenue:>16,.2f}{1:>9.0%}")

Region       Revenue (₹)    Share
---------------------------------
North       1,250,000.00    19.2%
South       2,480,500.50    38.1%
East          975,250.25    15.0%
West        1,810,000.00    27.8%
---------------------------------
Total       6,515,750.75     100%


**URL slugs.** NFKD splits accented letters into base letter plus mark; encoding to
ASCII with `errors="ignore"` drops the marks; every run of other characters becomes
one hyphen.

In [136]:
def slugify(title: str) -> str:
    """Turn a title into a URL slug: ASCII, lowercase, words joined by hyphens."""
    decomposed_title = unicodedata.normalize("NFKD", title)
    ascii_title = decomposed_title.encode("ascii", errors="ignore").decode("ascii")
    words = "".join(
        character if character.isalnum() else " " for character in ascii_title.lower()
    ).split()
    return "-".join(words)


for blog_title in ["Crème Brûlée: 10 Best Recipes (2026)!", "  Hello,   World  "]:
    print(f"{blog_title!r} -> {slugify(blog_title)!r}")

'Crème Brûlée: 10 Best Recipes (2026)!' -> 'creme-brulee-10-best-recipes-2026'
'  Hello,   World  ' -> 'hello-world'


**Safe SQL with t-strings.** `render_sql` turns a t-string into a parameterized query:
each interpolation becomes a `?` placeholder and its value goes into a separate tuple,
so the database never treats data as SQL. Compare it with an f-string built query on
an in-memory SQLite database. (`render_safe_html` from 1.4.3 covers the HTML side.)

In [137]:
def render_sql(template: Template) -> tuple[str, tuple[object, ...]]:
    """Turn a t-string into '?' placeholders plus a tuple of parameters."""
    query_parts: list[str] = []
    query_parameters: list[object] = []
    for template_part in template:
        if isinstance(template_part, Interpolation):
            query_parts.append("?")
            query_parameters.append(template_part.value)
        else:
            query_parts.append(template_part)
    return "".join(query_parts), tuple(query_parameters)


print(render_sql(t"SELECT id FROM orders WHERE customer = {'Asha'} AND total > {500}"))

('SELECT id FROM orders WHERE customer = ? AND total > ?', ('Asha', 500))


Now against a real (in-memory) database. The hostile name closes the quote in the
f-string version and matches every row; as a parameter it is just an odd name that
matches nothing.

In [138]:
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE orders (id INTEGER, customer TEXT, total INTEGER)")
connection.executemany("INSERT INTO orders VALUES (?, ?, ?)", [(1, "Asha", 900), (2, "Ravi", 300)])
hostile_name = "nobody' OR '1'='1"
unsafe_query = f"SELECT id FROM orders WHERE customer = '{hostile_name}'"
print(f"f-string query rows: {connection.execute(unsafe_query).fetchall()}  (injected!)")
safe_query, safe_parameters = render_sql(t"SELECT id FROM orders WHERE customer = {hostile_name}")
print(f"t-string query rows: {connection.execute(safe_query, safe_parameters).fetchall()}")
connection.close()

f-string query rows: [(1,), (2,)]  (injected!)
t-string query rows: []


**`bytes` in the real world.** Network protocols and file formats are bytes. Files
announce their type in the first few bytes (the "magic number"), hashes work on bytes,
and base64 turns bytes into ASCII text that is safe for JSON, emails and URLs.

In [139]:
FILE_SIGNATURES: dict[bytes, str] = {
    b"\x89PNG\r\n\x1a\n": "PNG image",
    b"%PDF-": "PDF document",
    b"PK\x03\x04": "ZIP archive (also .docx, .xlsx)",
    b"\xff\xd8\xff": "JPEG image",
}


def detect_file_type(file_header: bytes) -> str:
    """Name the file type from its first bytes, or 'unknown'."""
    for signature, file_type in FILE_SIGNATURES.items():
        if file_header.startswith(signature):
            return file_type
    return "unknown"


for sample_header in [b"%PDF-1.7\n%\xe2\xe3", b"\x89PNG\r\n\x1a\n\x00\x00", b"hello"]:
    print(f"{sample_header[:8]!r:36} -> {detect_file_type(sample_header)}")

b'%PDF-1.7'                          -> PDF document
b'\x89PNG\r\n\x1a\n'                 -> PNG image
b'hello'                             -> unknown


A JSON request body: its UTF-8 byte length (not its character count) is the HTTP
`Content-Length`; SHA-256 fingerprints it; base64 makes it ASCII-safe. The URL-safe
alphabet swaps `+` and `/` for `-` and `_`.

In [140]:
request_body = '{"item": "dosa", "price": "₹120"}'
payload_bytes = request_body.encode("utf-8")
print(f"{len(request_body)=} characters, {len(payload_bytes)=} bytes (Content-Length)")
payload_digest = hashlib.sha256(payload_bytes).hexdigest()
print(f"{payload_digest=}")
encoded_payload = base64.b64encode(payload_bytes)
print(f"{encoded_payload=}")
print(f"{base64.b64decode(encoded_payload).decode('utf-8')=}")
print(f"{base64.b64encode(b'\xfb\xff')=}, {base64.urlsafe_b64encode(b'\xfb\xff')=}")

len(request_body)=33 characters, len(payload_bytes)=35 bytes (Content-Length)
payload_digest='9fb30cb1ec0a47b5b0e8d4a3698a760e662aa6cfdc88d4a0cd2f97b49e87025e'
encoded_payload=b'eyJpdGVtIjogImRvc2EiLCAicHJpY2UiOiAi4oK5MTIwIn0='
base64.b64decode(encoded_payload).decode('utf-8')='{"item": "dosa", "price": "₹120"}'
base64.b64encode(b'\xfb\xff')=b'+/8=', base64.urlsafe_b64encode(b'\xfb\xff')=b'-_8='


### Lab 1.4

Three text-cleaning functions of the kind every web application needs. For each one,
replace the `raise NotImplementedError` in the stub, then run the check cell. Until
then the checks print FAIL with `NotImplementedError`; that is expected.

#### Exercise 1 — `normalize_username`

Write `normalize_username(raw_username: str) -> str`:

1. Normalize with **NFKC** (full-width `ＡＳＨＡ` becomes `ASHA`; `e` + combining accent
   becomes `é`).
2. Strip the edges and **casefold**.
3. Replace each run of inner whitespace with a single `_`.
4. Raise `ValueError` if the result is empty, or if it contains any character that is
   not alphanumeric (`str.isalnum`), `_` or `.`.

`normalize_username("  Ravi   Shankar ") == "ravi_shankar"`.

In [141]:
def normalize_username(raw_username: str) -> str:
    """Return the canonical form of a username, or raise ValueError."""
    raise NotImplementedError("Lab 1.4: implement normalize_username()")

Run the checks. They cover full-width letters, `ß`, a combining accent and two
invalid inputs.

In [142]:
run_checks(
    [
        ("strip and casefold", lambda: normalize_username("  Asha.Kumar "), "asha.kumar"),
        ("inner spaces become _", lambda: normalize_username("Ravi   Shankar"), "ravi_shankar"),
        ("full-width letters", lambda: normalize_username("ＡＳＨＡ２６"), "asha26"),
        ("ß casefolds to ss", lambda: normalize_username("STRAßE"), "strasse"),
        (
            "combining accent equals precomposed",
            lambda: normalize_username("Jose\u0301") == normalize_username("JOSÉ"),
            True,
        ),
        (
            "blank raises",
            lambda: raised_error_name(lambda: normalize_username("   ")),
            "ValueError",
        ),
        (
            "@ raises",
            lambda: raised_error_name(lambda: normalize_username("asha@home")),
            "ValueError",
        ),
    ]
)

FAIL  strip and casefold: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  inner spaces become _: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  full-width letters: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  ß casefolds to ss: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  combining accent equals precomposed: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  blank raises: expected 'ValueError', got 'NotImplementedError'
FAIL  @ raises: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 2 — `format_receipt_line`

Write `format_receipt_line(item_name: str, quantity: int, unit_price: Decimal) -> str`
that returns one fixed-width receipt line of exactly **49 characters**:

| Column | Width | Rule |
| --- | --- | --- |
| item name | 20 | left-aligned; a longer name is cut to 19 characters plus `…` |
| quantity | 4 | right-aligned |
| separator | 3 | the text `" x "` |
| unit price | 10 | right-aligned, `,` separators, 2 decimal places |
| line total | 12 | `quantity * unit_price`, right-aligned, `,` separators, 2 decimal places |

```text
Masala Dosa            2 x     120.00      240.00
```

You may assume the quantity, price and total fit their columns.

In [143]:
def format_receipt_line(item_name: str, quantity: int, unit_price: Decimal) -> str:
    """Return a 49-character receipt line: name, quantity, unit price and line total."""
    raise NotImplementedError("Lab 1.4: implement format_receipt_line()")

The checks compare exact strings, so every space counts.

In [144]:
run_checks(
    [
        (
            "basic line",
            lambda: format_receipt_line("Masala Dosa", 2, Decimal("120")),
            "Masala Dosa            2 x     120.00      240.00",
        ),
        (
            "long name is truncated with …",
            lambda: format_receipt_line("Paneer Butter Masala Special", 1, Decimal("320.5")),
            "Paneer Butter Masal…   1 x     320.50      320.50",
        ),
        (
            "thousands separators",
            lambda: format_receipt_line("Wedding Catering", 3, Decimal("125000")),
            "Wedding Catering       3 x 125,000.00  375,000.00",
        ),
        (
            "always 49 characters",
            lambda: {
                len(format_receipt_line(name, count, Decimal(price)))
                for name, count, price in [("Tea", 1, "10"), ("A" * 40, 99, "9999.99")]
            },
            {49},
        ),
    ]
)

FAIL  basic line: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
FAIL  long name is truncated with …: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
FAIL  thousands separators: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
FAIL  always 49 characters: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
0/4 checks passed


#### Exercise 3 — search-query normalization

Write `normalize_search_query(raw_query: str) -> str` so that differently typed
queries match the same search index entry:

1. Normalize with **NFKD**, then drop every combining mark (`unicodedata.combining`
   returns non-zero for them), so `é` becomes `e`.
2. Casefold.
3. Turn every character that is not alphanumeric into a space.
4. Collapse whitespace runs into single spaces and strip the edges.

`normalize_search_query("  Café   CRÈME! ") == "cafe creme"`.

> **Note:** dropping marks is right for accented Latin text but wrong for scripts such
> as Devanagari, where marks are part of the letters. Real search engines use
> per-language analyzers.

In [145]:
def normalize_search_query(raw_query: str) -> str:
    """Return a lowercase, accent-free, single-spaced version of a search query."""
    raise NotImplementedError("Lab 1.4: implement normalize_search_query()")

Run the checks.

In [146]:
run_checks(
    [
        ("accents and case", lambda: normalize_search_query("  Café   CRÈME! "), "cafe creme"),
        (
            "punctuation becomes spaces",
            lambda: normalize_search_query("Masala-Dosa,  Bengaluru"),
            "masala dosa bengaluru",
        ),
        ("full-width text", lambda: normalize_search_query("ＤＯＳＡ ２０２６"), "dosa 2026"),
        (
            "ß and SS match",
            lambda: normalize_search_query("Straße") == normalize_search_query("STRASSE"),
            True,
        ),
        ("empty query", lambda: normalize_search_query(""), ""),
        ("only punctuation", lambda: normalize_search_query("!!! ???"), ""),
    ]
)

FAIL  accents and case: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  punctuation becomes spaces: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  full-width text: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  ß and SS match: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  empty query: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  only punctuation: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
0/6 checks passed


### Recap — Unit 1.4

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `str` | Immutable sequence of code points; every method returns a new string | `string` |
| `len` | Counts code points, not UTF-16 units or visible characters | `.length` (UTF-16 units) |
| PEP 393 | CPython stores 1, 2 or 4 bytes per character, chosen per string | — |
| Slicing | `[start:stop:step]`, `[::-1]`; slices never raise | `slice()`, `at()` |
| `strip(chars)` | Removes a *set* of characters; use `removeprefix`/`removesuffix` for exact text | `trim()` |
| `split` / `partition` | `split()` for whitespace, `split(sep)` keeps empty fields, `partition` always gives 3 parts | `split()` |
| `isdecimal` | The numeric test that matches what `int()` accepts | `/^\d+$/` |
| f-strings | Expressions, `=` debugging, `!r`, nested specs, multi-line (3.12+) | template literals |
| Format spec | `fill align sign # width , .precision type`, plus `strftime` codes for dates | `toLocaleString`, `toFixed` |
| t-strings (3.14) | `Template` objects that a renderer escapes safely | tagged templates |
| Normalization | NFC before comparing; NFKC/NFKD also fold look-alike characters | `normalize("NFC")` |
| `casefold` | Use for caseless comparison (`ß` → `ss`) | `toLowerCase()` (not enough) |
| `bytes` / `bytearray` | Binary data; indexing gives `int`; encode/decode at the boundary | `Uint8Array`, `Buffer` |
| `errors=` | `strict`, `replace`, `ignore`, `backslashreplace` | `TextDecoder({fatal})` |
| UTF-8 mode (3.15) | `open()` defaults to UTF-8 everywhere (PEP 686) | — |
| `memoryview` | Zero-copy slices of a buffer | `subarray()` |

**Further reading**

- [Text sequence type — `str`](https://docs.python.org/3/library/stdtypes.html#text-sequence-type-str),
  [string methods](https://docs.python.org/3/library/stdtypes.html#string-methods)
- [Format specification mini-language](https://docs.python.org/3/library/string.html#format-specification-mini-language),
  [f-strings](https://docs.python.org/3/reference/lexical_analysis.html#f-strings),
  [`string.templatelib`](https://docs.python.org/3/library/string.templatelib.html)
- [Unicode HOWTO](https://docs.python.org/3/howto/unicode.html),
  [`unicodedata`](https://docs.python.org/3/library/unicodedata.html)
- [Binary sequence types — `bytes`, `bytearray`, `memoryview`](https://docs.python.org/3/library/stdtypes.html#binary-sequence-types-bytes-bytearray-memoryview),
  [codec error handlers](https://docs.python.org/3/library/codecs.html#error-handlers)
- [PEP 393 — Flexible string representation](https://peps.python.org/pep-0393/),
  [PEP 498 — Literal string interpolation](https://peps.python.org/pep-0498/),
  [PEP 701 — Syntactic formalization of f-strings](https://peps.python.org/pep-0701/),
  [PEP 750 — Template strings](https://peps.python.org/pep-0750/),
  [PEP 686 — Make UTF-8 mode default](https://peps.python.org/pep-0686/)

## Unit 1.5 — Collection Data Types I: Lists, Tuples and Ranges

**Objectives**

- Use the operations every sequence shares, and know which types support which.
- Explain how a `list` is stored, and the cost of each list operation.
- Use the complete `list` API, slice assignment and starred unpacking.
- Sort by several keys, stably, in mixed directions.
- Choose between `list`, `tuple`, named tuples, `range` and `array.array`.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `arr.push(x)` / `arr.pop()` | `items.append(x)` / `items.pop()` | `append` returns `None`, not the new length |
| `arr.shift()` / `arr.unshift(x)` | `items.pop(0)` / `items.insert(0, x)` | O(n): use `collections.deque` (1.6.7) for queues |
| `[10, 9, 1].sort()` → `[1, 10, 9]` | `sorted([10, 9, 1])` → `[1, 9, 10]` | Python compares values; mixing `int` and `str` raises `TypeError` |
| `arr.sort()` returns the array | `items.sort()` returns `None` | Use `sorted(items)` when you want a new list |
| `arr.slice(1, 3)`, `arr.at(-1)` | `items[1:3]`, `items[-1]` | Slices also take a step, and assigning to a slice replaces it |
| `Object.freeze([lat, lng])` | `(lat, lng)` tuple | A tuple is hashable, so it can be a dict key |
| `const [first, ...rest] = arr` | `first, *rest = items` | Works on any iterable, and the star can sit in the middle |
| `Array.from({length: 5}, (_, i) => i)` | `range(5)` | `range` is lazy: no list is built |

The next cell is this unit's setup.

In [147]:
import array
import csv
import dis
import io
import operator
import sqlite3
import sys
import timeit
from collections import namedtuple
from collections.abc import Callable, Hashable, Iterable
from typing import Any, NamedTuple


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.5.1 Common sequence operations

Every sequence type supports indexing, slicing, `in`, `len`, `min`, `max`, `count` and
`index`; most also support `+` (concatenate) and `*` (repeat). `index(x, start)`
starts searching at `start`.

In [148]:
weekly_orders = [12, 7, 19, 7, 25]
print(f"{weekly_orders[0]=}, {weekly_orders[-1]=}, {weekly_orders[1:4]=}")
print(f"{weekly_orders[::-1]=}, {19 in weekly_orders=}, {len(weekly_orders)=}")
print(f"{weekly_orders + [30]=}, {[0] * 3=}")
print(f"{min(weekly_orders)=}, {max(weekly_orders)=}")
print(f"{weekly_orders.count(7)=}, {weekly_orders.index(7)=}, {weekly_orders.index(7, 2)=}")

weekly_orders[0]=12, weekly_orders[-1]=25, weekly_orders[1:4]=[7, 19, 7]
weekly_orders[::-1]=[25, 7, 19, 7, 12], 19 in weekly_orders=True, len(weekly_orders)=5
weekly_orders + [30]=[12, 7, 19, 7, 25, 30], [0] * 3=[0, 0, 0]
min(weekly_orders)=7, max(weekly_orders)=25
weekly_orders.count(7)=2, weekly_orders.index(7)=1, weekly_orders.index(7, 2)=3


**Support table.** The same operations across five sequence types (✓ supported,
✗ raises `TypeError`):

| Operation | `list` | `tuple` | `str` | `range` | `bytes` |
| --- | --- | --- | --- | --- | --- |
| `s[i]` | ✓ | ✓ | ✓ | ✓ | ✓ (gives `int`) |
| `s[a:b]` | ✓ | ✓ | ✓ | ✓ (gives `range`) | ✓ |
| `x in s` | ✓ | ✓ | ✓ (substring) | ✓ (O(1) for ints) | ✓ |
| `len(s)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s + t` | ✓ | ✓ | ✓ | ✗ | ✓ |
| `s * n` | ✓ | ✓ | ✓ | ✗ | ✓ |
| `min(s)`, `max(s)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s.count(x)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s.index(x)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s[i] = x` | ✓ | ✗ | ✗ | ✗ | ✗ (use `bytearray`) |

The next cell checks every row of the table by running the operation on each type.

In [149]:
def supports(operation: Callable[[Any], object], sequence: Any) -> bool:
    """Return True if operation(sequence) runs without TypeError."""
    try:
        operation(sequence)
    except TypeError:
        return False
    return True


def assign_first(sequence: Any) -> None:
    """Try to replace the first item in place."""
    sequence[0] = sequence[0]


sequence_samples = {
    "list": [3, 1, 2],
    "tuple": (3, 1, 2),
    "str": "312",
    "range": range(3),
    "bytes": b"312",
}
sequence_operations: dict[str, Callable[[Any], object]] = {
    "s[i]": lambda sequence: sequence[0],
    "s[a:b]": lambda sequence: sequence[1:],
    "x in s": lambda sequence: sequence[0] in sequence,
    "len(s)": len,
    "s + t": lambda sequence: sequence + sequence,
    "s * n": lambda sequence: sequence * 2,
    "min, max": lambda sequence: (min(sequence), max(sequence)),
    "s.count(x)": lambda sequence: sequence.count(sequence[0]),
    "s.index(x)": lambda sequence: sequence.index(sequence[0]),
    "s[i] = x": assign_first,
}
print(f"{'operation':<12}" + "".join(f"{type_name:>7}" for type_name in sequence_samples))
for operation_label, operation in sequence_operations.items():
    support_marks = [
        "✓" if supports(operation, sample) else "✗" for sample in sequence_samples.values()
    ]
    print(f"{operation_label:<12}" + "".join(f"{mark:>7}" for mark in support_marks))

operation      list  tuple    str  range  bytes
s[i]              ✓      ✓      ✓      ✓      ✓
s[a:b]            ✓      ✓      ✓      ✓      ✓
x in s            ✓      ✓      ✓      ✓      ✓
len(s)            ✓      ✓      ✓      ✓      ✓
s + t             ✓      ✓      ✓      ✗      ✓
s * n             ✓      ✓      ✓      ✗      ✓
min, max          ✓      ✓      ✓      ✓      ✓
s.count(x)        ✓      ✓      ✓      ✓      ✓
s.index(x)        ✓      ✓      ✓      ✓      ✓
s[i] = x          ✓      ✗      ✗      ✗      ✗


### 1.5.2 Lists (`list`) — internals

A CPython `list` is a **dynamic array of pointers**. The list object holds its length,
its capacity (`allocated`) and a pointer to a separate array of `PyObject*` slots;
each slot points to an item object stored elsewhere. Items are never stored inline,
which is why one list can hold any mix of types.

```mermaid
flowchart LR
    header["list object<br/>ob_size = 3<br/>allocated = 4"] --> slots["pointer array<br/>slot 0 | slot 1 | slot 2 | (spare)"]
    slots --> item_int["int 10"]
    slots --> item_str["str 'dosa'"]
    slots --> item_float["float 2.5"]
```

<details><summary>ASCII fallback</summary>

```text
list object                pointer array (capacity 4)
+----------------+        +--------+--------+--------+--------+
| ob_size = 3    |  --->  | slot 0 | slot 1 | slot 2 | spare  |
| allocated = 4  |        +---|----+---|----+---|----+--------+
+----------------+            v        v        v
                           int 10  str 'dosa' float 2.5
```

</details>

When the array is full, `append` **over-allocates**: it grows by roughly 1/8 plus a
few slots, so most appends just fill a spare slot. That is why `append` is O(1)
*amortized*. `sys.getsizeof` measures the list and its slot array, not the items.

In [150]:
growing_list: list[int] = []
previous_size = sys.getsizeof(growing_list)
print(f"len=0    size={previous_size} bytes")
for item_count in range(1, 41):
    growing_list.append(item_count)
    current_size = sys.getsizeof(growing_list)
    if current_size != previous_size:
        added_slots = (current_size - previous_size) // 8
        print(f"len={item_count:<4} size={current_size} bytes (+{added_slots} slots)")
        previous_size = current_size
print(f"{sys.getsizeof([1]) == sys.getsizeof([10**1000])=}  (items are not counted)")

len=0    size=56 bytes
len=1    size=88 bytes (+4 slots)
len=5    size=120 bytes (+4 slots)
len=9    size=184 bytes (+8 slots)
len=17   size=248 bytes (+8 slots)
len=25   size=312 bytes (+8 slots)
len=33   size=376 bytes (+8 slots)
sys.getsizeof([1]) == sys.getsizeof([10**1000])=True  (items are not counted)


**Cost of list operations** (*n* = length of the list, *k* = length of the other
iterable or slice):

| Operation | Cost | Why |
| --- | --- | --- |
| `items[i]`, `items[i] = x` | O(1) | pointer arithmetic |
| `append(x)`, `pop()` | O(1) amortized | work at the end only |
| `pop(0)`, `insert(0, x)` | O(n) | every pointer shifts by one slot |
| `x in items`, `index(x)`, `remove(x)` | O(n) | linear scan comparing with `==` |
| `extend(other)`, `items[a:b]` | O(k) | copy *k* pointers |
| `sort()`, `sorted()` | O(n log n) | Timsort (O(n) when already sorted) |

The timings make the table concrete. Absolute numbers depend on the machine; the gaps
between rows are what matter.

In [151]:
timing_cases = [
    ("items[50_000]", "O(1)"),
    ("items.append(0); items.pop()", "O(1)"),
    ("items.insert(0, 0); items.pop(0)", "O(n)"),
    ("99_999 in items", "O(n)"),
]
run_count = 1_000
for statement, complexity in timing_cases:
    elapsed_seconds = timeit.timeit(
        statement, setup="items = list(range(100_000))", number=run_count
    )
    nanoseconds_per_run = elapsed_seconds / run_count * 1e9
    print(f"{statement:<34} {complexity:<5} {nanoseconds_per_run:>12,.0f} ns per run")

items[50_000]                      O(1)            11 ns per run
items.append(0); items.pop()       O(1)            24 ns per run


items.insert(0, 0); items.pop(0)   O(n)        28,440 ns per run

99_999 in items                    O(n)       568,221 ns per run


### 1.5.3 List methods (complete API)

**Adding.** `append` adds one object (even a list, which becomes a nested item);
`extend` adds every item of any iterable; `insert(i, x)` puts `x` before index `i`.
`items += iterable` is `extend`: it changes the list **in place**. `items + other`
builds a **new** list and requires `other` to be a list. Mutating methods return
`None`.

In [152]:
cart_items = ["idli"]
cart_items.append("dosa")
cart_items.extend(["vada", "chutney"])
cart_items.insert(1, "coffee")
print(f"{cart_items=}")
print(f"{cart_items.append('tea')=}  (mutating methods return None)")
nested_cart = ["idli"]
nested_cart.append(["vada", "chutney"])
print(f"{nested_cart=}  (append adds the list as one item)")
original_cart = cart_items
cart_items += ("sambar",)
print(f"{cart_items is original_cart=}  (+= extends in place, accepts any iterable)")
cart_items = cart_items + ["payasam"]
print(f"{cart_items is original_cart=}  (+ builds a new list)")
print(f"list + tuple: {raised_error_name(lambda: ['idli'] + ('vada',))}")

cart_items=['idli', 'coffee', 'dosa', 'vada', 'chutney']
cart_items.append('tea')=None  (mutating methods return None)
nested_cart=['idli', ['vada', 'chutney']]  (append adds the list as one item)
cart_items is original_cart=True  (+= extends in place, accepts any iterable)
cart_items is original_cart=False  (+ builds a new list)
list + tuple: TypeError


**Removing.** `remove(x)` deletes the **first** item equal to `x` and raises
`ValueError` if there is none. `pop(i)` removes and returns the item at `i` (default:
the last one). `clear()` empties the list. `del items[i]` and `del items[a:b]` delete
by position without returning anything.

In [153]:
order_queue = ["idli", "dosa", "vada", "dosa", "upma", "poha", "khichdi"]
order_queue.remove("dosa")
print(f"{order_queue=}  (only the first 'dosa' is gone)")
try:
    order_queue.remove("pizza")
except ValueError as error:
    print(f"remove: {type(error).__name__}: {error}")
last_order = order_queue.pop()
second_order = order_queue.pop(1)
print(f"{last_order=}, {second_order=}, {order_queue=}")
del order_queue[0]
print(f"after del [0]: {order_queue}")
del order_queue[1:3]
print(f"after del [1:3]: {order_queue}")
order_queue.clear()
print(f"after clear(): {order_queue}, pop on empty: {raised_error_name(order_queue.pop)}")

order_queue=['idli', 'vada', 'dosa', 'upma', 'poha', 'khichdi']  (only the first 'dosa' is gone)
remove: ValueError: list.remove(x): x not in list
last_order='khichdi', second_order='vada', order_queue=['idli', 'dosa', 'upma', 'poha']
after del [0]: ['dosa', 'upma', 'poha']
after del [1:3]: ['dosa']
after clear(): [], pop on empty: IndexError


**Searching.** `index(x, start, end)` returns the first position of `x` within
`items[start:end]` (positions are still counted from the start of the list). `count`
counts equal items. Both compare with `==`, as does `in`.

In [154]:
daily_statuses = ["ok", "late", "ok", "failed", "late", "ok"]
print(f"{daily_statuses.index('late')=}")
print(f"{daily_statuses.index('late', 2)=}  (search from position 2)")
print(f"{daily_statuses.index('ok', 3, 6)=}")
print(f"index outside the window: {raised_error_name(lambda: daily_statuses.index('failed', 4))}")
print(f"{daily_statuses.count('ok')=}, {'failed' in daily_statuses=}")
print(f"{[1, 1.0, True].count(1)=}  (1 == 1.0 == True)")

daily_statuses.index('late')=1
daily_statuses.index('late', 2)=4  (search from position 2)
daily_statuses.index('ok', 3, 6)=5
index outside the window: ValueError
daily_statuses.count('ok')=3, 'failed' in daily_statuses=True
[1, 1.0, True].count(1)=3  (1 == 1.0 == True)


**Ordering.** `items.sort()` sorts **in place** and returns `None`; `sorted(iterable)`
returns a new list and works on any iterable. Both take `key=` and `reverse=`.
Likewise `items.reverse()` reverses in place, while `reversed(items)` returns a lazy
iterator.

In [155]:
dish_names = ["vada", "Idli", "dosa", "Upma"]
sorted_copy = sorted(dish_names)
print(f"{sorted_copy=}, {dish_names=}  (original untouched)")
print(f"{dish_names.sort(key=str.lower)=}  (sort() returns None)")
print(f"{dish_names=}  (sorted in place, case-insensitive)")
print(f"{sorted(dish_names, key=len, reverse=True)=}")
dish_names.reverse()
print(f"{dish_names=}  (reversed in place)")
backwards_iterator = reversed(dish_names)
print(f"{type(backwards_iterator).__name__=}, {list(backwards_iterator)=}")
print(f"{sorted('dosa')=}, {sorted({3, 1, 2})=}  (any iterable)")

sorted_copy=['Idli', 'Upma', 'dosa', 'vada'], dish_names=['vada', 'Idli', 'dosa', 'Upma']  (original untouched)
dish_names.sort(key=str.lower)=None  (sort() returns None)
dish_names=['dosa', 'Idli', 'Upma', 'vada']  (sorted in place, case-insensitive)
sorted(dish_names, key=len, reverse=True)=['dosa', 'Idli', 'Upma', 'vada']
dish_names=['vada', 'Upma', 'Idli', 'dosa']  (reversed in place)
type(backwards_iterator).__name__='list_reverseiterator', list(backwards_iterator)=['dosa', 'Idli', 'Upma', 'vada']
sorted('dosa')=['a', 'd', 'o', 's'], sorted({3, 1, 2})=[1, 2, 3]  (any iterable)


**Copying.** `items.copy()`, `items[:]` and `list(items)` all make a **shallow** copy:
a new list holding the *same* item objects. Changing the new list does not affect the
old one, but mutating a shared inner object shows up in both. Unit 1.7 covers deep
copies.

In [156]:
menu_rows = [["idli", 40], ["dosa", 60]]
copy_by_method = menu_rows.copy()
copy_by_slice = menu_rows[:]
copy_by_constructor = list(menu_rows)
print(f"{copy_by_method is menu_rows=}, {copy_by_slice == menu_rows=}")
copy_by_method.append(["vada", 30])
print(f"{len(menu_rows)=}, {len(copy_by_method)=}  (outer lists are independent)")
copy_by_constructor[0][1] = 45
print(f"{menu_rows[0]=}  (inner lists are shared: the original changed too)")
print(f"{copy_by_slice[0] is menu_rows[0]=}")

copy_by_method is menu_rows=False, copy_by_slice == menu_rows=True
len(menu_rows)=2, len(copy_by_method)=3  (outer lists are independent)
menu_rows[0]=['idli', 45]  (inner lists are shared: the original changed too)
copy_by_slice[0] is menu_rows[0]=True


**Slice assignment** replaces a slice with the items of any iterable, and the lengths
may differ, so the list can grow or shrink. An empty slice `items[i:i]` inserts.
**Extended slices** (with a step) need exactly as many items as they select.
**Starred unpacking** collects "the rest" into a list.

In [157]:
seat_row = ["A1", "A2", "A3", "A4", "A5"]
seat_row[1:3] = ["B2"]
print(f"{seat_row=}  (two items replaced by one)")
seat_row[1:1] = ["NEW1", "NEW2"]
print(f"{seat_row=}  (empty slice: pure insert)")
seat_row[::2] = ["x", "y", "z"]
print(f"{seat_row=}  (extended slice, same length)")
try:
    seat_row[::2] = ["only one"]
except ValueError as error:
    print(f"{type(error).__name__}: {error}")
first_seat, *other_seats = seat_row
*_, last_seat = seat_row
front_seat, *middle_seats, back_seat = seat_row
print(f"{first_seat=}, {other_seats=}, {last_seat=}")
print(f"{front_seat=}, {middle_seats=}, {back_seat=}")

seat_row=['A1', 'B2', 'A4', 'A5']  (two items replaced by one)
seat_row=['A1', 'NEW1', 'NEW2', 'B2', 'A4', 'A5']  (empty slice: pure insert)
seat_row=['x', 'NEW1', 'y', 'B2', 'z', 'A5']  (extended slice, same length)
ValueError: attempt to assign sequence of size 1 to extended slice of size 3
first_seat='x', other_seats=['NEW1', 'y', 'B2', 'z', 'A5'], last_seat='A5'
front_seat='x', middle_seats=['NEW1', 'y', 'B2', 'z'], back_seat='A5'


### 1.5.4 Sorting in depth

Python sorts with **Timsort**, a merge sort that finds existing ordered runs, so
nearly sorted data costs close to O(n). (Since 3.11 CPython merges runs with the
*powersort* strategy.) The sort is **stable**: items with equal keys keep their
original relative order.

Stability enables **multi-pass sorting**: sort by the least important key first, then
by more important keys. Each pass keeps the order the earlier passes produced among
ties.

In [158]:
customer_orders = [("Ravi", "Pune"), ("Asha", "Delhi"), ("Meena", "Pune"), ("John", "Delhi")]
print(f"{sorted(customer_orders, key=lambda order: order[1])=}")
print("  ^ within each city the original order (Ravi before Meena) is kept")
by_name = sorted(customer_orders, key=lambda order: order[0])
by_city_then_name = sorted(by_name, key=lambda order: order[1])
print(f"{by_city_then_name=}")

sorted(customer_orders, key=lambda order: order[1])=[('Asha', 'Delhi'), ('John', 'Delhi'), ('Ravi', 'Pune'), ('Meena', 'Pune')]
  ^ within each city the original order (Ravi before Meena) is kept
by_city_then_name=[('Asha', 'Delhi'), ('John', 'Delhi'), ('Meena', 'Pune'), ('Ravi', 'Pune')]


**Key functions.** `key` is called once per item, and items are compared by the key's
result:

- `key=str.lower` for case-insensitive order.
- `operator.itemgetter(1)` reads `item[1]`; `itemgetter(2, 0)` returns a tuple of both.
- `operator.attrgetter("price")` reads `item.price` (works on any object with
  attributes).
- **Tuple keys** compare element by element. For mixed directions, negate numeric
  parts: `key=lambda p: (-p.rating, p.price)` means rating descending, price ascending.
  Strings cannot be negated, so use multi-pass sorting with `reverse=True` for them.

The products below use `NamedTuple`, covered in 1.5.6; here it is just a record with
attributes.

In [159]:
class Product(NamedTuple):
    """A product in a catalogue."""

    name: str
    price: int
    rating: float


catalogue = [
    Product("pen", 20, 4.2),
    Product("Notebook", 60, 4.8),
    Product("eraser", 5, 4.2),
    Product("Bag", 900, 4.8),
]
print(f"{sorted(['pen', 'Notebook', 'eraser'])=}")
print(f"{sorted(['pen', 'Notebook', 'eraser'], key=str.lower)=}")
print([product.name for product in sorted(catalogue, key=operator.itemgetter(1))])
print(sorted(catalogue, key=operator.itemgetter(2, 0))[:2])
print([product.name for product in sorted(catalogue, key=operator.attrgetter("rating"))])
best_then_cheapest = sorted(catalogue, key=lambda product: (-product.rating, product.price))
print([(product.name, product.rating, product.price) for product in best_then_cheapest])

sorted(['pen', 'Notebook', 'eraser'])=['Notebook', 'eraser', 'pen']
sorted(['pen', 'Notebook', 'eraser'], key=str.lower)=['eraser', 'Notebook', 'pen']
['eraser', 'pen', 'Notebook', 'Bag']
[Product(name='eraser', price=5, rating=4.2), Product(name='pen', price=20, rating=4.2)]
['pen', 'eraser', 'Notebook', 'Bag']
[('Notebook', 4.8, 60), ('Bag', 4.8, 900), ('eraser', 4.2, 5), ('pen', 4.2, 20)]


**JS bridge.** JavaScript's default `sort()` converts items to strings, so
`[10, 9, 1, 100].sort()` gives `[1, 10, 100, 9]`. Python compares the values
themselves, and refuses to compare unrelated types instead of guessing.

In [160]:
print(f"{sorted([10, 9, 1, 100])=}")
print(f"{sorted(['10', '9', '1', '100'])=}  (what JS does to numbers)")
try:
    sorted([3, "2", 1])
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{sorted([3, '2', 1], key=str)=}  (an explicit key makes it work)")

sorted([10, 9, 1, 100])=[1, 9, 10, 100]
sorted(['10', '9', '1', '100'])=['1', '10', '100', '9']  (what JS does to numbers)
TypeError: '<' not supported between instances of 'str' and 'int'
sorted([3, '2', 1], key=str)=[1, '2', 3]  (an explicit key makes it work)


### 1.5.5 Tuples (`tuple`)

**The comma makes the tuple**, not the parentheses: `(42)` is just the int 42, while
`42,` and `(42,)` are one-item tuples. `()` is the empty tuple. Writing several values
separated by commas **packs** them; assigning a tuple to several names **unpacks** it.
Unpacking can be nested and starred, and `a, b = b, a` swaps without a temporary.

> **Note:** the two `# fmt: skip` comments stop the formatter from rewriting `(42)`
> as `42` and `42,` as `(42,)`, which would hide the point.

In [161]:
not_a_tuple = (42)  # fmt: skip
one_item_tuple = 42,  # fmt: skip
also_one_item = (42,)
empty_tuple = ()
print(f"{type(not_a_tuple).__name__=}, {one_item_tuple=}, {also_one_item=}, {empty_tuple=}")
office_location = 12.97, 77.59
latitude, longitude = office_location
print(f"{office_location=}, {latitude=}, {longitude=}")
city_name, (city_latitude, city_longitude) = ("Bengaluru", (12.97, 77.59))
print(f"{city_name=}, {city_latitude=}, {city_longitude=}")
top_scorer, *other_scorers = ("Asha", "Ravi", "Meena")
print(f"{top_scorer=}, {other_scorers=}  (the starred name is always a list)")
home_team, away_team = "India", "Australia"
home_team, away_team = away_team, home_team
print(f"{home_team=}, {away_team=}")

type(not_a_tuple).__name__='int', one_item_tuple=(42,), also_one_item=(42,), empty_tuple=()
office_location=(12.97, 77.59), latitude=12.97, longitude=77.59
city_name='Bengaluru', city_latitude=12.97, city_longitude=77.59
top_scorer='Asha', other_scorers=['Ravi', 'Meena']  (the starred name is always a list)
home_team='Australia', away_team='India'


**Methods and hashing.** Tuples have just two methods, `count` and `index`. A tuple is
**hashable only if all its items are**, so a tuple of strings and numbers can be a
dict key or a set member, but a tuple holding a list cannot.

In [162]:
match_ratings = (5, 4, 5, 3, 5)
print(f"{match_ratings.count(5)=}, {match_ratings.index(3)=}")
print(f"{hash((1, 'a', (2, 3))) == hash((1, 'a', (2, 3)))=}")
try:
    hash((1, [2, 3]))
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
sales_by_country_and_product = {("IN", "dosa"): 1200, ("US", "dosa"): 150}
print(f"{sales_by_country_and_product[('IN', 'dosa')]=}")
print(f"{isinstance((1, 'a'), Hashable)=}, {isinstance((1, []), Hashable)=}")
print("  ^ the Hashable ABC checks the type only; hash() checks the items")

match_ratings.count(5)=3, match_ratings.index(3)=3
hash((1, 'a', (2, 3))) == hash((1, 'a', (2, 3)))=True
TypeError: unhashable type: 'list'
sales_by_country_and_product[('IN', 'dosa')]=1200
isinstance((1, 'a'), Hashable)=True, isinstance((1, []), Hashable)=True
  ^ the Hashable ABC checks the type only; hash() checks the items


#### Under the hood: tuples are smaller and built once

A tuple stores its item pointers inline, with no spare capacity and no separate array,
so it is smaller than the equivalent list. A tuple of constants is built once at
compile time: `dis` shows a single `LOAD_CONST`, while the list is rebuilt on every
run. (Bytecode names vary by Python version.)

In [163]:
show_size("tuple (1, 2, 3)", (1, 2, 3))
show_size("list  [1, 2, 3]", [1, 2, 3])
appended_list = []
for number in (1, 2, 3):
    appended_list.append(number)
show_size("list built by append", appended_list)
dis.dis(compile("rgb_tuple = (255, 153, 51)", "<tuple>", "exec"))
dis.dis(compile("rgb_list = [255, 153, 51]", "<list>", "exec"))

tuple (1, 2, 3): 72 bytes, type=tuple, id=140266921912544
list  [1, 2, 3]: 88 bytes, type=list, id=140266922037440
list built by append: 88 bytes, type=list, id=140266922042368
  0           RESUME                   0

  1           LOAD_CONST               1 ((255, 153, 51))
              STORE_NAME               0 (rgb_tuple)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE
  0           RESUME                   0

  1           BUILD_LIST               0
              LOAD_CONST               1 ((255, 153, 51))
              LIST_EXTEND              1
              STORE_NAME               0 (rgb_list)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


#### Predict the output: the `t[0] += [1]` puzzle

**Predict the output.**

```python
pair_of_lists = ([1, 2], "x")
try:
    pair_of_lists[0] += [3]
except TypeError as error:
    print(type(error).__name__)
print(pair_of_lists)
```

<details><summary>Answer</summary>

`TypeError`, then `([1, 2, 3], 'x')`. The statement **both** raises and mutates.
`pair_of_lists[0] += [3]` first runs `list.__iadd__`, which extends the inner list in
place and succeeds; then it tries to store the result back into slot 0 of the tuple,
which fails because tuples do not support item assignment.

</details>

Run the next cell to check your prediction.

In [164]:
pair_of_lists = ([1, 2], "x")
try:
    pair_of_lists[0] += [3]
except TypeError as error:
    print(type(error).__name__)
print(pair_of_lists)

TypeError
([1, 2, 3], 'x')


The bytecode shows the two steps: an in-place add (`BINARY_OP` with `+=`) followed by
`STORE_SUBSCR`, and the error comes from the store.

In [165]:
dis.dis(compile("pair_of_lists[0] += [3]", "<puzzle>", "exec"))

  0           RESUME                   0

  1           LOAD_NAME                0 (pair_of_lists)
              LOAD_SMALL_INT           0
              COPY                     2
              COPY                     2
              BINARY_OP               26 ([])
              LOAD_SMALL_INT           3
              BUILD_LIST               1
              BINARY_OP               13 (+=)
              SWAP                     3
              SWAP                     2
              STORE_SUBSCR
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


### 1.5.6 Named tuples

A **named tuple** is a tuple subclass whose items also have names. It is still
immutable, still unpacks, and costs no more memory than a plain tuple.
`collections.namedtuple` builds one from a field list:

| API | What it does |
| --- | --- |
| `_fields` | tuple of field names |
| `_make(iterable)` | build an instance from any iterable (a CSV row, a DB row) |
| `_asdict()` | a `dict` of field name → value |
| `_replace(**changes)` | a modified **copy** (the original is unchanged) |

The leading underscore avoids clashes with field names; these methods are public.

In [166]:
GeoPoint = namedtuple("GeoPoint", ["latitude", "longitude"])
office_point = GeoPoint(12.97, 77.59)
print(f"{office_point=}, {office_point.latitude=}, {office_point[1]=}")
point_latitude, point_longitude = office_point
print(f"{GeoPoint._fields=}")
print(f"{GeoPoint._make(['28.61', '77.21'])=}  (values are not converted)")
print(f"{office_point._asdict()=}")
moved_point = office_point._replace(longitude=77.60)
print(f"{moved_point=}, {office_point=}  (original unchanged)")
try:
    office_point.latitude = 0.0
except AttributeError as error:
    print(f"{type(error).__name__}: {error}")

office_point=GeoPoint(latitude=12.97, longitude=77.59), office_point.latitude=12.97, office_point[1]=77.59
GeoPoint._fields=('latitude', 'longitude')
GeoPoint._make(['28.61', '77.21'])=GeoPoint(latitude='28.61', longitude='77.21')  (values are not converted)
office_point._asdict()={'latitude': 12.97, 'longitude': 77.59}
moved_point=GeoPoint(latitude=12.97, longitude=77.6), office_point=GeoPoint(latitude=12.97, longitude=77.59)  (original unchanged)
AttributeError: can't set attribute


**`typing.NamedTuple`** is the class syntax for the same thing: type-annotated fields,
**defaults** (fields with defaults must come last), a docstring and **methods**.
Prefer it in new code. `LogEntry` comes back in 1.5.11 and in the Phase 1 project.

In [167]:
class LogEntry(NamedTuple):
    """One parsed line of an access log."""

    level: str
    message: str
    status_code: int = 200

    def is_error(self) -> bool:
        """Return True for ERROR entries and 5xx status codes."""
        return self.level == "ERROR" or self.status_code >= 500


placed_entry = LogEntry("INFO", "order placed")
failed_entry = LogEntry("WARNING", "gateway timeout", 504)
print(f"{placed_entry=}, {placed_entry.is_error()=}")
print(f"{failed_entry.is_error()=}, {LogEntry._field_defaults=}")
print(f"{isinstance(placed_entry, tuple)=}, {placed_entry == ('INFO', 'order placed', 200)=}")

placed_entry=LogEntry(level='INFO', message='order placed', status_code=200), placed_entry.is_error()=False
failed_entry.is_error()=True, LogEntry._field_defaults={'status_code': 200}
isinstance(placed_entry, tuple)=True, placed_entry == ('INFO', 'order placed', 200)=True


### 1.5.7 Ranges (`range`)

A `range` is a **lazy, immutable sequence** of integers. It stores only `start`,
`stop` and `step`, so its size never depends on its length, and `len`, indexing,
slicing and `in` (for ints) are computed arithmetically in **O(1)**. Negative steps
count down; `stop` is always excluded.

In [168]:
page_offsets = range(0, 95, 20)
print(f"{page_offsets=}, {list(page_offsets)=}")
print(f"{list(range(10, 0, -3))=}, {list(range(5, 0, -1))=}")
huge_range = range(0, 10**18, 7)
print(f"{len(huge_range)=:,}")
print(f"{7 * 10**15 in huge_range=}, {7 * 10**15 + 1 in huge_range=}")
print(f"{huge_range[10**12]=:,}, {huge_range[-1]=:,}")
print(f"{huge_range[::2]=}  (slicing returns another range)")
print(f"{sys.getsizeof(huge_range)=} bytes, {sys.getsizeof(range(3))=} bytes")
try:
    page_offsets[0] = 5
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

page_offsets=range(0, 95, 20), list(page_offsets)=[0, 20, 40, 60, 80]
list(range(10, 0, -3))=[10, 7, 4, 1], list(range(5, 0, -1))=[5, 4, 3, 2, 1]
len(huge_range)=142,857,142,857,142,858
7 * 10**15 in huge_range=True, 7 * 10**15 + 1 in huge_range=False
huge_range[10**12]=7,000,000,000,000, huge_range[-1]=999,999,999,999,999,999
huge_range[::2]=range(0, 1000000000000000006, 14)  (slicing returns another range)
sys.getsizeof(huge_range)=48 bytes, sys.getsizeof(range(3))=48 bytes
TypeError: 'range' object does not support item assignment


### 1.5.8 Arrays (`array.array`)

A list of floats stores a pointer per item **plus** a separate 24-byte `float` object
per item. `array.array` stores raw machine values of one type (type code `"d"` =
C double, 8 bytes) in one block. Use it for long numeric series when memory matters
(NumPy, covered later, does the same and adds fast math).

In [169]:
reading_count = 100_000
readings_list = [reading_index * 0.5 for reading_index in range(reading_count)]
readings_array = array.array("d", readings_list)
list_total_bytes = sys.getsizeof(readings_list) + sum(
    sys.getsizeof(reading) for reading in readings_list
)
array_total_bytes = sys.getsizeof(readings_array)
print(f"list of floats: {list_total_bytes:>10,} bytes (list + float objects)")
print(f"array('d'):     {array_total_bytes:>10,} bytes")
print(f"ratio: {list_total_bytes / array_total_bytes:.1f}x")
print(f"{readings_array.typecode=}, {readings_array.itemsize=}, {readings_array[:3]=}")
print(f"append a str: {raised_error_name(lambda: readings_array.append('x'))}")

list of floats:  3,200,984 bytes (list + float objects)
array('d'):        800,080 bytes
ratio: 4.0x
readings_array.typecode='d', readings_array.itemsize=8, readings_array[:3]=array('d', [0.0, 0.5, 1.0])
append a str: TypeError


### 1.5.9 Method reference — `list`

Every row gets a demo cell below.

| Method / operation | What it does | Returns | Cost |
| --- | --- | --- | --- |
| `append(x)` | add one item at the end | `None` | O(1) amortized |
| `extend(iterable)` / `+= iterable` | add every item of an iterable | `None` / same list | O(k) |
| `insert(i, x)` | insert before index `i` | `None` | O(n) |
| `remove(x)` | delete the first item equal to `x` (`ValueError` if absent) | `None` | O(n) |
| `pop()` / `pop(i)` | remove and return the last / i-th item (`IndexError` if empty) | item | O(1) / O(n) |
| `clear()` | remove every item | `None` | O(n) |
| `index(x[, start[, end]])` | position of the first `x` (`ValueError` if absent) | `int` | O(n) |
| `count(x)` | number of items equal to `x` | `int` | O(n) |
| `sort(*, key=None, reverse=False)` | stable in-place sort | `None` | O(n log n) |
| `reverse()` | reverse in place | `None` | O(n) |
| `copy()` | shallow copy (same as `[:]`, `list(x)`) | new `list` | O(n) |
| `items[i]`, `items[i] = x` | read / replace by index (negative indexes allowed) | item / — | O(1) |
| `items[a:b:c]`, `items[a:b] = iterable` | slice copy / slice replacement | new `list` / — | O(k) |
| `del items[i]`, `del items[a:b]` | delete by index or slice | — | O(n) |
| `x in items` | membership by `==` | `bool` | O(n) |
| `items + other`, `items * n` | concatenate / repeat into a new list | new `list` | O(n + m) |
| `len`, `min`, `max`, `sum`, `sorted`, `reversed`, `enumerate`, `any`, `all` | built-ins that work on lists | varies | O(n) |
| `==`, `<` | element-wise, lexicographic comparison | `bool` | O(n) |

#### `list`

**`append(x)`**: one item at the end; the call returns `None`.

In [170]:
shopping_list = ["rice"]
append_result = shopping_list.append("dal")
print(f"{shopping_list=}, {append_result=}")

shopping_list=['rice', 'dal'], append_result=None


**`extend(iterable)` / `+= iterable`**: every item of any iterable, in place.

In [171]:
shopping_list = ["rice"]
shopping_list.extend(("dal", "ghee"))
list_before_augmented_add = shopping_list
shopping_list += "ab"
print(f"{shopping_list=}  (a str is an iterable of characters)")
print(f"{shopping_list is list_before_augmented_add=}")

shopping_list=['rice', 'dal', 'ghee', 'a', 'b']  (a str is an iterable of characters)
shopping_list is list_before_augmented_add=True


**`insert(i, x)`**: before index `i`; an index past the end appends.

In [172]:
shopping_list = ["rice", "ghee"]
shopping_list.insert(1, "dal")
shopping_list.insert(0, "salt")
shopping_list.insert(100, "sugar")
shopping_list.insert(-1, "tea")
print(f"{shopping_list=}")

shopping_list=['salt', 'rice', 'dal', 'ghee', 'tea', 'sugar']


**`remove(x)`**: the first equal item; `ValueError` if absent.

In [173]:
shopping_list = ["rice", "dal", "rice"]
shopping_list.remove("rice")
print(f"{shopping_list=}")
print(f"remove missing: {raised_error_name(lambda: shopping_list.remove('oil'))}")

shopping_list=['dal', 'rice']
remove missing: ValueError


**`pop()` / `pop(i)`**: remove and return the last or the i-th item.

In [174]:
shopping_list = ["rice", "dal", "ghee", "salt"]
print(f"{shopping_list.pop()=}, {shopping_list.pop(0)=}, {shopping_list=}")
print(f"pop from empty: {raised_error_name(lambda: [].pop())}")
print(f"pop bad index: {raised_error_name(lambda: ['rice'].pop(5))}")

shopping_list.pop()='salt', shopping_list.pop(0)='rice', shopping_list=['dal', 'ghee']
pop from empty: IndexError
pop bad index: IndexError


**`clear()`**: empties the list in place, so every alias sees the change.

In [175]:
shopping_list = ["rice", "dal"]
same_list_alias = shopping_list
shopping_list.clear()
print(f"{shopping_list=}, {same_list_alias=}")

shopping_list=[], same_list_alias=[]


**`index(x[, start[, end]])`**: position of the first match in the window.

In [176]:
shopping_list = ["rice", "dal", "ghee", "dal"]
print(f"{shopping_list.index('dal')=}, {shopping_list.index('dal', 2)=}")
print(f"not in window: {raised_error_name(lambda: shopping_list.index('rice', 1, 3))}")

shopping_list.index('dal')=1, shopping_list.index('dal', 2)=3
not in window: ValueError


**`count(x)`**: how many items are `==` to `x`.

In [177]:
shopping_list = ["rice", "dal", "rice", "Rice"]
print(f"{shopping_list.count('rice')=}, {shopping_list.count('oil')=}")

shopping_list.count('rice')=2, shopping_list.count('oil')=0


**`sort(*, key=None, reverse=False)`**: stable, in place, keyword-only arguments.

In [178]:
shopping_list = ["ghee", "Rice", "dal", "salt"]
shopping_list.sort()
print(f"{shopping_list=}  (uppercase first)")
shopping_list.sort(key=str.lower, reverse=True)
print(f"{shopping_list=}")
print(f"positional key: {raised_error_name(lambda: shopping_list.sort(str.lower))}")

shopping_list=['Rice', 'dal', 'ghee', 'salt']  (uppercase first)
shopping_list=['salt', 'Rice', 'ghee', 'dal']
positional key: TypeError


**`reverse()`**: in place, returns `None`.

In [179]:
shopping_list = ["rice", "dal", "ghee"]
print(f"{shopping_list.reverse()=}, {shopping_list=}")

shopping_list.reverse()=None, shopping_list=['ghee', 'dal', 'rice']


**`copy()`**: a new list holding the same item objects.

In [180]:
shopping_list = ["rice", ["dal", "ghee"]]
copied_list = shopping_list.copy()
print(f"{copied_list == shopping_list=}, {copied_list is shopping_list=}")
print(f"{copied_list[1] is shopping_list[1]=}  (shallow)")

copied_list == shopping_list=True, copied_list is shopping_list=False
copied_list[1] is shopping_list[1]=True  (shallow)


**`items[i]`, `items[i] = x`**: read or replace one slot; negative indexes count from the end.

In [181]:
shopping_list = ["rice", "dal", "ghee"]
print(f"{shopping_list[0]=}, {shopping_list[-1]=}")
shopping_list[-1] = "butter"
print(f"{shopping_list=}")
print(f"assign past the end: {raised_error_name(lambda: shopping_list.__setitem__(5, 'x'))}")

shopping_list[0]='rice', shopping_list[-1]='ghee'
shopping_list=['rice', 'dal', 'butter']
assign past the end: IndexError


**`items[a:b:c]`, `items[a:b] = iterable`**: copy a slice, or replace one.

In [182]:
shopping_list = ["rice", "dal", "ghee", "salt", "tea"]
print(f"{shopping_list[1:4]=}, {shopping_list[::2]=}, {shopping_list[::-1]=}")
shopping_list[1:3] = ("moong dal", "toor dal", "urad dal")
print(f"{shopping_list=}")

shopping_list[1:4]=['dal', 'ghee', 'salt'], shopping_list[::2]=['rice', 'ghee', 'tea'], shopping_list[::-1]=['tea', 'salt', 'ghee', 'dal', 'rice']
shopping_list=['rice', 'moong dal', 'toor dal', 'urad dal', 'salt', 'tea']


**`del items[i]`, `del items[a:b]`**: delete by position or slice (an extended slice works too).

In [183]:
shopping_list = ["rice", "dal", "ghee", "salt", "tea", "jam"]
del shopping_list[0]
del shopping_list[-2:]
print(f"{shopping_list=}")
del shopping_list[::2]
print(f"{shopping_list=}")

shopping_list=['dal', 'ghee', 'salt']
shopping_list=['ghee']


**`x in items`**: a linear scan with `==`. CPython checks identity first, so an object
is always found in a list that holds it, even `nan`, which is not equal to itself.

In [184]:
shopping_list = ["rice", "dal"]
print(f"{'dal' in shopping_list=}, {'oil' not in shopping_list=}")
not_a_number = float("nan")
print(f"{not_a_number == not_a_number=}, {not_a_number in [not_a_number]=}")
print(f"{float('nan') in [float('nan')]=}  (two different nan objects)")

'dal' in shopping_list=True, 'oil' not in shopping_list=True
not_a_number == not_a_number=False, not_a_number in [not_a_number]=True
float('nan') in [float('nan')]=False  (two different nan objects)


**`items + other`, `items * n`**: new lists. Repetition copies **references**, which
matters when the item is mutable (see Lab 1.5, exercise 4).

In [185]:
breakfast_list = ["idli", "dosa"]
print(f"{breakfast_list + ['vada']=}, {breakfast_list * 2=}, {breakfast_list=}")
repeated_rows = [[]] * 3
repeated_rows[0].append("x")
print(f"{repeated_rows=}  (three references to one inner list)")

breakfast_list + ['vada']=['idli', 'dosa', 'vada'], breakfast_list * 2=['idli', 'dosa', 'idli', 'dosa'], breakfast_list=['idli', 'dosa']
repeated_rows=[['x'], ['x'], ['x']]  (three references to one inner list)


**Built-ins that work on lists**: `len`, `min`, `max`, `sum`, `sorted`, `reversed`, `enumerate`, `any`, `all`.

In [186]:
daily_sales = [1200, 950, 0, 1800, 1430]
print(f"{len(daily_sales)=}, {min(daily_sales)=}, {max(daily_sales)=}, {sum(daily_sales)=}")
print(f"{sorted(daily_sales)=}, {list(reversed(daily_sales))=}")
for day_number, sales_amount in enumerate(daily_sales, start=1):
    print(f"day {day_number}: ₹{sales_amount:,}")
print(f"{any(sale == 0 for sale in daily_sales)=}, {all(daily_sales)=}")
print(f"{max([], default=0)=}  (default avoids ValueError on empty input)")

len(daily_sales)=5, min(daily_sales)=0, max(daily_sales)=1800, sum(daily_sales)=5380
sorted(daily_sales)=[0, 950, 1200, 1430, 1800], list(reversed(daily_sales))=[1430, 1800, 0, 950, 1200]
day 1: ₹1,200
day 2: ₹950
day 3: ₹0
day 4: ₹1,800
day 5: ₹1,430
any(sale == 0 for sale in daily_sales)=True, all(daily_sales)=False
max([], default=0)=0  (default avoids ValueError on empty input)


**`==`, `<`**: compare item by item; the first difference decides, and a shorter list
that is a prefix of a longer one is smaller.

In [187]:
print(f"{[1, 2, 3] == [1, 2, 3]=}, {[1, 2] == [1.0, 2.0]=}")
print(f"{[1, 2, 3] < [1, 3]=}  (decided at index 1)")
print(f"{[1, 2] < [1, 2, 0]=}  (prefix is smaller)")
print(f"{['b'] > ['a', 'z']=}")
print(f"mixed types: {raised_error_name(lambda: [1, 'a'] < [1, 2])}")

[1, 2, 3] == [1, 2, 3]=True, [1, 2] == [1.0, 2.0]=True
[1, 2, 3] < [1, 3]=True  (decided at index 1)
[1, 2] < [1, 2, 0]=True  (prefix is smaller)
['b'] > ['a', 'z']=True
mixed types: TypeError


### 1.5.10 Method reference — `tuple`, `namedtuple` and `range`

| Type | Method / operation | What it does |
| --- | --- | --- |
| `tuple` | `count(x)`, `index(x[, start[, end]])` | the only two methods — tuples are immutable |
| `tuple` | indexing, slicing, `in`, `+`, `*`, `len`, comparison | same semantics as lists, but always returns new tuples |
| `tuple` | `hash(t)` | works when every item is hashable — enables dict keys and set members |
| `tuple` | unpacking `a, b = t`, `first, *rest = t` | the most common way to read tuples |
| `namedtuple` | `_make(iterable)`, `_asdict()`, `_replace(**changes)`, `_fields`, `_field_defaults` | construction, conversion, "modified copy", introspection |
| `range` | `start`, `stop`, `step`, `count(x)`, `index(x)` | attributes and O(1) arithmetic lookups |

#### `tuple`

**`count(x)`, `index(x[, start[, end]])`**: the only two methods.

In [188]:
weekday_codes = ("Mon", "Tue", "Wed", "Tue", "Fri")
tuple_method_names = [name for name in dir(tuple) if not name.startswith("_")]
print(f"{tuple_method_names=}")
print(f"{weekday_codes.count('Tue')=}, {weekday_codes.index('Tue', 2)=}")
print(f"missing: {raised_error_name(lambda: weekday_codes.index('Sun'))}")

tuple_method_names=['count', 'index']
weekday_codes.count('Tue')=2, weekday_codes.index('Tue', 2)=3
missing: ValueError


**Indexing, slicing, `in`, `+`, `*`, `len`, comparison**: like lists, but every result is a new tuple.

In [189]:
rgb_colour = (255, 153, 51)
print(f"{rgb_colour[0]=}, {rgb_colour[-2:]=}, {153 in rgb_colour=}, {len(rgb_colour)=}")
print(f"{rgb_colour + (128,)=}, {(0,) * 3=}")
print(f"{(1, 2, 3) < (1, 3)=}, {(1, 2) == (1, 2)=}, {(1, 2) == [1, 2]=}")

rgb_colour[0]=255, rgb_colour[-2:]=(153, 51), 153 in rgb_colour=True, len(rgb_colour)=3
rgb_colour + (128,)=(255, 153, 51, 128), (0,) * 3=(0, 0, 0)
(1, 2, 3) < (1, 3)=True, (1, 2) == (1, 2)=True, (1, 2) == [1, 2]=False


**`hash(t)`**: works only when every item is hashable.

In [190]:
route_distances = {("Delhi", "Agra"): 233, ("Mumbai", "Pune"): 148}
print(f"{route_distances[('Delhi', 'Agra')]=}")
visited_cells = {(0, 0), (0, 1), (0, 0)}
print(f"{visited_cells=}")
print(f"tuple holding a list: {raised_error_name(lambda: hash(('Delhi', ['Agra'])))}")

route_distances[('Delhi', 'Agra')]=233
visited_cells={(0, 1), (0, 0)}
tuple holding a list: TypeError


**Unpacking `a, b = t`, `first, *rest = t`**: the most common way to read a tuple; the counts must match.

In [191]:
delivery_record = ("A-17", "Asha", "Pune", 450)
order_id, customer_name, delivery_city, amount_due = delivery_record
first_field, *remaining_fields = delivery_record
print(f"{order_id=}, {amount_due=}, {first_field=}, {remaining_fields=}")
try:
    order_id, customer_name = delivery_record
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

order_id='A-17', amount_due=450, first_field='A-17', remaining_fields=['Asha', 'Pune', 450]
ValueError: too many values to unpack (expected 2, got 4)


#### `namedtuple`

**`_make`, `_asdict`, `_replace`, `_fields`, `_field_defaults`**: construction,
conversion, modified copy and introspection. `defaults=` applies to the *rightmost*
fields.

In [192]:
OrderLine = namedtuple("OrderLine", ["item", "quantity", "unit_price"], defaults=[1, 0])
csv_line = OrderLine._make(["dosa", 2, 60])
print(f"{csv_line=}")
print(f"{csv_line._asdict()=}")
print(f"{csv_line._replace(quantity=3)=}")
print(f"{OrderLine._fields=}, {OrderLine._field_defaults=}")
print(f"{OrderLine('tea')=}  (defaults filled in)")

csv_line=OrderLine(item='dosa', quantity=2, unit_price=60)
csv_line._asdict()={'item': 'dosa', 'quantity': 2, 'unit_price': 60}
csv_line._replace(quantity=3)=OrderLine(item='dosa', quantity=3, unit_price=60)
OrderLine._fields=('item', 'quantity', 'unit_price'), OrderLine._field_defaults={'quantity': 1, 'unit_price': 0}
OrderLine('tea')=OrderLine(item='tea', quantity=1, unit_price=0)  (defaults filled in)


#### `range`

**`start`, `stop`, `step`, `count(x)`, `index(x)`**: the three attributes, plus
lookups computed arithmetically.

In [193]:
even_hours = range(0, 24, 2)
print(f"{even_hours.start=}, {even_hours.stop=}, {even_hours.step=}")
print(f"{even_hours.count(14)=}, {even_hours.count(15)=}, {even_hours.index(14)=}")
print(f"{range(10**18).index(10**17)=:,}  (instant: no scan)")
print(f"not in range: {raised_error_name(lambda: even_hours.index(15))}")

even_hours.start=0, even_hours.stop=24, even_hours.step=2
even_hours.count(14)=1, even_hours.count(15)=0, even_hours.index(14)=7
range(10**18).index(10**17)=100,000,000,000,000,000  (instant: no scan)
not in range: ValueError


### 1.5.11 Real-world use cases

**`list`**: an ordered, changing collection: shopping-cart items, a page of API
results, rows read from a CSV file, or a stack (`append` + `pop`). Ordered processing
queues of moderate size also fit, as long as you take items from the end.

In [194]:
undo_stack: list[str] = []
for editor_action in ["type 'a'", "type 'b'", "bold"]:
    undo_stack.append(editor_action)
print(f"undo {undo_stack.pop()!r}; remaining stack: {undo_stack}")
api_page = [{"id": 1, "item": "idli"}, {"id": 2, "item": "dosa"}]
print(f"{[result['item'] for result in api_page]=}")
csv_text = "id,item,price\n1,idli,40\n2,dosa,60\n"
header_row, *data_rows = csv.reader(io.StringIO(csv_text))
print(f"{header_row=}, {data_rows=}")

undo 'bold'; remaining stack: ["type 'a'", "type 'b'"]
[result['item'] for result in api_page]=['idli', 'dosa']
header_row=['id', 'item', 'price'], data_rows=[['1', 'idli', '40'], ['2', 'dosa', '60']]


**`tuple`**: a fixed record whose positions mean something: database rows (the
`sqlite3` module returns tuples), `(latitude, longitude)`, several return values from
one function, composite dict keys like `(country, product)`, and constants that must
not change.

In [195]:
WEEKDAY_NAMES = ("Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun")


def lowest_and_highest(prices: list[int]) -> tuple[int, int]:
    """Return the lowest and highest price as one tuple."""
    return min(prices), max(prices)


cheapest_price, dearest_price = lowest_and_highest([40, 60, 35, 120])
print(f"{cheapest_price=}, {dearest_price=}, {WEEKDAY_NAMES[4]=}")
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE stores (city TEXT, latitude REAL, longitude REAL)")
connection.execute("INSERT INTO stores VALUES ('Pune', 18.52, 73.86)")
store_row = connection.execute("SELECT * FROM stores").fetchone()
connection.close()
print(f"{store_row=}, {type(store_row).__name__=}")
units_sold = {("IN", "dosa"): 1200, ("IN", "idli"): 800, ("US", "dosa"): 150}
print(f"{sum(count for (country, _), count in units_sold.items() if country == 'IN')=}")

cheapest_price=35, dearest_price=120, WEEKDAY_NAMES[4]='Fri'
store_row=('Pune', 18.52, 73.86), type(store_row).__name__='tuple'
sum(count for (country, _), count in units_sold.items() if country == 'IN')=2000


**`namedtuple` / `NamedTuple`**: a lightweight read-only record with field names,
such as a parsed log line or a CSV row. This reuses `LogEntry` from 1.5.6.

In [196]:
raw_log_lines = [
    "INFO|order placed|201",
    "ERROR|payment failed|502",
    "WARNING|slow response|200",
]
parsed_entries = []
for raw_log_line in raw_log_lines:
    level, message, status_text = raw_log_line.split("|")
    parsed_entries.append(LogEntry(level, message, int(status_text)))
print([entry.message for entry in parsed_entries if entry.is_error()])
print(f"{parsed_entries[0]._asdict()=}")

['payment failed']
parsed_entries[0]._asdict()={'level': 'INFO', 'message': 'order placed', 'status_code': 201}


**`range`**: index loops, pagination offsets (`range(0, total, page_size)`) and
membership checks on numeric intervals.

In [197]:
total_results, page_size = 95, 20
for page_number, page_offset in enumerate(range(0, total_results, page_size), start=1):
    page_end = min(page_offset + page_size, total_results)
    print(f"page {page_number}: results {page_offset + 1}-{page_end}")
working_age = range(18, 61)
print(f"{25 in working_age=}, {61 in working_age=}, {17.5 in working_age=}")

page 1: results 1-20
page 2: results 21-40
page 3: results 41-60
page 4: results 61-80
page 5: results 81-95
25 in working_age=True, 61 in working_age=False, 17.5 in working_age=False


**`array.array`**: large numeric series where memory matters, such as sensor readings.
The type code `"f"` stores 4-byte C floats, and `tobytes()` gives the raw block,
ready for a file or a socket.

In [198]:
temperature_readings = array.array("f")
for minute in range(60 * 24):
    temperature_readings.append(20 + (minute % 120) / 10)
average_temperature = sum(temperature_readings) / len(temperature_readings)
print(f"{len(temperature_readings)=}, {average_temperature=:.2f}")
print(f"{len(temperature_readings.tobytes())=:,} bytes for a day of readings")

len(temperature_readings)=1440, average_temperature=25.95
len(temperature_readings.tobytes())=5,760 bytes for a day of readings


### Lab 1.5

Three coding exercises and a set of predict-the-output questions on aliasing. For each
coding exercise, replace the `raise NotImplementedError` in the stub, then run the
check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — sort employee records by several keys

Order employees by **department ascending**, then **salary descending**, then
**name ascending**. Write it twice:

1. `sort_employees_multipass(employees)`: several stable `sorted()` passes, with no
   tuple keys.
2. `sort_employees_tuple_key(employees)`: one `sorted()` call with a tuple key.

Both return a **new** list and leave the input unchanged. Run the next cell first; it
defines `Employee` and the sample data.

In [199]:
class Employee(NamedTuple):
    """An employee record for the sorting exercise."""

    name: str
    department: str
    salary: int


SAMPLE_EMPLOYEES = [
    Employee("Asha", "Engineering", 120_000),
    Employee("Ravi", "Sales", 80_000),
    Employee("Meena", "Engineering", 150_000),
    Employee("John", "Sales", 80_000),
    Employee("Fatima", "Design", 95_000),
    Employee("Arjun", "Engineering", 120_000),
]

The two stubs:

In [200]:
def sort_employees_multipass(employees: list[Employee]) -> list[Employee]:
    """Sort by department, salary (high first), name using stable multi-pass sorting."""
    raise NotImplementedError("Lab 1.5: implement sort_employees_multipass()")


def sort_employees_tuple_key(employees: list[Employee]) -> list[Employee]:
    """Sort by department, salary (high first), name using one tuple key."""
    raise NotImplementedError("Lab 1.5: implement sort_employees_tuple_key()")

The check cell defines two helpers: `employee_names` (the names in order) and
`returns_new_list` (checks the input list is left alone).

In [201]:
def employee_names(employees: list[Employee]) -> list[str]:
    """Return just the names, in order."""
    return [employee.name for employee in employees]


def returns_new_list(sort_function: Callable[[list[Employee]], list[Employee]]) -> bool:
    """Return True if sort_function returns a new list and leaves its argument alone."""
    employees_argument = list(SAMPLE_EMPLOYEES)
    sorted_employees = sort_function(employees_argument)
    return sorted_employees is not employees_argument and employees_argument == SAMPLE_EMPLOYEES


expected_employee_order = ["Fatima", "Meena", "Arjun", "Asha", "John", "Ravi"]
run_checks(
    [
        (
            "multi-pass order",
            lambda: employee_names(sort_employees_multipass(SAMPLE_EMPLOYEES)),
            expected_employee_order,
        ),
        (
            "tuple-key order",
            lambda: employee_names(sort_employees_tuple_key(SAMPLE_EMPLOYEES)),
            expected_employee_order,
        ),
        (
            "both versions agree",
            lambda: (
                sort_employees_multipass(SAMPLE_EMPLOYEES)
                == sort_employees_tuple_key(SAMPLE_EMPLOYEES)
            ),
            True,
        ),
        ("multi-pass returns a new list", lambda: returns_new_list(sort_employees_multipass), True),
        ("tuple-key returns a new list", lambda: returns_new_list(sort_employees_tuple_key), True),
        ("empty list", lambda: sort_employees_tuple_key([]), []),
    ]
)

FAIL  multi-pass order: raised NotImplementedError: Lab 1.5: implement sort_employees_multipass()
FAIL  tuple-key order: raised NotImplementedError: Lab 1.5: implement sort_employees_tuple_key()
FAIL  both versions agree: raised NotImplementedError: Lab 1.5: implement sort_employees_multipass()
FAIL  multi-pass returns a new list: raised NotImplementedError: Lab 1.5: implement sort_employees_multipass()
FAIL  tuple-key returns a new list: raised NotImplementedError: Lab 1.5: implement sort_employees_tuple_key()
FAIL  empty list: raised NotImplementedError: Lab 1.5: implement sort_employees_tuple_key()
0/6 checks passed


#### Exercise 2 — `transpose(matrix)`

Write `transpose(matrix: list[list[T]]) -> list[list[T]]` using `zip(*matrix)`:
rows become columns. Return a list of **lists** (not tuples). `[]` transposes to
`[]`. If the rows have different lengths, raise `ValueError` (*hint:*
`zip(..., strict=True)`).

In [202]:
def transpose[T](matrix: list[list[T]]) -> list[list[T]]:
    """Return the transpose of a rectangular matrix given as a list of rows."""
    raise NotImplementedError("Lab 1.5: implement transpose()")

Run the checks.

In [203]:
run_checks(
    [
        ("2x3 becomes 3x2", lambda: transpose([[1, 2, 3], [4, 5, 6]]), [[1, 4], [2, 5], [3, 6]]),
        ("square", lambda: transpose([[1, 2], [3, 4]]), [[1, 3], [2, 4]]),
        ("single row", lambda: transpose([["a", "b"]]), [["a"], ["b"]]),
        ("empty matrix", lambda: transpose([]), []),
        ("rows are lists", lambda: type(transpose([[1, 2]])[0]).__name__, "list"),
        (
            "ragged rows raise",
            lambda: raised_error_name(lambda: transpose([[1, 2], [3]])),
            "ValueError",
        ),
    ]
)

FAIL  2x3 becomes 3x2: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  square: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  single row: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  empty matrix: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  rows are lists: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  ragged rows raise: expected 'ValueError', got 'NotImplementedError'
0/6 checks passed


#### Exercise 3 — `dedupe_preserving_order(items)`

Write `dedupe_preserving_order(items: Iterable[T]) -> list[T]` for hashable items:
keep the **first** occurrence of each item, in the original order. It must accept any
iterable, including a one-shot iterator. Note that `1`, `1.0` and `True` are equal
and hash the same, so they count as duplicates.

In [204]:
def dedupe_preserving_order[T](items: Iterable[T]) -> list[T]:
    """Return the items without duplicates, keeping first occurrences in order."""
    raise NotImplementedError("Lab 1.5: implement dedupe_preserving_order()")

Run the checks.

In [205]:
run_checks(
    [
        ("ints", lambda: dedupe_preserving_order([3, 1, 3, 2, 1]), [3, 1, 2]),
        ("strings", lambda: dedupe_preserving_order(["b", "a", "b", "c", "a"]), ["b", "a", "c"]),
        ("empty", lambda: dedupe_preserving_order([]), []),
        (
            "one-shot iterator",
            lambda: dedupe_preserving_order(iter("mississippi")),
            ["m", "i", "s", "p"],
        ),
        ("1, 1.0 and True are equal", lambda: dedupe_preserving_order([1, 1.0, True, 2]), [1, 2]),
        ("tuples", lambda: dedupe_preserving_order([(1, 2), (1, 2), (2, 1)]), [(1, 2), (2, 1)]),
    ]
)

FAIL  ints: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  strings: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  empty: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  one-shot iterator: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  1, 1.0 and True are equal: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  tuples: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
0/6 checks passed


#### Exercise 4 — predict the output (aliasing)

**Predict the output.**

```python
seating_grid = [[0] * 3] * 3
seating_grid[0][0] = 1
print(seating_grid)
```

<details><summary>Answer</summary>

`[[1, 0, 0], [1, 0, 0], [1, 0, 0]]`. `[0] * 3` builds one inner list, and `* 3` on the
outer list copies the **reference** to it three times. All three rows are the same
object, so changing one changes them all. (`[0] * 3` itself is fine: ints are
immutable, so sharing them is harmless.)

</details>

Run the next cell to check your prediction.

In [206]:
seating_grid = [[0] * 3] * 3
seating_grid[0][0] = 1
print(seating_grid)
print(f"{seating_grid[0] is seating_grid[1]=}")

[[1, 0, 0], [1, 0, 0], [1, 0, 0]]
seating_grid[0] is seating_grid[1]=True


**Predict the output.**

```python
seating_grid = [[0] * 3 for _ in range(3)]
seating_grid[0][0] = 1
print(seating_grid)
```

<details><summary>Answer</summary>

`[[1, 0, 0], [0, 0, 0], [0, 0, 0]]`. The comprehension evaluates `[0] * 3` once per
row, so each row is a separate list. This is the correct way to build a grid.

</details>

Run the next cell to check your prediction.

In [207]:
seating_grid = [[0] * 3 for _ in range(3)]
seating_grid[0][0] = 1
print(seating_grid)
print(f"{seating_grid[0] is seating_grid[1]=}")

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]
seating_grid[0] is seating_grid[1]=False


**Predict the output.**

```python
pending_orders = processed_orders = []
pending_orders.append("A-17")
print(processed_orders, pending_orders is processed_orders)
```

<details><summary>Answer</summary>

`['A-17'] True`. Chained assignment binds both names to **one** list object. Write
`pending_orders, processed_orders = [], []` for two lists.

</details>

Run the next cell to check your prediction.

In [208]:
pending_orders = processed_orders = []
pending_orders.append("A-17")
print(processed_orders, pending_orders is processed_orders)

['A-17'] True


**Predict the output.**

```python
shelf_rows = [[]] * 2
shelf_rows[0] += ["jam"]
shelf_rows[1] = shelf_rows[1] + ["tea"]
print(shelf_rows)
```

<details><summary>Answer</summary>

`[['jam'], ['jam', 'tea']]`. Both slots start as the same empty list. `+=` extends
that shared list in place, so both rows show `'jam'`. `shelf_rows[1] + ["tea"]` builds
a **new** list and rebinds only slot 1.

</details>

Run the next cell to check your prediction.

In [209]:
shelf_rows = [[]] * 2
shelf_rows[0] += ["jam"]
shelf_rows[1] = shelf_rows[1] + ["tea"]
print(shelf_rows)

[['jam'], ['jam', 'tea']]


### Recap — Unit 1.5

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Sequence operations | Indexing, slicing, `in`, `len`, `min`, `max`, `count`, `index` on every sequence; `range` has no `+`/`*` | Array methods |
| `list` internals | Dynamic array of pointers with spare capacity; `append` is O(1) amortized | `Array` |
| Front operations | `pop(0)` and `insert(0, x)` are O(n) | `shift`, `unshift` |
| Mutating methods | Return `None`; `sorted()` and `+` return new lists | `push` returns the length |
| `+=` vs `+` | `+=` extends in place (any iterable); `+` builds a new list | `push(...items)` vs `concat` |
| Slice assignment | Replaces a slice, may change the length; extended slices must match | `splice` |
| Copies | `copy()`, `[:]`, `list()` are shallow | `[...arr]`, `slice()` |
| Sorting | Stable Timsort; `key=`, `reverse=`, `itemgetter`, tuple keys with negation | `sort((a, b) => …)` |
| `tuple` | The comma makes it; immutable; hashable if all items are | `Object.freeze([])` |
| `t[0] += [x]` | Mutates the inner list *and* raises `TypeError` | — |
| Named tuples | `namedtuple` / `NamedTuple`: fields, defaults, `_replace`, `_asdict` | object literal |
| `range` | Lazy, O(1) `len`/`in`/index, constant memory | — |
| `array.array` | Typed, compact numeric storage | `Float64Array` |
| `[[0] * 3] * 3` | Repeats references; build grids with a comprehension | `Array(3).fill([])` has the same bug |

**Further reading**

- [Sequence types — `list`, `tuple`, `range`](https://docs.python.org/3/library/stdtypes.html#sequence-types-list-tuple-range),
  [more on lists](https://docs.python.org/3/tutorial/datastructures.html#more-on-lists)
- [Sorting techniques](https://docs.python.org/3/howto/sorting.html),
  [`operator.itemgetter` / `attrgetter`](https://docs.python.org/3/library/operator.html#operator.itemgetter)
- [`collections.namedtuple`](https://docs.python.org/3/library/collections.html#collections.namedtuple),
  [`typing.NamedTuple`](https://docs.python.org/3/library/typing.html#typing.NamedTuple)
- [`array`](https://docs.python.org/3/library/array.html),
  [Time complexity of CPython operations](https://wiki.python.org/moin/TimeComplexity)
- [PEP 3132 — Extended iterable unpacking](https://peps.python.org/pep-3132/),
  [PEP 448 — Additional unpacking generalizations](https://peps.python.org/pep-0448/),
  [PEP 695 — Type parameter syntax](https://peps.python.org/pep-0695/) (used by the
  `transpose[T]` stub)

## Unit 1.6 — Collection Data Types II: Dictionaries, Sets and the collections Module

**Objectives**

- Explain how a `dict` is stored (compact hash table, probing, resizing, key sharing)
  and why lookups are O(1) on average.
- State the hashing contract, and predict which keys collapse into one.
- Use the complete `dict` and `set` APIs, including views, merge operators and
  in-place set algebra.
- Choose between `dict`, `frozendict`, `MappingProxyType`, `set`, `frozenset` and the
  `collections` types (`deque`, `Counter`, `defaultdict`, `OrderedDict`, `ChainMap`).
- Build stacks, queues, priority queues, graphs, tries and linked lists from built-ins.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `const prices = {dosa: 60}` | `prices = {"dosa": 60}` | Keys are expressions: a bare `dosa` is a variable name |
| `prices.dosa`, `prices["dosa"]` | `prices["dosa"]` | No dot access; a missing key raises `KeyError` (use `.get`) |
| `new Map()` with object keys | `dict` with any hashable key | Mutable keys (lists, dicts, sets) are rejected |
| `obj[1]` is the same key as `obj["1"]` | `1` and `"1"` are different keys | But `1`, `1.0` and `True` are the **same** key |
| `{...defaults, ...overrides}` | `{**defaults, **overrides}` or `defaults \| overrides` | Both build a new dict; `\|=` updates in place |
| `new Set([1, 2])` | `{1, 2}` or `set(iterable)` | `{}` is an empty **dict**; use `set()` |
| `a.union(b)` (ES2025 Set methods) | `a \| b` or `a.union(b)` | Operators need sets on both sides; methods take any iterable |
| `Object.freeze(obj)` | `frozendict(...)` / `types.MappingProxyType(...)` | `frozendict` is new in 3.15 and hashable |
| `queue.shift()` | `queue.popleft()` on a `deque` | O(1); `list.pop(0)` is O(n) |
| `(groups[k] ??= []).push(x)` | `groups.setdefault(k, []).append(x)` or `defaultdict(list)` | `defaultdict` inserts on every missing-key *read* |

> **Note:** sets of strings print in an order that changes between interpreter runs
> (string hashes are randomized per process, see 1.6.2). Demos print `sorted(...)` of
> such sets so the saved outputs stay stable.

The next cell is this unit's setup.

In [210]:
import argparse
import array
import heapq
import json
import os
import re
import subprocess
import sys
import timeit
import tracemalloc
import types
from collections import ChainMap, Counter, OrderedDict, defaultdict, deque, namedtuple
from collections.abc import Callable, Hashable, Iterable, Mapping, MutableMapping
from decimal import Decimal
from typing import Any, NamedTuple


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


class LogEntry(NamedTuple):
    """One parsed line of an access log (as in 1.5.6)."""

    level: str
    message: str
    status_code: int = 200

    def is_error(self) -> bool:
        """Return True for ERROR entries and 5xx status codes."""
        return self.level == "ERROR" or self.status_code >= 500


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.6.1 Dictionaries (`dict`) — internals

Since CPython 3.6 a dict is a **compact hash table** built from two arrays:

- a sparse **indices** array: one small integer per slot (1 byte each while the
  table has at most 128 slots), holding the position of an entry or "empty";
- a dense **entries** array: `(hash, key, value)` rows, appended in insertion order.

Looking up `key` hashes it, picks a slot with `hash & (table_size - 1)`, reads the
entry index from that slot, and compares the stored hash and key. Because entries are
appended, iteration simply walks the entries array, which is why **insertion order is
guaranteed** (a language rule since 3.7). The split also saves memory: the wide rows
are dense, and only the 1-byte indices are sparse.

```mermaid
flowchart LR
    indices["indices (sparse, 8 slots)<br/>slot: 0 1 2 3 4 5 6 7<br/>value: - 1 - - 0 - 2 -"]
    entries["entries (dense, insertion order)<br/>0: hash, 'dosa', 60<br/>1: hash, 'idli', 40<br/>2: hash, 'vada', 30"]
    indices -- "slot = hash & 7<br/>value = entry position" --> entries
```

<details><summary>ASCII fallback</summary>

```text
indices (sparse)              entries (dense, in insertion order)
slot  value                   pos  hash      key     value
0     -                       0    h(dosa)   'dosa'  60
1     1  ------------------>  1    h(idli)   'idli'  40
2     -                       2    h(vada)   'vada'  30
3     -
4     0  ------------------>  (pos 0)
5     -
6     2  ------------------>  (pos 2)
7     -
```

</details>

The next cell shows the order rules: replacing a value keeps the key's position;
deleting and re-inserting moves the key to the end.

In [211]:
menu_prices = {"dosa": 60, "idli": 40, "vada": 30}
menu_prices["upma"] = 45
menu_prices["dosa"] = 65  # replacing a value keeps the original position
del menu_prices["idli"]
menu_prices["idli"] = 40  # re-inserting goes to the end
print(f"{list(menu_prices)=}")
print(f"{menu_prices=}")

list(menu_prices)=['dosa', 'vada', 'upma', 'idli']
menu_prices={'dosa': 65, 'vada': 30, 'upma': 45, 'idli': 40}


**Open addressing with perturbed probing.** Two keys can land in the same slot. CPython
does not chain them in a list; it probes other slots of the same table. The recurrence
(from `Objects/dictobject.c`) mixes in the high bits of the hash, so keys that collide
at first soon follow different paths:

```text
perturb = hash
slot = hash & mask
while slot is taken by a different key:
    perturb >>= 5
    slot = (slot * 5 + perturb + 1) & mask
```

The next cell runs that recurrence in pure Python. `hash(n)` is `n` for small
non-negative ints, so the slots are the same on every machine. `12345` and `12353`
both start in slot 1 of an 8-slot table (they differ by 8), then split apart.

In [212]:
def probe_sequence(hash_value: int, table_size: int, probe_count: int) -> list[int]:
    """Return the first slots CPython's dict tries for hash_value (table_size: power of 2)."""
    mask = table_size - 1
    perturb = hash_value
    slot = hash_value & mask
    tried_slots = [slot]
    for _ in range(probe_count - 1):
        perturb >>= 5
        slot = (slot * 5 + perturb + 1) & mask
        tried_slots.append(slot)
    return tried_slots


print(f"{probe_sequence(hash(12345), 8, 8)=}")
print(f"{probe_sequence(hash(12353), 8, 8)=}")
print(f"{probe_sequence(hash(1), 8, 8)=}  (perturb hits 0 after one shift)")

probe_sequence(hash(12345), 8, 8)=[1, 7, 0, 1, 6, 7, 4, 5]
probe_sequence(hash(12353), 8, 8)=[1, 0, 5, 2, 3, 0, 1, 6]
probe_sequence(hash(1), 8, 8)=[1, 6, 7, 4, 5, 2, 3, 0]  (perturb hits 0 after one shift)


**Resizing.** A table is kept at most about two-thirds full, so probes stay short.
When an insert would pass that limit, CPython allocates a bigger table and re-inserts
every entry: an O(n) step that happens rarely enough for inserts to stay O(1)
*amortized* (the same idea as `list` over-allocation in 1.5.2). Watch
`sys.getsizeof` jump. The exact sizes vary by Python version and platform.

In [213]:
growing_dict: dict[int, int] = {}
previous_size = sys.getsizeof(growing_dict)
print(f"0 keys: {previous_size} bytes")
for key_count in range(1, 100):
    growing_dict[key_count] = key_count
    current_size = sys.getsizeof(growing_dict)
    if current_size != previous_size:
        print(f"{key_count:>2} keys: {current_size:>5} bytes (table resized)")
        previous_size = current_size

0 keys: 64 bytes
 1 keys:   224 bytes (table resized)
 6 keys:   352 bytes (table resized)
11 keys:   632 bytes (table resized)
22 keys:  1168 bytes (table resized)
43 keys:  2264 bytes (table resized)
86 keys:  4688 bytes (table resized)


#### Under the hood: key-sharing dicts (PEP 412)

Every instance of a class keeps its attributes in a namespace dict. Most instances of
one class have the **same attribute names**, so CPython stores the keys once per class
and gives each instance only its values (a *split* or *key-sharing* dict; recent versions
keep the values inside the object itself until someone asks for `__dict__`).

`sys.getsizeof(instance.__dict__)` is misleading here (it reports a materialized dict),
so the next cell uses `tracemalloc` to measure the memory of 10 000 objects of each
kind. Expect instances to cost clearly less than plain dicts with the same three keys;
the exact numbers vary by version.

In [214]:
class MenuItem:
    """A plain class: its instances share one key table (PEP 412)."""

    def __init__(self, name: str, price: int, is_veg: bool) -> None:
        self.name = name
        self.price = price
        self.is_veg = is_veg


def bytes_per_object(factory: Callable[[int], object], object_count: int = 10_000) -> float:
    """Return the average traced memory of object_count objects built by factory."""
    tracemalloc.start()
    kept_objects = [factory(number) for number in range(object_count)]
    traced_bytes, _peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return traced_bytes / len(kept_objects)


instance_bytes = bytes_per_object(lambda number: MenuItem(f"dish {number}", 60, True))
dict_bytes = bytes_per_object(
    lambda number: {"name": f"dish {number}", "price": 60, "is_veg": True}
)
print(f"MenuItem instance: about {instance_bytes:.0f} bytes each")
print(f"plain dict:        about {dict_bytes:.0f} bytes each")
print(f"{instance_bytes < dict_bytes=}")

MenuItem instance: about 155 bytes each
plain dict:        about 242 bytes each
instance_bytes < dict_bytes=True


**Dict vs list membership.** `key in some_dict` hashes once and probes a slot or two;
`value in some_list` compares against every item. The next cell searches 10 000 SKU
codes for one that is missing (the worst case for the list). Timings vary by machine;
the ratio is what matters.

In [215]:
sku_codes = [f"SKU{number:05}" for number in range(10_000)]
sku_lookup = dict.fromkeys(sku_codes, True)
missing_sku = "SKU-MISSING"
repeat_count = 200
list_seconds = timeit.timeit(lambda: missing_sku in sku_codes, number=repeat_count)
dict_seconds = timeit.timeit(lambda: missing_sku in sku_lookup, number=repeat_count)
print(f"list membership: {list_seconds / repeat_count * 1e6:8.2f} µs per lookup")
print(f"dict membership: {dict_seconds / repeat_count * 1e6:8.3f} µs per lookup")
print(f"dict is about {list_seconds / dict_seconds:,.0f}x faster here (varies by machine)")

list membership:    53.58 µs per lookup
dict membership:    0.033 µs per lookup
dict is about 1,615x faster here (varies by machine)


### 1.6.2 Hashing

`hash(obj)` returns an `int` computed by `obj.__hash__()`. An object is **hashable**
when it has a hash that never changes during its lifetime **and** that agrees with
`__eq__`: if `a == b` then `hash(a) == hash(b)` (the reverse need not hold; that is a
collision). Dicts and sets rely on both halves: the hash picks the slot, `==` confirms
the match.

Integer hashes are deterministic (`hash(n) == n` for small ints, and `-1` is reserved
as an error marker in C, so `hash(-1) == -2`). Equal tuples hash equally.

In [216]:
print(f"{hash(42)=}, {hash(-1)=}, {hash(-2)=}")
print(f"{hash((1, 'dosa')) == hash((1, 'dosa'))=}")
print(f"{isinstance('dosa', Hashable)=}, {isinstance([], Hashable)=}")

hash(42)=42, hash(-1)=-2, hash(-2)=-2
hash((1, 'dosa')) == hash((1, 'dosa'))=True
isinstance('dosa', Hashable)=True, isinstance([], Hashable)=False


**The contract in practice.** Defining `__eq__` without `__hash__` sets `__hash__` to
`None`, so instances become unhashable: Python refuses to guess a hash that agrees
with your equality. The second class breaks the contract by hashing a field that
changes; after the change the object sits in the wrong slot and the set can no longer
find it.

In [217]:
class SkuCode:
    """Equal when the codes match; no __hash__, so it is unhashable."""

    def __init__(self, code: str) -> None:
        self.code = code

    def __eq__(self, other: object) -> bool:
        return isinstance(other, SkuCode) and self.code == other.code


print(f"{SkuCode.__hash__=}")
print(f"hash(SkuCode('A1')): {raised_error_name(lambda: hash(SkuCode('A1')))}")

SkuCode.__hash__=None
hash(SkuCode('A1')): TypeError


The broken version: the hash depends on a field that is later changed.

In [218]:
class MutableSkuCode(SkuCode):
    """Hashes a mutable field: breaks the hashing contract."""

    def __hash__(self) -> int:
        return hash(self.code)


tracked_sku = MutableSkuCode("A1")
tracked_skus = {tracked_sku}
tracked_sku.code = "B2"  # the stored hash is still hash("A1")
print(f"{tracked_sku in tracked_skus=}  (looks in the slot for 'B2')")
print(f"{MutableSkuCode('A1') in tracked_skus=}  (right slot, but == fails)")
print(f"{len(tracked_skus)=}  (the object is still there, unreachable)")

tracked_sku in tracked_skus=False  (looks in the slot for 'B2')
MutableSkuCode('A1') in tracked_skus=False  (right slot, but == fails)
len(tracked_skus)=1  (the object is still there, unreachable)


**`1`, `1.0` and `True` are one key.** `True == 1 == 1.0`, so the language requires
`hash(True) == hash(1) == hash(1.0)`. A dict treats them as the same key: the **first**
key object is kept, the **last** value wins.

In [219]:
print(f"{hash(1) == hash(1.0) == hash(True)=}")
print(f"{1 == 1.0 == True=}")
print(f"{ {1: 'int', True: 'bool'}=}")  # noqa: F601
print(f"{len({1, 1.0, True})=}")  # noqa: B033

hash(1) == hash(1.0) == hash(True)=True
1 == 1.0 == True=True
 {1: 'int', True: 'bool'}={1: 'bool'}
len({1, 1.0, True})=1


#### Predict the output: mixed numeric keys

**Predict the output.**

```python
seat_labels = {1: "one", True: "yes", 1.0: "float one"}
print(seat_labels)
```

<details><summary>Answer</summary>

`{1: 'float one'}`. All three keys are equal and hash the same, so they are one key.
The dict keeps the first key object (`1`) and stores each new value over the old
one, so the last value (`'float one'`) wins.

</details>

Run the next cell to check your prediction.

In [220]:
seat_labels = {1: "one", True: "yes", 1.0: "float one"}  # noqa: F601
print(seat_labels)

{1: 'float one'}


**String-hash randomization.** `str` and `bytes` hashes use a random key chosen when
the interpreter starts. Without it, an attacker who knows the hash function can send
thousands of keys that all collide (for example as form fields or JSON keys), turning
each dict insert into O(n) work: a **hash-flooding** denial of service (2011–2012,
CVE-2012-1150). Since Python 3.4 the function is **SipHash** (PEP 456), a keyed hash
designed against this attack (SipHash-2-4 at first, the faster SipHash-1-3 since
3.11); `sys.hash_info.algorithm` names the variant.

`PYTHONHASHSEED` controls the key: unset or `random` → a new key per process; an
integer → a fixed key (useful only for reproducing a bug). The next cell starts child
interpreters and compares their `hash("dosa")` values. It prints counts, never the
hashes themselves, because those differ per run.

In [221]:
def string_hash_in_child(hash_seed: str | None) -> str:
    """Return hash("dosa") from a fresh interpreter run with the given PYTHONHASHSEED."""
    child_environment = {
        name: value for name, value in os.environ.items() if name != "PYTHONHASHSEED"
    }
    if hash_seed is not None:
        child_environment["PYTHONHASHSEED"] = hash_seed
    completed = subprocess.run(
        [sys.executable, "-c", "print(hash('dosa'))"],
        capture_output=True,
        text=True,
        env=child_environment,
        check=True,
    )
    return completed.stdout.strip()


random_seed_hashes = {string_hash_in_child(None) for _ in range(3)}
fixed_seed_hashes = {string_hash_in_child("0") for _ in range(3)}
print(f"distinct hash('dosa') values, 3 runs, random key:      {len(random_seed_hashes)}")
print(f"distinct hash('dosa') values, 3 runs, PYTHONHASHSEED=0: {len(fixed_seed_hashes)}")
print(f"{sys.hash_info.algorithm=}, {sys.hash_info.width=}")
print(f"{sys.flags.hash_randomization=}")

distinct hash('dosa') values, 3 runs, random key:      3
distinct hash('dosa') values, 3 runs, PYTHONHASHSEED=0: 1
sys.hash_info.algorithm='siphash13', sys.hash_info.width=64
sys.flags.hash_randomization=1


**Unhashable types.** The mutable built-ins `list`, `dict`, `set` and `bytearray` have
`__hash__ = None`. A `tuple` is hashable only if every item is, so `isinstance(...,
Hashable)` can say yes while `hash()` still fails.

In [222]:
print(f"list key:  {raised_error_name(lambda: {['rice']: 1})}")
print(f"dict key:  {raised_error_name(lambda: {dict(): 1})}")
print(f"set key:   {raised_error_name(lambda: {set(): 1})}")
tuple_holding_list = ("rice", [1, 2])
print(f"{isinstance(tuple_holding_list, Hashable)=}")
print(f"hash(tuple_holding_list): {raised_error_name(lambda: hash(tuple_holding_list))}")

list key:  TypeError
dict key:  TypeError
set key:   TypeError
isinstance(tuple_holding_list, Hashable)=True
hash(tuple_holding_list): TypeError


**Tuple composite keys.** When a record is identified by several fields, use a tuple
of them as the key: no string concatenation, no separator bugs.

In [223]:
seat_bookings: dict[tuple[str, int, str], str] = {
    ("2026-11-01", 7, "A"): "Asha",
    ("2026-11-01", 7, "B"): "Ravi",
}
seat_bookings[("2026-11-02", 3, "A")] = "Meena"
print(f"{seat_bookings[('2026-11-01', 7, 'B')]=}")
print(f"{('2026-11-01', 7, 'C') in seat_bookings=}")
print(f"{[seat for seat in seat_bookings if seat[0] == '2026-11-01']=}")

seat_bookings[('2026-11-01', 7, 'B')]='Ravi'
('2026-11-01', 7, 'C') in seat_bookings=False
[seat for seat in seat_bookings if seat[0] == '2026-11-01']=[('2026-11-01', 7, 'A'), ('2026-11-01', 7, 'B')]


**Collisions cost time.** The next class returns the same hash for every key. All keys
land on the same probe path, so each insert compares against every earlier key:
building the dict becomes O(n²). Compare it with the same class using a proper hash.
Doubling the key count should roughly double the good time but roughly quadruple the
bad one (timings vary by machine).

In [224]:
class ConstantHashKey:
    """A key whose hash is always 42, so every key collides."""

    def __init__(self, value: int) -> None:
        self.value = value

    def __eq__(self, other: object) -> bool:
        return isinstance(other, ConstantHashKey) and self.value == other.value

    def __hash__(self) -> int:
        return 42


class WellHashedKey(ConstantHashKey):
    """The same key with a hash that spreads values over the table."""

    def __hash__(self) -> int:
        return hash(self.value)


def build_seconds(key_class: type[ConstantHashKey], key_count: int) -> float:
    """Time building a dict of key_count keys of key_class."""
    return timeit.timeit(
        lambda: {key_class(number): number for number in range(key_count)}, number=3
    )


for key_count in (250, 500, 1000):
    good_seconds = build_seconds(WellHashedKey, key_count)
    bad_seconds = build_seconds(ConstantHashKey, key_count)
    print(
        f"{key_count:>5} keys: good hash {good_seconds * 1e3:6.2f} ms, "
        f"constant hash {bad_seconds * 1e3:8.2f} ms"
    )

  250 keys: good hash   0.07 ms, constant hash     3.96 ms


  500 keys: good hash   0.22 ms, constant hash    18.79 ms


 1000 keys: good hash   0.27 ms, constant hash    71.87 ms


**Python has no separate "hash" type.** JavaScript has `Object`, `Map` and libraries
with "HashMap" classes; Java has `HashMap` and `Hashtable`. In Python the built-in
`dict` *is* the hash map, and `set` is the hash set. The abstract interfaces live in
`collections.abc` (`Mapping`, `MutableMapping`). A dict itself is not hashable, which
is what `frozendict` (1.6.5) fixes.

In [225]:
inventory = {"rice": 10}
print(f"{type(inventory).__name__=}")
print(f"{isinstance(inventory, Mapping)=}, {isinstance(inventory, MutableMapping)=}")
print(f"{dict.__hash__ is None=}  (the hash map itself is not hashable)")

type(inventory).__name__='dict'
isinstance(inventory, Mapping)=True, isinstance(inventory, MutableMapping)=True
dict.__hash__ is None=True  (the hash map itself is not hashable)


### 1.6.3 Dictionary methods (complete API)

This section teaches the API by task: access, modify, views, construction, copying.
1.6.9 repeats every method as a short reference demo.

**Access.** `d[key]` raises `KeyError` for a missing key. `get(key, default=None)`
never raises. `in` tests **keys**, not values. A stored `0` or `""` is a real value,
so `get` returns it rather than the default.

In [226]:
stock_levels = {"rice": 12, "dal": 0}
print(f"{stock_levels['rice']=}")
print(f"missing key with []: {raised_error_name(lambda: stock_levels['ghee'])}")
print(f"{stock_levels.get('ghee')=}, {stock_levels.get('ghee', 0)=}")
print(f"{stock_levels.get('dal', 99)=}  (0 is stored, so no default)")
print(f"{'dal' in stock_levels=}, {12 in stock_levels=}  (in checks keys)")

stock_levels['rice']=12
missing key with []: KeyError
stock_levels.get('ghee')=None, stock_levels.get('ghee', 0)=0
stock_levels.get('dal', 99)=0  (0 is stored, so no default)
'dal' in stock_levels=True, 12 in stock_levels=False  (in checks keys)


`setdefault(key, default)` returns the value for `key`, inserting `default` first if
the key is missing. It is the one-line "get or create" used for grouping. The default
is evaluated on every call, even when the key exists.

In [227]:
tags_by_dish: dict[str, list[str]] = {}
tags_by_dish.setdefault("dosa", []).append("veg")
tags_by_dish.setdefault("dosa", []).append("breakfast")
tags_by_dish.setdefault("kebab", []).append("grill")
print(f"{tags_by_dish=}")
print(f"{stock_levels.setdefault('rice', 99)=}  (existing value returned, not replaced)")
print(f"{stock_levels.setdefault('ghee', 3)=}, {stock_levels=}")

tags_by_dish={'dosa': ['veg', 'breakfast'], 'kebab': ['grill']}
stock_levels.setdefault('rice', 99)=12  (existing value returned, not replaced)
stock_levels.setdefault('ghee', 3)=3, stock_levels={'rice': 12, 'dal': 0, 'ghee': 3}


**Modify.** `d[key] = value` inserts or replaces. `update` accepts a mapping, an
iterable of `(key, value)` pairs, keyword arguments, or a mapping/pairs **plus**
keywords; later values win.

In [228]:
stock_levels = {"rice": 12}
stock_levels["dal"] = 5
stock_levels["rice"] = 10
stock_levels.update({"ghee": 2})
stock_levels.update([("salt", 7), ("tea", 3)])
stock_levels.update(sugar=4, rice=8)
stock_levels.update({"oil": 1}, oil=2)
print(f"{stock_levels=}")

stock_levels={'rice': 8, 'dal': 5, 'ghee': 2, 'salt': 7, 'tea': 3, 'sugar': 4, 'oil': 2}


**`|` and `|=`** (PEP 584, 3.9) merge dicts. `|` builds a new dict and needs a dict
on both sides; `|=` updates in place and, like `update`, also accepts pairs. On
duplicate keys the right side wins.

In [229]:
default_settings = {"theme": "light", "page_size": 20}
user_settings = {"page_size": 50}
merged_settings = default_settings | user_settings
print(f"{merged_settings=}")
print(f"{default_settings=}  (unchanged by |)")
default_settings |= [("language", "en")]
print(f"{default_settings=}  (|= accepts pairs)")
print(f"dict | list: {raised_error_name(lambda: default_settings | [('x', 1)])}")

merged_settings={'theme': 'light', 'page_size': 50}
default_settings={'theme': 'light', 'page_size': 20}  (unchanged by |)
default_settings={'theme': 'light', 'page_size': 20, 'language': 'en'}  (|= accepts pairs)
dict | list: TypeError


**Removing.** `pop(key, default)` removes and returns a value (`KeyError` without a
default); `popitem()` removes the **last inserted** pair (LIFO); `del d[key]` raises
`KeyError` if the key is missing; `clear()` empties the dict in place.

In [230]:
def delete_key(mapping: dict[str, int], key: str) -> None:
    """Run del mapping[key] (del is a statement, so it cannot go in a lambda)."""
    del mapping[key]


shopping_basket = {"rice": 2, "dal": 1, "ghee": 1}
print(f"{shopping_basket.pop('dal')=}, {shopping_basket.pop('salt', 0)=}")
print(f"pop missing, no default: {raised_error_name(lambda: shopping_basket.pop('salt'))}")
shopping_basket["tea"] = 3
print(f"{shopping_basket.popitem()=}  (the last inserted pair)")
delete_key(shopping_basket, "rice")
print(f"{shopping_basket=}")
print(f"del missing key: {raised_error_name(lambda: delete_key(shopping_basket, 'oil'))}")
shopping_basket.clear()
print(f"{shopping_basket=}, popitem on empty: {raised_error_name(shopping_basket.popitem)}")

shopping_basket.pop('dal')=1, shopping_basket.pop('salt', 0)=0
pop missing, no default: KeyError
shopping_basket.popitem()=('tea', 3)  (the last inserted pair)
shopping_basket={'ghee': 1}
del missing key: KeyError
shopping_basket={}, popitem on empty: KeyError


**Views.** `keys()`, `values()` and `items()` return *views*: small objects that read
the dict live, so later changes show through. They are not copies; wrap them in
`list()` for a snapshot.

In [231]:
menu_prices = {"dosa": 60, "idli": 40}
menu_keys = menu_prices.keys()
menu_items = menu_prices.items()
menu_prices["vada"] = 30
print(f"{menu_keys=}")
print(f"{menu_items=}")
print(f"{list(menu_prices.values())=}, {('idli', 40) in menu_items=}")

menu_keys=dict_keys(['dosa', 'idli', 'vada'])
menu_items=dict_items([('dosa', 60), ('idli', 40), ('vada', 30)])
list(menu_prices.values())=[60, 40, 30], ('idli', 40) in menu_items=True


`keys()` and `items()` views behave like sets: `&`, `|`, `-`, `^` work on them and
return real `set` objects (and the right operand can be any iterable). `values()`
views do not, because values need not be hashable or unique. The results are sets of
strings, so they are printed sorted.

In [232]:
store_a_stock = {"rice": 12, "dal": 5, "ghee": 2}
store_b_stock = {"dal": 3, "salt": 9}
print(f"{sorted(store_a_stock.keys() & store_b_stock.keys())=}")
print(f"{sorted(store_a_stock.keys() - store_b_stock.keys())=}")
print(f"{sorted(store_a_stock.keys() | store_b_stock.keys())=}")
print(f"{sorted(store_a_stock.keys() ^ store_b_stock.keys())=}")
print(f"{sorted(store_a_stock.keys() & ['dal', 'oil'])=}  (any iterable on the right)")
print(f"values() & ...: {raised_error_name(lambda: store_a_stock.values() & {5})}")

sorted(store_a_stock.keys() & store_b_stock.keys())=['dal']
sorted(store_a_stock.keys() - store_b_stock.keys())=['ghee', 'rice']
sorted(store_a_stock.keys() | store_b_stock.keys())=['dal', 'ghee', 'rice', 'salt']
sorted(store_a_stock.keys() ^ store_b_stock.keys())=['ghee', 'rice', 'salt']
sorted(store_a_stock.keys() & ['dal', 'oil'])=['dal']  (any iterable on the right)
values() & ...: TypeError


**Construction.** Five equivalent ways to build the same dict, plus `dict.fromkeys`
(every key gets the same value). `zip(..., strict=True)` raises `ValueError` if the
key and value lists have different lengths, instead of silently dropping items.

In [233]:
from_literal = {"dosa": 60, "idli": 40}
from_keywords = dict(dosa=60, idli=40)
from_pairs = dict([("dosa", 60), ("idli", 40)])
from_zip = dict(zip(["dosa", "idli"], [60, 40], strict=True))
from_comprehension = {dish: price for dish, price in [("dosa", 60), ("idli", 40)]}
print(f"{from_literal == from_keywords == from_pairs == from_zip == from_comprehension=}")
print(f"{dict.fromkeys(['dosa', 'idli'], 0)=}")
mismatched_zip = raised_error_name(lambda: dict(zip(["dosa"], [60, 40], strict=True)))
print(f"zip with different lengths: {mismatched_zip}")

from_literal == from_keywords == from_pairs == from_zip == from_comprehension=True
dict.fromkeys(['dosa', 'idli'], 0)={'dosa': 0, 'idli': 0}
zip with different lengths: ValueError


#### Predict the output: the `dict.fromkeys` shared-value trap

**Predict the output.**

```python
orders_by_table = dict.fromkeys(["T1", "T2"], [])
orders_by_table["T1"].append("dosa")
print(orders_by_table)
```

<details><summary>Answer</summary>

`{'T1': ['dosa'], 'T2': ['dosa']}`. `fromkeys` evaluates `[]` **once** and stores that
same list object under every key, so appending through one key shows under all of
them. It is the dict version of `[[0] * 3] * 3` from Lab 1.5.

</details>

Run the next cell to check your prediction.

In [234]:
orders_by_table = dict.fromkeys(["T1", "T2"], [])
orders_by_table["T1"].append("dosa")
print(orders_by_table)
print(f"{orders_by_table['T1'] is orders_by_table['T2']=}")

{'T1': ['dosa'], 'T2': ['dosa']}
orders_by_table['T1'] is orders_by_table['T2']=True


The fix: a comprehension evaluates `[]` once **per key**. (`dict.fromkeys` is still
fine with immutable values such as `0`, `None` or `False`.)

In [235]:
orders_by_table = {table: [] for table in ["T1", "T2"]}
orders_by_table["T1"].append("dosa")
print(f"{orders_by_table=}")

orders_by_table={'T1': ['dosa'], 'T2': []}


**Copying.** `copy()` and `{**d}` both make a **shallow** copy: a new outer dict whose
values are the same objects. Rebinding a key in the copy is independent; mutating a
shared inner list is not. (Unit 1.7 covers deep copies.)

In [236]:
dosa_recipe = {"ingredients": ["rice", "urad dal"], "price": 60}
copied_recipe = dosa_recipe.copy()
unpacked_recipe = {**dosa_recipe, "price": 65}
copied_recipe["price"] = 70  # rebinding: only the copy changes
copied_recipe["ingredients"].append("salt")  # mutating a shared list
print(f"{dosa_recipe=}")
print(f"{unpacked_recipe=}")
print(f"{copied_recipe['ingredients'] is dosa_recipe['ingredients']=}")

dosa_recipe={'ingredients': ['rice', 'urad dal', 'salt'], 'price': 60}
unpacked_recipe={'ingredients': ['rice', 'urad dal', 'salt'], 'price': 65}
copied_recipe['ingredients'] is dosa_recipe['ingredients']=True


**Changing size during iteration raises `RuntimeError`.** Adding or deleting keys
while a `for` loop walks the dict could skip or repeat entries after a resize, so
CPython detects the size change and stops. Replacing values is allowed. To delete
while looping, iterate over a snapshot (`list(d)`) or build a new dict.

In [237]:
stock_levels = {"rice": 12, "dal": 0, "ghee": 0, "salt": 7}
try:
    for item_name in stock_levels:
        if stock_levels[item_name] == 0:
            del stock_levels[item_name]
except RuntimeError as error:
    print(f"{type(error).__name__}: {error}")
for item_name in stock_levels:
    stock_levels[item_name] += 1  # replacing values is fine
for item_name in list(stock_levels):  # snapshot of the keys
    if stock_levels[item_name] == 1:
        del stock_levels[item_name]
print(f"{stock_levels=}")

RuntimeError: dictionary changed size during iteration
stock_levels={'rice': 13, 'salt': 8}


### 1.6.4 dict vs JS objects

A Python `dict` behaves much more like a JS `Map` than like a plain object.

| Feature | JS `Object` | JS `Map` | Python `dict` |
| --- | --- | --- | --- |
| Key types | strings and symbols only | any value | any **hashable** object (`str`, `int`, `tuple`, `frozenset`, …) |
| Key coercion | `obj[1]` and `obj["1"]` are the same key | none | none: `1` and `"1"` differ (but `1 == 1.0 == True` collapse, 1.6.2) |
| Prototype keys | inherited: `"toString" in {}` is `true` | none | none: methods are attributes, never keys |
| Order | integer-like keys first (ascending), then other strings in insertion order | insertion order | insertion order (guaranteed since 3.7) |
| Size | `Object.keys(obj).length` | `map.size` | `len(d)` |
| Missing key | `undefined` | `undefined` | `KeyError` with `d[key]`; `None`/default with `d.get(key)` |
| Dot access | `obj.name` | no | no: `d["name"]`; use `SimpleNamespace` or a class |
| JSON | native | needs conversion | `json.dumps(d)` (string keys only; see 1.6.12) |

The next cell checks the Python column.

In [238]:
mixed_keys = {1: "int one", "1": "string one", (2, 3): "tuple key"}
print(f"{len(mixed_keys)=}, {mixed_keys[1]=}, {mixed_keys['1']=}")
empty_dict: dict[str, int] = {}
print(f"{'keys' in empty_dict=}, {'__init__' in empty_dict=}  (methods are not keys)")
insertion_ordered = {"b": 1, 10: 2, "a": 3, 2: 4}
print(f"{list(insertion_ordered)=}  (JS would list 2 and 10 first)")
print(f"missing key: {raised_error_name(lambda: empty_dict['name'])}")

len(mixed_keys)=3, mixed_keys[1]='int one', mixed_keys['1']='string one'
'keys' in empty_dict=False, '__init__' in empty_dict=False  (methods are not keys)
list(insertion_ordered)=['b', 10, 'a', 2]  (JS would list 2 and 10 first)
missing key: KeyError


**`types.SimpleNamespace`** gives JS-style attribute access (`config.port`) over a
plain `__dict__`. It supports `==`, a readable `repr` and adding attributes later; a
missing attribute raises `AttributeError`. *New in 3.13:* it also accepts a mapping or
an iterable of pairs as a positional argument, handy for parsed JSON.

In [239]:
app_config = types.SimpleNamespace(debug=False, port=8000)
app_config.port = 8080
app_config.log_level = "INFO"
print(f"{app_config=}")
print(f"{app_config.port=}, {vars(app_config)=}")
print(f"missing attribute: {raised_error_name(lambda: app_config.timeout)}")
if sys.version_info >= (3, 13):
    parsed_config = types.SimpleNamespace(json.loads('{"host": "localhost"}'), port=5432)
    print(f"{parsed_config=}")
else:
    print("SimpleNamespace(mapping) needs Python 3.13+; skipped on this kernel")

app_config=namespace(debug=False, port=8080, log_level='INFO')
app_config.port=8080, vars(app_config)={'debug': False, 'port': 8080, 'log_level': 'INFO'}
missing attribute: AttributeError
parsed_config=namespace(host='localhost', port=5432)


### 1.6.5 Immutable mappings

**`frozendict`** (*New in 3.15*, PEP 814) is a built-in immutable dict. It is
hashable when all its values are, so it can be a dict key, a set member or a safe
module-level constant. It supports the read-only dict API (`[]`, `get`, `keys`,
`values`, `items`, `copy`, `fromkeys`, `in`, `len`, `==`), and `|` returns a **new**
`frozendict`. It compares equal to a `dict` with the same pairs.

The cell is guarded by a version check (convention 5).

In [240]:
def try_assign(mapping: Any, key: str, value: object) -> None:
    """Run mapping[key] = value (assignment cannot go in a lambda)."""
    mapping[key] = value


if sys.version_info >= (3, 15):
    GST_RATES = frozendict(food=5, electronics=18, luxury=28)
    print(f"{GST_RATES=}, {GST_RATES['food']=}")
    print(f"item assignment: {raised_error_name(lambda: try_assign(GST_RATES, 'food', 0))}")
    rates_with_services = GST_RATES | {"services": 18}
    print(f"{type(rates_with_services).__name__=}, {len(rates_with_services)=}, {len(GST_RATES)=}")
    rate_table_years = {GST_RATES: "FY2026"}  # hashable, so usable as a key
    print(f"{rate_table_years[frozendict(luxury=28, food=5, electronics=18)]=}")
    print(f"{GST_RATES == {'food': 5, 'electronics': 18, 'luxury': 28}=}")
    print(f"unhashable value: {raised_error_name(lambda: hash(frozendict(tags=['veg'])))}")
else:
    print("frozendict is new in Python 3.15; this kernel is older, so the demo is skipped")

GST_RATES=frozendict({'food': 5, 'electronics': 18, 'luxury': 28}), GST_RATES['food']=5
item assignment: TypeError
type(rates_with_services).__name__='frozendict', len(rates_with_services)=4, len(GST_RATES)=3
rate_table_years[frozendict(luxury=28, food=5, electronics=18)]='FY2026'
GST_RATES == {'food': 5, 'electronics': 18, 'luxury': 28}=True
unhashable value: TypeError


**`types.MappingProxyType`** wraps an existing dict in a read-only **live view**: no
writes through the proxy, but changes to the underlying dict show through. It is how
CPython exposes class namespaces (`SomeClass.__dict__`), and how a module can hand out
its settings without letting callers change them. It is not a snapshot and is hashable
only if the wrapped mapping is (a dict is not).

| | `dict` | `frozendict` | `MappingProxyType` |
| --- | --- | --- | --- |
| Writable | yes | no | no (through the proxy) |
| Sees later changes | — | no (it owns its data) | yes (a live view) |
| Hashable | no | yes, if the values are | no (wraps a dict) |
| Available | always | 3.15+ | 3.3+ |

In [241]:
_settings_store = {"currency": "INR", "tax_rate": 18}
SETTINGS_VIEW = types.MappingProxyType(_settings_store)
print(f"{SETTINGS_VIEW['currency']=}")
write_result = raised_error_name(lambda: try_assign(SETTINGS_VIEW, "currency", "USD"))
print(f"write through the proxy: {write_result}")
_settings_store["tax_rate"] = 12
print(f"{SETTINGS_VIEW['tax_rate']=}  (live: sees the change)")
print(f"hash(SETTINGS_VIEW): {raised_error_name(lambda: hash(SETTINGS_VIEW))}")
print(f"{type(int.__dict__).__name__=}  (class namespaces are exposed this way)")

SETTINGS_VIEW['currency']='INR'
write through the proxy: TypeError
SETTINGS_VIEW['tax_rate']=12  (live: sees the change)
hash(SETTINGS_VIEW): TypeError
type(int.__dict__).__name__='mappingproxy'  (class namespaces are exposed this way)


### 1.6.6 Sets (`set`) and frozen sets (`frozenset`)

A `set` is a hash table of keys with no values: same hashing rules as dict keys,
O(1) average `in`, no duplicates, no order guarantee. `{}` is an empty **dict**
(dicts came first), so an empty set is written `set()`.

In [242]:
empty_braces = {}
empty_set = set()
print(f"{type(empty_braces).__name__=}, {type(empty_set).__name__=}")
visited_cities = {"Pune", "Delhi", "Pune", "Goa"}  # noqa: B033
print(f"{sorted(visited_cities)=}, {len(visited_cities)=}")
print(f"{sorted(set('banana'))=}  (a str is an iterable of characters)")
print(f"list as an element: {raised_error_name(lambda: {['Pune']})}")

type(empty_braces).__name__='dict', type(empty_set).__name__='set'
sorted(visited_cities)=['Delhi', 'Goa', 'Pune'], len(visited_cities)=3
sorted(set('banana'))=['a', 'b', 'n']  (a str is an iterable of characters)
list as an element: TypeError


**Adding and removing.** `add(x)` adds one element; `update(*iterables)` adds every
element of each iterable. `remove(x)` raises `KeyError` if `x` is missing, while
`discard(x)` does nothing. `pop()` removes and returns an **arbitrary** element, so
the cell does not print which one. `clear()` empties the set.

In [243]:
user_roles = {"viewer"}
user_roles.add("editor")
user_roles.add("viewer")  # already present: no change
user_roles.update(["admin", "auditor"], ("editor",))
print(f"{sorted(user_roles)=}")
user_roles.discard("guest")  # absent: silently ignored
print(f"remove absent: {raised_error_name(lambda: user_roles.remove('guest'))}")
user_roles.remove("auditor")
popped_role = user_roles.pop()
print(f"{popped_role in {'viewer', 'editor', 'admin'}=}, {len(user_roles)=}")
user_roles.clear()
print(f"{user_roles=}")

sorted(user_roles)=['admin', 'auditor', 'editor', 'viewer']
remove absent: KeyError
popped_role in {'viewer', 'editor', 'admin'}=True, len(user_roles)=2
user_roles=set()


**Operators vs methods.** `|` union, `&` intersection, `-` difference, `^` symmetric
difference. The named methods do the same, but accept **any iterables**, and several
at once; the operators require sets (or frozensets) on both sides.

In [244]:
python_developers = {"asha", "ravi", "meena"}
javascript_developers = {"ravi", "john", "meena"}
print(f"{sorted(python_developers | javascript_developers)=}")
print(f"{sorted(python_developers & javascript_developers)=}")
print(f"{sorted(python_developers - javascript_developers)=}")
print(f"{sorted(python_developers ^ javascript_developers)=}")
print(f"{sorted(python_developers.union(['zoe'], ('kim',)))=}")
print(f"{sorted(python_developers.intersection(['ravi', 'kim']))=}")
print(f"set | list: {raised_error_name(lambda: python_developers | ['zoe'])}")

sorted(python_developers | javascript_developers)=['asha', 'john', 'meena', 'ravi']
sorted(python_developers & javascript_developers)=['meena', 'ravi']
sorted(python_developers - javascript_developers)=['asha']
sorted(python_developers ^ javascript_developers)=['asha', 'john']
sorted(python_developers.union(['zoe'], ('kim',)))=['asha', 'kim', 'meena', 'ravi', 'zoe']
sorted(python_developers.intersection(['ravi', 'kim']))=['ravi']
set | list: TypeError


**Relations.** `issubset` (`<=`), `issuperset` (`>=`) and `isdisjoint` (no common
elements). `<` and `>` test *proper* subsets (subset and not equal).

In [245]:
required_permissions = {"read", "write"}
granted_permissions = {"read", "write", "delete"}
print(f"{required_permissions.issubset(granted_permissions)=}")
print(f"{granted_permissions.issuperset(required_permissions)=}")
print(f"{required_permissions < granted_permissions=}  (proper subset)")
print(f"{required_permissions < required_permissions=}  (equal is not proper)")
print(f"{required_permissions.isdisjoint({'admin', 'billing'})=}")

required_permissions.issubset(granted_permissions)=True
granted_permissions.issuperset(required_permissions)=True
required_permissions < granted_permissions=True  (proper subset)
required_permissions < required_permissions=False  (equal is not proper)
required_permissions.isdisjoint({'admin', 'billing'})=True


**In-place variants.** `|=`, `&=`, `-=`, `^=` (and the methods `update`,
`intersection_update`, `difference_update`, `symmetric_difference_update`) change the
set itself instead of building a new one, so every alias sees the change.

In [246]:
active_users = {"asha", "ravi"}
same_set_alias = active_users
active_users |= {"meena"}
active_users -= {"ravi"}
active_users &= {"asha", "meena", "john"}
active_users ^= {"john"}
print(f"{sorted(active_users)=}, {active_users is same_set_alias=}")
active_users = active_users | {"kim"}  # plain | rebinds to a new set
print(f"{active_users is same_set_alias=}")

sorted(active_users)=['asha', 'john', 'meena'], active_users is same_set_alias=True
active_users is same_set_alias=False


**Set comprehensions** use braces with a single expression (no `key: value`).

In [247]:
raw_emails = ["Asha@Example.com", "ravi@example.com ", "asha@example.com"]
unique_domains = {email.strip().lower().split("@")[1] for email in raw_emails}
unique_emails = {email.strip().lower() for email in raw_emails}
print(f"{unique_domains=}")
print(f"{sorted(unique_emails)=}")

unique_domains={'example.com'}
sorted(unique_emails)=['asha@example.com', 'ravi@example.com']


**`frozenset`** is the immutable, hashable set. Use it where a set must be a dict key
or a set member: here, an undirected route where `{"Pune", "Mumbai"}` and
`{"Mumbai", "Pune"}` are the same key.

In [248]:
route_distances_km = {
    frozenset({"Pune", "Mumbai"}): 150,
    frozenset({"Delhi", "Agra"}): 230,
}
print(f"{route_distances_km[frozenset({'Mumbai', 'Pune'})]=}  (order does not matter)")
set_as_key_result = raised_error_name(lambda: {set(["Pune"]): 1})
print(f"plain set as a key: {set_as_key_result}")

route_distances_km[frozenset({'Mumbai', 'Pune'})]=150  (order does not matter)
plain set as a key: TypeError


### 1.6.7 The `collections` module

Five specialised containers. 1.6.11 lists every method again with one demo each.

#### `deque`: double-ended queue

A `deque` (say "deck") is a doubly linked list of fixed-size blocks: O(1) appends and
pops at **both** ends. `extendleft` adds items one at a time to the front, so their
order flips.

In [249]:
print_queue = deque(["report.pdf"])
print_queue.append("invoice.pdf")
print_queue.appendleft("urgent.pdf")
print(f"{print_queue=}")
print(f"{print_queue.popleft()=}, {print_queue.pop()=}, {print_queue=}")
print_queue.extendleft(["b.pdf", "a.pdf"])
print(f"{print_queue=}  (extendleft reversed the order)")

print_queue=deque(['urgent.pdf', 'report.pdf', 'invoice.pdf'])
print_queue.popleft()='urgent.pdf', print_queue.pop()='invoice.pdf', print_queue=deque(['report.pdf'])
print_queue=deque(['a.pdf', 'b.pdf', 'report.pdf'])  (extendleft reversed the order)


`rotate(n)` moves `n` items from the right end to the left (negative `n` goes the other way): a round-robin.

In [250]:
server_pool = deque(["server-a", "server-b", "server-c"])
server_pool.rotate(1)
print(f"after rotate(1):  {list(server_pool)}")
server_pool.rotate(-2)
print(f"after rotate(-2): {list(server_pool)}")

after rotate(1):  ['server-c', 'server-a', 'server-b']
after rotate(-2): ['server-b', 'server-c', 'server-a']


`maxlen` makes a **ring buffer**: once full, each append drops an item from the other
end. Ideal for "the last N readings".

In [251]:
recent_temperatures: deque[float] = deque(maxlen=3)
for reading in [21.5, 22.0, 22.4, 23.1, 22.8]:
    recent_temperatures.append(reading)
    print(f"after {reading}: {list(recent_temperatures)}")
print(f"{recent_temperatures.maxlen=}")

after 21.5: [21.5]
after 22.0: [21.5, 22.0]
after 22.4: [21.5, 22.0, 22.4]
after 23.1: [22.0, 22.4, 23.1]
after 22.8: [22.4, 23.1, 22.8]
recent_temperatures.maxlen=3


**Benchmark: `deque.popleft()` vs `list.pop(0)`.** Draining a queue from the front is
O(n) per pop on a list (every remaining pointer shifts left), so the whole drain is
O(n²); on a deque it is O(1) per pop. Tripling the size should roughly triple the
deque time and multiply the list time by about nine (timings vary by machine).

In [252]:
def drain_list_from_front(item_count: int) -> None:
    """Empty a list by popping index 0."""
    waiting_jobs = list(range(item_count))
    while waiting_jobs:
        waiting_jobs.pop(0)


def drain_deque_from_front(item_count: int) -> None:
    """Empty a deque with popleft()."""
    waiting_jobs = deque(range(item_count))
    while waiting_jobs:
        waiting_jobs.popleft()


def seconds_for(drain: Callable[[int], None], item_count: int) -> float:
    """Time one call of drain(item_count)."""
    return timeit.timeit(lambda: drain(item_count), number=1)


for item_count in (10_000, 30_000):
    list_ms = seconds_for(drain_list_from_front, item_count) * 1e3
    deque_ms = seconds_for(drain_deque_from_front, item_count) * 1e3
    print(f"{item_count:>6} items: list.pop(0) {list_ms:8.2f} ms, popleft() {deque_ms:6.2f} ms")

 10000 items: list.pop(0)     5.22 ms, popleft()   0.23 ms


 30000 items: list.pop(0)    54.86 ms, popleft()   0.80 ms


#### `Counter`: a dict of counts

`Counter(iterable)` counts hashable items; it is a `dict` subclass mapping item →
count. `most_common(n)` returns the top `n` pairs (ties keep first-seen order).
`update` **adds** counts (unlike `dict.update`, which replaces), `subtract` removes
them (counts may go to zero or below), `total()` sums the counts, and `elements()`
repeats each item by its count.

In [253]:
dish_orders = ["dosa", "idli", "dosa", "vada", "dosa", "idli"]
dish_counts = Counter(dish_orders)
print(f"{dish_counts=}")
print(f"{dish_counts.most_common(2)=}")
dish_counts.update(["vada", "vada"])
dish_counts.update({"dosa": 2})
dish_counts.subtract({"idli": 3})
print(f"{dish_counts=}  (idli went negative)")
print(f"{dish_counts.total()=}")
print(f"{list(Counter(dosa=2, vada=1).elements())=}")

dish_counts=Counter({'dosa': 3, 'idli': 2, 'vada': 1})
dish_counts.most_common(2)=[('dosa', 3), ('idli', 2)]
dish_counts=Counter({'dosa': 5, 'vada': 3, 'idli': -1})  (idli went negative)
dish_counts.total()=7
list(Counter(dosa=2, vada=1).elements())=['dosa', 'dosa', 'vada']


**Counter arithmetic.** `+` adds, `-` subtracts and **drops** zero and negative counts,
`&` keeps the minimum of each count, `|` the maximum.

In [254]:
morning_sales = Counter(dosa=5, idli=3, vada=1)
evening_sales = Counter(dosa=2, vada=4)
print(f"{morning_sales + evening_sales=}")
print(f"{morning_sales - evening_sales=}  (vada 1 - 4 dropped)")
print(f"{morning_sales & evening_sales=}  (minimum)")
print(f"{morning_sales | evening_sales=}  (maximum)")

morning_sales + evening_sales=Counter({'dosa': 7, 'vada': 5, 'idli': 3})
morning_sales - evening_sales=Counter({'dosa': 3, 'idli': 3})  (vada 1 - 4 dropped)
morning_sales & evening_sales=Counter({'dosa': 2, 'vada': 1})  (minimum)
morning_sales | evening_sales=Counter({'dosa': 5, 'vada': 4, 'idli': 3})  (maximum)


#### `defaultdict`: a dict that creates missing values

`defaultdict(factory)` calls `factory()` to create a value whenever a **missing key is
read** with `[]`, stores it, and returns it. The factory is any zero-argument
callable (`list`, `int`, `set`, a lambda) and is stored in `default_factory`. Reading a
key you did not mean to create still inserts it, which can surprise you.

In [255]:
dishes_by_customer: defaultdict[str, list[str]] = defaultdict(list)
for customer_name, dish in [("asha", "dosa"), ("ravi", "idli"), ("asha", "vada")]:
    dishes_by_customer[customer_name].append(dish)
print(f"{dict(dishes_by_customer)=}")
print(f"{dishes_by_customer.default_factory=}")
print(f"{dishes_by_customer['zoe']=}  (a read created the key)")
print(f"{'zoe' in dishes_by_customer=}")

dict(dishes_by_customer)={'asha': ['dosa', 'vada'], 'ravi': ['idli']}
dishes_by_customer.default_factory=<class 'list'>
dishes_by_customer['zoe']=[]  (a read created the key)
'zoe' in dishes_by_customer=True


**Nested defaultdicts.** The factory can itself build a defaultdict, giving two-level
counting without any "does the inner dict exist yet?" checks. A function that returns
`defaultdict(itself)` gives an auto-vivifying tree of any depth.

In [256]:
sales_by_city_and_dish: defaultdict[str, defaultdict[str, int]] = defaultdict(
    lambda: defaultdict(int)
)
sales_by_city_and_dish["Pune"]["dosa"] += 3
sales_by_city_and_dish["Pune"]["idli"] += 1
sales_by_city_and_dish["Goa"]["dosa"] += 2
print(json.dumps(sales_by_city_and_dish, sort_keys=True))


def auto_tree() -> defaultdict[str, Any]:
    """Return a defaultdict whose missing values are more auto_trees."""
    return defaultdict(auto_tree)


site_map = auto_tree()
site_map["shop"]["cart"]["checkout"] = "checkout.html"
print(json.dumps(site_map))

{"Goa": {"dosa": 2}, "Pune": {"dosa": 3, "idli": 1}}
{"shop": {"cart": {"checkout": "checkout.html"}}}


**The `__missing__` hook.** `dict.__getitem__` calls `self.__missing__(key)` when a
key is absent, if the subclass defines it; `defaultdict` is just a dict with a
`__missing__` that calls the factory and stores the result. Writing your own lets you
choose: this one returns `0` **without** inserting. `get()` and `in` never call
`__missing__`.

In [257]:
class ZeroDefaultDict(dict[str, int]):
    """A dict that answers 0 for missing keys without storing them."""

    def __missing__(self, key: str) -> int:
        print(f"  __missing__({key!r}) called")
        return 0


stock_counts = ZeroDefaultDict(rice=5)
print(f"{stock_counts['rice']=}")
print(f"{stock_counts['ghee']=}")
print(f"{stock_counts.get('ghee')=}  (get does not call __missing__)")
print(f"{'ghee' in stock_counts=}  (nothing was inserted)")

stock_counts['rice']=5
  __missing__('ghee') called
stock_counts['ghee']=0
stock_counts.get('ghee')=None  (get does not call __missing__)
'ghee' in stock_counts=False  (nothing was inserted)


#### `OrderedDict`: order you can rearrange

Plain dicts have kept insertion order since 3.7, so `OrderedDict` is now for the
things a dict cannot do: `move_to_end(key, last=True)` (or to the front with
`last=False`), `popitem(last=False)` to remove the **oldest** item, and **order-sensitive**
equality between two `OrderedDict`s. That makes it the classic base for an LRU cache
(Lab 1.6).

In [258]:
recently_viewed = OrderedDict.fromkeys(["home", "cart", "checkout"])
recently_viewed.move_to_end("home")
print(f"{list(recently_viewed)=}")
recently_viewed.move_to_end("checkout", last=False)
print(f"{list(recently_viewed)=}")
print(f"{recently_viewed.popitem(last=False)=}  (oldest first)")
print(f"{OrderedDict(a=1, b=2) == OrderedDict(b=2, a=1)=}")
print(f"{dict(a=1, b=2) == dict(b=2, a=1)=}")
print(f"{OrderedDict(a=1, b=2) == {'b': 2, 'a': 1}=}  (vs a dict: order ignored)")

list(recently_viewed)=['cart', 'checkout', 'home']
list(recently_viewed)=['checkout', 'cart', 'home']
recently_viewed.popitem(last=False)=('checkout', None)  (oldest first)
OrderedDict(a=1, b=2) == OrderedDict(b=2, a=1)=False
dict(a=1, b=2) == dict(b=2, a=1)=True
OrderedDict(a=1, b=2) == {'b': 2, 'a': 1}=True  (vs a dict: order ignored)


#### `ChainMap`: layered lookups

`ChainMap(first, second, ...)` searches several mappings in order without copying
them; the first mapping that has the key wins. Typical use: configuration layers such
as command line > environment > defaults. Writes and deletes affect only the first
mapping.

In [259]:
default_settings = {"host": "localhost", "port": 8000, "debug": False}
environment_settings = {"port": 9000}
command_line_settings = {"debug": True}
effective_settings = ChainMap(command_line_settings, environment_settings, default_settings)
print(f"{effective_settings['port']=}, {effective_settings['debug']=}")
print(f"{effective_settings['host']=}  (only the defaults have it)")
print(f"{dict(effective_settings)=}")
effective_settings["port"] = 7000
print(f"{command_line_settings=}, {environment_settings=}")

effective_settings['port']=9000, effective_settings['debug']=True
effective_settings['host']='localhost'  (only the defaults have it)
dict(effective_settings)={'host': 'localhost', 'port': 9000, 'debug': True}
command_line_settings={'debug': True, 'port': 7000}, environment_settings={'port': 9000}


### 1.6.8 Choosing a container

| Container | Ordered | Mutable | Duplicates | Hashable | Lookup | Reach for it when |
| --- | --- | --- | --- | --- | --- | --- |
| `list` | yes | yes | yes | no | by index O(1), by value O(n) | an ordered, growing sequence |
| `tuple` | yes | no | yes | if items are | by index O(1) | a fixed record or a composite key |
| `NamedTuple` | yes | no | yes | if fields are | by name or index | a small immutable record with field names |
| `dict` | insertion | yes | unique keys | no | by key O(1) | mapping keys to values |
| `set` | no | yes | no | no | `in` O(1) | membership, deduplication, set algebra |
| `frozenset` | no | no | no | yes | `in` O(1) | a set used as a key or set member |
| `deque` | yes | yes | yes | no | ends O(1), middle O(n) | queues, sliding windows, last N items |
| `Counter` | insertion | yes | counts | no | by key O(1) | counting and tallies |
| `defaultdict` | insertion | yes | unique keys | no | by key O(1) | grouping, building nested data |
| `frozendict` | insertion | no | unique keys | if values are | by key O(1) | constant lookup tables, mapping-valued keys (3.15+) |
| `array.array` | yes | yes | yes | no | by index O(1) | large homogeneous numeric data in compact memory |

#### Under the hood: shallow sizes of the same 1 000 ints

`sys.getsizeof` counts the container only (pointer arrays, hash tables), not the
`int` objects, except for `array`, which stores raw machine values. Sizes vary by
version and platform.

In [260]:
sample_numbers = list(range(1000))
same_numbers_in = {
    "list": sample_numbers,
    "tuple": tuple(sample_numbers),
    "set": set(sample_numbers),
    "frozenset": frozenset(sample_numbers),
    "dict (value None)": dict.fromkeys(sample_numbers),
    "deque": deque(sample_numbers),
    "array('q')": array.array("q", sample_numbers),
}
for container_name, container in same_numbers_in.items():
    print(f"{container_name:<18}{sys.getsizeof(container):>8,} bytes")

list                 8,056 bytes
tuple                8,048 bytes
set                 32,984 bytes
frozenset           32,984 bytes
dict (value None)   36,952 bytes
deque                8,680 bytes
array('q')           8,080 bytes


### 1.6.9 Method reference — `dict`

Average costs; every row gets a demo cell below.

| Method / operation | What it does | Returns | Cost |
| --- | --- | --- | --- |
| `d[key]` | read a value (`KeyError` if missing) | value | O(1) |
| `d[key] = value` | insert or replace | — | O(1) |
| `del d[key]` | delete (`KeyError` if missing) | — | O(1) |
| `key in d`, `len(d)`, `iter(d)` | membership on keys, size, iterate keys | `bool` / `int` / iterator | O(1) / O(1) / O(n) |
| `get(key, default=None)` | read without raising | value or default | O(1) |
| `setdefault(key, default=None)` | read, inserting `default` first if missing | value | O(1) |
| `pop(key[, default])` | remove and return (`KeyError` without default) | value | O(1) |
| `popitem()` | remove and return the last inserted pair | `(key, value)` | O(1) |
| `update(other=(), **kwargs)` | merge a mapping, pairs or keywords in place | `None` | O(k) |
| `d1 \| d2`, `d1 \|= d2` | merge into a new dict / in place (right wins) | new `dict` / same | O(n + m) |
| `keys()`, `values()`, `items()` | live views (keys/items support `&`, `\|`, `-`, `^`) | view | O(1) |
| `copy()` | shallow copy | new `dict` | O(n) |
| `clear()` | remove everything | `None` | O(n) |
| `dict.fromkeys(iterable, value=None)` | new dict, every key → the **same** value object | new `dict` | O(n) |
| `reversed(d)` | iterate keys newest-first | iterator | O(n) |
| `==` | equal if same key/value pairs (order ignored) | `bool` | O(n) |

#### `dict`

**`d[key]`**: read a value; a missing key raises `KeyError`.

In [261]:
price_list = {"dosa": 60, "idli": 40}
print(f"{price_list['dosa']=}")
print(f"missing key: {raised_error_name(lambda: price_list['vada'])}")

price_list['dosa']=60
missing key: KeyError


**`d[key] = value`**: insert a new key or replace the value of an existing one.

In [262]:
price_list = {"dosa": 60}
price_list["idli"] = 40  # insert
price_list["dosa"] = 65  # replace, position unchanged
print(f"{price_list=}")

price_list={'dosa': 65, 'idli': 40}


**`del d[key]`**: delete a key; `KeyError` if it is missing (`delete_key` is from 1.6.3).

In [263]:
price_list = {"dosa": 60, "idli": 40}
del price_list["idli"]
print(f"{price_list=}")
print(f"del missing: {raised_error_name(lambda: delete_key(price_list, 'idli'))}")

price_list={'dosa': 60}
del missing: KeyError


**`key in d`, `len(d)`, `iter(d)`**: membership tests keys; iterating a dict yields its keys in insertion order.

In [264]:
price_list = {"dosa": 60, "idli": 40, "vada": 30}
print(f"{'idli' in price_list=}, {40 in price_list=}, {len(price_list)=}")
key_iterator = iter(price_list)
print(f"{next(key_iterator)=}, {list(key_iterator)=}")

'idli' in price_list=True, 40 in price_list=False, len(price_list)=3
next(key_iterator)='dosa', list(key_iterator)=['idli', 'vada']


**`get(key, default=None)`**: read without raising.

In [265]:
price_list = {"dosa": 60}
print(f"{price_list.get('dosa')=}, {price_list.get('vada')=}, {price_list.get('vada', 0)=}")

price_list.get('dosa')=60, price_list.get('vada')=None, price_list.get('vada', 0)=0


**`setdefault(key, default=None)`**: return the value, inserting `default` first if the key is missing.

In [266]:
price_list = {"dosa": 60}
print(f"{price_list.setdefault('dosa', 99)=}, {price_list.setdefault('vada', 30)=}")
print(f"{price_list.setdefault('upma')=}, {price_list=}")

price_list.setdefault('dosa', 99)=60, price_list.setdefault('vada', 30)=30
price_list.setdefault('upma')=None, price_list={'dosa': 60, 'vada': 30, 'upma': None}


**`pop(key[, default])`**: remove and return a value; without a default a missing key raises `KeyError`.

In [267]:
price_list = {"dosa": 60, "idli": 40}
print(f"{price_list.pop('idli')=}, {price_list.pop('idli', None)=}, {price_list=}")
print(f"pop missing, no default: {raised_error_name(lambda: price_list.pop('idli'))}")

price_list.pop('idli')=40, price_list.pop('idli', None)=None, price_list={'dosa': 60}
pop missing, no default: KeyError


**`popitem()`**: remove and return the most recently inserted pair (LIFO); `KeyError` when empty.

In [268]:
price_list = {"dosa": 60, "idli": 40, "vada": 30}
print(f"{price_list.popitem()=}, {price_list.popitem()=}, {price_list=}")
print(f"popitem on empty: {raised_error_name({}.popitem)}")

price_list.popitem()=('vada', 30), price_list.popitem()=('idli', 40), price_list={'dosa': 60}
popitem on empty: KeyError


**`update(other=(), **kwargs)`**: merge in place from a mapping, an iterable of pairs and/or keywords; returns `None`.

In [269]:
price_list = {"dosa": 60}
update_result = price_list.update({"idli": 40}, vada=30)
price_list.update([("upma", 45)])
print(f"{price_list=}, {update_result=}")

price_list={'dosa': 60, 'idli': 40, 'vada': 30, 'upma': 45}, update_result=None

**`d1 | d2`, `d1 |= d2`**: merge into a new dict, or into `d1` in place; the right side wins on shared keys.

In [270]:
base_prices = {"dosa": 60, "idli": 40}
festival_prices = {"dosa": 50}
merged_prices = base_prices | festival_prices
print(f"{merged_prices=}, {base_prices=}")
prices_alias = base_prices
base_prices |= festival_prices
print(f"{base_prices=}, {base_prices is prices_alias=}")

merged_prices={'dosa': 50, 'idli': 40}, base_prices={'dosa': 60, 'idli': 40}
base_prices={'dosa': 50, 'idli': 40}, base_prices is prices_alias=True


**`keys()`, `values()`, `items()`**: live views; key and item views support `&`, `|`, `-`, `^`.

In [271]:
price_list = {"dosa": 60, "idli": 40}
price_items = price_list.items()
price_list["vada"] = 30
print(f"{price_items=}  (includes vada: the view is live)")
print(f"{list(price_list.values())=}")
print(f"{sorted(price_list.keys() - {'idli'})=}, {price_items & {('vada', 30)}=}")

price_items=dict_items([('dosa', 60), ('idli', 40), ('vada', 30)])  (includes vada: the view is live)
list(price_list.values())=[60, 40, 30]
sorted(price_list.keys() - {'idli'})=['dosa', 'vada'], price_items & {('vada', 30)}={('vada', 30)}


**`copy()`**: a shallow copy, a new dict holding the same value objects.

In [272]:
price_list = {"dosa": 60, "combo": ["dosa", "chai"]}
price_copy = price_list.copy()
price_copy["dosa"] = 70
print(f"{price_list['dosa']=}, {price_copy['dosa']=}")
print(f"{price_copy['combo'] is price_list['combo']=}  (shallow)")

price_list['dosa']=60, price_copy['dosa']=70
price_copy['combo'] is price_list['combo']=True  (shallow)


**`clear()`**: remove every item in place; aliases see the empty dict.

In [273]:
price_list = {"dosa": 60, "idli": 40}
price_alias = price_list
print(f"{price_list.clear()=}, {price_alias=}")

price_list.clear()=None, price_alias={}


**`dict.fromkeys(iterable, value=None)`**: a new dict where every key maps to the **same** value object.

In [274]:
stock_flags = dict.fromkeys(["dosa", "idli", "vada"], False)
shared_lists = dict.fromkeys(["T1", "T2"], [])
print(f"{stock_flags=}")
print(f"{shared_lists['T1'] is shared_lists['T2']=}  (one list, see 1.6.3)")

stock_flags={'dosa': False, 'idli': False, 'vada': False}
shared_lists['T1'] is shared_lists['T2']=True  (one list, see 1.6.3)


**`reversed(d)`**: iterate keys newest-first (since 3.8); works on the views too.

In [275]:
price_list = {"dosa": 60, "idli": 40, "vada": 30}
print(f"{list(reversed(price_list))=}")
print(f"{list(reversed(price_list.items()))=}")

list(reversed(price_list))=['vada', 'idli', 'dosa']
list(reversed(price_list.items()))=[('vada', 30), ('idli', 40), ('dosa', 60)]


**`==`**: dicts are equal when they hold the same key/value pairs; order is ignored.

In [276]:
print(f"{ {'dosa': 60, 'idli': 40} == {'idli': 40, 'dosa': 60}=}")
print(f"{ {'dosa': 60} == {'dosa': 60.0}=}  (60 == 60.0)")
print(f"{ {'dosa': 60} == {'dosa': 65}=}")

 {'dosa': 60, 'idli': 40} == {'idli': 40, 'dosa': 60}=True
 {'dosa': 60} == {'dosa': 60.0}=True  (60 == 60.0)
 {'dosa': 60} == {'dosa': 65}=False


### 1.6.10 Method reference — `set` and `frozenset`

| Method | Operator | What it does | Mutates? |
| --- | --- | --- | --- |
| `add(x)` | — | add one element | yes |
| `remove(x)` / `discard(x)` | — | delete (`remove` raises `KeyError`, `discard` does not) | yes |
| `pop()` | — | remove and return an arbitrary element | yes |
| `clear()`, `copy()` | — | empty / shallow copy | yes / no |
| `union(*others)` | `a \| b` | elements in either | no |
| `intersection(*others)` | `a & b` | elements in both | no |
| `difference(*others)` | `a - b` | elements only in `a` | no |
| `symmetric_difference(other)` | `a ^ b` | elements in exactly one | no |
| `update`, `intersection_update`, `difference_update`, `symmetric_difference_update` | `\|=`, `&=`, `-=`, `^=` | in-place versions | yes |
| `issubset(other)` / `issuperset(other)` | `<=` / `>=` (`<`, `>` proper) | subset tests | no |
| `isdisjoint(other)` | — | no common elements | no |

Methods accept any iterable; operators require sets on both sides. `frozenset` has only the non-mutating methods and is hashable. `x in s` is O(1) on average.

The demos use small integer ticket numbers: small ints hash to themselves, so these
sets print in the same order on every run.

#### `set`

**`add(x)`**: add one element; adding an existing element changes nothing.

In [277]:
open_tickets = {101, 102}
open_tickets.add(103)
open_tickets.add(101)
print(f"{open_tickets=}")

open_tickets={101, 102, 103}


**`remove(x)` / `discard(x)`**: delete an element; `remove` raises `KeyError` if it is missing, `discard` does not.

In [278]:
open_tickets = {101, 102, 103}
open_tickets.remove(101)
open_tickets.discard(999)
print(f"{open_tickets=}")
print(f"remove missing: {raised_error_name(lambda: open_tickets.remove(999))}")

open_tickets={102, 103}
remove missing: KeyError


**`pop()`**: remove and return an arbitrary element (`KeyError` on an empty set). Do not rely on which one.

In [279]:
open_tickets = {101, 102, 103}
closed_ticket = open_tickets.pop()
print(f"{closed_ticket in {101, 102, 103}=}, {len(open_tickets)=}")
print(f"pop on empty: {raised_error_name(set().pop)}")

closed_ticket in {101, 102, 103}=True, len(open_tickets)=2
pop on empty: KeyError


**`clear()`, `copy()`**: empty the set in place / make a shallow copy.

In [280]:
open_tickets = {101, 102}
ticket_snapshot = open_tickets.copy()
open_tickets.clear()
print(f"{open_tickets=}, {ticket_snapshot=}")

open_tickets=set(), ticket_snapshot={101, 102}


**`union(*others)` / `a | b`**: elements in either.

In [281]:
team_a_tickets = {101, 102, 103}
team_b_tickets = {103, 104}
print(f"{team_a_tickets | team_b_tickets=}")
print(f"{team_a_tickets.union(team_b_tickets, [105])=}")

team_a_tickets | team_b_tickets={101, 102, 103, 104}
team_a_tickets.union(team_b_tickets, [105])={101, 102, 103, 104, 105}


**`intersection(*others)` / `a & b`**: elements in both.

In [282]:
team_a_tickets = {101, 102, 103}
team_b_tickets = {103, 104, 101}
print(f"{team_a_tickets & team_b_tickets=}")
print(f"{team_a_tickets.intersection(team_b_tickets, [101, 999])=}")

team_a_tickets & team_b_tickets={101, 103}
team_a_tickets.intersection(team_b_tickets, [101, 999])={101}


**`difference(*others)` / `a - b`**: elements only in `a`.

In [283]:
team_a_tickets = {101, 102, 103}
team_b_tickets = {103, 104}
print(f"{team_a_tickets - team_b_tickets=}")
print(f"{team_a_tickets.difference(team_b_tickets, [102])=}")

team_a_tickets - team_b_tickets={101, 102}
team_a_tickets.difference(team_b_tickets, [102])={101}


**`symmetric_difference(other)` / `a ^ b`**: elements in exactly one of the two.

In [284]:
team_a_tickets = {101, 102, 103}
team_b_tickets = {103, 104}
print(f"{team_a_tickets ^ team_b_tickets=}")
print(f"{team_a_tickets.symmetric_difference([103, 104])=}")

team_a_tickets ^ team_b_tickets={104, 101, 102}
team_a_tickets.symmetric_difference([103, 104])={104, 101, 102}


**`update`, `intersection_update`, `difference_update`, `symmetric_difference_update` / `|=`, `&=`, `-=`, `^=`**: the in-place versions; they return `None` and change the set itself.

In [285]:
ticket_queue = {101, 102}
ticket_queue.update([103, 104])  # |=
print(f"after update:                      {ticket_queue}")
ticket_queue.intersection_update({102, 103, 104, 105})  # &=
print(f"after intersection_update:         {ticket_queue}")
ticket_queue.difference_update([104])  # -=
print(f"after difference_update:           {ticket_queue}")
ticket_queue.symmetric_difference_update({103, 106})  # ^=
print(f"after symmetric_difference_update: {ticket_queue}")
ticket_queue |= {110}
ticket_queue &= {102, 106, 110}
ticket_queue -= {106}
ticket_queue ^= {111}
print(f"after |= &= -= ^=:                 {ticket_queue}")

after update:                      {104, 101, 102, 103}
after intersection_update:         {104, 102, 103}
after difference_update:           {102, 103}
after symmetric_difference_update: {106, 102}
after |= &= -= ^=:                 {102, 110, 111}


**`issubset(other)` / `issuperset(other)`, `<=` / `>=`, `<` / `>`**: subset tests; `<` and `>` require a *proper* subset or superset.

In [286]:
urgent_tickets = {101, 103}
open_tickets = {101, 102, 103}
print(f"{urgent_tickets.issubset(open_tickets)=}, {urgent_tickets <= open_tickets=}")
print(f"{open_tickets.issuperset([101])=}, {open_tickets >= urgent_tickets=}")
print(f"{urgent_tickets < open_tickets=}, {open_tickets < open_tickets=}")
print(f"{open_tickets > urgent_tickets=}")

urgent_tickets.issubset(open_tickets)=True, urgent_tickets <= open_tickets=True
open_tickets.issuperset([101])=True, open_tickets >= urgent_tickets=True
urgent_tickets < open_tickets=True, open_tickets < open_tickets=False
open_tickets > urgent_tickets=True


**`isdisjoint(other)`**: `True` when the two share no element.

In [287]:
open_tickets = {101, 102}
print(f"{open_tickets.isdisjoint({201, 202})=}, {open_tickets.isdisjoint([102, 300])=}")

open_tickets.isdisjoint({201, 202})=True, open_tickets.isdisjoint([102, 300])=False


#### `frozenset`

The notes under the table, one demo each. **Methods accept any iterable; operators
require sets on both sides** (a `set` and a `frozenset` mix freely; the result takes
the type of the left operand).

In [288]:
open_tickets = {101, 102}
print(f"{open_tickets.union(range(103, 105))=}")
print(f"set | range: {raised_error_name(lambda: open_tickets | range(103, 105))}")
print(f"{type(frozenset({1}) | {2}).__name__=}, {type({2} | frozenset({1})).__name__=}")

open_tickets.union(range(103, 105))={104, 101, 102, 103}
set | range: TypeError
type(frozenset({1}) | {2}).__name__='frozenset', type({2} | frozenset({1})).__name__='set'


**`frozenset` has only the non-mutating methods and is hashable.** Mutators such as
`add` do not exist on it; it can be a dict key or an element of another set.

In [289]:
weekend_days = frozenset({"sat", "sun"})
print(f"{hasattr(weekend_days, 'add')=}, {hasattr(weekend_days, 'union')=}")
print(f"add: {raised_error_name(lambda: weekend_days.add('fri'))}")
print(f"{sorted(weekend_days | {'fri'})=}  (returns a new frozenset)")
schedules = {weekend_days: "relaxed", frozenset({"mon"}): "busy"}
print(f"{schedules[frozenset({'sun', 'sat'})]=}")
set_of_sets = {frozenset({1, 2}), frozenset({2, 1}), frozenset({3})}
print(f"{len(set_of_sets)=}")

hasattr(weekend_days, 'add')=False, hasattr(weekend_days, 'union')=True
add: AttributeError
sorted(weekend_days | {'fri'})=['fri', 'sat', 'sun']  (returns a new frozenset)
schedules[frozenset({'sun', 'sat'})]='relaxed'
len(set_of_sets)=2


**`x in s` is O(1) on average**: a hash and a probe, independent of the set size.
A list scans every element. Timings vary by machine.

In [290]:
ticket_numbers = list(range(100_000))
ticket_set = frozenset(ticket_numbers)
list_seconds = timeit.timeit(lambda: -1 in ticket_numbers, number=50)
set_seconds = timeit.timeit(lambda: -1 in ticket_set, number=50)
print(f"list: {list_seconds / 50 * 1e6:9.2f} µs per lookup")
print(f"set:  {set_seconds / 50 * 1e6:9.3f} µs per lookup")

list:    548.96 µs per lookup
set:      0.043 µs per lookup


### 1.6.11 Method reference — `collections`

| Type | Key methods and attributes |
| --- | --- |
| `deque` | `append`, `appendleft`, `pop`, `popleft`, `extend`, `extendleft`, `rotate(n)`, `insert`, `remove`, `count`, `index`, `reverse`, `clear`, `copy`, `maxlen` |
| `Counter` | `most_common(n)`, `elements()`, `update`, `subtract`, `total()`, `+ - & \|`; missing keys count as 0 |
| `defaultdict` | `default_factory`, the `__missing__` hook; all `dict` methods |
| `OrderedDict` | `move_to_end(key, last=True)`, `popitem(last=True)`, order-sensitive `==` |
| `ChainMap` | `maps`, `new_child()`, `parents`; writes go to the first mapping |
| `namedtuple` | see 1.5.10 |

Each row lists several APIs, so each API gets its own short demo below.

#### `deque`

**`append(x)`**: add to the right end, O(1).

In [291]:
call_queue = deque(["caller-1"])
call_queue.append("caller-2")
print(f"{call_queue=}")

call_queue=deque(['caller-1', 'caller-2'])


**`appendleft(x)`**: add to the left end, O(1).

In [292]:
call_queue = deque(["caller-1"])
call_queue.appendleft("vip-caller")
print(f"{call_queue=}")

call_queue=deque(['vip-caller', 'caller-1'])


**`pop()`**: remove and return from the right end; `IndexError` when empty.

In [293]:
call_queue = deque(["caller-1", "caller-2"])
print(f"{call_queue.pop()=}, {call_queue=}")
print(f"pop on empty: {raised_error_name(deque().pop)}")

call_queue.pop()='caller-2', call_queue=deque(['caller-1'])
pop on empty: IndexError


**`popleft()`**: remove and return from the left end, O(1); `IndexError` when empty.

In [294]:
call_queue = deque(["caller-1", "caller-2"])
print(f"{call_queue.popleft()=}, {call_queue=}")
print(f"popleft on empty: {raised_error_name(deque().popleft)}")

call_queue.popleft()='caller-1', call_queue=deque(['caller-2'])
popleft on empty: IndexError


**`extend(iterable)`**: append every item on the right, in order.

In [295]:
call_queue = deque(["caller-1"])
call_queue.extend(["caller-2", "caller-3"])
print(f"{call_queue=}")

call_queue=deque(['caller-1', 'caller-2', 'caller-3'])


**`extendleft(iterable)`**: appendleft every item, one at a time, so they end up reversed.

In [296]:
call_queue = deque(["caller-3"])
call_queue.extendleft(["caller-2", "caller-1"])
print(f"{call_queue=}")

call_queue=deque(['caller-1', 'caller-2', 'caller-3'])


**`rotate(n)`**: rotate `n` steps right (negative `n`: left).

In [297]:
shift_rota = deque(["asha", "ravi", "meena", "john"])
shift_rota.rotate(1)
print(f"{shift_rota=}")
shift_rota.rotate(-1)
print(f"{shift_rota=}")

shift_rota=deque(['john', 'asha', 'ravi', 'meena'])
shift_rota=deque(['asha', 'ravi', 'meena', 'john'])


**`insert(i, x)`**: insert before index `i` (O(n) towards the middle); `IndexError` on a full bounded deque.

In [298]:
call_queue = deque(["caller-1", "caller-3"])
call_queue.insert(1, "caller-2")
print(f"{call_queue=}")
bounded_queue = deque([1, 2], maxlen=2)
print(f"insert into full deque: {raised_error_name(lambda: bounded_queue.insert(0, 0))}")

call_queue=deque(['caller-1', 'caller-2', 'caller-3'])
insert into full deque: IndexError


**`remove(x)`**: delete the first item equal to `x`; `ValueError` if absent.

In [299]:
call_queue = deque(["caller-1", "spam", "caller-2", "spam"])
call_queue.remove("spam")
print(f"{call_queue=}")
print(f"remove missing: {raised_error_name(lambda: call_queue.remove('nobody'))}")

call_queue=deque(['caller-1', 'caller-2', 'spam'])
remove missing: ValueError


**`count(x)`**: how many items equal `x`.

In [300]:
call_queue = deque(["asha", "ravi", "asha"])
print(f"{call_queue.count('asha')=}, {call_queue.count('zoe')=}")

call_queue.count('asha')=2, call_queue.count('zoe')=0


**`index(x[, start[, stop]])`**: position of the first `x`; `ValueError` if absent.

In [301]:
call_queue = deque(["asha", "ravi", "asha"])
print(f"{call_queue.index('asha')=}, {call_queue.index('asha', 1)=}")
print(f"index missing: {raised_error_name(lambda: call_queue.index('zoe'))}")

call_queue.index('asha')=0, call_queue.index('asha', 1)=2
index missing: ValueError


**`reverse()`**: reverse in place; returns `None`.

In [302]:
call_queue = deque([1, 2, 3])
print(f"{call_queue.reverse()=}, {call_queue=}")

call_queue.reverse()=None, call_queue=deque([3, 2, 1])


**`clear()`**: remove every item.

In [303]:
call_queue = deque([1, 2, 3])
call_queue.clear()
print(f"{call_queue=}, {len(call_queue)=}")

call_queue=deque([]), len(call_queue)=0

**`copy()`**: shallow copy that keeps `maxlen`.

In [304]:
recent_calls = deque(["asha", "ravi"], maxlen=3)
calls_copy = recent_calls.copy()
calls_copy.append("meena")
print(f"{recent_calls=}, {calls_copy=}")

recent_calls=deque(['asha', 'ravi'], maxlen=3), calls_copy=deque(['asha', 'ravi', 'meena'], maxlen=3)


**`maxlen`**: read-only bound (`None` if unbounded); a full deque drops items from the opposite end.

In [305]:
recent_calls = deque(maxlen=2)
recent_calls.extend(["asha", "ravi", "meena"])
recent_calls.appendleft("john")  # drops from the right
print(f"{recent_calls=}, {recent_calls.maxlen=}, {deque().maxlen=}")

recent_calls=deque(['john', 'ravi'], maxlen=2), recent_calls.maxlen=2, deque().maxlen=None


#### `Counter`

**`most_common(n)`**: the `n` largest counts as `(item, count)` pairs; ties keep first-seen order. No argument returns all.

In [306]:
page_views = Counter(["/home", "/cart", "/home", "/pay", "/cart", "/home"])
print(f"{page_views.most_common(2)=}")
print(f"{page_views.most_common()=}")

page_views.most_common(2)=[('/home', 3), ('/cart', 2)]
page_views.most_common()=[('/home', 3), ('/cart', 2), ('/pay', 1)]


**`elements()`**: an iterator repeating each item by its count (items with count ≤ 0 are skipped).

In [307]:
ingredient_counts = Counter(egg=2, flour=1, salt=0)
print(f"{list(ingredient_counts.elements())=}")

list(ingredient_counts.elements())=['egg', 'egg', 'flour']


**`update(iterable_or_mapping)`**: **add** counts (`dict.update` would replace them).

In [308]:
page_views = Counter({"/home": 3})
page_views.update(["/home", "/cart"])
page_views.update({"/cart": 5})
print(f"{page_views=}")

page_views=Counter({'/cart': 6, '/home': 4})


**`subtract(iterable_or_mapping)`**: subtract counts in place; results may be zero or negative.

In [309]:
warehouse_stock = Counter(soap=5, tea=2)
warehouse_stock.subtract({"tea": 3, "soap": 1})
print(f"{warehouse_stock=}")

warehouse_stock=Counter({'soap': 4, 'tea': -1})


**`total()`**: the sum of all counts (since 3.10).

In [310]:
page_views = Counter({"/home": 3, "/cart": 2})
print(f"{page_views.total()=}, {sum(page_views.values())=}")

page_views.total()=5, sum(page_views.values())=5


**`+ - & |`**: add, subtract (dropping counts ≤ 0), minimum, maximum. Unary `+c` drops non-positive counts too.

In [311]:
shop_a = Counter(soap=3, tea=1)
shop_b = Counter(soap=1, tea=4, rice=2)
print(f"{shop_a + shop_b=}")
print(f"{shop_a - shop_b=}")
print(f"{shop_a & shop_b=}, {shop_a | shop_b=}")
print(f"{+Counter(soap=2, tea=-1, rice=0)=}")

shop_a + shop_b=Counter({'tea': 5, 'soap': 4, 'rice': 2})
shop_a - shop_b=Counter({'soap': 2})
shop_a & shop_b=Counter({'soap': 1, 'tea': 1}), shop_a | shop_b=Counter({'tea': 4, 'soap': 3, 'rice': 2})
+Counter(soap=2, tea=-1, rice=0)=Counter({'soap': 2})


**Missing keys count as 0**: reading one returns `0` and does **not** insert it (unlike `defaultdict`).

In [312]:
page_views = Counter({"/home": 3})
print(f"{page_views['/missing']=}, {'/missing' in page_views=}")
page_views["/new"] += 1
print(f"{page_views=}")

page_views['/missing']=0, '/missing' in page_views=False


page_views=Counter({'/home': 3, '/new': 1})


#### `defaultdict`

**`default_factory`**: the zero-argument callable used for missing keys; it can be
changed later, and `None` turns the defaultdict back into plain-dict behaviour.

In [313]:
tickets_by_agent = defaultdict(set)
tickets_by_agent["asha"].add(101)
print(f"{tickets_by_agent.default_factory=}, {dict(tickets_by_agent)=}")
tickets_by_agent.default_factory = None
print(f"missing key, factory None: {raised_error_name(lambda: tickets_by_agent['ravi'])}")

tickets_by_agent.default_factory=<class 'set'>, dict(tickets_by_agent)={'asha': {101}}


missing key, factory None: KeyError


**The `__missing__` hook**: `defaultdict.__missing__(key)` calls the factory, stores the value and returns it; `d[key]` calls it on a miss.

In [314]:
page_counts = defaultdict(int)
print(f"{page_counts.__missing__('/home')=}, {dict(page_counts)=}")
page_counts["/cart"] += 1  # __getitem__ miss → __missing__ → 0, then += 1
print(f"{dict(page_counts)=}")

page_counts.__missing__('/home')=0, dict(page_counts)={'/home': 0}
dict(page_counts)={'/home': 0, '/cart': 1}


**All `dict` methods**: it *is* a dict subclass. `get`, `in` and `setdefault` do not call the factory.

In [315]:
page_counts = defaultdict(int, {"/home": 3})
print(f"{isinstance(page_counts, dict)=}")
print(f"{page_counts.get('/cart')=}, {'/cart' in page_counts=}  (no insert)")
page_counts.update({"/pay": 1})
print(f"{sorted(page_counts.items())=}, {page_counts.pop('/pay')=}")

isinstance(page_counts, dict)=True
page_counts.get('/cart')=None, '/cart' in page_counts=False  (no insert)
sorted(page_counts.items())=[('/home', 3), ('/pay', 1)], page_counts.pop('/pay')=1


#### `OrderedDict`

**`move_to_end(key, last=True)`**: move a key to the end, or to the front with
`last=False`; `KeyError` if it is missing.

In [316]:
browser_tabs = OrderedDict.fromkeys(["mail", "docs", "news"])
browser_tabs.move_to_end("mail")
print(f"{list(browser_tabs)=}")
browser_tabs.move_to_end("news", last=False)
print(f"{list(browser_tabs)=}")
print(f"missing key: {raised_error_name(lambda: browser_tabs.move_to_end('chat'))}")

list(browser_tabs)=['docs', 'news', 'mail']
list(browser_tabs)=['news', 'docs', 'mail']
missing key: KeyError


**`popitem(last=True)`**: remove the newest pair, or the oldest with `last=False`.

In [317]:
browser_tabs = OrderedDict(mail=1, docs=2, news=3)
print(f"{browser_tabs.popitem()=}, {browser_tabs.popitem(last=False)=}, {browser_tabs=}")

browser_tabs.popitem()=('news', 3), browser_tabs.popitem(last=False)=('mail', 1), browser_tabs=OrderedDict({'docs': 2})


**Order-sensitive `==`**: between two `OrderedDict`s order matters; against a plain `dict` it does not.

In [318]:
first_order = OrderedDict(mail=1, docs=2)
second_order = OrderedDict(docs=2, mail=1)
print(f"{first_order == second_order=}, {first_order == dict(second_order)=}")

first_order == second_order=False, first_order == dict(second_order)=True


#### `ChainMap`

**`maps`**: the list of underlying mappings, searched front to back; it can be edited.

In [319]:
built_in_defaults = {"theme": "light", "font_size": 14}
user_overrides = {"theme": "dark"}
layered_settings = ChainMap(user_overrides, built_in_defaults)
print(f"{layered_settings.maps=}")
print(f"{layered_settings['theme']=}, {layered_settings['font_size']=}")

layered_settings.maps=[{'theme': 'dark'}, {'theme': 'light', 'font_size': 14}]
layered_settings['theme']='dark', layered_settings['font_size']=14


**`new_child(m=None)`**: a new ChainMap with an extra (empty or given) mapping in front, the others unchanged; handy for nested scopes.

In [320]:
built_in_defaults = {"theme": "light", "font_size": 14}
layered_settings = ChainMap(built_in_defaults)
session_settings = layered_settings.new_child({"font_size": 18})
print(f"{session_settings['font_size']=}, {layered_settings['font_size']=}")
print(f"{len(session_settings.maps)=}")

session_settings['font_size']=18, layered_settings['font_size']=14
len(session_settings.maps)=2


**`parents`**: a ChainMap of every mapping except the first (skip the innermost scope).

In [321]:
scope_chain = ChainMap({"x": "local"}, {"x": "enclosing"}, {"x": "global"})
print(f"{scope_chain['x']=}, {scope_chain.parents['x']=}")
print(f"{scope_chain.parents.parents['x']=}")

scope_chain['x']='local', scope_chain.parents['x']='enclosing'
scope_chain.parents.parents['x']='global'


**Writes go to the first mapping**: assignment, `del` and `pop` touch only `maps[0]`.

In [322]:
built_in_defaults = {"theme": "light"}
user_overrides: dict[str, str] = {}
layered_settings = ChainMap(user_overrides, built_in_defaults)
layered_settings["theme"] = "dark"
print(f"{user_overrides=}, {built_in_defaults=}")
del layered_settings["theme"]
print(f"{layered_settings['theme']=}  (the default shows through again)")
print(f"del a key only in a later map: {raised_error_name(lambda: layered_settings.pop('theme'))}")

user_overrides={'theme': 'dark'}, built_in_defaults={'theme': 'light'}
layered_settings['theme']='light'  (the default shows through again)
del a key only in a later map: KeyError


#### `namedtuple`

Covered in 1.5.6 and 1.5.10 (`_fields`, `_make`, `_asdict`, `_replace`, defaults).
One reminder cell:

In [323]:
Coordinates = namedtuple("Coordinates", ["latitude", "longitude"])
office_location = Coordinates(18.52, 73.86)
print(f"{Coordinates._fields=}, {office_location._replace(longitude=73.9)=}")

Coordinates._fields=('latitude', 'longitude'), office_location._replace(longitude=73.9)=Coordinates(latitude=18.52, longitude=73.9)


### 1.6.12 Common dictionary patterns

Ten patterns you will write every week. The first two reuse `LogEntry` from 1.5.6
(re-defined in this unit's setup), which the Phase 1 project builds on.

**Counting.** The manual `get(key, 0) + 1` loop and `Counter` give the same counts;
`Counter` also gives `most_common`.

In [324]:
access_log = [
    LogEntry("INFO", "GET /home"),
    LogEntry("ERROR", "db timeout", 500),
    LogEntry("INFO", "GET /cart"),
    LogEntry("WARNING", "slow query"),
    LogEntry("ERROR", "gateway timeout", 504),
    LogEntry("INFO", "GET /pay"),
]
level_counts_manual: dict[str, int] = {}
for log_entry in access_log:
    level_counts_manual[log_entry.level] = level_counts_manual.get(log_entry.level, 0) + 1
level_counts = Counter(log_entry.level for log_entry in access_log)
print(f"{level_counts_manual=}")
print(f"{level_counts.most_common(1)=}, {level_counts_manual == level_counts=}")

level_counts_manual={'INFO': 3, 'ERROR': 2, 'WARNING': 1}
level_counts.most_common(1)=[('INFO', 3)], level_counts_manual == level_counts=True


**Grouping.** `defaultdict(list)` and `setdefault(key, []).append(...)` both collect
items per key. Convert a defaultdict to a plain `dict` before returning it, so later
reads of missing keys raise instead of inserting.

In [325]:
messages_by_level: defaultdict[str, list[str]] = defaultdict(list)
for log_entry in access_log:
    messages_by_level[log_entry.level].append(log_entry.message)
status_codes_by_level: dict[str, list[int]] = {}
for log_entry in access_log:
    status_codes_by_level.setdefault(log_entry.level, []).append(log_entry.status_code)
print(f"{dict(messages_by_level)=}")
print(f"{status_codes_by_level=}")

dict(messages_by_level)={'INFO': ['GET /home', 'GET /cart', 'GET /pay'], 'ERROR': ['db timeout', 'gateway timeout'], 'WARNING': ['slow query']}
status_codes_by_level={'INFO': [200, 200, 200], 'ERROR': [500, 504], 'WARNING': [200]}


**Indexing records by ID.** One comprehension turns a list of rows into an O(1)
lookup. If two rows share an ID, the last one silently wins, so check the length.

In [326]:
customer_rows = [
    {"id": 7, "name": "Asha", "city": "Pune"},
    {"id": 3, "name": "Ravi", "city": "Goa"},
    {"id": 9, "name": "Meena", "city": "Pune"},
]
customers_by_id = {row["id"]: row for row in customer_rows}
print(f"{customers_by_id[3]['name']=}, {sorted(customers_by_id)=}")
print(f"{len(customers_by_id) == len(customer_rows)=}  (no duplicate IDs)")

customers_by_id[3]['name']='Ravi', sorted(customers_by_id)=[3, 7, 9]
len(customers_by_id) == len(customer_rows)=True  (no duplicate IDs)


**Inverting a mapping.** Swapping keys and values works when values are unique and
hashable. With duplicate values the naive inverse loses entries; group instead.

In [327]:
country_names = {"IN": "India", "NP": "Nepal", "LK": "Sri Lanka"}
country_codes = {name: code for code, name in country_names.items()}
print(f"{country_codes['Nepal']=}")
city_by_person = {"asha": "Pune", "ravi": "Goa", "meena": "Pune"}
naive_inverse = {city: person for person, city in city_by_person.items()}
print(f"{naive_inverse=}  (asha was overwritten)")
people_by_city: defaultdict[str, list[str]] = defaultdict(list)
for person, city in city_by_person.items():
    people_by_city[city].append(person)
print(f"{dict(people_by_city)=}")

country_codes['Nepal']='NP'
naive_inverse={'Pune': 'meena', 'Goa': 'ravi'}  (asha was overwritten)
dict(people_by_city)={'Pune': ['asha', 'meena'], 'Goa': ['ravi']}


**Sorting by value.** Sort `items()` with a key on the value, then rebuild a dict
(which keeps the sorted order). `max(d, key=d.get)` finds the key with the largest
value.

In [328]:
dish_ratings = {"dosa": 4.7, "idli": 4.2, "vada": 4.5}
ratings_high_to_low = dict(sorted(dish_ratings.items(), key=lambda pair: pair[1], reverse=True))
print(f"{ratings_high_to_low=}")
print(f"{max(dish_ratings, key=dish_ratings.__getitem__)=}")
print(f"{sorted(dish_ratings, key=dish_ratings.__getitem__)=}")

ratings_high_to_low={'dosa': 4.7, 'vada': 4.5, 'idli': 4.2}
max(dish_ratings, key=dish_ratings.__getitem__)='dosa'
sorted(dish_ratings, key=dish_ratings.__getitem__)=['idli', 'vada', 'dosa']


**Safe nested access.** `config.get("db", {}).get("port")` avoids `KeyError` at each
level. It fails when a key exists with the value `None` (the default is only used for
*missing* keys); `(config.get(key) or {})` covers that. For deeper paths, a small
helper is clearer.

In [329]:
service_config: dict[str, Any] = {"db": {"host": "localhost", "port": 5432}, "cache": None}
print(f"{service_config.get('db', {}).get('port')=}")
print(f"{service_config.get('queue', {}).get('url')=}")
none_value_result = raised_error_name(lambda: service_config.get("cache", {}).get("ttl"))
print(f"key present with None: {none_value_result}")
print(f"{(service_config.get('cache') or {}).get('ttl')=}")


def nested_get(mapping: Mapping[str, Any], *keys: str, default: object = None) -> object:
    """Follow keys through nested mappings; return default if any step is missing."""
    current_value: object = mapping
    for key in keys:
        if not isinstance(current_value, Mapping) or key not in current_value:
            return default
        current_value = current_value[key]
    return current_value


print(f"{nested_get(service_config, 'db', 'host')=}")
print(f"{nested_get(service_config, 'cache', 'ttl', default=60)=}")

service_config.get('db', {}).get('port')=5432
service_config.get('queue', {}).get('url')=None
key present with None: AttributeError
(service_config.get('cache') or {}).get('ttl')=None
nested_get(service_config, 'db', 'host')='localhost'
nested_get(service_config, 'cache', 'ttl', default=60)=60


**Merging with precedence.** Later sources win in both `{**a, **b}` and `a | b`, so
list the sources from lowest to highest priority. Both build a new dict; a `ChainMap`
(1.6.7) gives the same lookups without copying.

In [330]:
default_options = {"retries": 3, "timeout": 10, "region": "ap-south-1"}
team_options = {"timeout": 30}
request_options = {"retries": 5}
merged_by_unpacking = {**default_options, **team_options, **request_options}
merged_by_operator = default_options | team_options | request_options
print(f"{merged_by_unpacking=}")
print(f"{merged_by_unpacking == merged_by_operator=}")
print(f"{dict(ChainMap(request_options, team_options, default_options)) == merged_by_operator=}")

merged_by_unpacking={'retries': 5, 'timeout': 30, 'region': 'ap-south-1'}
merged_by_unpacking == merged_by_operator=True
dict(ChainMap(request_options, team_options, default_options)) == merged_by_operator=True


**Filtering with dict comprehensions.** Keep the pairs that pass a test, or keep only
an allow-list of keys (for example before returning a record from an API).

In [331]:
stock_levels = {"rice": 12, "dal": 0, "ghee": 2, "salt": 0}
in_stock = {item: quantity for item, quantity in stock_levels.items() if quantity > 0}
print(f"{in_stock=}")
user_record = {"name": "Asha", "email": "asha@example.in", "password_hash": "x9f3"}
public_field_names = {"name", "email"}
public_record = {key: value for key, value in user_record.items() if key in public_field_names}
print(f"{public_record=}")

in_stock={'rice': 12, 'ghee': 2}
public_record={'name': 'Asha', 'email': 'asha@example.in'}


**Iterating with `items()`.** Unpack each pair in the `for` target; add `enumerate`
for a running number. Iterating the dict itself gives only keys.

In [332]:
menu_prices = {"dosa": 60, "idli": 40, "vada": 30}
for line_number, (dish, price) in enumerate(menu_prices.items(), start=1):
    print(f"{line_number}. {dish:<6}{price:>5}")
print(f"{sum(menu_prices.values())=}")

1. dosa     60
2. idli     40
3. vada     30
sum(menu_prices.values())=130


**Converting to and from JSON.** A dict of strings, numbers, booleans, `None`, lists
and dicts round-trips through `json`. Three traps: **non-string keys become strings**
(so the round trip is not equal), tuple keys and `Decimal` values raise `TypeError`,
and `True`/`None` become `true`/`null`.

In [333]:
order_payload = {
    "order_id": 1001,
    "items": [{"sku": "TEA", "qty": 2}],
    "paid": True,
    "coupon": None,
}
payload_text = json.dumps(order_payload, sort_keys=True)
print(payload_text)
print(f"{json.loads(payload_text) == order_payload=}")
int_key_dict = {1: "one", 2: "two"}
int_key_round_trip = json.loads(json.dumps(int_key_dict))
print(f"{int_key_round_trip=}, {int_key_round_trip == int_key_dict=}")
print(f"tuple key: {raised_error_name(lambda: json.dumps({(1, 2): 'x'}))}")
print(f"Decimal value: {raised_error_name(lambda: json.dumps({'rate': Decimal('1.5')}))}")
print(json.dumps({"rate": Decimal("1.5")}, default=str))

{"coupon": null, "items": [{"qty": 2, "sku": "TEA"}], "order_id": 1001, "paid": true}
json.loads(payload_text) == order_payload=True
int_key_round_trip={'1': 'one', '2': 'two'}, int_key_round_trip == int_key_dict=False
tuple key: TypeError
Decimal value: TypeError
{"rate": "1.5"}


### 1.6.13 Classic data structures with built-ins

Python has no `Stack`, `Queue` or `Graph` classes in the core language: the built-in
containers already do the job.

**Stack (LIFO): a `list` with `append` / `pop`.** Both work at the end, O(1). The
bracket checker is the textbook stack exercise.

In [334]:
def brackets_balanced(text: str) -> bool:
    """Return True if every (, [ and { in text is closed in the right order."""
    closing_to_opening = {")": "(", "]": "[", "}": "{"}
    open_brackets: list[str] = []
    for character in text:
        if character in "([{":
            open_brackets.append(character)
        elif character in closing_to_opening:
            if not open_brackets or open_brackets.pop() != closing_to_opening[character]:
                return False
    return not open_brackets


undo_stack = ["type 'a'", "type 'b'", "bold"]
print(f"{undo_stack[-1]=} (peek), {undo_stack.pop()=}, {undo_stack=}")
print(f"{brackets_balanced('{[()]}')=}, {brackets_balanced('([)]')=}, {brackets_balanced('((')=}")

undo_stack[-1]='bold' (peek), undo_stack.pop()='bold', undo_stack=["type 'a'", "type 'b'"]
brackets_balanced('{[()]}')=True, brackets_balanced('([)]')=False, brackets_balanced('((')=False


**FIFO queue: a `deque` with `append` / `popleft`**, both O(1).

In [335]:
support_queue: deque[str] = deque()
for ticket_title in ["login fails", "refund request", "typo on page"]:
    support_queue.append(ticket_title)
while support_queue:
    print(f"serving: {support_queue.popleft()}")

serving: login fails
serving: refund request
serving: typo on page


**Priority queue: `heapq` on a list.** `heapq` keeps a plain list in min-heap order:
`heappush` and `heappop` are O(log n), and `heap[0]` is always the smallest item.
Push tuples `(priority, sequence_number, task)`: the sequence number breaks ties in
arrival order and stops Python from ever comparing two tasks. `heapify` turns a list
into a heap in O(n); `nsmallest` / `nlargest` answer top-k questions.

In [336]:
pending_jobs: list[tuple[int, int, str]] = []
incoming_jobs = [(3, "send newsletter"), (1, "capture payment"), (2, "resize image"), (1, "refund")]
for sequence_number, (priority, job_name) in enumerate(incoming_jobs):
    heapq.heappush(pending_jobs, (priority, sequence_number, job_name))
print(f"{pending_jobs[0]=}  (smallest priority first)")
while pending_jobs:
    priority, _sequence_number, job_name = heapq.heappop(pending_jobs)
    print(f"priority {priority}: {job_name}")
response_times_ms = [120, 45, 300, 80, 95]
heapq.heapify(response_times_ms)
print(f"{response_times_ms[0]=}, {heapq.nlargest(2, [120, 45, 300, 80, 95])=}")

pending_jobs[0]=(1, 1, 'capture payment')  (smallest priority first)
priority 1: capture payment
priority 1: refund
priority 2: resize image
priority 3: send newsletter
response_times_ms[0]=45, heapq.nlargest(2, [120, 45, 300, 80, 95])=[300, 120]


*New in 3.14:* `heapq` gained public max-heap functions (`heapify_max`,
`heappush_max`, `heappop_max`, …), so a max-heap no longer needs negated numbers.

In [337]:
if sys.version_info >= (3, 14):
    bid_amounts = [120, 450, 300, 80]
    heapq.heapify_max(bid_amounts)
    heapq.heappush_max(bid_amounts, 500)
    print(f"{heapq.heappop_max(bid_amounts)=}, {heapq.heappop_max(bid_amounts)=}")
else:
    print("max-heap functions need Python 3.14+; push negated values on older versions")

heapq.heappop_max(bid_amounts)=500, heapq.heappop_max(bid_amounts)=450


**Graphs as `dict[str, list[str]]`, BFS with a `deque`, and a set of visited nodes.**
An adjacency list maps each node to its neighbours. Breadth-first search explores in
rings of increasing distance, so the first time it reaches the goal is along a
shortest path (fewest edges). The `visited` set stops it from queueing a node twice:
O(1) membership is the reason it is a set and not a list.

In [338]:
metro_lines: dict[str, list[str]] = {
    "Airport": ["Central", "Park"],
    "Central": ["Airport", "Market", "Stadium"],
    "Park": ["Airport", "Museum"],
    "Market": ["Central", "Harbour"],
    "Stadium": ["Central"],
    "Museum": ["Park", "Harbour"],
    "Harbour": ["Market", "Museum"],
}


def bfs_shortest_path(graph: dict[str, list[str]], start: str, goal: str) -> list[str] | None:
    """Return a path with the fewest edges from start to goal, or None if unreachable."""
    visited = {start}
    came_from: dict[str, str] = {}
    frontier = deque([start])
    while frontier:
        node = frontier.popleft()
        if node == goal:
            path = [node]
            while path[-1] != start:
                path.append(came_from[path[-1]])
            return path[::-1]
        for neighbour in graph.get(node, []):
            if neighbour not in visited:
                visited.add(neighbour)
                came_from[neighbour] = node
                frontier.append(neighbour)
    return None


print(f"{bfs_shortest_path(metro_lines, 'Airport', 'Harbour')=}")
print(f"{bfs_shortest_path(metro_lines, 'Stadium', 'Museum')=}")
print(f"{bfs_shortest_path(metro_lines, 'Airport', 'Moon')=}")

bfs_shortest_path(metro_lines, 'Airport', 'Harbour')=['Airport', 'Central', 'Market', 'Harbour']
bfs_shortest_path(metro_lines, 'Stadium', 'Museum')=['Stadium', 'Central', 'Airport', 'Park', 'Museum']
bfs_shortest_path(metro_lines, 'Airport', 'Moon')=None


**DFS with an explicit stack.** Depth-first search swaps the queue for a stack (a
`list`). Pushing neighbours in reverse makes the visit order match the recursive
version, without Python's recursion limit (Unit 1.10).

In [339]:
def dfs_order(graph: dict[str, list[str]], start: str) -> list[str]:
    """Return nodes in depth-first visit order, using a list as the stack."""
    visited: set[str] = set()
    visit_order: list[str] = []
    stack = [start]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        visit_order.append(node)
        for neighbour in reversed(graph.get(node, [])):
            if neighbour not in visited:
                stack.append(neighbour)
    return visit_order


print(f"{dfs_order(metro_lines, 'Airport')=}")
print(f"{len(dfs_order(metro_lines, 'Airport')) == len(metro_lines)=}  (all reachable)")

dfs_order(metro_lines, 'Airport')=['Airport', 'Central', 'Market', 'Harbour', 'Museum', 'Park', 'Stadium']
len(dfs_order(metro_lines, 'Airport')) == len(metro_lines)=True  (all reachable)


**A trie as nested dicts.** Each level maps one character to the next level;
a special key marks the end of a word. Prefix search walks down the prefix, then
collects every word below it. `setdefault` builds missing levels in one line.

In [340]:
END_OF_WORD = "$"


def trie_insert(trie: dict[str, Any], word: str) -> None:
    """Add word to a nested-dict trie."""
    node = trie
    for character in word:
        node = node.setdefault(character, {})
    node[END_OF_WORD] = True


def trie_words_with_prefix(trie: dict[str, Any], prefix: str) -> list[str]:
    """Return every stored word starting with prefix, sorted."""
    node = trie
    for character in prefix:
        if character not in node:
            return []
        node = node[character]
    found_words: list[str] = []
    pending = [(node, prefix)]
    while pending:
        current_node, letters = pending.pop()
        for key, child in current_node.items():
            if key == END_OF_WORD:
                found_words.append(letters)
            else:
                pending.append((child, letters + key))
    return sorted(found_words)


city_trie: dict[str, Any] = {}
for city_name in ["pune", "punjab", "puri", "patna", "goa"]:
    trie_insert(city_trie, city_name)
print(f"{trie_words_with_prefix(city_trie, 'pu')=}, {trie_words_with_prefix(city_trie, 'x')=}")
print(json.dumps(city_trie["g"]))

trie_words_with_prefix(city_trie, 'pu')=['pune', 'punjab', 'puri'], trie_words_with_prefix(city_trie, 'x')=[]
{"o": {"a": {"$": true}}}


**A singly linked list with a small node class** (a preview of Phase 2, where classes
are covered properly). Pushing to the front and inserting after a node you already
hold are O(1); finding the n-th item is O(n). In everyday Python a `list` or `deque`
is almost always the better choice; the point here is to see the pointer structure.
The annotation `LinkedListNode | None` inside the class works unquoted because
annotations are evaluated lazily since 3.14 (PEP 649).

In [341]:
class LinkedListNode:
    """One node of a singly linked list."""

    def __init__(self, value: str, next_node: LinkedListNode | None = None) -> None:
        self.value = value
        self.next_node = next_node


def linked_list_values(head: LinkedListNode | None) -> list[str]:
    """Walk the list from head and collect the values."""
    collected_values: list[str] = []
    current_node = head
    while current_node is not None:
        collected_values.append(current_node.value)
        current_node = current_node.next_node
    return collected_values


playlist_head = LinkedListNode("intro", LinkedListNode("verse", LinkedListNode("chorus")))
playlist_head = LinkedListNode("countdown", playlist_head)  # O(1) push to the front
verse_node = playlist_head.next_node.next_node
verse_node.next_node = LinkedListNode("bridge", verse_node.next_node)  # O(1) insert after
print(f"{linked_list_values(playlist_head)=}")

linked_list_values(playlist_head)=['countdown', 'intro', 'verse', 'bridge', 'chorus']


**Big-O summary** for every operation used in this section (average case; *n* items,
*k* items involved).

| Structure | Operation | Built-in used | Cost |
| --- | --- | --- | --- |
| Stack | push / pop / peek | `list.append` / `list.pop()` / `items[-1]` | O(1) amortized / O(1) / O(1) |
| FIFO queue | enqueue / dequeue | `deque.append` / `deque.popleft` | O(1) / O(1) |
| FIFO queue (wrong tool) | dequeue | `list.pop(0)` | O(n) |
| Priority queue | push / pop / peek | `heappush` / `heappop` / `heap[0]` | O(log n) / O(log n) / O(1) |
| Priority queue | build / top k | `heapify` / `nlargest(k, …)` | O(n) / O(n log k) |
| Visited set | add / membership | `set.add` / `in` | O(1) / O(1) |
| Graph | neighbours of a node | `graph[node]` | O(1) to find, O(degree) to scan |
| BFS / DFS | whole traversal | `deque` / `list` + `set` | O(V + E) |
| Trie | insert / prefix lookup | nested `dict` + `setdefault` | O(word length) / O(prefix length + matches) |
| Linked list | push front / insert after a held node | node objects | O(1) / O(1) |
| Linked list | find n-th item / search | walk `next_node` | O(n) |

### 1.6.14 Real-world use cases

**`dict` (1 of 2): parsed JSON payloads, lookup tables, caches.** A webhook body
parses into nested dicts; a constant dict maps country codes to names; a dict keyed by
the arguments caches an expensive call (`functools.cache`, Unit 1.12, does this for
you).

In [342]:
webhook_body = '{"event": "payment.captured", "payment": {"id": "pay_29", "amount": 49900}}'
webhook_payload = json.loads(webhook_body)
amount_in_rupees = Decimal(webhook_payload["payment"]["amount"]) / 100
print(f"{webhook_payload['event']=}, {amount_in_rupees=}")

COUNTRY_NAMES = {"IN": "India", "US": "United States", "JP": "Japan"}
print(f"{COUNTRY_NAMES.get('JP', 'unknown')=}, {COUNTRY_NAMES.get('XX', 'unknown')=}")

exchange_rate_cache: dict[tuple[str, str], Decimal] = {}


def exchange_rate(source_currency: str, target_currency: str) -> Decimal:
    """Return a rate, computing it only on the first request for each pair."""
    cache_key = (source_currency, target_currency)
    if cache_key not in exchange_rate_cache:
        print(f"  (slow lookup for {cache_key})")
        exchange_rate_cache[cache_key] = (
            Decimal("83.25") if cache_key == ("USD", "INR") else Decimal("1")
        )
    return exchange_rate_cache[cache_key]


print(f"{exchange_rate('USD', 'INR')=}")
print(f"{exchange_rate('USD', 'INR')=}  (served from the cache)")

webhook_payload['event']='payment.captured', amount_in_rupees=Decimal('499')
COUNTRY_NAMES.get('JP', 'unknown')='Japan', COUNTRY_NAMES.get('XX', 'unknown')='unknown'
  (slow lookup for ('USD', 'INR'))
exchange_rate('USD', 'INR')=Decimal('83.25')
exchange_rate('USD', 'INR')=Decimal('83.25')  (served from the cache)


**`dict` (2 of 2): config objects, routing/dispatch tables, records indexed by ID.**
A dispatch table maps a request to a handler function, replacing a long `if`/`elif`
chain; web frameworks (Phase 6) route this way.

In [343]:
server_config = {"debug": False, "workers": 4, "allowed_hosts": ["shop.example.in"]}
products_by_sku = {
    product["sku"]: product
    for product in [{"sku": "TEA", "price": 249}, {"sku": "DAL", "price": 162}]
}


def list_products(request_body: dict[str, Any]) -> str:
    """Handle GET /products."""
    return f"200 OK: {sorted(products_by_sku)}"


def create_product(request_body: dict[str, Any]) -> str:
    """Handle POST /products."""
    products_by_sku[request_body["sku"]] = request_body
    return f"201 Created: {request_body['sku']}"


ROUTES: dict[tuple[str, str], Callable[[dict[str, Any]], str]] = {
    ("GET", "/products"): list_products,
    ("POST", "/products"): create_product,
}


def dispatch(method: str, path: str, request_body: dict[str, Any]) -> str:
    """Look up the handler for (method, path) and call it."""
    handler = ROUTES.get((method, path))
    return handler(request_body) if handler else "404 Not Found"


print(dispatch("POST", "/products", {"sku": "GHEE", "price": 640}))
print(dispatch("GET", "/products", {}))
print(dispatch("DELETE", "/products", {}))
print(f"{server_config['workers']=}, {products_by_sku['GHEE']['price']=}")

201 Created: GHEE
200 OK: ['DAL', 'GHEE', 'TEA']
404 Not Found
server_config['workers']=4, products_by_sku['GHEE']['price']=640


**`set`: deduplicating emails, permission checks, tags, comparing datasets, visited
sets.** Normalise before deduplicating (as `normalize_email` did in 1.4.7). Comparing
two exports of user IDs is one subtraction each way.

In [344]:
signup_emails = ["Asha@Example.in", "ravi@example.in", " asha@example.in", "RAVI@example.in"]
unique_emails = {email.strip().lower() for email in signup_emails}
print(f"{sorted(unique_emails)=}")

ROLE_PERMISSIONS = {
    "viewer": {"read"},
    "editor": {"read", "write"},
    "admin": {"read", "write", "delete"},
}
print(f"{'delete' in ROLE_PERMISSIONS['editor']=}")
print(f"{ROLE_PERMISSIONS['viewer'] <= ROLE_PERMISSIONS['admin']=}")

post_tags = {"python", "dicts", "performance"}
followed_tags = {"python", "rust"}
print(f"{sorted(post_tags & followed_tags)=}  (why this post is in the feed)")

users_in_billing = {101, 102, 103, 105}
users_in_crm = {101, 103, 104}
print(f"{users_in_billing - users_in_crm=}, {users_in_crm - users_in_billing=}")
print("visited set: see bfs_shortest_path and dfs_order in 1.6.13")

sorted(unique_emails)=['asha@example.in', 'ravi@example.in']
'delete' in ROLE_PERMISSIONS['editor']=False
ROLE_PERMISSIONS['viewer'] <= ROLE_PERMISSIONS['admin']=True
sorted(post_tags & followed_tags)=['python']  (why this post is in the feed)
users_in_billing - users_in_crm={105, 102}, users_in_crm - users_in_billing={104}
visited set: see bfs_shortest_path and dfs_order in 1.6.13


**`frozenset` / `frozendict`: cache keys, sets of sets, immutable configuration.** A
search filter given as a dict can be turned into a cache key with
`frozenset(filters.items())` (or `frozendict(filters)` on 3.15), so the same filters in
any order hit the same entry.

In [345]:
search_cache: dict[frozenset[tuple[str, str]], list[str]] = {}
first_filters = {"city": "Pune", "cuisine": "south indian"}
search_cache[frozenset(first_filters.items())] = ["Dosa Corner", "Idli House"]
same_filters_reordered = {"cuisine": "south indian", "city": "Pune"}
print(f"{search_cache[frozenset(same_filters_reordered.items())]=}")

ingredient_combos = {frozenset({"rice", "dal"}), frozenset({"dal", "rice"}), frozenset({"rice"})}
print(f"{len(ingredient_combos)=}  (a set of sets; order inside each set ignored)")

if sys.version_info >= (3, 15):
    FEATURE_FLAGS = frozendict(new_checkout=True, dark_mode=False)
    reordered_flags = frozendict(dark_mode=False, new_checkout=True)
    print(f"{FEATURE_FLAGS['new_checkout']=}")
    print(f"usable as one cache key: {hash(FEATURE_FLAGS) == hash(reordered_flags)}")
else:
    print("frozendict needs Python 3.15; skipped")

search_cache[frozenset(same_filters_reordered.items())]=['Dosa Corner', 'Idli House']
len(ingredient_combos)=2  (a set of sets; order inside each set ignored)
FEATURE_FLAGS['new_checkout']=True
usable as one cache key: True


**`deque` (1 of 2): BFS queues, sliding windows, last N events.** `bfs_shortest_path`
in 1.6.13 already uses a deque as its queue. A `maxlen` deque gives a moving average
over the last readings and an "N most recent events" history.

In [346]:
print(f"{bfs_shortest_path(metro_lines, 'Park', 'Market')=}  (a deque inside)")

latency_window: deque[int] = deque(maxlen=3)
for latency_ms in [120, 80, 100, 400, 90]:
    latency_window.append(latency_ms)
    moving_average = sum(latency_window) / len(latency_window)
    print(f"latest {latency_ms:>3} ms → moving average {moving_average:6.1f}")

recent_events: deque[str] = deque(maxlen=2)
for event_name in ["login", "view cart", "add item", "checkout"]:
    recent_events.append(event_name)
print(f"{list(recent_events)=}")

bfs_shortest_path(metro_lines, 'Park', 'Market')=['Park', 'Airport', 'Central', 'Market']  (a deque inside)
latest 120 ms → moving average  120.0
latest  80 ms → moving average  100.0
latest 100 ms → moving average  100.0
latest 400 ms → moving average  193.3
latest  90 ms → moving average  196.7
list(recent_events)=['add item', 'checkout']


**`deque` (2 of 2): rate-limiter timestamps.** A sliding-window limiter keeps the
times of recent requests; it drops timestamps older than the window from the left and
allows a request if fewer than `limit` remain. The times here are fixed numbers, so
the output is deterministic.

In [347]:
def allow_request(
    request_times: deque[float], now: float, limit: int, window_seconds: float
) -> bool:
    """Allow at most limit requests in any window_seconds-long window."""
    while request_times and now - request_times[0] >= window_seconds:
        request_times.popleft()
    if len(request_times) < limit:
        request_times.append(now)
        return True
    return False


client_request_times: deque[float] = deque()
for request_time in [0.0, 0.2, 0.4, 0.6, 1.1, 1.3]:
    allowed = allow_request(client_request_times, request_time, limit=3, window_seconds=1.0)
    print(f"t={request_time:.1f}s: {'allowed' if allowed else 'rejected (429)'}")

t=0.0s: allowed
t=0.2s: allowed
t=0.4s: allowed
t=0.6s: rejected (429)
t=1.1s: allowed
t=1.3s: allowed


**`Counter`: word frequencies, page-view analytics, inventory tallies.**

In [348]:
review_text = "Great dosa. Great chutney! The dosa was crisp; the chutney was fresh."
word_frequencies = Counter(re.findall(r"[a-z]+", review_text.lower()))
print(f"{word_frequencies.most_common(3)=}")

request_paths = ["/home", "/product/7", "/home", "/cart", "/product/7", "/home"]
print(f"{Counter(request_paths).most_common(2)=}")

inventory = Counter(tea=40, rice=25)
inventory += Counter(tea=10, ghee=12)  # shipment received
inventory -= Counter(rice=25)  # sold out: the zero count is dropped
print(f"{inventory=}, {inventory.total()=}")

word_frequencies.most_common(3)=[('great', 2), ('dosa', 2), ('chutney', 2)]
Counter(request_paths).most_common(2)=[('/home', 3), ('/product/7', 2)]
inventory=Counter({'tea': 50, 'ghee': 12}), inventory.total()=62


**`defaultdict`: grouping and adjacency lists.** Grouping records by a field (here
invoices by month; grouping orders by customer is Lab 1.6, exercise 1) and building an
adjacency list from an edge list are the same three lines.

In [349]:
invoices = [("2026-09", 1200), ("2026-10", 800), ("2026-09", 450), ("2026-10", 300)]
amounts_by_month: defaultdict[str, list[int]] = defaultdict(list)
for month, amount in invoices:
    amounts_by_month[month].append(amount)
print(f"{ {month: sum(amounts) for month, amounts in amounts_by_month.items()}=}")

road_edges = [("Pune", "Mumbai"), ("Pune", "Nashik"), ("Mumbai", "Goa")]
road_graph: defaultdict[str, list[str]] = defaultdict(list)
for city_a, city_b in road_edges:
    road_graph[city_a].append(city_b)
    road_graph[city_b].append(city_a)
print(f"{dict(road_graph)=}")

 {month: sum(amounts) for month, amounts in amounts_by_month.items()}={'2026-09': 1650, '2026-10': 1100}
dict(road_graph)={'Pune': ['Mumbai', 'Nashik'], 'Mumbai': ['Pune', 'Goa'], 'Nashik': ['Pune'], 'Goa': ['Mumbai']}


**`OrderedDict`: LRU caches; `ChainMap`: CLI > environment > defaults.** An LRU cache
needs exactly two `OrderedDict` moves: `move_to_end` on every hit and
`popitem(last=False)` when full (Lab 1.6, exercise 4, wraps them in a class). For
configuration, `argparse` with `argument_default=argparse.SUPPRESS` leaves unset
options out of the result, so they fall through to the next layer of the ChainMap.

In [350]:
page_cache = OrderedDict([("/home", "<home>"), ("/cart", "<cart>")])
page_cache.move_to_end("/home")  # a hit on /home makes it most recent
page_cache["/pay"] = "<pay>"
if len(page_cache) > 2:
    print(f"evicted {page_cache.popitem(last=False)[0]}")
print(f"{list(page_cache)=}")

argument_parser = argparse.ArgumentParser(argument_default=argparse.SUPPRESS)
argument_parser.add_argument("--port", type=int)
argument_parser.add_argument("--debug", action="store_true")
cli_settings = vars(argument_parser.parse_args(["--debug"]))
fake_environment = {"APP_PORT": "9000", "HOME": "/home/asha"}
env_settings = {
    name.removeprefix("APP_").lower(): int(value)
    for name, value in fake_environment.items()
    if name.startswith("APP_")
}
settings = ChainMap(cli_settings, env_settings, {"port": 8000, "debug": False, "host": "0.0.0.0"})
print(f"{dict(settings)=}")

evicted /cart
list(page_cache)=['/home', '/pay']
dict(settings)={'port': 9000, 'debug': True, 'host': '0.0.0.0'}


### Lab 1.6

Five exercises on one small order dataset. For each, replace the
`raise NotImplementedError` in the stub, then run the check cell. Until then the
checks print FAIL; that is expected.

Run the next cell first: it defines the `Order` record, `SAMPLE_ORDERS` and
`PRODUCT_CATALOG`, a `frozendict` of SKU → unit price (a read-only
`MappingProxyType` on kernels older than 3.15). `SOAP-3PK` is deliberately missing
from the catalog.

In [351]:
class Order(NamedTuple):
    """One order line for the Lab 1.6 exercises."""

    order_id: int
    customer: str
    sku: str
    quantity: int


SAMPLE_ORDERS = [
    Order(1001, "asha", "TEA-500G", 2),
    Order(1002, "ravi", "RICE-5KG", 1),
    Order(1003, "asha", "GHEE-1L", 1),
    Order(1004, "meena", "TEA-500G", 3),
    Order(1005, "ravi", "TEA-500G", 1),
    Order(1006, "john", "SOAP-3PK", 4),
    Order(1007, "meena", "RICE-5KG", 2),
    Order(1008, "asha", "DAL-1KG", 2),
]
catalog_prices = {
    "TEA-500G": Decimal("249.00"),
    "RICE-5KG": Decimal("525.50"),
    "GHEE-1L": Decimal("640.00"),
    "DAL-1KG": Decimal("162.25"),
}
if sys.version_info >= (3, 15):
    PRODUCT_CATALOG: Mapping[str, Decimal] = frozendict(catalog_prices)
else:
    PRODUCT_CATALOG = types.MappingProxyType(catalog_prices)
print(f"{len(SAMPLE_ORDERS)=}, {type(PRODUCT_CATALOG).__name__=}")

len(SAMPLE_ORDERS)=8, type(PRODUCT_CATALOG).__name__='frozendict'


#### Exercise 1 — `group_orders_by_customer(orders)`

Group orders by customer with a `defaultdict(list)`. Return a **plain `dict`** (so a
missing customer raises `KeyError` instead of being created), with customers in the
order they first appear and each customer's orders in input order.

In [352]:
def group_orders_by_customer(orders: Iterable[Order]) -> dict[str, list[Order]]:
    """Return {customer: [their orders in input order]} as a plain dict."""
    raise NotImplementedError("Lab 1.6: implement group_orders_by_customer()")

The check cell compares order IDs per customer, the customer order, the return type,
and the missing-customer behaviour.

In [353]:
run_checks(
    [
        (
            "order ids per customer",
            lambda: {
                customer: [order.order_id for order in customer_orders]
                for customer, customer_orders in group_orders_by_customer(SAMPLE_ORDERS).items()
            },
            {
                "asha": [1001, 1003, 1008],
                "ravi": [1002, 1005],
                "meena": [1004, 1007],
                "john": [1006],
            },
        ),
        (
            "customers in first-seen order",
            lambda: list(group_orders_by_customer(SAMPLE_ORDERS)),
            ["asha", "ravi", "meena", "john"],
        ),
        (
            "returns a plain dict",
            lambda: type(group_orders_by_customer(SAMPLE_ORDERS)).__name__,
            "dict",
        ),
        (
            "missing customer raises",
            lambda: raised_error_name(lambda: group_orders_by_customer(SAMPLE_ORDERS)["zoe"]),
            "KeyError",
        ),
        (
            "accepts an iterator",
            lambda: list(group_orders_by_customer(iter(SAMPLE_ORDERS[:2]))),
            ["asha", "ravi"],
        ),
        ("no orders", lambda: group_orders_by_customer([]), {}),
    ]
)

FAIL  order ids per customer: raised NotImplementedError: Lab 1.6: implement group_orders_by_customer()
FAIL  customers in first-seen order: raised NotImplementedError: Lab 1.6: implement group_orders_by_customer()
FAIL  returns a plain dict: raised NotImplementedError: Lab 1.6: implement group_orders_by_customer()
FAIL  missing customer raises: expected 'KeyError', got 'NotImplementedError'
FAIL  accepts an iterator: raised NotImplementedError: Lab 1.6: implement group_orders_by_customer()
FAIL  no orders: raised NotImplementedError: Lab 1.6: implement group_orders_by_customer()
0/6 checks passed


#### Exercise 2 — `top_k_words(text, k)`

Return the `k` most frequent words as `(word, count)` pairs. Words are compared
**case-insensitively** (`casefold`) and **ignoring punctuation**: a word is a run of
letters or digits, `re.findall(r"[^\W_]+", ...)`. Sort by count descending, then
**ties alphabetically**. `k <= 0` gives `[]`.

> **Note:** `Counter.most_common` breaks ties by first appearance, not alphabetically,
> so it is not enough on its own.

In [354]:
def top_k_words(text: str, k: int) -> list[tuple[str, int]]:
    """Return the k most frequent words, ties broken alphabetically."""
    raise NotImplementedError("Lab 1.6: implement top_k_words()")

Run the checks.

In [355]:
nursery_text = "The cat and the hat. THE CAT sat; a bat sat!"
run_checks(
    [
        ("top 3", lambda: top_k_words(nursery_text, 3), [("the", 3), ("cat", 2), ("sat", 2)]),
        (
            "ties alphabetical",
            lambda: top_k_words(nursery_text, 5),
            [("the", 3), ("cat", 2), ("sat", 2), ("a", 1), ("and", 1)],
        ),
        ("k larger than vocabulary", lambda: len(top_k_words(nursery_text, 50)), 7),
        ("k = 0", lambda: top_k_words(nursery_text, 0), []),
        ("empty text", lambda: top_k_words("", 3), []),
        ("caseless and unicode", lambda: top_k_words("Café café CAFÉ, tea", 1), [("café", 3)]),
        ("underscore splits words", lambda: top_k_words("snake_case snake", 1), [("snake", 2)]),
    ]
)

FAIL  top 3: raised NotImplementedError: Lab 1.6: implement top_k_words()
FAIL  ties alphabetical: raised NotImplementedError: Lab 1.6: implement top_k_words()
FAIL  k larger than vocabulary: raised NotImplementedError: Lab 1.6: implement top_k_words()
FAIL  k = 0: raised NotImplementedError: Lab 1.6: implement top_k_words()
FAIL  empty text: raised NotImplementedError: Lab 1.6: implement top_k_words()
FAIL  caseless and unicode: raised NotImplementedError: Lab 1.6: implement top_k_words()
FAIL  underscore splits words: raised NotImplementedError: Lab 1.6: implement top_k_words()
0/7 checks passed


#### Exercise 3 — `merge_configs(defaults, env, cli)`

Return a `ChainMap` where command-line values beat environment values, which beat
defaults. A `None` value in `cli` means "flag not given" and must not hide the lower
layers. Writes to the result must not change any of the three input dicts (put a
fresh dict first in the chain).

In [356]:
def merge_configs(
    defaults: Mapping[str, object], env: Mapping[str, object], cli: Mapping[str, object]
) -> ChainMap[str, object]:
    """Layer cli > env > defaults; None in cli means "not given"."""
    raise NotImplementedError("Lab 1.6: implement merge_configs()")

The check cell defines `writes_leave_inputs_alone`, which writes through the result
and confirms the three inputs did not change.

In [357]:
default_config = {"host": "localhost", "port": 8000, "debug": False, "log_level": "INFO"}
env_config = {"port": 9000, "log_level": "WARNING"}
cli_config = {"debug": True, "port": None}


def writes_leave_inputs_alone() -> bool:
    """Write through merge_configs' result and check that no input dict changed."""
    inputs_before = (dict(default_config), dict(env_config), dict(cli_config))
    merged_config = merge_configs(default_config, env_config, cli_config)
    merged_config["host"] = "example.in"
    merged_config["timeout"] = 30
    return (default_config, env_config, cli_config) == inputs_before


run_checks(
    [
        (
            "returns a ChainMap",
            lambda: type(merge_configs(default_config, env_config, cli_config)).__name__,
            "ChainMap",
        ),
        (
            "precedence cli > env > defaults",
            lambda: dict(merge_configs(default_config, env_config, cli_config)),
            {"host": "localhost", "port": 9000, "debug": True, "log_level": "WARNING"},
        ),
        (
            "None in cli is ignored",
            lambda: merge_configs(default_config, env_config, cli_config)["port"],
            9000,
        ),
        ("writes leave inputs alone", writes_leave_inputs_alone, True),
        ("empty layers", lambda: dict(merge_configs({"a": 1}, {}, {})), {"a": 1}),
    ]
)

FAIL  returns a ChainMap: raised NotImplementedError: Lab 1.6: implement merge_configs()
FAIL  precedence cli > env > defaults: raised NotImplementedError: Lab 1.6: implement merge_configs()
FAIL  None in cli is ignored: raised NotImplementedError: Lab 1.6: implement merge_configs()
FAIL  writes leave inputs alone: raised NotImplementedError: Lab 1.6: implement merge_configs()
FAIL  empty layers: raised NotImplementedError: Lab 1.6: implement merge_configs()
0/5 checks passed


#### Exercise 4 — `LRUCache(capacity)`

A least-recently-used cache on an `OrderedDict`:

- `get(key, default=None)`: return the value and mark the key most recently used;
  return `default` on a miss.
- `put(key, value)`: insert or update, mark it most recently used, and if the cache
  is now over capacity evict the **least** recently used key.
- `len(cache)` and `key in cache` (membership does **not** change recency).
- `capacity < 1` raises `ValueError`.

In [358]:
class LRUCache:
    """A fixed-capacity least-recently-used cache built on OrderedDict."""

    def __init__(self, capacity: int) -> None:
        raise NotImplementedError("Lab 1.6: implement LRUCache.__init__()")

    def get(self, key: Hashable, default: object = None) -> object:
        """Return the cached value (marking it recent) or default."""
        raise NotImplementedError("Lab 1.6: implement LRUCache.get()")

    def put(self, key: Hashable, value: object) -> None:
        """Store value, evicting the least recently used key if over capacity."""
        raise NotImplementedError("Lab 1.6: implement LRUCache.put()")

    def __len__(self) -> int:
        raise NotImplementedError("Lab 1.6: implement LRUCache.__len__()")

    def __contains__(self, key: object) -> bool:
        raise NotImplementedError("Lab 1.6: implement LRUCache.__contains__()")

The check cell replays short sequences of operations and records what the cache
returns.

In [359]:
def lru_basic_trace() -> list[object]:
    """get refreshes 'a', so putting 'c' evicts 'b'."""
    cache = LRUCache(2)
    cache.put("a", 1)
    cache.put("b", 2)
    first_get = cache.get("a")
    cache.put("c", 3)
    return [first_get, cache.get("b"), cache.get("c"), len(cache), "a" in cache]


def lru_update_trace() -> list[object]:
    """Updating 'a' refreshes it, so putting 'c' evicts 'b'."""
    cache = LRUCache(2)
    cache.put("a", 1)
    cache.put("b", 2)
    cache.put("a", 10)
    cache.put("c", 3)
    return [cache.get("a"), "b" in cache, len(cache)]


def lru_contains_trace() -> bool:
    """Membership does not refresh 'a', so putting 'c' evicts it."""
    cache = LRUCache(2)
    cache.put("a", 1)
    cache.put("b", 2)
    _ = "a" in cache
    cache.put("c", 3)
    return "a" in cache


run_checks(
    [
        ("get refreshes recency", lru_basic_trace, [1, None, 3, 2, True]),
        ("put updates and refreshes", lru_update_trace, [10, False, 2]),
        ("in does not refresh", lru_contains_trace, False),
        ("get default on miss", lambda: LRUCache(1).get("x", "fallback"), "fallback"),
        ("capacity 0 raises", lambda: raised_error_name(lambda: LRUCache(0)), "ValueError"),
    ]
)

FAIL  get refreshes recency: raised NotImplementedError: Lab 1.6: implement LRUCache.__init__()
FAIL  put updates and refreshes: raised NotImplementedError: Lab 1.6: implement LRUCache.__init__()
FAIL  in does not refresh: raised NotImplementedError: Lab 1.6: implement LRUCache.__init__()
FAIL  get default on miss: raised NotImplementedError: Lab 1.6: implement LRUCache.__init__()
FAIL  capacity 0 raises: expected 'ValueError', got 'NotImplementedError'
0/5 checks passed


#### Exercise 5 — order-analytics pipeline

Write `analyze_orders(orders, catalog)` returning a dict with exactly these keys:

| Key | Value | Build it with |
| --- | --- | --- |
| `revenue_by_customer` | `dict[str, Decimal]`: quantity × catalog price, catalog SKUs only | `defaultdict(Decimal)` |
| `top_skus` | the 3 best-selling catalog SKUs by units, as `(sku, units)` pairs | `Counter.most_common(3)` |
| `unique_customer_count` | number of distinct customers (all orders) | `set` |
| `recent_order_ids` | IDs of the last 5 orders, oldest first, as a `list` | `deque(maxlen=5)` |
| `unknown_skus` | sorted list of SKUs missing from the catalog | `set`, then `sorted` |

Orders with an unknown SKU count towards customers and recent orders, but not towards
revenue or `top_skus`. Return plain `dict`/`list` values, not the helper containers.

In [360]:
def analyze_orders(orders: Iterable[Order], catalog: Mapping[str, Decimal]) -> dict[str, object]:
    """Summarise orders: revenue, top SKUs, customers, recent IDs, unknown SKUs."""
    raise NotImplementedError("Lab 1.6: implement analyze_orders()")

Run the checks.

In [361]:
run_checks(
    [
        (
            "revenue by customer",
            lambda: analyze_orders(SAMPLE_ORDERS, PRODUCT_CATALOG)["revenue_by_customer"],
            {"asha": Decimal("1462.50"), "ravi": Decimal("774.50"), "meena": Decimal("1798.00")},
        ),
        (
            "top 3 SKUs by units",
            lambda: analyze_orders(SAMPLE_ORDERS, PRODUCT_CATALOG)["top_skus"],
            [("TEA-500G", 6), ("RICE-5KG", 3), ("DAL-1KG", 2)],
        ),
        (
            "unique customers",
            lambda: analyze_orders(SAMPLE_ORDERS, PRODUCT_CATALOG)["unique_customer_count"],
            4,
        ),
        (
            "last 5 order ids",
            lambda: analyze_orders(SAMPLE_ORDERS, PRODUCT_CATALOG)["recent_order_ids"],
            [1004, 1005, 1006, 1007, 1008],
        ),
        (
            "unknown SKUs",
            lambda: analyze_orders(SAMPLE_ORDERS, PRODUCT_CATALOG)["unknown_skus"],
            ["SOAP-3PK"],
        ),
        (
            "revenue is a plain dict",
            lambda: (
                type(analyze_orders(SAMPLE_ORDERS, PRODUCT_CATALOG)["revenue_by_customer"]).__name__
            ),
            "dict",
        ),
        (
            "no orders",
            lambda: analyze_orders([], PRODUCT_CATALOG),
            {
                "revenue_by_customer": {},
                "top_skus": [],
                "unique_customer_count": 0,
                "recent_order_ids": [],
                "unknown_skus": [],
            },
        ),
    ]
)

FAIL  revenue by customer: raised NotImplementedError: Lab 1.6: implement analyze_orders()
FAIL  top 3 SKUs by units: raised NotImplementedError: Lab 1.6: implement analyze_orders()
FAIL  unique customers: raised NotImplementedError: Lab 1.6: implement analyze_orders()
FAIL  last 5 order ids: raised NotImplementedError: Lab 1.6: implement analyze_orders()
FAIL  unknown SKUs: raised NotImplementedError: Lab 1.6: implement analyze_orders()
FAIL  revenue is a plain dict: raised NotImplementedError: Lab 1.6: implement analyze_orders()
FAIL  no orders: raised NotImplementedError: Lab 1.6: implement analyze_orders()
0/7 checks passed


### Recap — Unit 1.6

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `dict` internals | Compact hash table: sparse indices + dense entries; insertion-ordered; O(1) average | `Map` |
| Probing and resizing | Open addressing with a perturbed probe; table kept ≤ 2/3 full, resized in O(n) rarely | — |
| Key-sharing dicts | Instances of one class share a key table (PEP 412) | hidden classes in V8 |
| Hashable | Stable `__hash__` that agrees with `__eq__`; mutable built-ins are unhashable | `Map` uses SameValueZero, no hashing contract |
| `1 == 1.0 == True` | One dict key: first key kept, last value wins | `obj[1]` and `obj["1"]` collide instead |
| String-hash randomization | Per-process SipHash key against hash flooding; `PYTHONHASHSEED` | — |
| Access | `d[k]` raises `KeyError`; `get`, `setdefault`, `in` (keys) | `obj[k]` gives `undefined` |
| Merge | `\|`, `\|=`, `{**a, **b}`, `update`; the right side wins | `{...a, ...b}` |
| Views | `keys`/`values`/`items` are live; keys and items are set-like | `Object.keys()` returns a snapshot array |
| `dict.fromkeys` | Every key shares one value object | `Array(n).fill([])` |
| Size change while iterating | `RuntimeError`; iterate over `list(d)` | silently allowed |
| `SimpleNamespace` | Attribute access over a dict | plain object |
| `frozendict` / `MappingProxyType` | Immutable and hashable (3.15) / read-only live view | `Object.freeze` (shallow, not a key) |
| `set` / `frozenset` | Hash set with `\| & - ^`; methods take any iterable; `frozenset` is hashable | `Set` (+ ES2025 methods) |
| `deque` | O(1) at both ends; `maxlen` ring buffer | `shift()` is O(n) |
| `Counter` | Counts, `most_common`, `total`, `+ - & \|`; missing keys are 0 | `reduce` into an object |
| `defaultdict` | Factory runs on a missing-key **read** (`__missing__`) | `(m[k] ??= [])` |
| `OrderedDict` | `move_to_end`, `popitem(last=False)`, order-sensitive `==` | `Map` delete + set |
| `ChainMap` | Layered lookups without copying; writes go to the first map | `Object.assign` (copies) |
| `heapq` | Min-heap on a list; `(priority, seq, item)` tuples; max-heap API in 3.14 | no built-in |
| JSON | Non-string keys become strings; tuples and `Decimal` need help | `JSON.stringify` |

**Further reading**

- [Mapping types — `dict`](https://docs.python.org/3/library/stdtypes.html#mapping-types-dict),
  [set types — `set`, `frozenset`](https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset),
  [`frozendict`](https://docs.python.org/3.15/library/stdtypes.html#frozendict)
- [`collections`](https://docs.python.org/3/library/collections.html),
  [`heapq`](https://docs.python.org/3/library/heapq.html),
  [`types.MappingProxyType` / `SimpleNamespace`](https://docs.python.org/3/library/types.html)
- [`object.__hash__`](https://docs.python.org/3/reference/datamodel.html#object.__hash__),
  [`PYTHONHASHSEED`](https://docs.python.org/3/using/cmdline.html#envvar-PYTHONHASHSEED),
  [`json`](https://docs.python.org/3/library/json.html)
- [PEP 412 — Key-sharing dictionary](https://peps.python.org/pep-0412/),
  [PEP 456 — Secure and interchangeable hash algorithm](https://peps.python.org/pep-0456/),
  [PEP 468 — Preserving keyword argument order](https://peps.python.org/pep-0468/)
- [PEP 584 — Add union operators to dict](https://peps.python.org/pep-0584/),
  [PEP 814 — Add frozendict built-in type](https://peps.python.org/pep-0814/),
  [PEP 649 — Deferred evaluation of annotations](https://peps.python.org/pep-0649/)
- [Time complexity of CPython operations](https://wiki.python.org/moin/TimeComplexity)

## Unit 1.7 — Copying: Shallow Copies, Deep Copies and Structured Clones

**Objectives**

- Tell an alias from a copy, and prove which one you have with `is`.
- Make shallow copies of lists, dicts and sets in every idiomatic way, and predict
  which inner objects they still share.
- Use `copy.deepcopy`, explain its memo dict, and customize copying with `__copy__`
  and `__deepcopy__`.
- Know what cannot be copied and what copying costs.
- Replace defensive copies with immutable snapshots (tuples, `frozenset`,
  `frozendict`) where you can.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `const b = a` | `b = a` | Same: both names point at one object (an alias) |
| `[...items]`, `items.slice()` | `items.copy()`, `items[:]`, `list(items)` | Shallow in both: nested objects are shared |
| `{...config}`, `Object.assign({}, config)` | `config.copy()`, `{**config}`, `dict(config)` | Shallow in both |
| `structuredClone(value)` | `copy.deepcopy(value)` | `deepcopy` keeps class instances and copies functions by reference; `structuredClone` drops prototypes and rejects functions |
| `JSON.parse(JSON.stringify(x))` | `json.loads(json.dumps(x))` | Wrong in both: Python loses tuples, sets, `Decimal`, datetimes and int keys |
| `Object.freeze(obj)` | `tuple`, `frozenset`, `frozendict` | Python's frozen types are real immutable types, not a flag on an object |

The next cell is this unit's setup.

In [362]:
import copy
import datetime as dt
import json
import socket
import sys
import threading
import timeit
import tracemalloc
import types
from collections.abc import Callable, Mapping
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from typing import Any, NamedTuple


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


scratch_dir = find_repo_root() / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
PAISE = Decimal("0.01")
print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.7.1 Assignment vs copy

Unit 1.2 showed that assignment **never copies**: `b = a` binds a second name to the
same object. Changes made through either name are visible through both, because there
is only one object. A copy is a **new** object, made by an explicit call.

The next cell compares an alias with a copy. Equal `id()` values mean the same object
(the cell compares ids instead of printing them, since the numbers change every run).

In [363]:
cart_items = ["tea", "rice"]
same_cart = cart_items  # alias: a second name, no new object
copied_cart = cart_items.copy()  # a new list holding the same elements

same_cart.append("ghee")
print(f"{cart_items=}")
print(f"{copied_cart=}")
print(f"{id(same_cart) == id(cart_items)=}, {same_cart is cart_items=}")
print(f"{id(copied_cart) == id(cart_items)=}, {copied_cart is cart_items=}")
print(f"{copied_cart == ['tea', 'rice']=}  (equal values, different object)")

cart_items=['tea', 'rice', 'ghee']
copied_cart=['tea', 'rice']
id(same_cart) == id(cart_items)=True, same_cart is cart_items=True
id(copied_cart) == id(cart_items)=False, copied_cart is cart_items=False
copied_cart == ['tea', 'rice']=True  (equal values, different object)


Passing an object to a function is also assignment (the parameter is a new name for
the caller's object, 1.2.3), so a function that mutates its argument changes the
caller's data. That is the most common reason to copy.

In [364]:
def add_free_gift(order_items: list[str]) -> list[str]:
    """Append a gift item and return the list (mutates the argument!)."""
    order_items.append("free-sample")
    return order_items


customer_order = ["tea"]
gifted_order = add_free_gift(customer_order)
print(f"{customer_order=}  (the caller's list changed)")
print(f"{gifted_order is customer_order=}")

customer_order = ["tea"]
gifted_copy = add_free_gift(customer_order.copy())
print(f"{customer_order=}, {gifted_copy=}  (copy passed in, caller untouched)")

customer_order=['tea', 'free-sample']  (the caller's list changed)
gifted_order is customer_order=True
customer_order=['tea'], gifted_copy=['tea', 'free-sample']  (copy passed in, caller untouched)


### 1.7.2 Shallow copies

A **shallow copy** builds a new outer container and fills it with references to the
**same** element objects. Every built-in container has several spellings:

| Container | Ways to shallow-copy |
| --- | --- |
| `list` | `items.copy()`, `items[:]`, `list(items)`, `copy.copy(items)` |
| `dict` | `config.copy()`, `{**config}`, `dict(config)`, `copy.copy(config)` |
| `set` | `tags.copy()`, `set(tags)`, `copy.copy(tags)` |

Each of the next cells shows one spelling, and checks that the copy is a new object
(`is not`) whose elements are the same objects (`is`).

**`list.copy()`**: the explicit method, the clearest choice for lists.

In [365]:
price_rows = [[60, 40], [30, 45]]
rows_copy = price_rows.copy()
print(f"{rows_copy == price_rows=}, {rows_copy is price_rows=}")
print(f"{rows_copy[0] is price_rows[0]=}  (inner lists are shared)")

rows_copy == price_rows=True, rows_copy is price_rows=False
rows_copy[0] is price_rows[0]=True  (inner lists are shared)


**`[:]`**: a full slice; every slice of a list is a new list (Unit 1.5). Common in older code.

In [366]:
price_rows = [[60, 40], [30, 45]]
rows_copy = price_rows[:]
print(f"{rows_copy is price_rows=}, {rows_copy[1] is price_rows[1]=}")

rows_copy is price_rows=False, rows_copy[1] is price_rows[1]=True


**`list(x)`**: the constructor copies **any** iterable into a new list (a tuple, a set, a generator).

In [367]:
price_rows = [[60, 40], [30, 45]]
rows_copy = list(price_rows)
rows_from_tuple = list(((60, 40), (30, 45)))
print(f"{rows_copy is price_rows=}, {rows_copy[0] is price_rows[0]=}")
print(f"{rows_from_tuple=}  (outer type converted, inner tuples shared)")

rows_copy is price_rows=False, rows_copy[0] is price_rows[0]=True
rows_from_tuple=[(60, 40), (30, 45)]  (outer type converted, inner tuples shared)


**`dict.copy()`**: a new dict with the same key and value objects.

In [368]:
stock_levels = {"tea": [5, 3], "rice": [9]}
stock_copy = stock_levels.copy()
print(f"{stock_copy is stock_levels=}, {stock_copy['tea'] is stock_levels['tea']=}")

stock_copy is stock_levels=False, stock_copy['tea'] is stock_levels['tea']=True


**`{**d}`**: dict unpacking into a literal (like JS spread). It is a copy that can also add or override keys.

In [369]:
stock_levels = {"tea": [5, 3], "rice": [9]}
stock_copy = {**stock_levels, "ghee": [2]}
print(f"{stock_copy=}")
print(f"{stock_copy is stock_levels=}, {stock_copy['tea'] is stock_levels['tea']=}")

stock_copy={'tea': [5, 3], 'rice': [9], 'ghee': [2]}
stock_copy is stock_levels=False, stock_copy['tea'] is stock_levels['tea']=True


**`set.copy()`**: a new set. Set elements are hashable, usually immutable, so a shallow copy of a set is normally all you need.

In [370]:
ticket_numbers = {101, 102, 103}
tickets_copy = ticket_numbers.copy()
tickets_copy.add(104)
print(f"{sorted(ticket_numbers)=}, {sorted(tickets_copy)=}")
print(f"{tickets_copy is ticket_numbers=}")

sorted(ticket_numbers)=[101, 102, 103], sorted(tickets_copy)=[101, 102, 103, 104]
tickets_copy is ticket_numbers=False


**`copy.copy(x)`**: the generic shallow copy for any object. For built-in containers
it calls their own copy; for class instances it builds a new instance and copies the
attribute **dict** (shallowly). Immutable objects (`int`, `str`, `tuple`) come back
unchanged, because a copy of an immutable object is pointless.

In [371]:
class Basket:
    """A tiny class to show copy.copy on instances."""

    def __init__(self, owner: str, items: list[str]) -> None:
        self.owner = owner
        self.items = items


asha_basket = Basket("asha", ["tea"])
basket_copy = copy.copy(asha_basket)
print(f"{basket_copy is asha_basket=}, {basket_copy.items is asha_basket.items=}")
print(f"{copy.copy([1, 2]) == [1, 2]=}, {copy.copy({'a': 1}) == {'a': 1}=}")
point = (3, 4)
print(f"{copy.copy(point) is point=}, {copy.copy('dosa') == 'dosa'=}  (immutables)")

basket_copy is asha_basket=False, basket_copy.items is asha_basket.items=True
copy.copy([1, 2]) == [1, 2]=True, copy.copy({'a': 1}) == {'a': 1}=True
copy.copy(point) is point=True, copy.copy('dosa') == 'dosa'=True  (immutables)


**The outer container is new; inner objects are shared.** This is the trap. A nested
config dict is copied shallowly, then a *nested* value is changed through the copy.
The top-level key change stays local, but the nested change leaks into the original.

```mermaid
flowchart LR
    base["base_config (dict)"] --> db["{'host': ..., 'pool': [5, 10]}"]
    copy_["staging_config (new dict)"] --> db
```

<details><summary>ASCII fallback</summary>

```text
base_config ----\
                 +--> {"host": "db.local", "pool": [5, 10]}   (one shared dict)
staging_config -/
```

</details>

In [372]:
base_config = {
    "app_name": "shop",
    "database": {"host": "db.local", "pool_sizes": [5, 10]},
}
staging_config = base_config.copy()
staging_config["app_name"] = "shop-staging"  # top level: only the copy changes
staging_config["database"]["host"] = "staging-db.local"  # nested: shared!
staging_config["database"]["pool_sizes"].append(20)

print(f"{base_config['app_name']=}")
print(f"{base_config['database']=}  (the original changed too)")
print(f"{staging_config['database'] is base_config['database']=}")

base_config['app_name']='shop'
base_config['database']={'host': 'staging-db.local', 'pool_sizes': [5, 10, 20]}  (the original changed too)
staging_config['database'] is base_config['database']=True


**Predict the output.**

```python
menu = {"breakfast": ["idli"], "lunch": ["thali"]}
menu_copy = {**menu}
menu_copy["breakfast"] += ["dosa"]
menu_copy["lunch"] = menu_copy["lunch"] + ["curd"]
print(menu)
```

<details><summary>Answer</summary>

`{'breakfast': ['idli', 'dosa'], 'lunch': ['thali']}`. `+=` on a list **mutates** it
in place (`list.__iadd__` is `extend`), and that list is shared with `menu`. Then
`menu_copy["lunch"] + [...]` builds a **new** list and rebinds only the copy's key,
so `menu["lunch"]` is untouched.

</details>

Run the next cell to check your prediction.

In [373]:
menu = {"breakfast": ["idli"], "lunch": ["thali"]}
menu_copy = {**menu}
menu_copy["breakfast"] += ["dosa"]
menu_copy["lunch"] = menu_copy["lunch"] + ["curd"]
print(menu)
print(f"{menu_copy['lunch'] is menu['lunch']=}")

{'breakfast': ['idli', 'dosa'], 'lunch': ['thali']}


menu_copy['lunch'] is menu['lunch']=False


### 1.7.3 Deep copies

`copy.deepcopy(x)` copies the container **and, recursively, everything it contains**,
so the result shares no mutable object with the original. Immutable atoms (`int`,
`str`, `None`, functions, classes) are reused, since copying them gains nothing.

The same nested config as in 1.7.2, copied deeply this time:

In [374]:
base_config = {
    "app_name": "shop",
    "database": {"host": "db.local", "pool_sizes": [5, 10]},
}
staging_config = copy.deepcopy(base_config)
staging_config["database"]["host"] = "staging-db.local"
staging_config["database"]["pool_sizes"].append(20)

print(f"{base_config['database']=}  (unchanged)")
print(f"{staging_config['database']=}")
print(f"{staging_config['database'] is base_config['database']=}")
print(f"{staging_config['app_name'] is base_config['app_name']=}  (str reused)")

base_config['database']={'host': 'db.local', 'pool_sizes': [5, 10]}  (unchanged)
staging_config['database']={'host': 'staging-db.local', 'pool_sizes': [5, 10, 20]}
staging_config['database'] is base_config['database']=False
staging_config['app_name'] is base_config['app_name']=True  (str reused)


**The memo dict.** `deepcopy` keeps a dictionary `memo` that maps
`id(original) → copy` for every object it has already copied. Before copying an
object it looks it up in `memo`. That gives two guarantees:

1. **Shared references stay shared.** If one list appears twice in the original, the
   copy contains one new list, twice; not two independent lists.
2. **Cycles terminate.** A structure that contains itself is copied once, and the
   inner reference points at the copy instead of recursing forever.

First, shared references. `default_tags` appears under two keys:

In [375]:
default_tags = ["new"]
catalog = {"tea": default_tags, "rice": default_tags}
catalog_copy = copy.deepcopy(catalog)

print(f"{catalog_copy['tea'] is catalog_copy['rice']=}  (still one list in the copy)")
print(f"{catalog_copy['tea'] is default_tags=}  (but a new one)")
catalog_copy["tea"].append("sale")
print(f"{catalog_copy=}")

catalog_copy['tea'] is catalog_copy['rice']=True  (still one list in the copy)
catalog_copy['tea'] is default_tags=False  (but a new one)
catalog_copy={'tea': ['new', 'sale'], 'rice': ['new', 'sale']}


Then a cycle: a list that contains itself. Python prints it as `[...]`. A naive
recursive copy would never end; `deepcopy` finds the list in `memo` the second time
and reuses the copy it is building.

In [376]:
self_referencing = ["start"]
self_referencing.append(self_referencing)
print(f"{self_referencing=}")

cycle_copy = copy.deepcopy(self_referencing)
print(f"{cycle_copy=}")
print(f"{cycle_copy[1] is cycle_copy=}  (the copy points at itself)")
print(f"{cycle_copy is self_referencing=}")

self_referencing=['start', [...]]
cycle_copy=['start', [...]]
cycle_copy[1] is cycle_copy=True  (the copy points at itself)
cycle_copy is self_referencing=False


#### Under the hood: inspecting the memo

You can pass your own `memo` dict and look at it afterwards. Keys are `id()` values of
the originals (not printed, they change per run), values are their copies. There is
also one bookkeeping entry: `memo[id(memo)]` holds a list that keeps the originals
alive while copying, so their ids cannot be reused mid-copy.

In [377]:
shared_pool = [5, 10]
nested_settings = {"primary": shared_pool, "replica": shared_pool}
copy_memo: dict[int, Any] = {}
settings_copy = copy.deepcopy(nested_settings, copy_memo)

print(f"{len(copy_memo)=}")
print(f"{copy_memo[id(shared_pool)] is settings_copy['primary']=}")
print(f"{copy_memo[id(nested_settings)] is settings_copy=}")
print(f"{id(copy_memo) in copy_memo=}, {type(copy_memo[id(copy_memo)]).__name__=}")

len(copy_memo)=3
copy_memo[id(shared_pool)] is settings_copy['primary']=True
copy_memo[id(nested_settings)] is settings_copy=True
id(copy_memo) in copy_memo=True, type(copy_memo[id(copy_memo)]).__name__='list'


### 1.7.4 Customizing copies

A class can control both kinds of copy:

| Hook | Called by | Must return |
| --- | --- | --- |
| `__copy__(self)` | `copy.copy(obj)` | the shallow copy |
| `__deepcopy__(self, memo)` | `copy.deepcopy(obj)` | the deep copy; pass `memo` on to nested `deepcopy` calls |

The main reason to customize is to **share expensive read-only parts on purpose**: a
large lookup table, a compiled regex, a connection pool. Copying them is slow and
pointless, so the hook copies only the per-object state.

The next cell defines `PriceList`, which holds a big read-only `tax_table` and a small
mutable `overrides` dict. `__copy__` gives the copy its own `overrides` dict but keeps
the same table.

In [378]:
class PriceList:
    """Prices with a large shared tax table and small per-copy overrides."""

    def __init__(self, tax_table: Mapping[str, int], overrides: dict[str, Any]) -> None:
        self.tax_table = tax_table  # big and read-only: safe to share
        self.overrides = overrides  # small and mutable: each copy needs its own

    def __copy__(self) -> PriceList:
        print("  __copy__ called")
        return PriceList(self.tax_table, dict(self.overrides))

    def __deepcopy__(self, memo: dict[int, Any]) -> PriceList:
        print("  __deepcopy__ called")
        return PriceList(self.tax_table, copy.deepcopy(self.overrides, memo))


gst_table = types.MappingProxyType({f"hsn-{code}": 18 for code in range(10_000)})
store_prices = PriceList(gst_table, {"tea": 249})
print("shallow:")
shallow_prices = copy.copy(store_prices)
print(f"  {shallow_prices.tax_table is store_prices.tax_table=}")
print(f"  {shallow_prices.overrides is store_prices.overrides=}")

shallow:
  __copy__ called
  shallow_prices.tax_table is store_prices.tax_table=True
  shallow_prices.overrides is store_prices.overrides=False


`__deepcopy__` receives the memo and **must** pass it to every nested `deepcopy`
call; otherwise shared references and cycles inside the object are not handled. Here
the `overrides` dict holds lists (so it needs a real deep copy), while the 10 000-row
table is still shared on purpose.

In [379]:
store_prices = PriceList(gst_table, {"tea": 249, "combo": 0})
store_prices.overrides["bundle_skus"] = ["tea", "rice"]
print("deep:")
deep_prices = copy.deepcopy(store_prices)
print(f"  {deep_prices.tax_table is store_prices.tax_table=}  (shared on purpose)")
print(f"  {deep_prices.overrides is store_prices.overrides=}")
bundle_original = store_prices.overrides["bundle_skus"]
print(f"  {deep_prices.overrides['bundle_skus'] is bundle_original=}")

deep:
  __deepcopy__ called
  deep_prices.tax_table is store_prices.tax_table=True  (shared on purpose)
  deep_prices.overrides is store_prices.overrides=False
  deep_prices.overrides['bundle_skus'] is bundle_original=False


> **Note:** without these hooks, `copy` falls back to the pickle protocol
> (`__reduce_ex__`), which for a plain class means "new instance + copy of
> `__dict__`". Classes that define `__getstate__` / `__setstate__` for pickling get the
> same behaviour for copies. Python 3.13 added `copy.replace(obj, **changes)` for
> "copy with some fields changed"; it is covered with dataclasses in Phase 2.

The last cell of this section shows the default (no hooks): a deep copy of an instance
copies its `__dict__`, so every attribute is duplicated, including the big table.

In [380]:
class PlainPriceList:
    """The same data without copy hooks."""

    def __init__(self, tax_table: dict[str, int], overrides: dict[str, int]) -> None:
        self.tax_table = tax_table
        self.overrides = overrides


plain_prices = PlainPriceList(dict(gst_table), {"tea": 249})
plain_copy = copy.deepcopy(plain_prices)
print(f"{plain_copy.tax_table is plain_prices.tax_table=}  (10 000 rows copied)")
print(f"{plain_copy.__dict__.keys() == plain_prices.__dict__.keys()=}")

plain_copy.tax_table is plain_prices.tax_table=False  (10 000 rows copied)
plain_copy.__dict__.keys() == plain_prices.__dict__.keys()=True


### 1.7.5 Limits and cost

Some objects **cannot** be copied, because they stand for something outside the
Python heap (an OS lock, a file descriptor, a network connection) or for a paused
computation (a generator frame). Copying the Python object would not copy the outside
resource, so these types refuse: `copy` falls back to pickling and fails with
`TypeError: cannot pickle ...`.

| Object | Why it cannot be copied |
| --- | --- |
| `threading.Lock()` | a lock's state belongs to the OS; two "copies" could both be held |
| open file | the descriptor and file position are OS state |
| `socket.socket()` | a connection cannot be duplicated by copying memory |
| generator | its suspended frame cannot be cloned (use `itertools.tee` to split one, 1.13) |

Each of the next four cells tries `copy.deepcopy` on one of them.

**A lock:**

In [381]:
order_lock = threading.Lock()
try:
    copy.deepcopy(order_lock)
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: cannot pickle '_thread.lock' object


**An open file** (the unit's scratch file is written first, then opened for reading):

In [382]:
scratch_file = scratch_dir / "unit_1_7_copy_demo.txt"
scratch_file.write_text("order 1001\n", encoding="utf-8")
with scratch_file.open(encoding="utf-8") as order_file:
    try:
        copy.deepcopy(order_file)
    except TypeError as error:
        print(f"{type(error).__name__}: {error}")

TypeError: cannot pickle 'TextIOWrapper' instances


**A socket** (created but never connected, so no network is used):

In [383]:
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as api_socket:
    try:
        copy.deepcopy(api_socket)
    except TypeError as error:
        print(f"{type(error).__name__}: {error}")

TypeError: cannot pickle 'socket' object


**A generator** (shallow `copy.copy` fails too; there is no frame-cloning at all):

In [384]:
order_numbers = (number for number in range(1001, 1004))
for copy_function in (copy.copy, copy.deepcopy):
    try:
        copy_function(order_numbers)
    except TypeError as error:
        print(f"{copy_function.__name__}: {type(error).__name__}: {error}")

copy: TypeError: cannot pickle 'generator' object
deepcopy: TypeError: cannot pickle 'generator' object


**Cost.** A shallow copy touches only the outer container: O(n) pointer copies. A deep
copy visits **every** object, keeps the memo dict, and dispatches on each object's
type in Python code, so it is much slower and allocates much more. The next cell
times both on 10 000 nested order records.

> **Note:** the timings vary by machine and Python version; the ratio is what matters.

In [385]:
order_records = [
    {"order_id": order_id, "customer": "asha", "lines": [{"sku": "TEA-500G", "qty": 2}]}
    for order_id in range(10_000)
]
shallow_seconds = timeit.timeit(lambda: copy.copy(order_records), number=20) / 20
deep_seconds = timeit.timeit(lambda: copy.deepcopy(order_records), number=3) / 3
print(f"shallow copy: {shallow_seconds * 1000:8.3f} ms")
print(f"deep copy:    {deep_seconds * 1000:8.3f} ms")
print(f"deep is roughly {deep_seconds / shallow_seconds:,.0f}x slower")

shallow copy:    0.027 ms


deep copy:      17.788 ms
deep is roughly 653x slower


#### Under the hood: memory with `tracemalloc`

`tracemalloc` measures the bytes allocated while each copy runs. The shallow copy
allocates one new list (8 bytes per pointer plus a header); the deep copy allocates
10 000 dicts, 10 000 inner lists and 10 000 inner dicts. Exact numbers vary by version.

In [386]:
def allocated_bytes(make_copy: Callable[[], object]) -> int:
    """Return the bytes still allocated by the object make_copy returns."""
    tracemalloc.start()
    before_bytes, _ = tracemalloc.get_traced_memory()
    copied_object = make_copy()
    after_bytes, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    del copied_object
    return after_bytes - before_bytes


shallow_bytes = allocated_bytes(lambda: copy.copy(order_records))
deep_bytes = allocated_bytes(lambda: copy.deepcopy(order_records))
print(f"shallow copy: {shallow_bytes:>12,} bytes")
print(f"deep copy:    {deep_bytes:>12,} bytes")

shallow copy:       80,000 bytes


deep copy:       4,634,575 bytes


### 1.7.6 JS bridge: structuredClone

| JavaScript | Python | Depth |
| --- | --- | --- |
| `[...items]`, `items.slice()` | `items.copy()`, `items[:]`, `list(items)` | shallow |
| `{...config}`, `Object.assign({}, config)` | `config.copy()`, `{**config}`, `dict(config)` | shallow |
| `structuredClone(value)` | `copy.deepcopy(value)` | deep |

The biggest practical difference: `structuredClone` copies only "plain data". A class
instance comes back as a plain object without its prototype (its methods are gone),
and functions throw `DataCloneError`. `deepcopy` keeps the class, so methods still
work, and copies functions and classes by reference.

```javascript
class Cart { constructor() { this.items = [] } total() { return this.items.length } }
const clone = structuredClone(new Cart())
clone instanceof Cart   // false
clone.total             // undefined
```

The Python equivalent keeps everything:

In [387]:
class Cart:
    """A cart with a method, to show that deepcopy keeps the class."""

    def __init__(self, items: list[str], on_checkout: Callable[[], str]) -> None:
        self.items = items
        self.on_checkout = on_checkout

    def item_count(self) -> int:
        """Return the number of items."""
        return len(self.items)


def send_receipt() -> str:
    """Pretend to send a receipt."""
    return "receipt sent"


asha_cart = Cart(["tea", "rice"], send_receipt)
cart_clone = copy.deepcopy(asha_cart)
print(f"{type(cart_clone).__name__=}, {isinstance(cart_clone, Cart)=}")
print(f"{cart_clone.item_count()=}, {cart_clone.items is asha_cart.items=}")
print(f"{cart_clone.on_checkout is send_receipt=}  (functions are shared, not copied)")

type(cart_clone).__name__='Cart', isinstance(cart_clone, Cart)=True
cart_clone.item_count()=2, cart_clone.items is asha_cart.items=False
cart_clone.on_checkout is send_receipt=True  (functions are shared, not copied)


**Why `json.loads(json.dumps(x))` is the wrong deep copy.** It is the Python version of
the old `JSON.parse(JSON.stringify(x))` trick, and it fails for the same reason: JSON
has only objects with string keys, arrays, strings, numbers, booleans and null. Every
other type is either converted or rejected.

| Value | After a JSON round trip |
| --- | --- |
| tuple | becomes a list |
| set | `TypeError` (not serializable) |
| `Decimal` | `TypeError` (or a lossy float with `default=float`) |
| `datetime` | `TypeError` (or a string with `default=str`) |
| int dict keys | become strings: `{1: ...}` → `{"1": ...}` |

The next five cells show one failure each; `deepcopy` handles all of them.

**Tuples become lists:**

In [388]:
shipment = {"dimensions_cm": (30, 20, 10)}
json_copy = json.loads(json.dumps(shipment))
print(f"{json_copy=}, {type(json_copy['dimensions_cm']).__name__=}")
print(f"{copy.deepcopy(shipment)=}")

json_copy={'dimensions_cm': [30, 20, 10]}, type(json_copy['dimensions_cm']).__name__='list'


copy.deepcopy(shipment)={'dimensions_cm': (30, 20, 10)}


**Sets are rejected:**

In [389]:
product = {"tags": {"organic", "vegan"}}
try:
    json.dumps(product)
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{sorted(copy.deepcopy(product)['tags'])=}")

TypeError: Object of type set is not JSON serializable
sorted(copy.deepcopy(product)['tags'])=['organic', 'vegan']


**`Decimal` is rejected, and the usual workaround (`default=float`) silently loses exactness:**

In [390]:
invoice = {"total": Decimal("1999.99")}
print(f"plain dumps: {raised_error_name(lambda: json.dumps(invoice))}")
float_round_trip = json.loads(json.dumps({"total": Decimal("0.1") * 3}, default=float))
print(f"{float_round_trip=}  (a float now, not a Decimal)")
print(f"{copy.deepcopy(invoice)=}")

plain dumps: TypeError
float_round_trip={'total': 0.3}  (a float now, not a Decimal)
copy.deepcopy(invoice)={'total': Decimal('1999.99')}


**Datetimes are rejected, and `default=str` turns them into strings that do not compare or add like dates:**

In [391]:
delivery = {"due": dt.datetime(2026, 11, 3, 18, 30)}
print(f"plain dumps: {raised_error_name(lambda: json.dumps(delivery))}")
string_round_trip = json.loads(json.dumps(delivery, default=str))
print(f"{string_round_trip=}, {type(string_round_trip['due']).__name__=}")
print(f"{copy.deepcopy(delivery)=}")

plain dumps: TypeError
string_round_trip={'due': '2026-11-03 18:30:00'}, type(string_round_trip['due']).__name__='str'
copy.deepcopy(delivery)={'due': datetime.datetime(2026, 11, 3, 18, 30)}


**Int keys come back as strings, so lookups by the original key fail:**

In [392]:
stock_by_shelf = {1: "tea", 2: "rice"}
json_copy = json.loads(json.dumps(stock_by_shelf))
print(f"{json_copy=}")
print(f"lookup json_copy[1]: {raised_error_name(lambda: json_copy[1])}")
print(f"{copy.deepcopy(stock_by_shelf)[1]=}")

json_copy={'1': 'tea', '2': 'rice'}
lookup json_copy[1]: KeyError
copy.deepcopy(stock_by_shelf)[1]='tea'


### 1.7.7 Immutable snapshots

Defensive copying ("copy the input in case someone mutates it later") costs time and
memory on every call. If the data is **immutable**, nobody can mutate it, so it can be
shared freely and copying is unnecessary. `copy.copy` knows this and returns the same
object for immutable built-ins.

| Mutable | Immutable snapshot | Notes |
| --- | --- | --- |
| `list` | `tuple(items)` | immutable only if the elements are |
| `set` | `frozenset(tags)` | hashable, usable as a dict key |
| `dict` | `frozendict(config)` (*New in 3.15*) | hashable if the values are; `MappingProxyType` is a read-only *view* on older versions |

The next cell takes a tuple snapshot of an order's line items and shows that
`copy.copy` returns the same object, so passing it around costs nothing.

In [393]:
live_line_items = ["TEA-500G", "RICE-5KG"]
line_snapshot = tuple(live_line_items)
live_line_items.append("GHEE-1L")
print(f"{line_snapshot=}  (later appends do not reach the snapshot)")
print(f"{copy.copy(line_snapshot) is line_snapshot=}")
print(f"{copy.deepcopy(line_snapshot) is line_snapshot=}  (all elements immutable)")
print(f"item assignment: {raised_error_name(lambda: line_snapshot.__setitem__(0, 'X'))}")

line_snapshot=('TEA-500G', 'RICE-5KG')  (later appends do not reach the snapshot)
copy.copy(line_snapshot) is line_snapshot=True
copy.deepcopy(line_snapshot) is line_snapshot=True  (all elements immutable)
item assignment: AttributeError


A tuple is only as immutable as its contents. A tuple holding a list can still change
through that list, and `deepcopy` must then copy it (it returns a new tuple).

In [394]:
mixed_snapshot = ("asha", ["tea"])
mixed_snapshot[1].append("rice")
print(f"{mixed_snapshot=}  (the inner list changed)")
print(f"{copy.deepcopy(mixed_snapshot) is mixed_snapshot=}")
print(f"hash: {raised_error_name(lambda: hash(mixed_snapshot))}")

mixed_snapshot=('asha', ['tea', 'rice'])  (the inner list changed)
copy.deepcopy(mixed_snapshot) is mixed_snapshot=False
hash: TypeError


**`frozenset`**: an immutable set of permissions that can be shared between users without copying, and used as a dict key.

In [395]:
EDITOR_PERMISSIONS = frozenset({"read", "write"})
permissions_by_user = {"asha": EDITOR_PERMISSIONS, "ravi": EDITOR_PERMISSIONS}
print(f"{copy.copy(EDITOR_PERMISSIONS) is EDITOR_PERMISSIONS=}")
print(f"add: {raised_error_name(lambda: EDITOR_PERMISSIONS.add('admin'))}")
role_names = {EDITOR_PERMISSIONS: "editor"}
print(f"{role_names[frozenset({'write', 'read'})]=}")
print(f"{sorted(permissions_by_user['ravi'])=}")

copy.copy(EDITOR_PERMISSIONS) is EDITOR_PERMISSIONS=True
add: AttributeError
role_names[frozenset({'write', 'read'})]='editor'
sorted(permissions_by_user['ravi'])=['read', 'write']


**`frozendict`** (*New in 3.15*, PEP 814): an immutable dict. `copy.copy` returns the
same object. `copy.deepcopy` still walks the values (they could be mutable), so it can
return a new object; the point is that you do not need to call it at all.
On older kernels the cell uses `types.MappingProxyType`, a read-only **view** that
still changes if the dict behind it changes.

In [396]:
settings_source = {"currency": "INR", "gst_percent": 18}
if sys.version_info >= (3, 15):
    APP_SETTINGS = frozendict(settings_source)
    settings_source["gst_percent"] = 28  # the frozendict copied the data at creation
    print(f"{APP_SETTINGS=}")
    print(f"{copy.copy(APP_SETTINGS) is APP_SETTINGS=}")
    print(f"assign: {raised_error_name(lambda: APP_SETTINGS.__setitem__('currency', 'USD'))}")
    print(f"{hash(APP_SETTINGS) == hash(frozendict(gst_percent=18, currency='INR'))=}")
else:
    read_only_view = types.MappingProxyType(settings_source)
    settings_source["gst_percent"] = 28
    print(f"frozendict needs Python 3.15; MappingProxyType view: {dict(read_only_view)=}")

APP_SETTINGS=frozendict({'currency': 'INR', 'gst_percent': 18})
copy.copy(APP_SETTINGS) is APP_SETTINGS=True
assign: AttributeError
hash(APP_SETTINGS) == hash(frozendict(gst_percent=18, currency='INR'))=True


**Rule of thumb:** store configuration and reference data in immutable types from the
start, and copy only the mutable working state (orders being edited, a cart being
filled). Lab 1.7, exercise 2, needs a deep copy precisely because the orders are
mutable nested dicts.

### Lab 1.7

Two exercises. For each, replace the `raise NotImplementedError` in the stub, then run
the check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — `deep_merge(base, override)`

Return a **new** dict that merges `override` into `base`:

- a key only in one input is taken from that input;
- if both values are dicts, they are merged recursively;
- otherwise the `override` value wins (a list replaces a list, it is not appended).

The function must **never mutate** its inputs, and the result must not share any
mutable object with them: mutating the result afterwards must not change `base` or
`override` (the checks use `is` to verify this). Use `copy.deepcopy` for values taken
over unchanged.

In [397]:
def deep_merge(base: Mapping[str, Any], override: Mapping[str, Any]) -> dict[str, Any]:
    """Return a new dict with override merged into base recursively."""
    raise NotImplementedError("Lab 1.7: implement deep_merge()")

The check cell builds fresh inputs for every case with `make_merge_inputs()`, and
`merge_shares_nothing()` mutates the result to prove the inputs stay untouched.

In [398]:
def make_merge_inputs() -> tuple[dict[str, Any], dict[str, Any]]:
    """Return fresh (base, override) config dicts for the deep_merge checks."""
    base_settings = {
        "app": {"name": "shop", "debug": False},
        "database": {"host": "db.local", "pool_sizes": [5, 10]},
        "features": ["search"],
    }
    override_settings = {
        "app": {"debug": True},
        "database": {"host": "staging-db.local"},
        "features": ["search", "wishlist"],
        "region": "ap-south-1",
    }
    return base_settings, override_settings


def merge_shares_nothing() -> bool:
    """Mutate deep_merge's result and check that no input object changed or is shared."""
    base_settings, override_settings = make_merge_inputs()
    inputs_before = copy.deepcopy((base_settings, override_settings))
    merged = deep_merge(base_settings, override_settings)
    shares_objects = (
        merged is base_settings
        or merged["app"] is base_settings["app"]
        or merged["database"] is base_settings["database"]
        or merged["database"]["pool_sizes"] is base_settings["database"]["pool_sizes"]
        or merged["features"] is override_settings["features"]
    )
    merged["database"]["pool_sizes"].append(99)
    merged["features"].append("cart")
    merged["app"]["name"] = "changed"
    return not shares_objects and (base_settings, override_settings) == inputs_before


run_checks(
    [
        (
            "nested merge",
            lambda: deep_merge(*make_merge_inputs()),
            {
                "app": {"name": "shop", "debug": True},
                "database": {"host": "staging-db.local", "pool_sizes": [5, 10]},
                "features": ["search", "wishlist"],
                "region": "ap-south-1",
            },
        ),
        ("returns a dict", lambda: type(deep_merge(*make_merge_inputs())).__name__, "dict"),
        ("inputs untouched, nothing shared", merge_shares_nothing, True),
        ("empty override", lambda: deep_merge({"a": {"b": 1}}, {}), {"a": {"b": 1}}),
        ("empty base", lambda: deep_merge({}, {"a": [1]}), {"a": [1]}),
        ("dict replaces scalar", lambda: deep_merge({"a": 1}, {"a": {"b": 2}}), {"a": {"b": 2}}),
        ("scalar replaces dict", lambda: deep_merge({"a": {"b": 2}}, {"a": None}), {"a": None}),
        (
            "three levels deep",
            lambda: deep_merge({"a": {"b": {"c": 1, "d": 2}}}, {"a": {"b": {"d": 3}}}),
            {"a": {"b": {"c": 1, "d": 3}}},
        ),
    ]
)

FAIL  nested merge: raised NotImplementedError: Lab 1.7: implement deep_merge()
FAIL  returns a dict: raised NotImplementedError: Lab 1.7: implement deep_merge()
FAIL  inputs untouched, nothing shared: raised NotImplementedError: Lab 1.7: implement deep_merge()
FAIL  empty override: raised NotImplementedError: Lab 1.7: implement deep_merge()
FAIL  empty base: raised NotImplementedError: Lab 1.7: implement deep_merge()
FAIL  dict replaces scalar: raised NotImplementedError: Lab 1.7: implement deep_merge()
FAIL  scalar replaces dict: raised NotImplementedError: Lab 1.7: implement deep_merge()
FAIL  three levels deep: raised NotImplementedError: Lab 1.7: implement deep_merge()
0/8 checks passed


#### Exercise 2 — snapshot, discount, diff

`apply_discounts_in_place` (given below) is "legacy" code: it changes unit prices
**inside** the orders you pass. To report what it changed, take a snapshot first.

1. `diff_orders(before, after)`: compare two order lists with the same shape and
   return a `PriceChange(order_id, sku, old_unit_price, new_unit_price)` for every
   line whose unit price differs, in order.
2. `discount_and_diff(orders, discount_percent_by_sku)`: snapshot `orders` with
   `copy.deepcopy`, call `apply_discounts_in_place(orders, ...)` (so `orders` **is**
   changed, as the legacy code requires), and return `diff_orders(snapshot, orders)`.

A shallow `orders.copy()` would make the diff empty, because the line dicts would be
shared; one check guards against that.

Run the next cell first: it defines `PriceChange`, `make_sample_orders()` (fresh data
every call) and `apply_discounts_in_place`.

In [399]:
class PriceChange(NamedTuple):
    """One changed line found by diff_orders."""

    order_id: int
    sku: str
    old_unit_price: Decimal
    new_unit_price: Decimal


def make_sample_orders() -> list[dict[str, Any]]:
    """Return fresh nested order data for Lab 1.7, exercise 2."""
    return [
        {
            "order_id": 1001,
            "customer": "asha",
            "lines": [
                {"sku": "TEA-500G", "unit_price": Decimal("249.00"), "quantity": 2},
                {"sku": "GHEE-1L", "unit_price": Decimal("640.00"), "quantity": 1},
            ],
        },
        {
            "order_id": 1002,
            "customer": "ravi",
            "lines": [{"sku": "RICE-5KG", "unit_price": Decimal("525.50"), "quantity": 1}],
        },
    ]


def apply_discounts_in_place(
    orders: list[dict[str, Any]], discount_percent_by_sku: Mapping[str, int]
) -> None:
    """Legacy code: reduce unit prices inside orders, rounded to the paisa."""
    for order in orders:
        for line in order["lines"]:
            discount_percent = discount_percent_by_sku.get(line["sku"], 0)
            discounted = line["unit_price"] * (100 - discount_percent) / 100
            line["unit_price"] = discounted.quantize(PAISE, rounding=ROUND_HALF_UP)


print(f"{len(make_sample_orders())=}")

len(make_sample_orders())=2


The two stubs:

In [400]:
def diff_orders(before: list[dict[str, Any]], after: list[dict[str, Any]]) -> list[PriceChange]:
    """Return a PriceChange for every line whose unit price differs."""
    raise NotImplementedError("Lab 1.7: implement diff_orders()")


def discount_and_diff(
    orders: list[dict[str, Any]], discount_percent_by_sku: Mapping[str, int]
) -> list[PriceChange]:
    """Snapshot orders, apply discounts in place, return what changed."""
    raise NotImplementedError("Lab 1.7: implement discount_and_diff()")

The check cell uses a 10% discount on tea and 5% on rice; ghee is not discounted.

In [401]:
sample_discounts = {"TEA-500G": 10, "RICE-5KG": 5}
expected_changes = [
    PriceChange(1001, "TEA-500G", Decimal("249.00"), Decimal("224.10")),
    PriceChange(1002, "RICE-5KG", Decimal("525.50"), Decimal("499.23")),
]


def orders_after_discount() -> list[dict[str, Any]]:
    """Run discount_and_diff on fresh orders and return the (mutated) orders."""
    live_orders = make_sample_orders()
    discount_and_diff(live_orders, sample_discounts)
    return live_orders


def orders_with_rice_at_one_rupee() -> list[dict[str, Any]]:
    """Return fresh orders with the rice line edited by hand."""
    edited_orders = make_sample_orders()
    edited_orders[1]["lines"][0]["unit_price"] = Decimal("1.00")
    return edited_orders


run_checks(
    [
        (
            "diff of identical orders",
            lambda: diff_orders(make_sample_orders(), make_sample_orders()),
            [],
        ),
        (
            "discount_and_diff changes",
            lambda: discount_and_diff(make_sample_orders(), sample_discounts),
            expected_changes,
        ),
        (
            "orders were discounted in place",
            lambda: orders_after_discount()[0]["lines"][0]["unit_price"],
            Decimal("224.10"),
        ),
        ("no discounts, no changes", lambda: discount_and_diff(make_sample_orders(), {}), []),
        (
            "diff after a manual edit",
            lambda: diff_orders(make_sample_orders(), orders_with_rice_at_one_rupee()),
            [PriceChange(1002, "RICE-5KG", Decimal("525.50"), Decimal("1.00"))],
        ),
    ]
)

FAIL  diff of identical orders: raised NotImplementedError: Lab 1.7: implement diff_orders()
FAIL  discount_and_diff changes: raised NotImplementedError: Lab 1.7: implement discount_and_diff()
FAIL  orders were discounted in place: raised NotImplementedError: Lab 1.7: implement discount_and_diff()
FAIL  no discounts, no changes: raised NotImplementedError: Lab 1.7: implement discount_and_diff()
FAIL  diff after a manual edit: raised NotImplementedError: Lab 1.7: implement diff_orders()
0/5 checks passed


### Recap — Unit 1.7

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Assignment | Binds another name to the same object; never copies | `const b = a` |
| Shallow copy | New outer container, same inner objects | `[...a]`, `{...o}`, `Object.assign` |
| Shallow spellings | `x.copy()`, `x[:]`, `list(x)`, `dict(x)`, `{**d}`, `set(x)`, `copy.copy(x)` | `slice()`, spread |
| Nested data | Changes to inner objects leak through a shallow copy | same trap |
| `copy.deepcopy` | Recursively copies everything mutable; immutables reused | `structuredClone` |
| Memo dict | `id → copy`; keeps shared references shared and ends cycles | internal to `structuredClone` |
| `__copy__` / `__deepcopy__(memo)` | Customize copies; share expensive read-only parts on purpose | no hook |
| Uncopyable | Locks, files, sockets, generators raise `TypeError` | `DataCloneError` |
| Cost | Deep copy is far slower and allocates per object | same |
| Class instances | `deepcopy` keeps the class and methods | `structuredClone` drops the prototype |
| JSON round trip | Loses tuples, sets, `Decimal`, datetimes, int keys | `JSON.parse(JSON.stringify())` |
| Immutable snapshots | `tuple`, `frozenset`, `frozendict` need no defensive copy | `Object.freeze` (shallow) |

**Further reading**

- [`copy` — Shallow and deep copy operations](https://docs.python.org/3/library/copy.html)
- [Programming FAQ: How do I copy an object?](https://docs.python.org/3/faq/programming.html#how-do-i-copy-an-object-in-python)
- [`pickle` — object reduction (`__reduce_ex__`)](https://docs.python.org/3/library/pickle.html#object.__reduce_ex__)
- [`json`](https://docs.python.org/3/library/json.html),
  [`tracemalloc`](https://docs.python.org/3/library/tracemalloc.html)
- [`frozendict`](https://docs.python.org/3.15/library/stdtypes.html#frozendict),
  [PEP 814 — Add frozendict built-in type](https://peps.python.org/pep-0814/)
- [MDN: `structuredClone()`](https://developer.mozilla.org/en-US/docs/Web/API/Window/structuredClone)

## Unit 1.8 — Operators, Expressions and Control Flow

**Objectives**

- Use every operator family, read the precedence table, and write chained
  comparisons.
- Explain why `and` / `or` return one of their operands, and avoid the
  `value or default` trap.
- Write conditionals, guard clauses and structural pattern matching (`match`) with
  every pattern kind.
- Use `for`, `while`, `break`, `continue` and the `else` clause of loops.
- Use every assignment form, including the walrus operator, and `*` / `**` unpacking.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `a === b` | `a == b` (value), `a is b` (identity) | No coercion: `1 == "1"` is `False`; `==` calls `__eq__` |
| `&&`, `\|\|`, `!` | `and`, `or`, `not` | Words, not symbols; `not` always returns a `bool` |
| `a ?? b` | `a if a is not None else b` | No `??` operator; `a or b` also replaces `0` and `""` |
| `c ? a : b` | `a if c else b` | The condition sits in the middle |
| `a < b && b < c` | `a < b < c` | Chained comparisons are real syntax |
| `-7 % 3` is `-1` | `-7 % 3` is `2` | `%` takes the sign of the divisor; `//` floors |
| `5 & 1 == 0` is `5 & (1 == 0)` | `5 & 1 == 0` is `(5 & 1) == 0` | Bitwise operators bind tighter than comparisons |
| `switch` / `case` / `break` | `match` / `case` | No fall-through; patterns destructure and bind names |
| `for (let i = 0; i < n; i++)` | `for index in range(n)` | No C-style loop, no `++` |
| `for (const x of items)` | `for item in items` | Works on any iterable |
| `"key" in obj` | `key in mapping` | `in` on a list searches **values** |
| `const [first, ...rest] = items` | `first, *rest = items` | `rest` is always a `list` |

The next cell is this unit's setup. `compile_error(source)` compiles a snippet without
running it and reports a `SyntaxError`, so illegal code can be shown safely.

In [402]:
import dis
import re
import sys
from collections.abc import Callable, Hashable, Iterable
from http import HTTPStatus
from typing import Any


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def compile_error(source: str) -> str:
    """Compile source without running it; return "SyntaxError: <message>" or "compiles"."""
    try:
        compile(source, "<demo>", "exec")
    except SyntaxError as error:
        return f"SyntaxError: {error.msg}"
    return "compiles"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.8.1 Operators and precedence

| Family | Operators |
| --- | --- |
| Arithmetic | `+`, `-`, `*`, `/` (true division), `//` (floor division), `%`, `**`, `@` (matrix, for libraries) |
| Comparison | `==`, `!=`, `<`, `<=`, `>`, `>=` |
| Logical | `and`, `or`, `not` |
| Bitwise | `&`, `\|`, `^`, `~`, `<<`, `>>` |
| Membership | `in`, `not in` |
| Identity | `is`, `is not` |

**Arithmetic.** `/` always returns a `float`; `//` floors toward negative infinity,
and `%` takes the sign of the **divisor**, so `a == (a // b) * b + a % b` always holds.
JavaScript's `%` takes the sign of the dividend, so negative numbers differ.

In [403]:
items_in_stock, box_size = 17, 5
print(f"{items_in_stock + box_size=}, {items_in_stock - box_size=}, {items_in_stock * box_size=}")
print(f"{items_in_stock / box_size=}, {items_in_stock // box_size=}, {items_in_stock % box_size=}")
print(f"{2**10=}, {-items_in_stock=}, {+items_in_stock=}")
print(f"{-7 // 2=}, {-7 % 3=}  (JS: Math.trunc(-7 / 2) is -3, -7 % 3 is -1)")
print(f"{divmod(-7, 3)=}, {(-7 // 3) * 3 + (-7 % 3)=}")
print(f"{4 / 2=}  (true division always gives a float)")

items_in_stock + box_size=22, items_in_stock - box_size=12, items_in_stock * box_size=85
items_in_stock / box_size=3.4, items_in_stock // box_size=3, items_in_stock % box_size=2
2**10=1024, -items_in_stock=-17, +items_in_stock=17
-7 // 2=-4, -7 % 3=2  (JS: Math.trunc(-7 / 2) is -3, -7 % 3 is -1)
divmod(-7, 3)=(-3, 2), (-7 // 3) * 3 + (-7 % 3)=-7
4 / 2=2.0  (true division always gives a float)


**Comparison.** `==` compares values (calls `__eq__`) with **no type coercion**:
numbers of different types compare by value, but a string never equals a number.
Ordering between unrelated types raises `TypeError` instead of guessing. Sequences
compare element by element (lexicographically).

In [404]:
print(f"{1 == 1.0=}, {1 == '1'=}, {1 != '1'=}")
print(f"{'apple' < 'banana'=}, {'Zebra' < 'apple'=}  (code point order: 'Z' < 'a')")
print(f"{[1, 2, 3] < [1, 3]=}, {(2026, 11) >= (2026, 10)=}")
print(f"'10' < 5: {raised_error_name(lambda: '10' < 5)}")

1 == 1.0=True, 1 == '1'=False, 1 != '1'=True
'apple' < 'banana'=True, 'Zebra' < 'apple'=True  (code point order: 'Z' < 'a')
[1, 2, 3] < [1, 3]=True, (2026, 11) >= (2026, 10)=True
'10' < 5: TypeError


**Logical.** `and`, `or`, `not` work on truthiness (1.3.8). 1.8.2 shows that `and`
and `or` return an operand, not necessarily a `bool`.

In [405]:
is_logged_in, has_items = True, False
print(f"{is_logged_in and has_items=}, {is_logged_in or has_items=}, {not has_items=}")
print(f"{not []=}, {not 'text'=}")

is_logged_in and has_items=False, is_logged_in or has_items=True, not has_items=True
not []=True, not 'text'=False


**Bitwise.** They work on the binary digits of ints. Python ints have arbitrary
precision, so shifts never overflow (JavaScript truncates bitwise operands to 32 bits).
`~x` is `-x - 1`. A typical use is a set of permission flags packed into one int.

In [406]:
READ, WRITE, DELETE = 0b001, 0b010, 0b100
editor_permissions = READ | WRITE
print(f"{editor_permissions=:03b}, {editor_permissions & WRITE != 0=}")
print(f"{editor_permissions & DELETE != 0=}  (& binds tighter than !=)")
print(f"{editor_permissions ^ WRITE=:03b}  (toggle WRITE off)")
print(f"{~5=}, {1 << 4=}, {256 >> 2=}")
print(f"{1 << 100=}  (no 32-bit overflow)")

editor_permissions=011, editor_permissions & WRITE != 0=True
editor_permissions & DELETE != 0=False  (& binds tighter than !=)
editor_permissions ^ WRITE=001  (toggle WRITE off)
~5=-6, 1 << 4=16, 256 >> 2=64
1 << 100=1267650600228229401496703205376  (no 32-bit overflow)


**Membership.** `x in container` asks the container (`__contains__`): a list or
tuple searches values (O(n)), a `str` searches for a **substring**, a dict checks
**keys**, and a set checks elements in O(1) on average.

In [407]:
print(f"{'tea' in ['tea', 'rice']=}, {'ghee' not in ('tea', 'rice')=}")
print(f"{'err' in 'server error'=}  (substring)")
print(f"{'tea' in {'tea': 249}=}, {249 in {'tea': 249}=}  (keys only)")
print(f"{249 in {'tea': 249}.values()=}, {101 in {101, 102}=}")

'tea' in ['tea', 'rice']=True, 'ghee' not in ('tea', 'rice')=True
'err' in 'server error'=True  (substring)
'tea' in {'tea': 249}=True, 249 in {'tea': 249}=False  (keys only)
249 in {'tea': 249}.values()=True, 101 in {101, 102}=True


**Identity.** `is` checks that two names refer to the **same object**. Use it for
singletons (`None`, `True`, `False`, sentinels), never for comparing numbers or
strings (1.2.5).

In [408]:
discount_code = None
saved_cart = ["tea"]
same_cart = saved_cart
fresh_cart = ["tea"]
print(f"{discount_code is None=}, {saved_cart is same_cart=}, {saved_cart is not fresh_cart=}")
print(f"{saved_cart == fresh_cart=}  (equal values, different objects)")

discount_code is None=True, saved_cart is same_cart=True, saved_cart is not fresh_cart=True
saved_cart == fresh_cart=True  (equal values, different objects)


**Precedence**, from highest (binds tightest) to lowest:

| Level | Operators |
| --- | --- |
| 1 | `(...)`, `[...]`, `{...}` displays |
| 2 | `x[index]`, `x[start:stop]`, `f(args)`, `x.attribute` |
| 3 | `await x` |
| 4 | `**` (right-associative; binds tighter than a unary minus on its left) |
| 5 | `+x`, `-x`, `~x` |
| 6 | `*`, `@`, `/`, `//`, `%` |
| 7 | `+`, `-` |
| 8 | `<<`, `>>` |
| 9 | `&` |
| 10 | `^` |
| 11 | `\|` |
| 12 | `in`, `not in`, `is`, `is not`, `<`, `<=`, `>`, `>=`, `!=`, `==` (all one level, chainable) |
| 13 | `not x` |
| 14 | `and` |
| 15 | `or` |
| 16 | `x if condition else y` |
| 17 | `lambda` |
| 18 | `:=` |

The next cell shows the cases that surprise people. When in doubt, add parentheses.

In [409]:
print(f"{-2**2=}  (** first: -(2**2))")
print(f"{2**3**2=}  (right-associative: 2**(3**2))")
print(f"{1 + 2 * 3=}, {(1 + 2) * 3=}")
print(f"{5 & 1 == 0=}  ((5 & 1) == 0, unlike JS where == binds first)")
print(f"{not 1 == 2=}  (not (1 == 2))")
print(f"{True or False and False=}  (and before or)")

-2**2=-4  (** first: -(2**2))
2**3**2=512  (right-associative: 2**(3**2))
1 + 2 * 3=7, (1 + 2) * 3=9
5 & 1 == 0=False  ((5 & 1) == 0, unlike JS where == binds first)
not 1 == 2=True  (not (1 == 2))
True or False and False=True  (and before or)


**Predict the output.**

```python
print(-3**2, 2**-1, 7 // -2, -7 % 3)
```

<details><summary>Answer</summary>

`-9 0.5 -4 2`. `**` binds tighter than the unary minus on its left, so `-3**2` is
`-(3**2)`; a negative exponent gives a float; `//` floors toward negative infinity
(`-3.5` → `-4`); `%` takes the sign of the divisor, so `-7 % 3` is `2`.

</details>

Run the next cell to check your prediction.

In [410]:
print(-3**2, 2**-1, 7 // -2, -7 % 3)  # fmt: skip

-9 0.5 -4 2


**Chained comparisons.** `0 < age <= 120` means `0 < age and age <= 120`, except that
`age` is evaluated **once**. Any comparison operators can chain, which allows
`low <= value < high` range checks and `a == b == c`. It also allows confusing
chains such as `x < y > z`; avoid those.

In [411]:
def is_valid_age(age: int) -> bool:
    """Return True for a plausible human age."""
    return 0 < age <= 120


print(f"{is_valid_age(35)=}, {is_valid_age(0)=}, {is_valid_age(130)=}")
first_price, second_price, third_price = 249, 249, 249
print(f"{first_price == second_price == third_price=}")
print(f"{1 < 3 > 2=}  (legal but unclear: 1 < 3 and 3 > 2)")
print(f"{(1 < 2) == True=}, {1 < 2 == True=}  (1 < 2 and 2 == True)")  # noqa: E712

is_valid_age(35)=True, is_valid_age(0)=False, is_valid_age(130)=False
first_price == second_price == third_price=True
1 < 3 > 2=True  (legal but unclear: 1 < 3 and 3 > 2)
(1 < 2) == True=True, 1 < 2 == True=False  (1 < 2 and 2 == True)


#### Under the hood: a chained comparison in bytecode

`dis` shows how `0 < age <= 120` evaluates `age` once: it is loaded a single time,
`COPY` duplicates it for the second comparison, and a conditional jump skips the
second comparison when the first is false (short-circuit). Opcode names vary between
Python versions.

In [412]:
dis.dis(is_valid_age)

  1           RESUME                   0

  3           LOAD_SMALL_INT           0
              LOAD_FAST                0 (age)
              SWAP                     2
              COPY                     2
              COMPARE_OP               2 (<)
              COPY                     1
              TO_BOOL
              POP_JUMP_IF_FALSE        6 (to L1)
              NOT_TAKEN
              POP_TOP
              LOAD_SMALL_INT         120
              COMPARE_OP              42 (<=)
              RETURN_VALUE
      L1:     SWAP                     2
              POP_TOP
              RETURN_VALUE


### 1.8.2 Boolean operators return operands

`and` and `or` short-circuit like JavaScript's `&&` and `||`, and like them they
return **one of their operands**, not `True` / `False`:

| Expression | Returns |
| --- | --- |
| `a or b` | `a` if `a` is truthy, else `b` |
| `a and b` | `a` if `a` is falsy, else `b` |
| `not a` | always a `bool` |

The right operand is evaluated **only if needed**. The next cell uses a function
with a visible side effect to prove which operands run.

In [413]:
def loud(value: object) -> object:
    """Print the value, then return it (to see which operands are evaluated)."""
    print(f"  evaluated {value!r}")
    return value


print(f"result {loud('') or loud('guest')!r}")
print(f"result {loud('asha') or loud('guest')!r}  (right side skipped)")
print(f"result {loud(0) and loud(1 / 1)!r}  (right side skipped)")
print(f"{[] and 1 / 0=}  (1 / 0 never runs)")
print(f"{not 'asha'=}, {not 0=}, {type(not 'asha').__name__=}")

  evaluated ''
  evaluated 'guest'
result 'guest'
  evaluated 'asha'
result 'asha'  (right side skipped)
  evaluated 0
result 0  (right side skipped)
[] and 1 / 0=[]  (1 / 0 never runs)
not 'asha'=False, not 0=True, type(not 'asha').__name__='bool'


**`value or default` and its `0` trap.** `or` is a handy default, but it replaces
**every** falsy value: `0`, `0.0`, `""`, `[]`, `False`; not only `None`. If those are
valid inputs, it silently changes them.

Python has no `??` (null-coalescing) operator (PEP 505 proposed one and was deferred).
The explicit form is a conditional expression on `is not None`.

In [414]:
def items_per_page_with_or(requested: int | None) -> int:
    """Default to 20 when nothing was requested (buggy: 0 is replaced too)."""
    return requested or 20


def items_per_page(requested: int | None) -> int:
    """Default to 20 only when requested is None (the JS `requested ?? 20`)."""
    return requested if requested is not None else 20


for requested in (None, 50, 0):
    print(
        f"{requested=}: with or -> {items_per_page_with_or(requested)}, "
        f"with is not None -> {items_per_page(requested)}"
    )
display_name = "" or "Anonymous"
print(f"{display_name=}  (fine here: an empty name should get the default)")

requested=None: with or -> 20, with is not None -> 20
requested=50: with or -> 50, with is not None -> 50
requested=0: with or -> 20, with is not None -> 0
display_name='Anonymous'  (fine here: an empty name should get the default)


An `or` chain picks the first truthy value, which reads well for layered settings
when empty values really mean "not set":

In [415]:
cli_region, env_region, default_region = None, "", "ap-south-1"
chosen_region = cli_region or env_region or default_region
print(f"{chosen_region=}")

chosen_region='ap-south-1'


### 1.8.3 Conditionals

`if` / `elif` / `else` with a colon and an indented block; no parentheses around the
condition and no braces. Any object can be a condition (truthiness, 1.3.8). The first
true branch runs and the rest are skipped.

In [416]:
def shipping_fee(order_total: int) -> int:
    """Return the delivery fee in rupees for an order total."""
    if order_total >= 999:
        return 0
    elif order_total >= 499:
        return 40
    else:
        return 80


for order_total in (1200, 650, 120):
    print(f"{order_total=}: {shipping_fee(order_total)=}")

order_total=1200: shipping_fee(order_total)=0
order_total=650: shipping_fee(order_total)=40
order_total=120: shipping_fee(order_total)=80


**Conditional expression:** `value_if_true if condition else value_if_false` is an
*expression* (it has a value), the equivalent of JavaScript's `c ? a : b`. Keep it to
one simple condition; nested conditional expressions are hard to read, use
`if` / `elif` instead.

In [417]:
stock_count = 0
availability = "in stock" if stock_count > 0 else "sold out"
print(f"{availability=}")
plural_suffix = "" if stock_count == 1 else "s"
print(f"{stock_count} item{plural_suffix} left")

availability='sold out'
0 items left


**Guard clauses** handle the special cases first and `return` (or `raise`) early, so
the main path is not buried in nested `if` blocks. Both functions below behave the
same; the second is flat.

In [418]:
def ship_order_nested(order: dict[str, Any]) -> str:
    """Decide what to do with an order (nested style)."""
    if order["paid"]:
        if order["items"]:
            if order["address"]:
                return f"shipping order {order['order_id']}"
            else:
                return "missing address"
        else:
            return "empty order"
    else:
        return "awaiting payment"


def ship_order(order: dict[str, Any]) -> str:
    """Decide what to do with an order (guard clauses)."""
    if not order["paid"]:
        return "awaiting payment"
    if not order["items"]:
        return "empty order"
    if not order["address"]:
        return "missing address"
    return f"shipping order {order['order_id']}"


pending_order = {"order_id": 1001, "paid": True, "items": ["tea"], "address": ""}
print(f"{ship_order_nested(pending_order)=}, {ship_order(pending_order)=}")

ship_order_nested(pending_order)='missing address', ship_order(pending_order)='missing address'


A block cannot be empty: use `pass` as a placeholder (or `...`). There is no
`else if`; it is spelled `elif`.

In [419]:
log_level = "DEBUG"
if log_level == "DEBUG":
    pass  # nothing to do yet; the block must not be empty
print(f"{compile_error('if True:' + chr(10) + 'print(1)')=}")

compile_error('if True:' + chr(10) + 'print(1)')="SyntaxError: expected an indented block after 'if' statement on line 1"


### 1.8.4 Structural pattern matching (`match`)

`match subject:` compares one value against `case` patterns, top to bottom, and runs
the **first** case that matches (PEP 634, Python 3.10). A pattern describes the
*shape* of the data and can bind parts of it to names.

| Pattern | Example | Matches |
| --- | --- | --- |
| Literal | `case 200:` | equal value (`None`, `True`, `False` by identity) |
| Or | `case 401 \| 403:` | any of the alternatives |
| Capture | `case other:` | anything; binds it to `other` |
| Wildcard | `case _:` | anything; binds nothing |
| Sequence | `case ["deploy", env]:`, `case [cmd, *args]:` | lists/tuples of that length and shape (not `str`) |
| Mapping | `case {"type": event_type, **rest}:` | mappings with those keys (extra keys allowed) |
| Class | `case Point(x=0, y=y):`, `case Point(0, y):` | `isinstance` plus attributes |
| Value | `case HTTPStatus.NOT_FOUND:` | equal to a dotted name |
| Guard | `case [x, y] if x == y:` | the pattern **and** the condition |
| As | `case [("start" \| "stop") as action, name]:` | binds a sub-pattern's value |

Each of the next cells shows one pattern kind.

**Literal patterns** compare with `==` (and `is` for `None`, `True`, `False`):

In [420]:
def describe_status(status_code: int) -> str:
    """Describe an HTTP status code with literal patterns."""
    match status_code:
        case 200:
            return "OK"
        case 404:
            return "Not Found"
        case _:
            return "something else"


print(f"{describe_status(200)=}, {describe_status(404)=}, {describe_status(418)=}")

describe_status(200)='OK', describe_status(404)='Not Found', describe_status(418)='something else'


**Or patterns** (`|`) try several alternatives in one case:

In [421]:
def status_family(status_code: int) -> str:
    """Group status codes with or-patterns."""
    match status_code:
        case 200 | 201 | 204:
            return "success"
        case 401 | 403:
            return "auth problem"
        case _:
            return "other"


print(f"{status_family(201)=}, {status_family(403)=}, {status_family(500)=}")

status_family(201)='success', status_family(403)='auth problem', status_family(500)='other'


**Capture patterns**: a bare name matches **anything** and binds it. A capture as a
whole case is irrefutable, so it must be the last case.

In [422]:
def label_quantity(quantity: int) -> str:
    """Use a capture pattern for 'any other value'."""
    match quantity:
        case 0:
            return "none"
        case 1:
            return "one"
        case many:
            return f"{many} items"


print(f"{label_quantity(0)=}, {label_quantity(7)=}")

label_quantity(0)='none', label_quantity(7)='7 items'


**Wildcard `_`** matches anything and binds nothing (it is not a variable). It is
the `default:` of a `switch`, and it can stand in for parts of other patterns.

In [423]:
def first_word_kind(words: list[str]) -> str:
    """Use _ alone and inside a sequence pattern."""
    match words:
        case ["help", _]:
            return "help with a topic"
        case _:
            return "anything else"


print(f"{first_word_kind(['help', 'deploy'])=}, {first_word_kind(['status'])=}")

first_word_kind(['help', 'deploy'])='help with a topic', first_word_kind(['status'])='anything else'


**Sequence patterns** match lists and tuples by length and element patterns;
`*rest` collects any remaining elements into a list. Strings, bytes and bytearrays
are **not** treated as sequences here, so `"abc"` does not match `[a, b, c]`.

In [424]:
def describe_words(words: list[str]) -> str:
    """Match on the shape of a split command."""
    match words:
        case []:
            return "empty"
        case [single_word]:
            return f"one word: {single_word}"
        case [command, *arguments]:
            return f"{command} with {len(arguments)} argument(s): {arguments}"


print(describe_words([]))
print(describe_words(["status"]))
print(describe_words(["copy", "a.txt", "b.txt"]))
print(f"{describe_words('ab')=}  (a str never matches a sequence pattern)")

empty
one word: status
copy with 2 argument(s): ['a.txt', 'b.txt']
describe_words('ab')=None  (a str never matches a sequence pattern)


**Mapping patterns** check that the listed keys exist and match their value
patterns. Extra keys are **ignored**; `**rest` captures them into a dict.

In [425]:
def describe_event(event: dict[str, Any]) -> str:
    """Match dicts by their keys."""
    match event:
        case {"type": "login", "user": user_name}:
            return f"login by {user_name}"
        case {"type": event_type, **other_fields}:
            return f"{event_type} event with extra keys {sorted(other_fields)}"
        case _:
            return "not an event"


print(describe_event({"type": "login", "user": "asha", "ip": "10.0.0.1"}))
print(describe_event({"type": "logout", "user": "ravi", "reason": "timeout"}))
print(describe_event({"user": "meena"}))

login by asha
logout event with extra keys ['reason', 'user']
not an event


**Class patterns** check `isinstance` and then attributes. Keyword sub-patterns
(`Point(x=0)`) name attributes directly. Positional sub-patterns (`Point(0, y)`) use
the class attribute `__match_args__` to know which attribute each position means.
Dataclasses and named tuples generate `__match_args__` for you (Phase 2); here it is
written by hand. Built-in types such as `int(n)` and `str(s)` match the whole value.

In [426]:
class Point:
    """A 2-D point with __match_args__ for positional class patterns."""

    __match_args__ = ("x", "y")

    def __init__(self, x: int, y: int) -> None:
        self.x = x
        self.y = y


def where_is(location: object) -> str:
    """Use class patterns, positional and keyword."""
    match location:
        case Point(0, 0):
            return "origin"
        case Point(0, y):
            return f"on the y axis at {y}"
        case Point(x=x, y=0):
            return f"on the x axis at {x}"
        case Point():
            return "somewhere else"
        case int(number) | float(number):
            return f"a number: {number}"
        case _:
            return "not a point"


for location in (Point(0, 0), Point(0, 5), Point(3, 0), Point(2, 2), 7, "home"):
    print(where_is(location))

origin
on the y axis at 5
on the x axis at 3
somewhere else
a number: 7
not a point


**Guards**: `case pattern if condition:` matches only when the pattern matches **and**
the condition (which can use the bound names) is true. If the guard fails, matching
continues with the next case.

In [427]:
def classify_pair(pair: tuple[int, int]) -> str:
    """Use guards on top of a sequence pattern."""
    match pair:
        case (first, second) if first == second:
            return "equal"
        case (first, second) if first > second:
            return "descending"
        case (_, _):
            return "ascending"
        case _:
            return "not a pair"


print(f"{classify_pair((3, 3))=}, {classify_pair((5, 1))=}, {classify_pair((1, 5))=}")

classify_pair((3, 3))='equal', classify_pair((5, 1))='descending', classify_pair((1, 5))='ascending'


**`as` bindings** name the value matched by a sub-pattern, which is useful after an
or-pattern (you know *that* it matched, `as` tells you *which*).

In [428]:
def service_action(words: list[str]) -> str:
    """Bind the matched alternative with as."""
    match words:
        case [("start" | "stop" | "restart") as action, service_name]:
            return f"{action} {service_name}"
        case [_, *_] as whole_command:
            return f"unknown command {whole_command}"
        case _:
            return "empty"


print(f"{service_action(['restart', 'nginx'])=}")
print(f"{service_action(['reload', 'nginx'])=}")

service_action(['restart', 'nginx'])='restart nginx'
service_action(['reload', 'nginx'])="unknown command ['reload', 'nginx']"


**No fall-through.** Exactly one case runs (or none: then nothing happens, there is
no error). There is no `break`, and no way to fall into the next case. Compare:

| | JavaScript `switch` | Python `match` |
| --- | --- | --- |
| Comparison | `===` against each `case` value | structural patterns (shape, type, keys, values) |
| After a match | falls through until `break` | runs one case, then leaves |
| Several values | stacked `case 1: case 2:` | `case 1 \| 2:` |
| Destructuring | no | binds names from the subject |
| Conditions | no (use `if`) | guards: `case ... if condition:` |
| Default | `default:` | `case _:` |
| No match | nothing happens | nothing happens |

In [429]:
def count_matches(signal: str) -> list[str]:
    """Show that only the first matching case runs."""
    ran_cases = []
    match signal:
        case "go":
            ran_cases.append("first go case")
        case "go":  # also matches, but never runs
            ran_cases.append("second go case")
    return ran_cases


print(f"{count_matches('go')=}, {count_matches('stop')=}  (no match is not an error)")

count_matches('go')=['first go case'], count_matches('stop')=[]  (no match is not an error)


**The bare-name capture trap.** A bare name in a pattern is a **capture**, never a
constant lookup. `case NOT_FOUND:` does not compare with your variable `NOT_FOUND`;
it matches everything and **rebinds** `NOT_FOUND`. If more cases follow, Python
refuses to compile it; if it is the last case, it silently "works".

In [430]:
trap_source = """
NOT_FOUND = 404
match 200:
    case NOT_FOUND:
        print("not found")
    case _:
        print("other")
"""
print(compile_error(trap_source))

NOT_FOUND = 404
match 200:
    case NOT_FOUND:  # captures 200 and rebinds the name
        print("this runs for 200!")
print(f"{NOT_FOUND=}  (the 'constant' was overwritten)")

SyntaxError: name capture 'NOT_FOUND' makes remaining patterns unreachable
this runs for 200!
NOT_FOUND=200  (the 'constant' was overwritten)


**Dotted names are value patterns.** Any name with a dot (`HTTPStatus.NOT_FOUND`,
`config.LIMIT`) is looked up and compared with `==`. That is why enums and constants
on a class or module work as cases. `HTTPStatus` is an `IntEnum`, so its members also
equal plain ints.

In [431]:
def describe_with_enum(status_code: int) -> str:
    """Use dotted names as constant (value) patterns."""
    match status_code:
        case HTTPStatus.OK:
            return "OK"
        case HTTPStatus.NOT_FOUND:
            return "Not Found"
        case _:
            return "other"


print(f"{describe_with_enum(404)=}, {describe_with_enum(200)=}, {describe_with_enum(500)=}")

describe_with_enum(404)='Not Found', describe_with_enum(200)='OK', describe_with_enum(500)='other'


**Demo: a command parser.** A tiny deploy tool turns a command line into an action.
Sequence patterns pick the command apart, or-patterns accept aliases, guards
validate arguments, and `*` handles optional flags.

In [432]:
def run_deploy_command(command_text: str) -> str:
    """Parse a deploy-tool command and describe the action it would take."""
    match command_text.split():
        case ["deploy", ("staging" | "production") as environment]:
            return f"deploying to {environment}"
        case ["deploy", environment]:
            return f"error: unknown environment {environment!r}"
        case ["rollback", version] if re.fullmatch(r"v\d+\.\d+\.\d+", version):
            return f"rolling back to {version}"
        case ["rollback", _]:
            return "error: version must look like v1.2.3"
        case ["logs", service_name, *flags]:
            follow = "--follow" in flags
            return f"showing logs for {service_name} (follow={follow})"
        case ["status" | "st"]:
            return "showing status"
        case []:
            return "error: empty command"
        case [unknown_command, *_]:
            return f"error: unknown command {unknown_command!r}"


sample_commands = [
    "deploy staging",
    "deploy moon",
    "rollback v2.0.1",
    "rollback 2",
    "logs api --follow",
    "st",
    "",
    "dance",
]
for command_text in sample_commands:
    print(f"{command_text!r:22} -> {run_deploy_command(command_text)}")

'deploy staging'       -> deploying to staging
'deploy moon'          -> error: unknown environment 'moon'
'rollback v2.0.1'      -> rolling back to v2.0.1
'rollback 2'           -> error: version must look like v1.2.3
'logs api --follow'    -> showing logs for api (follow=True)
'st'                   -> showing status
''                     -> error: empty command
'dance'                -> error: unknown command 'dance'


**Demo: a webhook router.** A payment provider posts JSON events. Mapping patterns
route them by `type`, nested patterns pull out fields, class patterns (`int(amount)`)
check types, guards check values, and `**rest` keeps unknown fields. Phase 6 builds
a real FastAPI endpoint around this idea.

In [433]:
def route_webhook(event: dict[str, Any]) -> str:
    """Route a payment-provider webhook event to a handler description."""
    match event:
        case {"type": "payment.captured", "data": {"amount": int(amount), "currency": "INR"}}:
            return f"mark order paid: ₹{amount / 100:,.2f}"
        case {"type": "payment.failed", "data": {"reason": str(reason)}}:
            return f"notify customer: payment failed ({reason})"
        case {"type": "refund.created", "data": {"amount": int(amount)}} if amount > 1_000_000:
            return "refund over ₹10,000: needs manual approval"
        case {"type": "refund.created", "data": {"amount": int(amount)}}:
            return f"refund ₹{amount / 100:,.2f}"
        case {"type": str(event_type), **rest}:
            return f"ignored {event_type} (fields: {sorted(rest)})"
        case _:
            return "rejected: not a webhook event"


webhook_events = [
    {"type": "payment.captured", "data": {"amount": 49900, "currency": "INR"}},
    {"type": "payment.failed", "data": {"reason": "card declined"}},
    {"type": "refund.created", "data": {"amount": 2_500_000}},
    {"type": "refund.created", "data": {"amount": 19900}},
    {"type": "dispute.opened", "id": "evt_9", "data": {}},
    {"event": "payment.captured"},
]
for webhook_event in webhook_events:
    print(route_webhook(webhook_event))

mark order paid: ₹499.00
notify customer: payment failed (card declined)
refund over ₹10,000: needs manual approval
refund ₹199.00
ignored dispute.opened (fields: ['data', 'id'])
rejected: not a webhook event


#### Under the hood: `match` opcodes

`match` compiles to ordinary bytecode plus a few dedicated instructions:
`MATCH_SEQUENCE`, `MATCH_MAPPING`, `MATCH_KEYS` and `MATCH_CLASS`. The next cell
collects them from the two demo functions above. There is no jump
table: the cases are tried in order, as a chain of tests. Opcode names vary by
version.

In [434]:
match_opcodes = {
    instruction.opname
    for matcher in (run_deploy_command, route_webhook)
    for instruction in dis.get_instructions(matcher)
    if instruction.opname.startswith("MATCH_")
}
print(f"{sorted(match_opcodes)=}")

sorted(match_opcodes)=['MATCH_CLASS', 'MATCH_KEYS', 'MATCH_MAPPING', 'MATCH_SEQUENCE']


### 1.8.5 Loops

Python has two loops. `for` iterates over **any iterable** (list, tuple, str, dict,
set, range, file, generator); there is no C-style `for (init; cond; step)`. For
counting, iterate over a `range`.

In [435]:
for product_name in ["tea", "rice"]:
    print(f"list item: {product_name}")
for character in "OK":
    print(f"str character: {character}")
for sku, unit_price in {"TEA-500G": 249, "GHEE-1L": 640}.items():
    print(f"dict item: {sku} = {unit_price}")
for even_number in range(0, 7, 2):  # JS: for (let n = 0; n < 7; n += 2)
    print(f"range value: {even_number}")
print(f"{even_number=}  (the loop variable still exists after the loop)")

list item: tea
list item: rice
str character: O
str character: K
dict item: TEA-500G = 249
dict item: GHEE-1L = 640
range value: 0
range value: 2
range value: 4
range value: 6
even_number=6  (the loop variable still exists after the loop)


`while condition:` repeats while the condition is truthy, for loops where the number
of iterations is not known in advance.

In [436]:
remaining_stock = 10
orders_served = 0
while remaining_stock >= 3:
    remaining_stock -= 3
    orders_served += 1
print(f"{orders_served=}, {remaining_stock=}")

orders_served=3, remaining_stock=1


`break` leaves the innermost loop immediately; `continue` skips to the next iteration.

In [437]:
log_lines = ["INFO start", "DEBUG cache warm", "ERROR db down", "INFO retry"]
for log_line in log_lines:
    if log_line.startswith("DEBUG"):
        continue  # skip noise
    if log_line.startswith("ERROR"):
        print(f"stopping at: {log_line}")
        break
    print(f"processing: {log_line}")

processing: INFO start
stopping at: ERROR db down


**`for ... else`**: the `else` block runs when the loop finishes **without `break`**.
Read it as "no break". It replaces the `found = False` flag pattern for searches.

In [438]:
def find_first_error(lines: list[str]) -> str:
    """Return the first ERROR line, or a message if there is none."""
    for line in lines:
        if line.startswith("ERROR"):
            result = f"found: {line}"
            break
    else:  # no break: the loop ran to the end
        result = "no errors"
    return result


print(f"{find_first_error(log_lines)=}")
print(f"{find_first_error(['INFO ok'])=}")
print(f"{find_first_error([])=}  (empty loop: else still runs)")

find_first_error(log_lines)='found: ERROR db down'
find_first_error(['INFO ok'])='no errors'
find_first_error([])='no errors'  (empty loop: else still runs)


**`while ... else`** works the same way: `else` runs when the condition becomes false,
not when the loop is left with `break`. The example retries a flaky operation.

In [439]:
def connect_with_retries(responses: list[bool], max_attempts: int) -> str:
    """Try up to max_attempts times; responses[i] is the result of attempt i."""
    attempt = 0
    while attempt < max_attempts:
        if responses[attempt]:
            outcome = f"connected on attempt {attempt + 1}"
            break
        attempt += 1
    else:
        outcome = f"gave up after {max_attempts} attempts"
    return outcome


print(f"{connect_with_retries([False, True, True], 3)=}")
print(f"{connect_with_retries([False, False, False], 3)=}")

connect_with_retries([False, True, True], 3)='connected on attempt 2'
connect_with_retries([False, False, False], 3)='gave up after 3 attempts'


**Do-while emulation.** Python has no `do { ... } while (cond)`. Use `while True:`
with a `break` at the bottom, so the body always runs at least once.

In [440]:
pending_pages = ["page-3", "page-2", "page-1"]
fetched_pages = []
while True:
    fetched_pages.append(pending_pages.pop())  # the body runs at least once
    if not pending_pages:
        break
print(f"{fetched_pages=}")

fetched_pages=['page-1', 'page-2', 'page-3']


### 1.8.6 Assignment forms

| Form | Example | Notes |
| --- | --- | --- |
| Simple | `total = 0` | binds a name |
| Augmented | `total += price` | in place for mutable objects, rebinding for immutable ones |
| Multiple targets | `low = high = 0` | every name gets the **same** object |
| Unpacking | `name, price = row` | counts must match (or use `*`) |
| Walrus | `if (match := pattern.search(line)):` | an assignment **expression** (PEP 572) |

**Augmented assignment** (`+=`, `-=`, `*=`, `/=`, `//=`, `%=`, `**=`, `&=`, `|=`,
`^=`, `<<=`, `>>=`, `@=`) calls the in-place method (`__iadd__`, …) if the type has
one, so a list is changed **in place**; immutable types have none and get a new
object. There is no `++` / `--`.

In [441]:
running_total = 100
running_total += 49
running_total //= 2
print(f"{running_total=}")

cart = ["tea"]
cart_alias = cart
cart += ["rice"]  # list.__iadd__: extends the same list
print(f"{cart_alias=}, {cart is cart_alias=}")

point = (1, 2)
point_alias = point
point += (3,)  # tuples have no __iadd__: a new tuple is bound to point
print(f"{point_alias=}, {point is point_alias=}")
print(f"{compile_error('count = 0; count++')=}")

running_total=74
cart_alias=['tea', 'rice'], cart is cart_alias=True
point_alias=(1, 2), point is point_alias=False
compile_error('count = 0; count++')='SyntaxError: invalid syntax'


**Multiple targets**: `a = b = value` evaluates `value` once and binds every target to
**that one object**. Harmless for immutable values, a trap for mutable ones (Lab 1.5,
exercise 4).

In [442]:
low_score = high_score = 0
high_score += 10  # rebinding high_score; low_score keeps 0
print(f"{low_score=}, {high_score=}")

pending_ids = done_ids = []
pending_ids.append(1001)  # one list, two names
print(f"{done_ids=}  (the trap)")

low_score=0, high_score=10
done_ids=[1001]  (the trap)


**Unpacking assignment**: any iterable on the right, a matching number of targets on
the left. It swaps without a temporary, and nests.

In [443]:
sku, unit_price = ("TEA-500G", 249)
print(f"{sku=}, {unit_price=}")
first_slot, second_slot = "morning", "evening"
first_slot, second_slot = second_slot, first_slot  # swap: the right side is a tuple
print(f"{first_slot=}, {second_slot=}")
store_name, (latitude, longitude) = ("Indiranagar", (12.97, 77.64))
print(f"{store_name=}, {latitude=}, {longitude=}")
print(f"too many values: {raised_error_name(lambda: exec('a, b = 1, 2, 3'))}")

sku='TEA-500G', unit_price=249
first_slot='evening', second_slot='morning'
store_name='Indiranagar', latitude=12.97, longitude=77.64
too many values: ValueError


**The walrus operator `:=`** (PEP 572, Python 3.8) assigns **and** returns the value
inside an expression. It avoids computing something twice, or splitting "compute,
then test" into two statements. Typical places: `while` conditions, `if` with a regex
match, and comprehension filters.

In [444]:
order_line_pattern = re.compile(r"(?P<sku>[A-Z]+-\w+) x(?P<quantity>\d+)")
raw_line = "order 1001: TEA-500G x2"
if (line_match := order_line_pattern.search(raw_line)) is not None:
    print(f"{line_match['sku']=}, {int(line_match['quantity'])=}")

pending_jobs = ["resize", "upload", "notify"]
while (current_job := pending_jobs.pop(0) if pending_jobs else None) is not None:
    print(f"running {current_job}")

prices = [120, 499, 999, 1500]
with_gst = [taxed for price in prices if (taxed := round(price * 1.18)) > 600]
print(f"{with_gst=}")

line_match['sku']='TEA-500G', int(line_match['quantity'])=2
running resize
running upload
running notify
with_gst=[1179, 1770]


**Walrus scope rules:**

- In a comprehension, `:=` binds in the **enclosing** scope (on purpose, so you can
  keep the last value or a "witness"); the comprehension's own loop variable does not
  leak (1.9.3).
- It cannot rebind a comprehension's iteration variable.
- An unparenthesized `:=` is not allowed as a top-level statement (use `=`), and it
  is not allowed for attributes or subscripts (`obj.x := 1`).

In [445]:
order_amounts = [250, 900, 1200]
has_large = any((large_amount := candidate) > 1000 for candidate in order_amounts)
print(f"{has_large=}, {large_amount=}  (the witness leaked out on purpose)")
print(f"'candidate' defined outside: {'candidate' in dir()}")

print(f"{compile_error('total := 5')=}")
print(f"{compile_error('[i := 0 for i in range(3)]')=}")
print(f"{compile_error('cart.total := 5')=}")
print(f"{compile_error('(total := 5)')=}  (parenthesized is legal, but use = instead)")

has_large=True, large_amount=1200  (the witness leaked out on purpose)
'candidate' defined outside: True
compile_error('total := 5')='SyntaxError: invalid syntax'
compile_error('[i := 0 for i in range(3)]')="SyntaxError: assignment expression cannot rebind comprehension iteration variable 'i'"
compile_error('cart.total := 5')='SyntaxError: cannot use assignment expressions with attribute'
compile_error('(total := 5)')='compiles'  (parenthesized is legal, but use = instead)


### 1.8.7 Unpacking operators

`*` unpacks any iterable into positional items; `**` unpacks a mapping into
`key=value` pairs. They work in three places:

| Place | `*` | `**` |
| --- | --- | --- |
| Calls | `f(*args)` → positional arguments | `f(**options)` → keyword arguments |
| Literals | `[*a, *b]`, `(*a, x)`, `{*a, *b}` | `{**defaults, **overrides}` |
| Assignment targets | `first, *rest = items` | — |

**In calls**:

In [446]:
def format_address(street: str, city: str, pin_code: str, country: str = "IN") -> str:
    """Join address parts into one line."""
    return f"{street}, {city} {pin_code}, {country}"


address_parts = ["12 MG Road", "Bengaluru", "560001"]
address_options = {"country": "India"}
print(format_address(*address_parts))
print(format_address(*address_parts, **address_options))
print(*["tea", "rice", "ghee"], sep=" | ")

12 MG Road, Bengaluru 560001, IN
12 MG Road, Bengaluru 560001, India
tea | rice | ghee


**In list, tuple, set and dict literals**: build a new container from several
sources (the JS spread `[...a, ...b]` and `{...a, ...b}`). In dicts, later keys win.

In [447]:
breakfast_items, lunch_items = ["idli", "dosa"], ("thali", "dosa")
all_items = [*breakfast_items, *lunch_items, "chai"]
unique_items = {*breakfast_items, *lunch_items}
print(f"{all_items=}")
print(f"{sorted(unique_items)=}, {(*breakfast_items, 'vada')=}")

default_options = {"timeout": 10, "retries": 3}
request_options = {**default_options, "timeout": 30, "verbose": True}
print(f"{request_options=}")

all_items=['idli', 'dosa', 'thali', 'dosa', 'chai']
sorted(unique_items)=['dosa', 'idli', 'thali'], (*breakfast_items, 'vada')=('idli', 'dosa', 'vada')
request_options={'timeout': 30, 'retries': 3, 'verbose': True}


**In assignment targets**: one starred name collects "everything else" into a
**list** (even when the source is a tuple or a string). At most one starred target is
allowed per level.

In [448]:
first_step, *middle_steps, last_step = ("cart", "address", "payment", "confirm")
print(f"{first_step=}, {middle_steps=}, {last_step=}")
head_char, *tail_chars = "UPI"
print(f"{head_char=}, {tail_chars=}")
*_, latest_version = ["v1.0.0", "v1.1.0", "v2.0.0"]
print(f"{latest_version=}")
only_item, *nothing_else = ["solo"]
print(f"{nothing_else=}  (an empty list, not an error)")
print(f"{compile_error('*first, *second = [1, 2, 3]')=}")

first_step='cart', middle_steps=['address', 'payment'], last_step='confirm'
head_char='U', tail_chars=['P', 'I']
latest_version='v2.0.0'
nothing_else=[]  (an empty list, not an error)
compile_error('*first, *second = [1, 2, 3]')='SyntaxError: multiple starred expressions in assignment'


**Unpacking in comprehensions** (*New in 3.15*, PEP 798): `[*batch for batch in
batches]` flattens one level, and `{**layer for layer in layers}` merges dicts. On
older versions it is a `SyntaxError`, so the cell compiles the expressions from
strings behind the version check.

In [449]:
order_batches = [["TEA-500G", "RICE-5KG"], ["GHEE-1L"], []]
config_layers = [{"timeout": 10}, {"timeout": 30, "debug": True}]
if sys.version_info >= (3, 15):
    flat_skus = eval("[*batch for batch in order_batches]")
    merged_config = eval("{**layer for layer in config_layers}")
    print(f"{flat_skus=}")
    print(f"{merged_config=}")
else:
    print("unpacking in comprehensions needs Python 3.15; skipped")
flat_skus_classic = [sku for batch in order_batches for sku in batch]
print(f"{flat_skus_classic=}  (the pre-3.15 spelling)")

flat_skus=['TEA-500G', 'RICE-5KG', 'GHEE-1L']
merged_config={'timeout': 30, 'debug': True}
flat_skus_classic=['TEA-500G', 'RICE-5KG', 'GHEE-1L']  (the pre-3.15 spelling)


### Lab 1.8

Three exercises. For each, replace the `raise NotImplementedError` in the stub, then
run the check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — `parse_command(command_text)` with `match`

Parse a shopping-cart command line into a tuple. Split on whitespace and use one
`match` statement:

| Input | Result |
| --- | --- |
| `"add SKU"` | `("add", "SKU", 1)` |
| `"add SKU QTY"` (QTY all digits and > 0) | `("add", "SKU", QTY as int)` |
| `"add SKU QTY"` (anything else) | `("error", "quantity must be a positive integer")` |
| `"remove SKU"` | `("remove", "SKU")` |
| `"list"` or `"ls"` | `("list",)` |
| `"checkout"` | `("checkout",)` |
| `"quit"` or `"exit"` | `("quit",)` |
| blank line | `("error", "empty command")` |
| anything else | `("error", "unknown command: <first word>")` |

Commands are case-sensitive; SKUs are returned as typed.

In [450]:
def parse_command(command_text: str) -> tuple[Any, ...]:
    """Parse a cart command into a tuple, using match."""
    raise NotImplementedError("Lab 1.8: implement parse_command()")

Run the checks.

In [451]:
run_checks(
    [
        (
            "add default quantity",
            lambda: parse_command("add TEA-500G"),
            ("add", "TEA-500G", 1),
        ),
        ("add with quantity", lambda: parse_command("add RICE-5KG 3"), ("add", "RICE-5KG", 3)),
        ("extra spaces", lambda: parse_command("  add   GHEE-1L   2 "), ("add", "GHEE-1L", 2)),
        (
            "bad quantity",
            lambda: parse_command("add TEA-500G two"),
            ("error", "quantity must be a positive integer"),
        ),
        (
            "zero quantity",
            lambda: parse_command("add TEA-500G 0"),
            ("error", "quantity must be a positive integer"),
        ),
        ("remove", lambda: parse_command("remove TEA-500G"), ("remove", "TEA-500G")),
        ("list", lambda: parse_command("list"), ("list",)),
        ("ls alias", lambda: parse_command("ls"), ("list",)),
        ("checkout", lambda: parse_command("checkout"), ("checkout",)),
        ("quit", lambda: parse_command("quit"), ("quit",)),
        ("exit alias", lambda: parse_command("exit"), ("quit",)),
        ("blank", lambda: parse_command("   "), ("error", "empty command")),
        ("unknown", lambda: parse_command("dance now"), ("error", "unknown command: dance")),
        (
            "remove without sku",
            lambda: parse_command("remove"),
            ("error", "unknown command: remove"),
        ),
    ]
)

FAIL  add default quantity: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  add with quantity: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  extra spaces: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  bad quantity: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  zero quantity: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  remove: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  list: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  ls alias: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  checkout: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  quit: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  exit alias: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  blank: raised NotImplementedError: Lab 1.8: implement parse_command()
FAIL  unknown: raised NotImpl

#### Exercise 2 — `fizzbuzz(number)` with `match`

Return `"FizzBuzz"` for multiples of 15, `"Fizz"` for multiples of 3, `"Buzz"` for
multiples of 5, otherwise the number as a string. Match on the **tuple**
`(number % 3, number % 5)`: one `case` per outcome, no `if`.

In [452]:
def fizzbuzz(number: int) -> str:
    """Return the FizzBuzz word for number, using match on a tuple of remainders."""
    raise NotImplementedError("Lab 1.8: implement fizzbuzz()")

Run the checks.

In [453]:
run_checks(
    [
        ("1", lambda: fizzbuzz(1), "1"),
        ("3", lambda: fizzbuzz(3), "Fizz"),
        ("5", lambda: fizzbuzz(5), "Buzz"),
        ("15", lambda: fizzbuzz(15), "FizzBuzz"),
        ("0 is a multiple of 15", lambda: fizzbuzz(0), "FizzBuzz"),
        (
            "1 to 15",
            lambda: " ".join(fizzbuzz(number) for number in range(1, 16)),
            "1 2 Fizz 4 Buzz Fizz 7 8 Fizz Buzz 11 Fizz 13 14 FizzBuzz",
        ),
    ]
)

FAIL  1: raised NotImplementedError: Lab 1.8: implement fizzbuzz()
FAIL  3: raised NotImplementedError: Lab 1.8: implement fizzbuzz()
FAIL  5: raised NotImplementedError: Lab 1.8: implement fizzbuzz()
FAIL  15: raised NotImplementedError: Lab 1.8: implement fizzbuzz()
FAIL  0 is a multiple of 15: raised NotImplementedError: Lab 1.8: implement fizzbuzz()
FAIL  1 to 15: raised NotImplementedError: Lab 1.8: implement fizzbuzz()
0/6 checks passed


#### Exercise 3 — `find_first_duplicate(items)` with `for ... else`

Return the first item that appears **for the second time** while scanning left to
right (so for `[3, 1, 4, 1, 3]` the answer is `1`, found at index 3, before the
second `3`). Return `None` if every item is unique. Use a set of seen items, `break`
when you find a duplicate, and put the "no duplicate" result in the loop's `else`.
Items are hashable; any iterable must work.

In [454]:
def find_first_duplicate(items: Iterable[Hashable]) -> Hashable | None:
    """Return the first item seen twice, or None, using for ... else."""
    raise NotImplementedError("Lab 1.8: implement find_first_duplicate()")

Run the checks.

In [455]:
run_checks(
    [
        ("second occurrence wins", lambda: find_first_duplicate([3, 1, 4, 1, 3]), 1),
        ("adjacent", lambda: find_first_duplicate(["a", "a"]), "a"),
        ("no duplicate", lambda: find_first_duplicate([1, 2, 3]), None),
        ("empty", lambda: find_first_duplicate([]), None),
        ("string iterable", lambda: find_first_duplicate("python"), None),
        ("generator", lambda: find_first_duplicate(n % 4 for n in range(10)), 0),
        ("1 and True are equal", lambda: find_first_duplicate([1, True]), True),
    ]
)

FAIL  second occurrence wins: raised NotImplementedError: Lab 1.8: implement find_first_duplicate()
FAIL  adjacent: raised NotImplementedError: Lab 1.8: implement find_first_duplicate()
FAIL  no duplicate: raised NotImplementedError: Lab 1.8: implement find_first_duplicate()
FAIL  empty: raised NotImplementedError: Lab 1.8: implement find_first_duplicate()
FAIL  string iterable: raised NotImplementedError: Lab 1.8: implement find_first_duplicate()
FAIL  generator: raised NotImplementedError: Lab 1.8: implement find_first_duplicate()
FAIL  1 and True are equal: raised NotImplementedError: Lab 1.8: implement find_first_duplicate()
0/7 checks passed


### Recap — Unit 1.8

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `/`, `//`, `%` | True division; floor division; `%` takes the divisor's sign | `/`, `Math.floor(a / b)`, `%` (dividend's sign) |
| `==` / `is` | Value equality without coercion / same object | `===` (no deep equality) / — |
| Bitwise | Arbitrary-precision ints; bind tighter than comparisons | 32-bit; `==` binds tighter than `&` |
| `in` | Calls `__contains__`: values in lists, substrings in str, keys in dicts | `includes`, `in` (keys) |
| Precedence | `**` > unary > `* /` > `+ -` > shifts > `&` > `^` > `\|` > comparisons > `not` > `and` > `or` | similar, but not identical |
| Chained comparisons | `0 < age <= 120`; middle evaluated once | `0 < age && age <= 120` |
| `and` / `or` | Short-circuit and return an operand | `&&` / `\|\|` |
| `value or default` | Replaces every falsy value, including `0` and `""` | `\|\|` (same trap) |
| No `??` | `x if x is not None else default` | `x ?? default` |
| Conditional expression | `a if condition else b` | `condition ? a : b` |
| Guard clauses | Handle special cases first and return early | same |
| `match` | Structural patterns, first match wins, no fall-through | `switch` with `===` and fall-through |
| Capture trap | A bare name in a case captures; use dotted names for constants | — |
| `for` | Iterates any iterable; counting via `range` | `for...of` |
| Loop `else` | Runs when the loop ends without `break` | no equivalent |
| Do-while | `while True: ... if not condition: break` | `do { } while ()` |
| Augmented assignment | In place for mutable types, rebinds immutable ones; no `++` | `+=`, `++` |
| Walrus `:=` | Assignment expression; binds in the enclosing scope from comprehensions | assignment is an expression in JS |
| `*` / `**` | Unpack in calls, literals and assignment targets; in comprehensions (3.15) | `...` spread / rest |

**Further reading**

- [Expressions: operator precedence](https://docs.python.org/3/reference/expressions.html#operator-precedence),
  [comparisons](https://docs.python.org/3/reference/expressions.html#comparisons),
  [boolean operations](https://docs.python.org/3/reference/expressions.html#boolean-operations)
- [Compound statements: `if`, `while`, `for`, `match`](https://docs.python.org/3/reference/compound_stmts.html)
- [Tutorial: `match` statements](https://docs.python.org/3/tutorial/controlflow.html#match-statements),
  [`break`, `continue` and `else` on loops](https://docs.python.org/3/tutorial/controlflow.html#break-and-continue-statements-and-else-clauses-on-loops)
- [Simple statements: assignment, augmented assignment](https://docs.python.org/3/reference/simple_stmts.html#assignment-statements),
  [assignment expressions](https://docs.python.org/3/reference/expressions.html#assignment-expressions)
- [PEP 634 — Structural pattern matching: specification](https://peps.python.org/pep-0634/),
  [PEP 635 — motivation and rationale](https://peps.python.org/pep-0635/),
  [PEP 636 — tutorial](https://peps.python.org/pep-0636/)
- [PEP 572 — Assignment expressions](https://peps.python.org/pep-0572/),
  [PEP 448 — Additional unpacking generalizations](https://peps.python.org/pep-0448/),
  [PEP 3132 — Extended iterable unpacking](https://peps.python.org/pep-3132/),
  [PEP 798 — Unpacking in comprehensions](https://peps.python.org/pep-0798/),
  [PEP 505 — None-aware operators (deferred)](https://peps.python.org/pep-0505/)

## Unit 1.9 — Iteration, Comprehensions and Built-in Functions

**Objectives**

- Explain the iteration protocol (`iter`, `next`, `StopIteration`) and desugar a
  `for` loop by hand.
- Use `enumerate`, `zip`, `reversed`, `sorted` and `range` instead of index
  arithmetic.
- Write list, dict and set comprehensions, and know when a plain loop reads better.
- Know every commonly used built-in function, grouped by purpose, and which ones are
  dangerous.
- Choose between `map` / `filter` and comprehensions, and write idiomatic loops.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `for (const x of items)` | `for item in items` | Same idea: both use an iteration protocol |
| `obj[Symbol.iterator]()` | `iter(obj)` → calls `obj.__iter__()` | |
| `it.next()` → `{value, done}` | `next(it)` → value, or raises `StopIteration` | The end is an exception, not a flag |
| `items.map(f)`, `items.filter(f)` | `map(f, items)`, `filter(f, items)` | Python's are **lazy, single-use iterators**, not arrays |
| `items.map(...).filter(...)` | `[f(x) for x in items if cond(x)]` | Comprehensions are the idiomatic form |
| `items.entries()` | `enumerate(items)` | Gives `(index, item)` pairs; `start=` sets the first index |
| lodash `zip` | `zip(a, b, strict=True)` | Built in; truncates silently without `strict=True` |
| `items.some(f)`, `items.every(f)` | `any(...)`, `all(...)` | Take any iterable of truthy values |
| `Math.max(...items)` | `max(items, key=..., default=...)` | Works on any comparable values; `default` for empty input |
| `typeof x`, `x instanceof C` | `type(x)`, `isinstance(x, C)` | |
| `eval(code)` | `eval(code)` | Just as dangerous; parse data with `ast.literal_eval` or `json.loads` |

The next cell is this unit's setup.

In [456]:
import ast
import datetime as dt
import dis
import io
import operator
import subprocess
import sys
import timeit
from collections.abc import Callable, Iterator
from decimal import Decimal
from pathlib import Path
from typing import IO, Any, NamedTuple


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


scratch_dir = find_repo_root() / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.9.1 The iteration protocol

Two roles:

| Role | Has | Example |
| --- | --- | --- |
| **Iterable** | `__iter__()` returning a new iterator | `list`, `str`, `dict`, `range`, files |
| **Iterator** | `__next__()` returning the next item or raising `StopIteration`, and `__iter__()` returning itself | `iter([1, 2])`, generators, `map`, `zip` |

`iter(obj)` calls `obj.__iter__()`; `next(it)` calls `it.__next__()`. When the items
run out, `__next__` raises `StopIteration`. The next cell walks a list by hand.

In [457]:
queue_names = ["asha", "ravi"]
name_iterator = iter(queue_names)
print(f"{type(name_iterator).__name__=}")
print(f"{next(name_iterator)=}")
print(f"{next(name_iterator)=}")
try:
    next(name_iterator)
except StopIteration:
    print("StopIteration: the iterator is exhausted")
print(f"{list(name_iterator)=}  (an exhausted iterator stays empty)")
print(f"{iter(name_iterator) is name_iterator=}, {iter(queue_names) is queue_names=}")

type(name_iterator).__name__='list_iterator'
next(name_iterator)='asha'
next(name_iterator)='ravi'
StopIteration: the iterator is exhausted
list(name_iterator)=[]  (an exhausted iterator stays empty)
iter(name_iterator) is name_iterator=True, iter(queue_names) is queue_names=False


**The desugared `for` loop.** `for item in iterable: body` is exactly this:

```python
hidden_iterator = iter(iterable)
while True:
    try:
        item = next(hidden_iterator)
    except StopIteration:
        break
    body
```

The next cell runs both versions side by side.

In [458]:
order_totals = [120, 499, 999]

for order_total in order_totals:
    print(f"for loop:     {order_total}")

hidden_iterator = iter(order_totals)
while True:
    try:
        order_total = next(hidden_iterator)
    except StopIteration:
        break
    print(f"desugared:    {order_total}")

for loop:     120
for loop:     499
for loop:     999
desugared:    120
desugared:    499
desugared:    999


Any class can join the protocol. `Countdown` is an iterable: each `iter()` call
returns a fresh `CountdownIterator`, so it can be looped over more than once.
(Unit 1.13 writes the same thing in three lines with a generator.)

In [459]:
class CountdownIterator:
    """Iterator from start down to 1."""

    def __init__(self, start: int) -> None:
        self.current = start

    def __iter__(self) -> CountdownIterator:
        return self

    def __next__(self) -> int:
        if self.current < 1:
            raise StopIteration
        self.current -= 1
        return self.current + 1


class Countdown:
    """Iterable: a new iterator per loop."""

    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self) -> CountdownIterator:
        return CountdownIterator(self.start)


launch = Countdown(3)
print(f"{list(launch)=}, {list(launch)=}  (iterable: reusable)")
single_pass = iter(launch)
print(f"{list(single_pass)=}, {list(single_pass)=}  (iterator: one pass)")

list(launch)=[3, 2, 1], list(launch)=[3, 2, 1]  (iterable: reusable)
list(single_pass)=[3, 2, 1], list(single_pass)=[]  (iterator: one pass)


#### Under the hood: `GET_ITER` and `FOR_ITER`

In bytecode, a `for` loop is `GET_ITER` (call `iter()` once) followed by `FOR_ITER`
(call `next()`; on `StopIteration`, jump past the loop body). The `StopIteration` is
handled inside the interpreter, so no Python-level exception is raised for the normal
end of a loop. Opcode names vary between versions.

In [460]:
def total_of(prices: list[int]) -> int:
    """Sum prices with an explicit loop."""
    running_total = 0
    for price in prices:
        running_total += price
    return running_total


print([instruction.opname for instruction in dis.get_instructions(total_of)])

['RESUME', 'LOAD_SMALL_INT', 'STORE_FAST', 'LOAD_FAST', 'GET_ITER', 'FOR_ITER', 'STORE_FAST', 'LOAD_FAST_BORROW_LOAD_FAST_BORROW', 'BINARY_OP', 'STORE_FAST', 'JUMP_BACKWARD', 'END_FOR', 'POP_ITER', 'LOAD_FAST_BORROW', 'RETURN_VALUE']


**`iter(callable, sentinel)`**: the two-argument form calls `callable()` again and
again and stops when the result **equals** `sentinel`. It turns "read until empty"
loops into iterators. `stream.readline` returns `""` at the end of a text stream:

In [461]:
access_log = io.StringIO("GET /\nPOST /cart\nGET /checkout\n")
for log_line in iter(access_log.readline, ""):
    print(f"line: {log_line.rstrip()}")

line: GET /
line: POST /cart
line: GET /checkout


With a `lambda` (1.10.7) the callable can pass arguments, which gives **fixed-size
chunks**: read 4 bytes at a time until `read` returns `b""`. Lab 1.9, exercise 1,
wraps this in a reusable function.

In [462]:
upload_stream = io.BytesIO(b"PNG-image-bytes")
chunk_list = list(iter(lambda: upload_stream.read(4), b""))
print(f"{chunk_list=}")

chunk_list=[b'PNG-', b'imag', b'e-by', b'tes']

### 1.9.2 Iteration helpers

Five built-ins remove almost every need for index arithmetic. Each returns a lazy
iterator except `sorted` (a new list) and `range` (a lazy sequence object).

**`enumerate(iterable, start=0)`** yields `(index, item)` pairs; `start=1` gives
human numbering.

In [463]:
checkout_steps = ["cart", "address", "payment"]
for step_number, step_name in enumerate(checkout_steps, start=1):
    print(f"step {step_number}: {step_name}")
print(f"{list(enumerate('ab'))=}")

step 1: cart
step 2: address
step 3: payment
list(enumerate('ab'))=[(0, 'a'), (1, 'b')]


**`zip(*iterables, strict=False)`** pairs items up. By default it stops at the
**shortest** input and silently drops the rest; `strict=True` (3.10) raises
`ValueError` when the lengths differ, which is almost always what you want.

In [464]:
product_names = ["tea", "rice", "ghee"]
unit_prices = [249, 525]
print(f"{list(zip(product_names, unit_prices, strict=False))=}  (ghee lost silently)")
print(f"strict: {raised_error_name(lambda: list(zip(product_names, unit_prices, strict=True)))}")
rows = [("tea", 249), ("rice", 525)]
names_column, prices_column = zip(*rows, strict=True)  # zip(*rows) transposes
print(f"{names_column=}, {prices_column=}")

list(zip(product_names, unit_prices, strict=False))=[('tea', 249), ('rice', 525)]  (ghee lost silently)
strict: ValueError
names_column=('tea', 'rice'), prices_column=(249, 525)


**`reversed(sequence)`** iterates backwards without copying. It needs a sequence
(`len` + indexing) or a `__reversed__` method; dicts and their views support it since
3.8. A generator does not.

In [465]:
print(f"{list(reversed([1, 2, 3]))=}, {''.join(reversed('UPI'))=}")
print(f"{list(reversed({'a': 1, 'b': 2}))=}, {list(reversed(range(3)))=}")
print(f"generator: {raised_error_name(lambda: reversed(n for n in range(3)))}")

list(reversed([1, 2, 3]))=[3, 2, 1], ''.join(reversed('UPI'))='IPU'
list(reversed({'a': 1, 'b': 2}))=['b', 'a'], list(reversed(range(3)))=[2, 1, 0]
generator: TypeError


**`sorted(iterable, *, key=None, reverse=False)`** accepts any iterable and returns a
**new list**; the input is not changed (1.5.4 covers sorting in depth).

In [466]:
stock_levels = {"tea": 5, "rice": 12, "ghee": 2}
print(f"{sorted(stock_levels)=}  (keys)")
print(f"{sorted(stock_levels, key=stock_levels.get, reverse=True)=}  (by stock)")
print(f"{sorted('python')=}, {sorted({3, 1, 2})=}")

sorted(stock_levels)=['ghee', 'rice', 'tea']  (keys)
sorted(stock_levels, key=stock_levels.get, reverse=True)=['rice', 'tea', 'ghee']  (by stock)
sorted('python')=['h', 'n', 'o', 'p', 't', 'y'], sorted({3, 1, 2})=[1, 2, 3]


**`range(start, stop, step)`** is a lazy, immutable sequence of ints: constant memory,
`len`, indexing, slicing and O(1) `in` (1.5.7).

In [467]:
invoice_numbers = range(1000, 2000, 10)
print(f"{len(invoice_numbers)=}, {invoice_numbers[3]=}, {1500 in invoice_numbers=}")
print(f"{invoice_numbers[:3]=}, {list(range(5, 0, -2))=}")

len(invoice_numbers)=100, invoice_numbers[3]=1030, 1500 in invoice_numbers=True
invoice_numbers[:3]=range(1000, 1030, 10), list(range(5, 0, -2))=[5, 3, 1]


### 1.9.3 Comprehensions

A comprehension builds a container from an iterable in one expression:

| Kind | Syntax | Result |
| --- | --- | --- |
| List | `[expression for item in iterable if condition]` | `list` |
| Dict | `{key: value for item in iterable if condition}` | `dict` |
| Set | `{expression for item in iterable if condition}` | `set` |
| Generator expression | `(expression for item in iterable)` | lazy iterator (1.13) |

**List comprehension**, the replacement for `.map()` (and the `append` loop):

In [468]:
prices_in_rupees = [120, 499, 999]
prices_with_gst = [round(price * 1.18, 2) for price in prices_in_rupees]
print(f"{prices_with_gst=}")

prices_with_gst=[141.6, 588.82, 1178.82]


**Filters**: an `if` clause keeps only matching items (the `.filter()` part).

In [469]:
order_totals = [120, 1499, 650, 2300]
free_shipping_orders = [total for total in order_totals if total >= 999]
print(f"{free_shipping_orders=}")
labels = ["big" if total >= 999 else "small" for total in order_totals]
print(f"{labels=}  (a conditional *expression* in the output part is different)")

free_shipping_orders=[1499, 2300]
labels=['small', 'big', 'small', 'big']  (a conditional *expression* in the output part is different)


**Dict comprehension**: build a mapping, for example an index or an inverted dict.

In [470]:
product_names = ["tea", "rice", "ghee"]
name_lengths = {name: len(name) for name in product_names}
sku_by_name = {"tea": "TEA-500G", "rice": "RICE-5KG"}
name_by_sku = {sku: name for name, sku in sku_by_name.items()}
print(f"{name_lengths=}")
print(f"{name_by_sku=}")

name_lengths={'tea': 3, 'rice': 4, 'ghee': 4}
name_by_sku={'TEA-500G': 'tea', 'RICE-5KG': 'rice'}


**Set comprehension**: unique results (printed sorted, see 1.6 on string-set order).

In [471]:
customer_emails = ["Asha@Mail.in", "asha@mail.in", "ravi@mail.in"]
unique_emails = {email.lower() for email in customer_emails}
print(f"{sorted(unique_emails)=}")

sorted(unique_emails)=['asha@mail.in', 'ravi@mail.in']


**Nested `for` order.** Several `for` clauses read **left to right, outer to inner**,
exactly like the nested loops they replace:

```python
[cell for row in grid for cell in row]
# is
for row in grid:
    for cell in row:
        result.append(cell)
```

In [472]:
sizes, colours = ["S", "M"], ["red", "blue"]
variants = [f"{size}-{colour}" for size in sizes for colour in colours]
print(f"{variants=}")

seat_grid = [[1, 2, 3], [4, 5, 6]]
flat_seats = [seat for row in seat_grid for seat in row]
even_seats = [seat for row in seat_grid for seat in row if seat % 2 == 0]
print(f"{flat_seats=}, {even_seats=}")

variants=['S-red', 'S-blue', 'M-red', 'M-blue']
flat_seats=[1, 2, 3, 4, 5, 6], even_seats=[2, 4, 6]


**Own scope.** A comprehension's loop variable does **not** leak into the surrounding
code (a plain `for` loop's variable does). Since Python 3.12 (PEP 709) comprehensions
are inlined into the enclosing function for speed, but the isolation is kept.

In [473]:
comprehension_item = "before"
doubled_items = [comprehension_item * 2 for comprehension_item in "ab"]
print(f"{doubled_items=}, {comprehension_item=}  (unchanged)")

for loop_item in "ab":  # noqa: B007
    pass
print(f"{loop_item=}  (a for loop's variable survives the loop)")

doubled_items=['aa', 'bb'], comprehension_item='before'  (unchanged)
loop_item='b'  (a for loop's variable survives the loop)


#### Under the hood: inlined comprehensions (PEP 709)

Before 3.12, every comprehension compiled to a hidden function that was created and
called each time. Now the bytecode sits inline; `LOAD_FAST_AND_CLEAR` saves any
outer variable with the same name and restores it afterwards, which is how the
isolation survives. There is no `MAKE_FUNCTION` in the output.

In [474]:
def doubled(values: list[int]) -> list[int]:
    """Double every value with a comprehension."""
    return [value * 2 for value in values]


doubled_opnames = [instruction.opname for instruction in dis.get_instructions(doubled)]
print(f"{'MAKE_FUNCTION' in doubled_opnames=}, {'LOAD_FAST_AND_CLEAR' in doubled_opnames=}")

'MAKE_FUNCTION' in doubled_opnames=False, 'LOAD_FAST_AND_CLEAR' in doubled_opnames=True


**Readability limits.** One `for`, maybe one `if`: a comprehension reads like a
sentence. With two loops, two conditions and a nested conditional expression it
stops being readable; a plain loop with names for each step is better. Both versions
below compute the same result.

In [475]:
daily_orders = [[120, 1500, -1], [999, 2300], [450]]

hard_to_read = [
    total * 0.9 if total > 1000 else total
    for day in daily_orders
    for total in day
    if total > 0
    if total != 999
]


def discounted_totals(days: list[list[int]]) -> list[float]:
    """Skip invalid and exact-999 totals; give 10% off totals over 1000."""
    result = []
    for day in days:
        for total in day:
            if total <= 0 or total == 999:
                continue
            result.append(total * 0.9 if total > 1000 else total)
    return result


print(f"{hard_to_read == discounted_totals(daily_orders)=}, {hard_to_read=}")

hard_to_read == discounted_totals(daily_orders)=True, hard_to_read=[120, 1350.0, 2070.0, 450]


### 1.9.4 Built-in functions catalogue

Python has about 70 built-in functions (`import builtins; dir(builtins)`). These are
the ones you use every day, grouped by purpose. Each gets its own demo cell below,
under a heading per group.

| Group | Function | Signature | Example | JS equivalent |
| --- | --- | --- | --- | --- |
| Numeric | `abs` | `abs(x)` | `abs(-5)` → `5` | `Math.abs` |
| Numeric | `round` | `round(number, ndigits=None)` | `round(2.5)` → `2` | `Math.round` (rounds half up) |
| Numeric | `divmod` | `divmod(a, b)` | `divmod(17, 5)` → `(3, 2)` | — |
| Numeric | `pow` | `pow(base, exp, mod=None)` | `pow(3, 4, 5)` → `1` | `**`, `Math.pow` |
| Numeric | `sum` | `sum(iterable, /, start=0)` | `sum([1, 2], start=10)` → `13` | `reduce((a, b) => a + b, 0)` |
| Numeric | `min` | `min(iterable, *, key=None, default=...)` | `min([], default=0)` → `0` | `Math.min(...a)` |
| Numeric | `max` | `max(iterable, *, key=None, default=...)` | `max(names, key=len)` | `Math.max(...a)` |
| Iteration | `len` | `len(obj)` | `len("tea")` → `3` | `.length`, `.size` |
| Iteration | `iter` | `iter(obj)`, `iter(callable, sentinel)` | `iter([1, 2])` | `a[Symbol.iterator]()` |
| Iteration | `next` | `next(iterator, default)` | `next(it, None)` | `it.next().value` |
| Iteration | `map` | `map(function, *iterables, strict=False)` | `map(str.upper, names)` | `a.map(f)` (eager) |
| Iteration | `filter` | `filter(function, iterable)` | `filter(None, items)` | `a.filter(f)` (eager) |
| Iteration | `any` | `any(iterable)` | `any(x > 3 for x in a)` | `a.some(f)` |
| Iteration | `all` | `all(iterable)` | `all([])` → `True` | `a.every(f)` |
| Iteration | `enumerate` | `enumerate(iterable, start=0)` | `enumerate(a, start=1)` | `a.entries()` |
| Iteration | `zip` | `zip(*iterables, strict=False)` | `zip(a, b, strict=True)` | lodash `zip` |
| Iteration | `reversed` | `reversed(sequence)` | `reversed([1, 2])` | `a.toReversed()` |
| Iteration | `sorted` | `sorted(iterable, *, key=None, reverse=False)` | `sorted(a, key=len)` | `a.toSorted(cmp)` |
| Conversion | `int` | `int(x=0)`, `int(text, base=10)` | `int("ff", 16)` → `255` | `parseInt(s, 16)` |
| Conversion | `float` | `float(x=0.0)` | `float("1e3")` → `1000.0` | `parseFloat`, `Number` |
| Conversion | `str` | `str(object="")` | `str(42)` → `'42'` | `String(x)` |
| Conversion | `bool` | `bool(x=False)` | `bool([])` → `False` | `Boolean(x)` |
| Conversion | `list` | `list(iterable=())` | `list("ab")` → `['a', 'b']` | `Array.from` |
| Conversion | `tuple` | `tuple(iterable=())` | `tuple([1, 2])` → `(1, 2)` | — |
| Conversion | `dict` | `dict(mapping_or_pairs, **kwargs)` | `dict(a=1)` → `{'a': 1}` | `Object.fromEntries` |
| Conversion | `set` | `set(iterable=())` | `set("aab")` → `{'a', 'b'}` | `new Set(a)` |
| Conversion | `frozenset` | `frozenset(iterable=())` | `frozenset({1, 2})` | — |
| Conversion | `bytes` | `bytes(source, encoding)` | `bytes("₹", "utf-8")` | `new TextEncoder().encode` |
| Conversion | `chr` | `chr(code_point)` | `chr(8377)` → `'₹'` | `String.fromCodePoint` |
| Conversion | `ord` | `ord(character)` | `ord("A")` → `65` | `s.codePointAt(0)` |
| Conversion | `bin` | `bin(integer)` | `bin(5)` → `'0b101'` | `n.toString(2)` |
| Conversion | `hex` | `hex(integer)` | `hex(255)` → `'0xff'` | `n.toString(16)` |
| Conversion | `oct` | `oct(integer)` | `oct(8)` → `'0o10'` | `n.toString(8)` |
| Conversion | `format` | `format(value, format_spec="")` | `format(0.25, ".0%")` → `'25%'` | `Intl.NumberFormat` |
| Conversion | `repr` | `repr(object)` | `repr("tea")` → `"'tea'"` | `JSON.stringify` (roughly) |
| Conversion | `ascii` | `ascii(object)` | `ascii("₹")` → `"'\u20b9'"` | — |
| Introspection | `type` | `type(obj)`, `type(name, bases, namespace)` | `type(3)` → `int` | `typeof` |
| Introspection | `isinstance` | `isinstance(obj, class_or_tuple)` | `isinstance(3, int \| float)` | `instanceof` |
| Introspection | `issubclass` | `issubclass(cls, class_or_tuple)` | `issubclass(bool, int)` → `True` | `C.prototype instanceof B` |
| Introspection | `id` | `id(obj)` | `id(a) == id(b)` | — |
| Introspection | `hash` | `hash(obj)` | `hash(1) == hash(1.0)` | — |
| Introspection | `callable` | `callable(obj)` | `callable(len)` → `True` | `typeof f === "function"` |
| Introspection | `dir` | `dir(obj)` | `dir("")` | `Object.getOwnPropertyNames` |
| Introspection | `vars` | `vars(obj)` | `vars(instance)` | (the object itself) |
| Introspection | `globals` | `globals()` | `"x" in globals()` | `globalThis` |
| Introspection | `locals` | `locals()` | `locals()` in a function | — |
| Introspection | `getattr` | `getattr(obj, name, default)` | `getattr(o, "x", 0)` | `o[name] ?? 0` |
| Introspection | `setattr` | `setattr(obj, name, value)` | `setattr(o, "x", 1)` | `o[name] = 1` |
| Introspection | `hasattr` | `hasattr(obj, name)` | `hasattr(o, "x")` | `name in o` |
| Introspection | `delattr` | `delattr(obj, name)` | `delattr(o, "x")` | `delete o[name]` |
| Introspection | `help` | `help(obj)` | `help(divmod)` | — (MDN) |
| I/O | `print` | `print(*objects, sep=" ", end="\n", file=None, flush=False)` | `print(a, b, sep=", ")` | `console.log` |
| I/O | `input` | `input(prompt="")` | `input("Name: ")` | `prompt()` / `readline` |
| I/O | `open` | `open(file, mode="r", encoding=None, ...)` | `open(p, encoding="utf-8")` | `fs.readFileSync` |
| Dangerous | `eval` | `eval(expression, globals=None, locals=None)` | `eval("2 + 3")` → `5` | `eval` |
| Dangerous | `exec` | `exec(source, globals=None, locals=None)` | `exec("x = 1", namespace)` | `new Function(...)()` |
| Dangerous | `compile` | `compile(source, filename, mode)` | `compile("x * 2", "<rule>", "eval")` | `new Function` |
| Safe alternative | `ast.literal_eval` | `ast.literal_eval(text)` | `literal_eval("[1, 2]")` | `JSON.parse` |

#### Numeric

**`abs(x)`**: absolute value; for a complex number, its magnitude. Works on any type
with `__abs__` (`Decimal`, `Fraction`).

In [476]:
print(f"{abs(-5)=}, {abs(-2.5)=}, {abs(3 + 4j)=}, {abs(Decimal('-19.99'))=}")

abs(-5)=5, abs(-2.5)=2.5, abs(3 + 4j)=5.0, abs(Decimal('-19.99'))=Decimal('19.99')


**`round(number, ndigits=None)`**: without `ndigits` it returns an `int`. Ties go to
the **even** neighbour (banker's rounding), unlike JavaScript's `Math.round`. Some
"ties" are not ties at all: `2.675` is stored as `2.67499999…` (1.3.2), so money
should use `Decimal.quantize` instead.

In [477]:
print(f"{round(2.5)=}, {round(3.5)=}, {round(-0.5)=}  (ties to even)")
print(f"{round(3.14159, 2)=}, {round(1234, -2)=}, {type(round(2.7)).__name__=}")
print(f"{round(2.675, 2)=}  (2.675 is really 2.67499999...)")

round(2.5)=2, round(3.5)=4, round(-0.5)=0  (ties to even)
round(3.14159, 2)=3.14, round(1234, -2)=1200, type(round(2.7)).__name__='int'
round(2.675, 2)=2.67  (2.675 is really 2.67499999...)


**`divmod(a, b)`**: `(a // b, a % b)` in one call, handy for unit conversions.

In [478]:
total_seconds = 3725
minutes, seconds = divmod(total_seconds, 60)
hours, minutes = divmod(minutes, 60)
print(f"{hours}h {minutes}m {seconds}s, {divmod(-7, 2)=}")

1h 2m 5s, divmod(-7, 2)=(-4, 1)


**`pow(base, exp, mod=None)`**: like `**`, plus a fast three-argument form for
modular arithmetic (`pow(base, -1, mod)` is the modular inverse, 3.8).

In [479]:
print(f"{pow(2, 10)=}, {pow(2, -1)=}, {pow(3, 4, 5)=}")
print(f"{pow(38, -1, 97)=}, {38 * pow(38, -1, 97) % 97=}")

pow(2, 10)=1024, pow(2, -1)=0.5, pow(3, 4, 5)=1
pow(38, -1, 97)=23, 38 * pow(38, -1, 97) % 97=1


**`sum(iterable, /, start=0)`**: adds numbers. `start` is the initial value: it sets
the result for an empty input and its type (here `Decimal("0.00")` keeps money as
`Decimal`). `sum` refuses strings; use `"".join(...)`.

In [480]:
line_totals = [Decimal("498.00"), Decimal("640.00")]
print(f"{sum(line_totals)=}, {sum([], start=Decimal('0.00'))=}, {sum([])=}")
print(f"{sum([1, 2, 3], start=10)=}")
print(f"strings: {raised_error_name(lambda: sum(['a', 'b'], start=''))}")

sum(line_totals)=Decimal('1138.00'), sum([], start=Decimal('0.00'))=Decimal('0.00'), sum([])=0
sum([1, 2, 3], start=10)=16
strings: TypeError


**`min(iterable, *, key=None, default=...)`** (or `min(a, b, ...)`): the smallest
item. `key` compares by a derived value and returns the original item; `default` is
returned for an empty iterable instead of raising `ValueError`.

In [481]:
stock_levels = {"tea": 5, "rice": 12, "ghee": 2}
print(f"{min(4, 9, 1)=}, {min(stock_levels, key=stock_levels.get)=}")
print(f"{min([], default=None)=}, empty: {raised_error_name(lambda: min([]))}")

min(4, 9, 1)=1, min(stock_levels, key=stock_levels.get)='ghee'
min([], default=None)=None, empty: ValueError


**`max(iterable, *, key=None, default=...)`**: the largest item; on ties, the **first** one wins.

In [482]:
product_names = ["tea", "rice", "ghee"]
print(f"{max(product_names, key=len)=}  (rice and ghee tie: first wins)")
print(f"{max([], default=0)=}, {max('apple', 'banana')=}")

max(product_names, key=len)='rice'  (rice and ghee tie: first wins)
max([], default=0)=0, max('apple', 'banana')='banana'


#### Iteration

**`len(obj)`**: the number of items, from `__len__`. O(1) for built-in containers.
Iterators and generators have no length.

In [483]:
print(f"{len('₹99')=}, {len([1, 2])=}, {len({'a': 1})=}, {len(range(0, 10, 3))=}")
print(f"generator: {raised_error_name(lambda: len(n for n in range(3)))}")

len('₹99')=3, len([1, 2])=2, len({'a': 1})=1, len(range(0, 10, 3))=4
generator: TypeError


**`iter(obj)` / `iter(callable, sentinel)`**: get an iterator (1.9.1); a non-iterable raises `TypeError`.

In [484]:
letter_iterator = iter("ab")
print(f"{next(letter_iterator)=}, {iter(letter_iterator) is letter_iterator=}")
print(f"iter(5): {raised_error_name(lambda: iter(5))}")

next(letter_iterator)='a', iter(letter_iterator) is letter_iterator=True
iter(5): TypeError


**`next(iterator, default)`**: the next item; with a default, it returns that instead
of raising `StopIteration`. `next(generator, None)` is the idiom for "first match or
None".

In [485]:
order_totals = [120, 1499, 650]
first_large = next((total for total in order_totals if total > 1000), None)
first_huge = next((total for total in order_totals if total > 5000), None)
print(f"{first_large=}, {first_huge=}")

first_large=1499, first_huge=None


**`map(function, *iterables, strict=False)`**: a lazy iterator of `function(item)`.
With several iterables it calls `function(a, b)`. `strict=True` is *New in 3.14*
(same meaning as in `zip`).

In [486]:
print(f"{list(map(str.upper, ['tea', 'rice']))=}")
print(f"{list(map(operator.mul, [249, 525], [2, 1]))=}")  # noqa: B912
if sys.version_info >= (3, 14):
    print(f"strict: {raised_error_name(lambda: list(map(pow, [2, 3], [3], strict=True)))}")
else:
    print("map(strict=True) needs Python 3.14; skipped")

list(map(str.upper, ['tea', 'rice']))=['TEA', 'RICE']
list(map(operator.mul, [249, 525], [2, 1]))=[498, 525]
strict: ValueError


**`filter(function, iterable)`**: a lazy iterator of items where `function(item)` is truthy; `filter(None, ...)` keeps truthy items.

In [487]:
raw_tags = ["new", "", "sale", None, "organic"]
print(f"{list(filter(None, raw_tags))=}")
print(f"{list(filter(str.isdigit, ['12', 'ab', '7']))=}")

list(filter(None, raw_tags))=['new', 'sale', 'organic']
list(filter(str.isdigit, ['12', 'ab', '7']))=['12', '7']


**`any(iterable)`**: `True` if any item is truthy; stops at the first one. `any([])` is `False`.

In [488]:
days_overdue = [0, 3, 45, 2]
print(f"{any(days > 30 for days in days_overdue)=}, {any([])=}")

any(days > 30 for days in days_overdue)=True, any([])=False


**`all(iterable)`**: `True` if every item is truthy; stops at the first falsy one. `all([])` is `True` (nothing fails).

In [489]:
required_fields = {"name": "Asha", "email": "asha@mail.in", "phone": ""}
print(f"{all(required_fields.values())=}, {all([])=}")

all(required_fields.values())=False, all([])=True


**`enumerate(iterable, start=0)`**: `(index, item)` pairs (1.9.2).

In [490]:
print(f"{list(enumerate(['tea', 'rice'], start=1))=}")

list(enumerate(['tea', 'rice'], start=1))=[(1, 'tea'), (2, 'rice')]


**`zip(*iterables, strict=False)`**: tuples of parallel items (1.9.2).

In [491]:
print(f"{dict(zip(['tea', 'rice'], [249, 525], strict=True))=}")

dict(zip(['tea', 'rice'], [249, 525], strict=True))={'tea': 249, 'rice': 525}


**`reversed(sequence)`**: a reverse iterator, no copy (1.9.2).

In [492]:
print(f"{list(reversed(['v1', 'v2', 'v3']))=}")

list(reversed(['v1', 'v2', 'v3']))=['v3', 'v2', 'v1']


**`sorted(iterable, *, key=None, reverse=False)`**: a new sorted list (1.9.2, 1.5.4).

In [493]:
print(f"{sorted(['Rice', 'tea', 'Ghee'], key=str.casefold)=}")

sorted(['Rice', 'tea', 'Ghee'], key=str.casefold)=['Ghee', 'Rice', 'tea']


#### Conversion

**`int(x)` / `int(text, base=10)`**: from a float it truncates toward zero; from text
it accepts surrounding whitespace and `_` separators, but not a decimal point.

In [494]:
print(f"{int('42')=}, {int(' 7 ')=}, {int('1_000')=}, {int('ff', 16)=}, {int('0b101', 0)=}")
print(f"{int(3.9)=}, {int(-3.9)=}  (truncates toward zero)")
print(f"int('3.5'): {raised_error_name(lambda: int('3.5'))}")

int('42')=42, int(' 7 ')=7, int('1_000')=1000, int('ff', 16)=255, int('0b101', 0)=5
int(3.9)=3, int(-3.9)=-3  (truncates toward zero)
int('3.5'): ValueError


**`float(x)`**: parses decimal and scientific notation, `'nan'` and `'inf'`.

In [495]:
print(f"{float('1e3')=}, {float(' 2.5 ')=}, {float('inf')=}, {float('nan')=}")
print(f"float('1,5'): {raised_error_name(lambda: float('1,5'))}")

float('1e3')=1000.0, float(' 2.5 ')=2.5, float('inf')=inf, float('nan')=nan
float('1,5'): ValueError


**`str(object)`**: the readable text form (`__str__`). Trap: `str(b'x')` gives `"b'x'"`; decode bytes instead.

In [496]:
print(f"{str(42)=}, {str(None)=}, {str(['tea'])=}")
print(f"{str(b'tea')=}  vs  {b'tea'.decode('utf-8')=}")

str(42)='42', str(None)='None', str(['tea'])="['tea']"
str(b'tea')="b'tea'"  vs  b'tea'.decode('utf-8')='tea'


**`bool(x)`**: truthiness (1.3.8). Trap: any non-empty string is `True`, including `'False'`.

In [497]:
print(f"{bool(0)=}, {bool([])=}, {bool('False')=}, {bool('0')=}")

bool(0)=False, bool([])=False, bool('False')=True, bool('0')=True


**`list(iterable)`**: a new list from any iterable.

In [498]:
print(f"{list('UPI')=}, {list(range(3))=}, {list({'a': 1, 'b': 2})=}")

list('UPI')=['U', 'P', 'I'], list(range(3))=[0, 1, 2], list({'a': 1, 'b': 2})=['a', 'b']

**`tuple(iterable)`**: a new tuple (an immutable snapshot, 1.7.7).

In [499]:
print(f"{tuple(['tea', 'rice'])=}, {tuple('ab')=}")

tuple(['tea', 'rice'])=('tea', 'rice'), tuple('ab')=('a', 'b')


**`dict(mapping_or_pairs, **kwargs)`**: from a mapping, from `(key, value)` pairs, or from keywords.

In [500]:
print(f"{dict([('tea', 249), ('rice', 525)])=}")
print(f"{dict(tea=249, rice=525)=}, {dict({'tea': 249}, ghee=640)=}")

dict([('tea', 249), ('rice', 525)])={'tea': 249, 'rice': 525}
dict(tea=249, rice=525)={'tea': 249, 'rice': 525}, dict({'tea': 249}, ghee=640)={'tea': 249, 'ghee': 640}


**`set(iterable)`**: unique elements (`set()` is the only way to write an empty set).

In [501]:
print(f"{sorted(set('banana'))=}, {set()=}, {type({}).__name__=}")

sorted(set('banana'))=['a', 'b', 'n'], set()=set(), type({}).__name__='dict'


**`frozenset(iterable)`**: an immutable, hashable set.

In [502]:
weekend_days = frozenset({"sat", "sun"})
print(f"{sorted(weekend_days)=}, {hash(weekend_days) == hash(frozenset({'sun', 'sat'}))=}")

sorted(weekend_days)=['sat', 'sun'], hash(weekend_days) == hash(frozenset({'sun', 'sat'}))=True


**`bytes(source, encoding)`**: encode text, zero-fill a size, or copy a list of ints 0–255.

In [503]:
print(f"{bytes('₹99', 'utf-8')=}, {bytes(3)=}, {bytes([72, 105])=}")

bytes('₹99', 'utf-8')=b'\xe2\x82\xb999', bytes(3)=b'\x00\x00\x00', bytes([72, 105])=b'Hi'


**`chr(code_point)`**: the character for a Unicode code point.

In [504]:
print(f"{chr(65)=}, {chr(8377)=}, {chr(0x1F600)=}")

chr(65)='A', chr(8377)='₹', chr(0x1F600)='😀'


**`ord(character)`**: the code point of a single character.

In [505]:
print(f"{ord('A')=}, {ord('₹')=}, {hex(ord('₹'))=}")
print(f"ord('ab'): {raised_error_name(lambda: ord('ab'))}")

ord('A')=65, ord('₹')=8377, hex(ord('₹'))='0x20b9'
ord('ab'): TypeError


**`bin(integer)`**: binary text with a `0b` prefix (`format(n, 'b')` drops it).

In [506]:
print(f"{bin(10)=}, {bin(-10)=}, {format(10, '08b')=}")

bin(10)='0b1010', bin(-10)='-0b1010', format(10, '08b')='00001010'


**`hex(integer)`**: hexadecimal text with a `0x` prefix.

In [507]:
print(f"{hex(255)=}, {hex(8377)=}, {format(255, 'X')=}")

hex(255)='0xff', hex(8377)='0x20b9', format(255, 'X')='FF'


**`oct(integer)`**: octal text with a `0o` prefix (Unix file permissions are octal).

In [508]:
print(f"{oct(8)=}, {oct(0o755)=}, {0o755 == 493=}")

oct(8)='0o10', oct(0o755)='0o755', 0o755 == 493=True


**`format(value, format_spec)`**: the format-spec mini-language of f-strings (1.4.3), as a function.

In [509]:
print(f"{format(1234567.891, ',.2f')=}, {format(0.25, '.0%')=}, {format(42, '>6')=}")

format(1234567.891, ',.2f')='1,234,567.89', format(0.25, '.0%')='25%', format(42, '>6')='    42'


**`repr(object)`**: the unambiguous developer form (`__repr__`); for built-ins, valid Python that recreates the value.

In [510]:
price_text = "249"
print(f"{str(price_text)=}, {repr(price_text)=}")
print(f"{eval(repr([1, 'two', None])) == [1, 'two', None]=}")

str(price_text)='249', repr(price_text)="'249'"
eval(repr([1, 'two', None])) == [1, 'two', None]=True


**`ascii(object)`**: like `repr`, but non-ASCII characters are escaped.

In [511]:
print(f"{ascii('₹99 café')=}")

ascii('₹99 café')="'\\u20b999 caf\\xe9'"

#### Introspection

**`type(obj)`**: the class of an object. The rarely used three-argument form
`type(name, bases, namespace)` creates a new class at run time (Phase 2 covers
metaclasses).

In [512]:
sample_quantity, sample_name = 3, "tea"
print(f"{type(sample_quantity)=}, {type(sample_name).__name__=}, {type(type)=}")
DynamicProduct = type("DynamicProduct", (), {"currency": "INR"})
print(f"{DynamicProduct.__name__=}, {DynamicProduct().currency=}")

type(sample_quantity)=<class 'int'>, type(sample_name).__name__='str', type(type)=<class 'type'>
DynamicProduct.__name__='DynamicProduct', DynamicProduct().currency='INR'


**`isinstance(obj, class_or_tuple)`**: also true for subclasses. Accepts a tuple or a
`X | Y` union (3.10). `bool` is a subclass of `int`, a classic surprise.

In [513]:
print(f"{isinstance(3, int)=}, {isinstance(3.0, int | float)=}, {isinstance('3', (int, float))=}")
print(f"{isinstance(True, int)=}  (bool subclasses int)")

isinstance(3, int)=True, isinstance(3.0, int | float)=True, isinstance('3', (int, float))=False
isinstance(True, int)=True  (bool subclasses int)


**`issubclass(cls, class_or_tuple)`**: the same test for classes.

In [514]:
print(f"{issubclass(bool, int)=}, {issubclass(int, object)=}, {issubclass(str, int)=}")

issubclass(bool, int)=True, issubclass(int, object)=True, issubclass(str, int)=False


**`id(obj)`**: an integer unique among objects alive at the same time (the address in CPython). Compare ids; never store or print them as data.

In [515]:
first_cart = ["tea"]
second_cart = first_cart
print(f"{id(first_cart) == id(second_cart)=}, {id(first_cart) == id(['tea'])=}")

id(first_cart) == id(second_cart)=True, id(first_cart) == id(['tea'])=False


**`hash(obj)`**: the hash used by dicts and sets (1.6.2); equal objects have equal hashes; mutable built-ins have none.

In [516]:
print(f"{hash(1) == hash(1.0) == hash(True)=}, {hash((1, 2)) == hash((1, 2))=}")
print(f"hash([]): {raised_error_name(lambda: hash([]))}")

hash(1) == hash(1.0) == hash(True)=True, hash((1, 2)) == hash((1, 2))=True
hash([]): TypeError


**`callable(obj)`**: `True` for functions, classes and objects with `__call__`.

In [517]:
print(f"{callable(len)=}, {callable(str)=}, {callable('len')=}, {callable(lambda: 0)=}")

callable(len)=True, callable(str)=True, callable('len')=False, callable(lambda: 0)=True


**`dir(obj)`**: the attribute names an object has (sorted), including dunders; filter them for a quick API overview.

In [518]:
public_str_methods = [name for name in dir(str) if not name.startswith("_")]
print(f"{len(public_str_methods)=}, {public_str_methods[:6]=}")

len(public_str_methods)=47, public_str_methods[:6]=['capitalize', 'casefold', 'center', 'count', 'encode', 'endswith']


**`vars(obj)`**: the object's `__dict__` (its attributes as a dict); works for instances, classes and modules.

In [519]:
class Customer:
    """A small class for the attribute built-ins."""

    def __init__(self, name: str, city: str) -> None:
        self.name = name
        self.city = city


asha = Customer("Asha", "Pune")
print(f"{vars(asha)=}, {vars(asha) is asha.__dict__=}")

vars(asha)={'name': 'Asha', 'city': 'Pune'}, vars(asha) is asha.__dict__=True


**`globals()`**: the current module's namespace dict. In a notebook, that is every name defined so far.

In [520]:
print(f"{'Customer' in globals()=}, {type(globals()).__name__=}")
print(f"{globals()['asha'] is asha=}")

'Customer' in globals()=True, type(globals()).__name__='dict'
globals()['asha'] is asha=True


**`locals()`**: the local names of the current function. *Changed in 3.13* (PEP 667):
inside a function it returns a **fresh snapshot** on every call, so later
assignments do not show up in an earlier result, and writing to it never changes a
variable.

In [521]:
def checkout_summary(order_total: int) -> None:
    """Show locals() as a snapshot."""
    delivery_fee = 40
    snapshot = locals()
    delivery_fee = 0
    print(f"{snapshot=}")
    print(f"{locals()['delivery_fee']=}  (a new snapshot sees the change)")


checkout_summary(1200)

snapshot={'order_total': 1200, 'delivery_fee': 40}
locals()['delivery_fee']=0  (a new snapshot sees the change)


**`getattr(obj, name, default)`**: read an attribute whose name is only known at run
time (from a column header, a config file, a request). With a literal name, just
write `obj.name`. The default avoids `AttributeError`.

In [522]:
for field_name in ("city", "phone"):
    print(f"{field_name}: {getattr(asha, field_name, 'not given')}")
missing_field = "phone"
print(f"no default: {raised_error_name(lambda: getattr(asha, missing_field))}")

city: Pune
phone: not given
no default: AttributeError


**`setattr(obj, name, value)`**: assign an attribute by name, for example from a dict of updates.

In [523]:
profile_updates = {"city": "Mumbai", "phone": "98200 00000"}
for field_name, new_value in profile_updates.items():
    setattr(asha, field_name, new_value)
print(f"{vars(asha)=}")

vars(asha)={'name': 'Asha', 'city': 'Mumbai', 'phone': '98200 00000'}


**`hasattr(obj, name)`**: `True` if `getattr` would succeed.

In [524]:
print(f"{hasattr(asha, 'name')=}, {hasattr(asha, 'email')=}, {hasattr('tea', 'upper')=}")

hasattr(asha, 'name')=True, hasattr(asha, 'email')=False, hasattr('tea', 'upper')=True


**`delattr(obj, name)`**: delete an attribute by name (`del obj.name`).

In [525]:
for field_name in ("city", "phone"):
    delattr(asha, field_name)
print(f"{vars(asha)=}, {hasattr(asha, 'city')=}")

vars(asha)={'name': 'Asha'}, hasattr(asha, 'city')=False


**`help(obj)`**: prints the signature and docstring (1.10.8 writes docstrings that `help` can show).

In [526]:
help(divmod)

Help on built-in function divmod in module builtins:

divmod(x, y, /)
    Return the tuple (x//y, x%y).  Invariant: div*y + mod == x.



#### I/O

**`print(*objects, sep=" ", end="\n", file=None, flush=False)`**: `sep` goes between
objects, `end` after the last one, `file` redirects the output (any object with
`write`), and `flush=True` pushes it out immediately (progress output, logs).

In [527]:
print("tea", "rice", "ghee", sep=" | ")
print("loading", end="... ")
print("done")
report_buffer = io.StringIO()
print("total", 1498, sep="=", file=report_buffer, flush=True)
print(f"{report_buffer.getvalue()=}")

tea | rice | ghee
loading... done
report_buffer.getvalue()='total=1498\n'


**`input(prompt="")`**: prints the prompt and reads one line from standard input,
**without** the newline, always as a `str`. A notebook run headless has no keyboard,
so the demo starts a child interpreter and feeds its standard input.

In [528]:
child_code = 'name = input("Name: "); qty = int(input("Qty: ")); print(f"|{name}|{qty * 2}|")'
completed = subprocess.run(
    [sys.executable, "-c", child_code],
    input="Asha\n3\n",
    capture_output=True,
    text=True,
    check=True,
)
print(f"{completed.stdout=}")

completed.stdout='Name: Qty: |Asha|6|\n'


**`open(file, mode="r", encoding=None, ...)`**: open a file. Always pass
`encoding="utf-8"` for text, and use `with` so the file is closed (1.14). Modes:
`"r"`, `"w"` (truncate), `"a"` (append), `"x"` (create only), plus `"b"` for bytes.

In [529]:
receipt_path = scratch_dir / "unit_1_9_receipt.txt"
with open(receipt_path, "w", encoding="utf-8") as receipt_file:
    receipt_file.write("tea 2 x ₹249\n")
    receipt_file.write("ghee 1 x ₹640\n")
with open(receipt_path, encoding="utf-8") as receipt_file:
    receipt_lines = [line.rstrip() for line in receipt_file]
print(f"{receipt_lines=}, {receipt_file.closed=}")

receipt_lines=['tea 2 x ₹249', 'ghee 1 x ₹640'], receipt_file.closed=True


#### Dangerous

> **Warning:** `eval`, `exec` and `compile` run arbitrary code. Never pass them text
> that came from a user, a file or the network. Restricting `globals` does **not**
> make them safe.

**`eval(expression, globals=None, locals=None)`**: evaluates one expression and returns
its value. The cell shows why it is dangerous: an "expression" can import modules,
and an empty `__builtins__` can still be escaped through the object graph.

In [530]:
print(f"{eval('2 + 3 * 4')=}, {eval('unit_price * 2', {'unit_price': 249})=}")
sneaky_input = "__import__('os').getcwd()"
print(f"{type(eval(sneaky_input)).__name__=}  (it imported os and ran a function)")
escape_route = "().__class__.__mro__[1].__subclasses__()"
reachable_classes = eval(escape_route, {"__builtins__": {}})
print(f"{len(reachable_classes) > 100=}  (every loaded class is reachable without builtins)")

eval('2 + 3 * 4')=14, eval('unit_price * 2', {'unit_price': 249})=498
type(eval(sneaky_input)).__name__='str'  (it imported os and ran a function)
len(reachable_classes) > 100=True  (every loaded class is reachable without builtins)


**`exec(source, globals=None, locals=None)`**: runs statements; returns `None`. Pass a dict to keep the names it defines out of your namespace.

In [531]:
plugin_namespace: dict[str, Any] = {}
exec_result = exec("def bonus(total):\n    return total // 10", plugin_namespace)
print(f"{exec_result=}, {plugin_namespace['bonus'](1500)=}")

exec_result=None, plugin_namespace['bonus'](1500)=150


**`compile(source, filename, mode)`**: turns source into a code object without running
it. Modes: `"eval"` (one expression), `"exec"` (statements), `"single"` (one
interactive statement). Compiling once and evaluating many times avoids re-parsing.

In [532]:
discount_rule = compile("total * 0.9 if total > 1000 else total", "<discount-rule>", "eval")
print(f"{type(discount_rule).__name__=}, {discount_rule.co_filename=}")
print([eval(discount_rule, {"total": total}) for total in (500, 2000)])

type(discount_rule).__name__='code', discount_rule.co_filename='<discount-rule>'
[500, 1800.0]


**`ast.literal_eval(text)`**: the **safe** alternative for data. It parses only Python
literals (strings, bytes, numbers, tuples, lists, dicts, sets, booleans, `None`) and
rejects everything else, including names and calls. For JSON text, use `json.loads`.

In [533]:
saved_settings = "{'currency': 'INR', 'tiers': (499, 999), 'beta': None}"
print(f"{ast.literal_eval(saved_settings)=}")
print(f"call: {raised_error_name(lambda: ast.literal_eval(sneaky_input))}")
print(f"name: {raised_error_name(lambda: ast.literal_eval('unit_price * 2'))}")

ast.literal_eval(saved_settings)={'currency': 'INR', 'tiers': (499, 999), 'beta': None}
call: ValueError
name: ValueError


### 1.9.5 `map` / `filter` vs comprehensions

| | `map` / `filter` | List comprehension |
| --- | --- | --- |
| Result | lazy iterator | list, built immediately |
| Reusable | no: one pass | yes |
| Reads well when | applying an **existing** named function | any expression, with a filter |
| Needs `lambda` for an expression | yes | no |

**Laziness**: `map` does no work until something consumes it. The `loud_gst`
function prints when it runs, so you can see when the work happens.

In [534]:
def loud_gst(price: int) -> float:
    """Add 18% GST and print that the work happened."""
    print(f"  computing GST for {price}")
    return round(price * 1.18, 2)


lazy_prices = map(loud_gst, [100, 200])
print("map created, nothing computed yet")
print(f"{next(lazy_prices)=}")
print("comprehension:")
eager_prices = [loud_gst(price) for price in [100, 200]]

map created, nothing computed yet
  computing GST for 100
next(lazy_prices)=118.0
comprehension:
  computing GST for 100
  computing GST for 200


**Single use**: a `map` / `filter` object is an iterator, so a second pass finds it
empty, silently. Convert with `list()` if you need the values twice.

In [535]:
valid_quantities = filter(lambda quantity: quantity > 0, [3, 0, 2])
print(f"{sum(valid_quantities)=}")
print(f"{sum(valid_quantities)=}  (already consumed)")

sum(valid_quantities)=5
sum(valid_quantities)=0  (already consumed)


**Readability**: with an existing function, `map` is short and clear. With a `lambda`
and a `filter`, the comprehension wins.

In [536]:
product_names = ["tea", "rice", "ghee"]
prices_in_rupees = [80, 249, 640, 1500]
print(f"{list(map(str.title, product_names))=}  (fine)")
with_lambdas = list(
    map(lambda price: price * 2, filter(lambda price: price > 200, prices_in_rupees))
)
with_comprehension = [price * 2 for price in prices_in_rupees if price > 200]
print(f"{with_lambdas == with_comprehension=}, {with_comprehension=}")

list(map(str.title, product_names))=['Tea', 'Rice', 'Ghee']  (fine)
with_lambdas == with_comprehension=True, with_comprehension=[498, 1280, 3000]


**Timing**: loop + `append` vs comprehension vs `map` with a built-in vs `map` with a
`lambda`. Comprehensions beat the explicit loop (no method lookup per item); `map`
with a C function such as `str.upper` is fastest; `map` with a `lambda` loses its edge
because each item pays for a Python function call.

> **Note:** timings vary by machine and Python version.

In [537]:
sample_words = ["tea", "rice", "ghee", "dal"] * 2_500


def upper_with_loop() -> list[str]:
    """Uppercase with an explicit loop."""
    result = []
    for word in sample_words:
        result.append(word.upper())
    return result


timing_candidates = {
    "loop + append": upper_with_loop,
    "comprehension": lambda: [word.upper() for word in sample_words],
    "map(str.upper)": lambda: list(map(str.upper, sample_words)),
    "map(lambda)": lambda: list(map(lambda word: word.upper(), sample_words)),
}
for label, candidate in timing_candidates.items():
    seconds = min(timeit.repeat(candidate, number=50, repeat=3)) / 50
    print(f"{label:15} {seconds * 1e6:8.1f} µs per 10 000 words")

loop + append      272.3 µs per 10 000 words


comprehension      211.2 µs per 10 000 words


map(str.upper)     179.1 µs per 10 000 words


map(lambda)        314.9 µs per 10 000 words


**Predict the output.**

```python
quantities = map(int, ["10", "20", "30"])
print(sum(quantities), sum(quantities))
print(any([]), all([]))
```

<details><summary>Answer</summary>

`60 0` then `False True`. The first `sum` consumes the `map` iterator, so the second
sum sees no items and returns its start value `0`. `any([])` has no truthy item;
`all([])` has no falsy item, so it is vacuously `True`.

</details>

Run the next cell to check your prediction.

In [538]:
quantities = map(int, ["10", "20", "30"])
print(sum(quantities), sum(quantities))
print(any([]), all([]))

60 0
False True


### 1.9.6 Pythonic idioms

Each pair below does the same work: first the style that comes over from C or
JavaScript, then the Python idiom.

**`enumerate` over `range(len())`**: you need the index *and* the item.

In [539]:
queue_names = ["asha", "ravi", "meena"]
for position in range(len(queue_names)):  # un-Pythonic
    print(f"{position}: {queue_names[position]}", end="  ")
print()
for position, queue_name in enumerate(queue_names):  # idiomatic
    print(f"{position}: {queue_name}", end="  ")
print()

0: asha  1: ravi  2: meena  
0: asha  1: ravi  2: meena  


**`zip` over parallel indexing**: walk two sequences together.

In [540]:
cities = ["Pune", "Delhi"]
temperatures = [31, 38]
for index in range(len(cities)):  # un-Pythonic
    print(f"{cities[index]}: {temperatures[index]}°C", end="  ")
print()
for city, temperature in zip(cities, temperatures, strict=True):  # idiomatic
    print(f"{city}: {temperature}°C", end="  ")
print()

Pune: 31°C  Delhi: 38°C  
Pune: 31°C  Delhi: 38°C  


**`in`** instead of a search loop or a chain of `==` comparisons.

In [541]:
payment_method = "upi"
is_instant_long = payment_method == "upi" or payment_method == "card" or payment_method == "wallet"
is_instant = payment_method in {"upi", "card", "wallet"}
print(f"{is_instant_long=}, {is_instant=}")

is_instant_long=True, is_instant=True


**`any` / `all`** instead of a flag variable and `break`.

In [542]:
stock_levels = [5, 0, 12]
has_sold_out = False  # un-Pythonic
for stock in stock_levels:
    if stock == 0:
        has_sold_out = True
        break
print(f"{has_sold_out=}, {any(stock == 0 for stock in stock_levels)=}")
print(f"{all(stock > 0 for stock in stock_levels)=}")

has_sold_out=True, any(stock == 0 for stock in stock_levels)=True
all(stock > 0 for stock in stock_levels)=False


**`dict.get`** instead of checking the key first.

In [543]:
discount_by_code = {"WELCOME10": 10, "FESTIVE20": 20}
entered_code = "SUMMER5"
if entered_code in discount_by_code:  # un-Pythonic
    discount_percent = discount_by_code[entered_code]
else:
    discount_percent = 0
print(f"{discount_percent=}, {discount_by_code.get(entered_code, 0)=}")

discount_percent=0, discount_by_code.get(entered_code, 0)=0


**EAFP preview** ("easier to ask forgiveness than permission"): try the operation and
handle the failure, instead of checking every precondition first (LBYL, "look before
you leap"). It avoids a race between the check and the use, and the check is not
done twice. Unit 1.14 covers exceptions in depth.

In [544]:
def parse_quantity_lbyl(text: str) -> int | None:
    """Look before you leap: check, then convert."""
    if text.strip().isdigit():
        return int(text)
    return None


def parse_quantity_eafp(text: str) -> int | None:
    """Easier to ask forgiveness: convert, handle the failure."""
    try:
        return int(text)
    except ValueError:
        return None


for raw_quantity in ("3", " 4 ", "-2", "two"):
    print(
        f"{raw_quantity!r:6} LBYL={parse_quantity_lbyl(raw_quantity)}, "
        f"EAFP={parse_quantity_eafp(raw_quantity)}"
    )

'3'    LBYL=3, EAFP=3
' 4 '  LBYL=4, EAFP=4
'-2'   LBYL=None, EAFP=-2
'two'  LBYL=None, EAFP=None


Note the difference for `"-2"`: `isdigit` does not accept a sign, so the LBYL version
rejects a value that `int` parses fine. Checks written by hand tend to drift from what
the operation really accepts.

### Lab 1.9

Three exercises. For each, replace the `raise NotImplementedError` in the stub, then
run the check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — `read_in_chunks(stream, size)`

Return an **iterator** over chunks of at most `size` characters (text stream) or bytes
(binary stream), built with `iter(callable, sentinel)`: no `while` loop and no
generator. It must work for both kinds of stream.

- Hint: `stream.read(0)` returns the empty value of the right type (`""` or `b""`).
- Raise `ValueError` immediately if `size < 1`.
- Read lazily: nothing is read until the first `next()`.

In [545]:
def read_in_chunks(stream: IO[Any], size: int) -> Iterator[Any]:
    """Return an iterator of chunks of at most size items from stream."""
    raise NotImplementedError("Lab 1.9: implement read_in_chunks()")

The check cell uses in-memory streams. `reads_lazily()` checks that the stream
position moves only when a chunk is requested.

In [546]:
def reads_lazily() -> list[int]:
    """Return the stream positions before and after the first next()."""
    lazy_stream = io.StringIO("abcdefgh")
    chunk_iterator = read_in_chunks(lazy_stream, 3)
    position_before = lazy_stream.tell()
    next(chunk_iterator)
    return [position_before, lazy_stream.tell()]


def returns_an_iterator() -> bool:
    """Return True if read_in_chunks returns an iterator (iter(x) is x)."""
    chunk_iterator = read_in_chunks(io.StringIO("x"), 1)
    return iter(chunk_iterator) is chunk_iterator


run_checks(
    [
        (
            "text chunks",
            lambda: list(read_in_chunks(io.StringIO("abcdefgh"), 3)),
            ["abc", "def", "gh"],
        ),
        (
            "binary chunks",
            lambda: list(read_in_chunks(io.BytesIO(b"\x00\x01\x02\x03"), 2)),
            [b"\x00\x01", b"\x02\x03"],
        ),
        ("empty stream", lambda: list(read_in_chunks(io.StringIO(""), 4)), []),
        ("size larger than data", lambda: list(read_in_chunks(io.BytesIO(b"hi"), 100)), [b"hi"]),
        ("returns an iterator", returns_an_iterator, True),
        ("reads lazily", reads_lazily, [0, 3]),
        (
            "size 0 raises",
            lambda: raised_error_name(lambda: read_in_chunks(io.StringIO("x"), 0)),
            "ValueError",
        ),
    ]
)

FAIL  text chunks: raised NotImplementedError: Lab 1.9: implement read_in_chunks()
FAIL  binary chunks: raised NotImplementedError: Lab 1.9: implement read_in_chunks()
FAIL  empty stream: raised NotImplementedError: Lab 1.9: implement read_in_chunks()
FAIL  size larger than data: raised NotImplementedError: Lab 1.9: implement read_in_chunks()
FAIL  returns an iterator: raised NotImplementedError: Lab 1.9: implement read_in_chunks()
FAIL  reads lazily: raised NotImplementedError: Lab 1.9: implement read_in_chunks()
FAIL  size 0 raises: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 2 — rewrite five un-Pythonic loops

The next cell defines five working but un-Pythonic functions. Write the idiomatic
versions (same names without `_unpythonic`) using the tools from 1.9.2–1.9.6:

| Function | Use |
| --- | --- |
| `numbered_lines(lines)` | `enumerate(..., start=1)` and a comprehension |
| `line_totals(unit_prices, quantities)` | `zip(..., strict=True)`: different lengths must raise `ValueError` |
| `has_overdue(days_overdue)` | `any(...)` |
| `count_statuses(statuses)` | `dict.get` or `collections.Counter` |
| `even_squares(numbers)` | a comprehension with a filter |

The checks compare your versions with the originals; they cannot judge style, so
compare with the reference solution afterwards.

In [547]:
def numbered_lines_unpythonic(lines: list[str]) -> list[str]:
    """Number lines from 1."""
    result = []
    for index in range(len(lines)):
        result.append(f"{index + 1}. {lines[index]}")
    return result


def line_totals_unpythonic(unit_prices: list[int], quantities: list[int]) -> list[int]:
    """Multiply parallel lists."""
    totals = []
    for index in range(len(unit_prices)):
        totals.append(unit_prices[index] * quantities[index])
    return totals


def has_overdue_unpythonic(days_overdue: list[int]) -> bool:
    """Return True if any invoice is more than 30 days overdue."""
    found = False
    for days in days_overdue:
        if days > 30:
            found = True
            break
    return found


def count_statuses_unpythonic(statuses: list[str]) -> dict[str, int]:
    """Count how often each status occurs."""
    counts = {}
    for status in statuses:
        if status in counts:
            counts[status] = counts[status] + 1
        else:
            counts[status] = 1
    return counts


def even_squares_unpythonic(numbers: list[int]) -> list[int]:
    """Square the even numbers."""
    squares = []
    for number in numbers:
        if number % 2 == 0:
            squares.append(number * number)
    return squares

The five stubs:

In [548]:
def numbered_lines(lines: list[str]) -> list[str]:
    """Number lines from 1 (enumerate)."""
    raise NotImplementedError("Lab 1.9: implement numbered_lines()")


def line_totals(unit_prices: list[int], quantities: list[int]) -> list[int]:
    """Multiply parallel lists (zip, strict)."""
    raise NotImplementedError("Lab 1.9: implement line_totals()")


def has_overdue(days_overdue: list[int]) -> bool:
    """Return True if any invoice is more than 30 days overdue (any)."""
    raise NotImplementedError("Lab 1.9: implement has_overdue()")


def count_statuses(statuses: list[str]) -> dict[str, int]:
    """Count how often each status occurs (dict.get or Counter)."""
    raise NotImplementedError("Lab 1.9: implement count_statuses()")


def even_squares(numbers: list[int]) -> list[int]:
    """Square the even numbers (comprehension)."""
    raise NotImplementedError("Lab 1.9: implement even_squares()")

The check cell runs each pair on the same inputs.

In [549]:
sample_lines = ["GET /", "POST /cart"]
sample_statuses = ["paid", "failed", "paid", "refunded", "paid"]
run_checks(
    [
        (
            "numbered_lines",
            lambda: numbered_lines(sample_lines),
            numbered_lines_unpythonic(sample_lines),
        ),
        ("numbered_lines empty", lambda: numbered_lines([]), []),
        (
            "line_totals",
            lambda: line_totals([249, 640], [2, 1]),
            line_totals_unpythonic([249, 640], [2, 1]),
        ),
        (
            "line_totals strict",
            lambda: raised_error_name(lambda: line_totals([249, 640], [2])),
            "ValueError",
        ),
        ("has_overdue true", lambda: has_overdue([3, 45]), has_overdue_unpythonic([3, 45])),
        ("has_overdue false", lambda: has_overdue([30, 0]), False),
        ("has_overdue empty", lambda: has_overdue([]), False),
        (
            "count_statuses",
            lambda: count_statuses(sample_statuses),
            count_statuses_unpythonic(sample_statuses),
        ),
        ("count_statuses empty", lambda: count_statuses([]), {}),
        (
            "even_squares",
            lambda: even_squares([1, 2, 3, 4, -6]),
            even_squares_unpythonic([1, 2, 3, 4, -6]),
        ),
    ]
)

FAIL  numbered_lines: raised NotImplementedError: Lab 1.9: implement numbered_lines()
FAIL  numbered_lines empty: raised NotImplementedError: Lab 1.9: implement numbered_lines()
FAIL  line_totals: raised NotImplementedError: Lab 1.9: implement line_totals()
FAIL  line_totals strict: expected 'ValueError', got 'NotImplementedError'
FAIL  has_overdue true: raised NotImplementedError: Lab 1.9: implement has_overdue()
FAIL  has_overdue false: raised NotImplementedError: Lab 1.9: implement has_overdue()
FAIL  has_overdue empty: raised NotImplementedError: Lab 1.9: implement has_overdue()
FAIL  count_statuses: raised NotImplementedError: Lab 1.9: implement count_statuses()
FAIL  count_statuses empty: raised NotImplementedError: Lab 1.9: implement count_statuses()
FAIL  even_squares: raised NotImplementedError: Lab 1.9: implement even_squares()
0/10 checks passed


#### Exercise 3 — summarize webhook events

Write `summarize_webhooks(events)` returning a dict with exactly these keys, using
`sum`, `max` and `sorted` (no explicit loops needed):

| Key | Value |
| --- | --- |
| `"event_count"` | number of events |
| `"captured_total"` | sum of `amount` (paise) over `"payment.captured"` events; `0` if none |
| `"largest_event_id"` | `event_id` of the event with the largest `amount`; first one on ties; `None` for no events |
| `"ids_by_time"` | event ids sorted by `received_at` (oldest first) |
| `"event_types"` | the distinct event types, sorted |

Run the next cell first: it defines `WebhookEvent` and `SAMPLE_WEBHOOK_EVENTS`
(deliberately not in time order).

In [550]:
class WebhookEvent(NamedTuple):
    """One webhook delivery from a payment provider (amount in paise)."""

    event_id: str
    event_type: str
    amount: int
    received_at: dt.datetime


SAMPLE_WEBHOOK_EVENTS = [
    WebhookEvent("evt_3", "payment.captured", 49_900, dt.datetime(2026, 11, 3, 10, 5)),
    WebhookEvent("evt_1", "payment.captured", 129_900, dt.datetime(2026, 11, 3, 9, 15)),
    WebhookEvent("evt_4", "refund.created", 19_900, dt.datetime(2026, 11, 3, 11, 0)),
    WebhookEvent("evt_2", "payment.failed", 129_900, dt.datetime(2026, 11, 3, 9, 40)),
    WebhookEvent("evt_5", "payment.captured", 9_900, dt.datetime(2026, 11, 3, 8, 55)),
]
print(f"{len(SAMPLE_WEBHOOK_EVENTS)=}")

len(SAMPLE_WEBHOOK_EVENTS)=5


The stub:

In [551]:
def summarize_webhooks(events: list[WebhookEvent]) -> dict[str, object]:
    """Summarize webhook events with sum, max and sorted."""
    raise NotImplementedError("Lab 1.9: implement summarize_webhooks()")

The check cell: note that `evt_1` and `evt_2` tie on the largest amount, and `evt_1` comes first in the list.

In [552]:
run_checks(
    [
        (
            "sample summary",
            lambda: summarize_webhooks(SAMPLE_WEBHOOK_EVENTS),
            {
                "event_count": 5,
                "captured_total": 189_700,
                "largest_event_id": "evt_1",
                "ids_by_time": ["evt_5", "evt_1", "evt_2", "evt_3", "evt_4"],
                "event_types": ["payment.captured", "payment.failed", "refund.created"],
            },
        ),
        (
            "no events",
            lambda: summarize_webhooks([]),
            {
                "event_count": 0,
                "captured_total": 0,
                "largest_event_id": None,
                "ids_by_time": [],
                "event_types": [],
            },
        ),
        (
            "no captures",
            lambda: summarize_webhooks(SAMPLE_WEBHOOK_EVENTS[2:4])["captured_total"],
            0,
        ),
    ]
)

FAIL  sample summary: raised NotImplementedError: Lab 1.9: implement summarize_webhooks()
FAIL  no events: raised NotImplementedError: Lab 1.9: implement summarize_webhooks()
FAIL  no captures: raised NotImplementedError: Lab 1.9: implement summarize_webhooks()
0/3 checks passed


### Recap — Unit 1.9

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Iterable / iterator | `__iter__` gives an iterator; `__next__` gives items, then raises `StopIteration` | `[Symbol.iterator]`, `next()` → `{value, done}` |
| `for` loop | `iter()` once, `next()` until `StopIteration` (`GET_ITER` / `FOR_ITER`) | `for...of` |
| `iter(callable, sentinel)` | Call until the sentinel comes back: chunked reads | — |
| `enumerate`, `zip(strict=True)` | Index + item; parallel iteration that checks lengths | `entries()`; no built-in zip |
| `reversed`, `sorted`, `range` | Reverse iterator; new sorted list; lazy int sequence | `toReversed`, `toSorted` |
| Comprehensions | `[expr for x in it if cond]`, dict and set forms; own scope; inlined since 3.12 | `map` + `filter` |
| Nested `for` order | Left to right = outer to inner | nested `flatMap` |
| Numeric built-ins | `round` ties to even; `sum(start=)`; `min`/`max` with `key` and `default` | `Math.*`, `reduce` |
| `map` / `filter` | Lazy, single-use iterators; best with existing functions | eager array methods |
| `any` / `all` | Short-circuit truth tests; `all([])` is `True` | `some` / `every` |
| Conversion built-ins | `int`, `float`, `str`, `bytes`, `chr`, `ord`, `bin`/`hex`/`oct`, `format`, `repr`, `ascii` | `parseInt`, `String`, `toString(16)` |
| Introspection | `type`, `isinstance`, `dir`, `vars`, `getattr`/`setattr`/`hasattr`/`delattr`, `help` | `typeof`, `instanceof`, `Reflect` |
| `locals()` | A snapshot in functions since 3.13 (PEP 667) | — |
| `eval` / `exec` / `compile` | Run arbitrary code; never on untrusted input | `eval`, `new Function` |
| `ast.literal_eval` | Safe parsing of Python literals | `JSON.parse` |
| Idioms | `enumerate`, `zip`, `in`, `any`/`all`, `dict.get`, EAFP | — |

**Further reading**

- [Built-in functions](https://docs.python.org/3/library/functions.html)
- [Iterator types](https://docs.python.org/3/library/stdtypes.html#iterator-types),
  [glossary: iterable](https://docs.python.org/3/glossary.html#term-iterable),
  [glossary: EAFP](https://docs.python.org/3/glossary.html#term-EAFP)
- [Tutorial: list comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions),
  [displays for lists, sets and dicts](https://docs.python.org/3/reference/expressions.html#displays-for-lists-sets-and-dictionaries)
- [`ast.literal_eval`](https://docs.python.org/3/library/ast.html#ast.literal_eval),
  [`dis`](https://docs.python.org/3/library/dis.html)
- [PEP 234 — Iterators](https://peps.python.org/pep-0234/),
  [PEP 202 — List comprehensions](https://peps.python.org/pep-0202/),
  [PEP 274 — Dict comprehensions](https://peps.python.org/pep-0274/)
- [PEP 618 — Add optional length-checking to zip](https://peps.python.org/pep-0618/),
  [PEP 709 — Inlined comprehensions](https://peps.python.org/pep-0709/),
  [PEP 667 — Consistent views of namespaces](https://peps.python.org/pep-0667/)

## Unit 1.10 — Functions: Definition and Arguments

**Objectives**

- Explain that `def` runs at run time and builds a function object with inspectable
  attributes.
- Use every parameter kind (positional-only, standard, `*args`, keyword-only,
  `**kwargs`), in the right order, and read them with `inspect.signature`.
- Forward arguments with `*` / `**`, and avoid the mutable-default trap.
- Pass functions around: callbacks, dispatch tables, `lambda`.
- Write PEP 257 docstrings and annotations, and know what annotations do **not** do.
- Write a recursive function with a base case, and know the recursion limit.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `function f() {}` is hoisted | `def f():` runs when reached | Calling before the `def` raises `NameError` |
| `return;` / no return → `undefined` | no return → `None` | |
| `function f(a, b = 2)` | `def f(a, b=2):` | Python evaluates the default **once**, at definition |
| `function f(...args)` | `def f(*args):` | `args` is a **tuple** |
| `function f({ page, size })` | `def f(*, page, size):` | Keyword-only parameters, no object needed |
| — | `def f(x, /):` | Positional-only parameters |
| `f(...values)` | `f(*values)`, `f(**options)` | `**` unpacks a dict into keyword arguments |
| `(x) => x * 2` | `lambda x: x * 2` | One expression only; no statements |
| JSDoc / TypeScript types | docstrings and annotations | Annotations are **not** checked at run time |
| `f.name`, `f.length` | `f.__name__`, `inspect.signature(f)` | Many more attributes (`__defaults__`, `__code__`, …) |

The next cell is this unit's setup (`compile_error` is the helper from Unit 1.8).

In [553]:
import datetime as dt
import dis
import inspect
import math
import operator
import sys
import traceback
from collections.abc import Callable
from typing import Any
from urllib.parse import urlencode  # noqa: F401 (used in Lab 1.10)


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def compile_error(source: str) -> str:
    """Compile source without running it; return "SyntaxError: <message>" or "compiles"."""
    try:
        compile(source, "<demo>", "exec")
    except SyntaxError as error:
        return f"SyntaxError: {error.msg}"
    return "compiles"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.10.1 `def` is an executable statement

`def name(...):` is a statement that **runs**: it builds a function object from
the compiled body and binds it to `name`. Until that line has run, the name does not
exist. There is no hoisting, unlike JavaScript function declarations.

The next cell runs a small module in a fresh namespace (so earlier notebook cells
cannot interfere): the call before the `def` fails, the call after it works.

In [554]:
module_source = """
print(greet("asha"))

def greet(name):
    return f"hello {name}"
"""
try:
    exec(module_source, {})
except NameError as error:
    print(f"{type(error).__name__}: {error}")

fixed_namespace: dict[str, Any] = {}
exec("def greet(name):\n    return name\nresult = greet('asha')", fixed_namespace)
print(f"{fixed_namespace['result']=}")

NameError: name 'greet' is not defined
fixed_namespace['result']='asha'


#### Under the hood: `MAKE_FUNCTION`

The bytecode of a `def` shows it: the body was compiled ahead of time into a code
object (a constant), and at run time `MAKE_FUNCTION` wraps it into a function object,
which `STORE_NAME` binds to the name. Opcode names vary by version.

In [555]:
definition_code = compile("def greet(name):\n    return name\n", "<demo>", "exec")
for instruction in dis.get_instructions(definition_code):
    if inspect.iscode(instruction.argval):
        detail = f"<code object {instruction.argval.co_name}>"  # no address printed
    else:
        detail = instruction.argrepr
    print(f"{instruction.opname:16} {detail}")

RESUME           
LOAD_CONST       <code object greet>
MAKE_FUNCTION    
STORE_NAME       greet
LOAD_COMMON_CONSTANT None
RETURN_VALUE     


Because `def` is a statement, it can sit inside an `if`: **conditional definitions**
pick an implementation once, at import time. Defining the same name twice simply
rebinds it; the last definition wins.

In [556]:
if sys.version_info >= (3, 15):

    def describe_runtime() -> str:
        """Implementation for new interpreters."""
        return "3.15+ implementation"

else:

    def describe_runtime() -> str:
        """Fallback implementation."""
        return "fallback implementation"


def price_label(price: int) -> str:
    """First version."""
    return f"Rs {price}"


def price_label(price: int) -> str:  # rebinding the name; the first object is gone
    """Second version."""
    return f"₹{price}"


print(f"{describe_runtime()=}, {price_label(249)=}")

describe_runtime()='3.15+ implementation', price_label(249)='₹249'


**Function attributes.** A function is an object like any other, with attributes:

| Attribute | Holds |
| --- | --- |
| `__name__` | the name given in `def` |
| `__qualname__` | the dotted path from the module (`Class.method`, `outer.<locals>.inner`) |
| `__doc__` | the docstring, or `None` |
| `__defaults__` | tuple of default values for positional parameters, or `None` |
| `__kwdefaults__` | dict of defaults for keyword-only parameters, or `None` |
| `__code__` | the compiled code object (bytecode, variable names, argument counts) |
| `__globals__` | the module namespace the function looks up global names in |
| `__annotations__` | dict of parameter and return annotations |
| `__dict__` | arbitrary attributes you attach yourself |

The next cell defines one function used by every attribute demo below.

In [557]:
def apply_discount(price: float, percent: float = 10, *, round_to: int = 2) -> float:
    """Return price reduced by percent, rounded to round_to decimal places."""
    discounted_price = price * (100 - percent) / 100
    return round(discounted_price, round_to)


print(f"{apply_discount(249)=}, {apply_discount(249, 50, round_to=0)=}")

apply_discount(249)=224.1, apply_discount(249, 50, round_to=0)=124.0


**`__name__`**: the name from the `def` line (it does not change if you assign the function to another name).

In [558]:
discount_function = apply_discount
print(f"{apply_discount.__name__=}, {discount_function.__name__=}")

apply_discount.__name__='apply_discount', discount_function.__name__='apply_discount'


**`__qualname__`**: the qualified name, showing where the function was defined.

In [559]:
class Checkout:
    """A class with one method, to show qualified names."""

    def total(self) -> int:
        """Return a fixed total."""
        return 0


def make_formatter() -> Callable[[int], str]:
    """Return a nested function."""

    def format_price(price: int) -> str:
        """Format a price in rupees."""
        return f"₹{price}"

    return format_price


print(f"{apply_discount.__qualname__=}")
print(f"{Checkout.total.__qualname__=}")
print(f"{make_formatter().__qualname__=}")

apply_discount.__qualname__='apply_discount'
Checkout.total.__qualname__='Checkout.total'
make_formatter().__qualname__='make_formatter.<locals>.format_price'


**`__doc__`**: the docstring (the first statement of the body, if it is a string literal).

In [560]:
def no_docstring() -> None:
    pass


print(f"{apply_discount.__doc__=}")
print(f"{make_formatter().__doc__=}, {no_docstring.__doc__=}")

apply_discount.__doc__='Return price reduced by percent, rounded to round_to decimal places.'
make_formatter().__doc__='Format a price in rupees.', no_docstring.__doc__=None


**`__defaults__`**: defaults of the positional parameters, as a tuple aligned to the *last* ones.

In [561]:
print(f"{apply_discount.__defaults__=}  (percent=10)")

apply_discount.__defaults__=(10,)  (percent=10)


**`__kwdefaults__`**: defaults of the keyword-only parameters, as a dict.

In [562]:
print(f"{apply_discount.__kwdefaults__=}")

apply_discount.__kwdefaults__={'round_to': 2}


**`__code__`**: the code object: argument counts, local variable names, the bytecode itself.

In [563]:
discount_code = apply_discount.__code__
print(f"{discount_code.co_argcount=}, {discount_code.co_kwonlyargcount=}")
print(f"{discount_code.co_varnames=}")
print(f"{len(discount_code.co_code) > 0=}  (raw bytecode bytes)")

discount_code.co_argcount=2, discount_code.co_kwonlyargcount=1
discount_code.co_varnames=('price', 'percent', 'round_to', 'discounted_price')
len(discount_code.co_code) > 0=True  (raw bytecode bytes)


**`__globals__`**: the namespace of the module where the function was defined; here, the notebook's globals.

In [564]:
print(f"{apply_discount.__globals__ is globals()=}")
print(f"{'apply_discount' in apply_discount.__globals__=}")

apply_discount.__globals__ is globals()=True
'apply_discount' in apply_discount.__globals__=True


**`__annotations__`**: the annotations, by parameter name, plus `'return'`.

In [565]:
print(f"{apply_discount.__annotations__=}")

apply_discount.__annotations__={'price': <class 'float'>, 'percent': <class 'float'>, 'round_to': <class 'int'>, 'return': <class 'float'>}


**`__dict__`**: attributes you attach yourself (version tags, counters, registry flags).

In [566]:
apply_discount.version = 2
apply_discount.owner = "pricing-team"
print(f"{apply_discount.__dict__=}, {apply_discount.version=}")

apply_discount.__dict__={'version': 2, 'owner': 'pricing-team'}, apply_discount.version=2


### 1.10.2 Return values

**Implicit `None`**: a function without `return`, or with a bare `return`, returns
`None`. Methods that change an object in place follow this convention (`list.sort()`,
`list.append()`, `dict.update()`), so `items = items.sort()` loses the list.

In [567]:
def log_order(order_id: int) -> None:
    """Print a log line; no return statement."""
    print(f"  order {order_id} logged")


log_result = log_order(1001)
print(f"{log_result=}")
product_names = ["tea", "rice"]
sort_result = product_names.sort()
print(f"{sort_result=}  (sort works in place and returns None)")

  order 1001 logged
log_result=None
sort_result=None  (sort works in place and returns None)


**Returning several values**: `return a, b` returns **one tuple**; the caller unpacks
it. For more than two or three values, return a `NamedTuple` (1.5.6) or a dataclass
(Phase 2) so the fields have names.

In [568]:
def price_range(prices: list[int]) -> tuple[int, int]:
    """Return the lowest and highest price."""
    return min(prices), max(prices)


range_result = price_range([249, 80, 640])
lowest_price, highest_price = price_range([249, 80, 640])
print(f"{range_result=}, {type(range_result).__name__=}")
print(f"{lowest_price=}, {highest_price=}")

range_result=(80, 640), type(range_result).__name__='tuple'
lowest_price=80, highest_price=640


**Guard clauses** (1.8.3) are early returns: deal with invalid or trivial input
first, so the rest of the function handles only the normal case.

In [569]:
def average_order_value(order_totals: list[float]) -> float | None:
    """Return the mean order total, or None for no orders."""
    if not order_totals:
        return None
    if any(total < 0 for total in order_totals):
        raise ValueError("order totals cannot be negative")
    return sum(order_totals) / len(order_totals)


print(f"{average_order_value([])=}, {average_order_value([100, 250])=}")
print(f"negative: {raised_error_name(lambda: average_order_value([-5]))}")

average_order_value([])=None, average_order_value([100, 250])=175.0
negative: ValueError


### 1.10.3 Parameter kinds

| Kind | Syntax | Caller may pass | `inspect` kind |
| --- | --- | --- | --- |
| Positional-only | before `/` | by position only | `POSITIONAL_ONLY` |
| Positional-or-keyword | plain names | by position or by name | `POSITIONAL_OR_KEYWORD` |
| Default value | `name=value` | may be omitted | (any kind) |
| Variadic positional | `*args` | any number of extra positionals → a **tuple** | `VAR_POSITIONAL` |
| Keyword-only | after `*` or `*args` | by name only | `KEYWORD_ONLY` |
| Variadic keyword | `**kwargs` | any number of extra keywords → a **dict** | `VAR_KEYWORD` |

**Positional-or-keyword** (the default kind): the caller chooses. Passing the same
parameter twice is a `TypeError`.

In [570]:
def format_greeting(name: str, greeting: str) -> str:
    """Greet name with greeting."""
    return f"{greeting}, {name}!"


print(format_greeting("Asha", "Namaste"))
print(format_greeting(greeting="Hello", name="Ravi"))
print(f"twice: {raised_error_name(lambda: format_greeting('Asha', name='Ravi'))}")

Namaste, Asha!
Hello, Ravi!
twice: TypeError


**Defaults** make a parameter optional. Parameters with defaults must come after
those without (among the positional ones).

In [571]:
def add_gst(amount: float, rate_percent: float = 18) -> float:
    """Add GST at rate_percent (default 18%)."""
    return round(amount * (1 + rate_percent / 100), 2)


print(f"{add_gst(100)=}, {add_gst(100, 5)=}, {add_gst(100, rate_percent=28)=}")
print(f"{compile_error('def broken(rate=18, amount): pass')=}")

add_gst(100)=118.0, add_gst(100, 5)=105.0, add_gst(100, rate_percent=28)=128.0
compile_error('def broken(rate=18, amount): pass')='SyntaxError: parameter without a default follows parameter with a default'


**`*args`** collects any extra positional arguments into a **tuple**.

In [572]:
def order_total(*line_amounts: float) -> float:
    """Sum any number of line amounts."""
    print(f"  {line_amounts=}, {type(line_amounts).__name__=}")
    return sum(line_amounts)


print(f"{order_total(249, 640)=}")
print(f"{order_total()=}")

  line_amounts=(249, 640), type(line_amounts).__name__='tuple'
order_total(249, 640)=889
  line_amounts=(), type(line_amounts).__name__='tuple'
order_total()=0


**Keyword-only parameters** come after `*` (or after `*args`). They must be passed by
name, which keeps call sites readable for flags and options: `is_admin=True` instead
of a bare `True`.

In [573]:
def create_user(name: str, *, is_admin: bool = False, send_welcome: bool = True) -> str:
    """Create a user; the flags must be named."""
    return f"{name} (admin={is_admin}, welcome={send_welcome})"


print(create_user("asha", is_admin=True))
print(f"positional flag: {raised_error_name(lambda: create_user('asha', True))}")

asha (admin=True, welcome=True)
positional flag: TypeError


**Positional-only parameters** come before `/` (PEP 570, Python 3.8). The caller cannot
use their names, so the author can rename them later without breaking anyone, and
the names stay free for `**kwargs`. Many built-ins work this way: `len(obj, /)`,
`divmod(x, y, /)`.

In [574]:
def distance_km(start_km: float, end_km: float, /) -> float:
    """Distance between two markers on a highway."""
    return abs(end_km - start_km)


print(f"{distance_km(12.5, 40.0)=}")
print(f"by name: {raised_error_name(lambda: distance_km(start_km=1, end_km=2))}")
print(f"len by name: {raised_error_name(lambda: len(obj=[1, 2]))}")

distance_km(12.5, 40.0)=27.5
by name: TypeError
len by name: TypeError


**`**kwargs`** collects any extra keyword arguments into a **dict** (insertion-ordered, PEP 468).

In [575]:
def log_event(event_name: str, **fields: object) -> str:
    """Format an event with arbitrary named fields."""
    print(f"  {fields=}, {type(fields).__name__=}")
    return " ".join([event_name, *(f"{key}={value}" for key, value in fields.items())])


print(log_event("checkout", user="asha", total=889, coupon=None))

  fields={'user': 'asha', 'total': 889, 'coupon': None}, type(fields).__name__='dict'
checkout user=asha total=889 coupon=None


**The full signature order** is fixed:

```python
def f(pos_only, /, standard, *, kw_only, **kwargs): ...
def g(pos_only, /, standard=1, *args, kw_only, kw_with_default=2, **kwargs): ...
```

Why each kind exists:

| Kind | Use it when |
| --- | --- |
| Positional-only | the name means nothing to callers (`x`, `obj`), may change, or must not clash with `**kwargs` |
| Positional-or-keyword | the ordinary case |
| `*args` | the function naturally takes "any number of" things (`print`, `max`) |
| Keyword-only | options and boolean flags, so call sites say what they mean |
| `**kwargs` | forwarding to another function, or open-ended named fields |

The next cell calls the full form and shows where each argument lands.

In [576]:
def every_kind(
    pos_only: object,
    /,
    standard: object = 1,
    *args: object,
    kw_only: object,
    kw_with_default: object = 2,
    **kwargs: object,
) -> dict[str, object]:
    """Return how the arguments were bound."""
    return {
        "pos_only": pos_only,
        "standard": standard,
        "args": args,
        "kw_only": kw_only,
        "kw_with_default": kw_with_default,
        "kwargs": kwargs,
    }


print(every_kind("a", "b", "c", "d", kw_only="e", extra="f"))
print(every_kind("a", kw_only="e", pos_only="goes into kwargs"))

{'pos_only': 'a', 'standard': 'b', 'args': ('c', 'd'), 'kw_only': 'e', 'kw_with_default': 2, 'kwargs': {'extra': 'f'}}
{'pos_only': 'a', 'standard': 1, 'args': (), 'kw_only': 'e', 'kw_with_default': 2, 'kwargs': {'pos_only': 'goes into kwargs'}}


The last call shows the positional-only benefit: `pos_only="..."` is not a duplicate,
it lands in `**kwargs`.

**Reading kinds with `inspect.signature`**: frameworks (FastAPI in Phase 6, pytest
fixtures in Phase 4) inspect signatures to decide what to pass.

In [577]:
for parameter in inspect.signature(every_kind).parameters.values():
    default = "—" if parameter.default is inspect.Parameter.empty else parameter.default
    print(f"{parameter.name:16} {parameter.kind.name:22} default={default}")
print(f"{inspect.signature(divmod)=}")

pos_only         POSITIONAL_ONLY        default=—
standard         POSITIONAL_OR_KEYWORD  default=1
args             VAR_POSITIONAL         default=—
kw_only          KEYWORD_ONLY           default=—
kw_with_default  KEYWORD_ONLY           default=2
kwargs           VAR_KEYWORD            default=—
inspect.signature(divmod)=<Signature (x, y, /)>


### 1.10.4 Argument unpacking

At a **call**, `*sequence` spreads items into positional arguments and `**mapping`
spreads key/value pairs into keyword arguments (1.8.7). It is how you call a function
with arguments you built as data.

In [578]:
def ship_parcel(weight_kg: float, destination: str, *, express: bool = False) -> str:
    """Describe a shipment."""
    return f"{weight_kg} kg to {destination} ({'express' if express else 'standard'})"


parcel_values = [2.5, "Pune"]
parcel_options = {"express": True}
print(ship_parcel(*parcel_values, **parcel_options))
print(f"unknown option: {raised_error_name(lambda: ship_parcel(*parcel_values, gift=True))}")

2.5 kg to Pune (express)
unknown option: TypeError


**The forwarding wrapper**: `def wrapper(*args, **kwargs)` accepts **any** arguments
and passes them on unchanged with `func(*args, **kwargs)`. It is the shape of every
decorator (Unit 1.12).

In [579]:
def call_logged(func: Callable[..., Any], *args: Any, **kwargs: Any) -> Any:
    """Call func with the given arguments and log the call and its result."""
    result = func(*args, **kwargs)
    print(f"  {func.__name__}(args={args}, kwargs={kwargs}) -> {result!r}")
    return result


call_logged(ship_parcel, 1.0, "Delhi")
call_logged(ship_parcel, 4.2, destination="Goa", express=True)
largest_by_size = call_logged(max, [3, -9, 4], key=abs)

  ship_parcel(args=(1.0, 'Delhi'), kwargs={}) -> '1.0 kg to Delhi (standard)'
  ship_parcel(args=(4.2,), kwargs={'destination': 'Goa', 'express': True}) -> '4.2 kg to Goa (express)'
  max(args=([3, -9, 4],), kwargs={'key': <built-in function abs>}) -> -9


### 1.10.5 Default values

A default expression is evaluated **once**, when the `def` statement runs, and the
resulting object is stored in `__defaults__`. Every call that omits the argument gets
**that same object**. JavaScript evaluates defaults on every call, so this surprises
JS developers.

In [580]:
def announce_default() -> int:
    """Print when the default is computed."""
    print("  default computed now (at def time)")
    return 18


print("defining:")


def tax_rate(rate: int = announce_default()) -> int:
    """Return rate (default computed once)."""
    return rate


print("calling twice:")
print(f"{tax_rate()=}, {tax_rate()=}")

defining:
  default computed now (at def time)
calling twice:
tax_rate()=18, tax_rate()=18


**The mutable-default trap.** A list (or dict, or set) as a default is created once
and shared by every call, so it accumulates. `__defaults__` shows the shared list
growing.

In [581]:
def add_to_cart_buggy(item: str, cart: list[str] = []) -> list[str]:  # noqa: B006
    """Append item to cart (buggy: the default list is shared)."""
    cart.append(item)
    return cart


print(f"{add_to_cart_buggy.__defaults__=}")
first_cart = add_to_cart_buggy("tea")
second_cart = add_to_cart_buggy("rice")
print(f"{second_cart=}, {first_cart is second_cart=}")
print(f"{add_to_cart_buggy.__defaults__=}")

add_to_cart_buggy.__defaults__=([],)
second_cart=['tea', 'rice'], first_cart is second_cart=True
add_to_cart_buggy.__defaults__=(['tea', 'rice'],)


**The `None` pattern**: use `None` as the default and create the object inside the
function, so each call gets a fresh one. (`ruff` rule B006 flags the buggy version.)

In [582]:
def add_to_cart(item: str, cart: list[str] | None = None) -> list[str]:
    """Append item to cart, or to a new cart if none is given."""
    if cart is None:
        cart = []
    cart.append(item)
    return cart


print(f"{add_to_cart('tea')=}, {add_to_cart('rice')=}")
shared_cart = ["ghee"]
print(f"{add_to_cart('dal', shared_cart)=}, {shared_cart=}")

add_to_cart('tea')=['tea'], add_to_cart('rice')=['rice']
add_to_cart('dal', shared_cart)=['ghee', 'dal'], shared_cart=['ghee', 'dal']


**`sentinel()`** (*New in 3.15*, PEP 661): sometimes `None` is a valid value. An
"update profile" function needs three cases: argument not passed (keep the email),
`None` (clear it), or a string (change it). A sentinel is a unique object with a clear
`repr`, used as the default and checked with `is`. Create it **once** at module level:
every `sentinel(...)` call makes a new, distinct object. Before 3.15 the usual
spelling is `MISSING = object()`.

In [583]:
if sys.version_info >= (3, 15):
    NOT_GIVEN = sentinel("NOT_GIVEN")
else:
    NOT_GIVEN = object()


def update_email(profile: dict[str, Any], email: str | None | object = NOT_GIVEN) -> dict[str, Any]:
    """Return a copy of profile: keep, clear (None) or replace the email."""
    updated_profile = dict(profile)
    if email is not NOT_GIVEN:
        updated_profile["email"] = email
    return updated_profile


asha_profile = {"name": "Asha", "email": "asha@mail.in"}
print(f"{update_email(asha_profile)=}")
print(f"{update_email(asha_profile, None)=}")
print(f"{update_email(asha_profile, 'asha@work.in')=}")
print(f"{NOT_GIVEN!r}  (readable in signatures and logs)")

update_email(asha_profile)={'name': 'Asha', 'email': 'asha@mail.in'}
update_email(asha_profile, None)={'name': 'Asha', 'email': None}
update_email(asha_profile, 'asha@work.in')={'name': 'Asha', 'email': 'asha@work.in'}
NOT_GIVEN  (readable in signatures and logs)


**Frozen `datetime.now()` defaults**: the same trap with time. `created_at=now()` is
computed once, so every order made without an explicit time gets the timestamp of the
moment the function was **defined**. Use the `None` pattern (the cell compares
timestamps instead of printing them, since they change every run).

In [584]:
def create_order_buggy(
    order_id: int,
    created_at: dt.datetime = dt.datetime.now(),  # noqa: B008
) -> dict[str, Any]:
    """Create an order (buggy: created_at is frozen at definition time)."""
    return {"order_id": order_id, "created_at": created_at}


def create_order(order_id: int, created_at: dt.datetime | None = None) -> dict[str, Any]:
    """Create an order stamped with the current time unless one is given."""
    if created_at is None:
        created_at = dt.datetime.now()
    return {"order_id": order_id, "created_at": created_at}


definition_time = create_order_buggy.__defaults__[0]
later_order = create_order_buggy(1002)
print(f"{later_order['created_at'] is definition_time=}  (frozen)")
print(f"{create_order(1003)['created_at'] > definition_time=}  (fresh per call)")

later_order['created_at'] is definition_time=True  (frozen)
create_order(1003)['created_at'] > definition_time=True  (fresh per call)


**Predict the output.**

```python
def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags

print(add_tag("new"), add_tag("sale"), add_tag("eco", []))
```

<details><summary>Answer</summary>

`['new', 'sale'] ['new', 'sale'] ['eco']`. Both of the first two calls append to the
**one** default list, and both return that same list object; by the time `print`
runs, it holds both tags. The third call passes its own list, so it is unaffected.

</details>

Run the next cell to check your prediction.

In [585]:
def add_tag(tag, tags=[]):  # noqa: B006
    tags.append(tag)
    return tags


print(add_tag("new"), add_tag("sale"), add_tag("eco", []))

['new', 'sale'] ['new', 'sale'] ['eco']


### 1.10.6 First-class and higher-order functions

Functions are ordinary objects: you can bind them to other names, store them in lists
and dicts, pass them as arguments, and return them. A **higher-order** function is one
that takes or returns a function.

**In variables, lists and dicts**:

In [586]:
def strip_spaces(text: str) -> str:
    """Remove surrounding whitespace."""
    return text.strip()


clean_text = strip_spaces  # no call: the function object itself
text_cleaners = [strip_spaces, str.lower, str.title]
formatter_by_style = {"upper": str.upper, "lower": str.lower, "title": str.title}

raw_name = "  aSHA kUMAR "
print(f"{clean_text(raw_name)=}")
for cleaner in text_cleaners:
    raw_name = cleaner(raw_name)
print(f"{raw_name=}")
print(f"{formatter_by_style['upper']('masala chai')=}")

clean_text(raw_name)='aSHA kUMAR'
raw_name='Asha Kumar'
formatter_by_style['upper']('masala chai')='MASALA CHAI'


**Callbacks**: the caller decides what happens next by passing functions. Same idea as
JS callbacks; in Python they are usually plain functions or `lambda`s.

In [587]:
def process_payment(
    amount: int, on_success: Callable[[int], str], on_failure: Callable[[str], str]
) -> str:
    """Pretend to charge amount and call the matching callback."""
    if amount <= 0:
        return on_failure("amount must be positive")
    return on_success(amount)


def send_receipt(amount: int) -> str:
    """Success callback."""
    return f"receipt sent for ₹{amount}"


print(process_payment(889, send_receipt, lambda reason: f"failed: {reason}"))
print(process_payment(0, send_receipt, lambda reason: f"failed: {reason}"))

receipt sent for ₹889
failed: amount must be positive


A function can also **return** a function, configured by its arguments (how it
remembers `percent` is a closure, Unit 1.11):

In [588]:
def make_discount(percent: int) -> Callable[[float], float]:
    """Return a function that applies a fixed discount."""

    def apply(price: float) -> float:
        return round(price * (100 - percent) / 100, 2)

    return apply


festive_discount = make_discount(20)
print(f"{festive_discount(1000)=}, {festive_discount.__qualname__=}")

festive_discount(1000)=800.0, festive_discount.__qualname__='make_discount.<locals>.apply'


**Dispatch tables with `operator` functions.** A dict from a key to a function
replaces a long `if` / `elif` chain, and the `operator` module provides every operator
as a function (`operator.lt` is `<`, `operator.add` is `+`), so you do not need a
`lambda` for each. The demo filters products by a rule read from a query string, such
as `price < 500`. (Lab 1.10, exercise 3, builds an arithmetic one.)

In [589]:
COMPARISONS: dict[str, Callable[[Any, Any], bool]] = {
    "<": operator.lt,
    "<=": operator.le,
    "==": operator.eq,
    "!=": operator.ne,
    ">=": operator.ge,
    ">": operator.gt,
}
catalog = [
    {"name": "tea", "price": 249, "rating": 4.5},
    {"name": "ghee", "price": 640, "rating": 4.8},
    {"name": "dal", "price": 162, "rating": 3.9},
]


def filter_products(products: list[dict[str, Any]], rule: str) -> list[str]:
    """Apply a 'field operator value' rule such as 'price < 500'."""
    field_name, symbol, raw_value = rule.split()
    compare = COMPARISONS[symbol]
    get_field = operator.itemgetter(field_name)
    threshold = float(raw_value)
    return [product["name"] for product in products if compare(get_field(product), threshold)]


print(f"{filter_products(catalog, 'price < 500')=}")
print(f"{filter_products(catalog, 'rating >= 4.5')=}")

filter_products(catalog, 'price < 500')=['tea', 'dal']
filter_products(catalog, 'rating >= 4.5')=['tea', 'ghee']


### 1.10.7 `lambda`

`lambda parameters: expression` creates a small anonymous function. The body is a
**single expression** (its value is returned): no statements, no assignments (except
`:=`), no annotations, no docstring.

In [590]:
print(f"{(lambda price: price * 2)(249)=}")
print(f"{(lambda first, second=10: first + second)(5)=}")
print(f"{compile_error('lambda order: return order')=}")

(lambda price: price * 2)(249)=498
(lambda first, second=10: first + second)(5)=15
compile_error('lambda order: return order')='SyntaxError: invalid syntax'


**`key=` usage** is the classic place for a `lambda`: a one-off function passed
straight to `sorted`, `min`, `max` or `groupby`. When the key is just an item or an
attribute, `operator.itemgetter` / `attrgetter` say the same thing.

In [591]:
print(f"{[p['name'] for p in sorted(catalog, key=lambda product: product['price'])]=}")
print(f"{max(catalog, key=lambda product: product['rating'])['name']=}")
print(f"{[p['name'] for p in sorted(catalog, key=operator.itemgetter('price'))]=}")

[p['name'] for p in sorted(catalog, key=lambda product: product['price'])]=['dal', 'tea', 'ghee']
max(catalog, key=lambda product: product['rating'])['name']='ghee'
[p['name'] for p in sorted(catalog, key=operator.itemgetter('price'))]=['dal', 'tea', 'ghee']


**`<lambda>` in tracebacks**: every lambda is named `<lambda>`, so an error inside one
tells you little about *which* lambda failed. A `def` has a real name.

In [592]:
def failing_frame_name(func: Callable[[], object]) -> str:
    """Call func and return the name of the innermost frame in the traceback."""
    try:
        func()
    except Exception as error:
        return traceback.extract_tb(error.__traceback__)[-1].name
    return "no error"


def unit_price_from_total() -> float:
    """Divide by a zero quantity on purpose."""
    return 498 / 0


print(f"{failing_frame_name(lambda: 498 / 0)=}")
print(f"{failing_frame_name(unit_price_from_total)=}")
print(f"{(lambda: 0).__name__=}")

failing_frame_name(lambda: 498 / 0)='<lambda>'
failing_frame_name(unit_price_from_total)='unit_price_from_total'
(lambda: 0).__name__='<lambda>'


**When to prefer `def`**: as soon as the function gets a name, needs more than one
expression, deserves a docstring or a type hint, or is reused. Assigning a lambda to a
name is the clearest sign (PEP 8 says so, and `ruff` rule E731 flags it); the `def`
version is no longer and has a proper `__name__`.

In [593]:
add_delivery_fee = lambda total: total + 40  # noqa: E731


def add_delivery_fee_def(total: int) -> int:
    """Add the flat delivery fee."""
    return total + 40


print(f"{add_delivery_fee.__name__=}, {add_delivery_fee_def.__name__=}")

add_delivery_fee.__name__='<lambda>', add_delivery_fee_def.__name__='add_delivery_fee_def'


### 1.10.8 Docstrings and annotations

**PEP 257 docstrings**: a string literal as the first statement of a function, class
or module. Use triple double quotes; start with a one-line summary in the imperative
("Return …", not "Returns …"), ending with a period; for more, add a blank line and
the details.

**Google style** adds named sections that tools such as Sphinx (with napoleon) and
IDEs understand:

```python
def split_bill(total, people):
    """Split total into equal shares.

    Args:
        total: Amount in rupees.
        people: Number of people, at least 1.

    Returns:
        One share per person.

    Raises:
        ValueError: If people is less than 1.
    """
```

`help()` prints the signature and the docstring; `inspect.getdoc()` returns the
docstring with the indentation cleaned up.

In [594]:
def delivery_eta_days(distance_km: float, *, express: bool = False) -> int:
    """Estimate delivery time in whole days.

    Args:
        distance_km: Road distance from the warehouse.
        express: Use the express courier (twice as fast).

    Returns:
        Days until delivery, at least 1.

    Raises:
        ValueError: If distance_km is negative.
    """
    if distance_km < 0:
        raise ValueError("distance cannot be negative")
    speed_km_per_day = 800 if express else 400
    return max(1, math.ceil(distance_km / speed_km_per_day))


help(delivery_eta_days)

Help on function delivery_eta_days in module __main__:

delivery_eta_days(distance_km: float, *, express: bool = False) -> int
    Estimate delivery time in whole days.

    Args:
        distance_km: Road distance from the warehouse.
        express: Use the express courier (twice as fast).

    Returns:
        Days until delivery, at least 1.

    Raises:
        ValueError: If distance_km is negative.



`inspect.getdoc` gives the cleaned text, useful for tooling (the first line is the summary):

In [595]:
print(f"{inspect.getdoc(delivery_eta_days).splitlines()[0]=}")
print(f"{delivery_eta_days(1200)=}, {delivery_eta_days(1200, express=True)=}")

inspect.getdoc(delivery_eta_days).splitlines()[0]='Estimate delivery time in whole days.'
delivery_eta_days(1200)=3, delivery_eta_days(1200, express=True)=2


**Annotations are not enforced.** Python stores them and tools read them (type
checkers in Phase 4, FastAPI and Pydantic in Phase 6), but the interpreter does not
check arguments against them. A wrong type passes through until some operation
fails, or silently produces nonsense.

In [596]:
def double_quantity(quantity: int) -> int:
    """Double a quantity."""
    return quantity * 2


print(f"{double_quantity(3)=}")
print(f"{double_quantity('3')=}  (no error: str * 2 repeats the text)")
print(f"{double_quantity.__annotations__=}")

double_quantity(3)=6
double_quantity('3')='33'  (no error: str * 2 repeats the text)
double_quantity.__annotations__={'quantity': <class 'int'>, 'return': <class 'int'>}


**Lazy annotation evaluation** (*New in 3.14*, PEP 649 / PEP 749). Annotations are no
longer evaluated when the `def` runs; they are computed only when someone asks. So a
function can name a class that is defined **later** (no quotes, no
`from __future__ import annotations`). The `annotationlib` module asks in
different formats:

| Format | Returns |
| --- | --- |
| `Format.VALUE` | the evaluated objects (raises `NameError` if a name is still undefined) |
| `Format.FORWARDREF` | values where possible, `ForwardRef` placeholders otherwise |
| `Format.STRING` | the annotations as source strings |

In [597]:
if sys.version_info >= (3, 14):
    import annotationlib

    def print_invoice(invoice: Invoice) -> Receipt:  # noqa: F821
        """Annotations name classes that do not exist yet."""
        return Receipt()  # noqa: F821

    Format = annotationlib.Format
    print(f"{annotationlib.get_annotations(print_invoice, format=Format.STRING)=}")
    forward_refs = annotationlib.get_annotations(print_invoice, format=Format.FORWARDREF)
    print(f"{type(forward_refs['invoice']).__name__=}")
    value_error = raised_error_name(lambda: print_invoice.__annotations__)
    print(f"VALUE before the classes exist: {value_error}")

    class Invoice:
        """Defined after the function."""

    class Receipt:
        """Defined after the function."""

    print(f"{print_invoice.__annotations__=}")
else:
    print("lazy annotations need Python 3.14; skipped")

annotationlib.get_annotations(print_invoice, format=Format.STRING)={'invoice': 'Invoice', 'return': 'Receipt'}
type(forward_refs['invoice']).__name__='ForwardRef'
VALUE before the classes exist: NameError
print_invoice.__annotations__={'invoice': <class '__main__.Invoice'>, 'return': <class '__main__.Receipt'>}


### 1.10.9 Recursion basics

A recursive function calls itself on a smaller problem. It needs a **base case** that
returns without recursing; every other call must move toward it.

```text
factorial(4) = 4 * factorial(3)
                 = 3 * factorial(2)
                     = 2 * factorial(1)
                         = 1            (base case)
```

The next cell writes factorial recursively and iteratively and checks both against
`math.factorial`.

In [598]:
def factorial_recursive(number: int) -> int:
    """Return number! recursively."""
    if number <= 1:  # base case
        return 1
    return number * factorial_recursive(number - 1)


def factorial_iterative(number: int) -> int:
    """Return number! with a loop."""
    product = 1
    for factor in range(2, number + 1):
        product *= factor
    return product


print(f"{factorial_recursive(10)=}, {factorial_iterative(10)=}")
versions_agree = all(
    factorial_recursive(n) == factorial_iterative(n) == math.factorial(n) for n in range(30)
)
print(f"{versions_agree=}  (0! to 29!)")

factorial_recursive(10)=3628800, factorial_iterative(10)=3628800
versions_agree=True  (0! to 29!)


Recursion fits naturally nested data, such as summing a nested price structure,
where a loop would need an explicit stack (1.6.13).

In [599]:
def total_price(node: int | list[Any]) -> int:
    """Sum all ints in an arbitrarily nested list."""
    if isinstance(node, int):  # base case
        return node
    return sum(total_price(child) for child in node)


gift_hamper = [249, [640, [162, 80]], [], 525]
print(f"{total_price(gift_hamper)=}")

total_price(gift_hamper)=1656


**Recursion-limit preview.** Every call adds a frame to the call stack. CPython caps
the depth (`sys.getrecursionlimit()`, usually 1000) and raises `RecursionError`
instead of crashing. Python does **not** optimize tail calls, so deep recursion
should be rewritten as a loop. A missing base case hits the limit immediately. Unit
1.12 looks at the call stack in detail.

In [600]:
def countdown_without_base_case(number: int) -> int:
    """Never stops: there is no base case."""
    return countdown_without_base_case(number - 1)


print(f"{sys.getrecursionlimit()=}")
print(f"no base case: {raised_error_name(lambda: countdown_without_base_case(5))}")
print(f"too deep: {raised_error_name(lambda: factorial_recursive(50_000))}")
print(f"{factorial_iterative(5_000).bit_length()=}  (the loop has no depth limit)")

sys.getrecursionlimit()=1000
no base case: RecursionError
too deep: RecursionError
factorial_iterative(5_000).bit_length()=54233  (the loop has no depth limit)


### Lab 1.10

Three exercises. For each, replace the `raise NotImplementedError` in the stub, then
run the check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — `build_query_string(base_url, /, *, page=1, page_size=20, **filters)`

Return `base_url + "?" + query`, where the query lists `page`, then `page_size`, then
every filter **in the order given**, URL-encoded with `urllib.parse.urlencode`
(imported in the setup cell; spaces become `+`).

- Filters whose value is `None` are skipped.
- Raise `ValueError` if `page < 1` or `page_size` is not between 1 and 100.
- Keep the exact signature: `base_url` positional-only (so a filter may be called
  `base_url`), `page` and `page_size` keyword-only. The two signature checks
  ("page is keyword-only", "parameter kinds") already pass on the stub; they guard
  against changing it.

In [601]:
def build_query_string(
    base_url: str, /, *, page: int = 1, page_size: int = 20, **filters: object
) -> str:
    """Return base_url with page, page_size and the non-None filters as a query string."""
    raise NotImplementedError("Lab 1.10: implement build_query_string()")

Run the checks.

In [602]:
run_checks(
    [
        (
            "defaults",
            lambda: build_query_string("/api/products"),
            "/api/products?page=1&page_size=20",
        ),
        (
            "filters in call order",
            lambda: build_query_string("/api/products", page=2, category="tea", max_price=500),
            "/api/products?page=2&page_size=20&category=tea&max_price=500",
        ),
        (
            "None filters skipped",
            lambda: build_query_string("/search", q="masala chai", city=None),
            "/search?page=1&page_size=20&q=masala+chai",
        ),
        (
            "filter named base_url",
            lambda: build_query_string("/api", base_url="mirror"),
            "/api?page=1&page_size=20&base_url=mirror",
        ),
        (
            "page is keyword-only",
            lambda: raised_error_name(lambda: build_query_string("/api", 2)),
            "TypeError",
        ),
        (
            "page 0",
            lambda: raised_error_name(lambda: build_query_string("/api", page=0)),
            "ValueError",
        ),
        (
            "page_size 101",
            lambda: raised_error_name(lambda: build_query_string("/api", page_size=101)),
            "ValueError",
        ),
        (
            "parameter kinds",
            lambda: [
                parameter.kind.name
                for parameter in inspect.signature(build_query_string).parameters.values()
            ],
            ["POSITIONAL_ONLY", "KEYWORD_ONLY", "KEYWORD_ONLY", "VAR_KEYWORD"],
        ),
    ]
)

FAIL  defaults: raised NotImplementedError: Lab 1.10: implement build_query_string()
FAIL  filters in call order: raised NotImplementedError: Lab 1.10: implement build_query_string()
FAIL  None filters skipped: raised NotImplementedError: Lab 1.10: implement build_query_string()
FAIL  filter named base_url: raised NotImplementedError: Lab 1.10: implement build_query_string()
PASS  page is keyword-only
FAIL  page 0: expected 'ValueError', got 'NotImplementedError'
FAIL  page_size 101: expected 'ValueError', got 'NotImplementedError'
PASS  parameter kinds
2/8 checks passed


#### Exercise 2 — `apply_pipeline(value, *transforms)`

Apply each transform to the result of the previous one, left to right, and return the
final value. With no transforms, return `value` itself (the same object).

In [603]:
def apply_pipeline(value: Any, *transforms: Callable[[Any], Any]) -> Any:
    """Return value passed through every transform, left to right."""
    raise NotImplementedError("Lab 1.10: implement apply_pipeline()")

Run the checks.

In [604]:
slug_steps = [str.strip, str.lower, lambda text: text.replace(" ", "-")]
original_list = [3, 1, 2]
run_checks(
    [
        ("slug pipeline", lambda: apply_pipeline("  Masala Chai ", *slug_steps), "masala-chai"),
        ("numbers", lambda: apply_pipeline(10, lambda n: n + 5, lambda n: n * 2), 30),
        ("order matters", lambda: apply_pipeline(10, lambda n: n * 2, lambda n: n + 5), 25),
        (
            "no transforms returns the same object",
            lambda: apply_pipeline(original_list) is original_list,
            True,
        ),
        ("built-ins as transforms", lambda: apply_pipeline([3, 1, 2], sorted, tuple), (1, 2, 3)),
        ("types can change", lambda: apply_pipeline("42", int, float, str), "42.0"),
    ]
)

FAIL  slug pipeline: raised NotImplementedError: Lab 1.10: implement apply_pipeline()
FAIL  numbers: raised NotImplementedError: Lab 1.10: implement apply_pipeline()
FAIL  order matters: raised NotImplementedError: Lab 1.10: implement apply_pipeline()
FAIL  no transforms returns the same object: raised NotImplementedError: Lab 1.10: implement apply_pipeline()
FAIL  built-ins as transforms: raised NotImplementedError: Lab 1.10: implement apply_pipeline()
FAIL  types can change: raised NotImplementedError: Lab 1.10: implement apply_pipeline()
0/6 checks passed


#### Exercise 3 — `calculate(operation_name, left, right)` with a dispatch table

Fill the `OPERATIONS` dict with **`operator` module functions** (no `lambda`) for these
names: `"add"`, `"subtract"`, `"multiply"`, `"divide"` (true division),
`"floor_divide"`, `"modulo"`, `"power"`. Then `calculate` looks the name up and calls
the function: no `if` / `elif` chain. An unknown name raises
`ValueError("unknown operation: <name>")`. Division by zero raises
`ZeroDivisionError` naturally.

In [605]:
OPERATIONS: dict[str, Callable[[float, float], float]] = {}  # fill in


def calculate(operation_name: str, left: float, right: float) -> float:
    """Apply the named operation from OPERATIONS to left and right."""
    raise NotImplementedError("Lab 1.10: implement calculate()")

Run the checks.

In [606]:
run_checks(
    [
        ("add", lambda: calculate("add", 2, 3), 5),
        ("subtract", lambda: calculate("subtract", 2, 3), -1),
        ("multiply", lambda: calculate("multiply", 249, 2), 498),
        ("divide", lambda: calculate("divide", 7, 2), 3.5),
        ("floor_divide", lambda: calculate("floor_divide", -7, 2), -4),
        ("modulo", lambda: calculate("modulo", -7, 3), 2),
        ("power", lambda: calculate("power", 2, 10), 1024),
        (
            "divide by zero",
            lambda: raised_error_name(lambda: calculate("divide", 1, 0)),
            "ZeroDivisionError",
        ),
        (
            "unknown operation",
            lambda: raised_error_name(lambda: calculate("sqrt", 9, 0)),
            "ValueError",
        ),
        (
            "table has the seven names",
            lambda: sorted(OPERATIONS),
            ["add", "divide", "floor_divide", "modulo", "multiply", "power", "subtract"],
        ),
        ("table uses operator functions", lambda: OPERATIONS.get("add") is operator.add, True),
    ]
)

FAIL  add: raised NotImplementedError: Lab 1.10: implement calculate()
FAIL  subtract: raised NotImplementedError: Lab 1.10: implement calculate()
FAIL  multiply: raised NotImplementedError: Lab 1.10: implement calculate()
FAIL  divide: raised NotImplementedError: Lab 1.10: implement calculate()
FAIL  floor_divide: raised NotImplementedError: Lab 1.10: implement calculate()
FAIL  modulo: raised NotImplementedError: Lab 1.10: implement calculate()
FAIL  power: raised NotImplementedError: Lab 1.10: implement calculate()
FAIL  divide by zero: expected 'ZeroDivisionError', got 'NotImplementedError'
FAIL  unknown operation: expected 'ValueError', got 'NotImplementedError'
FAIL  table has the seven names: expected ['add', 'divide', 'floor_divide', 'modulo', 'multiply', 'power', 'subtract'], got []
FAIL  table uses operator functions: expected True, got False
0/11 checks passed


### Recap — Unit 1.10

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `def` | A statement that builds a function object at run time (`MAKE_FUNCTION`); no hoisting | function declarations are hoisted |
| Function attributes | `__name__`, `__qualname__`, `__doc__`, `__defaults__`, `__kwdefaults__`, `__code__`, `__globals__`, `__annotations__`, `__dict__` | `name`, `length` |
| Return values | No `return` → `None`; `return a, b` returns one tuple | `undefined`; return an array/object |
| Parameter kinds | `pos_only, /, standard, *args, kw_only, **kwargs` | rest parameters, destructured options |
| `*args` / `**kwargs` | Extra positionals as a tuple / extra keywords as a dict | `...args` (array) |
| `inspect.signature` | Read parameter kinds and defaults (frameworks rely on it) | `fn.length`, `toString()` parsing |
| Call unpacking | `f(*values, **options)`; forwarding wrapper `f(*args, **kwargs)` | `f(...values)` |
| Defaults | Evaluated once at definition; never use a mutable default | evaluated per call |
| `None` / `sentinel()` | `None` for "fresh object per call"; `sentinel()` (3.15) when `None` is a valid value | `undefined` vs `null` |
| First-class functions | Store, pass and return functions; dispatch tables with `operator` | same |
| `lambda` | One expression; fine for `key=`; named functions get `def` | arrow functions (full bodies allowed) |
| Docstrings | PEP 257 summary line + Google-style sections; `help()` shows them | JSDoc |
| Annotations | Stored, not enforced; lazily evaluated since 3.14 (`annotationlib`) | TypeScript types (erased) |
| Recursion | Base case first; limit ~1000 frames; no tail-call optimization | similar stack limits |

**Further reading**

- [Tutorial: defining functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions),
  [more on defining functions](https://docs.python.org/3/tutorial/controlflow.html#more-on-defining-functions)
- [Function definitions (reference)](https://docs.python.org/3/reference/compound_stmts.html#function-definitions),
  [user-defined function attributes](https://docs.python.org/3/reference/datamodel.html#user-defined-functions)
- [`inspect.signature`](https://docs.python.org/3/library/inspect.html#introspecting-callables-with-the-signature-object),
  [`operator`](https://docs.python.org/3/library/operator.html),
  [`annotationlib`](https://docs.python.org/3/library/annotationlib.html)
- [PEP 257 — Docstring conventions](https://peps.python.org/pep-0257/),
  [Google Python style guide: docstrings](https://google.github.io/styleguide/pyguide.html#38-comments-and-docstrings)
- [PEP 570 — Positional-only parameters](https://peps.python.org/pep-0570/),
  [PEP 3102 — Keyword-only arguments](https://peps.python.org/pep-3102/),
  [PEP 468 — Preserving keyword argument order](https://peps.python.org/pep-0468/)
- [PEP 661 — Sentinel values](https://peps.python.org/pep-0661/),
  [PEP 649 — Deferred evaluation of annotations](https://peps.python.org/pep-0649/),
  [PEP 749 — Implementing PEP 649](https://peps.python.org/pep-0749/)

## Unit 1.11 — Scope, Namespaces and Closures

**Objectives**

- Name the four kinds of namespace (built-in, module, class, function) and inspect
  them with `globals()`, `locals()` and the `builtins` module.
- Resolve any name with the LEGB rule, and spot a shadowed built-in.
- Explain why Python has no block scope, and which constructs *do* create a scope.
- Explain `UnboundLocalError` from the compiler's point of view (`co_varnames`,
  `LOAD_FAST_CHECK`).
- Use `global` and `nonlocal` correctly, and prefer passing state explicitly.
- Read a closure's cell objects, and fix the late-binding trap.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `let` / `const` block scope | no block scope: function, class, module, comprehension | A name set inside `if` / `for` survives the block |
| `var` hoisting (`undefined`) | assignment anywhere makes a name local | Reading it early raises `UnboundLocalError`, not `undefined` |
| `let` temporal dead zone | `UnboundLocalError` | Same idea, decided at compile time |
| Assigning an outer variable just works | `nonlocal` / `global` needed to **rebind** | Without them the assignment creates a new local |
| `globalThis`, `window` | `globals()` (module), `builtins` | Each module has its own global namespace |
| Closures capture variables | Closures capture **cells** | Same late binding; no per-iteration `let` binding in loops |
| `for (let i …)` callbacks | `lambda i=i: …` or `functools.partial` | Python loops reuse one variable |

The next cell is this unit's setup (`compile_error` is the helper from Unit 1.8).

In [607]:
import builtins
import dis
import functools
import inspect
import io
import math
import re  # noqa: F401 (used in Lab 1.11)
import sys
import timeit
from collections.abc import Callable
from typing import Any


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def compile_error(source: str) -> str:
    """Compile source without running it; return "SyntaxError: <message>" or "compiles"."""
    try:
        compile(source, "<demo>", "exec")
    except SyntaxError as error:
        return f"SyntaxError: {error.msg}"
    return "compiles"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.11.1 Namespaces

A **namespace** is a mapping from names to objects. Python keeps several of them, with
different lifetimes:

| Namespace | Created | Lives until | Look at it with |
| --- | --- | --- | --- |
| Built-in | interpreter start | interpreter exit | `import builtins; vars(builtins)` |
| Module (global) | first import of the module | interpreter exit (usually) | `globals()`, `module.__dict__` |
| Class | the `class` body runs | becomes the class's `__dict__` | `vars(SomeClass)` |
| Function (local) | **each call** | the call returns (unless a closure keeps a cell) | `locals()` inside the function |

**Module namespace.** In a notebook the module is `__main__`. `globals()` returns the
module's own `__dict__`, the real thing, not a copy: writing to it creates a variable.

In [608]:
main_module = sys.modules[__name__]
print(f"{__name__=}")
print(f"{globals() is main_module.__dict__=}")

globals()["created_through_globals"] = "hello"
print(f"{created_through_globals=}")  # noqa: F821 (created on the line above)
print(f"{'show_size' in globals()=}, {'len' in globals()=}")

__name__='__main__'
globals() is main_module.__dict__=True
created_through_globals='hello'
'show_size' in globals()=True, 'len' in globals()=False


Note the last line: `len` is **not** in `globals()`. Built-ins live in their own
namespace, which is searched after the module namespace (1.11.2).

**Function namespace.** Every call gets a fresh local namespace. Inside a function,
`locals()` returns a dict *snapshot* of it (*Changed in 3.13*, PEP 667: writes to the
snapshot never change the real local variables).

In [609]:
def price_with_tax(base_price: int, tax_percent: int = 18) -> int:
    """Show the local namespace of one call."""
    tax_amount = base_price * tax_percent // 100
    print(f"  locals()={locals()}")
    return base_price + tax_amount


print(price_with_tax(500))
print(price_with_tax(1000, 5))


def try_to_write_locals() -> int:
    """Writing to the locals() snapshot does not create a variable."""
    local_snapshot = locals()
    local_snapshot["discount"] = 50
    return len(local_snapshot)


print(f"{try_to_write_locals()=}, the variable 'discount' was never created")

  locals()={'base_price': 500, 'tax_percent': 18, 'tax_amount': 90}
590
  locals()={'base_price': 1000, 'tax_percent': 5, 'tax_amount': 50}
1050
try_to_write_locals()=1, the variable 'discount' was never created


**Class namespace.** A `class` body runs once, in its own namespace, which then becomes
the class's `__dict__` (wrapped in a read-only `mappingproxy`). The trap: names
defined in the class body are **not** visible inside methods. Methods must go through
`self` or the class.

In [610]:
class MenuItem:
    """A class whose body defines a name and two methods."""

    tax_percent = 5
    print(f"  inside the class body: {sorted(locals())}")

    def tax_without_self(self) -> int:
        """Looks for tax_percent in the local, global and built-in scopes only."""
        return tax_percent  # noqa: F821 (deliberate: class scope is skipped)

    def tax_with_self(self) -> int:
        """Finds the class attribute through the instance."""
        return self.tax_percent


print(f"{type(vars(MenuItem)).__name__=}, {vars(MenuItem)['tax_percent']=}")
print(f"{MenuItem().tax_with_self()=}")
print(f"tax_without_self(): {raised_error_name(MenuItem().tax_without_self)}")

  inside the class body: ['__doc__', '__firstlineno__', '__module__', '__qualname__', 'tax_percent']
type(vars(MenuItem)).__name__='mappingproxy', vars(MenuItem)['tax_percent']=5
MenuItem().tax_with_self()=5
tax_without_self(): NameError


**Built-in namespace.** `len`, `print`, `ValueError` and friends live in the
`builtins` module. Each module also has `__builtins__` (an implementation detail:
in `__main__` it is the module itself, elsewhere its `__dict__`). Always use
`import builtins` in your own code.

In [611]:
builtin_names = [name for name in vars(builtins) if not name.startswith("_")]
print(f"{len(builtin_names)=} public built-in names")
print(f"{builtins.len is len=}, {builtins.ValueError is ValueError=}")
print(f"first few: {sorted(builtin_names)[:6]}")
print(f"{type(__builtins__).__name__=}  (in __main__; a dict in other modules)")

len(builtin_names)=155 public built-in names
builtins.len is len=True, builtins.ValueError is ValueError=True
first few: ['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError']
type(__builtins__).__name__='module'  (in __main__; a dict in other modules)


#### Under the hood: function locals are not a dict

Module and class namespaces are real dicts. Function locals are not: the compiler
gives each local a slot number, and the frame stores them in a C array ("fast
locals"). `locals()` builds a dict from that array on demand. The code object lists
the slots in `co_varnames` and counts them in `co_nlocals`.

In [612]:
price_code = price_with_tax.__code__
print(f"{price_code.co_varnames=}")
print(f"{price_code.co_nlocals=}, {price_code.co_argcount=}")
print(f"{inspect.isfunction(price_with_tax)=}, {type(vars(MenuItem)).__name__=}")

price_code.co_varnames=('base_price', 'tax_percent', 'tax_amount')
price_code.co_nlocals=3, price_code.co_argcount=2
inspect.isfunction(price_with_tax)=True, type(vars(MenuItem)).__name__='mappingproxy'


### 1.11.2 The LEGB rule

To find the object a name refers to, Python searches four scopes in order and stops
at the first hit:

```mermaid
flowchart LR
    lookup(["name lookup"]) --> local["L: Local<br/>current function"]
    local -- "not found" --> enclosing["E: Enclosing<br/>outer functions, inside out"]
    enclosing -- "not found" --> global_scope["G: Global<br/>module namespace"]
    global_scope -- "not found" --> builtin["B: Built-in<br/>builtins module"]
    builtin -- "not found" --> error(["NameError"])
```

<details><summary>ASCII fallback</summary>

```text
name --> Local --> Enclosing (inside out) --> Global (module) --> Built-in --> NameError
         (stop at the first scope that has the name)
```

</details>

Class bodies are **not** an enclosing scope for the methods inside them (1.11.1).

The next cell defines the same name at three levels. Each print shows which binding
the lookup reaches first.

In [613]:
currency_symbol = "Rs"  # global


def outer_report() -> Callable[[], str]:
    """Enclosing function with its own currency_symbol."""
    currency_symbol = "₹"  # enclosing

    def inner_report() -> str:
        """Uses the nearest currency_symbol and the built-in len."""
        return f"{currency_symbol} (name length {len(currency_symbol)})"

    return inner_report


def global_report() -> str:
    """No local or enclosing binding: falls through to the global."""
    return currency_symbol


print(f"{outer_report()()=}")
print(f"{global_report()=}")

outer_report()()='₹ (name length 1)'
global_report()='Rs'


`inspect.getclosurevars` reports where each free name of a function was found:
enclosing (`nonlocals`), global or built-in.

In [614]:
print(inspect.getclosurevars(outer_report()))
print(inspect.getclosurevars(global_report))

ClosureVars(nonlocals={'currency_symbol': '₹'}, globals={}, builtins={'len': <built-in function len>}, unbound=set())
ClosureVars(nonlocals={}, globals={'currency_symbol': 'Rs'}, builtins={}, unbound=set())


**Shadowing built-ins.** Because the global scope is searched before the built-in
scope, `list = [...]` at module level hides the built-in `list` for the whole module.
The error comes much later, far from the cause. `del list` removes the global and
uncovers the built-in again.

The next cell runs the mistake in a throwaway namespace, so this notebook's `list`
stays intact.

In [615]:
shadowing_module = """
list = ["idli", "dosa"]
try:
    letters = list("abc")
except TypeError as error:
    print(f"  {type(error).__name__}: {error}")
del list
print(f"  after del: {list('abc')}")
"""
exec(shadowing_module, {})
print(f"{list('ok')=}  (this notebook's list is untouched)")

  TypeError: 'list' object is not callable
  after del: ['a', 'b', 'c']
list('ok')=['o', 'k']  (this notebook's list is untouched)


A parameter or local with a built-in's name shadows it only inside that function.
That is common and usually harmless (`id`, `type`, `input` are popular parameter
names), until the function needs the built-in itself.

In [616]:
def describe_order(id: int, type: str) -> str:
    """Parameters named id and type hide the built-ins inside this function."""
    try:
        return f"order {id} of kind {type}, class name {type(id).__name__}"
    except TypeError as error:
        return f"{builtins.type(error).__name__}: {error}"


print(describe_order(42, "express"))
order_id = 42
print(f"{type(order_id).__name__=}  (outside the function, type is the built-in again)")

TypeError: 'str' object is not callable
type(order_id).__name__='int'  (outside the function, type is the built-in again)


**Predict the output.**

```python
stock = 100

def outer():
    stock = 50
    def inner():
        return stock
    stock = 25
    return inner()

print(outer(), stock)
```

<details><summary>Answer</summary>

`25 100`. `inner` finds `stock` in the enclosing scope, and it reads the variable
**when called**, by which time `outer` has rebound it to 25. The global `stock` is
never touched: both assignments in `outer` create an `outer` local.

</details>

Run the next cell to check your prediction.

In [617]:
stock = 100


def outer() -> int:
    """Rebind the enclosing variable before calling inner."""
    stock = 50

    def inner() -> int:
        """Reads stock from the enclosing scope at call time."""
        return stock

    stock = 25
    return inner()


print(outer(), stock)

25 100


### 1.11.3 No block scope

Only modules, classes, functions, comprehensions and lambdas create scopes. Blocks
(`if`, `for`, `while`, `with`, `try`) do **not**: a name bound inside them is an
ordinary variable of the surrounding function or module and survives the block.

In [618]:
def block_survivors(order_total: int) -> dict[str, Any]:
    """Names bound inside if / for / with / try are still there afterwards."""
    if order_total > 1000:
        shipping_label = "free shipping"
    else:
        shipping_label = "Rs 49 shipping"

    for line_number in range(1, 4):  # noqa: B007
        pass  # the loop variable keeps its last value

    with io.StringIO("receipt text") as log_file:
        pass  # the buffer is closed, but the name still exists

    try:
        parsed_quantity = int("3")
    except ValueError:
        parsed_quantity = 0

    return {
        "shipping_label": shipping_label,
        "line_number": line_number,
        "log_file_closed": log_file.closed,
        "parsed_quantity": parsed_quantity,
    }


print(block_survivors(1500))

{'shipping_label': 'free shipping', 'line_number': 3, 'log_file_closed': True, 'parsed_quantity': 3}


Two consequences:

- If a `for` loop never runs, its variable is **never bound**, and reading it later
  fails.
- `except ... as error` is the one exception to the rule: Python **deletes** `error`
  at the end of the `except` block (to break a reference cycle through the
  traceback). Copy it to another name if you need it later.

In [619]:
def last_item(items: list[str]) -> str:
    """Return the loop variable after the loop (broken for empty lists)."""
    for item in items:  # noqa: B007
        pass
    return item


def error_after_except() -> str:
    """The 'as' name is deleted when the except block ends."""
    try:
        int("not a number")
    except ValueError as error:
        saved_error = error
    return f"{type(saved_error).__name__}; error itself: {'error' in locals()}"


print(f"{last_item(['tea', 'coffee'])=}")
print(f"last_item([]): {raised_error_name(lambda: last_item([]))}")
print(f"{error_after_except()=}")

last_item(['tea', 'coffee'])='coffee'
last_item([]): UnboundLocalError
error_after_except()='ValueError; error itself: False'


**Comprehensions and lambdas do have their own scope.** The loop variable of a
comprehension does not leak (since Python 3; Python 2 list comprehensions leaked).
Since 3.12, list/dict/set comprehensions are *inlined* into the surrounding function
(PEP 709) for speed, but the isolation is kept. The one deliberate leak is the walrus
operator `:=`, which binds in the enclosing scope (PEP 572).

In [620]:
def comprehension_scope() -> dict[str, Any]:
    """Comprehension and lambda variables stay inside; := leaks on purpose."""
    prices = [120, 80, 250]
    doubled = [price * 2 for price in prices]
    with_tax = [(taxed := price * 118 // 100) for price in prices]
    add_one = lambda value: value + 1  # noqa: E731
    return {
        "doubled": doubled,
        "price leaked": "price" in locals(),
        "value leaked": "value" in locals(),
        "taxed (walrus)": taxed,
        "with_tax": with_tax,
        "add_one(1)": add_one(1),
    }


print(comprehension_scope())

{'doubled': [240, 160, 500], 'price leaked': False, 'value leaked': False, 'taxed (walrus)': 295, 'with_tax': [141, 94, 295], 'add_one(1)': 2}


### 1.11.4 Compile-time scope decisions

Scope is decided when the function is **compiled**, not while it runs. If a name is
assigned *anywhere* in a function body (`=`, `+=`, `for name in`, `import name`,
`def name`, `except ... as name`, …), it is local for the **whole** function, even on
lines before the assignment. Reading it before it has a value raises
`UnboundLocalError`.

In [621]:
stock_count = 10


def sell_one() -> int:
    """Reads stock_count, then assigns it: the compiler made it local."""
    print(f"  selling one of {stock_count}")  # noqa: F823
    stock_count -= 1
    return stock_count


try:
    sell_one()
except UnboundLocalError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{issubclass(UnboundLocalError, NameError)=}")

UnboundLocalError: cannot access local variable 'stock_count' where it is not associated with a value
issubclass(UnboundLocalError, NameError)=True


The assignment does not even need to run. A branch that can never execute still makes
the name local, because the decision was made from the source text.

In [622]:
def read_with_dead_branch() -> int:
    """An assignment that never runs still makes stock_count local."""
    if False:
        stock_count = 0
    return stock_count


print(f"read_with_dead_branch(): {raised_error_name(read_with_dead_branch)}")

read_with_dead_branch(): UnboundLocalError


The code object records the decision. `co_varnames` lists the local slots
(parameters first); `co_names` lists the global and attribute names the code uses.

In [623]:
def build_receipt(customer_name: str) -> str:
    """Uses a parameter, a local, a global function and an attribute."""
    greeting = customer_name.title()
    return f"{greeting}: {len(greeting)} letters, pi≈{math.pi:.2f}"


receipt_code = build_receipt.__code__
print(f"{receipt_code.co_varnames=}")
print(f"{receipt_code.co_names=}")
print(f"{sell_one.__code__.co_varnames=}  (stock_count is a local slot)")

receipt_code.co_varnames=('customer_name', 'greeting')
receipt_code.co_names=('title', 'len', 'math', 'pi')
sell_one.__code__.co_varnames=('stock_count',)  (stock_count is a local slot)


#### Under the hood: `LOAD_FAST_CHECK`

Locals are read with `LOAD_FAST` (or `LOAD_FAST_BORROW`, *New in 3.14*), which skips
the "is it bound?" test when the compiler can prove the variable has a value. When it
cannot (as in `sell_one`), it emits `LOAD_FAST_CHECK`, which raises
`UnboundLocalError` if the slot is empty. Globals use `LOAD_GLOBAL`. Opcode names vary
between versions.

In [624]:
for function in (sell_one, build_receipt):
    load_instructions = [
        f"{instruction.opname} {instruction.argval}"
        for instruction in dis.get_instructions(function)
        if instruction.opname.startswith(("LOAD_FAST", "LOAD_GLOBAL", "LOAD_ATTR"))
    ]
    print(f"{function.__name__}: {load_instructions}")

sell_one_opnames = {instruction.opname for instruction in dis.get_instructions(sell_one)}
print(f"{'LOAD_FAST_CHECK' in sell_one_opnames=}")

sell_one: ['LOAD_GLOBAL print', 'LOAD_FAST_CHECK stock_count', 'LOAD_FAST_BORROW stock_count', 'LOAD_FAST_BORROW stock_count']
build_receipt: ['LOAD_FAST_BORROW customer_name', 'LOAD_ATTR title', 'LOAD_FAST_BORROW greeting', 'LOAD_GLOBAL len', 'LOAD_FAST_BORROW greeting', 'LOAD_GLOBAL math', 'LOAD_ATTR pi']
'LOAD_FAST_CHECK' in sell_one_opnames=True


**JS bridge: `var` hoisting and the `let` TDZ.**

| | JavaScript `var` | JavaScript `let` / `const` | Python |
| --- | --- | --- | --- |
| Scope | function | block | function (no blocks) |
| Read before assignment | `undefined` (hoisted) | `ReferenceError` (temporal dead zone) | `UnboundLocalError` |
| Decided | at parse time | at parse time | at compile time |

```javascript
function sellOne() {
  console.log(stock);   // var: undefined; let: ReferenceError
  let stock = 9;
}
```

Python behaves like `let`: the name is reserved for the whole function, but has no
value until the assignment runs. The fixes are in 1.11.5: pass the value in, return
the new value, or declare `global` / `nonlocal`.

In [625]:
def sell_one_fixed(current_stock: int) -> int:
    """Take the value as a parameter and return the new value."""
    return current_stock - 1


stock_count = sell_one_fixed(stock_count)
print(f"{stock_count=}")

stock_count=9


### 1.11.5 `global` and `nonlocal`

| Statement | Lets a function **rebind** | Typical use |
| --- | --- | --- |
| `global name` | a module-level name | module-level counters, caches, configuration (sparingly) |
| `nonlocal name` | a name in the nearest enclosing function | closures that keep state (1.11.6) |

`global` makes assignments inside the function write to the module namespace.

In [626]:
request_count = 0


def record_request() -> int:
    """Rebind the module-level counter."""
    global request_count
    request_count += 1
    return request_count


record_request()
record_request()
print(f"{request_count=}")

request_count=2


**Rebinding vs mutating.** `global` / `nonlocal` are only needed to make the name
point at a **new object**. Calling a method that changes the object in place
(`append`, `update`, item assignment) is a read of the name, so no declaration is
needed.

In [627]:
audit_log: list[str] = []


def log_event(message: str) -> None:
    """Mutates the global list: no 'global' needed."""
    audit_log.append(message)


def reset_log_wrong() -> None:
    """Assignment creates a new local; the global list is unchanged."""
    audit_log = []  # noqa: F841 (deliberate: local variable, never used)


log_event("login")
log_event("checkout")
reset_log_wrong()
print(f"{audit_log=}  (reset_log_wrong did nothing)")

audit_log=['login', 'checkout']  (reset_log_wrong did nothing)


`nonlocal` does the same for an enclosing function's variable. Without it, `+=` inside
the inner function would make `total` local and fail with `UnboundLocalError`.

In [628]:
def make_cart_total() -> Callable[[int], int]:
    """Return an add function that keeps a running total."""
    total = 0

    def add(amount: int) -> int:
        """Rebind the enclosing total."""
        nonlocal total
        total += amount
        return total

    return add


add_to_cart = make_cart_total()
add_to_cart(249)
print(f"{add_to_cart(640)=}")

add_to_cart(640)=889


Both statements are checked at compile time: `nonlocal` must name a variable of an
enclosing *function* (not a global), and a name may not be used before its `global`
declaration.

In [629]:
print(compile_error("def outer():\n    nonlocal missing_name\n"))
print(compile_error("rate = 1\ndef show():\n    print(rate)\n    global rate\n"))
print(compile_error("def outer():\n    total = 0\n    def add():\n        nonlocal total\n"))

SyntaxError: no binding for nonlocal 'missing_name' found
SyntaxError: name 'rate' is used prior to global declaration
compiles


**Why global state hurts testing.** A function that reads or writes module state gives
different answers depending on what ran before it. Tests then pass alone and fail
together (or the other way round). Below, two "tests" of `process_order` share the
hidden `processed_order_ids` set.

In [630]:
processed_order_ids: set[int] = set()


def process_order(order_id: int) -> str:
    """Depends on hidden module state."""
    if order_id in processed_order_ids:
        return "duplicate"
    processed_order_ids.add(order_id)
    return "processed"


def test_new_order_is_processed() -> bool:
    return process_order(101) == "processed"


def test_repeat_order_is_duplicate() -> bool:
    process_order(101)
    return process_order(101) == "duplicate"


print(f"order A then B: {test_new_order_is_processed()}, {test_repeat_order_is_duplicate()}")
processed_order_ids.clear()
print(f"order B then A: {test_repeat_order_is_duplicate()}, {test_new_order_is_processed()}")

order A then B: True, True
order B then A: True, False


The fix is to make the state an **explicit argument** (or an object attribute, Phase
2). Each test then builds its own state, and the function has no hidden inputs.

In [631]:
def process_order_explicit(order_id: int, processed_ids: set[int]) -> str:
    """Same logic; the state is passed in."""
    if order_id in processed_ids:
        return "duplicate"
    processed_ids.add(order_id)
    return "processed"


print(f"{process_order_explicit(101, set())=}")
seen_ids = {101}
print(f"{process_order_explicit(101, seen_ids)=}")

process_order_explicit(101, set())='processed'
process_order_explicit(101, seen_ids)='duplicate'


### 1.11.6 Closures

A **closure** is a function that remembers variables of the enclosing function after
that function has returned. Unit 1.10.6 used one (`make_discount`) without explaining
it. The classic example is a counter: `make_counter` returns a function whose `count`
lives on between calls, and every call to `make_counter` creates independent state.

In [632]:
def make_counter(start: int = 0) -> Callable[[], int]:
    """Return a function that counts up from start."""
    count = start

    def increment() -> int:
        """Rebind the enclosing count."""
        nonlocal count
        count += 1
        return count

    return increment


ticket_counter = make_counter()
invoice_counter = make_counter(start=1000)
print(ticket_counter(), ticket_counter(), ticket_counter())
print(invoice_counter(), invoice_counter())

1 2 3
1001 1002


A closure that only **reads** the enclosing variable needs no `nonlocal`.
`make_multiplier` is a function factory: each returned function carries its own
`factor`.

In [633]:
def make_multiplier(factor: float) -> Callable[[float], float]:
    """Return a function that multiplies by factor."""

    def multiply(value: float) -> float:
        return value * factor

    return multiply


double = make_multiplier(2)
add_gst = make_multiplier(1.18)
print(f"{double(21)=}, {add_gst(500)=}")

double(21)=42, add_gst(500)=590.0


#### Under the hood: cells

The compiler sees that `count` is used by an inner function, so it does not store it
in a plain local slot. It stores it in a **cell object**, a tiny box holding one
reference. Both functions point at the same cell:

| Attribute | On | Holds |
| --- | --- | --- |
| `co_cellvars` | outer code object | names that inner functions capture |
| `co_freevars` | inner code object | names it reads from enclosing scopes |
| `__closure__` | inner function | a tuple of cell objects, one per free variable |
| `cell.cell_contents` | each cell | the current value |

In [634]:
print(f"{make_counter.__code__.co_cellvars=}")
print(f"{ticket_counter.__code__.co_freevars=}")
print(f"{type(ticket_counter.__closure__[0]).__name__=}")
print(f"{ticket_counter.__closure__[0].cell_contents=}")
print(f"{invoice_counter.__closure__[0].cell_contents=}")
print(f"{add_gst.__code__.co_freevars=}, {add_gst.__closure__[0].cell_contents=}")

make_counter.__code__.co_cellvars=('count',)
ticket_counter.__code__.co_freevars=('count',)
type(ticket_counter.__closure__[0]).__name__='cell'
ticket_counter.__closure__[0].cell_contents=3
invoice_counter.__closure__[0].cell_contents=1002
add_gst.__code__.co_freevars=('factor',), add_gst.__closure__[0].cell_contents=1.18


A cell holds the **variable**, not a frozen value. Two closures created in the same
call share their cells, and a change made after the closure was created is visible
through it. A function without free variables has `__closure__ = None`.

In [635]:
def make_price_tools(initial_price: int) -> tuple[Callable[[], int], Callable[[int], None]]:
    """Return a getter and a setter that share one cell."""
    price = initial_price

    def get_price() -> int:
        return price

    def set_price(new_price: int) -> None:
        nonlocal price
        price = new_price

    return get_price, set_price


get_price, set_price = make_price_tools(250)
set_price(199)
print(f"{get_price()=}")
print(f"{get_price.__closure__[0] is set_price.__closure__[0]=}")
print(f"{show_size.__closure__=}")

get_price()=199
get_price.__closure__[0] is set_price.__closure__[0]=True
show_size.__closure__=None


The bytecode shows the cell machinery: `MAKE_CELL` in the outer function, then
`STORE_DEREF` / `LOAD_DEREF` instead of `STORE_FAST` / `LOAD_FAST`. The inner function
starts with `COPY_FREE_VARS`, which copies the cells from `__closure__` into its
frame. Opcode names vary between versions.

In [636]:
for function in (make_counter, ticket_counter):
    opnames = [instruction.opname for instruction in dis.get_instructions(function)]
    cell_ops = [name for name in opnames if "DEREF" in name or "CELL" in name or "FREE" in name]
    print(f"{function.__qualname__}: {cell_ops}")

make_counter: ['MAKE_CELL', 'STORE_DEREF']
make_counter.<locals>.increment: ['COPY_FREE_VARS', 'LOAD_DEREF', 'STORE_DEREF', 'LOAD_DEREF']


> **Note:** JavaScript closures work the same way (they capture variables, not
> values). The difference is that JavaScript can assign an outer variable directly,
> while Python needs `nonlocal` to rebind it.

### 1.11.7 Late binding

Because closures capture variables, they read the value **when they are called**, not
when they are created. Functions made in a loop all share the loop variable, so they
all see its last value.

**Predict the output.**

```python
callbacks = [lambda: i for i in range(3)]
print([callback() for callback in callbacks])
```

<details><summary>Answer</summary>

`[2, 2, 2]`. The three lambdas share **one** variable `i` (one cell). By the time they
are called, the comprehension has finished and `i` is 2.

</details>

Run the next cell to check your prediction.

In [637]:
callbacks = [lambda: i for i in range(3)]  # noqa: B023
print([callback() for callback in callbacks])
print(f"{callbacks[0].__closure__[0] is callbacks[2].__closure__[0]=}")

[2, 2, 2]
callbacks[0].__closure__[0] is callbacks[2].__closure__[0]=True


**Fix 1: a default argument.** Defaults are evaluated once, when the function is
created (1.10.5), so `i=i` freezes the current value into each lambda. This is the
common idiom; its downside is that callers could pass `i` by mistake.

In [638]:
frozen_callbacks = [lambda i=i: i for i in range(3)]
print([callback() for callback in frozen_callbacks])
print(f"{frozen_callbacks[1].__defaults__=}, {frozen_callbacks[1].__closure__=}")

[0, 1, 2]
frozen_callbacks[1].__defaults__=(1,), frozen_callbacks[1].__closure__=None


**Fix 2: `functools.partial`.** `partial(function, value)` stores the value when it
is created, with no extra parameter to misuse. A small factory function (like
`make_multiplier`) works too, because every call creates a new cell.

In [639]:
def label_for(slot_number: int) -> str:
    """Return a delivery-slot label."""
    return f"slot {slot_number}"


partial_callbacks = [functools.partial(label_for, slot) for slot in range(3)]
factory_callbacks = [make_multiplier(slot) for slot in range(3)]
print([callback() for callback in partial_callbacks])
print([callback(10) for callback in factory_callbacks])

['slot 0', 'slot 1', 'slot 2']
[0, 10, 20]


**JS bridge: `var` in loops.** JavaScript had the same bug with `var` and fixed it
with `let`, which creates a **new binding per iteration**. Python has no per-iteration
binding, so it behaves like `var`:

```javascript
for (var i = 0; i < 3; i++) setTimeout(() => console.log(i));  // 3 3 3
for (let i = 0; i < 3; i++) setTimeout(() => console.log(i));  // 0 1 2
```

The same trap appears in real code whenever callbacks are registered in a loop, for
example one handler per button or per route. The next cell shows it with a plain
`for` loop, which shares the variable just like the comprehension.

In [640]:
button_handlers: dict[str, Callable[[], str]] = {}
for button_name in ["save", "cancel", "delete"]:
    button_handlers[button_name] = lambda: f"clicked {button_name}"  # noqa: B023

print(f"{button_handlers['save']()=}  (late binding)")

for button_name in ["save", "cancel", "delete"]:
    button_handlers[button_name] = lambda name=button_name: f"clicked {name}"

print(f"{button_handlers['save']()=}  (fixed with a default)")

button_handlers['save']()='clicked delete'  (late binding)
button_handlers['save']()='clicked save'  (fixed with a default)


### 1.11.8 Name lookup cost

The scope decision also picks the instruction used to read a name:

| Instruction | Scope | Cost |
| --- | --- | --- |
| `LOAD_FAST` / `LOAD_FAST_BORROW` | local | index into the frame's array |
| `LOAD_DEREF` | enclosing (cell) | array index + one pointer hop through the cell |
| `LOAD_GLOBAL` | global, then built-in | dict lookups (cached by the specializing interpreter, 3.11+) |
| `LOAD_ATTR` | `module.name` | one more lookup on top of the global |

The next cell shows all of them in one function.

In [641]:
def make_scaler(scale: float) -> Callable[[list[float]], list[float]]:
    """Return a function that reads a local, a cell, a global and a built-in."""

    def scale_values(values: list[float]) -> list[float]:
        rounded_values = [round(value * scale, 2) for value in values]
        return [math.floor(value) for value in rounded_values]

    return scale_values


loads = {
    f"{instruction.opname} {instruction.argval}"
    for instruction in dis.get_instructions(make_scaler(1.5))
    if instruction.opname.startswith(("LOAD_FAST", "LOAD_DEREF", "LOAD_GLOBAL", "LOAD_ATTR"))
}
print(sorted(loads))

['LOAD_ATTR floor', 'LOAD_DEREF scale', 'LOAD_FAST rounded_values', 'LOAD_FAST values', 'LOAD_FAST_AND_CLEAR value', 'LOAD_FAST_BORROW value', 'LOAD_GLOBAL math', 'LOAD_GLOBAL round']


**Timing a local alias.** The classic micro-optimization binds a global or a method
to a local name before a hot loop: `sqrt = math.sqrt`, `append = results.append`.
The specializing interpreter (3.11+) has narrowed the gap a lot, so the gain is
usually small. Measure before you do it, and only in genuinely hot loops. Timings vary
by machine and Python version.

In [642]:
def roots_global(count: int) -> list[float]:
    """math.sqrt and results.append looked up on every iteration."""
    results: list[float] = []
    for number in range(count):
        results.append(math.sqrt(number))
    return results


def roots_local_alias(count: int) -> list[float]:
    """Both looked up once, then read as locals."""
    results: list[float] = []
    append, sqrt = results.append, math.sqrt
    for number in range(count):
        append(sqrt(number))
    return results


global_seconds = min(timeit.repeat(lambda: roots_global(10_000), number=50, repeat=5))
alias_seconds = min(timeit.repeat(lambda: roots_local_alias(10_000), number=50, repeat=5))
print(f"global lookups: {global_seconds * 1000:.1f} ms, local alias: {alias_seconds * 1000:.1f} ms")
print(f"speed-up: {global_seconds / alias_seconds:.2f}x  (varies by machine and version)")

global lookups: 19.6 ms, local alias: 21.1 ms
speed-up: 0.93x  (varies by machine and version)


### Lab 1.11

Three exercises. For each, replace the `raise NotImplementedError` in the stub, then
run the check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — `make_rate_counter(window_seconds)`

Return **two closures that share state**, `(record_hit, hits_in_window)`:

- `record_hit(timestamp)` remembers that a hit happened at `timestamp` (seconds, a
  float; timestamps arrive in increasing order).
- `hits_in_window(now)` returns how many recorded hits satisfy
  `now - window_seconds < timestamp <= now`.
- Raise `ValueError` if `window_seconds <= 0`.
- Two calls to `make_rate_counter` give independent counters. Use a closure, not a
  global or a class.

In [643]:
def make_rate_counter(
    window_seconds: float,
) -> tuple[Callable[[float], None], Callable[[float], int]]:
    """Return (record_hit, hits_in_window) closures sharing one list of timestamps."""
    raise NotImplementedError("Lab 1.11: implement make_rate_counter()")

Run the checks. `rate_counter_trace` records hits at 0, 10, 20, 30 and 65 seconds in a
60-second window and asks the count at several moments.

In [644]:
def rate_counter_trace() -> list[int]:
    """Counts at t=30, 59.9, 60, 65 and 85 after hits at 0, 10, 20, 30 and 65."""
    record_hit, hits_in_window = make_rate_counter(60)
    for timestamp in (0, 10, 20, 30):
        record_hit(timestamp)
    counts = [hits_in_window(30), hits_in_window(59.9), hits_in_window(60)]
    record_hit(65)
    counts += [hits_in_window(65), hits_in_window(85)]
    return counts


def rate_counters_independent() -> tuple[int, int]:
    """Hits recorded on one counter are not seen by another."""
    record_first, count_first = make_rate_counter(10)
    _, count_second = make_rate_counter(10)
    record_first(1.0)
    return count_first(1.0), count_second(1.0)


run_checks(
    [
        ("window counts", rate_counter_trace, [4, 4, 3, 4, 2]),
        ("independent counters", rate_counters_independent, (1, 0)),
        ("both are closures", lambda: all(f.__closure__ for f in make_rate_counter(5)), True),
        ("window 0", lambda: raised_error_name(lambda: make_rate_counter(0)), "ValueError"),
    ]
)

FAIL  window counts: raised NotImplementedError: Lab 1.11: implement make_rate_counter()
FAIL  independent counters: raised NotImplementedError: Lab 1.11: implement make_rate_counter()
FAIL  both are closures: raised NotImplementedError: Lab 1.11: implement make_rate_counter()
FAIL  window 0: expected 'ValueError', got 'NotImplementedError'
0/4 checks passed


#### Exercise 2 — fix three buggy functions

The next cell defines three buggy functions and shows each bug. Write fixed versions
**with the same names minus `_buggy`**, keeping the signatures:

1. `add_tag(tag, tags=None)`: return a list of the given tags plus `tag`. The buggy
   version shares one default list between calls (mutable default).
2. `make_accumulator()`: return a function `add(amount)` returning the running total.
   The buggy version raises `UnboundLocalError`.
3. `make_price_formatters(currency_codes)`: return one formatter per currency code;
   each takes an amount and returns e.g. `"INR 250.00"`. The buggy version formats
   every amount with the last currency (late binding).

`add_tag` must not change a list passed in by the caller.

In [645]:
def add_tag_buggy(tag: str, tags: list[str] = []) -> list[str]:  # noqa: B006 (the bug)
    tags.append(tag)
    return tags


def make_accumulator_buggy() -> Callable[[float], float]:
    total = 0.0

    def add(amount: float) -> float:
        total += amount  # noqa: F823 (the bug)
        return total

    return add


def make_price_formatters_buggy(currency_codes: list[str]) -> list[Callable[[float], str]]:
    return [lambda amount: f"{code} {amount:.2f}" for code in currency_codes]  # noqa: B023


print(f"{add_tag_buggy('new')=}, {add_tag_buggy('sale')=}")
print(f"accumulator: {raised_error_name(lambda: make_accumulator_buggy()(5))}")
print([formatter(250) for formatter in make_price_formatters_buggy(["INR", "USD"])])

add_tag_buggy('new')=['new'], add_tag_buggy('sale')=['new', 'sale']
accumulator: UnboundLocalError
['USD 250.00', 'USD 250.00']


Write the fixed versions here.

In [646]:
def add_tag(tag: str, tags: list[str] | None = None) -> list[str]:
    """Return a new list holding the given tags followed by tag."""
    raise NotImplementedError("Lab 1.11: fix add_tag()")


def make_accumulator() -> Callable[[float], float]:
    """Return add(amount), which returns the running total."""
    raise NotImplementedError("Lab 1.11: fix make_accumulator()")


def make_price_formatters(currency_codes: list[str]) -> list[Callable[[float], str]]:
    """Return one formatter per currency code, e.g. 'INR 250.00'."""
    raise NotImplementedError("Lab 1.11: fix make_price_formatters()")

Run the checks.

In [647]:
def accumulator_trace() -> list[float]:
    """Running totals from one accumulator."""
    add = make_accumulator()
    return [add(100), add(49.5), add(-20)]


def caller_list_untouched() -> tuple[list[str], list[str]]:
    """The caller's list is not modified."""
    existing_tags = ["new"]
    return add_tag("sale", existing_tags), existing_tags


run_checks(
    [
        ("add_tag fresh default", lambda: (add_tag("new"), add_tag("sale")), (["new"], ["sale"])),
        ("add_tag keeps caller list", caller_list_untouched, (["new", "sale"], ["new"])),
        ("accumulator", accumulator_trace, [100, 149.5, 129.5]),
        (
            "accumulators independent",
            lambda: (make_accumulator()(1), make_accumulator()(2)),
            (1, 2),
        ),
        (
            "formatters",
            lambda: [f(250) for f in make_price_formatters(["INR", "USD", "EUR"])],
            ["INR 250.00", "USD 250.00", "EUR 250.00"],
        ),
        (
            "formatter per currency",
            lambda: [f(9.5) for f in make_price_formatters(["INR", "USD"])][0],
            "INR 9.50",
        ),
    ]
)

FAIL  add_tag fresh default: raised NotImplementedError: Lab 1.11: fix add_tag()
FAIL  add_tag keeps caller list: raised NotImplementedError: Lab 1.11: fix add_tag()
FAIL  accumulator: raised NotImplementedError: Lab 1.11: fix make_accumulator()
FAIL  accumulators independent: raised NotImplementedError: Lab 1.11: fix make_accumulator()
FAIL  formatters: raised NotImplementedError: Lab 1.11: fix make_price_formatters()
FAIL  formatter per currency: raised NotImplementedError: Lab 1.11: fix make_price_formatters()
0/6 checks passed


#### Exercise 3 — closure-based field validation

Write three **validator factories**. Each returns a validator: a function that takes a
string and returns an error message, or `None` if the value is valid.

| Factory | Error message (exactly) |
| --- | --- |
| `min_length(length)` | `"must be at least {length} characters"` |
| `matches_pattern(pattern, description)` | `"must be {description}"` (use `re.fullmatch`) |
| `one_of(allowed_values)` | `"must be one of: a, b, c"` (allowed values sorted, joined with `", "`) |

Then `validate_record(record, rules)`: `rules` maps a field name to a list of
validators. Return a dict mapping each field to its list of error messages, in rule
order, leaving out fields with no errors. A field missing from `record` is
validated as `""`.

In [648]:
type Validator = Callable[[str], str | None]


def min_length(length: int) -> Validator:
    """Return a validator that rejects values shorter than length."""
    raise NotImplementedError("Lab 1.11: implement min_length()")


def matches_pattern(pattern: str, description: str) -> Validator:
    """Return a validator that rejects values that do not fully match pattern."""
    raise NotImplementedError("Lab 1.11: implement matches_pattern()")


def one_of(allowed_values: set[str]) -> Validator:
    """Return a validator that rejects values outside allowed_values."""
    raise NotImplementedError("Lab 1.11: implement one_of()")


def validate_record(
    record: dict[str, str], rules: dict[str, list[Validator]]
) -> dict[str, list[str]]:
    """Return {field: [error messages]} for every field that has errors."""
    raise NotImplementedError("Lab 1.11: implement validate_record()")

Run the checks. `signup_rules()` builds a realistic rule set for a sign-up form.

In [649]:
def signup_rules() -> dict[str, list[Validator]]:
    """Rules for a sign-up form, built from the three factories."""
    return {
        "password": [min_length(8), matches_pattern(r".*\d.*", "a value containing a digit")],
        "username": [min_length(3), matches_pattern(r"[a-z_]+", "lowercase letters or _")],
        "plan": [one_of({"free", "pro", "team"})],
    }


run_checks(
    [
        ("min_length ok", lambda: min_length(3)("abc"), None),
        ("min_length short", lambda: min_length(8)("abc"), "must be at least 8 characters"),
        ("pattern ok", lambda: matches_pattern(r"\d{6}", "a 6-digit PIN code")("560001"), None),
        (
            "pattern is a full match",
            lambda: matches_pattern(r"\d{6}", "a 6-digit PIN code")("5600012"),
            "must be a 6-digit PIN code",
        ),
        ("one_of ok", lambda: one_of({"free", "pro"})("pro"), None),
        ("one_of message", lambda: one_of({"pro", "free"})("gold"), "must be one of: free, pro"),
        (
            "valid record",
            lambda: validate_record(
                {"password": "chai2024!", "username": "asha", "plan": "pro"}, signup_rules()
            ),
            {},
        ),
        (
            "invalid record",
            lambda: validate_record({"password": "chai", "username": "Asha K"}, signup_rules()),
            {
                "password": ["must be at least 8 characters", "must be a value containing a digit"],
                "username": ["must be lowercase letters or _"],
                "plan": ["must be one of: free, pro, team"],
            },
        ),
        ("validators are closures", lambda: min_length(8).__closure__ is not None, True),
    ]
)

FAIL  min_length ok: raised NotImplementedError: Lab 1.11: implement min_length()
FAIL  min_length short: raised NotImplementedError: Lab 1.11: implement min_length()
FAIL  pattern ok: raised NotImplementedError: Lab 1.11: implement matches_pattern()
FAIL  pattern is a full match: raised NotImplementedError: Lab 1.11: implement matches_pattern()
FAIL  one_of ok: raised NotImplementedError: Lab 1.11: implement one_of()
FAIL  one_of message: raised NotImplementedError: Lab 1.11: implement one_of()
FAIL  valid record: raised NotImplementedError: Lab 1.11: implement min_length()
FAIL  invalid record: raised NotImplementedError: Lab 1.11: implement min_length()
FAIL  validators are closures: raised NotImplementedError: Lab 1.11: implement min_length()
0/9 checks passed


### Recap — Unit 1.11

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Namespaces | Built-in, module (`globals()`), class (`__dict__`), function (a new one per call) | global object, module scope, function scope |
| `locals()` | A dict snapshot of a function's fast locals; writes are ignored (3.13+) | — |
| LEGB | Local → Enclosing → Global → Built-in → `NameError`; class bodies are skipped | scope chain |
| Shadowing | `list = [...]` hides the built-in for the whole module; `del list` restores it | shadowing `Array`, `name` |
| No block scope | `if` / `for` / `with` / `try` names survive; `except … as e` is deleted | `var` (`let` is block-scoped) |
| Comprehension scope | Loop variables stay inside (inlined since 3.12); `:=` leaks on purpose | — |
| Compile-time scope | Any assignment makes a name local for the whole function → `UnboundLocalError` | `let` TDZ (`ReferenceError`) |
| `co_varnames` / `co_names` | Local slots / global and attribute names; `LOAD_FAST_CHECK` guards maybe-unbound locals | — |
| `global` / `nonlocal` | Needed to rebind, not to mutate; prefer passing state explicitly | assignment just works |
| Closures | Inner functions keep enclosing variables alive in cells (`__closure__`, `cell_contents`) | closures |
| Late binding | Closures read variables at call time; fix with `x=x` defaults or `functools.partial` | `var` in loops; fixed by `let` |
| Lookup cost | `LOAD_FAST` < `LOAD_DEREF` < `LOAD_GLOBAL`; alias only in measured hot loops | — |

**Further reading**

- [Tutorial: Python scopes and namespaces](https://docs.python.org/3/tutorial/classes.html#python-scopes-and-namespaces)
- [Execution model: naming and binding](https://docs.python.org/3/reference/executionmodel.html#naming-and-binding)
- [The `global` statement](https://docs.python.org/3/reference/simple_stmts.html#the-global-statement),
  [the `nonlocal` statement](https://docs.python.org/3/reference/simple_stmts.html#the-nonlocal-statement)
- [FAQ: why am I getting an UnboundLocalError?](https://docs.python.org/3/faq/programming.html#why-am-i-getting-an-unboundlocalerror-when-the-variable-has-a-value),
  [FAQ: lambdas defined in a loop](https://docs.python.org/3/faq/programming.html#why-do-lambdas-defined-in-a-loop-with-different-values-all-return-the-same-result)
- [`builtins`](https://docs.python.org/3/library/builtins.html),
  [`inspect.getclosurevars`](https://docs.python.org/3/library/inspect.html#inspect.getclosurevars),
  [`dis`](https://docs.python.org/3/library/dis.html)
- [PEP 227 — Statically nested scopes](https://peps.python.org/pep-0227/),
  [PEP 3104 — Access to names in outer scopes (`nonlocal`)](https://peps.python.org/pep-3104/),
  [PEP 572 — Assignment expressions](https://peps.python.org/pep-0572/)
- [PEP 709 — Inlined comprehensions](https://peps.python.org/pep-0709/),
  [PEP 667 — Consistent views of namespaces](https://peps.python.org/pep-0667/)

## Unit 1.12 — Decorators, functools and the Call Stack

**Objectives**

- Explain that `@decorator` is `f = decorator(f)` run at definition time, and
  predict the order of stacked decorators.
- Write decorators that keep the wrapped function's metadata (`functools.wraps`),
  take arguments (factories), keep state, and work on methods.
- Read and reuse a gallery of real-world decorators (`timer`, `retry`, `log_calls`,
  `memoize`, registration), and see how `@app.get("/users")` works.
- Use the key `functools` and `operator` tools.
- Walk the call stack through frame objects, and rewrite deep recursion with an
  explicit stack.
- Introspect callables with `inspect.signature` and build a tiny dependency injector.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| Higher-order function `withLogging(fn)` | decorator `@log_calls` | `@` syntax applies it once, at definition time |
| TC39 / TypeScript decorators | Python decorators | Python's work on any function or class, with no build step |
| Lost `fn.name` after wrapping | `functools.wraps` | Without it, `__name__`, `__doc__` and the signature are lost |
| `fn.bind(null, x)` | `functools.partial(fn, x)` | `partial` can also fix keyword arguments |
| Memoize helpers (lodash) | `functools.cache`, `lru_cache` | Built in, with stats; arguments must be hashable |
| `obj => obj.price` | `operator.attrgetter("price")` | Faster, picklable, supports dotted paths |
| `new Error().stack` | `traceback.extract_stack()`, `inspect.stack()` | Real frame objects with locals |
| Express `app.get("/users", handler)` | `@app.get("/users")` | The decorator registers the function and returns it |

The next cell is this unit's setup.

In [650]:
import contextlib
import datetime as dt
import functools
import gc
import inspect
import io
import operator
import re
import sys
import time
import timeit
import traceback
import types
import warnings
import weakref
from collections.abc import Callable
from decimal import Decimal
from typing import Any, NamedTuple


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.12.1 Decorator mechanics

A **decorator** is any callable that takes a function and returns something to bind
in its place (usually a new function). The `@` line is pure syntax sugar:

```python
@announce                     def greet(): ...
def greet(): ...         ≡    greet = announce(greet)
```

The decorator runs **once, when the `def` statement runs**, not when the function is
called. The next cell writes `announce`, which prints when it is applied and returns
the function unchanged, then shows the manual form does exactly the same.

In [651]:
def announce(func: Callable[..., Any]) -> Callable[..., Any]:
    """Print when applied; return the function unchanged."""
    print(f"  announce applied to {func.__name__}")
    return func


print("before the def")


@announce
def greet(name: str) -> str:
    """Return a greeting."""
    return f"namaste, {name}"


print("after the def, before any call")
print(f"{greet('asha')=}")


def greet_manually(name: str) -> str:
    """Same function, decorated by hand."""
    return f"namaste, {name}"


greet_manually = announce(greet_manually)

before the def
  announce applied to greet
after the def, before any call
greet('asha')='namaste, asha'
  announce applied to greet_manually


**Import-time execution.** Because decorators run when the `def` runs, and a module's
top-level `def`s run when the module is imported, decorators execute **at import
time**. Frameworks rely on this: importing a module full of `@app.get(...)` handlers
is what registers the routes. The next cell runs a small module from a string
(`exec` stands in for `import`) and shows the registration happening before any
handler is called.

In [652]:
handler_module_source = """
registered_handlers = []

def handler(func):
    registered_handlers.append(func.__name__)
    print(f"  registering {func.__name__} at import time")
    return func

@handler
def list_orders():
    return []

@handler
def create_order():
    return {}
"""
module_namespace: dict[str, Any] = {}
exec(handler_module_source, module_namespace)
print(f"{module_namespace['registered_handlers']=}  (no handler was called)")

  registering list_orders at import time
  registering create_order at import time
module_namespace['registered_handlers']=['list_orders', 'create_order']  (no handler was called)


**Stacking order.** With several decorators, the one **closest to the `def` is applied
first** (bottom-up), so the outermost wrapper is the top one, and at call time the
wrappers **run top-down**:

```python
@bold
@italic
def headline(): ...      ≡    headline = bold(italic(headline))
```

```mermaid
sequenceDiagram
    participant Caller
    participant bold as bold wrapper
    participant italic as italic wrapper
    participant headline as headline()
    Caller->>bold: headline()
    bold->>italic: call inner
    italic->>headline: call inner
    headline-->>italic: "Sale"
    italic-->>bold: "<i>Sale</i>"
    bold-->>Caller: "<b><i>Sale</i></b>"
```

<details><summary>ASCII fallback</summary>

```text
definition time (bottom-up):  italic(headline) -> then bold(that)
call time (top-down):
  Caller -> bold wrapper -> italic wrapper -> headline()
  Caller <- "<b><i>Sale</i></b>" <- "<i>Sale</i>" <- "Sale"
```

</details>

In [653]:
def bold(func: Callable[[], str]) -> Callable[[], str]:
    """Wrap the result in <b> tags."""
    print("  applying bold")

    def bold_wrapper() -> str:
        print("  bold wrapper runs")
        return f"<b>{func()}</b>"

    return bold_wrapper


def italic(func: Callable[[], str]) -> Callable[[], str]:
    """Wrap the result in <i> tags."""
    print("  applying italic")

    def italic_wrapper() -> str:
        print("  italic wrapper runs")
        return f"<i>{func()}</i>"

    return italic_wrapper


@bold
@italic
def headline() -> str:
    return "Sale"


print(f"{headline()=}")

  applying italic
  applying bold
  bold wrapper runs
  italic wrapper runs
headline()='<b><i>Sale</i></b>'


**Predict the output.**

```python
@italic
@bold
def banner():
    return "Diwali offers"

print(banner())
```

<details><summary>Answer</summary>

First, at definition time, `applying bold` then `applying italic` (bottom-up). Then
the call prints `italic wrapper runs`, `bold wrapper runs` (top-down) and the result
`<i><b>Diwali offers</b></i>`. Swapping the decorators swaps the nesting.

</details>

Run the next cell to check your prediction.

In [654]:
@italic
@bold
def banner() -> str:
    return "Diwali offers"


print(banner())

  applying bold
  applying italic
  italic wrapper runs
  bold wrapper runs
<i><b>Diwali offers</b></i>


### 1.12.2 Writing decorators

The standard shape: an outer function receives `func`, defines a **wrapper** that
accepts any arguments (`*args, **kwargs`), does something before and/or after calling
`func(*args, **kwargs)`, and **returns the result**. It is the forwarding wrapper from
Unit 1.10.4 (`call_logged`), packaged so it applies once at definition time.

The first attempt below works but has a flaw: the decorated function now reports
the wrapper's name, docstring and signature.

In [655]:
def log_calls_naive(func: Callable[..., Any]) -> Callable[..., Any]:
    """Log every call (without functools.wraps)."""

    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"  calling {func.__name__}{args}")
        return func(*args, **kwargs)

    return wrapper


@log_calls_naive
def add_gst(amount: float, rate_percent: float = 18) -> float:
    """Return amount with GST added."""
    return round(amount * (100 + rate_percent) / 100, 2)


print(f"{add_gst(500)=}")
print(f"{add_gst.__name__=}, {add_gst.__doc__=}")
print(f"{inspect.signature(add_gst)=}")

  calling add_gst(500,)
add_gst(500)=590.0
add_gst.__name__='wrapper', add_gst.__doc__=None
inspect.signature(add_gst)=<Signature (*args: Any, **kwargs: Any) -> Any>


**`functools.wraps`** fixes that. It is itself a decorator, applied to the wrapper,
that copies the attributes listed in `functools.WRAPPER_ASSIGNMENTS` (on 3.14+:
`__module__`, `__name__`, `__qualname__`, `__doc__`, `__annotate__`,
`__type_params__`), merges `__dict__`, and sets `wrapper.__wrapped__ = func`.
`inspect.signature` follows `__wrapped__`, so tools (and FastAPI) see the real
signature. Always use it.

In [656]:
def log_calls_wrapped(func: Callable[..., Any]) -> Callable[..., Any]:
    """Log every call, keeping the wrapped function's metadata."""

    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"  calling {func.__name__}{args}")
        return func(*args, **kwargs)

    return wrapper


@log_calls_wrapped
def add_gst(amount: float, rate_percent: float = 18) -> float:
    """Return amount with GST added."""
    return round(amount * (100 + rate_percent) / 100, 2)


print(f"{add_gst.__name__=}, {add_gst.__doc__=}")
print(f"{inspect.signature(add_gst)=}")
print(f"{functools.WRAPPER_ASSIGNMENTS=}")

add_gst.__name__='add_gst', add_gst.__doc__='Return amount with GST added.'
inspect.signature(add_gst)=<Signature (amount: float, rate_percent: float = 18) -> float>
functools.WRAPPER_ASSIGNMENTS=('__module__', '__name__', '__qualname__', '__doc__', '__annotate__', '__type_params__')


**`__wrapped__`** gives access to the original, undecorated function, for example to
test it without the logging. `inspect.unwrap` follows a whole chain of
`__wrapped__` links.

In [657]:
original_add_gst = add_gst.__wrapped__
print(f"{original_add_gst(1000, 5)=}  (no log line: the wrapper is bypassed)")
print(f"{inspect.unwrap(add_gst) is original_add_gst=}")

original_add_gst(1000, 5)=1050.0  (no log line: the wrapper is bypassed)
inspect.unwrap(add_gst) is original_add_gst=True


**Returning results.** The most common decorator bug is forgetting `return` in the
wrapper: every decorated function then returns `None`. Python raises no error.

In [658]:
def log_calls_broken(func: Callable[..., Any]) -> Callable[..., Any]:
    """Bug: the wrapper calls func but does not return its result."""

    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        func(*args, **kwargs)

    return wrapper


@log_calls_broken
def order_total(prices: list[int]) -> int:
    """Return the sum of prices."""
    return sum(prices)


print(f"{order_total([249, 640])=}  (the result is lost)")

order_total([249, 640])=None  (the result is lost)


**Typed decorators** (*New in 3.12* syntax, PEP 695 and PEP 612). A type checker can
keep the wrapped function's exact signature if the decorator is generic over a
`ParamSpec` (`**P`) and a return type `R`. Type checking is covered in Phase 4; the
runtime behaviour is identical.

In [659]:
def traced[**P, R](func: Callable[P, R]) -> Callable[P, R]:
    """Log calls; type checkers keep func's parameters and return type."""

    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        result = func(*args, **kwargs)
        print(f"  {func.__name__} -> {result!r}")
        return result

    return wrapper


@traced
def delivery_days(pincode: str, express: bool = False) -> int:
    """Return the delivery estimate in days."""
    return 1 if express else 4


delivery_days("560001", express=True)
print(f"{traced.__type_params__=}")

  delivery_days -> 1
traced.__type_params__=(P, R)


### 1.12.3 Decorator factories

To configure a decorator (`@retry(times=3)`), add one more level: a **factory**
that takes the options and returns the decorator.

| Level | Called with | Returns |
| --- | --- | --- |
| `retry(times=3)` | the options | `decorator` |
| `decorator(func)` | the function (at definition time) | `wrapper` |
| `wrapper(*args, **kwargs)` | the call arguments (every call) | the result |

`@retry(times=3)` therefore means `fetch = retry(times=3)(fetch)`. This first version
retries on any exception and re-raises the last one when the attempts run out. The
gallery in 1.12.6 adds backoff and an exception filter.

In [660]:
def retry_simple(times: int = 3) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Return a decorator that calls func up to times times."""

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as error:
                    print(f"  attempt {attempt} failed: {error}")
                    if attempt == times:
                        raise

        return wrapper

    return decorator

The next cell decorates a flaky function that fails twice before succeeding, then
one that always fails.

In [661]:
flaky_call_count = 0


@retry_simple(times=3)
def fetch_exchange_rate() -> float:
    """Fail on the first two calls, then succeed."""
    global flaky_call_count
    flaky_call_count += 1
    if flaky_call_count < 3:
        raise ConnectionError(f"timeout #{flaky_call_count}")
    return 83.25


@retry_simple(times=2)
def fetch_from_dead_server() -> float:
    """Always fails."""
    raise ConnectionError("connection refused")


print(f"{fetch_exchange_rate()=}")
print(f"dead server: {raised_error_name(fetch_from_dead_server)}")

  attempt 1 failed: timeout #1
  attempt 2 failed: timeout #2
fetch_exchange_rate()=83.25
  attempt 1 failed: connection refused
  attempt 2 failed: connection refused
dead server: ConnectionError


**Optional-argument decorators.** Sometimes both `@timer` and `@timer(unit="s")`
should work. Make the function positional-only and optional, the options
keyword-only:

- `@timer`: Python calls `timer(func)`, so `func` is given → decorate it directly.
- `@timer(unit="s")`: the call has no function, so `func is None` → return the
  decorator, which Python then applies to the function.

`timer` below is a **shared helper**: the gallery (1.12.6) and the Phase 1 project
reuse it. `report` lets the caller send the message somewhere other than `print`.
Timings vary by machine.

In [662]:
TIME_UNITS = {"s": 1.0, "ms": 1_000.0, "us": 1_000_000.0}


def timer(
    func: Callable[..., Any] | None = None,
    /,
    *,
    unit: str = "ms",
    report: Callable[[str], object] = print,
) -> Callable[..., Any]:
    """Report how long each call takes. Use as @timer or @timer(unit="s")."""
    if unit not in TIME_UNITS:
        raise ValueError(f"unit must be one of {sorted(TIME_UNITS)}, got {unit!r}")

    def decorator(target: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(target)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            started = time.perf_counter()
            try:
                return target(*args, **kwargs)
            finally:
                elapsed = (time.perf_counter() - started) * TIME_UNITS[unit]
                report(f"{target.__qualname__} took {elapsed:.3f} {unit}")

        return wrapper

    return decorator if func is None else decorator(func)

Both forms in action. The `try` / `finally` in the wrapper means the time is reported
even when the function raises.

In [663]:
@timer
def build_report(row_count: int) -> int:
    """Sum some squares."""
    return sum(row * row for row in range(row_count))


@timer(unit="us")
def lookup_price(sku: str) -> int:
    """A fast dictionary lookup."""
    return {"TEA-500G": 249}.get(sku, 0)


print(f"{build_report(200_000)=}")
print(f"{lookup_price('TEA-500G')=}")
print(f"{build_report.__name__=}, unknown unit: {raised_error_name(lambda: timer(unit='h'))}")

build_report took 5.701 ms
build_report(200_000)=2666646666700000
lookup_price took 1.045 us
lookup_price('TEA-500G')=249
build_report.__name__='build_report', unknown unit: ValueError


### 1.12.4 Stateful and class-based decorators

A decorator can remember things between calls. Three places to keep the state:

| Where | How | Good for |
| --- | --- | --- |
| Function attribute | `wrapper.last_result = …` | state the caller should be able to read |
| Closure variable | `nonlocal` in the wrapper | private state |
| Instance attribute | a class with `__call__` | richer state and helper methods |

**Function attribute.** The wrapper is a function object, so it can carry attributes
(Unit 1.10.1). Callers read them directly.

In [664]:
def remember_last_result(func: Callable[..., Any]) -> Callable[..., Any]:
    """Store the most recent result on the wrapper as last_result."""

    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        wrapper.last_result = func(*args, **kwargs)
        return wrapper.last_result

    wrapper.last_result = None
    return wrapper


@remember_last_result
def shipping_fee(weight_kg: float) -> int:
    """Return the fee in rupees."""
    return 49 if weight_kg <= 1 else 99


shipping_fee(0.5)
shipping_fee(3.0)
print(f"{shipping_fee.last_result=}")

shipping_fee.last_result=99


**Closure variable.** `run_once` keeps a private flag and cached result with
`nonlocal`: the first call runs the function, later calls return the same result.
Useful for expensive one-time setup.

In [665]:
def run_once(func: Callable[..., Any]) -> Callable[..., Any]:
    """Call func the first time only; afterwards return the first result."""
    has_run = False
    first_result: Any = None

    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        nonlocal has_run, first_result
        if not has_run:
            first_result = func(*args, **kwargs)
            has_run = True
        return first_result

    return wrapper


@run_once
def load_settings() -> dict[str, str]:
    """Pretend to read a settings file."""
    print("  reading settings (only once)")
    return {"currency": "INR"}


print(load_settings(), load_settings())
print(f"{load_settings.__code__.co_freevars=}")

  reading settings (only once)
{'currency': 'INR'} {'currency': 'INR'}
load_settings.__code__.co_freevars=('first_result', 'func', 'has_run')


**Class-based decorator.** A class whose instances are callable (`__call__`) can be a
decorator: `CallHistory(func)` builds an instance that replaces the function.
`functools.update_wrapper(self, func)` copies the metadata, as `wraps` does for
functions.

In [666]:
class CallHistory:
    """Decorator that records the arguments of every call."""

    def __init__(self, func: Callable[..., Any]) -> None:
        functools.update_wrapper(self, func)
        self.func = func
        self.history: list[tuple[tuple[Any, ...], dict[str, Any]]] = []

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        self.history.append((args, kwargs))
        return self.func(*args, **kwargs)


@CallHistory
def apply_coupon(total: int, code: str) -> int:
    """Take 10% off with any coupon."""
    return total * 9 // 10


apply_coupon(1000, "DIWALI")
apply_coupon(500, code="NEWUSER")
print(f"{apply_coupon.history=}")
print(f"{type(apply_coupon).__name__=}, {apply_coupon.__name__=}")

apply_coupon.history=[((1000, 'DIWALI'), {}), ((500,), {'code': 'NEWUSER'})]
type(apply_coupon).__name__='CallHistory', apply_coupon.__name__='apply_coupon'


**The method trap.** Put the same class-based decorator on a **method** and calls
break. Functions become bound methods because they have a `__get__` method (they are
*descriptors*, Phase 2): `cart.add` calls `add.__get__(cart, Cart)`, which fixes
`self`. A `CallHistory` instance has no `__get__`, so `cart.add` returns the plain
instance and `self` is never passed.

In [667]:
class Cart:
    """A cart whose add method uses the class-based decorator."""

    def __init__(self) -> None:
        self.items: list[str] = []

    @CallHistory
    def add(self, sku: str) -> int:
        self.items.append(sku)
        return len(self.items)


shopping_cart = Cart()
try:
    shopping_cart.add("TEA-500G")
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{hasattr(Cart.__dict__['add'], '__get__')=}")

TypeError: Cart.add() missing 1 required positional argument: 'sku'
hasattr(Cart.__dict__['add'], '__get__')=False


**The `__get__` fix.** Give the decorator class a `__get__` that returns a bound
method (`types.MethodType(self, instance)`) when accessed through an instance. Plain
function decorators never have this problem: their wrapper is a function, which
already has `__get__`.

In [668]:
class MethodSafeCallHistory(CallHistory):
    """CallHistory that also works on methods."""

    def __get__(self, instance: object, owner: type | None = None) -> Any:
        if instance is None:  # accessed on the class: return the decorator itself
            return self
        return types.MethodType(self, instance)


class SafeCart:
    """Same cart, method-safe decorator."""

    def __init__(self) -> None:
        self.items: list[str] = []

    @MethodSafeCallHistory
    def add(self, sku: str) -> int:
        self.items.append(sku)
        return len(self.items)


safe_cart = SafeCart()
print(f"{safe_cart.add('TEA-500G')=}, {safe_cart.add('RICE-5KG')=}")
print(f"{len(SafeCart.add.history)=}")

safe_cart.add('TEA-500G')=1, safe_cart.add('RICE-5KG')=2
len(SafeCart.add.history)=2


### 1.12.5 Decorating methods and classes

**Methods.** A function decorator works on methods unchanged: the wrapper is a plain
function, so it becomes a bound method, and `self` simply arrives as `args[0]`.

In [669]:
def audit(func: Callable[..., Any]) -> Callable[..., Any]:
    """Log the class name and method name of each call."""

    @functools.wraps(func)
    def wrapper(self: Any, *args: Any, **kwargs: Any) -> Any:
        print(f"  audit: {type(self).__name__}.{func.__name__}{args}")
        return func(self, *args, **kwargs)

    return wrapper


class Inventory:
    """Stock levels keyed by SKU."""

    def __init__(self) -> None:
        self.stock: dict[str, int] = {}

    @audit
    def receive(self, sku: str, quantity: int) -> int:
        self.stock[sku] = self.stock.get(sku, 0) + quantity
        return self.stock[sku]


warehouse = Inventory()
print(f"{warehouse.receive('TEA-500G', 40)=}")

  audit: Inventory.receive('TEA-500G', 40)
warehouse.receive('TEA-500G', 40)=40


**Class decorators.** `@decorator` above a `class` statement means
`Cls = decorator(Cls)`: the decorator receives the finished class. The most common
use is **registration**: collect classes in a registry so they can be looked up by
name later (serializers, plugins, ORM models, command handlers). The factory form
takes the registry key.

In [670]:
PAYMENT_PROVIDERS: dict[str, type] = {}


def payment_provider(name: str) -> Callable[[type], type]:
    """Class decorator factory: register the class under name."""

    def decorator(cls: type) -> type:
        PAYMENT_PROVIDERS[name] = cls
        cls.provider_name = name
        return cls  # return the class itself, unchanged apart from the attribute

    return decorator


@payment_provider("upi")
class UpiPayment:
    def charge(self, amount: int) -> str:
        return f"UPI collect request for ₹{amount}"


@payment_provider("card")
class CardPayment:
    def charge(self, amount: int) -> str:
        return f"card charged ₹{amount}"


print(f"{sorted(PAYMENT_PROVIDERS)=}")
print(PAYMENT_PROVIDERS["upi"]().charge(499), f"{UpiPayment.provider_name=}")

sorted(PAYMENT_PROVIDERS)=['card', 'upi']
UPI collect request for ₹499 UpiPayment.provider_name='upi'


A class decorator can also **add behaviour**, for example a `__repr__` built from the
instance attributes. `@dataclass` (Phase 2) is the standard library's big example of
a class decorator that generates methods.

In [671]:
def add_repr(cls: type) -> type:
    """Give cls a __repr__ listing its instance attributes."""

    def __repr__(self: Any) -> str:
        fields = ", ".join(f"{name}={value!r}" for name, value in vars(self).items())
        return f"{type(self).__name__}({fields})"

    cls.__repr__ = __repr__
    return cls


@add_repr
class Customer:
    def __init__(self, name: str, city: str) -> None:
        self.name = name
        self.city = city


print(Customer("Asha", "Pune"))

Customer(name='Asha', city='Pune')


### 1.12.6 Real-world decorator gallery

| Decorator | Kind | What it does |
| --- | --- | --- |
| `timer` | optional-argument | report the duration of each call (defined in 1.12.3) |
| `retry` | factory | retry with exponential backoff, only on listed exceptions |
| `log_calls` | plain | log arguments and the result (or the exception) |
| `require_role` | factory | refuse the call unless the user has a role |
| `validate_types` | factory | check argument types given explicitly |
| `register` | factory | add the function to a registry |
| `deprecated` | factory | warn on every call |
| `memoize` | plain | cache results by arguments |
| `App.get` | method factory | register a route handler (how FastAPI works) |

#### `timer`

Reused from 1.12.3. Here `report` collects the messages in a list instead of printing,
as a logger would. Timings vary by machine.

In [672]:
timing_log: list[str] = []


@timer(unit="ms", report=timing_log.append)
def render_invoice(line_count: int) -> str:
    """Build a long invoice string."""
    return "\n".join(f"line {number}" for number in range(line_count))


render_invoice(10_000)
render_invoice(50_000)
print(f"{len(timing_log)=}, first entry starts with: {timing_log[0].split(' took ')[0]!r}")

len(timing_log)=2, first entry starts with: 'render_invoice'


#### `retry` with backoff and an exception filter

The production version of 1.12.3's `retry_simple`, and a **shared helper** the
Phase 1 project reuses:

- only exceptions listed in `exceptions` are retried; anything else (a bug, such as
  `ValueError`) propagates at once;
- between attempts it sleeps `first_delay * backoff ** (attempt - 1)` seconds
  (exponential backoff), so a struggling service gets time to recover;
- `sleep` is a parameter so tests (and this notebook) can record the delays instead of
  waiting.

In [673]:
def retry(
    times: int = 3,
    *,
    exceptions: tuple[type[Exception], ...] = (Exception,),
    first_delay: float = 0.1,
    backoff: float = 2.0,
    sleep: Callable[[float], object] = time.sleep,
) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Retry on the listed exceptions, sleeping first_delay * backoff**n in between."""
    if times < 1:
        raise ValueError("times must be at least 1")

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
                    sleep(first_delay * backoff ** (attempt - 1))
            raise AssertionError("unreachable")

        return wrapper

    return decorator

The next cell records the delays (`sleep=recorded_delays.append`): 0.1 s, 0.2 s,
0.4 s before the fourth attempt succeeds. A `ValueError` is not in the filter, so it
is raised after a single attempt.

In [674]:
recorded_delays: list[float] = []
gateway_attempts: list[int] = []


@retry(times=5, exceptions=(ConnectionError, TimeoutError), sleep=recorded_delays.append)
def call_payment_gateway(amount: int) -> str:
    """Time out three times, then succeed."""
    gateway_attempts.append(amount)
    if len(gateway_attempts) <= 3:
        raise TimeoutError("gateway timeout")
    return f"paid ₹{amount}"


@retry(times=5, exceptions=(ConnectionError,), sleep=recorded_delays.append)
def parse_amount(text: str) -> int:
    """A bug, not a network problem: must not be retried."""
    return int(text)


print(f"{call_payment_gateway(499)=}, {len(gateway_attempts)=}, {recorded_delays=}")
print(f"parse_amount('abc'): {raised_error_name(lambda: parse_amount('abc'))}, {recorded_delays=}")

call_payment_gateway(499)='paid ₹499', len(gateway_attempts)=4, recorded_delays=[0.1, 0.2, 0.4]
parse_amount('abc'): ValueError, recorded_delays=[0.1, 0.2, 0.4]


#### `log_calls`

Logs the arguments and the result, or the exception (and re-raises it: a logging
decorator must never swallow errors).

In [675]:
def log_calls(func: Callable[..., Any]) -> Callable[..., Any]:
    """Print each call with its arguments, then its result or exception."""

    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        arguments = [repr(arg) for arg in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        call_text = f"{func.__name__}({', '.join(arguments)})"
        try:
            result = func(*args, **kwargs)
        except Exception as error:
            print(f"  {call_text} raised {type(error).__name__}: {error}")
            raise
        print(f"  {call_text} -> {result!r}")
        return result

    return wrapper


@log_calls
def divide_bill(total: int, people: int) -> float:
    return total / people


divide_bill(1200, people=4)
print(f"by zero: {raised_error_name(lambda: divide_bill(1200, 0))}")

  divide_bill(1200, people=4) -> 300.0
  divide_bill(1200, 0) raised ZeroDivisionError: division by zero
by zero: ZeroDivisionError


#### `require_role`

An authorization guard: the decorated function takes a `user` keyword argument, and
the wrapper raises `PermissionError` unless the user has the required role. Phase 6
does this with FastAPI dependencies.

In [676]:
def require_role(role: str) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Refuse the call unless kwargs['user'] has role in its roles."""

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(func)
        def wrapper(*args: Any, user: dict[str, Any], **kwargs: Any) -> Any:
            if role not in user.get("roles", ()):
                raise PermissionError(f"{user['name']} needs role {role!r}")
            return func(*args, user=user, **kwargs)

        return wrapper

    return decorator


@require_role("admin")
def refund_order(order_id: int, *, user: dict[str, Any]) -> str:
    return f"order {order_id} refunded by {user['name']}"


print(refund_order(1042, user={"name": "Meena", "roles": ["admin"]}))
try:
    refund_order(1042, user={"name": "Ravi", "roles": ["support"]})
except PermissionError as error:
    print(f"{type(error).__name__}: {error}")

order 1042 refunded by Meena
PermissionError: Ravi needs role 'admin'


#### `validate_types`

The expected types are passed to the factory **explicitly**
(`@validate_types(quantity=int)`). `inspect.signature(func).bind(...)` maps the
positional and keyword arguments to parameter names, so both call styles are checked.
(Lab 1.12 builds `validate_arguments`, which reads the types from the annotations
instead.)

In [677]:
def validate_types(**expected_types: type) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Raise TypeError when a named argument is not of its expected type."""

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        signature = inspect.signature(func)

        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            bound = signature.bind(*args, **kwargs)
            for name, expected_type in expected_types.items():
                if name in bound.arguments and not isinstance(bound.arguments[name], expected_type):
                    actual = type(bound.arguments[name]).__name__
                    raise TypeError(f"{name} must be {expected_type.__name__}, got {actual}")
            return func(*args, **kwargs)

        return wrapper

    return decorator


@validate_types(sku=str, quantity=int)
def reserve_stock(sku: str, quantity: int) -> str:
    return f"reserved {quantity} x {sku}"


print(reserve_stock("TEA-500G", quantity=3))
print(f"quantity '3': {raised_error_name(lambda: reserve_stock('TEA-500G', '3'))}")

reserved 3 x TEA-500G
quantity '3': TypeError


#### `register`

A function registry: the decorator stores the function under a key and returns it
**unchanged**, so it can still be called directly. This is the plugin pattern (here,
report exporters chosen by format name at run time).

In [678]:
EXPORTERS: dict[str, Callable[[list[dict[str, Any]]], str]] = {}


def register(format_name: str) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Store the decorated function in EXPORTERS under format_name."""

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        EXPORTERS[format_name] = func
        return func

    return decorator


@register("csv")
def export_csv(rows: list[dict[str, Any]]) -> str:
    return "\n".join(",".join(str(value) for value in row.values()) for row in rows)


@register("tsv")
def export_tsv(rows: list[dict[str, Any]]) -> str:
    return "\n".join("\t".join(str(value) for value in row.values()) for row in rows)


sales_rows = [{"sku": "TEA-500G", "units": 3}, {"sku": "RICE-5KG", "units": 1}]
print(f"{sorted(EXPORTERS)=}")
print(EXPORTERS["csv"](sales_rows))

sorted(EXPORTERS)=['csv', 'tsv']
TEA-500G,3
RICE-5KG,1


#### `deprecated`

Warn callers that a function is going away. `warnings.warn(..., DeprecationWarning,
stacklevel=2)` points the warning at the **caller's** line. The cell captures the
warnings with `warnings.catch_warnings(record=True)` so they can be printed.

In [679]:
def deprecated(reason: str) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Emit a DeprecationWarning with reason on every call."""

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            message = f"{func.__name__} is deprecated: {reason}"
            warnings.warn(message, DeprecationWarning, stacklevel=2)
            return func(*args, **kwargs)

        return wrapper

    return decorator


@deprecated("use calculate_tax() instead")
def compute_vat(amount: int) -> float:
    return amount * 0.18


with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    compute_vat(100)
print([f"{w.category.__name__}: {w.message}" for w in caught_warnings])

['DeprecationWarning: compute_vat is deprecated: use calculate_tax() instead']


The standard library has its own: `warnings.deprecated` (*New in 3.13*, PEP 702)
warns at run time **and** tells type checkers to flag every use. It also sets
`__deprecated__` on the function.

In [680]:
if sys.version_info >= (3, 13):

    @warnings.deprecated("use calculate_tax() instead")
    def compute_sales_tax(amount: int) -> float:
        return amount * 0.18

    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always")
        compute_sales_tax(100)
    print([w.category.__name__ for w in caught_warnings], f"{compute_sales_tax.__deprecated__=}")
else:
    print("warnings.deprecated needs Python 3.13; skipped")

['DeprecationWarning'] compute_sales_tax.__deprecated__='use calculate_tax() instead'


#### `memoize`

Cache results in a dict keyed by the arguments. Repeated calls with the same
arguments skip the work. This is what `functools.cache` does (1.12.7, with a C
implementation and statistics); writing it once shows there is no magic.

In [681]:
def memoize(func: Callable[..., Any]) -> Callable[..., Any]:
    """Cache results by positional arguments (which must be hashable)."""
    results_by_args: dict[tuple[Any, ...], Any] = {}

    @functools.wraps(func)
    def wrapper(*args: Any) -> Any:
        if args not in results_by_args:
            results_by_args[args] = func(*args)
        return results_by_args[args]

    wrapper.cache = results_by_args
    return wrapper


@memoize
def ways_to_pay(amount: int, coins: tuple[int, ...] = (1, 2, 5, 10)) -> int:
    """Count the ways to make amount from coins (exponential without a cache)."""
    if amount == 0:
        return 1
    if amount < 0 or not coins:
        return 0
    return ways_to_pay(amount - coins[0], coins) + ways_to_pay(amount, coins[1:])


print(f"{ways_to_pay(200)=}, {len(ways_to_pay.cache)=} cached results")

ways_to_pay(200)=15211, len(ways_to_pay.cache)=1018 cached results

#### How `@app.get("/users")` works

A web framework's route decorator is a **registration decorator factory implemented as
a method**: `app.get(path)` returns a decorator that stores `(method, path) → handler`
in the app's route table and returns the handler unchanged. Dispatching a request is
then a dictionary lookup. This version matches exact paths only; Lab 1.12 adds path
parameters like `/users/{user_id}`.

In [682]:
class App:
    """A tiny web app: a route table filled by decorators."""

    def __init__(self) -> None:
        self.routes: dict[tuple[str, str], Callable[[], Any]] = {}

    def route(self, method: str, path: str) -> Callable[[Callable[[], Any]], Callable[[], Any]]:
        def decorator(handler: Callable[[], Any]) -> Callable[[], Any]:
            self.routes[(method, path)] = handler
            return handler

        return decorator

    def get(self, path: str) -> Callable[[Callable[[], Any]], Callable[[], Any]]:
        return self.route("GET", path)

    def handle(self, method: str, path: str) -> tuple[int, Any]:
        handler = self.routes.get((method, path))
        return (404, {"detail": "Not Found"}) if handler is None else (200, handler())

Register two handlers and dispatch a few requests.

In [683]:
app = App()


@app.get("/users")
def list_users() -> list[str]:
    return ["asha", "ravi"]


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}


print(f"{list(app.routes)=}")
print(app.handle("GET", "/users"), app.handle("GET", "/health"), app.handle("GET", "/admin"))
print(f"{list_users()=}  (still callable directly)")

list(app.routes)=[('GET', '/users'), ('GET', '/health')]
(200, ['asha', 'ravi']) (200, {'status': 'ok'}) (404, {'detail': 'Not Found'})
list_users()=['asha', 'ravi']  (still callable directly)


### 1.12.7 `functools`

| Tool | What it does | JS equivalent |
| --- | --- | --- |
| `partial(func, *args, **kwargs)` | freeze some arguments | `func.bind(null, a)` |
| `partialmethod` | `partial` for methods defined in a class body | — |
| `cache` | unbounded memoization | lodash `memoize` |
| `lru_cache(maxsize=128)` | bounded memoization with stats (`cache_info`, `cache_clear`) | — |
| `cached_property` | compute an attribute once per instance | lazy getter |
| `reduce(func, iterable, initial)` | fold a sequence into one value | `array.reduce` |
| `singledispatch` | choose an implementation by the first argument's type | — |
| `total_ordering` | fill in `<=`, `>`, `>=` from `==` and `<` | — |
| `update_wrapper` / `wraps` | copy metadata onto a wrapper | — |

#### `partial`

`functools.partial(func, *args, **kwargs)` returns a callable with some arguments
fixed. It keeps `.func`, `.args` and `.keywords` for introspection.

In [684]:
def price_with_tax(amount: float, rate_percent: float, *, round_to: int = 2) -> float:
    """Return amount plus tax."""
    return round(amount * (100 + rate_percent) / 100, round_to)


with_gst = functools.partial(price_with_tax, rate_percent=18)
whole_rupees = functools.partial(price_with_tax, round_to=0)
print(f"{with_gst(499)=}, {whole_rupees(499, 5)=}")
print(f"{with_gst.func.__name__=}, {with_gst.args=}, {with_gst.keywords=}")

with_gst(499)=588.82, whole_rupees(499, 5)=524.0


with_gst.func.__name__='price_with_tax', with_gst.args=(), with_gst.keywords={'rate_percent': 18}


`functools.Placeholder` (*New in 3.14*) reserves a positional slot, so you can fix a
**later** positional argument and leave an earlier one open.

In [685]:
if sys.version_info >= (3, 14):
    boxes_of_seven = functools.partial(divmod, functools.Placeholder, 7)
    print(f"{boxes_of_seven(50)=}  (full boxes, leftover items)")
else:
    print("functools.Placeholder needs Python 3.14; skipped")

boxes_of_seven(50)=(7, 1)  (full boxes, leftover items)


#### `partialmethod`

`partial` objects are not descriptors, so they do not receive `self` when stored in a
class body. `partialmethod` is the version that works as a method.

In [686]:
class Order:
    """An order whose status helpers are partial methods."""

    def __init__(self) -> None:
        self.status = "new"

    def set_status(self, status: str) -> str:
        self.status = status
        return status

    mark_shipped = functools.partialmethod(set_status, "shipped")
    mark_cancelled = functools.partialmethod(set_status, "cancelled")


festival_order = Order()
print(f"{festival_order.mark_shipped()=}, {festival_order.status=}")

festival_order.mark_shipped()='shipped', festival_order.status='shipped'


#### `cache`

`@functools.cache` is an unbounded memoizer (the same as `lru_cache(maxsize=None)`):
a plain dict lookup in C, the fastest option when the set of arguments is small.

In [687]:
@functools.cache
def fibonacci(position: int) -> int:
    """Return the position-th Fibonacci number (exponential without the cache)."""
    return position if position < 2 else fibonacci(position - 1) + fibonacci(position - 2)


print(f"{fibonacci(90)=}")
print(f"{fibonacci.cache_info()=}")

fibonacci(90)=2880067194370816120
fibonacci.cache_info()=CacheInfo(hits=88, misses=91, maxsize=None, currsize=91)


#### `lru_cache`

`@functools.lru_cache(maxsize=N)` keeps at most `N` results and evicts the **least
recently used** one. `cache_info()` reports hits, misses and the current size;
`cache_clear()` empties it.

In [688]:
@functools.lru_cache(maxsize=2)
def exchange_rate(currency: str) -> float:
    """Pretend to call a rate service."""
    print(f"  fetching {currency}")
    return {"USD": 83.2, "EUR": 90.1, "GBP": 105.4}[currency]


for currency in ["USD", "EUR", "USD", "GBP", "EUR"]:
    exchange_rate(currency)
print(f"{exchange_rate.cache_info()=}")
exchange_rate.cache_clear()
print(f"after cache_clear: {exchange_rate.cache_info()=}")

  fetching USD
  fetching EUR
  fetching GBP
  fetching EUR
exchange_rate.cache_info()=CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)
after cache_clear: exchange_rate.cache_info()=CacheInfo(hits=0, misses=0, maxsize=2, currsize=0)


**Hashable arguments.** The arguments become a dict key, so they must be hashable:
a list argument raises `TypeError`. Pass a tuple (or a `frozenset`) instead.

In [689]:
@functools.lru_cache(maxsize=128)
def basket_total(prices: tuple[int, ...]) -> int:
    """Sum a basket given as a tuple."""
    return sum(prices)


print(f"{basket_total((249, 640))=}")
try:
    basket_total([249, 640])
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

basket_total((249, 640))=889
TypeError: unhashable type: 'list'


**Memory risk on methods.** `lru_cache` on a method caches `self` as part of every
key. The cache belongs to the class, so it keeps **every instance alive** as long as
its entries remain, even after your code has dropped all references. The `weakref`
below shows the object surviving `del` until the cache is cleared. Prefer
`cached_property`, a cache keyed by plain values outside the class, or a per-instance
cache.

In [690]:
class QuoteService:
    """Caches quotes with lru_cache on a method (risky)."""

    @functools.lru_cache(maxsize=None)  # noqa: B019, UP033 (the leak is the point)
    def quote(self, sku: str) -> int:
        return len(sku) * 100


quote_service = QuoteService()
service_ref = weakref.ref(quote_service)
quote_service.quote("TEA-500G")
del quote_service
gc.collect()
print(f"after del: still alive = {service_ref() is not None}")
QuoteService.quote.cache_clear()
gc.collect()
print(f"after cache_clear: still alive = {service_ref() is not None}")

after del: still alive = True


after cache_clear: still alive = False


#### `cached_property`

Computes an attribute on first access and stores it in the instance `__dict__`;
later accesses are ordinary attribute reads. Delete the attribute to recompute. It
avoids the method-cache leak above because the cache lives on the instance.

In [691]:
class SalesReport:
    """A report whose total is expensive to compute."""

    def __init__(self, amounts: list[int]) -> None:
        self.amounts = amounts

    @functools.cached_property
    def total(self) -> int:
        print("  computing total")
        return sum(self.amounts)


weekly_report = SalesReport([1200, 450, 980])
print(weekly_report.total, weekly_report.total)
print(f"{'total' in vars(weekly_report)=}")
del weekly_report.total
print(weekly_report.total)

  computing total
2630 2630
'total' in vars(weekly_report)=True
  computing total
2630


#### `reduce`

`functools.reduce(func, iterable, initial)` folds a sequence into one value, like
JavaScript's `array.reduce`. Pass `initial` so empty input works (*Changed in 3.14*:
`initial` may be given by keyword). Prefer `sum`, `max`, `math.prod` or a loop when
they fit: they read better.

In [692]:
discount_steps = [0.9, 0.95, 0.98]  # three successive discounts
final_factor = functools.reduce(operator.mul, discount_steps, 1.0)
print(f"{final_factor=:.4f}, price after discounts: {round(1000 * final_factor, 2)}")
print(f"{functools.reduce(operator.add, [], 0)=}  (initial covers empty input)")
print(f"empty without initial: {raised_error_name(lambda: functools.reduce(operator.add, []))}")

final_factor=0.8379, price after discounts: 837.9
functools.reduce(operator.add, [], 0)=0  (initial covers empty input)
empty without initial: TypeError


#### `singledispatch`

Turns a function into a **generic function**: the implementation is chosen by the type
of the first argument. Register implementations with `@func.register` and a type
annotation. Subclasses use the nearest registered base class (`bool` → `int`).

In [693]:
@functools.singledispatch
def format_amount(amount: object) -> str:
    """Fallback for unsupported types."""
    raise TypeError(f"cannot format {type(amount).__name__}")


@format_amount.register
def _(amount: int) -> str:
    return f"₹{amount:,}"


@format_amount.register
def _(amount: Decimal) -> str:
    return f"₹{amount:,.2f}"


print(format_amount(125000), format_amount(Decimal("999.5")), format_amount(True))
print(f"str: {raised_error_name(lambda: format_amount('12'))}")
print(f"{sorted(cls.__name__ for cls in format_amount.registry)=}")

₹125,000 ₹999.50 ₹1
str: TypeError
sorted(cls.__name__ for cls in format_amount.registry)=['Decimal', 'int', 'object']


#### `total_ordering`

Define `__eq__` and **one** of `__lt__`, `__le__`, `__gt__`, `__ge__`; the class
decorator `total_ordering` adds the other three. (Phase 2 covers the comparison
methods; dataclasses can do this with `order=True`.)

In [694]:
@functools.total_ordering
class Version:
    """A release version compared by its numbers."""

    def __init__(self, text: str) -> None:
        self.numbers = tuple(int(part) for part in text.split("."))

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Version) and self.numbers == other.numbers

    def __lt__(self, other: Version) -> bool:
        return self.numbers < other.numbers


print(f"{Version('3.15.1') >= Version('3.9.0')=}, {Version('3.12.0') > Version('3.12.1')=}")
print(f"added: {[name for name in ('__le__', '__gt__', '__ge__') if name in vars(Version)]}")

Version('3.15.1') >= Version('3.9.0')=True, Version('3.12.0') > Version('3.12.1')=False
added: ['__le__', '__gt__', '__ge__']


#### `update_wrapper`

`functools.wraps(func)` is shorthand for `functools.update_wrapper(wrapper, func)`.
Call `update_wrapper` directly when the wrapper is not a `def` you can decorate,
for example an instance (as in `CallHistory`, 1.12.4) or a function built elsewhere.

In [695]:
def apply_festival_discount(price: int) -> int:
    """Take 15% off for the festival."""
    return price * 85 // 100


def generic_wrapper(*args: Any, **kwargs: Any) -> Any:
    return apply_festival_discount(*args, **kwargs)


print(f"before: {generic_wrapper.__name__=}")
functools.update_wrapper(generic_wrapper, apply_festival_discount)
print(f"after:  {generic_wrapper.__name__=}, {generic_wrapper.__doc__=}")
print(f"{generic_wrapper.__wrapped__ is apply_festival_discount=}, {generic_wrapper(1000)=}")

before: generic_wrapper.__name__='generic_wrapper'
after:  generic_wrapper.__name__='apply_festival_discount', generic_wrapper.__doc__='Take 15% off for the festival.'
generic_wrapper.__wrapped__ is apply_festival_discount=True, generic_wrapper(1000)=850


### 1.12.8 `operator`

The `operator` module exposes Python's operators and lookups as functions, ready to
pass as `key=` or to `map`, `reduce` and dispatch tables (Unit 1.10.6).

| Tool | Builds a function equivalent to | Typical use |
| --- | --- | --- |
| `itemgetter("price")` | `lambda row: row["price"]` | sort dicts / tuples |
| `itemgetter(0, 2)` | `lambda row: (row[0], row[2])` | pick several fields |
| `attrgetter("customer.city")` | `lambda obj: obj.customer.city` | sort objects, dotted paths |
| `methodcaller("strip", "-")` | `lambda obj: obj.strip("-")` | call a method on each item |
| `add`, `mul`, `neg`, `contains`, `truth`, … | the matching operator | `reduce`, dispatch tables |

#### `itemgetter`

In [696]:
products = [
    {"sku": "TEA-500G", "price": 249, "rating": 4.5},
    {"sku": "RICE-5KG", "price": 640, "rating": 4.1},
    {"sku": "SOAP-3PK", "price": 162, "rating": 4.5},
]
by_price = sorted(products, key=operator.itemgetter("price"))
print([product["sku"] for product in by_price])
get_sku_and_price = operator.itemgetter("sku", "price")
print([get_sku_and_price(product) for product in products])

['SOAP-3PK', 'TEA-500G', 'RICE-5KG']
[('TEA-500G', 249), ('RICE-5KG', 640), ('SOAP-3PK', 162)]


#### `attrgetter` (with dotted paths)

`attrgetter("customer.city")` follows the dots, like `obj.customer.city`. Several
names give a tuple, which makes a multi-level sort key.

In [697]:
class Buyer(NamedTuple):
    name: str
    city: str


class Purchase(NamedTuple):
    order_id: int
    customer: Buyer
    amount: int


purchases = [
    Purchase(3, Buyer("Ravi", "Pune"), 640),
    Purchase(1, Buyer("Asha", "Delhi"), 249),
    Purchase(2, Buyer("Meena", "Pune"), 162),
]
by_city_then_amount = sorted(purchases, key=operator.attrgetter("customer.city", "amount"))
print([purchase.order_id for purchase in by_city_then_amount])
print(f"{operator.attrgetter('customer.name')(purchases[0])=}")

[1, 2, 3]
operator.attrgetter('customer.name')(purchases[0])='Ravi'


#### `methodcaller`

`methodcaller(name, *args)` builds a function that calls `obj.name(*args)` on whatever
it receives.

In [698]:
raw_codes = ["--tea--", "-rice-", "soap---"]
strip_dashes = operator.methodcaller("strip", "-")
print(list(map(strip_dashes, raw_codes)))
print(list(map(operator.methodcaller("upper"), raw_codes)))

['tea', 'rice', 'soap']
['--TEA--', '-RICE-', 'SOAP---']


#### Operator functions

Every operator has a function: arithmetic (`add`, `mul`, `truediv`, `neg`),
comparison (`lt`, `eq`), membership (`contains`), truth (`truth`, `not_`), indexing
(`getitem`) and calling (`call`, *New in 3.11*). The last line pairs two lists
with `map(..., strict=True)` (*New in 3.14*, Unit 1.9.4).

In [699]:
print(f"{operator.add(249, 640)=}, {operator.mul(3, 249)=}, {operator.neg(50)=}")
print(f"{operator.lt(162, 249)=}, {operator.contains(['tea', 'rice'], 'tea')=}")
print(f"{operator.truth([])=}, {operator.getitem({'sku': 'TEA'}, 'sku')=}")
print(f"{operator.call(max, 3, 9, 4)=}")
print(f"{list(map(operator.mul, [249, 640], [2, 1], strict=True))=}  (line totals)")

operator.add(249, 640)=889, operator.mul(3, 249)=747, operator.neg(50)=-50
operator.lt(162, 249)=True, operator.contains(['tea', 'rice'], 'tea')=True
operator.truth([])=False, operator.getitem({'sku': 'TEA'}, 'sku')='TEA'
operator.call(max, 3, 9, 4)=9
list(map(operator.mul, [249, 640], [2, 1], strict=True))=[498, 640]  (line totals)


**Speed vs `lambda`.** `itemgetter` / `attrgetter` are implemented in C, so they are
usually faster than the equivalent `lambda` as a sort key, and they can be pickled
(useful with multiprocessing, Phase 5). Timings vary by machine and version.

In [700]:
many_rows = [(index % 97, index) for index in range(50_000)]
lambda_seconds = min(
    timeit.repeat(lambda: sorted(many_rows, key=lambda row: row[0]), number=10, repeat=5)
)
getter_seconds = min(
    timeit.repeat(lambda: sorted(many_rows, key=operator.itemgetter(0)), number=10, repeat=5)
)
print(f"lambda key: {lambda_seconds * 1000:.1f} ms, itemgetter: {getter_seconds * 1000:.1f} ms")
print(f"itemgetter speed-up: {lambda_seconds / getter_seconds:.2f}x  (varies by machine)")

lambda key: 22.4 ms, itemgetter: 15.8 ms
itemgetter speed-up: 1.41x  (varies by machine)


### 1.12.9 Frames and the call stack

Every function call creates a **frame object**: the running state of one call.
Frames link to their caller, forming the call stack.

| Attribute | Holds |
| --- | --- |
| `f_code` | the code object being run (`co_name`, `co_filename`, …) |
| `f_locals` | the local variables (a write-through `FrameLocalsProxy`, *New in 3.13*) |
| `f_back` | the caller's frame (`None` at the bottom) |
| `f_lineno` | the line currently executing |
| `f_globals` | the module namespace |

`sys._getframe()` returns the current frame (the underscore marks it as CPython
specific, but it is widely used); `sys._getframe(1)` the caller's.

In [701]:
def checkout(cart_total: int) -> None:
    """Inspect the current frame and its caller."""
    current_frame = sys._getframe()
    print(f"  {current_frame.f_code.co_name=}")
    print(f"  {sorted(current_frame.f_locals)=}, {current_frame.f_locals['cart_total']=}")
    print(f"  {current_frame.f_lineno=} (line inside this cell)")
    print(f"  {current_frame.f_back.f_code.co_name=}")
    print(f"  {type(current_frame.f_locals).__name__=}")


def place_order() -> None:
    """Caller of checkout."""
    checkout(cart_total=889)


place_order()

  current_frame.f_code.co_name='checkout'
  sorted(current_frame.f_locals)=['cart_total', 'current_frame'], current_frame.f_locals['cart_total']=889
  current_frame.f_lineno=6 (line inside this cell)
  current_frame.f_back.f_code.co_name='place_order'
  type(current_frame.f_locals).__name__='FrameLocalsProxy'


Since 3.13 (PEP 667), `frame.f_locals` inside a function is a **write-through proxy**:
assigning through it changes the real local variable. (Contrast `locals()`, which is
a snapshot, Unit 1.11.1.) Debuggers rely on this.

In [702]:
def adjust_through_frame() -> int:
    """Change a local through the frame's f_locals proxy."""
    discount_percent = 10
    sys._getframe().f_locals["discount_percent"] = 25
    return discount_percent


print(f"{adjust_through_frame()=}")

adjust_through_frame()=25


Following `f_back` walks up the stack. The next cell walks three levels and prints the
function names (the frames below this cell belong to Jupyter, so the walk stops after
four).

In [703]:
def level_three() -> list[str]:
    """Walk up the stack through f_back."""
    names = []
    frame = sys._getframe()
    while frame is not None and len(names) < 4:
        names.append(frame.f_code.co_name)
        frame = frame.f_back
    return names


def level_two() -> list[str]:
    return level_three()


def level_one() -> list[str]:
    return level_two()


print(level_one())

['level_three', 'level_two', 'level_one', '<module>']


**`inspect.stack()`** returns a list of `FrameInfo` records (frame, filename, line
number, function name, source line) from the current frame outward. It is slower
than `sys._getframe` because it reads source lines.

In [704]:
def who_called_me() -> list[str]:
    """Return function name and line of the innermost frames."""
    return [f"{info.function}:{info.lineno}" for info in inspect.stack()[:3]]


def handle_request() -> list[str]:
    return who_called_me()


print(handle_request())
print(f"{type(inspect.stack()[0]).__name__=}")

['who_called_me:3', 'handle_request:7', '<module>:10']
type(inspect.stack()[0]).__name__='FrameInfo'


**`traceback.extract_stack()`** returns lightweight `FrameSummary` records
(filename, line number, name, source line), **outermost first**, without keeping the
frames alive. It is what you would log; a traceback printed after an exception has
the same shape.

In [705]:
def record_stack() -> list[str]:
    """Return the innermost three stack entries, outermost first."""
    return [f"{summary.name}:{summary.line}" for summary in traceback.extract_stack()[-3:]]


def process_payment() -> list[str]:
    return record_stack()


for entry in process_payment():
    print(f"  {entry}")

  <module>:for entry in process_payment():
  process_payment:return record_stack()
  record_stack:return [f"{summary.name}:{summary.line}" for summary in traceback.extract_stack()[-3:]]


**JS bridge: execution contexts.** The ideas map one to one:

| JavaScript | Python |
| --- | --- |
| execution context | frame object |
| call stack | chain of frames via `f_back` |
| lexical environment / variable environment | `f_locals` (+ cells for closures) |
| global execution context | the module frame (`<module>`) and `f_globals` |
| `new Error().stack` (a string) | `traceback.extract_stack()` (structured records) |
| `RangeError: Maximum call stack size exceeded` | `RecursionError` (1.12.10) |

### 1.12.10 Recursion limits

Each Python-level call adds a frame. CPython limits the depth with a counter,
`sys.getrecursionlimit()` (default 1000), and raises `RecursionError` when it is
exceeded, instead of crashing the process.

In [706]:
def nesting_depth(levels: int) -> int:
    """Recurse levels times."""
    return 0 if levels == 0 else 1 + nesting_depth(levels - 1)


print(f"{sys.getrecursionlimit()=}")
print(f"{nesting_depth(900)=}")
try:
    nesting_depth(5_000)
except RecursionError as error:
    print(f"{type(error).__name__}: {error}")

sys.getrecursionlimit()=1000
nesting_depth(900)=900
RecursionError: maximum recursion depth exceeded


**No tail-call optimization.** Even when the recursive call is the very last thing a
function does (a tail call), CPython keeps every frame: tracebacks stay complete,
which Guido van Rossum considered more important. A tail-recursive loop over 5 000
items still fails.

In [707]:
def sum_tail_recursive(numbers: list[int], index: int = 0, running_total: int = 0) -> int:
    """Tail-recursive sum: the recursive call is the last operation."""
    if index == len(numbers):
        return running_total
    return sum_tail_recursive(numbers, index + 1, running_total + numbers[index])


print(f"{sum_tail_recursive(list(range(500)))=}")
print(f"5 000 items: {raised_error_name(lambda: sum_tail_recursive(list(range(5_000))))}")

sum_tail_recursive(list(range(500)))=124750
5 000 items: RecursionError


`sys.setrecursionlimit(n)` raises the limit. It is a blunt tool: each frame also uses
C stack in some paths, and a limit far above what the C stack can hold can crash the
interpreter instead of raising. Change it temporarily and restore it.

In [708]:
original_limit = sys.getrecursionlimit()
try:
    sys.setrecursionlimit(6_000)
    print(f"{nesting_depth(5_000)=} with limit {sys.getrecursionlimit()}")
finally:
    sys.setrecursionlimit(original_limit)
print(f"restored: {sys.getrecursionlimit()=}")

nesting_depth(5_000)=5000 with limit 6000
restored: sys.getrecursionlimit()=1000


**Explicit-stack rewrites.** The robust fix is to replace the call stack with a
**list used as a stack** (1.6.13): push the work still to do, pop and process until
empty. The next cell builds a category tree 10 000 levels deep (by loop, not
recursion), where the recursive sum fails and the explicit-stack version succeeds.

In [709]:
def nested_total_recursive(node: int | list[Any]) -> int:
    """Sum all ints in a nested list, recursively."""
    return node if isinstance(node, int) else sum(nested_total_recursive(child) for child in node)


def nested_total_iterative(root: int | list[Any]) -> int:
    """Same result with an explicit stack: no recursion limit."""
    total, pending = 0, [root]
    while pending:
        node = pending.pop()
        if isinstance(node, int):
            total += node
        else:
            pending.extend(node)
    return total


deep_tree: list[Any] = [1]
for _ in range(10_000):
    deep_tree = [1, deep_tree]
print(f"recursive: {raised_error_name(lambda: nested_total_recursive(deep_tree))}")
print(f"{nested_total_iterative(deep_tree)=}")

recursive: RecursionError
nested_total_iterative(deep_tree)=10001


### 1.12.11 Introspecting callables

`inspect.signature(func)` returns a `Signature`: an ordered mapping of `Parameter`
objects. Each parameter has a `name`, `kind`, `default` and `annotation`;
`Parameter.empty` marks a missing default or annotation.

In [710]:
def create_shipment(
    order_id: int, /, city: str, express: bool = False, *, insured_value: float = 0.0
) -> str:
    """Create a shipment."""
    return f"{order_id} to {city}"


shipment_signature = inspect.signature(create_shipment)
print(f"{shipment_signature=}")
for parameter in shipment_signature.parameters.values():
    default = "—" if parameter.default is inspect.Parameter.empty else repr(parameter.default)
    annotation_name = parameter.annotation.__name__
    print(f"  {parameter.name:14} {parameter.kind.name:22} default={default:6} {annotation_name=}")
print(f"{shipment_signature.return_annotation=}")

shipment_signature=<Signature (order_id: int, /, city: str, express: bool = False, *, insured_value: float = 0.0) -> str>
  order_id       POSITIONAL_ONLY        default=—      annotation_name='int'
  city           POSITIONAL_OR_KEYWORD  default=—      annotation_name='str'
  express        POSITIONAL_OR_KEYWORD  default=False  annotation_name='bool'
  insured_value  KEYWORD_ONLY           default=0.0    annotation_name='float'
shipment_signature.return_annotation=<class 'str'>


**`bind`** maps call arguments to parameter names **without calling** the function,
exactly as a real call would, and raises `TypeError` for an invalid call.
**`apply_defaults`** then fills in the parameters that were not given. Frameworks use
this to validate and inspect calls.

In [711]:
bound_call = shipment_signature.bind(1042, "Pune", insured_value=5000.0)
print(f"{bound_call.arguments=}")
bound_call.apply_defaults()
print(f"after apply_defaults: {bound_call.arguments=}")
print(f"{bound_call.args=}, {bound_call.kwargs=}")
print(f"missing city: {raised_error_name(lambda: shipment_signature.bind(1042))}")
by_name_error = raised_error_name(lambda: shipment_signature.bind(order_id=1, city="Goa"))
print(f"order_id by name: {by_name_error}  (positional-only)")

bound_call.arguments={'order_id': 1042, 'city': 'Pune', 'insured_value': 5000.0}
after apply_defaults: bound_call.arguments={'order_id': 1042, 'city': 'Pune', 'express': False, 'insured_value': 5000.0}
bound_call.args=(1042, 'Pune', False), bound_call.kwargs={'insured_value': 5000.0}
missing city: TypeError
order_id by name: TypeError  (positional-only)


**A tiny dependency injector.** FastAPI reads a handler's signature and supplies each
parameter itself. The core idea fits in one function: look at each parameter's
**annotation**, find a provider registered for that type, call it, and pass the
result in. Explicit arguments win; parameters with defaults may be left out.

In [712]:
class Settings:
    """Application settings."""

    currency = "INR"


class Database:
    """A fake database connection."""

    def __init__(self, url: str) -> None:
        self.url = url

    def count_orders(self) -> int:
        return 42


def inject(
    func: Callable[..., Any], providers: dict[type, Callable[[], Any]], **explicit: Any
) -> Any:
    """Call func, building each argument from explicit values or providers by annotation."""
    arguments: dict[str, Any] = {}
    for name, parameter in inspect.signature(func).parameters.items():
        if name in explicit:
            arguments[name] = explicit[name]
        elif parameter.annotation in providers:
            arguments[name] = providers[parameter.annotation]()
        elif parameter.default is inspect.Parameter.empty:
            raise TypeError(f"cannot provide parameter {name!r}")
    return func(**arguments)

The handler below only declares what it needs; `inject` supplies it. In a test you
would register a different `Database` provider: that is the whole appeal of
dependency injection.

In [713]:
app_providers: dict[type, Callable[[], Any]] = {
    Settings: Settings,
    Database: lambda: Database("sqlite:///shop.db"),
}


def order_dashboard(db: Database, settings: Settings, title: str = "Orders") -> str:
    """A handler that declares its dependencies as annotated parameters."""
    return f"{title}: {db.count_orders()} orders, currency {settings.currency}, db={db.url}"


print(inject(order_dashboard, app_providers))
print(inject(order_dashboard, app_providers, title="Today"))
test_providers = {**app_providers, Database: lambda: Database("sqlite:///:memory:")}
print(inject(order_dashboard, test_providers))

Orders: 42 orders, currency INR, db=sqlite:///shop.db
Today: 42 orders, currency INR, db=sqlite:///shop.db
Orders: 42 orders, currency INR, db=sqlite:///:memory:


### Lab 1.12

Seven exercises. For each, replace the `raise NotImplementedError` in the stub, then
run the check cell. Until then the checks print FAIL; that is expected.

> **Note:** exercises 1 and 2 rebuild `timer` and `retry` from 1.12.3 / 1.12.6 with
> extra hooks for testing. Write them without scrolling back up. The stubs replace
> the demo versions in this notebook's namespace; later units re-define what they
> need.

The next cell defines the check helpers used below: `fake_clock(*readings)` returns a
clock function that returns the given readings one per call.

In [714]:
def fake_clock(*readings: float) -> Callable[[], float]:
    """Return a clock that returns readings one by one."""
    reading_iterator = iter(readings)
    return lambda: next(reading_iterator)


def captured_stdout(thunk: Callable[[], object]) -> tuple[object, str]:
    """Run thunk and return (its result, what it printed)."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        result = thunk()
    return result, buffer.getvalue()

#### Exercise 1 — `@timer` with optional arguments

Write `timer(func=None, /, *, unit="ms", clock=time.perf_counter, report=print)`:

- works both as `@timer` and as `@timer(unit="s", ...)`;
- `unit` is `"s"`, `"ms"` or `"us"`; anything else raises `ValueError` **when
  `timer(...)` is called**;
- each call reads `clock()` before and after, and calls
  `report(f"{func.__name__} took {elapsed:.2f} {unit}")`, then returns the result;
- keeps the function's metadata (`__name__`, `__wrapped__`).

In [715]:
def timer(
    func: Callable[..., Any] | None = None,
    /,
    *,
    unit: str = "ms",
    clock: Callable[[], float] = time.perf_counter,
    report: Callable[[str], object] = print,
) -> Callable[..., Any]:
    """Report each call's duration. Use as @timer or @timer(unit=..., clock=..., report=...)."""
    raise NotImplementedError("Lab 1.12: implement timer()")

Run the checks.

In [716]:
def timer_with_options() -> tuple[int, list[str]]:
    """@timer(...) with a fake clock: 10.0 s → 10.25 s."""
    messages: list[str] = []

    @timer(unit="ms", clock=fake_clock(10.0, 10.25), report=messages.append)
    def brew_chai(cups: int) -> int:
        return cups * 150

    return brew_chai(2), messages


def timer_bare() -> tuple[object, bool, str]:
    """Bare @timer: real clock, printed message."""

    def pack_order() -> str:
        """Pack it."""
        return "packed"

    result, printed = captured_stdout(lambda: timer(pack_order)())
    return (
        result,
        bool(re.fullmatch(r"pack_order took \d+\.\d{2} ms\n", printed)),
        timer(pack_order).__name__,
    )


run_checks(
    [
        ("@timer(...) result and message", timer_with_options, (300, ["brew_chai took 250.00 ms"])),
        ("bare @timer", timer_bare, ("packed", True, "pack_order")),
        (
            "seconds",
            lambda: captured_stdout(lambda: timer(unit="s", clock=fake_clock(1.0, 3.5))(len)("ab"))[
                1
            ],
            "len took 2.50 s\n",
        ),
        ("bad unit", lambda: raised_error_name(lambda: timer(unit="hours")), "ValueError"),
        ("__wrapped__", lambda: timer(len).__wrapped__ is len, True),
    ]
)

FAIL  @timer(...) result and message: raised NotImplementedError: Lab 1.12: implement timer()
FAIL  bare @timer: raised NotImplementedError: Lab 1.12: implement timer()
FAIL  seconds: raised NotImplementedError: Lab 1.12: implement timer()
FAIL  bad unit: expected 'ValueError', got 'NotImplementedError'
FAIL  __wrapped__: raised NotImplementedError: Lab 1.12: implement timer()
0/5 checks passed


#### Exercise 2 — `@retry(times, exceptions, backoff_factor)`

Write `retry(times=3, exceptions=(Exception,), backoff_factor=0.0, *, sleep=time.sleep)`:

- call the function up to `times` times in total;
- when it raises one of `exceptions` and attempts remain, call
  `sleep(backoff_factor * 2 ** (attempt - 1))` (attempt counts from 1), then retry;
- after the last attempt, re-raise the last exception;
- other exceptions propagate immediately, with no retry;
- `times < 1` raises `ValueError` when `retry(...)` is called;
- use `functools.wraps`.

In [717]:
def retry(
    times: int = 3,
    exceptions: tuple[type[BaseException], ...] = (Exception,),
    backoff_factor: float = 0.0,
    *,
    sleep: Callable[[float], object] = time.sleep,
) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Retry on exceptions with exponential backoff."""
    raise NotImplementedError("Lab 1.12: implement retry()")

Run the checks. `flaky(failures, error)` builds a function that raises `error` for its
first `failures` calls and then returns `"ok"`; it records each call in `.calls`.

In [718]:
def flaky(failures: int, error: Exception) -> Callable[[], str]:
    """Return a function that fails failures times, then returns "ok"."""
    calls: list[int] = []

    def unreliable() -> str:
        calls.append(len(calls) + 1)
        if len(calls) <= failures:
            raise error
        return "ok"

    unreliable.calls = calls
    return unreliable


def retry_trace(failures: int, times: int, error: Exception) -> tuple[str, int, list[float]]:
    """Return (result or exception name, number of calls, recorded delays)."""
    delays: list[float] = []
    target = flaky(failures, error)
    decorated = retry(times, (ConnectionError,), 0.5, sleep=delays.append)(target)
    outcome = raised_error_name(decorated)
    return ("ok" if outcome == "no error" else outcome), len(target.calls), delays


run_checks(
    [
        (
            "succeeds on third attempt",
            lambda: retry_trace(2, 3, ConnectionError()),
            ("ok", 3, [0.5, 1.0]),
        ),
        (
            "gives up and re-raises",
            lambda: retry_trace(5, 3, ConnectionError()),
            ("ConnectionError", 3, [0.5, 1.0]),
        ),
        (
            "other exceptions not retried",
            lambda: retry_trace(5, 3, KeyError("x")),
            ("KeyError", 1, []),
        ),
        ("first try ok", lambda: retry_trace(0, 3, ConnectionError()), ("ok", 1, [])),
        ("times 0", lambda: raised_error_name(lambda: retry(0)), "ValueError"),
        ("wraps", lambda: retry(2)(flaky(0, ValueError())).__name__, "unreliable"),
    ]
)

FAIL  succeeds on third attempt: raised NotImplementedError: Lab 1.12: implement retry()
FAIL  gives up and re-raises: raised NotImplementedError: Lab 1.12: implement retry()
FAIL  other exceptions not retried: raised NotImplementedError: Lab 1.12: implement retry()
FAIL  first try ok: raised NotImplementedError: Lab 1.12: implement retry()
FAIL  times 0: expected 'ValueError', got 'NotImplementedError'
FAIL  wraps: raised NotImplementedError: Lab 1.12: implement retry()
0/6 checks passed


#### Exercise 3 — `@count_calls`

Write `count_calls(func)`: the wrapper has a `call_count` attribute that starts at 0
and goes up by one on **every** call, even when the function raises. Each decorated
function has its own count. Keep the metadata.

In [719]:
def count_calls(func: Callable[..., Any]) -> Callable[..., Any]:
    """Count every call in wrapper.call_count."""
    raise NotImplementedError("Lab 1.12: implement count_calls()")

Run the checks.

In [720]:
def count_calls_trace() -> tuple[int, int, int, str]:
    """Count calls on two functions, one of which raises."""

    @count_calls
    def ping() -> str:
        return "pong"

    @count_calls
    def fail() -> None:
        raise RuntimeError("boom")

    start = ping.call_count
    ping()
    ping()
    raised_error_name(fail)
    return start, ping.call_count, fail.call_count, ping.__name__


run_checks(
    [
        ("counts, raising calls included", count_calls_trace, (0, 2, 1, "ping")),
        ("returns the result", lambda: count_calls(max)(3, 9), 9),
    ]
)

FAIL  counts, raising calls included: raised NotImplementedError: Lab 1.12: implement count_calls()
FAIL  returns the result: raised NotImplementedError: Lab 1.12: implement count_calls()
0/2 checks passed


#### Exercise 4 — `@validate_arguments` using `inspect.signature`

Write `validate_arguments(func)`: on each call, bind the arguments with
`inspect.signature(func).bind(*args, **kwargs)` and apply the defaults. For every
parameter whose annotation is a **class** (`isinstance(annotation, type)`), check the
value with `isinstance`; parameters without an annotation, or with a non-class
annotation (such as `list[int]` or `int | None`), are not checked. Careful: the
"no annotation" marker `inspect.Parameter.empty` is itself a class. On a mismatch
raise `TypeError(f"{name} must be {annotation.__name__}, got {type(value).__name__}")`.
An invalid call (missing argument) raises `TypeError` from `bind`.

In [721]:
def validate_arguments(func: Callable[..., Any]) -> Callable[..., Any]:
    """Check annotated argument types on every call."""
    raise NotImplementedError("Lab 1.12: implement validate_arguments()")

Run the checks.

In [722]:
def make_add_item() -> Callable[..., Any]:
    """A decorated function with checked and unchecked parameters."""

    @validate_arguments
    def add_item(
        sku: str, quantity: int = 1, *, note: str = "", tags: list[str] | None = None, extra=None
    ) -> str:
        return f"{quantity} x {sku}"

    return add_item


def type_error_message(thunk: Callable[[], object]) -> str:
    """Return the TypeError message thunk raises, or 'no error'."""
    try:
        thunk()
    except TypeError as error:
        return str(error)
    return "no error"


run_checks(
    [
        ("valid call", lambda: make_add_item()("TEA", 2), "2 x TEA"),
        ("defaults pass", lambda: make_add_item()("TEA"), "1 x TEA"),
        (
            "bad positional",
            lambda: type_error_message(lambda: make_add_item()("TEA", "2")),
            "quantity must be int, got str",
        ),
        (
            "bad keyword-only",
            lambda: type_error_message(lambda: make_add_item()("TEA", note=5)),
            "note must be str, got int",
        ),
        (
            "unchecked annotations",
            lambda: make_add_item()("TEA", tags="x", extra=object()),
            "1 x TEA",
        ),
        ("missing argument", lambda: raised_error_name(lambda: make_add_item()()), "TypeError"),
        ("keeps name", lambda: make_add_item().__name__, "add_item"),
    ]
)

FAIL  valid call: raised NotImplementedError: Lab 1.12: implement validate_arguments()
FAIL  defaults pass: raised NotImplementedError: Lab 1.12: implement validate_arguments()
FAIL  bad positional: raised NotImplementedError: Lab 1.12: implement validate_arguments()
FAIL  bad keyword-only: raised NotImplementedError: Lab 1.12: implement validate_arguments()
FAIL  unchecked annotations: raised NotImplementedError: Lab 1.12: implement validate_arguments()
FAIL  missing argument: expected 'TypeError', got 'NotImplementedError'
FAIL  keeps name: raised NotImplementedError: Lab 1.12: implement validate_arguments()
0/7 checks passed


#### Exercise 5 — `serialize()` with `singledispatch`

Write a generic function `serialize(value)` that converts a value into something
`json.dumps` accepts:

| Input type | Output |
| --- | --- |
| `str`, `int`, `float`, `bool`, `None` | unchanged |
| `Decimal` | `str(value)` |
| `dt.date` and `dt.datetime` | `value.isoformat()` |
| `list`, `tuple` | a list, each item serialized |
| `set`, `frozenset` | a **sorted** list of the serialized items |
| `dict` | a dict with `str(key)` keys and serialized values |
| anything else | `TypeError(f"cannot serialize {type(value).__name__}")` |

Use `@functools.singledispatch` and one `register` per row (no `isinstance` chain).

In [723]:
@functools.singledispatch
def serialize(value: object) -> Any:
    """Convert value into JSON-compatible data."""
    raise NotImplementedError("Lab 1.12: implement serialize() and its registrations")

Run the checks.

In [724]:
sample_order = {
    "order_id": 1042,
    "placed_on": dt.date(2026, 11, 5),
    "total": Decimal("889.50"),
    "items": ("TEA-500G", "RICE-5KG"),
    "tags": {"gift", "express"},
    "paid": True,
    "coupon": None,
}
run_checks(
    [
        ("Decimal", lambda: serialize(Decimal("249.50")), "249.50"),
        ("datetime", lambda: serialize(dt.datetime(2026, 11, 5, 9, 30)), "2026-11-05T09:30:00"),
        ("set sorted", lambda: serialize({3, 1, 2}), [1, 2, 3]),
        ("dict keys become str", lambda: serialize({1: Decimal("1.5")}), {"1": "1.5"}),
        (
            "nested order",
            lambda: serialize(sample_order),
            {
                "order_id": 1042,
                "placed_on": "2026-11-05",
                "total": "889.50",
                "items": ["TEA-500G", "RICE-5KG"],
                "tags": ["express", "gift"],
                "paid": True,
                "coupon": None,
            },
        ),
        ("unsupported type", lambda: raised_error_name(lambda: serialize(object())), "TypeError"),
        ("uses singledispatch", lambda: Decimal in serialize.registry, True),
    ]
)

FAIL  Decimal: raised NotImplementedError: Lab 1.12: implement serialize() and its registrations
FAIL  datetime: raised NotImplementedError: Lab 1.12: implement serialize() and its registrations
FAIL  set sorted: raised NotImplementedError: Lab 1.12: implement serialize() and its registrations
FAIL  dict keys become str: raised NotImplementedError: Lab 1.12: implement serialize() and its registrations
FAIL  nested order: raised NotImplementedError: Lab 1.12: implement serialize() and its registrations
FAIL  unsupported type: expected 'TypeError', got 'NotImplementedError'
FAIL  uses singledispatch: expected True, got False
0/7 checks passed


#### Exercise 6 — `call_with_params(func, params)`

`params` is a dict of **strings**, as they arrive in a URL query string. Call `func`
with them, converted by the parameters' annotations:

- for each parameter of `func`: if its name is in `params`, convert the string with
  the annotation when it is `int`, `float` or `bool` (`"true"`, `"1"`, `"yes"` →
  `True`; `"false"`, `"0"`, `"no"` → `False`, case-insensitive; anything else
  raises `ValueError`), otherwise pass the string through;
- if the name is missing and the parameter has a default, leave it out;
- if it is missing and has no default, raise
  `TypeError(f"missing required parameter: {name}")`;
- keys in `params` that are not parameters are ignored.

In [725]:
def call_with_params(func: Callable[..., Any], params: dict[str, str]) -> Any:
    """Call func with string params converted by its annotations."""
    raise NotImplementedError("Lab 1.12: implement call_with_params()")

Run the checks.

In [726]:
def search_products(
    query: str, page: int = 1, max_price: float = 1e9, in_stock: bool = False
) -> tuple[Any, ...]:
    """A handler with typed query parameters."""
    return query, page, max_price, in_stock


def missing_message() -> str:
    """Return the TypeError message for a missing required parameter."""
    try:
        call_with_params(search_products, {"page": "2"})
    except TypeError as error:
        return str(error)
    return "no error"


run_checks(
    [
        (
            "converts",
            lambda: call_with_params(
                search_products,
                {"query": "tea", "page": "2", "max_price": "499.5", "in_stock": "Yes"},
            ),
            ("tea", 2, 499.5, True),
        ),
        (
            "defaults",
            lambda: call_with_params(search_products, {"query": "rice"}),
            ("rice", 1, 1e9, False),
        ),
        (
            "extra keys ignored",
            lambda: call_with_params(search_products, {"query": "x", "utm_source": "ad"}),
            ("x", 1, 1e9, False),
        ),
        (
            "bool false",
            lambda: call_with_params(search_products, {"query": "x", "in_stock": "0"})[3],
            False,
        ),
        ("missing required", missing_message, "missing required parameter: query"),
        (
            "bad int",
            lambda: raised_error_name(
                lambda: call_with_params(search_products, {"query": "x", "page": "two"})
            ),
            "ValueError",
        ),
        (
            "bad bool",
            lambda: raised_error_name(
                lambda: call_with_params(search_products, {"query": "x", "in_stock": "maybe"})
            ),
            "ValueError",
        ),
    ]
)

FAIL  converts: raised NotImplementedError: Lab 1.12: implement call_with_params()
FAIL  defaults: raised NotImplementedError: Lab 1.12: implement call_with_params()
FAIL  extra keys ignored: raised NotImplementedError: Lab 1.12: implement call_with_params()
FAIL  bool false: raised NotImplementedError: Lab 1.12: implement call_with_params()
FAIL  missing required: raised NotImplementedError: Lab 1.12: implement call_with_params()
FAIL  bad int: expected 'ValueError', got 'NotImplementedError'
FAIL  bad bool: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 7 — a mini router with typed path parameters

Write a `Router` class:

- `router.get(path_template)` and `router.post(path_template)` are decorator factories
  that register the handler and **return it unchanged**;
- a template is split on `/`; a segment like `{user_id}` captures that part of the
  path, other segments must match exactly;
- `router.dispatch(method, path)` finds the **first** registered route whose method
  and segments match, converts each captured value with the handler parameter's
  annotation (`int`, `float` or `str`), and returns `handler(**captured)`;
- a route whose conversion fails (`"/users/abc"` for an `int`) does not match;
- when nothing matches, raise `LookupError(f"no route for {method} {path}")`.

In [727]:
class Router:
    """Register handlers with @router.get / @router.post and dispatch paths to them."""

    def __init__(self) -> None:
        raise NotImplementedError("Lab 1.12: implement Router")

    def get(self, path_template: str) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
        raise NotImplementedError

    def post(self, path_template: str) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
        raise NotImplementedError

    def dispatch(self, method: str, path: str) -> Any:
        raise NotImplementedError

Run the checks. `make_shop_router()` registers four routes.

In [728]:
def make_shop_router() -> Router:
    """A router with literal, typed and multi-parameter routes."""
    router = Router()

    @router.get("/health")
    def health() -> dict[str, str]:
        return {"status": "ok"}

    @router.get("/users/{user_id}")
    def get_user(user_id: int) -> dict[str, Any]:
        return {"user_id": user_id, "type": type(user_id).__name__}

    @router.get("/users/{user_id}/orders/{order_code}")
    def get_order(user_id: int, order_code: str) -> tuple[int, str]:
        return user_id, order_code

    @router.post("/users/{user_id}")
    def update_user(user_id: int) -> str:
        return f"updated {user_id}"

    return router


def decorator_returns_handler() -> bool:
    """router.get(...) must return the original function."""

    def handler() -> None:
        return None

    return Router().get("/x")(handler) is handler


run_checks(
    [
        ("literal route", lambda: make_shop_router().dispatch("GET", "/health"), {"status": "ok"}),
        (
            "typed param",
            lambda: make_shop_router().dispatch("GET", "/users/42"),
            {"user_id": 42, "type": "int"},
        ),
        (
            "two params",
            lambda: make_shop_router().dispatch("GET", "/users/7/orders/A-19"),
            (7, "A-19"),
        ),
        ("method matters", lambda: make_shop_router().dispatch("POST", "/users/7"), "updated 7"),
        (
            "conversion failure is 404",
            lambda: raised_error_name(lambda: make_shop_router().dispatch("GET", "/users/abc")),
            "LookupError",
        ),
        (
            "unknown path",
            lambda: raised_error_name(lambda: make_shop_router().dispatch("GET", "/admin")),
            "LookupError",
        ),
        (
            "segment count matters",
            lambda: raised_error_name(lambda: make_shop_router().dispatch("GET", "/users/7/x")),
            "LookupError",
        ),
        ("decorator returns handler", decorator_returns_handler, True),
    ]
)

FAIL  literal route: raised NotImplementedError: Lab 1.12: implement Router
FAIL  typed param: raised NotImplementedError: Lab 1.12: implement Router
FAIL  two params: raised NotImplementedError: Lab 1.12: implement Router
FAIL  method matters: raised NotImplementedError: Lab 1.12: implement Router
FAIL  conversion failure is 404: expected 'LookupError', got 'NotImplementedError'
FAIL  unknown path: expected 'LookupError', got 'NotImplementedError'
FAIL  segment count matters: expected 'LookupError', got 'NotImplementedError'
FAIL  decorator returns handler: raised NotImplementedError: Lab 1.12: implement Router
0/8 checks passed


### Recap — Unit 1.12

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `@decorator` | `f = decorator(f)`, run once when the `def` runs (import time) | higher-order function call |
| Stacking | Applied bottom-up, wrappers run top-down | `a(b(fn))` |
| Wrapper shape | `*args, **kwargs`, call, **return the result** | rest/spread wrapper |
| `functools.wraps` | Copies name/doc/annotations, sets `__wrapped__`; signature survives | — |
| Factories | `@retry(times=3)` = three levels: options → decorator → wrapper | decorator factories |
| Optional arguments | `func=None, /, *, options`: works as `@timer` and `@timer(unit="s")` | — |
| State | Function attributes, closures (`nonlocal`) or a class with `__call__` | closure state |
| Method trap | Class-based decorators need `__get__` to bind `self` | `this` binding |
| Class decorators | Receive the class; registries, added methods (`@dataclass`) | class decorators |
| Gallery | `timer`, `retry` (backoff + filter), `log_calls`, `require_role`, `validate_types`, `register`, `deprecated`, `memoize`, `@app.get` | Express middleware/routes |
| `functools` | `partial`, `partialmethod`, `cache`, `lru_cache`, `cached_property`, `reduce`, `singledispatch`, `total_ordering`, `update_wrapper` | `bind`, memoize, `reduce` |
| `lru_cache` caveats | Arguments must be hashable; on methods it keeps instances alive | — |
| `operator` | `itemgetter`, `attrgetter` (dotted), `methodcaller`, operator functions; faster than `lambda` | arrow functions |
| Frames | `sys._getframe()`, `f_code`, `f_locals` (write-through proxy), `f_back`, `f_lineno` | execution contexts |
| Stack views | `inspect.stack()` (FrameInfo), `traceback.extract_stack()` (FrameSummary) | `new Error().stack` |
| Recursion | Limit ~1000, `RecursionError`, no tail calls; rewrite with an explicit stack | `RangeError` |
| `inspect.signature` | `Parameter` kinds/defaults/annotations, `bind`, `apply_defaults`; basis of DI | `fn.length` (much less) |

**Further reading**

- [Glossary: decorator](https://docs.python.org/3/glossary.html#term-decorator),
  [function definitions (decorators)](https://docs.python.org/3/reference/compound_stmts.html#function-definitions),
  [class definitions](https://docs.python.org/3/reference/compound_stmts.html#class-definitions)
- [`functools`](https://docs.python.org/3/library/functools.html),
  [`operator`](https://docs.python.org/3/library/operator.html),
  [`inspect`](https://docs.python.org/3/library/inspect.html),
  [`traceback`](https://docs.python.org/3/library/traceback.html),
  [`warnings.deprecated`](https://docs.python.org/3/library/warnings.html#warnings.deprecated)
- [`sys._getframe`](https://docs.python.org/3/library/sys.html#sys._getframe),
  [`sys.setrecursionlimit`](https://docs.python.org/3/library/sys.html#sys.setrecursionlimit),
  [frame objects](https://docs.python.org/3/reference/datamodel.html#frame-objects)
- [PEP 318 — Decorators for functions and methods](https://peps.python.org/pep-0318/),
  [PEP 3129 — Class decorators](https://peps.python.org/pep-3129/),
  [PEP 614 — Relaxing grammar restrictions on decorators](https://peps.python.org/pep-0614/)
- [PEP 443 — Single-dispatch generic functions](https://peps.python.org/pep-0443/),
  [PEP 362 — Function signature object](https://peps.python.org/pep-0362/),
  [PEP 612 — Parameter specification variables](https://peps.python.org/pep-0612/)
- [PEP 667 — Consistent views of namespaces](https://peps.python.org/pep-0667/),
  [PEP 702 — Marking deprecations](https://peps.python.org/pep-0702/)

## Unit 1.13 — Iterators, Generators and itertools

**Objectives**

- Tell an iterable (gives fresh iterators) from an iterator (single use), and write
  both as classes.
- Write generator functions and expressions, and explain their lazy start, states,
  suspended frames and return values.
- Build lazy, pull-based pipelines that process data of any size in constant memory.
- Delegate with `yield from`, and drive two-way generators with `send`, `throw` and
  `close`.
- See how `async def` coroutines grew out of generators, by driving one by hand.
- Use every `itertools` building block, and know where the recipes live.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `[Symbol.iterator]()` | `__iter__()` | Called by `for`, `list()`, unpacking, `in`, … |
| `iterator.next()` → `{value, done}` | `__next__()` returns the value, raises `StopIteration` | No result object; the end is an exception |
| `function*` / `yield` | `def` containing `yield` | Any `yield` in the body makes it a generator function |
| `yield*` | `yield from` | Also forwards `send` / `throw` and returns the sub-generator's return value |
| `gen.next(value)` | `gen.send(value)` | The first call must send `None` (priming) |
| `gen.return()` / `gen.throw(e)` | `gen.close()` / `gen.throw(e)` | `close` raises `GeneratorExit` inside the generator |
| `async function` / `await` | `async def` / `await` | Coroutines are generator machinery underneath; you can drive one with `send` |
| Iterator helpers (`.map`, `.take`, ES2025) | `itertools`, generator expressions | Python has had lazy tools since 2.3 |

The next cell is this unit's setup (`scratch_dir` is the git-ignored folder used by
file demos).

In [729]:
import datetime as dt
import dis
import importlib.util
import inspect
import io
import itertools
import operator
import sys
import tracemalloc
from collections import Counter, deque  # noqa: F401 (deque: Lab 1.13)
from collections.abc import Callable, Generator, Iterable, Iterator
from pathlib import Path
from typing import Any, NamedTuple


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


scratch_dir = find_repo_root() / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.13.1 Iterable vs iterator

Unit 1.9.1 introduced the protocol; this section pins down the difference that causes
real bugs.

| | Iterable | Iterator |
| --- | --- | --- |
| Method | `__iter__()` returns an iterator | `__next__()` returns the next item or raises `StopIteration`; `__iter__()` returns `self` |
| Examples | `list`, `tuple`, `str`, `dict`, `set`, `range` | file objects, generators, `map`, `filter`, `zip`, `enumerate`, `iter(list)` |
| Loop it twice | works: each loop gets a **fresh** iterator | the second loop is **empty**: iterators are single use |

Every iterator is also iterable (its `__iter__` returns itself), which is why you can
loop over a `map` object, but only once.

In [730]:
basket_prices = [249, 640, 162]
print(f"{iter(basket_prices) is iter(basket_prices)=}  (fresh iterator each time)")

with_gst = map(lambda price: round(price * 1.18), basket_prices)
print(f"{iter(with_gst) is with_gst=}  (an iterator returns itself)")
print(f"first pass:  {list(with_gst)}")
print(f"second pass: {list(with_gst)}  (already consumed)")

iter(basket_prices) is iter(basket_prices)=False  (fresh iterator each time)
iter(with_gst) is with_gst=True  (an iterator returns itself)
first pass:  [294, 755, 191]
second pass: []  (already consumed)


The `collections.abc` classes check the protocol: `Iterable` needs `__iter__`,
`Iterator` needs both methods. The bug pattern below: a function loops over its
argument twice. With a list it works; with a single-use iterator, the second loop
silently sees nothing.

In [731]:
def price_report(prices: Iterable[int]) -> str:
    """Loops over prices twice: fine for iterables, wrong for iterators."""
    total = sum(prices)
    count = sum(1 for _ in prices)
    return f"total={total}, count={count}"


print(f"{isinstance(basket_prices, Iterable)=}, {isinstance(basket_prices, Iterator)=}")
print(f"{isinstance(iter(basket_prices), Iterator)=}")
print(f"list:     {price_report(basket_prices)}")
print(f"iterator: {price_report(iter(basket_prices))}  (count is wrong)")

isinstance(basket_prices, Iterable)=True, isinstance(basket_prices, Iterator)=False
isinstance(iter(basket_prices), Iterator)=True
list:     total=1051, count=3
iterator: total=1051, count=0  (count is wrong)


> **Note:** when a function needs two passes, either accept a `Sequence` / `list`, or
> call `list(prices)` once at the top. Guard against the bug with
> `if iter(prices) is prices: prices = list(prices)`.

**JS bridge: `Symbol.iterator`.** The protocols are the same idea with different
spelling:

```javascript
const prices = [249, 640];
const it = prices[Symbol.iterator]();
it.next();  // { value: 249, done: false }
it.next();  // { value: 640, done: false }
it.next();  // { value: undefined, done: true }
```

Python's `next()` returns the value directly and signals the end with an exception,
which `for` catches for you. `next(iterator, default)` returns `default` instead.

In [732]:
price_iterator = iter([249, 640])
print(next(price_iterator), next(price_iterator))
print(f"{next(price_iterator, 'done')=}")
print(f"next() after the end: {raised_error_name(lambda: next(price_iterator))}")

249 640
next(price_iterator, 'done')='done'
next() after the end: StopIteration


### 1.13.2 Custom iterator classes

**Separate the iterable from its iterator.** The container (`DeliveryWindow`) stores
the data and returns a **new** iterator object from `__iter__`; the iterator
(`DeliveryWindowIterator`) holds the position. Then any number of loops, even nested
ones, work independently.

In [733]:
class DeliveryWindowIterator:
    """Iterator over the days of a DeliveryWindow; holds the position."""

    def __init__(self, first_day: dt.date, day_count: int) -> None:
        self.next_day = first_day
        self.days_left = day_count

    def __iter__(self) -> DeliveryWindowIterator:
        return self

    def __next__(self) -> dt.date:
        if self.days_left == 0:
            raise StopIteration
        current_day = self.next_day
        self.next_day += dt.timedelta(days=1)
        self.days_left -= 1
        return current_day


class DeliveryWindow:
    """Iterable: a range of delivery dates; every loop gets a new iterator."""

    def __init__(self, first_day: dt.date, day_count: int) -> None:
        self.first_day = first_day
        self.day_count = day_count

    def __iter__(self) -> DeliveryWindowIterator:
        return DeliveryWindowIterator(self.first_day, self.day_count)

The nested loop pairs every day with every day (3 × 3 = 9 pairs) because the inner
and outer loops each got their own iterator.

In [734]:
festival_window = DeliveryWindow(dt.date(2026, 11, 7), 3)
print([day.isoformat() for day in festival_window])
print([day.isoformat() for day in festival_window])  # works again
pairs = [(first.day, second.day) for first in festival_window for second in festival_window]
print(f"{len(pairs)=}")

['2026-11-07', '2026-11-08', '2026-11-09']
['2026-11-07', '2026-11-08', '2026-11-09']
len(pairs)=9


**The shortcut that breaks.** Making the container its own iterator (`__iter__`
returns `self`) looks simpler, but the position lives on the container. The nested
loop shares one position: the inner loop consumes days 2 and 3 during the outer
loop's first pass, so it produces 2 pairs instead of 9, and a later loop is empty.

In [735]:
class OneShotWindow:
    """Bug: the container is its own iterator."""

    def __init__(self, first_day: dt.date, day_count: int) -> None:
        self.next_day = first_day
        self.days_left = day_count

    def __iter__(self) -> OneShotWindow:
        return self

    def __next__(self) -> dt.date:
        if self.days_left == 0:
            raise StopIteration
        self.days_left -= 1
        self.next_day += dt.timedelta(days=1)
        return self.next_day - dt.timedelta(days=1)


one_shot = OneShotWindow(dt.date(2026, 11, 7), 3)
nested_pairs = [(first.day, second.day) for first in one_shot for second in one_shot]
print(f"{len(nested_pairs)=}, {list(one_shot)=}")

len(nested_pairs)=2, list(one_shot)=[]


**The `__getitem__` sequence fallback.** Before the iterator protocol existed (2.2),
`for` worked on anything with `__getitem__`: Python calls it with 0, 1, 2, … until it
raises `IndexError`. `iter()` still supports that, wrapping the object in a generic
`iterator`. The catch: `isinstance(obj, Iterable)` is **False**, because the ABC only
checks for `__iter__`. The reliable test is to call `iter(obj)`.

In [736]:
class SeatRow:
    """Old-style sequence: only __getitem__, no __iter__."""

    def __init__(self, seat_count: int) -> None:
        self.seat_count = seat_count

    def __getitem__(self, index: int) -> str:
        if index >= self.seat_count:
            raise IndexError(index)
        return f"A{index + 1}"


front_row = SeatRow(4)
print(f"{list(front_row)=}, {'A3' in front_row=}")
print(f"{type(iter(front_row)).__name__=}")
print(f"{isinstance(front_row, Iterable)=}  (the ABC misses the fallback)")

list(front_row)=['A1', 'A2', 'A3', 'A4'], 'A3' in front_row=True
type(iter(front_row)).__name__='iterator'
isinstance(front_row, Iterable)=False  (the ABC misses the fallback)


### 1.13.3 Generator functions

Writing iterator classes by hand is tedious. A **generator function**, a `def` that
contains `yield`, does it for you. Calling it does **not** run the body: it returns a
generator object (an iterator). Each `next()` runs the body until the next `yield`,
hands out the yielded value, and **suspends** the function with all its local
variables intact. When the body ends, `StopIteration` is raised.

In [737]:
def order_statuses() -> Iterator[str]:
    """Yield the stages of an order, printing as the body runs."""
    print("  body starts")
    yield "placed"
    print("  resumed after 'placed'")
    yield "packed"
    print("  resumed after 'packed'")
    yield "shipped"
    print("  body ends")


status_generator = order_statuses()
print(f"{type(status_generator).__name__=}  (nothing printed yet: lazy start)")
print(f"{next(status_generator)=}")
print(f"{next(status_generator)=}")
print(f"rest: {list(status_generator)}")

type(status_generator).__name__='generator'  (nothing printed yet: lazy start)
  body starts
next(status_generator)='placed'
  resumed after 'placed'
next(status_generator)='packed'
  resumed after 'packed'
  body ends
rest: ['shipped']


**States.** `inspect.getgeneratorstate(gen)` reports one of four states. `GEN_RUNNING`
can only be observed from inside the generator itself, so the next cell's generator
inspects itself through a global name.

| State | Meaning |
| --- | --- |
| `GEN_CREATED` | created, body not started |
| `GEN_RUNNING` | currently executing |
| `GEN_SUSPENDED` | paused at a `yield` |
| `GEN_CLOSED` | finished (returned, raised, or closed) |

In [738]:
def self_reporting() -> Iterator[str]:
    """Yield its own state while running."""
    yield inspect.getgeneratorstate(watched_generator)


watched_generator = self_reporting()
states = [inspect.getgeneratorstate(watched_generator)]
states.append(next(watched_generator))
states.append(inspect.getgeneratorstate(watched_generator))
states.append(raised_error_name(lambda: next(watched_generator)))
states.append(inspect.getgeneratorstate(watched_generator))
print(states)

['GEN_CREATED', 'GEN_RUNNING', 'GEN_SUSPENDED', 'StopIteration', 'GEN_CLOSED']


#### Under the hood: suspended frames

A generator owns its frame (`gi_frame`). While suspended, the frame keeps the current
line and all locals; that is what "pausing" means. `gi_suspended` (*New in 3.11*)
says whether it is paused at a `yield`. When the generator finishes, the frame is
released and `gi_frame` becomes `None`.

In [739]:
def running_totals(amounts: list[int]) -> Iterator[int]:
    """Yield the running total after each amount."""
    total = 0
    for amount in amounts:
        total += amount
        yield total


totals = running_totals([249, 640, 162])
next(totals)
next(totals)
frame = totals.gi_frame
print(f"{totals.gi_suspended=}, {frame.f_locals['total']=}, {frame.f_locals['amount']=}")
print(f"{frame.f_lineno - totals.gi_code.co_firstlineno=} (line offset of the yield)")
print(f"{list(totals)=}, {totals.gi_frame=}")

totals.gi_suspended=True, frame.f_locals['total']=889, frame.f_locals['amount']=640
frame.f_lineno - totals.gi_code.co_firstlineno=5 (line offset of the yield)
list(totals)=[1051], totals.gi_frame=None


The compiler marks generator functions with the `CO_GENERATOR` flag. Their bytecode
starts with `RETURN_GENERATOR` (which builds the generator object instead of running
the body) and pauses at `YIELD_VALUE`. Opcode names vary between versions.

In [740]:
print(f"{bool(running_totals.__code__.co_flags & inspect.CO_GENERATOR)=}")
print(f"{inspect.isgeneratorfunction(running_totals)=}, {inspect.isgenerator(totals)=}")
opnames = [instruction.opname for instruction in dis.get_instructions(running_totals)]
print(f"{opnames[0]=}, {'YIELD_VALUE' in opnames=}")

bool(running_totals.__code__.co_flags & inspect.CO_GENERATOR)=True
inspect.isgeneratorfunction(running_totals)=True, inspect.isgenerator(totals)=True
opnames[0]='RETURN_GENERATOR', 'YIELD_VALUE' in opnames=True


**`return` in a generator.** A `return value` statement ends the generator; the
value travels as `StopIteration.value`. A `for` loop discards it, but code that calls
`next()` itself (and `yield from`, 1.13.6) can read it.

In [741]:
def take_until_budget(prices: list[int], budget: int) -> Generator[int, None, int]:
    """Yield prices while they fit the budget; return the money left."""
    for price in prices:
        if price > budget:
            break
        budget -= price
        yield price
    return budget


shopper = take_until_budget([249, 640, 162], budget=1000)
bought = [next(shopper), next(shopper)]
try:
    next(shopper)
except StopIteration as stop:
    print(f"{bought=}, money left: {stop.value}")

bought=[249, 640], money left: 111


> **Note:** the type hint `Generator[YieldType, SendType, ReturnType]` describes all
> three channels. `Iterator[int]` is enough when only `yield` is used. `SendType` and
> `ReturnType` default to `None` (*New in 3.13*, PEP 696), so `Generator[str, int]`
> means "yields `str`, receives `int`, returns nothing".

### 1.13.4 Generator expressions

A **generator expression** is a comprehension in parentheses: `(expr for x in
iterable if condition)`. It builds a generator, not a list. When it is the only
argument of a call, the extra parentheses can be dropped:
`sum(price * 2 for price in prices)`.

In [742]:
basket_prices = [249, 640, 162, 980]
doubled_lazily = (price * 2 for price in basket_prices if price < 900)
print(f"{type(doubled_lazily).__name__=}")
print(f"{sum(price * 2 for price in basket_prices)=}")
print(f"{max(len(sku) for sku in ['TEA-500G', 'RICE-5KG', 'SOAP'])=}")

type(doubled_lazily).__name__='generator'
sum(price * 2 for price in basket_prices)=4062
max(len(sku) for sku in ['TEA-500G', 'RICE-5KG', 'SOAP'])=8


**Predict the output.**

```python
expensive = (price for price in [249, 640, 980] if price > 500)
print(list(expensive))
print(list(expensive))
print(980 in expensive)
```

<details><summary>Answer</summary>

`[640, 980]`, then `[]`, then `False`. A generator expression is an iterator: the
first `list()` consumes it, so the second sees nothing, and `in` searches what is left
(nothing).

</details>

Run the next cell to check your prediction.

In [743]:
expensive = (price for price in [249, 640, 980] if price > 500)
print(list(expensive))
print(list(expensive))
print(980 in expensive)

[640, 980]
[]
False


One subtle rule: the **first** `in` iterable is evaluated immediately, when the
expression is created; everything else is lazy. So a bad name in the first `for`
fails at once, while an error in the expression part waits until the item is pulled.

In [744]:
print(f"bad first iterable: {raised_error_name(lambda: (x for x in missing_prices))}")  # noqa: F821
lazy_division = (1000 // quantity for quantity in [4, 0, 2])
print(f"created fine; first item: {next(lazy_division)}")
print(f"second item: {raised_error_name(lambda: next(lazy_division))}")

bad first iterable: NameError
created fine; first item: 250
second item: ZeroDivisionError


**Memory.** A list comprehension materializes every element; a generator expression
holds only its frame, whatever the input size. `sys.getsizeof` shows the containers;
`tracemalloc` shows the **peak** memory of summing a million squares each way
(including the int objects themselves). Exact numbers vary by version.

In [745]:
def peak_bytes(thunk: Callable[[], object]) -> int:
    """Return the peak traced memory while running thunk."""
    tracemalloc.start()
    try:
        thunk()
        return tracemalloc.get_traced_memory()[1]
    finally:
        tracemalloc.stop()


squares_list = [number * number for number in range(1_000_000)]
squares_generator = (number * number for number in range(1_000_000))
print(f"list object: {sys.getsizeof(squares_list):,} bytes")
print(f"generator object: {sys.getsizeof(squares_generator)} bytes")
del squares_list
list_peak = peak_bytes(lambda: sum([number * number for number in range(1_000_000)]))
generator_peak = peak_bytes(lambda: sum(number * number for number in range(1_000_000)))
print(f"peak with list: {list_peak / 1e6:.1f} MB, with generator: {generator_peak / 1e3:.1f} kB")

list object: 8,448,728 bytes
generator object: 216 bytes


peak with list: 40.4 MB, with generator: 0.4 kB


| Use | When |
| --- | --- |
| List comprehension | you need the results more than once, `len()`, indexing, or sorting |
| Generator expression | you consume the results once (`sum`, `max`, `any`, `join`, a loop), or the input is large or infinite |

### 1.13.5 Lazy pipelines

Chain generators so that each stage consumes the previous one: **read → parse →
filter → aggregate**. Nothing happens until the last stage asks for a value; then the
request travels up the chain, and **one item at a time** flows down. Memory stays
constant whatever the input size, and work stops as soon as the consumer stops.

```mermaid
flowchart RL
    aggregate["sum()<br/>aggregate"] -- "next()" --> filter_stage["only_city()<br/>filter"]
    filter_stage -- "next()" --> parse_stage["parse_sales()<br/>parse"]
    parse_stage -- "next()" --> read_stage["read_lines()<br/>read"]
    read_stage -. "one line" .-> parse_stage
    parse_stage -. "one Sale" .-> filter_stage
    filter_stage -. "one matching Sale" .-> aggregate
```

<details><summary>ASCII fallback</summary>

```text
pull:   sum() --next()--> only_city() --next()--> parse_sales() --next()--> read_lines()
push:   sum() <--Sale---- only_city() <--Sale---- parse_sales() <--line---- read_lines()
(one item travels the whole chain before the next line is read)
```

</details>

The stages below print as they work, so the trace shows the interleaving.

In [746]:
class Sale(NamedTuple):
    city: str
    sku: str
    amount: int


def read_lines(stream: io.TextIOBase) -> Iterator[str]:
    """Stage 1: yield non-empty lines."""
    for line in stream:
        if line.strip():
            print(f"    read  {line.strip()!r}")
            yield line


def parse_sales(lines: Iterable[str]) -> Iterator[Sale]:
    """Stage 2: turn 'city,sku,amount' lines into Sale records."""
    for line in lines:
        city, sku, amount = line.strip().split(",")
        yield Sale(city, sku, int(amount))


def only_city(sales: Iterable[Sale], city: str) -> Iterator[Sale]:
    """Stage 3: keep one city's sales."""
    for sale in sales:
        if sale.city == city:
            print(f"    keep  {sale.sku} {sale.amount}")
            yield sale

Stage 4 is `sum`. Building the pipeline prints nothing; `sum` then pulls each line
through every stage before the next line is read.

In [747]:
sales_csv = "Pune,TEA-500G,249\nDelhi,RICE-5KG,640\n\nPune,SOAP-3PK,162\nGoa,TEA-500G,249\n"
pune_sales = only_city(parse_sales(read_lines(io.StringIO(sales_csv))), "Pune")
print("pipeline built, nothing read yet")
pune_total = sum(sale.amount for sale in pune_sales)
print(f"{pune_total=}")

pipeline built, nothing read yet
    read  'Pune,TEA-500G,249'
    keep  TEA-500G 249
    read  'Delhi,RICE-5KG,640'
    read  'Pune,SOAP-3PK,162'
    keep  SOAP-3PK 162
    read  'Goa,TEA-500G,249'
pune_total=411


**Early termination.** Asking only for the first Pune sale reads **one** line; the
rest of the input is never touched. With a 10 GB file that is the difference between
milliseconds and minutes.

In [748]:
first_pune_sale = next(only_city(parse_sales(read_lines(io.StringIO(sales_csv))), "Pune"))
print(f"{first_pune_sale=}")

    read  'Pune,TEA-500G,249'
    keep  TEA-500G 249
first_pune_sale=Sale(city='Pune', sku='TEA-500G', amount=249)


### 1.13.6 `yield from`

`yield from sub_iterable` hands control to another iterable until it is exhausted:
every item it produces is yielded to our caller. For plain iteration it is the same as
`for item in sub_iterable: yield item`, just shorter. (It does more for two-way
generators, 1.13.7.)

In [749]:
def warehouse_skus() -> Iterator[str]:
    yield from ["TEA-500G", "RICE-5KG"]


def store_skus() -> Iterator[str]:
    yield from ("SOAP-3PK", "TEA-500G")


def all_skus() -> Iterator[str]:
    """Delegate to two sub-generators in turn."""
    yield from warehouse_skus()
    yield from store_skus()


print(list(all_skus()))

['TEA-500G', 'RICE-5KG', 'SOAP-3PK', 'TEA-500G']


**Recursive tree walks.** A recursive generator yields its own items and delegates
the subtrees with `yield from`. The category tree below is nested dicts; the walk
yields every path, depth first.

In [750]:
category_tree: dict[str, dict[str, Any]] = {
    "Grocery": {"Tea": {}, "Rice": {"Basmati": {}, "Sona Masoori": {}}},
    "Home": {"Cleaning": {"Soap": {}}},
}


def category_paths(tree: dict[str, Any], prefix: str = "") -> Iterator[str]:
    """Yield 'A/B/C' paths for every node of a nested-dict tree."""
    for name, subtree in tree.items():
        path = f"{prefix}/{name}" if prefix else name
        yield path
        yield from category_paths(subtree, path)


for path in category_paths(category_tree):
    print(f"  {path}")

  Grocery
  Grocery/Tea
  Grocery/Rice
  Grocery/Rice/Basmati
  Grocery/Rice/Sona Masoori
  Home
  Home/Cleaning
  Home/Cleaning/Soap


**Capturing return values.** `result = yield from sub_generator()` evaluates to the
sub-generator's `return` value (its `StopIteration.value`). This lets a sub-generator
produce items **and** report a summary. Below, each section of an inventory file
yields its items and returns how many it had.

In [751]:
def section_items(section_name: str, items: list[str]) -> Generator[str, None, int]:
    """Yield 'section: item' strings; return the item count."""
    for item in items:
        yield f"{section_name}: {item}"
    return len(items)


def inventory(sections: dict[str, list[str]]) -> Generator[str, None, dict[str, int]]:
    """Yield every item; return the count per section."""
    counts = {}
    for section_name, items in sections.items():
        counts[section_name] = yield from section_items(section_name, items)
    return counts


inventory_walk = inventory({"Shelf A": ["tea", "rice"], "Shelf B": ["soap"]})
print(list(itertools.islice(inventory_walk, 10)))
print(f"{inspect.getgeneratorstate(inventory_walk)=}")


def collect_return(generator: Generator[Any, None, Any]) -> Any:
    """Exhaust a generator and return its return value."""
    while True:
        try:
            next(generator)
        except StopIteration as stop:
            return stop.value


print(f"{collect_return(inventory({'Shelf A': ['tea', 'rice'], 'Shelf B': ['soap']}))=}")

['Shelf A: tea', 'Shelf A: rice', 'Shelf B: soap']
inspect.getgeneratorstate(inventory_walk)='GEN_CLOSED'
collect_return(inventory({'Shelf A': ['tea', 'rice'], 'Shelf B': ['soap']}))={'Shelf A': 2, 'Shelf B': 1}


### 1.13.7 Two-way generators

`yield` is also an **expression**: `received = yield value` hands out `value`, and
when the generator is resumed with `gen.send(something)`, `received` becomes
`something`. `next(gen)` is the same as `gen.send(None)`.

| Method | Effect at the paused `yield` |
| --- | --- |
| `gen.send(value)` | `yield` evaluates to `value`; runs to the next `yield`, returns its value |
| `gen.throw(exception)` | the exception is raised at the `yield` |
| `gen.close()` | `GeneratorExit` is raised at the `yield`; the generator must finish |

**Priming.** A new generator has not reached a `yield` yet, so there is nothing to
receive a value: the first call must be `next(gen)` or `gen.send(None)`.

In [752]:
def duplicate_detector() -> Generator[bool | None, str]:
    """Receive order ids; answer True if the id was seen before."""
    seen_ids: set[str] = set()
    answer: bool | None = None
    while True:
        order_id = yield answer
        answer = order_id in seen_ids
        seen_ids.add(order_id)


detector = duplicate_detector()
print(f"priming: {next(detector)=}")
print([detector.send(order_id) for order_id in ["A-1", "A-2", "A-1", "A-3", "A-2"]])
unprimed = duplicate_detector()
try:
    unprimed.send("A-1")
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

priming: next(detector)=None
[False, False, True, False, True]
TypeError: can't send non-None value to a just-started generator


Forgetting to prime is so common that coroutine-style generators were often wrapped
in a small decorator (Unit 1.12) that calls `next()` once.

In [753]:
def primed[**P](
    generator_function: Callable[P, Generator[Any, Any, Any]],
) -> Callable[P, Generator[Any, Any, Any]]:
    """Decorator: create the generator and advance it to the first yield."""

    def start(*args: P.args, **kwargs: P.kwargs) -> Generator[Any, Any, Any]:
        generator = generator_function(*args, **kwargs)
        next(generator)
        return generator

    return start


ready_detector = primed(duplicate_detector)()
print([ready_detector.send(order_id) for order_id in ["B-7", "B-7"]])

[False, True]


**`throw`** raises an exception at the paused `yield`. The generator may handle it
and continue (its next `yield` value is what `throw` returns), or let it propagate to
the caller.

In [754]:
def resilient_reader(lines: list[str]) -> Iterator[str]:
    """Yield lines; on a ValueError from the caller, report and carry on."""
    for line in lines:
        try:
            yield line
        except ValueError as error:
            yield f"skipped after error: {error}"


reader = resilient_reader(["line 1", "line 2", "line 3"])
print(next(reader))
print(reader.throw(ValueError("bad checksum")))
print(next(reader), next(reader))

line 1
skipped after error: bad checksum
line 2 line 3


**`close` and cleanup.** `close()` raises `GeneratorExit` (a `BaseException`) at the
paused `yield`. Put cleanup in a `finally` block: it runs whether the generator
finishes, fails, is closed explicitly, or is garbage-collected half-way. A generator
that **yields** in response to `GeneratorExit` is a bug, reported as `RuntimeError`.

In [755]:
def database_rows(table: str) -> Iterator[str]:
    """Pretend to stream rows from a connection; always release it."""
    print(f"  open connection for {table}")
    try:
        for row_number in itertools.count(1):
            yield f"{table} row {row_number}"
    finally:
        print(f"  close connection for {table}")


rows = database_rows("orders")
print(next(rows), "|", next(rows))
rows.close()
print(f"{inspect.getgeneratorstate(rows)=}, {issubclass(GeneratorExit, Exception)=}")
for row in database_rows("customers"):  # noqa: B007
    break  # leaving the loop drops the generator; finally runs when it is collected

  open connection for orders
orders row 1 | orders row 2
  close connection for orders
inspect.getgeneratorstate(rows)='GEN_CLOSED', issubclass(GeneratorExit, Exception)=False
  open connection for customers
  close connection for customers


The `RuntimeError` case, for completeness:

In [756]:
def refuses_to_stop() -> Iterator[int]:
    """Bug: yields again after GeneratorExit."""
    try:
        yield 1
    except GeneratorExit:
        yield 2


stubborn = refuses_to_stop()
next(stubborn)
try:
    stubborn.close()
except RuntimeError as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: generator ignored GeneratorExit


### 1.13.8 From generators to async

`async` / `await` did not appear from nowhere: each step below added one capability
to generators, and the last one gave coroutines their own syntax.

| PEP | Python | Added | Why it mattered |
| --- | --- | --- | --- |
| [255](https://peps.python.org/pep-0255/) | 2.2 | `yield`: simple generators | functions that pause and resume |
| [342](https://peps.python.org/pep-0342/) | 2.5 | `send`, `throw`, `close`; `yield` as an expression | generators become **coroutines**: a scheduler can resume them with results |
| [380](https://peps.python.org/pep-0380/) | 3.3 | `yield from` | a coroutine can call another and get its return value (asyncio 3.4 was built on this) |
| [492](https://peps.python.org/pep-0492/) | 3.5 | `async def`, `await`, `__await__` | native coroutines, a distinct type that cannot be confused with a generator |
| [525](https://peps.python.org/pep-0525/) | 3.6 | async generators | `yield` inside `async def` |

A coroutine is driven exactly like a two-way generator: `coroutine.send(value)` runs
it to the next point where something deep inside an `await` **yields**, and
`StopIteration.value` carries its result. The event loop (Phase 5) is "just" the code
that calls `send` and decides when.

The next cell defines an awaitable, `Delay`, whose `__await__` is a generator that
yields a request to whoever drives the coroutine, and receives a reply.

In [757]:
class Delay:
    """Awaitable: ask the driver to wait, receive its reply."""

    def __init__(self, seconds: float) -> None:
        self.seconds = seconds

    def __await__(self) -> Generator[tuple[str, float], str, str]:
        reply = yield ("sleep", self.seconds)  # goes out through coroutine.send()
        return reply


async def fetch_order(order_id: int) -> dict[str, Any]:
    """A native coroutine that awaits twice."""
    first_reply = await Delay(0.5)
    second_reply = await Delay(0.2)
    return {"order_id": order_id, "replies": [first_reply, second_reply]}

Drive it by hand. Calling `fetch_order(42)` creates a coroutine object without running
it (like a generator); each `send` runs it to the next yielded request.

In [758]:
order_coroutine = fetch_order(42)
print(f"{inspect.getcoroutinestate(order_coroutine)=}")
first_request = order_coroutine.send(None)
print(f"{first_request=}, {inspect.getcoroutinestate(order_coroutine)=}")
second_request = order_coroutine.send("woke after 0.5 s")
print(f"{second_request=}")
try:
    order_coroutine.send("woke after 0.2 s")
except StopIteration as stop:
    print(f"result: {stop.value}")

inspect.getcoroutinestate(order_coroutine)='CORO_CREATED'
first_request=('sleep', 0.5), inspect.getcoroutinestate(order_coroutine)='CORO_SUSPENDED'
second_request=('sleep', 0.2)
result: {'order_id': 42, 'replies': ['woke after 0.5 s', 'woke after 0.2 s']}


A **minimal event loop** is a loop around `send`. This one runs several coroutines on
a virtual clock: it always resumes the coroutine whose wake-up time is earliest, so
the two orders interleave and both finish at t=0.7 s instead of 1.4 s: they waited
concurrently. Real asyncio does the same with sockets and timers (Phase 5).

In [759]:
def run_all(coroutines: list[Any]) -> list[Any]:
    """Run coroutines that await Delay, interleaving them on a virtual clock."""
    ready = [(0.0, index, coroutine, None) for index, coroutine in enumerate(coroutines)]
    results: list[Any] = [None] * len(coroutines)
    while ready:
        ready.sort(key=lambda entry: entry[0])
        now, index, coroutine, reply = ready.pop(0)
        try:
            _, seconds = coroutine.send(reply)
        except StopIteration as stop:
            results[index] = stop.value
            print(f"  t={now:.1f}s order {stop.value['order_id']} done")
            continue
        ready.append((now + seconds, index, coroutine, f"woke at {now + seconds:.1f}s"))
    return results


finished = run_all([fetch_order(1), fetch_order(2)])
print(finished[1])

  t=0.7s order 1 done
  t=0.7s order 2 done
{'order_id': 2, 'replies': ['woke at 0.5s', 'woke at 0.7s']}


### 1.13.9 The `itertools` module

`itertools` provides fast, memory-efficient building blocks that return **iterators**.
Every row of the table has its own demo below, grouped as in the table. Infinite
iterators are always capped with `islice`, `zip` or `takewhile`.

| Group | Function | Yields | Example → result |
| --- | --- | --- | --- |
| Infinite | `count(start=0, step=1)` | start, start+step, … | `count(100, 5)` → 100 105 110 … |
| Infinite | `cycle(iterable)` | the items, repeated forever | `cycle("AB")` → A B A B … |
| Infinite | `repeat(obj, times=None)` | `obj`, `times` times (or forever) | `repeat(0, 3)` → 0 0 0 |
| Terminating | `accumulate(iterable, func=add, *, initial=None)` | running results | `accumulate([1, 2, 3])` → 1 3 6 |
| Terminating | `batched(iterable, n, *, strict=False)` | tuples of `n` items | `batched("ABCDE", 2)` → AB CD E |
| Terminating | `chain(*iterables)` | items of each iterable in turn | `chain("AB", "CD")` → A B C D |
| Terminating | `chain.from_iterable(iterable)` | same, from one iterable of iterables | `chain.from_iterable(["AB", "CD"])` → A B C D |
| Terminating | `compress(data, selectors)` | data items whose selector is true | `compress("ABC", [1, 0, 1])` → A C |
| Terminating | `dropwhile(predicate, iterable)` | everything after the predicate first fails | `dropwhile(lambda x: x < 3, [1, 4, 1])` → 4 1 |
| Terminating | `takewhile(predicate, iterable)` | items until the predicate first fails | `takewhile(lambda x: x < 3, [1, 4, 1])` → 1 |
| Terminating | `filterfalse(predicate, iterable)` | items where the predicate is false | `filterfalse(str.isdigit, "a1b")` → a b |
| Terminating | `groupby(iterable, key=None)` | `(key, group)` for each run of equal keys | `groupby("AAB")` → (A, AA) (B, B) |
| Terminating | `islice(iterable, stop)` / `(start, stop, step)` | a slice of any iterator | `islice("ABCDEF", 1, 5, 2)` → B D |
| Terminating | `pairwise(iterable)` | overlapping pairs | `pairwise("ABC")` → AB BC |
| Terminating | `starmap(function, iterable)` | `function(*args)` for each args tuple | `starmap(pow, [(2, 3)])` → 8 |
| Terminating | `tee(iterable, n=2)` | `n` independent iterators | `tee("AB")` → two iterators |
| Terminating | `zip_longest(*iterables, fillvalue=None)` | zip to the **longest** input | `zip_longest("AB", "x", fillvalue="-")` → Ax B- |
| Combinatoric | `product(*iterables, repeat=1)` | Cartesian product | `product("AB", "12")` → A1 A2 B1 B2 |
| Combinatoric | `permutations(iterable, r=None)` | ordered arrangements | `permutations("ABC", 2)` → AB AC BA … (6) |
| Combinatoric | `combinations(iterable, r)` | unordered selections, no repeats | `combinations("ABC", 2)` → AB AC BC |
| Combinatoric | `combinations_with_replacement(iterable, r)` | unordered selections, repeats allowed | `…("AB", 2)` → AA AB BB |

#### Infinite iterators

**`count(start, step)`**: an endless counter, often zipped with finite data to number
it, or used for generated ids.

In [760]:
invoice_numbers = itertools.count(1001)
print([f"INV-{next(invoice_numbers)}" for _ in range(3)])
print(list(zip(["tea", "rice", "soap"], itertools.count(10, 10), strict=False)))

['INV-1001', 'INV-1002', 'INV-1003']
[('tea', 10), ('rice', 20), ('soap', 30)]


**`cycle(iterable)`**: repeat the items forever (round-robin assignment, alternating styles).

In [761]:
delivery_agents = itertools.cycle(["Asha", "Ravi"])
orders = ["A-1", "A-2", "A-3", "A-4", "A-5"]
print([(order, next(delivery_agents)) for order in orders])

[('A-1', 'Asha'), ('A-2', 'Ravi'), ('A-3', 'Asha'), ('A-4', 'Ravi'), ('A-5', 'Asha')]


**`repeat(obj, times)`**: the same object again and again; handy as a constant argument to `map`.

In [762]:
print(list(itertools.repeat("-", 5)))
print(list(map(pow, [2, 3, 4], itertools.repeat(2))))  # noqa: B912 (repeat is infinite)

['-', '-', '-', '-', '-']
[4, 9, 16]


#### Terminating iterators

**`accumulate(iterable, func, initial=None)`**: running totals by default; any
two-argument function works (`max` gives a running maximum).

In [763]:
daily_sales = [450, 1200, 300, 1500]
print(f"{list(itertools.accumulate(daily_sales))=}")
print(f"{list(itertools.accumulate(daily_sales, max))=}")
print(f"{list(itertools.accumulate(daily_sales, operator.add, initial=5000))=}")

list(itertools.accumulate(daily_sales))=[450, 1650, 1950, 3450]
list(itertools.accumulate(daily_sales, max))=[450, 1200, 1200, 1500]
list(itertools.accumulate(daily_sales, operator.add, initial=5000))=[5000, 5450, 6650, 6950, 8450]


**`batched(iterable, n)`** (*New in 3.12*): chunks of `n` items; the last may be
shorter. `strict=True` (*New in 3.13*) raises `ValueError` for a short last batch.

In [764]:
order_ids = ["A-1", "A-2", "A-3", "A-4", "A-5"]
print(list(itertools.batched(order_ids, 2, strict=False)))
if sys.version_info >= (3, 13):
    print(
        f"strict: {raised_error_name(lambda: list(itertools.batched(order_ids, 2, strict=True)))}"
    )
else:
    print("batched(strict=True) needs Python 3.13; skipped")

[('A-1', 'A-2'), ('A-3', 'A-4'), ('A-5',)]
strict: ValueError


**`chain(*iterables)`**: one iterator over several iterables, without copying them.

In [765]:
morning_orders, evening_orders = ["A-1", "A-2"], ("B-1",)
print(list(itertools.chain(morning_orders, evening_orders, "C")))

['A-1', 'A-2', 'B-1', 'C']


**`chain.from_iterable(iterable)`**: the same when the iterables arrive as one
(possibly lazy) iterable. It flattens exactly **one** level.

In [766]:
orders_per_store = {"Pune": ["A-1", "A-2"], "Goa": ["B-1"]}
print(list(itertools.chain.from_iterable(orders_per_store.values())))

['A-1', 'A-2', 'B-1']


**`compress(data, selectors)`**: keep the items whose selector is truthy (a boolean mask).

In [767]:
skus = ["TEA-500G", "RICE-5KG", "SOAP-3PK"]
in_stock = [True, False, True]
print(list(itertools.compress(skus, in_stock)))

['TEA-500G', 'SOAP-3PK']


**`dropwhile(predicate, iterable)`**: skip items while the predicate holds, then yield everything (it never checks again).

In [768]:
log_lines = ["# header", "# generated 2026-11-05", "INFO start", "# inline comment", "ERROR stop"]
print(list(itertools.dropwhile(lambda line: line.startswith("#"), log_lines)))

['INFO start', '# inline comment', 'ERROR stop']


**`takewhile(predicate, iterable)`**: yield items while the predicate holds, then stop (safe on infinite input).

In [769]:
sorted_prices = [99, 149, 249, 640, 980]
print(list(itertools.takewhile(lambda price: price < 300, sorted_prices)))
print(list(itertools.takewhile(lambda n: n * n < 50, itertools.count(1))))

[99, 149, 249]
[1, 2, 3, 4, 5, 6, 7]


**`filterfalse(predicate, iterable)`**: the complement of `filter`.

In [770]:
quantities = ["3", "two", "10", "x"]
print(f"{list(filter(str.isdigit, quantities))=}")
print(f"{list(itertools.filterfalse(str.isdigit, quantities))=}")

list(filter(str.isdigit, quantities))=['3', '10']
list(itertools.filterfalse(str.isdigit, quantities))=['two', 'x']


**`groupby(iterable, key)`**: groups **consecutive** items with the same key, like
Unix `uniq`, not SQL `GROUP BY`. Sort by the same key first to get one group per key.
Each group is an iterator that is invalidated when you advance to the next group, so
consume it immediately.

In [771]:
sales = [("Pune", 249), ("Pune", 162), ("Goa", 640), ("Pune", 80)]
unsorted_groups = [
    (city, len(list(group))) for city, group in itertools.groupby(sales, key=operator.itemgetter(0))
]
print(f"{unsorted_groups=}  (runs, not totals)")
by_city = sorted(sales, key=operator.itemgetter(0))
print(
    {
        city: sum(amount for _, amount in group)
        for city, group in itertools.groupby(by_city, key=operator.itemgetter(0))
    }
)

unsorted_groups=[('Pune', 2), ('Goa', 1), ('Pune', 1)]  (runs, not totals)
{'Goa': 640, 'Pune': 491}


**`islice(iterable, stop)` / `islice(iterable, start, stop, step)`**: slicing for any
iterator (no negative indices). It consumes the skipped items.

In [772]:
print(list(itertools.islice(itertools.count(1), 5)))
print(list(itertools.islice("ABCDEFGH", 2, 7, 2)))
report_lines = iter(["header", "row 1", "row 2", "row 3"])
print(f"{list(itertools.islice(report_lines, 1, None))=}  (skip a header)")

[1, 2, 3, 4, 5]
['C', 'E', 'G']
list(itertools.islice(report_lines, 1, None))=['row 1', 'row 2', 'row 3']  (skip a header)


**`pairwise(iterable)`** (*New in 3.10*): consecutive overlapping pairs, e.g. for differences.

In [773]:
stock_levels = [120, 95, 95, 40]
print(list(itertools.pairwise(stock_levels)))
print(f"changes: {[later - earlier for earlier, later in itertools.pairwise(stock_levels)]}")

[(120, 95), (95, 95), (95, 40)]
changes: [-25, 0, -55]


**`starmap(function, iterable)`**: like `map`, but each item is unpacked into the arguments.

In [774]:
line_items = [(249, 2), (640, 1), (162, 3)]
print(list(itertools.starmap(operator.mul, line_items)))

[498, 640, 486]


**`tee(iterable, n)`**: split one iterator into `n` independent ones. It buffers the
items one copy has seen and another has not, so if one runs far ahead, memory grows.
Do not use the original iterator after calling `tee`.

In [775]:
price_feed = iter([249, 640, 162])
for_total, for_maximum = itertools.tee(price_feed)
print(f"{sum(for_total)=}, {max(for_maximum)=}")

sum(for_total)=1051, max(for_maximum)=640


**`zip_longest(*iterables, fillvalue)`**: like `zip`, but runs to the longest input, padding the gaps.

In [776]:
skus = ["TEA-500G", "RICE-5KG", "SOAP-3PK"]
stock_counts = [40, 12]
print(list(itertools.zip_longest(skus, stock_counts, fillvalue=0)))

[('TEA-500G', 40), ('RICE-5KG', 12), ('SOAP-3PK', 0)]


#### Combinatoric iterators

**`product(*iterables, repeat=1)`**: the Cartesian product, the same as nested `for`
loops.

In [777]:
sizes, colours = ["S", "M"], ["red", "blue"]
print([f"{size}-{colour}" for size, colour in itertools.product(sizes, colours)])
print(f"{len(list(itertools.product('01', repeat=4)))=}  (4-bit strings)")

['S-red', 'S-blue', 'M-red', 'M-blue']
len(list(itertools.product('01', repeat=4)))=16  (4-bit strings)


**`permutations(iterable, r)`**: ordered arrangements; `n! / (n - r)!` of them.

In [778]:
podium = list(itertools.permutations(["Asha", "Ravi", "Meena"], 2))
print(f"{len(podium)=}, first three: {podium[:3]}")

len(podium)=6, first three: [('Asha', 'Ravi'), ('Asha', 'Meena'), ('Ravi', 'Asha')]


**`combinations(iterable, r)`**: unordered selections without repeats; `n! / (r! (n - r)!)` of them.

In [779]:
combo_items = ["tea", "rice", "soap", "oil"]
print(list(itertools.combinations(combo_items, 2)))

[('tea', 'rice'), ('tea', 'soap'), ('tea', 'oil'), ('rice', 'soap'), ('rice', 'oil'), ('soap', 'oil')]


**`combinations_with_replacement(iterable, r)`**: unordered selections where an item may repeat.

In [780]:
coin_values = [1, 2, 5]
two_coin_amounts = sorted(
    {sum(pair) for pair in itertools.combinations_with_replacement(coin_values, 2)}
)
print(f"{list(itertools.combinations_with_replacement(coin_values, 2))=}")
print(f"{two_coin_amounts=}")

list(itertools.combinations_with_replacement(coin_values, 2))=[(1, 1), (1, 2), (1, 5), (2, 2), (2, 5), (5, 5)]
two_coin_amounts=[2, 3, 4, 6, 7, 10]


#### The itertools recipes and `more-itertools`

The [itertools documentation](https://docs.python.org/3/library/itertools.html#itertools-recipes)
ends with **recipes**: short functions built from the blocks above, meant to be copied
into your code. Five of them:

| Recipe | Does |
| --- | --- |
| `take(n, iterable)` | first `n` items as a list |
| `nth(iterable, n, default=None)` | the item at index `n` |
| `all_equal(iterable)` | are all items equal? |
| `unique_everseen(iterable, key=None)` | unique items, first occurrence order |
| `partition(predicate, iterable)` | (items where false, items where true) |

In [781]:
def take(n: int, iterable: Iterable[Any]) -> list[Any]:
    """Return the first n items as a list."""
    return list(itertools.islice(iterable, n))


def nth(iterable: Iterable[Any], n: int, default: Any = None) -> Any:
    """Return the nth item or a default."""
    return next(itertools.islice(iterable, n, None), default)


def all_equal(iterable: Iterable[Any]) -> bool:
    """Return True if all items are equal."""
    groups = itertools.groupby(iterable)
    return next(groups, True) and not next(groups, False)


def unique_everseen(
    iterable: Iterable[Any], key: Callable[[Any], Any] | None = None
) -> Iterator[Any]:
    """Yield unique items, preserving first-seen order."""
    seen: set[Any] = set()
    for item in iterable:
        marker = item if key is None else key(item)
        if marker not in seen:
            seen.add(marker)
            yield item


def partition(
    predicate: Callable[[Any], bool], iterable: Iterable[Any]
) -> tuple[Iterator[Any], Iterator[Any]]:
    """Split into (items where predicate is false, items where it is true)."""
    for_false, for_true = itertools.tee(iterable)
    return itertools.filterfalse(predicate, for_false), filter(predicate, for_true)

Each recipe on a small input:

In [782]:
print(f"{take(3, itertools.count(1001))=}")
print(f"{nth('ABCDE', 2)=}, {nth('AB', 9, 'none')=}")
print(f"{all_equal([5, 5, 5])=}, {all_equal('aab')=}")
print(f"{list(unique_everseen(['Tea', 'rice', 'TEA', 'Rice'], key=str.lower))=}")
cheap, expensive = partition(lambda price: price > 500, [249, 640, 162, 980])
print(f"{list(cheap)=}, {list(expensive)=}")

take(3, itertools.count(1001))=[1001, 1002, 1003]
nth('ABCDE', 2)='C', nth('AB', 9, 'none')='none'
all_equal([5, 5, 5])=True, all_equal('aab')=False
list(unique_everseen(['Tea', 'rice', 'TEA', 'Rice'], key=str.lower))=['Tea', 'rice']
list(cheap)=[249, 162], list(expensive)=[640, 980]


The third-party [`more-itertools`](https://more-itertools.readthedocs.io/) package
implements all the recipes and many more (`chunked`, `windowed`, `peekable`, `first`,
`one`, `spy`, …). It is installed in Phase 4 (`uv add more-itertools`), so the next
cell checks whether it is available and skips cleanly if not.

In [783]:
if importlib.util.find_spec("more_itertools") is not None:
    import more_itertools

    print(f"{more_itertools.first([249, 640])=}")
    print(f"{list(more_itertools.chunked(range(7), 3))=}")
else:
    print("more-itertools is not installed (it is added in Phase 4); skipped")

more-itertools is not installed (it is added in Phase 4); skipped


### Lab 1.13

Seven exercises. For each, replace the `raise NotImplementedError` in the stub, then
run the check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — re-iterable `Countdown`

Unit 1.9.1 wrote `Countdown` with a separate `CountdownIterator` class. Rewrite it
with **no iterator class**: make `__iter__` a generator method that yields
`start, start - 1, …, 1`. Every loop must get a fresh generator, so the object can be
looped over many times (and nested). `Countdown(0)` yields nothing;
`Countdown(-1)` raises `ValueError` in `__init__`.

In [784]:
class Countdown:
    """Re-iterable countdown from start to 1."""

    def __init__(self, start: int) -> None:
        raise NotImplementedError("Lab 1.13: implement Countdown")

    def __iter__(self) -> Iterator[int]:
        raise NotImplementedError("Lab 1.13: implement Countdown.__iter__")

Run the checks.

In [785]:
run_checks(
    [
        ("counts down", lambda: list(Countdown(3)), [3, 2, 1]),
        (
            "re-iterable",
            lambda: (lambda launch: (list(launch), list(launch)))(Countdown(2)),
            ([2, 1], [2, 1]),
        ),
        (
            "nested loops",
            lambda: len([(a, b) for launch in [Countdown(3)] for a in launch for b in launch]),
            9,
        ),
        ("zero", lambda: list(Countdown(0)), []),
        ("negative", lambda: raised_error_name(lambda: Countdown(-1)), "ValueError"),
        ("__iter__ is a generator", lambda: inspect.isgenerator(iter(Countdown(3))), True),
    ]
)

FAIL  counts down: raised NotImplementedError: Lab 1.13: implement Countdown
FAIL  re-iterable: raised NotImplementedError: Lab 1.13: implement Countdown
FAIL  nested loops: raised NotImplementedError: Lab 1.13: implement Countdown
FAIL  zero: raised NotImplementedError: Lab 1.13: implement Countdown
FAIL  negative: expected 'ValueError', got 'NotImplementedError'
FAIL  __iter__ is a generator: raised NotImplementedError: Lab 1.13: implement Countdown
0/6 checks passed


#### Exercise 2 — `read_log_records(path)`

A **generator function** that opens the log file (UTF-8, with `with`) and yields one
dict per record line, `{"date", "time", "level", "message"}`, using
`line.split(maxsplit=3)` (the format of `parse_log_line` from Unit 1.4.7). Strip each
line; skip blank lines, lines starting with `#`, and lines with fewer than four
fields. It must be lazy: the file is opened only when iteration starts, and read one
line at a time.

In [786]:
def read_log_records(path: Path) -> Iterator[dict[str, str]]:
    """Yield {"date", "time", "level", "message"} for each record line of the file."""
    raise NotImplementedError("Lab 1.13: implement read_log_records()")

Run the checks. The cell writes a six-line sample log (with a comment, a blank line and
a malformed line) to `scratch_dir`.

In [787]:
SAMPLE_LOG_TEXT = (
    "# service log\n"
    "2026-11-05 09:30:01 INFO user asha logged in\n"
    "\n"
    "2026-11-05 09:30:05 ERROR payment failed: card declined\n"
    "garbage line\n"
    "2026-11-05 09:31:00 WARNING disk 91% full\n"
)
sample_log_path = scratch_dir / "unit_1_13_sample.log"
sample_log_path.write_text(SAMPLE_LOG_TEXT, encoding="utf-8")
missing_log_path = scratch_dir / "unit_1_13_missing.log"
run_checks(
    [
        ("record count", lambda: len(list(read_log_records(sample_log_path))), 3),
        (
            "first record",
            lambda: next(read_log_records(sample_log_path)),
            {
                "date": "2026-11-05",
                "time": "09:30:01",
                "level": "INFO",
                "message": "user asha logged in",
            },
        ),
        (
            "levels",
            lambda: [r["level"] for r in read_log_records(sample_log_path)],
            ["INFO", "ERROR", "WARNING"],
        ),
        ("is a generator", lambda: inspect.isgenerator(read_log_records(sample_log_path)), True),
        (
            "lazy open",
            lambda: raised_error_name(lambda: read_log_records(missing_log_path)),
            "no error",
        ),
        (
            "error on first next",
            lambda: raised_error_name(lambda: next(read_log_records(missing_log_path))),
            "FileNotFoundError",
        ),
    ]
)

FAIL  record count: raised NotImplementedError: Lab 1.13: implement read_log_records()


FAIL  first record: raised NotImplementedError: Lab 1.13: implement read_log_records()
FAIL  levels: raised NotImplementedError: Lab 1.13: implement read_log_records()
FAIL  is a generator: raised NotImplementedError: Lab 1.13: implement read_log_records()
FAIL  lazy open: expected 'no error', got 'NotImplementedError'
FAIL  error on first next: expected 'FileNotFoundError', got 'NotImplementedError'
0/6 checks passed


#### Exercise 3 — `flatten(nested)`

A recursive generator (use `yield from`) that yields the leaves of arbitrarily nested
**lists and tuples**, left to right. Everything else, including strings and dicts, is
a leaf and is yielded as is.

In [788]:
def flatten(nested: Iterable[Any]) -> Iterator[Any]:
    """Yield the leaves of nested lists and tuples."""
    raise NotImplementedError("Lab 1.13: implement flatten()")

Run the checks.

In [789]:
run_checks(
    [
        ("deep", lambda: list(flatten([1, [2, [3, [4, (5, 6)]]], 7])), [1, 2, 3, 4, 5, 6, 7]),
        (
            "strings are leaves",
            lambda: list(flatten(["tea", ["rice", ("soap",)]])),
            ["tea", "rice", "soap"],
        ),
        ("dicts are leaves", lambda: list(flatten([{"a": 1}, [[]]])), [{"a": 1}]),
        ("empty", lambda: list(flatten([[], [[]]])), []),
        ("tuple at the top", lambda: list(flatten((1, (2, [3])))), [1, 2, 3]),
        ("is a generator", lambda: inspect.isgenerator(flatten([])), True),
    ]
)

FAIL  deep: raised NotImplementedError: Lab 1.13: implement flatten()
FAIL  strings are leaves: raised NotImplementedError: Lab 1.13: implement flatten()
FAIL  dicts are leaves: raised NotImplementedError: Lab 1.13: implement flatten()
FAIL  empty: raised NotImplementedError: Lab 1.13: implement flatten()
FAIL  tuple at the top: raised NotImplementedError: Lab 1.13: implement flatten()
FAIL  is a generator: raised NotImplementedError: Lab 1.13: implement flatten()
0/6 checks passed


#### Exercise 4 — `running_average()` coroutine

A generator used with `send`: after priming with `next()` (which yields `None`), each
`send(number)` yields the average of **all numbers sent so far**, as a float. Each
call to `running_average()` starts fresh.

In [790]:
def running_average() -> Generator[float | None, float]:
    """Receive numbers with send(); yield the running average."""
    raise NotImplementedError("Lab 1.13: implement running_average()")

Run the checks.

In [791]:
def averages_for(numbers: list[float]) -> tuple[float | None, list[float | None]]:
    """Prime a running_average and send numbers; return (priming value, averages)."""
    averager = running_average()
    priming_value = next(averager)
    return priming_value, [averager.send(number) for number in numbers]


run_checks(
    [
        ("averages", lambda: averages_for([10, 20, 60]), (None, [10.0, 15.0, 30.0])),
        ("floats", lambda: averages_for([1, 2])[1], [1.0, 1.5]),
        ("independent", lambda: (averages_for([100])[1], averages_for([2])[1]), ([100.0], [2.0])),
        ("is a generator", lambda: inspect.isgenerator(running_average()), True),
    ]
)

FAIL  averages: raised NotImplementedError: Lab 1.13: implement running_average()
FAIL  floats: raised NotImplementedError: Lab 1.13: implement running_average()
FAIL  independent: raised NotImplementedError: Lab 1.13: implement running_average()
FAIL  is a generator: raised NotImplementedError: Lab 1.13: implement running_average()
0/4 checks passed


#### Exercise 5 — `paginate(fetch_page)`

`fetch_page(page_number)` returns a list of items (pages are numbered from 1) and an
**empty list** after the last page, like a paginated API. Write a generator that
yields every item of every page, one at a time, and fetches a page **only when the
consumer needs its first item**.

In [792]:
def paginate(fetch_page: Callable[[int], list[Any]]) -> Iterator[Any]:
    """Yield items from pages 1, 2, … until fetch_page returns an empty list."""
    raise NotImplementedError("Lab 1.13: implement paginate()")

Run the checks. `make_fake_api(items, page_size)` returns a `fetch_page` function that
records every page number it is asked for.

In [793]:
def make_fake_api(items: list[Any], page_size: int) -> Callable[[int], list[Any]]:
    """Return fetch_page(page_number) serving items in pages; records requested pages."""
    requested_pages: list[int] = []

    def fetch_page(page_number: int) -> list[Any]:
        requested_pages.append(page_number)
        start = (page_number - 1) * page_size
        return items[start : start + page_size]

    fetch_page.requested_pages = requested_pages
    return fetch_page


def lazy_fetch_trace() -> tuple[list[str], list[int]]:
    """Take 3 items with page size 2: only pages 1 and 2 may be fetched."""
    fetch_page = make_fake_api(["A-1", "A-2", "A-3", "A-4", "A-5"], page_size=2)
    first_three = list(itertools.islice(paginate(fetch_page), 3))
    return first_three, fetch_page.requested_pages


run_checks(
    [
        (
            "all items",
            lambda: list(paginate(make_fake_api(list(range(7)), 3))),
            [0, 1, 2, 3, 4, 5, 6],
        ),
        ("fetches lazily", lazy_fetch_trace, (["A-1", "A-2", "A-3"], [1, 2])),
        ("no items", lambda: list(paginate(make_fake_api([], 3))), []),
        (
            "nothing fetched before iteration",
            lambda: (lambda api: (paginate(api), api.requested_pages)[1])(make_fake_api([1], 1)),
            [],
        ),
    ]
)

FAIL  all items: raised NotImplementedError: Lab 1.13: implement paginate()
FAIL  fetches lazily: raised NotImplementedError: Lab 1.13: implement paginate()
FAIL  no items: raised NotImplementedError: Lab 1.13: implement paginate()
FAIL  nothing fetched before iteration: raised NotImplementedError: Lab 1.13: implement paginate()
0/4 checks passed


#### Exercise 6 — `sliding_windows(iterable, size)`

Yield tuples of `size` consecutive items: `sliding_windows("ABCD", 3)` →
`("A", "B", "C")`, `("B", "C", "D")`. Fewer than `size` items → nothing. It must work on
infinite iterators and keep at most `size` items in memory (hint: `deque(maxlen=size)`).
`size < 1` raises `ValueError` (when iteration starts is fine).

In [794]:
def sliding_windows(iterable: Iterable[Any], size: int) -> Iterator[tuple[Any, ...]]:
    """Yield overlapping windows of size consecutive items."""
    raise NotImplementedError("Lab 1.13: implement sliding_windows()")

Run the checks.

In [795]:
run_checks(
    [
        (
            "windows of 3",
            lambda: list(sliding_windows("ABCD", 3)),
            [("A", "B", "C"), ("B", "C", "D")],
        ),
        ("size 1", lambda: list(sliding_windows([7, 8], 1)), [(7,), (8,)]),
        ("too few items", lambda: list(sliding_windows([1, 2], 3)), []),
        (
            "infinite input",
            lambda: list(itertools.islice(sliding_windows(itertools.count(), 2), 3)),
            [(0, 1), (1, 2), (2, 3)],
        ),
        (
            "moving average",
            lambda: [sum(w) / 2 for w in sliding_windows([10, 20, 40], 2)],
            [15.0, 30.0],
        ),
        ("size 0", lambda: raised_error_name(lambda: list(sliding_windows([1], 0))), "ValueError"),
    ]
)

FAIL  windows of 3: raised NotImplementedError: Lab 1.13: implement sliding_windows()
FAIL  size 1: raised NotImplementedError: Lab 1.13: implement sliding_windows()
FAIL  too few items: raised NotImplementedError: Lab 1.13: implement sliding_windows()
FAIL  infinite input: raised NotImplementedError: Lab 1.13: implement sliding_windows()
FAIL  moving average: raised NotImplementedError: Lab 1.13: implement sliding_windows()
FAIL  size 0: expected 'ValueError', got 'NotImplementedError'
0/6 checks passed


#### Exercise 7 — streaming analysis of a 1M-line log

Write two functions:

- `summarize_log(records)`: one pass over **any iterable** of record dicts (as yielded
  by `read_log_records`), returning
  `{"total": <record count>, "by_level": {level: count, …} sorted by level name,
  "first_error": <message of the first ERROR record, or None>}`. Do not build a list
  of the records.
- `analyze_log_file(path)`: `summarize_log(read_log_records(path))`.

The next cell provides `write_big_log(path, line_count)` (deterministic data),
`analyze_log_file_eager(path)` (the same summary computed by loading the whole file
into memory first, the pattern to avoid) and `peak_memory(function, *args)` (result
and peak bytes, measured with `tracemalloc`). The check writes a 1 000 000-line log
(about 45 MB), compares both versions, prints their peak memory, and deletes the
file. It takes several seconds once your solution works.

In [796]:
LEVEL_PATTERN = (
    "INFO",
    "INFO",
    "DEBUG",
    "INFO",
    "WARNING",
    "INFO",
    "DEBUG",
    "INFO",
    "INFO",
    "ERROR",
)


def write_big_log(path: Path, line_count: int) -> None:
    """Write a comment, a blank line and line_count deterministic records."""
    with path.open("w", encoding="utf-8") as log_file:
        log_file.write("# synthetic service log\n\n")
        log_file.writelines(
            f"2026-11-05 {index // 3600 % 24:02d}:{index // 60 % 60:02d}:{index % 60:02d} "
            f"{LEVEL_PATTERN[index % 10]} request {index} handled\n"
            for index in range(line_count)
        )


def analyze_log_file_eager(path: Path) -> dict[str, Any]:
    """Reference summary that reads the whole file into memory first."""
    all_lines = path.read_text(encoding="utf-8").splitlines()
    level_counts: Counter[str] = Counter()
    first_error = None
    for line in all_lines:
        fields = line.strip().split(maxsplit=3)
        if len(fields) < 4 or fields[0].startswith("#"):
            continue
        level_counts[fields[2]] += 1
        if fields[2] == "ERROR" and first_error is None:
            first_error = fields[3]
    return {
        "total": level_counts.total(),
        "by_level": dict(sorted(level_counts.items())),
        "first_error": first_error,
    }


def peak_memory(function: Callable[..., Any], *args: Any) -> tuple[Any, int]:
    """Return (function(*args), peak traced bytes during the call)."""
    tracemalloc.start()
    try:
        result = function(*args)
        return result, tracemalloc.get_traced_memory()[1]
    finally:
        tracemalloc.stop()

Write your two functions here.

In [797]:
def summarize_log(records: Iterable[dict[str, str]]) -> dict[str, Any]:
    """Return {"total", "by_level", "first_error"} in a single pass over records."""
    raise NotImplementedError("Lab 1.13: implement summarize_log()")


def analyze_log_file(path: Path) -> dict[str, Any]:
    """Stream the file through read_log_records into summarize_log."""
    raise NotImplementedError("Lab 1.13: implement analyze_log_file()")

Run the checks.

In [798]:
def streaming_vs_eager() -> tuple[Any, ...]:
    """Analyze a 1M-line log both ways; return the comparison."""
    big_log_path = scratch_dir / "unit_1_13_big.log"
    try:
        write_big_log(big_log_path, 1_000_000)
        streamed, streaming_peak = peak_memory(analyze_log_file, big_log_path)
        eager, eager_peak = peak_memory(analyze_log_file_eager, big_log_path)
        print(f"  peak memory: streaming {streaming_peak / 1e3:,.0f} kB", end=", ")
        print(f"eager {eager_peak / 1e6:,.0f} MB")
        return (
            streamed == eager,
            streamed["total"],
            streamed["by_level"],
            streamed["first_error"],
            streaming_peak * 20 < eager_peak,
        )
    finally:
        big_log_path.unlink(missing_ok=True)


small_records = [
    {"date": "d", "time": "t", "level": "WARNING", "message": "slow"},
    {"date": "d", "time": "t", "level": "ERROR", "message": "first failure"},
    {"date": "d", "time": "t", "level": "ERROR", "message": "second failure"},
]
run_checks(
    [
        (
            "small input",
            lambda: summarize_log(small_records),
            {"total": 3, "by_level": {"ERROR": 2, "WARNING": 1}, "first_error": "first failure"},
        ),
        ("accepts an iterator", lambda: summarize_log(iter(small_records))["total"], 3),
        (
            "no records",
            lambda: summarize_log([]),
            {"total": 0, "by_level": {}, "first_error": None},
        ),
        (
            "1M lines: same result, 20x less memory",
            streaming_vs_eager,
            (
                True,
                1_000_000,
                {"DEBUG": 200_000, "ERROR": 100_000, "INFO": 600_000, "WARNING": 100_000},
                "request 9 handled",
                True,
            ),
        ),
    ]
)

FAIL  small input: raised NotImplementedError: Lab 1.13: implement summarize_log()
FAIL  accepts an iterator: raised NotImplementedError: Lab 1.13: implement summarize_log()
FAIL  no records: raised NotImplementedError: Lab 1.13: implement summarize_log()


FAIL  1M lines: same result, 20x less memory: raised NotImplementedError: Lab 1.13: implement analyze_log_file()
0/4 checks passed


### Recap — Unit 1.13

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Iterable vs iterator | Iterables give fresh iterators; iterators are single use (`iter(it) is it`) | `Symbol.iterator` / iterator objects |
| Custom iterators | Keep the position in a separate iterator object; `__getitem__` is the legacy fallback | `[Symbol.iterator]()` returning a new object |
| Generator functions | `yield` makes a lazy iterator; states via `getgeneratorstate`; frame in `gi_frame` | `function*` |
| Generator `return` | The value rides on `StopIteration.value` | `{value, done: true}` |
| Generator expressions | `(x for x in data)`: single use, constant memory; first iterable evaluated eagerly | iterator helpers |
| Lazy pipelines | Chain generators; pull-based, one item at a time, stops early | streams |
| `yield from` | Delegates iteration, recursion made easy, returns the sub-generator's value | `yield*` |
| Two-way generators | `send` (prime first), `throw`, `close` → `GeneratorExit`; clean up in `finally` | `next(v)`, `throw`, `return` |
| Generators → async | PEP 255 → 342 → 380 → 492: coroutines are driven with `send`, like generators | `async` / `await` |
| `itertools` | Infinite, terminating and combinatoric building blocks; `groupby` needs sorted input | — |
| Recipes / `more-itertools` | Copy recipes from the docs, or install `more-itertools` (Phase 4) | lodash |

**Further reading**

- [Glossary: iterable](https://docs.python.org/3/glossary.html#term-iterable),
  [iterator](https://docs.python.org/3/glossary.html#term-iterator),
  [generator](https://docs.python.org/3/glossary.html#term-generator)
- [Iterator types](https://docs.python.org/3/library/stdtypes.html#iterator-types),
  [generator expressions](https://docs.python.org/3/reference/expressions.html#generator-expressions),
  [yield expressions](https://docs.python.org/3/reference/expressions.html#yield-expressions)
- [Functional programming HOWTO](https://docs.python.org/3/howto/functional.html),
  [`itertools`](https://docs.python.org/3/library/itertools.html) and its
  [recipes](https://docs.python.org/3/library/itertools.html#itertools-recipes),
  [`more-itertools`](https://more-itertools.readthedocs.io/)
- [`inspect.getgeneratorstate`](https://docs.python.org/3/library/inspect.html#inspect.getgeneratorstate),
  [coroutine objects](https://docs.python.org/3/reference/datamodel.html#coroutine-objects)
- [PEP 234 — Iterators](https://peps.python.org/pep-0234/),
  [PEP 255 — Simple generators](https://peps.python.org/pep-0255/),
  [PEP 289 — Generator expressions](https://peps.python.org/pep-0289/)
- [PEP 342 — Coroutines via enhanced generators](https://peps.python.org/pep-0342/),
  [PEP 380 — `yield from`](https://peps.python.org/pep-0380/),
  [PEP 492 — `async` and `await`](https://peps.python.org/pep-0492/),
  [PEP 525 — Asynchronous generators](https://peps.python.org/pep-0525/)

## Unit 1.14 — Exceptions and Context Managers

**Objectives**

- Read the exception hierarchy and predict which `except` clause catches what.
- Trace every path through `try` / `except` / `else` / `finally`, including the 3.14
  syntax and warning changes.
- Raise, re-raise and chain exceptions so tracebacks tell the whole story.
- Design a custom exception hierarchy that maps to HTTP status codes.
- Raise and handle several errors at once with exception groups and `except*`.
- Choose between EAFP and LBYL with measurements, and log exceptions correctly.
- Write context managers as classes and with `contextlib`, and see both in bytecode.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `throw value` | `raise ExceptionInstance` | Only `BaseException` subclasses can be raised; `throw "text"` has no equivalent |
| `catch (e)` (catches everything) | `except SomeError as error:` | Python filters by class; a bare `except:` also catches Ctrl+C and `sys.exit` |
| — | `else:` clause | Runs only when the `try` body raised nothing |
| `finally` | `finally` | Same, and a `return` in `finally` now warns (3.14) |
| `new Error(msg, { cause })` | `raise New(...) from original` | Python also records the *implicit* context automatically |
| `AggregateError` | `ExceptionGroup` + `except*` | `except*` handles each matching subgroup, re-raises the rest |
| `try { } finally { res.close() }` / `using` (TC39) | `with resource:` | The manager's `__exit__` sees the exception and may suppress it |
| `console.error(err)` | `logger.exception("message")` | Logs the message **and** the traceback, at ERROR level |

The next cell is this unit's setup (`scratch_dir` is the git-ignored folder used by
file demos).

In [799]:
import builtins
import dis
import functools
import inspect
import io
import json  # noqa: F401 (Lab 1.14)
import logging
import os
import sqlite3
import subprocess
import sys
import time
import timeit
import traceback
import warnings
from collections.abc import Callable, Iterable, Iterator
from contextlib import (
    AbstractContextManager,
    ExitStack,
    chdir,
    closing,
    contextmanager,
    nullcontext,
    redirect_stdout,
    suppress,
)
from pathlib import Path
from types import TracebackType
from typing import IO, Any


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def compile_error(source: str) -> str:
    """Compile source without running it; return "SyntaxError: <message>" or "compiles"."""
    try:
        compile(source, "<demo>", "exec")
    except SyntaxError as error:
        return f"SyntaxError: {error.msg}"
    return "compiles"


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


scratch_dir = find_repo_root() / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.14.1 The exception hierarchy

Every exception is an instance of a class that derives from **`BaseException`**.
Directly under it sit a few classes that are *not errors* but control signals, and
**`Exception`**, the base of everything you normally handle:

| Class | Raised by | Should `except Exception` catch it? |
| --- | --- | --- |
| `SystemExit` | `sys.exit()` | No: the program asked to stop |
| `KeyboardInterrupt` | Ctrl+C | No: the user asked to stop |
| `GeneratorExit` | `generator.close()` (Unit 1.13.7) | No: the generator is being shut down |
| `BaseExceptionGroup` | several errors at once (1.14.6) | Only its `ExceptionGroup` subclass |
| `Exception` | every ordinary error | Yes |

So `except Exception:` is the widest net that still lets the program be stopped.
Custom exceptions should derive from `Exception` (or a subclass), never directly from
`BaseException`.

```mermaid
graph TD
    BaseException --> SystemExit
    BaseException --> KeyboardInterrupt
    BaseException --> GeneratorExit
    BaseException --> BaseExceptionGroup
    BaseException --> Exception
    BaseExceptionGroup --> ExceptionGroup
    Exception --> ExceptionGroup
    Exception --> ArithmeticError --> ZeroDivisionError
    Exception --> LookupError --> KeyError
    LookupError --> IndexError
    Exception --> OSError --> FileNotFoundError
    OSError --> PermissionError
    OSError --> ConnectionError
    OSError --> TimeoutError
    Exception --> ValueError --> UnicodeError
    Exception --> TypeError
    Exception --> AttributeError
    Exception --> NameError --> UnboundLocalError
    Exception --> RuntimeError --> RecursionError
    RuntimeError --> NotImplementedError
    Exception --> StopIteration
    Exception --> ImportError --> ModuleNotFoundError
    Exception --> Warning --> DeprecationWarning
```

<details><summary>ASCII fallback</summary>

```text
BaseException
├── SystemExit
├── KeyboardInterrupt
├── GeneratorExit
├── BaseExceptionGroup
│   └── ExceptionGroup        (also an Exception)
└── Exception
    ├── ArithmeticError ── ZeroDivisionError, OverflowError
    ├── LookupError ────── KeyError, IndexError
    ├── OSError ────────── FileNotFoundError, PermissionError,
    │                      ConnectionError, TimeoutError, …
    ├── ValueError ─────── UnicodeError (UnicodeDecodeError, …)
    ├── TypeError, AttributeError
    ├── NameError ──────── UnboundLocalError
    ├── RuntimeError ───── RecursionError, NotImplementedError
    ├── StopIteration, ImportError ── ModuleNotFoundError
    └── Warning ────────── DeprecationWarning, UserWarning, …
```

</details>

The cell below shows the practical consequence: `except Exception` lets `sys.exit()`
and Ctrl+C through.

In [800]:
def catches_with_exception_clause(error: BaseException) -> bool:
    """Raise error inside try / except Exception; report whether the clause caught it."""
    try:
        raise error
    except Exception:
        return True
    except BaseException:
        return False


for raised in (ValueError("bad"), KeyError("sku"), SystemExit(1), KeyboardInterrupt()):
    print(
        f"{type(raised).__name__:<18} caught by except Exception: "
        f"{catches_with_exception_clause(raised)}"
    )

ValueError         caught by except Exception: True
KeyError           caught by except Exception: True
SystemExit         caught by except Exception: False
KeyboardInterrupt  caught by except Exception: False


#### Under the hood: `__mro__` and what `except` matches

`except SomeClass:` matches when `isinstance(error, SomeClass)` is true, that is when
`SomeClass` appears anywhere in the error's **method resolution order**
(`type(error).__mro__`). So `except OSError:` catches `FileNotFoundError`, and
`except LookupError:` catches both `KeyError` and `IndexError`.

In [801]:
print(" -> ".join(cls.__name__ for cls in FileNotFoundError.__mro__))
print(" -> ".join(cls.__name__ for cls in KeyError.__mro__))
print(" -> ".join(cls.__name__ for cls in ExceptionGroup.__mro__))

try:
    Path("no-such-file.txt").read_text(encoding="utf-8")
except OSError as error:
    print(f"except OSError caught {type(error).__name__}: errno={error.errno}")

FileNotFoundError -> OSError -> Exception -> BaseException -> object
KeyError -> LookupError -> Exception -> BaseException -> object
ExceptionGroup -> BaseExceptionGroup -> Exception -> BaseException -> object
except OSError caught FileNotFoundError: errno=2


The tree in the diagram is a selection. The next cell builds the real tree of
built-in exceptions from `builtins`, two levels below `Exception`, using
`__subclasses__()`. The exact list grows slowly between versions.

In [802]:
BUILTIN_EXCEPTIONS = {
    value
    for value in vars(builtins).values()
    if isinstance(value, type) and issubclass(value, BaseException)
}


def exception_tree(root: type[BaseException], depth: int, max_depth: int) -> Iterator[str]:
    """Yield indented names of the built-in subclasses of root, depth first."""
    yield "    " * depth + root.__name__
    if depth == max_depth:
        return
    children = [cls for cls in root.__subclasses__() if cls in BUILTIN_EXCEPTIONS]
    for child in sorted(children, key=lambda cls: cls.__name__):
        yield from exception_tree(child, depth + 1, max_depth)


print(f"{len(BUILTIN_EXCEPTIONS)} built-in exception classes (warnings included)")
top_level = list(exception_tree(BaseException, 0, 1))
print("Under BaseException:", ", ".join(name.strip() for name in top_level[1:]))
print("\n".join(line for line in exception_tree(Exception, 0, 1)))

70 built-in exception classes (warnings included)
Under BaseException: BaseExceptionGroup, Exception, GeneratorExit, KeyboardInterrupt, SystemExit
Exception
    ArithmeticError
    AssertionError
    AttributeError
    BufferError
    EOFError
    ExceptionGroup
    ImportError
    LookupError
    MemoryError
    NameError
    OSError
    ReferenceError
    RuntimeError
    StopAsyncIteration
    StopIteration
    SyntaxError
    SystemError
    TypeError
    ValueError
    Warning


### 1.14.2 `try` / `except` / `else` / `finally`

The full statement has four parts. `else` runs only if the `try` body raised nothing;
`finally` runs **always**, on every way out.

| Path | Runs, in order | Afterwards |
| --- | --- | --- |
| No exception | `try` → `else` → `finally` | continues after the statement |
| Exception matched by an `except` | `try` (up to the error) → that `except` → `finally` | continues after the statement |
| Exception not matched | `try` (up to the error) → `finally` | the exception propagates |
| Exception raised inside `except` | `try` → `except` (up to the new error) → `finally` | the **new** exception propagates; `__context__` is the old one |
| Exception raised inside `else` | `try` → `else` (up to the error) → `finally` | propagates: this statement's `except` clauses do **not** see it |
| `return` in `try` | `try` (return value evaluated) → `finally` | the function returns that value |
| `break` / `continue` in `try` | `try` → `finally` | the loop jump happens |

Why `else`? Code placed there is not protected by the `except` clauses, so you do not
accidentally catch errors from code you did not mean to guard. The next cell walks
the first five paths and records each step.

In [803]:
def checkout_steps(scenario: str, steps: list[str]) -> None:
    """Append the name of each block that runs for the given scenario."""
    try:
        steps.append("try")
        if scenario in ("error handled", "error in except"):
            raise ValueError("bad quantity")
        if scenario == "error not matched":
            raise KeyError("sku")
    except ValueError:
        steps.append("except")
        if scenario == "error in except":
            raise RuntimeError("handler failed")  # noqa: B904 (implicit chaining demo)
    else:
        steps.append("else")
        if scenario == "error in else":
            raise TypeError("else failed")
    finally:
        steps.append("finally")
    steps.append("after")


for scenario in (
    "no error",
    "error handled",
    "error not matched",
    "error in except",
    "error in else",
):
    recorded_steps: list[str] = []
    try:
        checkout_steps(scenario, recorded_steps)
    except Exception as error:
        recorded_steps.append(f"propagated {type(error).__name__}")
    print(f"{scenario:<18} {' -> '.join(recorded_steps)}")

no error           try -> else -> finally -> after
error handled      try -> except -> finally -> after
error not matched  try -> finally -> propagated KeyError
error in except    try -> except -> finally -> propagated RuntimeError
error in else      try -> else -> finally -> propagated TypeError


The last two rows of the table: `finally` runs even when `try` leaves through
`return` or `break`. The return value is computed first, then `finally` runs, then
the function actually returns.

In [804]:
def price_with_audit(audit_log: list[str]) -> int:
    """Return from inside try; finally still runs before the caller gets the value."""
    try:
        audit_log.append("computing price")
        return 499
    finally:
        audit_log.append("finally after return")


audit_log: list[str] = []
print(f"{price_with_audit(audit_log)=}, {audit_log=}")

loop_log: list[str] = []
for attempt in range(3):
    try:
        if attempt == 1:
            break
    finally:
        loop_log.append(f"finally for attempt {attempt}")
print(f"{loop_log=}")

price_with_audit(audit_log)=499, audit_log=['computing price', 'finally after return']
loop_log=['finally for attempt 0', 'finally for attempt 1']


**Multiple clauses.** Clauses are tested top to bottom and the **first match wins**;
the rest are skipped. Put the most specific classes first. A broad clause placed
early silently shadows the specific ones below it (Python does not warn).

In [805]:
def classify_failure(thunk: Callable[[], object]) -> str:
    """Specific clauses first, broad ones last."""
    try:
        thunk()
    except KeyError:
        return "KeyError clause"
    except LookupError:
        return "LookupError clause"
    except Exception as error:
        return f"Exception clause ({type(error).__name__})"
    return "no error"


def classify_failure_wrong_order(thunk: Callable[[], object]) -> str:
    """The broad clause comes first, so the KeyError clause can never run."""
    try:
        thunk()
    except LookupError:
        return "LookupError clause"
    except KeyError:
        return "KeyError clause (unreachable)"
    return "no error"


stock = {"tea": 4}
print(f"{classify_failure(lambda: stock['coffee'])=}")
print(f"{classify_failure(lambda: [][0])=}")
print(f"{classify_failure(lambda: int('four'))=}")
print(f"{classify_failure_wrong_order(lambda: stock['coffee'])=}")

classify_failure(lambda: stock['coffee'])='KeyError clause'
classify_failure(lambda: [][0])='LookupError clause'
classify_failure(lambda: int('four'))='Exception clause (ValueError)'
classify_failure_wrong_order(lambda: stock['coffee'])='LookupError clause'


**Tuples.** One clause can handle several classes with a parenthesized tuple; with
`as error` you then branch on the type if needed.

In [806]:
def first_item_name(order: dict[str, list[str]]) -> str:
    """Handle a missing key and an empty list in one clause."""
    try:
        return order["items"][0]
    except (KeyError, IndexError) as error:
        return f"no items ({type(error).__name__})"


print(first_item_name({"items": ["rice"]}))
print(first_item_name({}))
print(first_item_name({"items": []}))

rice
no items (KeyError)
no items (IndexError)


**Unparenthesized `except A, B:`** (*New in 3.14*, PEP 758). The parentheses may now
be dropped, **but only without `as`**. In Python 2, `except A, B:` meant "catch `A`
and bind it to the name `B`", which is why it was a syntax error until 3.14.

```python
except KeyError, IndexError:          # 3.14+: fine
except (KeyError, IndexError) as e:   # needs parentheses with `as`
```

The cell runs the new form through `exec` inside a version guard, so older kernels can
still parse the cell.

In [807]:
NEW_EXCEPT_SOURCE = """
try:
    {}["missing"]
except KeyError, IndexError:
    print("3.14 syntax: caught without parentheses")
"""

if sys.version_info >= (3, 14):
    exec(inspect.cleandoc(NEW_EXCEPT_SOURCE))
else:
    print("Skipped: unparenthesized except needs Python 3.14+")

print(compile_error("try:\n    pass\nexcept KeyError, IndexError as error:\n    pass"))

3.14 syntax: caught without parentheses
SyntaxError: multiple exception types must be parenthesized when using 'as'


**`as error` is deleted at the end of the clause.** Python runs `del error` when the
`except` block ends, because the exception holds its traceback, the traceback holds
the frame, and the frame holds `error`: a reference cycle that would keep every local
variable alive. To keep the exception, bind it to another name inside the block.

In [808]:
try:
    int("twelve")
except ValueError as conversion_error:
    kept_error = conversion_error
    print(f"inside:  {'conversion_error' in globals()=}")
print(f"after:   {'conversion_error' in globals()=}")
print(f"kept:    {kept_error!r}")

inside:  'conversion_error' in globals()=True
after:   'conversion_error' in globals()=False
kept:    ValueError("invalid literal for int() with base 10: 'twelve'")


**Bare `except:` dangers.** `except:` with no class is the same as
`except BaseException:`. It catches `SystemExit` and `KeyboardInterrupt`, so a
program cannot be stopped, and it hides typos (`NameError`) as if they were expected
failures. If you truly must catch everything (a top-level crash reporter), catch
`BaseException` explicitly and **re-raise**.

In [809]:
def shutdown_service() -> str:
    """Ask the interpreter to exit, but a bare except swallows the request."""
    try:
        sys.exit(0)
    except:  # noqa: E722 (the trap being demonstrated)
        return "sys.exit() was swallowed; the program keeps running"


def report_crash_and_reraise(thunk: Callable[[], object]) -> str:
    """The acceptable pattern: catch everything, record it, re-raise."""
    try:
        thunk()
    except BaseException as error:
        print(f"crash reporter saw {type(error).__name__}")
        raise
    return "no error"


print(shutdown_service())
try:
    report_crash_and_reraise(lambda: sys.exit(3))
except SystemExit as exit_request:
    print(f"SystemExit still propagated, code={exit_request.code}")

sys.exit() was swallowed; the program keeps running
crash reporter saw SystemExit
SystemExit still propagated, code=3


**`return` in `finally`** (*Changed in 3.14*, PEP 765). A `return`, `break` or
`continue` inside `finally` silently discards any exception in flight and any earlier
return value. Since 3.14 the compiler emits a `SyntaxWarning`. The cell compiles such
a function from a string (so the warning is captured, not printed by the notebook),
then calls it: the `ZeroDivisionError` vanishes.

In [810]:
SWALLOWING_SOURCE = """
def average_price(total: int, count: int) -> float | str:
    try:
        return total / count
    finally:
        return "finally won"
"""

with warnings.catch_warnings(record=True) as compile_warnings:
    warnings.simplefilter("always")
    swallowing_code = compile(inspect.cleandoc(SWALLOWING_SOURCE), "<demo>", "exec")
for caught_warning in compile_warnings:
    print(f"{caught_warning.category.__name__}: {caught_warning.message}")

demo_namespace: dict[str, Any] = {}
exec(swallowing_code, demo_namespace)
print(f"{demo_namespace['average_price'](100, 0)=}  (ZeroDivisionError lost)")

demo_namespace['average_price'](100, 0)='finally won'  (ZeroDivisionError lost)


#### Predict the output

**Predict the output.**

```python
def apply_coupons() -> list[str]:
    applied = ["SAVE10"]
    try:
        return applied
    finally:
        applied.append("FREESHIP")


print(apply_coupons())
```

<details><summary>Answer</summary>

`['SAVE10', 'FREESHIP']`. The `return` evaluates `applied` (a reference to the list)
before `finally` runs; `finally` then mutates that same list. Returning an `int` the
same way would be unaffected, because rebinding is not mutation (Unit 1.2).

</details>

Run the next cell to check your prediction.

In [811]:
def apply_coupons() -> list[str]:
    """Return a list that finally mutates."""
    applied = ["SAVE10"]
    try:
        return applied
    finally:
        applied.append("FREESHIP")


print(apply_coupons())

['SAVE10', 'FREESHIP']


### 1.14.3 Raising and chaining

| Form | Meaning |
| --- | --- |
| `raise ValueError("msg")` | raise a new instance |
| `raise ValueError` | the class is instantiated with no arguments |
| `raise` (inside `except`) | re-raise the exception being handled, traceback intact |
| `raise New(...) from original` | explicit chaining: sets `__cause__` |
| `raise New(...)` inside `except` | implicit chaining: sets `__context__` automatically |
| `raise New(...) from None` | hide the context (`__suppress_context__ = True`) |

Only instances or subclasses of `BaseException` can be raised; anything else is a
`TypeError`.

In [812]:
def raise_value(value: object) -> None:
    """Raise whatever is passed in."""
    raise value  # type: ignore[misc]


for candidate in (ValueError("negative quantity"), ValueError, "just a string"):
    try:
        raise_value(candidate)
    except Exception as error:
        print(f"{candidate!r:<36} -> {type(error).__name__}: {error}")

ValueError('negative quantity')      -> ValueError: negative quantity
<class 'ValueError'>                 -> ValueError: 
'just a string'                      -> TypeError: exceptions must derive from BaseException


**Bare re-raise.** Inside an `except` block, a plain `raise` re-raises the current
exception with its original traceback, which is how you add a side effect (logging,
a metric, a rollback) without changing what the caller sees. Outside an `except`
block there is nothing to re-raise.

In [813]:
failed_charges: list[str] = []


def charge_card(card_number: str) -> None:
    """Record the failure, then let the original error continue."""
    try:
        int(card_number)
    except ValueError:
        failed_charges.append(card_number)
        raise


print(f"{raised_error_name(lambda: charge_card('4111-x'))=}, {failed_charges=}")


def reraise_with_nothing_active() -> None:
    """A bare raise with no exception being handled."""
    raise


try:
    reraise_with_nothing_active()
except RuntimeError as error:
    print(f"RuntimeError: {error}")

raised_error_name(lambda: charge_card('4111-x'))='ValueError', failed_charges=['4111-x']
RuntimeError: No active exception to reraise


**Explicit chaining with `from`.** When you translate a low-level error into a
domain error, chain it: `raise LookupError(...) from error` stores the original in
`__cause__`, and the traceback shows both, joined by *"The above exception was the
direct cause of the following exception"*.

`chain_lines(error)` keeps only the unindented lines of the formatted traceback (the
exception lines and the joining sentences); the indented `File …` lines contain
machine-specific paths.

In [814]:
def chain_lines(error: BaseException) -> list[str]:
    """Return the unindented lines of the full traceback (no file paths)."""
    formatted = "".join(traceback.format_exception(error))
    return [line for line in formatted.splitlines() if line and not line[0].isspace()]


CATALOG = {"TEA-500G": 249}


def unit_price(sku: str) -> int:
    """Translate KeyError into a domain-level LookupError, keeping the cause."""
    try:
        return CATALOG[sku]
    except KeyError as error:
        raise LookupError(f"unknown SKU {sku!r}") from error


try:
    unit_price("SOAP-3PK")
except LookupError as error:
    print(f"{error.__cause__=}")
    print("\n".join(chain_lines(error)))

error.__cause__=KeyError('SOAP-3PK')
Traceback (most recent call last):
KeyError: 'SOAP-3PK'
The above exception was the direct cause of the following exception:
Traceback (most recent call last):
LookupError: unknown SKU 'SOAP-3PK'


**Implicit chaining.** Raising inside an `except` block without `from` still records
the exception being handled, in `__context__`, and the traceback says *"During
handling of the above exception, another exception occurred"*. This usually signals a
bug in the handler itself, here a typo-like `TypeError`.

In [815]:
def parse_discount(text: str) -> int:
    """The handler itself fails, so the second error carries the first as context."""
    try:
        return int(text)
    except ValueError:
        return len(None)  # type: ignore[arg-type]  (bug in the handler)


try:
    parse_discount("ten")
except TypeError as error:
    print(f"{error.__cause__=}, {error.__context__=}")
    print("\n".join(chain_lines(error)))

error.__cause__=None, error.__context__=ValueError("invalid literal for int() with base 10: 'ten'")
Traceback (most recent call last):
ValueError: invalid literal for int() with base 10: 'ten'
During handling of the above exception, another exception occurred:
Traceback (most recent call last):
TypeError: object of type 'NoneType' has no len()


**`from None`** hides the context when it is noise to the caller (an implementation
detail). Note that `__context__` is still stored; `__suppress_context__ = True` only
tells the traceback printer not to show it.

In [816]:
def environment_name(settings: dict[str, str]) -> str:
    """Raise a clean error; the internal KeyError is hidden from the traceback."""
    try:
        return settings["APP_ENV"]
    except KeyError:
        raise RuntimeError("APP_ENV is not configured") from None


try:
    environment_name({})
except RuntimeError as error:
    print(f"{error.__cause__=}, {error.__context__=}, {error.__suppress_context__=}")
    print("\n".join(chain_lines(error)))

error.__cause__=None, error.__context__=KeyError('APP_ENV'), error.__suppress_context__=True
Traceback (most recent call last):
RuntimeError: APP_ENV is not configured


#### Predict the output

**Predict the output.**

```python
try:
    try:
        raise ValueError("payment declined")
    finally:
        raise RuntimeError("cleanup failed")
except Exception as error:
    print(type(error).__name__, "|", type(error.__context__).__name__)
```

<details><summary>Answer</summary>

`RuntimeError | ValueError`. An exception raised in `finally` replaces the one in
flight; the original survives only as `__context__`. That is why cleanup code should
not raise.

</details>

Run the next cell to check your prediction.

In [817]:
try:
    try:
        raise ValueError("payment declined")
    finally:
        raise RuntimeError("cleanup failed")
except Exception as error:
    print(type(error).__name__, "|", type(error.__context__).__name__)

RuntimeError | ValueError


### 1.14.4 Exception objects

An exception is an ordinary object. The attributes worth knowing:

| Attribute / function | What it holds |
| --- | --- |
| `args` | the tuple of constructor arguments; `str(error)` is `args[0]` if there is one argument, else `str(args)` |
| `__traceback__` | linked list of traceback entries (frame + line), most recent last |
| `__cause__`, `__context__`, `__suppress_context__` | chaining (1.14.3) |
| `add_note(text)` / `__notes__` | extra lines shown after the message (*New in 3.11*, PEP 678) |
| `traceback.format_exception(error)` | the full printed traceback as a list of strings |

**`args`** first. Some built-ins add attributes on top: `OSError` splits its
arguments into `errno`, `strerror` and `filename`. `KeyError` uses the `repr` of the
key as its message, which is why it prints with quotes.

In [818]:
single = ValueError("quantity must be positive")
several = ValueError("quantity must be positive", -3)
print(f"{single.args=}, {str(single)=}")
print(f"{several.args=}, {str(several)=}")
print(f"{str(KeyError('sku'))=}  (repr of the key)")

try:
    open(scratch_dir / "no-such-report.txt", encoding="utf-8")
except OSError as error:
    print(f"{error.errno=}, {error.strerror=}, {error.filename.endswith('report.txt')=}")

single.args=('quantity must be positive',), str(single)='quantity must be positive'
several.args=('quantity must be positive', -3), str(several)="('quantity must be positive', -3)"
str(KeyError('sku'))="'sku'"  (repr of the key)
error.errno=2, error.strerror='No such file or directory', error.filename.endswith('report.txt')=True


**`__traceback__`** is a chain of `traceback` objects linked by `tb_next`. Each one
points at a frame (`tb_frame`) and a line number (`tb_lineno`), which is exactly what
the printed traceback lists. Walking it shows the call path from where the exception
was caught down to where it was raised.

In [819]:
def load_cart(cart_id: int) -> dict[str, int]:
    """Fails two calls deep."""
    return parse_cart_row("rice,two")


def parse_cart_row(row: str) -> dict[str, int]:
    """Raise ValueError on a bad quantity."""
    sku, quantity = row.split(",")
    return {sku: int(quantity)}


try:
    load_cart(7)
except ValueError as error:
    frame_names = []
    traceback_entry: TracebackType | None = error.__traceback__
    while traceback_entry is not None:
        frame_names.append(traceback_entry.tb_frame.f_code.co_name)
        traceback_entry = traceback_entry.tb_next
    print(" -> ".join(frame_names))

<module> -> load_cart -> parse_cart_row


**`add_note`** (*New in 3.11*, PEP 678) attaches context without changing the
exception type or message: which row of a file, which request, which retry attempt.
Notes are stored in `__notes__` and printed under the message.

In [820]:
def import_price_rows(rows: list[str]) -> list[int]:
    """Parse prices; on failure, note which row broke and re-raise."""
    prices = []
    for row_number, row in enumerate(rows, start=1):
        try:
            prices.append(int(row))
        except ValueError as error:
            error.add_note(f"while importing row {row_number}: {row!r}")
            error.add_note("hint: prices are whole rupees")
            raise
    return prices


try:
    import_price_rows(["249", "640", "1,299"])
except ValueError as error:
    print(f"{error.__notes__=}")
    print("\n".join(chain_lines(error)))

error.__notes__=["while importing row 3: '1,299'", 'hint: prices are whole rupees']
Traceback (most recent call last):
ValueError: invalid literal for int() with base 10: '1,299'
while importing row 3: '1,299'
hint: prices are whole rupees


**`traceback.format_exception(error)`** (single-argument form since 3.10) returns the
same text the interpreter would print, as a list of strings ending in newlines. Use it
to put tracebacks in log records or API error reports. `format_exception_only` gives
only the final line(s), message plus notes. The frame lines contain file paths, so
the cell prints only the first line, the line count and the last lines.

In [821]:
try:
    import_price_rows(["10", "ten"])
except ValueError as error:
    full_lines = "".join(traceback.format_exception(error)).splitlines()
    print(f"first line: {full_lines[0]}")
    print(f"{len(full_lines)} lines in total; last three:")
    print("\n".join(full_lines[-3:]))
    print(f"{traceback.format_exception_only(error)=}")

first line: Traceback (most recent call last):
10 lines in total; last three:
ValueError: invalid literal for int() with base 10: 'ten'
while importing row 2: 'ten'
hint: prices are whole rupees
traceback.format_exception_only(error)=["ValueError: invalid literal for int() with base 10: 'ten'\n", "while importing row 2: 'ten'\n", 'hint: prices are whole rupees\n']


### 1.14.5 Custom exception hierarchies

An application defines **one base class** for its own errors, then specific
subclasses. Callers can catch everything from your code with `except AppError`, or one
case precisely. Each class carries what a handler needs as attributes, here the HTTP
status code and a stable machine-readable `error_code`.

| Class | Also a | HTTP status | Extra attributes |
| --- | --- | --- | --- |
| `AppError` | `Exception` | 500 | `message`, `details` |
| `ValidationError` | `ValueError` | 422 | `field` |
| `NotFoundError` | `LookupError` | 404 | `resource`, `resource_id` |

Inheriting from a matching built-in as well (`ValidationError(AppError, ValueError)`)
keeps older `except ValueError` code working. Class-level defaults mean subclasses
only override what differs.

In [822]:
class AppError(Exception):
    """Base class for all errors raised by the shop application."""

    status_code = 500
    error_code = "internal_error"

    def __init__(self, message: str, *, details: dict[str, Any] | None = None) -> None:
        super().__init__(message)
        self.message = message
        self.details = details or {}

    def to_dict(self) -> dict[str, Any]:
        """Return a JSON-ready error body."""
        return {"error": self.error_code, "message": self.message, **self.details}


class ValidationError(AppError, ValueError):
    """Client sent a value that breaks a rule."""

    status_code = 422
    error_code = "validation_error"

    def __init__(self, field: str, message: str) -> None:
        super().__init__(message, details={"field": field})
        self.field = field


class NotFoundError(AppError, LookupError):
    """A requested resource does not exist."""

    status_code = 404
    error_code = "not_found"

    def __init__(self, resource: str, resource_id: object) -> None:
        super().__init__(f"{resource} {resource_id!r} not found", details={"resource": resource})
        self.resource = resource
        self.resource_id = resource_id

One handler at the edge of the application turns any `AppError` into a status code
and body; anything else becomes a generic 500 so internal details do not leak. This is
the shape of FastAPI exception handlers in Phase 6.

In [823]:
PRODUCTS = {101: {"name": "Masala tea", "price": 249}}


def get_product(product_id: int) -> dict[str, Any]:
    """Raise domain errors instead of returning error codes."""
    if product_id <= 0:
        raise ValidationError("product_id", "product_id must be positive")
    if product_id not in PRODUCTS:
        raise NotFoundError("product", product_id)
    return PRODUCTS[product_id]


def handle_request(product_id: int) -> tuple[int, dict[str, Any]]:
    """Map exceptions to (status, body), as a web framework would."""
    try:
        return 200, get_product(product_id)
    except AppError as error:
        return error.status_code, error.to_dict()
    except Exception:
        return 500, {"error": "internal_error", "message": "unexpected error"}


for requested_id in (101, 0, 999):
    print(handle_request(requested_id))

(200, {'name': 'Masala tea', 'price': 249})
(422, {'error': 'validation_error', 'message': 'product_id must be positive', 'field': 'product_id'})
(404, {'error': 'not_found', 'message': 'product 999 not found', 'resource': 'product'})


Because of the second base class, the same exceptions are caught by built-in
categories too, and `isinstance` checks follow the MRO:

In [824]:
missing = NotFoundError("order", 42)
print(" -> ".join(cls.__name__ for cls in NotFoundError.__mro__))
print(f"{isinstance(missing, LookupError)=}, {missing.status_code=}, {missing.resource_id=}")
print(f"{raised_error_name(lambda: get_product(-1))=}")
try:
    get_product(-1)
except ValueError as error:
    print(f"plain except ValueError still works: {error}")

NotFoundError -> AppError -> LookupError -> Exception -> BaseException -> object
isinstance(missing, LookupError)=True, missing.status_code=404, missing.resource_id=42
raised_error_name(lambda: get_product(-1))='ValidationError'
plain except ValueError still works: product_id must be positive


### 1.14.6 Exception groups

*New in 3.11* (PEP 654). Some operations fail in several independent ways at once:
validating a form, running tasks concurrently (`asyncio.TaskGroup`, Phase 5),
retrying several hosts. An **`ExceptionGroup(message, exceptions)`** bundles them into
one raisable object. Groups can be nested.

| Class | Can contain |
| --- | --- |
| `ExceptionGroup` | only `Exception` subclasses (it is itself an `Exception`) |
| `BaseExceptionGroup` | any `BaseException`; constructing it with only `Exception`s actually returns an `ExceptionGroup` |

In [825]:
nightly_failures = ExceptionGroup(
    "nightly jobs failed",
    [
        ConnectionError("payments API unreachable"),
        ValueError("bad row in inventory.csv"),
        ExceptionGroup("email batch", [TimeoutError("smtp 1"), TimeoutError("smtp 2")]),
    ],
)
print(f"{nightly_failures.message=}")
print(f"{len(nightly_failures.exceptions)=}")
print(f"{str(nightly_failures)=}")
print(f"{type(BaseExceptionGroup('m', [ValueError()])).__name__=}")
print(f"{raised_error_name(lambda: ExceptionGroup('m', [KeyboardInterrupt()]))=}")

nightly_failures.message='nightly jobs failed'


len(nightly_failures.exceptions)=3
str(nightly_failures)='nightly jobs failed (3 sub-exceptions)'
type(BaseExceptionGroup('m', [ValueError()])).__name__='ExceptionGroup'
raised_error_name(lambda: ExceptionGroup('m', [KeyboardInterrupt()]))='TypeError'


**`except*`** handles groups. Each `except* SomeError` clause receives a **subgroup**
with all matching leaves (nesting preserved) and runs at most once. Clauses are tried
in order; leaves no clause handles are **re-raised** in a new group. A plain
exception that reaches `except*` is wrapped in a group first.

`check_services()` fails three ways; the `except*` handlers deal with network
problems, and the `ValueError` propagates out to the outer `try`.

In [826]:
def check_services() -> None:
    """Collect every health-check failure instead of stopping at the first."""
    failures: list[Exception] = []
    for service, problem in [
        ("payments", ConnectionError("refused")),
        ("search", TimeoutError("5 s")),
        ("config", ValueError("missing API key")),
        ("email", TimeoutError("10 s")),
    ]:
        problem.add_note(f"service: {service}")
        failures.append(problem)
    raise ExceptionGroup("health check failed", failures)


try:
    try:
        check_services()
    except* ConnectionError as network_group:
        print(f"reconnect later: {[str(e) for e in network_group.exceptions]}")
    except* TimeoutError as slow_group:
        print(
            f"{len(slow_group.exceptions)} timeouts: "
            f"{[e.__notes__[0] for e in slow_group.exceptions]}"
        )
except ExceptionGroup as leftover:
    print(f"re-raised: {leftover.message!r} with {leftover.exceptions!r}")

reconnect later: ['refused']
2 timeouts: ['service: search', 'service: email']
re-raised: 'health check failed' with (ValueError('missing API key'),)


Rules that differ from `except`: a plain exception is wrapped; `except` and `except*`
cannot be mixed on one `try`; `return`, `break` and `continue` are not allowed in an
`except*` block (the remainder still has to be re-raised).

In [827]:
try:
    raise ValueError("single error")
except* ValueError as wrapped_group:
    print(f"{wrapped_group!r}")

print(compile_error("try:\n    pass\nexcept* ValueError:\n    pass\nexcept TypeError:\n    pass"))
print(compile_error("def f():\n    try:\n        pass\n    except* ValueError:\n        return 1"))

ExceptionGroup('', (ValueError('single error'),))
SyntaxError: cannot have both 'except' and 'except*' on the same 'try'


SyntaxError: 'break', 'continue' and 'return' cannot appear in an except* block


**`split` and `subgroup`** are what `except*` uses underneath, and you can call them
directly. Both take an exception class (or tuple), or a predicate function called on
every leaf, and preserve the nesting.

| Method | Returns |
| --- | --- |
| `group.subgroup(condition)` | a group of the matching leaves, or `None` |
| `group.split(condition)` | `(matching_group_or_None, rest_group_or_None)` |

In [828]:
timeouts = nightly_failures.subgroup(TimeoutError)
print(f"{timeouts!r}")

network_part, other_part = nightly_failures.split((ConnectionError, TimeoutError))
print(f"{network_part.exceptions=}")
print(f"{other_part.exceptions=}")

mentions_smtp = nightly_failures.subgroup(lambda leaf: "smtp" in str(leaf))
print(f"{mentions_smtp!r}")
print(f"{nightly_failures.subgroup(KeyError)=}")

ExceptionGroup('nightly jobs failed', [ExceptionGroup('email batch', [TimeoutError('smtp 1'), TimeoutError('smtp 2')])])
network_part.exceptions=(ConnectionError('payments API unreachable'), ExceptionGroup('email batch', [TimeoutError('smtp 1'), TimeoutError('smtp 2')]))
other_part.exceptions=(ValueError('bad row in inventory.csv'),)
ExceptionGroup('nightly jobs failed', [ExceptionGroup('email batch', [TimeoutError('smtp 1'), TimeoutError('smtp 2')])])
nightly_failures.subgroup(KeyError)=None


### 1.14.7 EAFP vs LBYL

Unit 1.9.6 previewed the two styles with `parse_quantity_lbyl` / `parse_quantity_eafp`:

| Style | Shape | Python's default? |
| --- | --- | --- |
| LBYL, *look before you leap* | `if can_do(x): do(x)` | no |
| EAFP, *easier to ask forgiveness than permission* | `try: do(x)` / `except Failure:` | yes |

**TOCTOU races** (time of check to time of use). With LBYL, the world can change
between the check and the use: another process deletes the file, another thread pops
the key. The check passed, the use still fails. The demo simulates the other process
with an `on_gap` callback that deletes the file between the two steps.

In [829]:
def read_stock_lbyl(path: Path, on_gap: Callable[[], object]) -> str | None:
    """Check, then use. on_gap stands for whatever happens in between."""
    if path.exists():
        on_gap()
        return path.read_text(encoding="utf-8")
    return None


def read_stock_eafp(path: Path, on_gap: Callable[[], object]) -> str | None:
    """Just try; there is no gap between a check and a use."""
    on_gap()
    try:
        return path.read_text(encoding="utf-8")
    except FileNotFoundError:
        return None


stock_path = scratch_dir / "unit_1_14_stock.txt"
for reader in (read_stock_lbyl, read_stock_eafp):
    stock_path.write_text("tea=4", encoding="utf-8")
    try:
        print(f"{reader.__name__}: {reader(stock_path, stock_path.unlink)!r}")
    except FileNotFoundError as error:
        print(f"{reader.__name__}: check passed, then {type(error).__name__}")

read_stock_lbyl: check passed, then FileNotFoundError
read_stock_eafp: None


**Zero-cost `try`** (*Changed in 3.11*). Older CPython executed a `SETUP_FINALLY`
instruction on entry to every `try`. Since 3.11, the compiler records the protected
ranges in a side **exception table** (shown in 1.14.12), so entering a `try` costs
nothing; only actually *raising* is expensive (creating the exception object and
traceback, unwinding, matching clauses).

The measurement: a dict lookup with and without a surrounding `try`, then the miss
case handled by `except KeyError` versus `dict.get`. Timings vary by machine; the
ratios are what matter.

In [830]:
menu = {"tea": 30, "coffee": 60}


def lookup_plain(prices: dict[str, int]) -> int:
    """No try at all."""
    return prices["tea"]


def lookup_in_try(prices: dict[str, int]) -> int:
    """Same lookup inside a try that never fires."""
    try:
        return prices["tea"]
    except KeyError:
        return 0


def miss_with_except(prices: dict[str, int]) -> int:
    """The key is missing: an exception is raised and caught every call."""
    try:
        return prices["juice"]
    except KeyError:
        return 0


def miss_with_get(prices: dict[str, int]) -> int:
    """The key is missing: get returns the default, no exception."""
    return prices.get("juice", 0)


CALLS = 200_000
for candidate in (lookup_plain, lookup_in_try, miss_with_except, miss_with_get):
    best = min(timeit.repeat(functools.partial(candidate, menu), number=CALLS, repeat=5))
    print(f"{candidate.__name__:<17} {best / CALLS * 1e9:6.1f} ns per call")

lookup_plain        56.6 ns per call


lookup_in_try       41.3 ns per call

miss_with_except   152.4 ns per call


miss_with_get       51.0 ns per call


The first two lines are about equal (zero-cost entry); the raised-and-caught miss is
several times slower than `get`. That gives the rule for **when LBYL is better**:

| Situation | Prefer | Why |
| --- | --- | --- |
| Failure is rare | EAFP | the `try` is free, the check would run every time |
| Failure is common (a miss on most calls) | LBYL / `get` / `in` | raising costs far more than a check |
| The resource can change underneath you (files, shared state) | EAFP | no TOCTOU gap |
| The operation has side effects that cannot be undone halfway | LBYL | validate everything **before** the first write |
| You need a helpful message for a user | LBYL | the check knows which rule failed |

The side-effect row in code: charging and then discovering the stock is short leaves
the system half-updated, so check first.

In [831]:
def place_order(stock: dict[str, int], charges: list[int], sku: str, quantity: int) -> str:
    """Validate before any side effect; then charge and reserve."""
    if stock.get(sku, 0) < quantity:
        return f"rejected: only {stock.get(sku, 0)} {sku} left"
    charges.append(quantity * 30)
    stock[sku] -= quantity
    return "placed"


warehouse = {"tea": 2}
card_charges: list[int] = []
print(place_order(warehouse, card_charges, "tea", 5), card_charges, warehouse)
print(place_order(warehouse, card_charges, "tea", 2), card_charges, warehouse)

rejected: only 2 tea left [] {'tea': 2}
placed [60] {'tea': 0}


### 1.14.8 Logging exceptions

Inside an `except` block, **`logger.exception(message)`** logs at `ERROR` level and
appends the full traceback (it is `logger.error(message, exc_info=True)`). Call it
only while an exception is being handled. Pass values as `%s` arguments instead of an
f-string: the message is formatted only if the record is actually emitted.

The demo sends the logger's output to an in-memory stream so the cell can print the
interesting lines (the traceback frames contain machine-specific paths). Phase 4
covers logging configuration properly.

In [832]:
log_stream = io.StringIO()
payments_logger = logging.getLogger("shop.payments")
payments_logger.handlers.clear()
stream_handler = logging.StreamHandler(log_stream)
stream_handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
payments_logger.addHandler(stream_handler)
payments_logger.setLevel(logging.INFO)
payments_logger.propagate = False


def capture_payment(order_id: int, amount_text: str) -> int:
    """Log the failure with its traceback, then re-raise for the caller."""
    try:
        return int(amount_text)
    except ValueError:
        payments_logger.exception("could not capture payment for order %s", order_id)
        raise


print(f"{raised_error_name(lambda: capture_payment(42, '49x'))=}")
logged_lines = log_stream.getvalue().splitlines()
print(f"{len(logged_lines)} log lines; first two and last:")
print("\n".join([*logged_lines[:2], logged_lines[-1]]))

raised_error_name(lambda: capture_payment(42, '49x'))='ValueError'
5 log lines; first two and last:
ERROR shop.payments: could not capture payment for order 42
Traceback (most recent call last):
ValueError: invalid literal for int() with base 10: '49x'


**Never swallow errors.** Every `except` must do one of: handle the failure fully
(fall back to a value the caller expects), translate it (`raise … from`), or record
it and re-raise. The anti-patterns:

| Anti-pattern | What goes wrong |
| --- | --- |
| `except Exception: pass` | bugs (typos, wrong types) disappear silently |
| `except Exception: return None` | the caller cannot tell "no result" from "broken" |
| log and continue, when the caller needs to know | the program carries on with bad state |
| `except` around a large block | catches errors from lines you never meant to guard |

The cell shows the first one hiding a real bug: a misspelled attribute raises
`AttributeError`, which the broad handler turns into a plausible default.

In [833]:
def shipping_fee_swallowed(order: dict[str, Any]) -> int:
    """Bug: .uper() is a typo, but the broad except hides it."""
    try:
        return 0 if order["city"].uper() == "PUNE" else 50  # type: ignore[attr-defined]
    except Exception:
        return 50


def shipping_fee(order: dict[str, Any]) -> int:
    """Catch only the failure you expect: a missing city."""
    try:
        city = order["city"]
    except KeyError:
        return 50
    return 0 if city.upper() == "PUNE" else 50


print(f"{shipping_fee_swallowed({'city': 'Pune'})=}  (wrong, and no error)")
print(f"{shipping_fee({'city': 'Pune'})=}")
print(f"{shipping_fee({})=}")

shipping_fee_swallowed({'city': 'Pune'})=50  (wrong, and no error)
shipping_fee({'city': 'Pune'})=0
shipping_fee({})=50


### 1.14.9 The `with` statement

`with EXPRESSION as name:` works with any **context manager**: an object with two
methods.

| Method | Called | Receives | Returns |
| --- | --- | --- | --- |
| `__enter__(self)` | on entry | — | the value bound to `name` (often `self`) |
| `__exit__(self, exc_type, exc, tb)` | on every exit | `None, None, None`, or the exception details | truthy to **suppress** the exception |

The manager below prints what `__exit__` receives on a normal exit and on an error.

In [834]:
class ExitInspector:
    """Show the arguments __exit__ receives."""

    def __enter__(self) -> str:
        print("__enter__ called")
        return "value bound by as"

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc: BaseException | None,
        tb: TracebackType | None,
    ) -> bool:
        print(f"__exit__({exc_type=}, {exc=}, traceback given: {tb is not None})")
        return False


with ExitInspector() as bound_value:
    print(f"body: {bound_value=}")
try:
    with ExitInspector():
        raise ValueError("body failed")
except ValueError as error:
    print(f"propagated: {error!r}")

__enter__ called
body: bound_value='value bound by as'
__exit__(exc_type=None, exc=None, traceback given: False)
__enter__ called
__exit__(exc_type=<class 'ValueError'>, exc=ValueError('body failed'), traceback given: True)
propagated: ValueError('body failed')


**Suppression by returning `True`.** If `__exit__` returns a truthy value, the
exception is discarded and execution continues after the `with` block. Use this
deliberately and narrowly (here: only the listed classes). Returning truthy by accident
(for example returning a non-empty list) swallows every error.

In [835]:
class IgnoreErrors:
    """Suppress only the listed exception classes (a hand-written contextlib.suppress)."""

    def __init__(self, *error_types: type[Exception]) -> None:
        self.error_types = error_types

    def __enter__(self) -> None:
        return None

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc: BaseException | None,
        tb: TracebackType | None,
    ) -> bool:
        return exc_type is not None and issubclass(exc_type, self.error_types)


with IgnoreErrors(KeyError):
    {}["missing"]
    print("not reached")
print("continued after a suppressed KeyError")


def convert_inside_ignore() -> int:
    """A ValueError is not in the list, so it is not suppressed."""
    with IgnoreErrors(KeyError):
        return int("x")
    return 0


print(f"{raised_error_name(convert_inside_ignore)=}")

continued after a suppressed KeyError
raised_error_name(convert_inside_ignore)='ValueError'


**The desugared form** (PEP 343). `with` is shorthand for roughly the code below;
like every special method, `__enter__` and `__exit__` are looked up on the type:

```python
manager = EXPRESSION
enter = type(manager).__enter__      # looked up on the type, like other dunders
exit_ = type(manager).__exit__
value = enter(manager)
try:
    name = value
    BODY
except BaseException as error:
    if not exit_(manager, type(error), error, error.__traceback__):
        raise                         # not suppressed
else:
    exit_(manager, None, None, None)
```

The cell runs this expansion by hand with `ExitInspector` and prints the same lines as
the real `with` above.

In [836]:
manager = ExitInspector()
enter_method = type(manager).__enter__
exit_method = type(manager).__exit__
entered_value = enter_method(manager)
try:
    print(f"body: {entered_value=}")
    raise ValueError("body failed")
except BaseException as error:
    if not exit_method(manager, type(error), error, error.__traceback__):
        print(f"would re-raise: {error!r}")
else:
    exit_method(manager, None, None, None)

__enter__ called
body: entered_value='value bound by as'
__exit__(exc_type=<class 'ValueError'>, exc=ValueError('body failed'), traceback given: True)
would re-raise: ValueError('body failed')


**Multiple managers.** `with A() as a, B() as b:` is the same as nesting: entered left
to right, exited **right to left** (LIFO). Since 3.10 the list can be wrapped in
parentheses across lines. If entering the second manager fails, the first one is
still exited.

In [837]:
class Tracked:
    """Log enter/exit; optionally fail on enter."""

    def __init__(self, name: str, events: list[str], fail_on_enter: bool = False) -> None:
        self.name, self.events, self.fail_on_enter = name, events, fail_on_enter

    def __enter__(self) -> Tracked:
        if self.fail_on_enter:
            raise OSError(f"cannot open {self.name}")
        self.events.append(f"enter {self.name}")
        return self

    def __exit__(self, *exc_info: object) -> None:
        self.events.append(f"exit {self.name}")


events: list[str] = []
with (
    Tracked("database", events) as database,
    Tracked("cache", events) as cache,
):
    events.append(f"body uses {database.name} and {cache.name}")
print(events)

events = []
try:
    with Tracked("database", events), Tracked("cache", events, fail_on_enter=True):
        events.append("body")
except OSError as error:
    events.append(f"caught {error}")
print(events)

['enter database', 'enter cache', 'body uses database and cache', 'exit cache', 'exit database']
['enter database', 'exit database', 'caught cannot open cache']


#### Example: `Timer`

A timing context manager measures a block, not a whole function (the `@timer`
decorator from 1.12.3 times calls). `__exit__` reports even when the block raises,
and does not suppress. Lab 1.14 asks for a richer `StopwatchTimer`; the Phase 1
Project times its run with this `Timer`.

In [838]:
class Timer:
    """Time a with block and report "<label> took <ms> ms"."""

    def __init__(self, label: str, report: Callable[[str], object] = print) -> None:
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        self._started = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        self.elapsed_seconds = time.perf_counter() - self._started
        self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")

Using it, including on a failing block (durations vary by machine):

In [839]:
with Timer("sum of squares") as squares_timer:
    total = sum(number * number for number in range(200_000))
print(f"{total=}, {squares_timer.elapsed_seconds > 0=}")

try:
    with Timer("failing block"):
        raise ConnectionError("upstream down")
except ConnectionError as error:
    print(f"still reported, then propagated: {error!r}")

sum of squares took 10.93 ms
total=2666646666700000, squares_timer.elapsed_seconds > 0=True
failing block took 0.00 ms
still reported, then propagated: ConnectionError('upstream down')


#### Example: `DatabaseTransaction`

The classic use: commit if the block succeeds, roll back if it raises, and let the
exception continue. The demo uses an in-memory SQLite database (standard library, no
server). A `CHECK` constraint makes an overdraft fail halfway through a transfer; the
rollback undoes the first `UPDATE`.

In [840]:
class DatabaseTransaction:
    """Commit on success, roll back on any exception (never suppress)."""

    def __init__(self, connection: sqlite3.Connection) -> None:
        self.connection = connection

    def __enter__(self) -> sqlite3.Cursor:
        self.connection.execute("BEGIN")
        return self.connection.cursor()

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc: BaseException | None,
        tb: TracebackType | None,
    ) -> bool:
        if exc_type is None:
            self.connection.commit()
        else:
            self.connection.rollback()
        return False


bank = sqlite3.connect(":memory:", isolation_level=None)
bank.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
inserted = bank.executemany("INSERT INTO accounts VALUES (?, ?)", [("asha", 500), ("ravi", 100)])
print(f"{inserted.rowcount} accounts created")

2 accounts created


A transfer that succeeds, then one that overdraws `ravi`:

In [841]:
def transfer(connection: sqlite3.Connection, source: str, target: str, amount: int) -> str:
    """Move money inside one transaction."""
    try:
        with DatabaseTransaction(connection) as cursor:
            cursor.execute(
                "UPDATE accounts SET balance = balance + ? WHERE name = ?", (amount, target)
            )
            cursor.execute(
                "UPDATE accounts SET balance = balance - ? WHERE name = ?", (amount, source)
            )
    except sqlite3.IntegrityError as error:
        return f"rolled back ({error})"
    return "committed"


def balances(connection: sqlite3.Connection) -> dict[str, int]:
    """Current balances by name."""
    return dict(connection.execute("SELECT name, balance FROM accounts ORDER BY name"))


print(transfer(bank, "asha", "ravi", 200), balances(bank))
print(transfer(bank, "ravi", "asha", 1_000), balances(bank))

committed {'asha': 300, 'ravi': 300}
rolled back (CHECK constraint failed: balance >= 0) {'asha': 300, 'ravi': 300}


> **Note:** a `sqlite3.Connection` is itself a context manager, but its `__exit__`
> commits or rolls back and does **not close** the connection. Use
> `contextlib.closing(connection)` (1.14.10) to close it.

### 1.14.10 `contextlib`

The standard library's toolbox for context managers. Each row gets a demo below.

| API | What it does | Typical use |
| --- | --- | --- |
| `@contextmanager` | turns a one-`yield` generator into a context manager | most hand-written managers |
| `suppress(*errors)` | ignores the listed exceptions | "delete if present" |
| `closing(thing)` | calls `thing.close()` on exit | objects with `close()` but no `with` support |
| `redirect_stdout(stream)` | sends `print` output to another stream | capturing output in tests |
| `nullcontext(value)` | does nothing, returns `value` | "maybe a manager" code paths |
| `chdir(path)` | changes the working directory, restores it on exit (*New in 3.11*) | scripts, tests |
| `ExitStack()` | enters a variable number of managers and callbacks, unwinds them LIFO | "open N things" |
| `AbstractContextManager` | ABC with a default `__enter__` returning `self` | class-based managers, `isinstance` checks |

#### `@contextmanager`

The code before `yield` is `__enter__`, the yielded value is bound by `as`, the code
after `yield` is `__exit__`. If the body raises, the exception is **thrown into the
generator at the `yield`**, so cleanup must sit in `finally`, or it is skipped. The
demo leases a connection from a pool; the unsafe version leaks it on an error.

In [842]:
@contextmanager
def leased_connection_unsafe(pool: list[str]) -> Iterator[str]:
    """Bug: no try/finally, so an exception in the body skips the return to the pool."""
    connection = pool.pop()
    yield connection
    pool.append(connection)


@contextmanager
def leased_connection(pool: list[str]) -> Iterator[str]:
    """Return the connection to the pool on every exit."""
    connection = pool.pop()
    try:
        yield connection
    finally:
        pool.append(connection)


for lease in (leased_connection_unsafe, leased_connection):
    pool = ["conn-1", "conn-2"]
    with suppress(RuntimeError), lease(pool) as connection:
        raise RuntimeError(f"query failed on {connection}")
    print(f"{lease.__name__:<25} pool after an error: {pool}")

leased_connection_unsafe  pool after an error: ['conn-1']
leased_connection         pool after an error: ['conn-1', 'conn-2']


A `@contextmanager` object also works as a **decorator** (it is a
`ContextDecorator`): each call of the decorated function runs inside a fresh `with`.

In [843]:
shared_pool = ["conn-1"]


@leased_connection(shared_pool)
def nightly_report() -> str:
    """Runs while a connection is leased from shared_pool."""
    return f"pool during the call: {shared_pool}"


print(nightly_report())
print(f"pool after the call: {shared_pool}")

pool during the call: []
pool after the call: ['conn-1']


#### `suppress`

`with suppress(FileNotFoundError):` replaces a `try` / `except …: pass`. Use it only
when ignoring the error is truly the intent. (For files specifically,
`Path.unlink(missing_ok=True)` does the same.)

In [844]:
temporary_report = scratch_dir / "unit_1_14_temp_report.txt"
temporary_report.write_text("draft", encoding="utf-8")
for attempt in (1, 2):
    with suppress(FileNotFoundError):
        temporary_report.unlink()
        print(f"attempt {attempt}: deleted")
        continue
    print(f"attempt {attempt}: already gone, FileNotFoundError suppressed")

attempt 1: deleted
attempt 2: already gone, FileNotFoundError suppressed


#### `closing`

For an object that has `close()` but is not a context manager (older APIs, some
third-party clients), `closing(thing)` returns `thing` from `__enter__` and calls
`thing.close()` on exit.

In [845]:
class LegacyPrinterClient:
    """Has close() but no __enter__/__exit__."""

    def __init__(self) -> None:
        self.closed = False

    def print_label(self, text: str) -> str:
        return f"printed {text!r}"

    def close(self) -> None:
        self.closed = True


with closing(LegacyPrinterClient()) as printer:
    print(printer.print_label("ORDER-42"))
print(f"{printer.closed=}")


def use_without_closing() -> None:
    """A plain with fails: the class has no __enter__/__exit__."""
    with LegacyPrinterClient():  # type: ignore[attr-defined]
        pass


print(f"{raised_error_name(use_without_closing)=}")

printed 'ORDER-42'
printer.closed=True
raised_error_name(use_without_closing)='TypeError'


#### `redirect_stdout`

Temporarily points `sys.stdout` at another stream, so everything `print` writes is
captured. `redirect_stderr` does the same for `sys.stderr`. It changes global state,
so it is not safe across threads.

In [846]:
def noisy_import_step() -> int:
    """Prints progress as a side effect."""
    print("importing 3 rows")
    print("done")
    return 3


captured_output = io.StringIO()
with redirect_stdout(captured_output):
    imported_count = noisy_import_step()
print(f"{imported_count=}, {captured_output.getvalue()=}")

imported_count=3, captured_output.getvalue()='importing 3 rows\ndone\n'


#### `nullcontext`

A manager that does nothing and returns its argument. It removes an `if` when a
resource is sometimes managed and sometimes not, for example "accept a path (open and
close it) or an already-open stream (leave it open)".

In [847]:
def count_lines(source: Path | io.StringIO) -> int:
    """Open a path ourselves, or use a caller-owned stream without closing it."""
    manager = source.open(encoding="utf-8") if isinstance(source, Path) else nullcontext(source)
    with manager as stream:
        return sum(1 for _ in stream)


lines_path = scratch_dir / "unit_1_14_lines.txt"
lines_path.write_text("a\nb\nc\n", encoding="utf-8")
caller_stream = io.StringIO("x\ny\n")
print(f"{count_lines(lines_path)=}, {count_lines(caller_stream)=}, {caller_stream.closed=}")
with nullcontext(42) as answer:
    print(f"{answer=}")

count_lines(lines_path)=3, count_lines(caller_stream)=2, caller_stream.closed=False
answer=42


#### `chdir`

*New in 3.11.* Changes the process working directory for the block and always changes
it back. Like `redirect_stdout` it changes process-wide state, so avoid it in threaded
or async code; prefer passing absolute paths.

In [848]:
directory_before = Path.cwd()
with chdir(scratch_dir):
    print(f"inside:  cwd name = {Path.cwd().name}")
    print(f"{Path('unit_1_14_lines.txt').exists()=}  (relative to the new cwd)")
print(f"after:   restored = {Path.cwd() == directory_before}")

inside:  cwd name = _scratch
Path('unit_1_14_lines.txt').exists()=True  (relative to the new cwd)
after:   restored = True


#### `ExitStack`

For a number of managers known only at run time. `enter_context(manager)` enters one
and schedules its exit; `callback(function, *args)` schedules any cleanup function;
everything unwinds in reverse order when the stack exits, even after an error.
`pop_all()` moves the pending cleanups to a new stack: the pattern for "acquire
everything, and only keep it if all acquisitions succeed".

In [849]:
cleanup_log: list[str] = []
with ExitStack() as stack:
    for name in ("orders", "payments", "audit"):
        stack.enter_context(Tracked(name, cleanup_log))
    stack.callback(cleanup_log.append, "callback: flush metrics")
    cleanup_log.append("body runs")
print(cleanup_log)

['enter orders', 'enter payments', 'enter audit', 'body runs', 'callback: flush metrics', 'exit audit', 'exit payments', 'exit orders']


`pop_all()`: keep the acquired managers open past the `with`, then close them later.

In [850]:
def acquire_all(names: list[str], events: list[str]) -> ExitStack:
    """Enter every manager; if one fails, the ones already entered are exited."""
    with ExitStack() as stack:
        for name in names:
            stack.enter_context(Tracked(name, events, fail_on_enter=(name == "broken")))
        return stack.pop_all()


events = []
kept_open = acquire_all(["orders", "payments"], events)
print(f"after acquire: {events}")
kept_open.close()
print(f"after close:   {events}")

events = []
print(f"{raised_error_name(lambda: acquire_all(['orders', 'broken'], events))=}, {events=}")

after acquire: ['enter orders', 'enter payments']
after close:   ['enter orders', 'enter payments', 'exit payments', 'exit orders']
raised_error_name(lambda: acquire_all(['orders', 'broken'], events))='OSError', events=['enter orders', 'exit orders']


#### `AbstractContextManager`

Subclassing it gives a default `__enter__` that returns `self`, so a class-based
manager only writes `__exit__`. It is also an ABC with a structural
`__subclasshook__`: any class with both methods counts as a context manager for
`isinstance`, inheritance or not.

In [851]:
class AuditScope(AbstractContextManager):
    """Only __exit__ is written; __enter__ comes from the ABC."""

    def __init__(self) -> None:
        self.entries: list[str] = []

    def __exit__(self, *exc_info: object) -> None:
        self.entries.append("scope closed")


with AuditScope() as scope:
    scope.entries.append("work")
print(f"{scope.entries=}")
print(f"{isinstance(Timer('x'), AbstractContextManager)=}  (structural, no inheritance)")
print(f"{isinstance(LegacyPrinterClient(), AbstractContextManager)=}")

scope.entries=['work', 'scope closed']
isinstance(Timer('x'), AbstractContextManager)=True  (structural, no inheritance)
isinstance(LegacyPrinterClient(), AbstractContextManager)=False


### 1.14.11 `assert` and `warnings`

`assert condition, message` raises `AssertionError(message)` when the condition is
false. It is for **internal invariants** ("this can never happen if my code is
right"), never for validating input: running Python with **`-O`** sets `__debug__` to
`False` and the compiler **removes every assert**. The cell runs the same script in
child interpreters with and without `-O`.

In [852]:
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0"}  # plain tracebacks (3.13+ colours them)
ASSERT_SCRIPT = (
    "assert 1 + 1 == 3, 'arithmetic invariant broken'\n"
    "print('assert was stripped; __debug__ =', __debug__)"
)
for flags in ([], ["-O"]):
    completed = subprocess.run(
        [sys.executable, *flags, "-c", ASSERT_SCRIPT],
        capture_output=True,
        text=True,
        check=False,
        env=CHILD_ENV,
    )
    last_line = (completed.stdout or completed.stderr).strip().splitlines()[-1]
    print(f"python {' '.join(flags) or '(no -O)':<8} exit={completed.returncode}  {last_line}")

python (no -O)  exit=1  AssertionError: arithmetic invariant broken


python -O       exit=0  assert was stripped; __debug__ = False


A classic trap: `assert (condition, "message")` asserts a non-empty **tuple**, which
is always true. The compiler warns (`SyntaxWarning`).

In [853]:
with warnings.catch_warnings(record=True) as assert_warnings:
    warnings.simplefilter("always")
    compile("assert (1 == 2, 'never fails')", "<demo>", "exec")
print([f"{w.category.__name__}: {w.message}" for w in assert_warnings])

['SyntaxWarning: assertion is always true, perhaps remove parentheses?']


**`warnings.warn(message, category, stacklevel)`** reports something the caller
should fix without stopping the program. `stacklevel=2` blames the caller's line
instead of the line inside your function.

| Category | Shown by default? | Use for |
| --- | --- | --- |
| `UserWarning` (default) | yes | general "you probably didn't mean this" |
| `DeprecationWarning` | only when triggered from `__main__` | API going away, for developers |
| `FutureWarning` | yes | behaviour will change, for end users |
| `RuntimeWarning` | yes | dubious runtime situations |
| `SyntaxWarning` | yes | dubious syntax (compile time) |

The demo records warnings with `catch_warnings(record=True)` so the output is
deterministic.

In [854]:
def legacy_price_in_rupees(price_in_paise: int) -> float:
    """Old API: warn callers, keep working."""
    warnings.warn("use price_in_paise / 100 with Decimal", DeprecationWarning, stacklevel=2)
    return price_in_paise / 100


with warnings.catch_warnings(record=True) as recorded:
    warnings.simplefilter("always")
    print(f"{legacy_price_in_rupees(24_900)=}")
for recorded_warning in recorded:
    print(f"{recorded_warning.category.__name__}: {recorded_warning.message}")

legacy_price_in_rupees(24_900)=249.0


**Filters** decide what happens to each warning. `warnings.simplefilter(action,
category)` and `warnings.filterwarnings(action, message=regex, category=…,
module=regex)` add filters; the first matching filter wins. The same filters can be
set from outside with `-W action::Category` or `PYTHONWARNINGS`.

| Action | Effect |
| --- | --- |
| `"default"` | show once per location |
| `"always"` | show every time |
| `"once"` | show once per message, anywhere |
| `"module"` | show once per module |
| `"ignore"` | never show |
| `"error"` | raise the warning as an exception |

In [855]:
def noisy_conversion() -> None:
    """Emit two different warnings."""
    warnings.warn("rounding to whole rupees", UserWarning, stacklevel=2)
    warnings.warn("legacy currency code INR-OLD", DeprecationWarning, stacklevel=2)


with warnings.catch_warnings(record=True) as kept:
    warnings.simplefilter("always")
    warnings.filterwarnings("ignore", message="rounding", category=UserWarning)
    noisy_conversion()
    noisy_conversion()
print([str(w.message) for w in kept])

['legacy currency code INR-OLD', 'legacy currency code INR-OLD']


**`catch_warnings()`** saves the filter list and restores it on exit, so changes made
inside never leak. `record=True` collects `WarningMessage` objects instead of printing.

In [856]:
filters_before = len(warnings.filters)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    warnings.filterwarnings("error", category=RuntimeWarning)
    print(f"inside:  {len(warnings.filters) - filters_before} filters added")
print(f"after:   {len(warnings.filters) == filters_before=}")

inside:  2 filters added
after:   len(warnings.filters) == filters_before=True


**Escalating to errors.** The `"error"` action turns a warning into an exception,
which is how test suites make sure no deprecated API is used (`pytest -W error`,
Phase 4). In-process with a filter, then for a whole child interpreter with `-W`:

In [857]:
with warnings.catch_warnings():
    warnings.simplefilter("error", DeprecationWarning)
    try:
        legacy_price_in_rupees(100)
    except DeprecationWarning as error:
        print(f"raised as an exception: {type(error).__name__}: {error}")

WARN_SCRIPT = (
    "import warnings\nwarnings.warn('old checkout API', DeprecationWarning)\nprint('continued')"
)
for flags in ([], ["-W", "error::DeprecationWarning"]):
    completed = subprocess.run(
        [sys.executable, *flags, "-c", WARN_SCRIPT],
        capture_output=True,
        text=True,
        check=False,
        env=CHILD_ENV,
    )
    outcome = completed.stdout.strip() or completed.stderr.strip().splitlines()[-1]
    print(f"{' '.join(flags) or '(default)':<28} exit={completed.returncode}  {outcome}")

raised as an exception: DeprecationWarning: use price_in_paise / 100 with Decimal


(default)                    exit=0  continued


-W error::DeprecationWarning exit=1  DeprecationWarning: old checkout API


### 1.14.12 Under the hood

**The exception table.** Since 3.11 there is no instruction that "enters" a `try`.
Instead, each code object carries an exception table: *if an exception is raised
between instruction offsets `start` and `end`, jump to `target` with the value stack
cut to `depth`*. `dis` prints it after the instructions as `ExceptionTable:`, with
`L` labels. Instruction names change between versions; the structure is the point.

What to notice in the listing:

- the `try` body (`L1` to `L2`) has no setup instruction at all: that is the
  zero-cost entry measured in 1.14.7;
- `PUSH_EXC_INFO`, `CHECK_EXC_MATCH` and `POP_JUMP_IF_FALSE` implement
  `except ValueError` (an `isinstance`-style test, 1.14.1);
- the handler ends with `STORE_FAST` / `DELETE_FAST` of `error`: the hidden
  `del error` from 1.14.2;
- `RERAISE` re-raises when no clause matches.

In [858]:
def parse_quantity(text: str) -> int:
    """try / except with an as-name."""
    try:
        return int(text)
    except ValueError as error:
        print(error)
        return 0


dis.dis(parse_quantity)

   1           RESUME                   0

   3           NOP

   4   L1:     LOAD_GLOBAL              1 (int + NULL)
               LOAD_FAST_BORROW         0 (text)
               CALL                     1
       L2:     RETURN_VALUE

  --   L3:     PUSH_EXC_INFO

   5           LOAD_GLOBAL              2 (ValueError)
               CHECK_EXC_MATCH
               POP_JUMP_IF_FALSE       23 (to L7)
               NOT_TAKEN
               STORE_FAST               1 (error)

   6   L4:     LOAD_GLOBAL              5 (print + NULL)
               LOAD_FAST                1 (error)
               CALL                     1
               POP_TOP

   7   L5:     POP_EXCEPT
               LOAD_COMMON_CONSTANT     7 (None)
               STORE_FAST               1 (error)
               DELETE_FAST              1 (error)
               LOAD_SMALL_INT           0
               RETURN_VALUE

  --   L6:     LOAD_COMMON_CONSTANT     7 (None)
               STORE_FAST               1 (error)
  

The same table as data: `dis.Bytecode(...).exception_entries` lists the entries
(offsets in bytes), and the raw encoded table lives in `co_exceptiontable`.

In [859]:
for entry in dis.Bytecode(parse_quantity).exception_entries:
    print(
        f"offsets {entry.start:>3}-{entry.end:<3} -> handler {entry.target:>3}  "
        f"depth={entry.depth} lasti={entry.lasti}"
    )
print(f"{len(parse_quantity.__code__.co_exceptiontable)=} bytes")

offsets   6-26  -> handler  28  depth=0 lasti=False
offsets  28-50  -> handler  94  depth=1 lasti=True
offsets  50-72  -> handler  84  depth=1 lasti=True
offsets  84-94  -> handler  94  depth=1 lasti=True
len(parse_quantity.__code__.co_exceptiontable)=16 bytes


**`with` bytecode.** In 3.14+, `LOAD_SPECIAL` looks up `__enter__` and `__exit__` on
the **type** (3.11–3.13 used `BEFORE_WITH`). The body is covered by an exception-table
entry whose handler runs `WITH_EXCEPT_START`, which calls `__exit__(type, value,
traceback)` and then tests the result with `TO_BOOL`: truthy means suppress. On the
normal path, `__exit__(None, None, None)` is a plain `CALL` with three `None`s.

In [860]:
def first_line(path: Path) -> str:
    """A with statement around a file."""
    with path.open(encoding="utf-8") as stream:
        return stream.readline()


dis.dis(first_line)

   1           RESUME                   0

   3           LOAD_FAST_BORROW         0 (path)
               LOAD_ATTR                1 (open + NULL|self)
               LOAD_CONST               1 ('utf-8')
               LOAD_CONST               2 (('encoding',))
               CALL_KW                  1
               COPY                     1
               LOAD_SPECIAL             1 (__exit__)
               SWAP                     2
               SWAP                     3
               LOAD_SPECIAL             0 (__enter__)
               CALL                     0
       L1:     STORE_FAST               1 (stream)

   4           LOAD_FAST_BORROW         1 (stream)
               LOAD_ATTR                3 (readline + NULL|self)
               CALL                     0

   3   L2:     SWAP                     3
               SWAP                     2
               LOAD_COMMON_CONSTANT     7 (None)
               LOAD_COMMON_CONSTANT     7 (None)
               LOAD_COMMON_

### Lab 1.14

Six exercises. For each, replace the `raise NotImplementedError` in the stub, then
run the check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — `parse_age(text)` with notes and chaining

Return the age as an `int` after stripping whitespace.

- Not a whole number (`int()` fails): raise
  `ValueError(f"invalid age: {text!r}")` **chained** from the original error
  (`__cause__`), with the note `"age must be a whole number, e.g. 42"`.
- Outside 0–150: raise `ValueError(f"age out of range: {age}")` with the note
  `"age must be between 0 and 150"` and no cause.

In [861]:
def parse_age(text: str) -> int:
    """Parse an age in years, with helpful notes on failure."""
    raise NotImplementedError("Lab 1.14: implement parse_age()")

Run the checks. `caught_error(thunk)` returns the exception a call raises (or `None`),
so the checks can look at its message, notes and cause.

In [862]:
def caught_error(thunk: Callable[[], object]) -> BaseException | None:
    """Return the exception thunk raises, or None."""
    try:
        thunk()
    except BaseException as error:
        return error
    return None


def error_report(thunk: Callable[[], object]) -> tuple[str, str, list[str], str]:
    """(type, message, notes, cause type) of the raised exception."""
    error = caught_error(thunk)
    return (
        type(error).__name__,
        str(error),
        getattr(error, "__notes__", []),
        type(error.__cause__).__name__ if error is not None else "",
    )


run_checks(
    [
        ("plain", lambda: parse_age("42"), 42),
        ("whitespace", lambda: parse_age(" 7 "), 7),
        ("bounds", lambda: (parse_age("0"), parse_age("150")), (0, 150)),
        (
            "not a number",
            lambda: error_report(lambda: parse_age("abc")),
            (
                "ValueError",
                "invalid age: 'abc'",
                ["age must be a whole number, e.g. 42"],
                "ValueError",
            ),
        ),
        ("decimal", lambda: error_report(lambda: parse_age("4.5"))[1], "invalid age: '4.5'"),
        (
            "too large",
            lambda: error_report(lambda: parse_age("200")),
            ("ValueError", "age out of range: 200", ["age must be between 0 and 150"], "NoneType"),
        ),
        ("negative", lambda: error_report(lambda: parse_age("-1"))[1], "age out of range: -1"),
    ]
)

FAIL  plain: raised NotImplementedError: Lab 1.14: implement parse_age()
FAIL  whitespace: raised NotImplementedError: Lab 1.14: implement parse_age()
FAIL  bounds: raised NotImplementedError: Lab 1.14: implement parse_age()
FAIL  not a number: expected ('ValueError', "invalid age: 'abc'", ['age must be a whole number, e.g. 42'], 'ValueError'), got ('NotImplementedError', 'Lab 1.14: implement parse_age()', [], 'NoneType')
FAIL  decimal: expected "invalid age: '4.5'", got 'Lab 1.14: implement parse_age()'
FAIL  too large: expected ('ValueError', 'age out of range: 200', ['age must be between 0 and 150'], 'NoneType'), got ('NotImplementedError', 'Lab 1.14: implement parse_age()', [], 'NoneType')
FAIL  negative: expected 'age out of range: -1', got 'Lab 1.14: implement parse_age()'
0/7 checks passed


#### Exercise 2 — `load_config(path)` converting errors with `from`

Read a JSON file (UTF-8) and return the resulting `dict`. Convert every failure into
the given `ConfigError`, chained with `from`:

| Failure | Message | `__cause__` |
| --- | --- | --- |
| file missing | `f"config file not found: {path.name}"` | the `FileNotFoundError` |
| invalid JSON | `f"invalid JSON in {path.name} at line {lineno}"` | the `json.JSONDecodeError` (it has `.lineno`) |
| top level is not an object | `f"config must be a JSON object, got {type_name}"` | `None` |

In [863]:
class ConfigError(Exception):
    """Raised when a configuration file cannot be loaded."""


def load_config(path: Path) -> dict[str, Any]:
    """Load a JSON object from path, raising ConfigError on any failure."""
    raise NotImplementedError("Lab 1.14: implement load_config()")

Run the checks. The cell writes three small files to `scratch_dir`.

In [864]:
good_config = scratch_dir / "lab_1_14_good.json"
good_config.write_text('{"debug": true, "port": 8000}', encoding="utf-8")
broken_config = scratch_dir / "lab_1_14_broken.json"
broken_config.write_text('{"debug": true,\n "port": }', encoding="utf-8")
list_config = scratch_dir / "lab_1_14_list.json"
list_config.write_text("[1, 2]", encoding="utf-8")
missing_config = scratch_dir / "lab_1_14_missing.json"
run_checks(
    [
        ("loads", lambda: load_config(good_config), {"debug": True, "port": 8000}),
        (
            "missing",
            lambda: error_report(lambda: load_config(missing_config)),
            (
                "ConfigError",
                "config file not found: lab_1_14_missing.json",
                [],
                "FileNotFoundError",
            ),
        ),
        (
            "invalid JSON",
            lambda: error_report(lambda: load_config(broken_config)),
            (
                "ConfigError",
                "invalid JSON in lab_1_14_broken.json at line 2",
                [],
                "JSONDecodeError",
            ),
        ),
        (
            "not an object",
            lambda: error_report(lambda: load_config(list_config)),
            ("ConfigError", "config must be a JSON object, got list", [], "NoneType"),
        ),
    ]
)

FAIL  loads: raised NotImplementedError: Lab 1.14: implement load_config()
FAIL  missing: expected ('ConfigError', 'config file not found: lab_1_14_missing.json', [], 'FileNotFoundError'), got ('NotImplementedError', 'Lab 1.14: implement load_config()', [], 'NoneType')
FAIL  invalid JSON: expected ('ConfigError', 'invalid JSON in lab_1_14_broken.json at line 2', [], 'JSONDecodeError'), got ('NotImplementedError', 'Lab 1.14: implement load_config()', [], 'NoneType')
FAIL  not an object: expected ('ConfigError', 'config must be a JSON object, got list', [], 'NoneType'), got ('NotImplementedError', 'Lab 1.14: implement load_config()', [], 'NoneType')
0/4 checks passed


#### Exercise 3 — a `StopwatchTimer` context-manager class

`StopwatchTimer(clock=time.perf_counter)`, where `clock` returns seconds as a float:

- `__enter__` records the start time and returns the stopwatch; entering one that is
  already running raises `RuntimeError`. Entering it again later starts over (laps
  cleared).
- `__exit__` records the stop time and never suppresses exceptions.
- `elapsed` (a property): `0.0` before the first start; `clock() - start` while
  running; `stop - start` afterwards (it no longer changes).
- `running`: `True` only inside the block.
- `lap()`: seconds since the previous lap (or the start); appends it to the `laps`
  list and returns it; `RuntimeError` if not running.

In [865]:
class StopwatchTimer:
    """Context-manager stopwatch with laps and an injectable clock."""

    def __init__(self, clock: Callable[[], float] = time.perf_counter) -> None:
        raise NotImplementedError("Lab 1.14: implement StopwatchTimer")

Run the checks. `fake_clock(*readings)` returns the given readings one per call, so
every duration is exact.

In [866]:
def fake_clock(*readings: float) -> Callable[[], float]:
    """A clock that returns the given readings in order."""
    remaining = list(readings)
    return lambda: remaining.pop(0)


def stopwatch_laps() -> tuple[list[float], float]:
    """Readings: enter 0, lap 1, lap 3, exit 6."""
    with StopwatchTimer(clock=fake_clock(0.0, 1.0, 3.0, 6.0)) as stopwatch:
        stopwatch.lap()
        stopwatch.lap()
    return stopwatch.laps, stopwatch.elapsed


def stopwatch_while_running() -> tuple[bool, float, bool, float]:
    """Readings: enter 10, elapsed read 14, exit 15."""
    stopwatch = StopwatchTimer(clock=fake_clock(10.0, 14.0, 15.0))
    with stopwatch:
        during = (stopwatch.running, stopwatch.elapsed)
    return (*during, stopwatch.running, stopwatch.elapsed)


def stopwatch_on_error() -> tuple[str, float]:
    """The body raises; the time is still recorded and the error propagates."""
    stopwatch = StopwatchTimer(clock=fake_clock(0.0, 2.5))

    def failing_block() -> None:
        with stopwatch:
            raise ValueError("body failed")

    return raised_error_name(failing_block), stopwatch.elapsed


def stopwatch_nested() -> str:
    """Entering a running stopwatch must fail."""
    stopwatch = StopwatchTimer(clock=fake_clock(0.0, 1.0, 2.0, 3.0))
    with stopwatch:
        return raised_error_name(lambda: stopwatch.__enter__())


def stopwatch_real_clock() -> bool:
    """The default clock measures real time."""
    with StopwatchTimer() as stopwatch:
        time.sleep(0.01)
    return stopwatch.elapsed >= 0.01


run_checks(
    [
        (
            "returns self",
            lambda: (lambda s: s.__enter__() is s)(StopwatchTimer(fake_clock(0.0))),
            True,
        ),
        (
            "before start",
            lambda: (lambda s: (s.elapsed, s.running, s.laps))(StopwatchTimer()),
            (0.0, False, []),
        ),
        ("laps and total", stopwatch_laps, ([1.0, 2.0], 6.0)),
        ("elapsed while running, frozen after", stopwatch_while_running, (True, 4.0, False, 5.0)),
        ("error propagates, time recorded", stopwatch_on_error, ("ValueError", 2.5)),
        (
            "lap when stopped",
            lambda: raised_error_name(lambda: StopwatchTimer().lap()),
            "RuntimeError",
        ),
        ("nested enter", stopwatch_nested, "RuntimeError"),
        ("real clock", stopwatch_real_clock, True),
    ]
)

FAIL  returns self: raised NotImplementedError: Lab 1.14: implement StopwatchTimer
FAIL  before start: raised NotImplementedError: Lab 1.14: implement StopwatchTimer
FAIL  laps and total: raised NotImplementedError: Lab 1.14: implement StopwatchTimer
FAIL  elapsed while running, frozen after: raised NotImplementedError: Lab 1.14: implement StopwatchTimer
FAIL  error propagates, time recorded: raised NotImplementedError: Lab 1.14: implement StopwatchTimer
FAIL  lap when stopped: expected 'RuntimeError', got 'NotImplementedError'
FAIL  nested enter: raised NotImplementedError: Lab 1.14: implement StopwatchTimer
FAIL  real clock: raised NotImplementedError: Lab 1.14: implement StopwatchTimer
0/8 checks passed


#### Exercise 4 — `temporary_env(**variables)` with `@contextmanager`

Set environment variables (`os.environ`) for the duration of a `with` block. A value
of `None` means "unset this variable inside the block". On exit, **even after an
exception**, restore exactly the previous state: variables that existed get their old
value back, variables that did not exist are removed. Yield nothing. (The last check
only looks at the structure, so it already passes on the stub.)

In [867]:
@contextmanager
def temporary_env(**variables: str | None) -> Iterator[None]:
    """Temporarily set (or unset, with None) environment variables."""
    raise NotImplementedError("Lab 1.14: implement temporary_env()")
    yield

Run the checks. They use two variable names that are removed first.

In [868]:
for variable_name in ("LAB_1_14_MODE", "LAB_1_14_REGION"):
    os.environ.pop(variable_name, None)


def env_inside_and_after() -> tuple[str | None, str | None]:
    """Value inside the block, then after it."""
    with temporary_env(LAB_1_14_MODE="test"):
        inside = os.environ.get("LAB_1_14_MODE")
    return inside, os.environ.get("LAB_1_14_MODE")


def env_restores_existing() -> tuple[str | None, str | None, str | None]:
    """An existing variable: overridden, unset, then restored."""
    os.environ["LAB_1_14_REGION"] = "ap-south-1"
    try:
        with temporary_env(LAB_1_14_REGION="eu-west-1"):
            overridden = os.environ.get("LAB_1_14_REGION")
        with temporary_env(LAB_1_14_REGION=None):
            unset = os.environ.get("LAB_1_14_REGION")
        return overridden, unset, os.environ.get("LAB_1_14_REGION")
    finally:
        os.environ.pop("LAB_1_14_REGION", None)


def env_after_error() -> tuple[str, str | None]:
    """The block raises; the variable must still be removed."""

    def failing_block() -> None:
        with temporary_env(LAB_1_14_MODE="broken"):
            raise RuntimeError("body failed")

    return raised_error_name(failing_block), os.environ.get("LAB_1_14_MODE")


run_checks(
    [
        ("set inside, removed after", env_inside_and_after, ("test", None)),
        ("existing restored", env_restores_existing, ("eu-west-1", None, "ap-south-1")),
        ("restored after an error", env_after_error, ("RuntimeError", None)),
        (
            "built with @contextmanager",
            lambda: inspect.isgeneratorfunction(getattr(temporary_env, "__wrapped__", None)),
            True,
        ),
    ]
)

FAIL  set inside, removed after: raised NotImplementedError: Lab 1.14: implement temporary_env()
FAIL  existing restored: raised NotImplementedError: Lab 1.14: implement temporary_env()
FAIL  restored after an error: expected ('RuntimeError', None), got ('NotImplementedError', None)
PASS  built with @contextmanager
1/4 checks passed


#### Exercise 5 — `open_many(paths)` with `ExitStack`

A `@contextmanager` that opens every path and yields the list of open file objects, in
order. Use one `ExitStack`: every file must be closed when the block exits (also after
an error), and if opening the *k*-th file fails, the files already opened are closed
before the error propagates. `opener` opens one path; it defaults to
`open(path, encoding="utf-8")` (the checks pass a recording opener).

In [869]:
@contextmanager
def open_many(
    paths: Iterable[Path],
    opener: Callable[[Path], IO[str]] | None = None,
) -> Iterator[list[IO[str]]]:
    """Open all paths; close all of them on exit."""
    raise NotImplementedError("Lab 1.14: implement open_many()")
    yield []

Run the checks. The cell writes three small files.

In [870]:
part_paths = []
for part_number in (1, 2, 3):
    part_path = scratch_dir / f"lab_1_14_part_{part_number}.txt"
    part_path.write_text(f"part {part_number}\n", encoding="utf-8")
    part_paths.append(part_path)


def make_recording_opener() -> tuple[Callable[[Path], IO[str]], list[IO[str]]]:
    """An opener that remembers every file it opened."""
    opened: list[IO[str]] = []

    def recording_opener(path: Path) -> IO[str]:
        stream = open(path, encoding="utf-8")  # closed by open_many
        opened.append(stream)
        return stream

    return recording_opener, opened


def read_all_parts() -> tuple[list[str], bool]:
    """Read every part; afterwards all files must be closed."""
    opener, opened = make_recording_opener()
    with open_many(part_paths, opener) as streams:
        contents = [stream.read().strip() for stream in streams]
    return contents, all(stream.closed for stream in opened)


def partial_failure() -> tuple[str, int, bool]:
    """The third path is missing: the two opened files must be closed."""
    opener, opened = make_recording_opener()
    paths = [*part_paths[:2], scratch_dir / "lab_1_14_missing_part.txt"]

    def open_them() -> None:
        with open_many(paths, opener):
            pass

    return raised_error_name(open_them), len(opened), all(s.closed for s in opened)


def closed_after_body_error() -> tuple[str, bool]:
    """The body raises: files are closed and the error propagates."""
    opener, opened = make_recording_opener()

    def failing_block() -> None:
        with open_many(part_paths, opener):
            raise KeyError("body failed")

    return raised_error_name(failing_block), all(s.closed for s in opened)


def no_paths() -> list[IO[str]]:
    """An empty iterable yields an empty list."""
    with open_many([]) as streams:
        return streams


def default_opener() -> list[str]:
    """Without an opener, files are opened as UTF-8 text."""
    with open_many(part_paths[:1]) as streams:
        return [stream.read().strip() for stream in streams]


run_checks(
    [
        ("reads and closes", read_all_parts, (["part 1", "part 2", "part 3"], True)),
        ("failed open closes the others", partial_failure, ("FileNotFoundError", 2, True)),
        ("closed after a body error", closed_after_body_error, ("KeyError", True)),
        ("default opener", default_opener, ["part 1"]),
        ("no paths", no_paths, []),
    ]
)

FAIL  reads and closes: raised NotImplementedError: Lab 1.14: implement open_many()
FAIL  failed open closes the others: expected ('FileNotFoundError', 2, True), got ('NotImplementedError', 0, True)
FAIL  closed after a body error: expected ('KeyError', True), got ('NotImplementedError', True)
FAIL  default opener: raised NotImplementedError: Lab 1.14: implement open_many()
FAIL  no paths: raised NotImplementedError: Lab 1.14: implement open_many()
0/5 checks passed


#### Exercise 6 — `validate_records` raising an `ExceptionGroup`, handled with `except*`

**`validate_records(records)`** checks every field of every record and collects
**all** problems instead of stopping at the first:

| Field | Rule | Error |
| --- | --- | --- |
| missing field | `name`, `age` and `email` are required | `KeyError(field_name)` |
| `name` | a `str` that is not blank | `ValueError("name must be a non-empty string")` |
| `age` | an `int` (not a `bool`) between 0 and 150 | `ValueError("age must be an integer between 0 and 150")` |
| `email` | a `str` containing `@` | `ValueError("email must contain '@'")` |

Check the fields in the order name, age, email. Add the note `f"record {index}"`
(0-based) to each error. If there are errors, raise
`ExceptionGroup("invalid records", errors)`; otherwise return the same `records` list.

**`summarize_validation(records)`** calls it and handles the group with **`except*`**:
`except* KeyError` builds `"missing"` entries `f"{note}: {field_name}"`, and
`except* ValueError` builds `"invalid"` entries `f"{note}: {message}"`. Return
`{"missing": [...], "invalid": [...]}` (empty lists when everything is valid).

In [871]:
def validate_records(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Return records unchanged, or raise an ExceptionGroup of every problem."""
    raise NotImplementedError("Lab 1.14: implement validate_records()")


def summarize_validation(records: list[dict[str, Any]]) -> dict[str, list[str]]:
    """Handle the ExceptionGroup with except* and summarize it."""
    raise NotImplementedError("Lab 1.14: implement summarize_validation()")

Run the checks.

In [872]:
sample_signups = [
    {"name": "Asha", "age": 31, "email": "asha@example.com"},
    {"name": "  ", "age": 200, "email": "ravi@example.com"},
    {"name": "Meena", "email": "meena.example.com"},
    {"name": "John", "age": True, "email": "john@example.com"},
]
valid_signups = sample_signups[:1]


def group_shape(thunk: Callable[[], object]) -> tuple[str, str, int, list[str], int]:
    """(type, message, leaf count, first leaf's notes, KeyError count)."""
    error = caught_error(thunk)
    if not isinstance(error, ExceptionGroup):
        return (type(error).__name__, str(error), 0, [], 0)
    missing_group = error.subgroup(KeyError)
    return (
        type(error).__name__,
        error.message,
        len(error.exceptions),
        error.exceptions[0].__notes__,
        len(missing_group.exceptions) if missing_group else 0,
    )


run_checks(
    [
        (
            "valid returns the same list",
            lambda: validate_records(valid_signups) is valid_signups,
            True,
        ),
        (
            "group of every problem",
            lambda: group_shape(lambda: validate_records(sample_signups)),
            ("ExceptionGroup", "invalid records", 5, ["record 1"], 1),
        ),
        (
            "summary",
            lambda: summarize_validation(sample_signups),
            {
                "missing": ["record 2: age"],
                "invalid": [
                    "record 1: name must be a non-empty string",
                    "record 1: age must be an integer between 0 and 150",
                    "record 2: email must contain '@'",
                    "record 3: age must be an integer between 0 and 150",
                ],
            },
        ),
        (
            "summary of valid records",
            lambda: summarize_validation(valid_signups),
            {"missing": [], "invalid": []},
        ),
        ("empty input", lambda: validate_records([]), []),
    ]
)

FAIL  valid returns the same list: raised NotImplementedError: Lab 1.14: implement validate_records()
FAIL  group of every problem: expected ('ExceptionGroup', 'invalid records', 5, ['record 1'], 1), got ('NotImplementedError', 'Lab 1.14: implement validate_records()', 0, [], 0)
FAIL  summary: raised NotImplementedError: Lab 1.14: implement summarize_validation()
FAIL  summary of valid records: raised NotImplementedError: Lab 1.14: implement summarize_validation()
FAIL  empty input: raised NotImplementedError: Lab 1.14: implement validate_records()
0/5 checks passed


### Recap — Unit 1.14

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Hierarchy | Catch `Exception` subclasses; `SystemExit` / `KeyboardInterrupt` are not errors | `Error` subclasses |
| Matching | `except C` matches when `C` is in the error's `__mro__`; first matching clause wins | `instanceof` checks in `catch` |
| `else` / `finally` | `else` only without an exception; `finally` on every exit | `finally` |
| 3.14 changes | `except A, B:` without parentheses; `return` in `finally` warns | — |
| `as error` | deleted after the clause (breaks the traceback cycle) | block-scoped `catch (e)` |
| Chaining | `from` sets `__cause__`, raising in `except` sets `__context__`, `from None` hides it | `new Error(msg, { cause })` |
| Exception objects | `args`, `__traceback__`, `add_note`, `traceback.format_exception` | `err.stack` |
| Custom hierarchy | One `AppError` base, subclasses carry status codes and fields | `class AppError extends Error` |
| Exception groups | `ExceptionGroup` + `except*`, `split`, `subgroup` | `AggregateError` |
| EAFP vs LBYL | `try` is free since 3.11, raising is not; LBYL for common misses and side effects | — |
| Logging | `logger.exception` inside `except`; never swallow errors | `console.error(err)` |
| `with` | `__enter__` / `__exit__`, truthy return suppresses, exits LIFO | `try/finally`, `using` (TC39) |
| `contextlib` | `@contextmanager` + `finally`, `suppress`, `closing`, `ExitStack`, … | — |
| `assert` / `warnings` | asserts vanish under `-O`; warnings are filterable, can be made errors | `console.assert`, `console.warn` |
| Bytecode | Exception table instead of setup opcodes; `LOAD_SPECIAL` + `WITH_EXCEPT_START` for `with` | — |

**Further reading**

- [Tutorial: errors and exceptions](https://docs.python.org/3/tutorial/errors.html),
  [built-in exceptions](https://docs.python.org/3/library/exceptions.html),
  [the `try` statement](https://docs.python.org/3/reference/compound_stmts.html#the-try-statement)
- [`traceback`](https://docs.python.org/3/library/traceback.html),
  [`logging`](https://docs.python.org/3/library/logging.html),
  [Logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [The `with` statement](https://docs.python.org/3/reference/compound_stmts.html#the-with-statement),
  [context manager types](https://docs.python.org/3/library/stdtypes.html#typecontextmanager),
  [`contextlib`](https://docs.python.org/3/library/contextlib.html)
- [`warnings`](https://docs.python.org/3/library/warnings.html),
  [the `assert` statement](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement),
  [`dis`](https://docs.python.org/3/library/dis.html)
- [PEP 343 — The `with` statement](https://peps.python.org/pep-0343/),
  [PEP 3134 — Exception chaining](https://peps.python.org/pep-3134/),
  [PEP 654 — Exception groups and `except*`](https://peps.python.org/pep-0654/),
  [PEP 678 — Notes](https://peps.python.org/pep-0678/)
- [PEP 758 — `except` without parentheses](https://peps.python.org/pep-0758/),
  [PEP 765 — Disallow `return`/`break`/`continue` that exit `finally`](https://peps.python.org/pep-0765/)

## Phase 1 Project — Log Analyzer

Build a small but complete tool that analyzes a web server's access log, using only
what Phase 1 covered. The pipeline:

```mermaid
graph LR
    A[write_synthetic_log] --> B[read_lines]
    B --> C[parse_records]
    C -->|AccessRecord| D[aggregate]
    C -->|MalformedLineError| E[malformed list]
    D --> F[render_report]
    E --> F
    F --> G["upload_report (@retry)"]
```

<details><summary>ASCII fallback</summary>

```text
write_synthetic_log -> read_lines -> parse_records --AccessRecord--------> aggregate
                                          |                                    |
                                          +--MalformedLineError--> list         |
                                                                    |          v
                                                                    +--> render_report
                                                                             |
                                                                             v
                                                                  upload_report (@retry)
```

</details>

| Step | Requirement | Phase 1 tools |
| --- | --- | --- |
| 1 | Generate a seeded synthetic access log, with malformed lines | `random.Random(seed)` (1.3), f-strings (1.4), `with open` (1.14) |
| 2 | Read it lazily with generators | generator functions (1.13), `enumerate` (1.9) |
| 3 | Parse with `re` and `match` | named groups, `re.fullmatch` (1.4), `match` with guards (1.8), `NamedTuple` (1.5) |
| 4 | Collect `MalformedLineError`s without stopping | custom exceptions (1.14.5), `try` / `except` / `else` (1.14.2) |
| 5 | Status codes, top endpoints, p95 latency | `Counter`, `defaultdict` (1.6), `sorted` with a key (1.5), `re.sub`, `@timer` (1.12) |
| 6 | Print an aligned report | format specs (1.4.3) |
| 7 | Time the whole run with a context manager | `Timer` (1.14.9) |
| 8 | Wrap a flaky `upload_report()` in `@retry`; strict mode | `retry` (1.12.6), `ExceptionGroup` and `except*` (1.14.6) |
| 9 | Self-check | `run_checks` (shared setup) |

The setup cell imports what the project needs, so this section runs on its own after
the shared setup cell.

In [873]:
import datetime as dt
import functools
import hashlib
import inspect
import itertools
import math
import random
import re
import sys
import time
import tracemalloc
from collections import Counter, defaultdict
from collections.abc import Callable, Iterable, Iterator
from pathlib import Path
from typing import Any, NamedTuple


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


scratch_dir = find_repo_root() / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


**Reused helpers.** These three are copied from earlier units, unchanged, so the
project does not depend on cells far above (Lab 1.12's stubs also shadow the names
`timer` and `retry`):

| Helper | From | Used for |
| --- | --- | --- |
| `timer(unit=, report=)` | 1.12.3 | timing `aggregate` |
| `retry(times, exceptions=, first_delay=, backoff=, sleep=)` | 1.12.6 | the flaky upload |
| `Timer(label, report=)` | 1.14.9 | timing the whole run |

In [874]:
TIME_UNITS = {"s": 1.0, "ms": 1_000.0, "us": 1_000_000.0}


def timer(
    func: Callable[..., Any] | None = None,
    /,
    *,
    unit: str = "ms",
    report: Callable[[str], object] = print,
) -> Callable[..., Any]:
    """Report how long each call takes. Use as @timer or @timer(unit="s")."""
    if unit not in TIME_UNITS:
        raise ValueError(f"unit must be one of {sorted(TIME_UNITS)}, got {unit!r}")

    def decorator(target: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(target)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            started = time.perf_counter()
            try:
                return target(*args, **kwargs)
            finally:
                elapsed = (time.perf_counter() - started) * TIME_UNITS[unit]
                report(f"{target.__qualname__} took {elapsed:.3f} {unit}")

        return wrapper

    return decorator if func is None else decorator(func)


def retry(
    times: int = 3,
    *,
    exceptions: tuple[type[Exception], ...] = (Exception,),
    first_delay: float = 0.1,
    backoff: float = 2.0,
    sleep: Callable[[float], object] = time.sleep,
) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Retry on the listed exceptions, sleeping first_delay * backoff**n in between."""
    if times < 1:
        raise ValueError("times must be at least 1")

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
                    sleep(first_delay * backoff ** (attempt - 1))
            raise AssertionError("unreachable")

        return wrapper

    return decorator


class Timer:
    """Time a with block and report "<label> took <ms> ms"."""

    def __init__(self, label: str, report: Callable[[str], object] = print) -> None:
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        self._started = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        self.elapsed_seconds = time.perf_counter() - self._started
        self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")

### Step 1 — Generate a seeded synthetic access log

Each well-formed line has six fields:

```text
2026-11-06T09:00:00Z 10.0.2.17 "GET /api/products/381 HTTP/1.1" 200 23ms
timestamp            client IP  method path              protocol  status latency
```

The generator uses its own `random.Random(seed)` instance (not the global `random`),
so the same seed always gives the same file and nothing else in the notebook is
affected. Endpoints, status codes and latencies are drawn with weights; latency is
log-normal around a typical value per endpoint (real latencies have a long tail), and
5xx responses are slower. About 2% of lines are deliberately broken in one of four
ways. The function returns a `Counter` of what it injected, so the analysis can be
checked against it.

In [875]:
ENDPOINTS = [  # (method, path template, weight, typical latency in ms)
    ("GET", "/api/products", 30, 40),
    ("GET", "/api/products/{id}", 25, 25),
    ("POST", "/api/cart/items", 12, 60),
    ("GET", "/api/orders/{id}", 10, 80),
    ("GET", "/health", 10, 2),
    ("POST", "/api/checkout", 6, 250),
    ("DELETE", "/api/cart/items/{id}", 4, 30),
    ("PUT", "/api/users/{id}/address", 3, 90),
]
STATUS_WEIGHTS = {
    200: 800,
    201: 50,
    204: 20,
    304: 30,
    400: 30,
    401: 10,
    404: 30,
    429: 10,
    500: 10,
    502: 5,
    503: 5,
}
MALFORMED_KINDS = ("truncated", "garbage", "unknown method", "status out of range")
GARBAGE_LINES = ("-", "<html>502 Bad Gateway</html>", "Connection reset by peer", "GET /")


def well_formed_line(rng: random.Random, timestamp: dt.datetime) -> str:
    """One valid access-log line drawn from the weighted tables."""
    method, template, _, typical_ms = rng.choices(ENDPOINTS, weights=[e[2] for e in ENDPOINTS])[0]
    path = template.replace("{id}", str(rng.randint(1, 5_000)))
    if template == "/api/products" and rng.random() < 0.3:
        path += f"?page={rng.randint(2, 9)}"
    status = rng.choices(list(STATUS_WEIGHTS), weights=list(STATUS_WEIGHTS.values()))[0]
    latency_ms = max(1, round(rng.lognormvariate(math.log(typical_ms), 0.6)))
    if status >= 500:
        latency_ms *= 5
    client_ip = f"10.0.{rng.randint(0, 3)}.{rng.randint(1, 254)}"
    stamp = timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")
    return f'{stamp} {client_ip} "{method} {path} HTTP/1.1" {status} {latency_ms}ms'

`break_line` produces the four kinds of damage: a line cut short (a crashed writer),
a garbage line (an upstream error page), an unknown HTTP method, and an impossible
status code. The last two still *look* like log lines, so only the `match` step can
reject them.

In [876]:
def break_line(rng: random.Random, line: str, kind: str) -> str:
    """Damage a valid line in the given way."""
    match kind:
        case "truncated":
            return line[: rng.randint(10, len(line) - 6)]
        case "garbage":
            return rng.choice(GARBAGE_LINES)
        case "unknown method":
            return re.sub(r'"[A-Z]+ ', f'"{rng.choice(["FETCH", "BREW"])} ', line, count=1)
        case "status out of range":
            return re.sub(r'" \d{3} ', f'" {rng.choice(["000", "999"])} ', line, count=1)
        case _:
            raise ValueError(f"unknown kind: {kind!r}")


def write_synthetic_log(
    path: Path, line_count: int, *, seed: int = 42, malformed_rate: float = 0.02
) -> Counter[str]:
    """Write line_count lines to path; return how many of each malformed kind were written."""
    rng = random.Random(seed)
    timestamp = dt.datetime(2026, 11, 6, 9, 0, 0)
    injected: Counter[str] = Counter()
    with path.open("w", encoding="utf-8") as log_file:
        for _ in range(line_count):
            timestamp += dt.timedelta(milliseconds=rng.randint(100, 1_500))
            line = well_formed_line(rng, timestamp)
            if rng.random() < malformed_rate:
                kind = rng.choice(MALFORMED_KINDS)
                injected[kind] += 1
                line = break_line(rng, line, kind)
            log_file.write(line + "\n")
    return injected

Write 20 000 lines to the scratch folder and look at the first few, and at the file
size. Because the generator is seeded, these lines are identical on every run.

In [877]:
LOG_LINE_COUNT = 20_000
log_path = scratch_dir / "phase1_access.log"
injected_kinds = write_synthetic_log(log_path, LOG_LINE_COUNT, seed=42)

injected_total = sum(injected_kinds.values())
print(f"{injected_total} malformed lines injected: {dict(sorted(injected_kinds.items()))}")
print(f"file size: {log_path.stat().st_size / 1024:,.0f} KiB")
with log_path.open(encoding="utf-8") as preview_file:
    for line in itertools.islice(preview_file, 4):
        print(line, end="")

431 malformed lines injected: {'garbage': 100, 'status out of range': 110, 'truncated': 116, 'unknown method': 105}
file size: 1,400 KiB
2026-11-06T09:00:01Z 10.0.0.152 "GET /api/products?page=4 HTTP/1.1" 200 216ms
2026-11-06T09:00:01Z 10.0.3.57 "GET /api/products HTTP/1.1" 200 17ms
2026-11-06T09:00:02Z 10.0.2.40 "GET /health HTTP/1.1" 200 1ms
2026-11-06T09:00:03Z 10.0.0.250 "GET /api/products?page=7 HTTP/1.1" 200 129ms


### Step 2 — Read it lazily with generators

`read_lines` is a generator function (Unit 1.13): calling it does nothing until the
first `next()`, then it opens the file and yields one `(line_number, line)` pair at a
time. Line numbers start at 1 (as editors show them) and blank lines are skipped. The
`with` inside the generator closes the file when iteration ends, or when the
generator is closed or garbage-collected early.

In [878]:
def read_lines(path: Path) -> Iterator[tuple[int, str]]:
    """Yield (line_number, line) for each non-blank line, without the newline."""
    with path.open(encoding="utf-8") as log_file:
        for line_number, raw_line in enumerate(log_file, start=1):
            line = raw_line.rstrip("\n")
            if line.strip():
                yield line_number, line


numbered_lines = read_lines(log_path)
print(f"{inspect.isgenerator(numbered_lines)=}, {inspect.getgeneratorstate(numbered_lines)=}")
print(next(numbered_lines))
print(f"{inspect.getgeneratorstate(numbered_lines)=}")
numbered_lines.close()
print(f"after close(): {inspect.getgeneratorstate(numbered_lines)=}")

inspect.isgenerator(numbered_lines)=True, inspect.getgeneratorstate(numbered_lines)='GEN_CREATED'


(1, '2026-11-06T09:00:01Z 10.0.0.152 "GET /api/products?page=4 HTTP/1.1" 200 216ms')
inspect.getgeneratorstate(numbered_lines)='GEN_SUSPENDED'
after close(): inspect.getgeneratorstate(numbered_lines)='GEN_CLOSED'


**Proof that it is lazy.** Counting the lines through the generator holds one line at
a time; reading the file eagerly (`read_text().splitlines()`) holds all of it. The
cell measures both peaks with `tracemalloc`. The streaming peak (roughly the text
reader's buffers, about 150 kB here) stays the same for any file size; the eager peak
grows with the file. Exact bytes vary by version, so the cell prints the comparison.

In [879]:
def traced_peak(thunk: Callable[[], object]) -> tuple[object, int]:
    """Run thunk under tracemalloc; return (result, peak bytes)."""
    tracemalloc.start()
    try:
        result = thunk()
        return result, tracemalloc.get_traced_memory()[1]
    finally:
        tracemalloc.stop()


streamed_count, streaming_peak = traced_peak(lambda: sum(1 for _ in read_lines(log_path)))
eager_count, eager_peak = traced_peak(
    lambda: len(log_path.read_text(encoding="utf-8").splitlines())
)
print(f"{streamed_count=}, {eager_count=}")
print(f"streaming peak at least 10x smaller than eager: {streaming_peak * 10 < eager_peak}")

streamed_count=20000, eager_count=20000
streaming peak at least 10x smaller than eager: True


### Step 3 — Parse with `re` and `match`

Parsing happens in two stages:

1. **`re.fullmatch`** checks the *shape* of the line and extracts the fields with
   named groups. Anything that does not have the shape (truncated, garbage) fails
   here.
2. A **`match`** statement on `(method, status)` applies the *rules* with guards:
   only real HTTP methods, only status codes 100–599.

Every rejection raises `MalformedLineError`, which carries the line number, the raw
line and a short `reason`. It derives from a project base class and from
`ValueError` (the pattern from 1.14.5). Good lines become an `AccessRecord`
`NamedTuple` (1.5.6).

In [880]:
class LogAnalyzerError(Exception):
    """Base class for errors raised by the log analyzer."""


class MalformedLineError(LogAnalyzerError, ValueError):
    """A log line that could not be parsed."""

    def __init__(self, line_number: int, line: str, reason: str) -> None:
        super().__init__(f"line {line_number}: {reason}")
        self.line_number = line_number
        self.line = line
        self.reason = reason


class AccessRecord(NamedTuple):
    """One parsed access-log line."""

    timestamp: str
    client_ip: str
    method: str
    path: str
    status: int
    latency_ms: int

The pattern, piece by piece (adjacent raw strings are joined by the compiler, so each
field gets its own line):

In [881]:
ACCESS_LINE_PATTERN = re.compile(
    r"(?P<timestamp>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z)"
    r" (?P<client_ip>\d{1,3}(?:\.\d{1,3}){3})"
    r' "(?P<method>[A-Z]+) (?P<path>/\S*) HTTP/1\.1"'
    r" (?P<status>\d{3})"
    r" (?P<latency_ms>\d+)ms"
)
HTTP_METHODS = frozenset({"GET", "HEAD", "POST", "PUT", "PATCH", "DELETE", "OPTIONS"})


def parse_access_line(line_number: int, line: str) -> AccessRecord:
    """Parse one line, or raise MalformedLineError with the reason."""
    found = ACCESS_LINE_PATTERN.fullmatch(line)
    if found is None:
        raise MalformedLineError(line_number, line, "unrecognized format")
    match (found["method"], int(found["status"])):
        case (method, _) if method not in HTTP_METHODS:
            raise MalformedLineError(line_number, line, "unknown method")
        case (_, status) if not 100 <= status <= 599:
            raise MalformedLineError(line_number, line, "status out of range")
        case (method, status):
            return AccessRecord(
                found["timestamp"],
                found["client_ip"],
                method,
                found["path"],
                status,
                int(found["latency_ms"]),
            )

Trying the parser on one line of each kind. The broken lines raise, and the cell
prints the reason of each `MalformedLineError`:

In [882]:
sample_lines = [
    '2026-11-06T09:00:00Z 10.0.2.17 "GET /api/products/381 HTTP/1.1" 200 23ms',
    '2026-11-06T09:00:01Z 10.0.1.5 "POST /api/check',
    "<html>502 Bad Gateway</html>",
    '2026-11-06T09:00:02Z 10.0.3.9 "BREW /api/products HTTP/1.1" 200 41ms',
    '2026-11-06T09:00:03Z 10.0.0.4 "GET /health HTTP/1.1" 999 2ms',
]
for sample_number, sample_line in enumerate(sample_lines, start=1):
    try:
        print(parse_access_line(sample_number, sample_line))
    except MalformedLineError as error:
        print(f"{type(error).__name__}: {error}")

AccessRecord(timestamp='2026-11-06T09:00:00Z', client_ip='10.0.2.17', method='GET', path='/api/products/381', status=200, latency_ms=23)
MalformedLineError: line 2: unrecognized format
MalformedLineError: line 3: unrecognized format
MalformedLineError: line 4: unknown method
MalformedLineError: line 5: status out of range


### Step 4 — Collect `MalformedLineError`s without stopping

One bad line must not abort the analysis of twenty thousand good ones. `parse_records`
is a generator stage between `read_lines` and the aggregation: it catches **only**
`MalformedLineError` (a bug elsewhere still surfaces), appends it to a list the
caller owns, and moves on. The `yield` sits in the `else` clause, so the `except`
guards only the parsing call (1.14.2).

In [883]:
def parse_records(
    numbered_lines: Iterable[tuple[int, str]], malformed: list[MalformedLineError]
) -> Iterator[AccessRecord]:
    """Yield parsed records; collect malformed lines into the given list."""
    for line_number, line in numbered_lines:
        try:
            record = parse_access_line(line_number, line)
        except MalformedLineError as error:
            malformed.append(error)
        else:
            yield record


sample_malformed: list[MalformedLineError] = []
sample_records = list(parse_records(enumerate(sample_lines, start=1), sample_malformed))
print(f"{len(sample_records)} record(s), {len(sample_malformed)} malformed:")
for error in sample_malformed:
    print(f"  line {error.line_number}: {error.reason:<20} {error.line[:40]!r}")

1 record(s), 4 malformed:
  line 2: unrecognized format  '2026-11-06T09:00:01Z 10.0.1.5 "POST /api'
  line 3: unrecognized format  '<html>502 Bad Gateway</html>'
  line 4: unknown method       '2026-11-06T09:00:02Z 10.0.3.9 "BREW /api'
  line 5: status out of range  '2026-11-06T09:00:03Z 10.0.0.4 "GET /heal'


### Step 5 — Aggregate status codes, top endpoints and p95 latency

Three helpers first:

- **`normalize_endpoint`** groups `/api/orders/17` and `/api/orders/4012` as one
  endpoint, `GET /api/orders/{id}`: it drops the query string and replaces every
  all-digit path segment with `re.sub`.
- **`percentile(values, percent)`** uses the *nearest-rank* method: sort, then take
  the value at rank `ceil(percent / 100 * n)`. The p95 latency is the time that 95%
  of requests beat; unlike the mean, it is not hidden by many fast requests.
- **`status_class`** names a status family with `match` on `status // 100`.

In [884]:
ID_SEGMENT = re.compile(r"/\d+(?=/|$)")


def normalize_endpoint(method: str, path: str) -> str:
    """'GET /api/orders/42?x=1' -> 'GET /api/orders/{id}'."""
    route = ID_SEGMENT.sub("/{id}", path.partition("?")[0])
    return f"{method} {route}"


def percentile(values: Iterable[int], percent: float) -> int:
    """Nearest-rank percentile; ValueError for no values."""
    ordered = sorted(values)
    if not ordered:
        raise ValueError("percentile of no values")
    rank = max(1, math.ceil(percent / 100 * len(ordered)))
    return ordered[rank - 1]


def status_class(status: int) -> str:
    """Name the status family."""
    match status // 100:
        case 1:
            return "informational"
        case 2:
            return "success"
        case 3:
            return "redirect"
        case 4:
            return "client error"
        case _:
            return "server error"


print(normalize_endpoint("GET", "/api/orders/42?expand=items"))
print(normalize_endpoint("PUT", "/api/users/7/address"))
print(f"{percentile(range(1, 101), 95)=}, {percentile([120, 15, 30], 50)=}")
print([status_class(status) for status in (101, 204, 304, 404, 503)])

GET /api/orders/{id}
PUT /api/users/{id}/address
percentile(range(1, 101), 95)=95, percentile([120, 15, 30], 50)=30
['informational', 'success', 'redirect', 'client error', 'server error']


`aggregate` makes **one pass** over the records, so it works on the lazy pipeline
directly:

| Container | Holds |
| --- | --- |
| `Counter[int]` | requests per status code |
| `Counter[str]` | requests per normalized endpoint |
| `defaultdict(list)` | latencies per endpoint (for the per-endpoint p95) |
| `list[int]` | every latency (for the overall p95) |

The latency lists are the only part that grows with the input; a production tool
would use a streaming quantile sketch instead. `@timer` (1.12.3) records how long the
pass takes into `timing_lines`, which Step 7 prints.

In [885]:
class AccessSummary(NamedTuple):
    """Aggregated view of the parsed records."""

    record_count: int
    status_counts: Counter[int]
    endpoint_counts: Counter[str]
    latencies_by_endpoint: dict[str, list[int]]
    all_latencies: list[int]


timing_lines: list[str] = []


@timer(unit="ms", report=timing_lines.append)
def aggregate(records: Iterable[AccessRecord]) -> AccessSummary:
    """Count statuses and endpoints and collect latencies, in one pass."""
    status_counts: Counter[int] = Counter()
    endpoint_counts: Counter[str] = Counter()
    latencies_by_endpoint: defaultdict[str, list[int]] = defaultdict(list)
    all_latencies: list[int] = []
    for record in records:
        endpoint = normalize_endpoint(record.method, record.path)
        status_counts[record.status] += 1
        endpoint_counts[endpoint] += 1
        latencies_by_endpoint[endpoint].append(record.latency_ms)
        all_latencies.append(record.latency_ms)
    return AccessSummary(
        len(all_latencies),
        status_counts,
        endpoint_counts,
        dict(latencies_by_endpoint),
        all_latencies,
    )

`top_endpoints` ranks with `sorted` and a tuple key (count descending, then name
ascending, the pattern of Lab 1.6's `top_k_words`), so ties come out in a stable,
documented order rather than in insertion order as `Counter.most_common` would.

In [886]:
def top_endpoints(summary: AccessSummary, count: int = 5) -> list[tuple[str, int, int]]:
    """(endpoint, requests, p95 ms) for the busiest endpoints."""
    ranked = sorted(summary.endpoint_counts.items(), key=lambda item: (-item[1], item[0]))
    return [
        (endpoint, requests, percentile(summary.latencies_by_endpoint[endpoint], 95))
        for endpoint, requests in ranked[:count]
    ]


sample_summary = aggregate(sample_records)
print(sample_summary.status_counts, top_endpoints(sample_summary))

Counter({200: 1}) [('GET /api/products/{id}', 1, 23)]


### Step 6 — Render an aligned report

Every line of the report is exactly `REPORT_WIDTH` (60) characters, built from format
specs (1.4.3): `<` / `>` alignment, `,` thousands separators, `.1%` percentages, and
`-<60` to fill a section rule with dashes. Malformed examples are cut to fit. The
report is a pure function of its inputs, so it is easy to test (Step 9).

In [887]:
REPORT_WIDTH = 60


def fit(text: str) -> str:
    """Cut or pad text to exactly REPORT_WIDTH characters."""
    if len(text) > REPORT_WIDTH:
        text = text[: REPORT_WIDTH - 1] + "…"
    return f"{text:<{REPORT_WIDTH}}"


def render_report(summary: AccessSummary, malformed: list[MalformedLineError]) -> str:
    """Build the full text report."""
    total_lines = summary.record_count + len(malformed)
    lines = [
        "ACCESS LOG REPORT".center(REPORT_WIDTH, "="),
        f"{'Lines read':<30}{total_lines:>30,}",
        f"{'Records parsed':<30}{summary.record_count:>30,}",
        f"{'Malformed lines':<30}{len(malformed):>30,}",
        f"{'Overall p95 latency (ms)':<30}{percentile(summary.all_latencies, 95):>30,}",
        f"{'Status codes ':-<{REPORT_WIDTH}}",
        f"{'Status':<8}{'Class':<22}{'Requests':>15}{'Share':>15}",
    ]
    for status, requests in sorted(summary.status_counts.items()):
        share = requests / summary.record_count
        lines.append(f"{status:<8}{status_class(status):<22}{requests:>15,}{share:>15.1%}")
    lines.append(f"{'Top endpoints ':-<{REPORT_WIDTH}}")
    lines.append(f"{'Endpoint':<34}{'Requests':>13}{'p95 ms':>13}")
    for endpoint, requests, p95_ms in top_endpoints(summary):
        lines.append(f"{endpoint:<34}{requests:>13,}{p95_ms:>13,}")
    lines.append(f"{'Malformed lines by reason ':-<{REPORT_WIDTH}}")
    for reason, reason_count in sorted(Counter(e.reason for e in malformed).items()):
        lines.append(f"{reason:<34}{reason_count:>26,}")
    for error in malformed[:3]:
        lines.append(fit(f"  e.g. line {error.line_number}: {error.line!r}"))
    lines.append("=" * REPORT_WIDTH)
    return "\n".join(lines)


print(render_report(sample_summary, sample_malformed))

=====================ACCESS LOG REPORT======================
Lines read                                                 5
Records parsed                                             1
Malformed lines                                            4
Overall p95 latency (ms)                                  23
Status codes -----------------------------------------------
Status  Class                        Requests          Share
200     success                             1         100.0%
Top endpoints ----------------------------------------------
Endpoint                               Requests       p95 ms
GET /api/products/{id}                        1           23
Malformed lines by reason ----------------------------------
status out of range                                        1
unknown method                                             1
unrecognized format                                        2
  e.g. line 2: '2026-11-06T09:00:01Z 10.0.1.5 "POST /api/ch…
  e.g. line 3: '<html>50

### Step 7 — Run the analysis, timed with a context manager

`analyze_access_log` wires the stages together. Nothing is read until `aggregate`
pulls the first record; the whole file then streams through
`read_lines → parse_records → aggregate` in a single pass.

With `strict=True` it still collects **every** malformed line first, then raises them
together as one `ExceptionGroup` (1.14.6), so a caller sees all problems at once
instead of only the first. Step 8 handles that with `except*`.

In [888]:
def analyze_access_log(
    path: Path, *, strict: bool = False
) -> tuple[AccessSummary, list[MalformedLineError]]:
    """Stream the log through the pipeline; optionally fail on malformed lines."""
    malformed: list[MalformedLineError] = []
    summary = aggregate(parse_records(read_lines(path), malformed))
    if strict and malformed:
        raise ExceptionGroup(f"{len(malformed)} malformed lines in {path.name}", malformed)
    return summary, malformed

The run itself, inside `with Timer(...)`. The two timing lines at the end (the
`Timer` block and the `@timer` on `aggregate`) vary by machine; the report above them
is identical on every run.

In [889]:
timing_lines.clear()
with Timer("full analysis", report=timing_lines.append):
    access_summary, malformed_lines = analyze_access_log(log_path)
    report_text = render_report(access_summary, malformed_lines)

print(report_text)
print("\n".join(timing_lines))

=====================ACCESS LOG REPORT======================
Lines read                                            20,000
Records parsed                                        19,569
Malformed lines                                          431
Overall p95 latency (ms)                                 225
Status codes -----------------------------------------------
Status  Class                        Requests          Share
200     success                        15,690          80.2%
201     success                           982           5.0%
204     success                           384           2.0%
304     redirect                          612           3.1%
400     client error                      575           2.9%
401     client error                      187           1.0%
404     client error                      580           3.0%
429     client error                      197           1.0%
500     server error                      164           0.8%
502     server error    

### Step 8 — Upload with `@retry`, and strict mode with `except*`

The report goes to a (simulated) report server that drops the first two connections.
`upload_report` is wrapped in `@retry` (1.12.6), which retries only on
`ConnectionError` / `TimeoutError` with exponential backoff. `sleep` records the
delays instead of waiting, as in 1.12.6; in production you would keep the default
`time.sleep`. The server writes what it receives to `scratch_dir`.

In [890]:
class FlakyReportServer:
    """Refuses the first `failures` uploads, then stores the report in a file."""

    def __init__(self, failures: int, destination: Path) -> None:
        self.failures_left = failures
        self.destination = destination
        self.attempts = 0

    def receive(self, report: str) -> int:
        """Store the report and return its size in bytes, or raise ConnectionError."""
        self.attempts += 1
        if self.failures_left > 0:
            self.failures_left -= 1
            raise ConnectionError(f"connection reset (attempt {self.attempts})")
        self.destination.write_text(report + "\n", encoding="utf-8")
        return self.destination.stat().st_size


report_server = FlakyReportServer(failures=2, destination=scratch_dir / "phase1_log_report.txt")
upload_delays: list[float] = []


@retry(
    times=4,
    exceptions=(ConnectionError, TimeoutError),
    first_delay=0.05,
    sleep=upload_delays.append,
)
def upload_report(report: str) -> str:
    """Send the report to the server (flaky)."""
    return f"uploaded {report_server.receive(report):,} bytes"


print(upload_report(report_text))
print(f"{report_server.attempts=}, {upload_delays=}")

uploaded 2,015 bytes
report_server.attempts=3, upload_delays=[0.05, 0.1]


**Strict mode.** A pipeline that must not accept bad input runs with `strict=True`.
The caller handles the whole group with `except* MalformedLineError`: one clause
receives every malformed line at once. Any *other* exception type in the group would
not be swallowed here; it would be re-raised.

In [891]:
try:
    analyze_access_log(log_path, strict=True)
except* MalformedLineError as malformed_group:
    reasons = Counter(error.reason for error in malformed_group.exceptions)
    print(f"strict run rejected: {malformed_group.message}")
    print(f"by reason: {dict(sorted(reasons.items()))}")
    first_problem = malformed_group.exceptions[0]
    print(f"first problem: {first_problem} -> {first_problem.line[:45]!r}")

strict run rejected: 431 malformed lines in phase1_access.log
by reason: {'status out of range': 110, 'unknown method': 105, 'unrecognized format': 216}
first problem: line 64: unrecognized format -> '-'


### Step 9 — Self-check

The project's own tests, with the shared `run_checks`. They tie the output back to the
input: every line is either a record or a malformed line, the malformed reasons match
what Step 1 injected (truncated and garbage lines both fail the regex, as
`"unrecognized format"`), and the report is aligned. All checks should pass.

In [892]:
EXPECTED_REASON = {
    "truncated": "unrecognized format",
    "garbage": "unrecognized format",
    "unknown method": "unknown method",
    "status out of range": "status out of range",
}


def expected_reasons(injected: Counter[str]) -> Counter[str]:
    """Translate injected kinds into the reasons the parser should report."""
    reasons: Counter[str] = Counter()
    for kind, kind_count in injected.items():
        reasons[EXPECTED_REASON[kind]] += kind_count
    return reasons


def same_seed_same_file() -> tuple[bool, bool]:
    """(seed 42 twice gives identical bytes, seed 7 gives different bytes)."""
    digests = []
    for run_number, seed in enumerate((42, 42, 7)):
        check_path = scratch_dir / f"phase1_access_check_{run_number}.log"
        write_synthetic_log(check_path, 500, seed=seed)
        digests.append(hashlib.sha256(check_path.read_bytes()).hexdigest())
        check_path.unlink()
    return digests[0] == digests[1], digests[0] != digests[2]

Run the checks.

In [893]:
run_checks(
    [
        ("deterministic generator", same_seed_same_file, (True, True)),
        (
            "every line accounted for",
            lambda: access_summary.record_count + len(malformed_lines),
            LOG_LINE_COUNT,
        ),
        (
            "reasons match the injected damage",
            lambda: Counter(error.reason for error in malformed_lines),
            expected_reasons(injected_kinds),
        ),
        (
            "status counts add up",
            lambda: sum(access_summary.status_counts.values()),
            access_summary.record_count,
        ),
        ("percentile", lambda: (percentile(range(1, 101), 95), percentile([5], 95)), (95, 5)),
        (
            "percentile of nothing",
            lambda: raised_error_name(lambda: percentile([], 95)),
            "ValueError",
        ),
        (
            "normalize_endpoint",
            lambda: normalize_endpoint("DELETE", "/api/cart/items/12?force=1"),
            "DELETE /api/cart/items/{id}",
        ),
        (
            "parse error type",
            lambda: raised_error_name(lambda: parse_access_line(1, "garbage")),
            "MalformedLineError",
        ),
        (
            "report aligned",
            lambda: {len(line) for line in report_text.splitlines()},
            {REPORT_WIDTH},
        ),
        ("upload retried twice", lambda: (report_server.attempts, upload_delays), (3, [0.05, 0.1])),
        (
            "strict mode raises a group",
            lambda: raised_error_name(lambda: analyze_access_log(log_path, strict=True)),
            "ExceptionGroup",
        ),
        ("pipeline is lazy", lambda: inspect.isgenerator(parse_records([], [])), True),
    ]
)

PASS  deterministic generator
PASS  every line accounted for
PASS  reasons match the injected damage
PASS  status counts add up
PASS  percentile
PASS  percentile of nothing
PASS  normalize_endpoint
PASS  parse error type
PASS  report aligned
PASS  upload retried twice


PASS  strict mode raises a group
PASS  pipeline is lazy
12/12 checks passed


**Extensions to try:** read a `.gz` log with `gzip.open(path, "rt")` (nothing else
changes, because every stage takes an iterable); add a `--since` time filter as one
more generator stage; report requests per minute with
`Counter(record.timestamp[:16] for record in records)`; replace the latency lists
with a fixed-size reservoir sample to make memory constant.

### Phase 1 review — 30 predict-the-output questions

Thirty short snippets, at least two from every unit of the phase. For each one,
decide what it prints **before** opening the answer, then run the proof cell. If a
question surprises you, go back to the unit it names.

| Unit | Questions |
| --- | --- |
| 1.1 | 2 |
| 1.2 | 2 |
| 1.3 | 2 |
| 1.4 | 2 |
| 1.5 | 3 |
| 1.6 | 3 |
| 1.7 | 2 |
| 1.8 | 2 |
| 1.9 | 2 |
| 1.10 | 2 |
| 1.11 | 2 |
| 1.12 | 2 |
| 1.13 | 2 |
| 1.14 | 2 |

#### Question 1 — Unit 1.1

**Predict the output.**

```python
def checkout_total() -> int:
    return subtotal + delivery_fee


subtotal = 400
delivery_fee = 40
print(checkout_total())
```

<details><summary>Answer</summary>

`440`. Compiling the `def` only creates the function; global names inside it are
looked up when it **runs** (`LOAD_GLOBAL`), and by then both exist.

</details>

Run the next cell to check your prediction.

In [894]:
def checkout_total() -> int:
    return subtotal + delivery_fee


subtotal = 400
delivery_fee = 40
print(checkout_total())

440


#### Question 2 — Unit 1.1

**Predict the output.**

```python
print(eval("[n * n for n in range(4)]"))
try:
    eval("total = 5")
except SyntaxError:
    print("SyntaxError")
```

<details><summary>Answer</summary>

`[0, 1, 4, 9]`, then `SyntaxError`. `eval` accepts only an **expression**; an
assignment is a statement (use `exec` for statements, or `:=` for an assignment
expression).

</details>

Run the next cell to check your prediction.

In [895]:
print(eval("[n * n for n in range(4)]"))
try:
    eval("total = 5")
except SyntaxError:
    print("SyntaxError")

[0, 1, 4, 9]
SyntaxError


#### Question 3 — Unit 1.2

**Predict the output.**

```python
prices = [100, 200]
alias = prices
alias += [300]
alias = alias + [400]
print(prices, alias)
```

<details><summary>Answer</summary>

`[100, 200, 300] [100, 200, 300, 400]`. `+=` on a list mutates it in place (both
names see it); `alias + [400]` builds a **new** list and rebinds only `alias`.

</details>

Run the next cell to check your prediction.

In [896]:
prices = [100, 200]
alias = prices
alias += [300]
alias = alias + [400]
print(prices, alias)

[100, 200, 300] [100, 200, 300, 400]


#### Question 4 — Unit 1.2

**Predict the output.**

```python
def add_item(cart: list[str], item: str) -> None:
    cart.append(item)
    cart = ["reset"]


basket = ["rice"]
add_item(basket, "dal")
print(basket)
```

<details><summary>Answer</summary>

`['rice', 'dal']`. The parameter is a second name for the caller's list: `append`
mutates that object, while `cart = [...]` only rebinds the local name (call by object
reference).

</details>

Run the next cell to check your prediction.

In [897]:
def add_item(cart: list[str], item: str) -> None:
    cart.append(item)
    cart = ["reset"]


basket = ["rice"]
add_item(basket, "dal")
print(basket)

['rice', 'dal']


#### Question 5 — Unit 1.3

**Predict the output.**

```python
print(0.1 + 0.2 == 0.3, round(2.5), round(3.5), -7 // 2, -7 % 2)
```

<details><summary>Answer</summary>

`False 2 4 -4 1`. Binary floats cannot store 0.1 exactly; `round` rounds halves to
even (banker's rounding); `//` floors toward minus infinity, and `%` takes the sign of
the divisor.

</details>

Run the next cell to check your prediction.

In [898]:
print(0.1 + 0.2 == 0.3, round(2.5), round(3.5), -7 // 2, -7 % 2)

False 2 4 -4 1


#### Question 6 — Unit 1.3

**Predict the output.**

```python
from decimal import Decimal

print(True + True, isinstance(True, int), Decimal("0.1") * 3 == Decimal("0.3"), bool("False"))
```

<details><summary>Answer</summary>

`2 True True True`. `bool` is a subclass of `int` (`True == 1`); `Decimal` built from
strings is exact; any non-empty string is truthy, including `"False"`.

</details>

Run the next cell to check your prediction.

In [899]:
from decimal import Decimal

print(True + True, isinstance(True, int), Decimal("0.1") * 3 == Decimal("0.3"), bool("False"))

2 True True True


#### Question 7 — Unit 1.4

**Predict the output.**

```python
word = "naïve"
print(len(word), len(word.encode("utf-8")), word[::-1].upper())
```

<details><summary>Answer</summary>

`5 6 EVÏAN`. `len` counts code points (5); UTF-8 needs two bytes for `ï`, so 6 bytes;
slicing and `upper` work on code points too. (A decomposed `ï`, `i` plus a combining
mark, would make the length 6: see 1.4.4.)

</details>

Run the next cell to check your prediction.

In [900]:
word = "naïve"
print(len(word), len(word.encode("utf-8")), word[::-1].upper())

5 6 EVÏAN


#### Question 8 — Unit 1.4

**Predict the output.**

```python
print("a,b,,c".split(","), "a b  c".split(), "-".join("abc"), repr("Hello"[10:]))
```

<details><summary>Answer</summary>

`['a', 'b', '', 'c'] ['a', 'b', 'c'] a-b-c ''`. With a separator, `split` keeps empty
fields; with no argument it splits on runs of whitespace; `join` iterates the string;
slicing past the end returns `''` instead of raising.

</details>

Run the next cell to check your prediction.

In [901]:
print("a,b,,c".split(","), "a b  c".split(), "-".join("abc"), repr("Hello"[10:]))

['a', 'b', '', 'c'] ['a', 'b', 'c'] a-b-c ''


#### Question 9 — Unit 1.5

**Predict the output.**

```python
grid = [[0] * 2] * 2
grid[0][0] = 9
print(grid)
```

<details><summary>Answer</summary>

`[[9, 0], [9, 0]]`. The outer `* 2` copies the **reference** to one inner list, so
both rows are the same object. Build rows with `[[0] * 2 for _ in range(2)]`.

</details>

Run the next cell to check your prediction.

In [902]:
grid = [[0] * 2] * 2
grid[0][0] = 9
print(grid)

[[9, 0], [9, 0]]


#### Question 10 — Unit 1.5

**Predict the output.**

```python
order = ("tea", ["cup"])
try:
    order[1] += ["saucer"]
except TypeError as error:
    print(type(error).__name__)
print(order)
```

<details><summary>Answer</summary>

`TypeError`, then `('tea', ['cup', 'saucer'])`. `+=` first extends the list in place
(that succeeds), then tries to store the result back into the tuple slot (that fails).
The mutation has already happened.

</details>

Run the next cell to check your prediction.

In [903]:
order = ("tea", ["cup"])
try:
    order[1] += ["saucer"]
except TypeError as error:
    print(type(error).__name__)
print(order)

TypeError
('tea', ['cup', 'saucer'])


#### Question 11 — Unit 1.5

**Predict the output.**

```python
names = ["bea", "Al", "cy", "Bo"]
print(sorted(names))
print(sorted(names, key=str.lower))
print(names.sort())
```

<details><summary>Answer</summary>

`['Al', 'Bo', 'bea', 'cy']`, `['Al', 'bea', 'Bo', 'cy']`, `None`. Uppercase letters
sort before lowercase by code point; `key=str.lower` compares case-insensitively;
`list.sort()` sorts in place and returns `None`.

</details>

Run the next cell to check your prediction.

In [904]:
names = ["bea", "Al", "cy", "Bo"]
print(sorted(names))
print(sorted(names, key=str.lower))
print(names.sort())

['Al', 'Bo', 'bea', 'cy']
['Al', 'bea', 'Bo', 'cy']
None


#### Question 12 — Unit 1.6

**Predict the output.**

```python
labels = {1: "int", 1.0: "float", True: "bool"}
print(labels)
```

<details><summary>Answer</summary>

`{1: 'bool'}`. `1`, `1.0` and `True` are equal and hash the same, so they are one
key. The **first** key object is kept, the **last** value wins.

</details>

Run the next cell to check your prediction.

In [905]:
labels = {1: "int", 1.0: "float", True: "bool"}  # noqa: F601 (the point of the question)
print(labels)

{1: 'bool'}


#### Question 13 — Unit 1.6

**Predict the output.**

```python
prices = {"tea": 10, "coffee": 20}
prices["tea"] = 15
prices |= {"juice": 30, "tea": 5}
print(list(prices), prices["tea"])
```

<details><summary>Answer</summary>

`['tea', 'coffee', 'juice'] 5`. Dicts keep insertion order; updating an existing key
(by assignment or `|=`) changes its value but **not** its position.

</details>

Run the next cell to check your prediction.

In [906]:
prices = {"tea": 10, "coffee": 20}
prices["tea"] = 15
prices |= {"juice": 30, "tea": 5}
print(list(prices), prices["tea"])

['tea', 'coffee', 'juice'] 5


#### Question 14 — Unit 1.6

**Predict the output.**

```python
from collections import Counter

letters = Counter("mississippi")
print(letters.most_common(2), letters["z"], "z" in letters)
```

<details><summary>Answer</summary>

`[('i', 4), ('s', 4)] 0 False`. Ties in `most_common` keep first-seen order (`i`
appears before `s`); a missing key reads as `0` but is **not** inserted.

</details>

Run the next cell to check your prediction.

In [907]:
from collections import Counter

letters = Counter("mississippi")
print(letters.most_common(2), letters["z"], "z" in letters)

[('i', 4), ('s', 4)] 0 False


#### Question 15 — Unit 1.7

**Predict the output.**

```python
order = {"items": ["pen"], "total": 10}
snapshot = order.copy()
snapshot["items"].append("ink")
snapshot["total"] = 25
print(order)
```

<details><summary>Answer</summary>

`{'items': ['pen', 'ink'], 'total': 10}`. A shallow copy is a new dict holding the
**same** values: the shared list sees the `append`, while rebinding `total` in the
copy does not touch the original.

</details>

Run the next cell to check your prediction.

In [908]:
order = {"items": ["pen"], "total": 10}
snapshot = order.copy()
snapshot["items"].append("ink")
snapshot["total"] = 25
print(order)

{'items': ['pen', 'ink'], 'total': 10}


#### Question 16 — Unit 1.7

**Predict the output.**

```python
import copy

shared = [1]
pair = [shared, shared]
clone = copy.deepcopy(pair)
clone[0].append(2)
print(clone, clone[0] is clone[1], pair)
```

<details><summary>Answer</summary>

`[[1, 2], [1, 2]] True [[1], [1]]`. `deepcopy` copies the inner list once and uses its
`memo` to reuse that copy, so the clone keeps the original's sharing structure while
sharing nothing with the original.

</details>

Run the next cell to check your prediction.

In [909]:
import copy

shared = [1]
pair = [shared, shared]
clone = copy.deepcopy(pair)
clone[0].append(2)
print(clone, clone[0] is clone[1], pair)

[[1, 2], [1, 2]] True [[1], [1]]


#### Question 17 — Unit 1.8

**Predict the output.**

```python
print(0 or [] or "default", 0 and 1 / 0, None or 0, [] and "never")
```

<details><summary>Answer</summary>

`default 0 0 []`. `or` returns the first truthy operand (or the last one); `and`
returns the first falsy operand (or the last one) and short-circuits, so `1 / 0` never
runs.

</details>

Run the next cell to check your prediction.

In [910]:
print(0 or [] or "default", 0 and 1 / 0, None or 0, [] and "never")

default 0 0 []


#### Question 18 — Unit 1.8

**Predict the output.**

```python
def describe(command: object) -> str:
    match command:
        case [action]:
            return f"one: {action}"
        case [action, *rest]:
            return f"many: {action} +{len(rest)}"
        case str():
            return "text"
        case _:
            return "other"


print(describe("go"), describe(["go"]), describe(("go", "fast", "now")), describe({}))
```

<details><summary>Answer</summary>

`text one: go many: go +2 other`. Sequence patterns match lists and tuples but
deliberately **not** strings, so `"go"` falls through to `str()`; a dict is not a
sequence at all.

</details>

Run the next cell to check your prediction.

In [911]:
def describe(command: object) -> str:
    match command:
        case [action]:
            return f"one: {action}"
        case [action, *rest]:
            return f"many: {action} +{len(rest)}"
        case str():
            return "text"
        case _:
            return "other"


print(describe("go"), describe(["go"]), describe(("go", "fast", "now")), describe({}))

text one: go many: go +2 other


#### Question 19 — Unit 1.9

**Predict the output.**

```python
squares = (n * n for n in range(4))
print(sum(squares), sum(squares))
print(list(zip("ab", [1, 2, 3], strict=False)))
```

<details><summary>Answer</summary>

`14 0`, then `[('a', 1), ('b', 2)]`. A generator expression is single use, so the
second `sum` sees nothing; `zip` stops at the shortest input (unless `strict=True`).

</details>

Run the next cell to check your prediction.

In [912]:
squares = (n * n for n in range(4))
print(sum(squares), sum(squares))
print(list(zip("ab", [1, 2, 3], strict=False)))

14 0
[('a', 1), ('b', 2)]


#### Question 20 — Unit 1.9

**Predict the output.**

```python
print(any([]), all([]), max([], default="none"), list(enumerate("xy", start=1)))
```

<details><summary>Answer</summary>

`False True none [(1, 'x'), (2, 'y')]`. `any` of nothing is `False`, `all` of nothing
is `True` (no counter-example); `max` of an empty iterable needs `default`.

</details>

Run the next cell to check your prediction.

In [913]:
print(any([]), all([]), max([], default="none"), list(enumerate("xy", start=1)))

False True none [(1, 'x'), (2, 'y')]


#### Question 21 — Unit 1.10

**Predict the output.**

```python
def add_tag(tag: str, tags: list[str] = []) -> list[str]:
    tags.append(tag)
    return tags


print(add_tag("a"), add_tag("b"))
```

<details><summary>Answer</summary>

`['a', 'b'] ['a', 'b']`. The default list is created **once**, at `def` time, and
shared by every call. Both calls return that same list, and `print` shows it after
both appends.

</details>

Run the next cell to check your prediction.

In [914]:
def add_tag(tag: str, tags: list[str] = []) -> list[str]:  # noqa: B006 (the trap)
    tags.append(tag)
    return tags


print(add_tag("a"), add_tag("b"))

['a', 'b'] ['a', 'b']


#### Question 22 — Unit 1.10

**Predict the output.**

```python
def book_room(room_number: int, /, guest: str, *, nights: int = 1) -> str:
    return f"{room_number}:{guest}:{nights}"


print(book_room(101, guest="Asha", nights=2))
print(book_room(102, "Ravi"))
try:
    book_room(room_number=103, guest="Meena")
except TypeError:
    print("TypeError")
```

<details><summary>Answer</summary>

`101:Asha:2`, `102:Ravi:1`, `TypeError`. Parameters before `/` are positional-only, so
`room_number=` is rejected; `guest` can be passed either way; `nights` (after `*`) only
by keyword.

</details>

Run the next cell to check your prediction.

In [915]:
def book_room(room_number: int, /, guest: str, *, nights: int = 1) -> str:
    return f"{room_number}:{guest}:{nights}"


print(book_room(101, guest="Asha", nights=2))
print(book_room(102, "Ravi"))
try:
    book_room(room_number=103, guest="Meena")
except TypeError:
    print("TypeError")

101:Asha:2
102:Ravi:1
TypeError


#### Question 23 — Unit 1.11

**Predict the output.**

```python
multipliers = [lambda x: x * factor for factor in range(3)]
print([multiply(10) for multiply in multipliers])
```

<details><summary>Answer</summary>

`[20, 20, 20]`. Each lambda closes over the **variable** `factor`, not its value at
creation; when they run, the loop has finished with `factor == 2` (late binding). Fix
with a default: `lambda x, factor=factor: x * factor`.

</details>

Run the next cell to check your prediction.

In [916]:
multipliers = [lambda x: x * factor for factor in range(3)]  # noqa: B023 (late binding)
print([multiply(10) for multiply in multipliers])

[20, 20, 20]


#### Question 24 — Unit 1.11

**Predict the output.**

```python
visits = 0


def record_visit() -> None:
    visits += 1


try:
    record_visit()
except UnboundLocalError:
    print("UnboundLocalError")
print(visits)
```

<details><summary>Answer</summary>

`UnboundLocalError`, then `0`. Assigning to `visits` anywhere in the function makes it
**local** for the whole body (decided at compile time), so `visits += 1` reads a local
that has no value yet. `global visits` (or better, a parameter) fixes it.

</details>

Run the next cell to check your prediction.

In [917]:
visits = 0


def record_visit() -> None:
    """Increment the visit counter (broken on purpose)."""
    visits += 1  # noqa: F823, F841 (the trap)


try:
    record_visit()
except UnboundLocalError:
    print("UnboundLocalError")
print(visits)

UnboundLocalError
0


#### Question 25 — Unit 1.12

**Predict the output.**

```python
def tag(label):
    def decorator(func):
        def wrapper():
            return f"<{label}>{func()}</{label}>"
        return wrapper
    return decorator


@tag("b")
@tag("i")
def greet():
    return "hi"


print(greet(), greet.__name__)
```

<details><summary>Answer</summary>

`<b><i>hi</i></b> wrapper`. Decorators apply bottom-up (`tag("b")(tag("i")(greet))`),
so the outer tag wraps the inner one. Without `functools.wraps`, the result keeps the
wrapper's name.

</details>

Run the next cell to check your prediction.

In [918]:
def tag(label: str) -> Callable[[Callable[[], str]], Callable[[], str]]:
    """Wrap a function's result in an HTML tag."""

    def decorator(func: Callable[[], str]) -> Callable[[], str]:
        def wrapper() -> str:
            return f"<{label}>{func()}</{label}>"

        return wrapper

    return decorator


@tag("b")
@tag("i")
def greet() -> str:
    """Say hi."""
    return "hi"


print(greet(), greet.__name__)

<b><i>hi</i></b> wrapper


#### Question 26 — Unit 1.12

**Predict the output.**

```python
import functools


@functools.lru_cache(maxsize=128)
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(10), fib.cache_info())
```

<details><summary>Answer</summary>

`55 CacheInfo(hits=8, misses=11, maxsize=128, currsize=11)`. Each of `fib(0)` …
`fib(10)` is computed once (11 misses); for `n` from 3 to 10, the `fib(n - 2)` call
finds the value its sibling branch just cached (8 hits).

</details>

Run the next cell to check your prediction.

In [919]:
import functools


@functools.lru_cache(maxsize=128)
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(10), fib.cache_info())

55 CacheInfo(hits=8, misses=11, maxsize=128, currsize=11)


#### Question 27 — Unit 1.13

**Predict the output.**

```python
def countdown(start: int):
    print("starting")
    while start > 0:
        yield start
        start -= 1


launch = countdown(2)
print("created")
print(list(launch), list(launch))
```

<details><summary>Answer</summary>

`created`, then `starting`, then `[2, 1] []`. Calling a generator function runs none
of its body; the first `next()` (inside `list`) starts it. A generator is single use,
so the second `list` is empty.

</details>

Run the next cell to check your prediction.

In [920]:
def countdown(start: int) -> Iterator[int]:
    """Yield start, start - 1, ..., 1."""
    print("starting")
    while start > 0:
        yield start
        start -= 1


launch = countdown(2)
print("created")
print(list(launch), list(launch))

created
starting
[2, 1] []


#### Question 28 — Unit 1.13

**Predict the output.**

```python
import itertools

print([(key, len(list(group))) for key, group in itertools.groupby("aabaa")])
print(list(itertools.islice(itertools.count(10, 5), 3)))
```

<details><summary>Answer</summary>

`[('a', 2), ('b', 1), ('a', 2)]`, then `[10, 15, 20]`. `groupby` groups **consecutive**
equal items only (sort first to group globally); `islice` takes a finite slice of an
infinite `count`.

</details>

Run the next cell to check your prediction.

In [921]:
import itertools

print([(key, len(list(group))) for key, group in itertools.groupby("aabaa")])
print(list(itertools.islice(itertools.count(10, 5), 3)))

[('a', 2), ('b', 1), ('a', 2)]
[10, 15, 20]


#### Question 29 — Unit 1.14

**Predict the output.**

```python
def checkout() -> str:
    try:
        print("try")
        return "from try"
    except ValueError:
        return "from except"
    else:
        return "from else"
    finally:
        print("finally")


print(checkout())
```

<details><summary>Answer</summary>

`try`, `finally`, `from try`. `else` runs only when the `try` body finishes
**without** an exception **and** without leaving through `return`/`break`/`continue`;
`finally` runs on the way out, before the caller receives the value.

</details>

Run the next cell to check your prediction.

In [922]:
def checkout() -> str:
    """Return from try; else is skipped, finally runs."""
    try:
        print("try")
        return "from try"
    except ValueError:
        return "from except"
    else:
        return "from else"
    finally:
        print("finally")


print(checkout())

try
finally
from try


#### Question 30 — Unit 1.14

**Predict the output.**

```python
try:
    try:
        {}["sku"]
    except KeyError:
        raise ValueError("bad order") from None
except ValueError as outer:
    print(repr(outer.__cause__), repr(outer.__context__), outer.__suppress_context__)
```

<details><summary>Answer</summary>

`None KeyError('sku') True`. `from None` sets no cause and only **hides** the context:
`__context__` still holds the `KeyError`, and `__suppress_context__` tells the
traceback printer not to show it.

</details>

Run the next cell to check your prediction.

In [923]:
try:
    try:
        {}["sku"]
    except KeyError:
        raise ValueError("bad order") from None
except ValueError as outer:
    print(repr(outer.__cause__), repr(outer.__context__), outer.__suppress_context__)

None KeyError('sku') True


### Phase 1 complete

You can now explain any core Python snippet in terms of objects, names, types and
frames, and you have used every built-in type, the `collections` and `itertools`
toolkits, functions, closures, decorators, generators, exceptions and context
managers, together, in one working tool.

**Next: Phase 2 — Object-Oriented Programming and the Data Model.** Classes,
dataclasses and the dunder methods behind `len()`, `in`, `with` and iteration, so the
built-in behaviour you predicted here becomes behaviour you can give your own types.